In [1]:
#@title 00. Import Stage 5 decision and freeze candidate prospective-test design

from pathlib import Path
from datetime import datetime, timezone
from google.colab import drive

import json
import platform
import sys

import numpy as np
import pandas as pd


# ============================================================
# 1. Mount Google Drive safely
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

else:
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")


assert DRIVE_ROOT.is_dir(), DRIVE_ROOT


# ============================================================
# 2. Project paths
# ============================================================

PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_RECORD_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

FAILURE_AUDIT_ROOT = (
    RETINAL_RECORD_ROOT
    / "External_Replication_Failure_Audit_v0.1"
)

PROSPECTIVE_DESIGN_ROOT = (
    RETINAL_RECORD_ROOT
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
)

PROSPECTIVE_DESIGN_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE5_DECISION_PATH = (
    FAILURE_AUDIT_ROOT
    / "Stage5_Final_Retinal_Feasibility_Decision_v0.1.json"
)

STAGE5_REQUIREMENTS_PATH = (
    FAILURE_AUDIT_ROOT
    / "Stage5_Prospective_Retinal_Test_Requirements_v0.1.csv"
)


assert PROJECT_ROOT.is_dir(), PROJECT_ROOT
assert STAGE5_DECISION_PATH.is_file(), STAGE5_DECISION_PATH
assert STAGE5_REQUIREMENTS_PATH.is_file(), STAGE5_REQUIREMENTS_PATH


# ============================================================
# 3. Import the sealed Stage 5 decision
# ============================================================

with open(
    STAGE5_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage5_decision = json.load(file)


stage5_requirements = pd.read_csv(
    STAGE5_REQUIREMENTS_PATH
)


EXPECTED_STAGE5_DECISION = (
    "RETIRE_BILATERAL_LOCALITY_ENDPOINT_AND_PROCEED_"
    "TO_ONE_REDESIGNED_PROSPECTIVE_RETINAL_BLIND_TEST"
)


assert (
    stage5_decision["decision"]
    ==
    EXPECTED_STAGE5_DECISION
), stage5_decision["decision"]


assert (
    stage5_decision[
        "authorised_cross_modal_expansion"
    ]
    is False
)


assert (
    stage5_decision[
        "authorised_next_step"
    ]
    ==
    "ONE_REDESIGNED_PROSPECTIVE_RETINAL_BLIND_TEST"
)


# ============================================================
# 4. Freeze the scientific endpoint
# ============================================================

endpoint_definition = {
    "endpoint_id": (
        "MODERATE_OR_WORSE_DR_GRADE_GE_2"
    ),
    "endpoint_name": (
        "Moderate-or-worse diabetic retinopathy"
    ),
    "positive_class": (
        "ICDR-equivalent DR grade greater than or equal to 2"
    ),
    "negative_class": (
        "ICDR-equivalent DR grade 0 or 1"
    ),
    "unit_of_analysis": (
        "single fundus image"
    ),
    "primary_metric": (
        "ROC AUC"
    ),
    "secondary_metrics": [
        "average precision",
        "balanced accuracy",
        "sensitivity",
        "specificity",
    ],
    "prohibited_endpoint_names": [
        (
            "Referable DR, unless DME handling is explicitly "
            "and consistently harmonised across every dataset"
        ),
        (
            "Left-versus-right bilateral severity direction"
        ),
    ],
    "reason_for_endpoint": (
        "A grade-threshold endpoint is more consistently "
        "harmonisable across public five-grade DR datasets "
        "than the retired bilateral severity-direction task."
    ),
}


# ============================================================
# 5. Freeze candidate dataset roles
# ============================================================

dataset_rows = [
    {
        "dataset": "EyePACS_2015",
        "candidate_role": "source",
        "current_label_status": (
            "ALREADY_UNBLINDED"
        ),
        "proposed_use": (
            "Train a source-domain moderate-or-worse DR model"
        ),
        "target_evaluation_authorised": False,
        "reason": (
            "Already used and unblinded during the external "
            "replication pilot."
        ),
    },
    {
        "dataset": "DeepDRiD",
        "candidate_role": "source",
        "current_label_status": (
            "ALREADY_UNBLINDED"
        ),
        "proposed_use": (
            "Train an independent source-domain "
            "moderate-or-worse DR model"
        ),
        "target_evaluation_authorised": False,
        "reason": (
            "Already used for discovery and mechanism analysis."
        ),
    },
    {
        "dataset": "APTOS_2019",
        "candidate_role": "target_candidate",
        "current_label_status": (
            "DO_NOT_ACCESS_FOR_MODEL_EVALUATION"
        ),
        "proposed_use": (
            "Candidate prospectively sealed target domain"
        ),
        "target_evaluation_authorised": (
            "PENDING_FEASIBILITY_AND_LABEL_QUARANTINE"
        ),
        "reason": (
            "Five-grade DR dataset with a different acquisition "
            "environment and substantial sample size."
        ),
    },
    {
        "dataset": "IDRiD",
        "candidate_role": "target_candidate",
        "current_label_status": (
            "DO_NOT_ACCESS_FOR_MODEL_EVALUATION"
        ),
        "proposed_use": (
            "Candidate second prospectively sealed target domain"
        ),
        "target_evaluation_authorised": (
            "PENDING_FEASIBILITY_AND_LABEL_QUARANTINE"
        ),
        "reason": (
            "Official image-level DR grading dataset with a "
            "different acquisition cohort and official split."
        ),
    },
    {
        "dataset": "Messidor_2",
        "candidate_role": "excluded_candidate",
        "current_label_status": (
            "NO_OFFICIAL_DR_GROUND_TRUTH"
        ),
        "proposed_use": (
            "Not authorised for the primary confirmatory test"
        ),
        "target_evaluation_authorised": False,
        "reason": (
            "The official dataset does not provide official "
            "DR ground-truth annotations."
        ),
    },
]


candidate_datasets = pd.DataFrame(
    dataset_rows
)


# ============================================================
# 6. Freeze candidate transfer edges
# ============================================================

edge_rows = [
    {
        "edge_id": "E1",
        "source_dataset": "EyePACS_2015",
        "target_dataset": "APTOS_2019",
        "endpoint": endpoint_definition[
            "endpoint_id"
        ],
        "status": (
            "CANDIDATE_PENDING_FEASIBILITY_AUDIT"
        ),
    },
    {
        "edge_id": "E2",
        "source_dataset": "DeepDRiD",
        "target_dataset": "APTOS_2019",
        "endpoint": endpoint_definition[
            "endpoint_id"
        ],
        "status": (
            "CANDIDATE_PENDING_FEASIBILITY_AUDIT"
        ),
    },
    {
        "edge_id": "E3",
        "source_dataset": "EyePACS_2015",
        "target_dataset": "IDRiD",
        "endpoint": endpoint_definition[
            "endpoint_id"
        ],
        "status": (
            "CANDIDATE_PENDING_FEASIBILITY_AUDIT"
        ),
    },
    {
        "edge_id": "E4",
        "source_dataset": "DeepDRiD",
        "target_dataset": "IDRiD",
        "endpoint": endpoint_definition[
            "endpoint_id"
        ],
        "status": (
            "CANDIDATE_PENDING_FEASIBILITY_AUDIT"
        ),
    },
]


candidate_edges = pd.DataFrame(
    edge_rows
)


# ============================================================
# 7. Freeze the sequential test workflow
# ============================================================

workflow_rows = [
    {
        "stage": 0,
        "name": (
            "Candidate-design import and endpoint freeze"
        ),
        "allowed_information": (
            "Stage 5 outputs and public dataset documentation"
        ),
        "target_labels_allowed": False,
        "checkpoint": (
            "Endpoint and candidate roles must be frozen "
            "before dataset acquisition."
        ),
    },
    {
        "stage": 1,
        "name": (
            "Target dataset feasibility and provenance audit"
        ),
        "allowed_information": (
            "File inventory, dataset documentation, image "
            "counts, licensing and non-diagnostic metadata"
        ),
        "target_labels_allowed": False,
        "checkpoint": (
            "Confirm that at least two targets are obtainable, "
            "independent and technically usable."
        ),
    },
    {
        "stage": 2,
        "name": (
            "Label quarantine and deterministic split sealing"
        ),
        "allowed_information": (
            "Labels may be accessed only by the quarantine "
            "procedure to create development and sealed "
            "evaluation partitions."
        ),
        "target_labels_allowed": (
            "QUARANTINE_PROCEDURE_ONLY"
        ),
        "checkpoint": (
            "Save hashes and remove evaluation labels from all "
            "subsequent modelling tables."
        ),
    },
    {
        "stage": 3,
        "name": (
            "Development-only target recoverability gate"
        ),
        "allowed_information": (
            "Source labels and target-development labels only"
        ),
        "target_labels_allowed": (
            "DEVELOPMENT_PARTITION_ONLY"
        ),
        "checkpoint": (
            "Retain only targets whose endpoint is demonstrably "
            "recoverable in development data."
        ),
    },
    {
        "stage": 4,
        "name": (
            "Label-free target-support and baseline extraction"
        ),
        "allowed_information": (
            "Source labels, target images and unlabelled sealed "
            "target manifests"
        ),
        "target_labels_allowed": False,
        "checkpoint": (
            "Compute CDO variables and ordinary baselines "
            "before transfer performance is revealed."
        ),
    },
    {
        "stage": 5,
        "name": (
            "Prospective transfer-performance prediction"
        ),
        "allowed_information": (
            "Source performance, target-support variables and "
            "ordinary baseline variables"
        ),
        "target_labels_allowed": False,
        "checkpoint": (
            "Seal a numerical performance interval, rank order "
            "and predicted failure channel for every edge."
        ),
    },
    {
        "stage": 6,
        "name": (
            "One-time sealed evaluation"
        ),
        "allowed_information": (
            "Previously quarantined evaluation labels"
        ),
        "target_labels_allowed": (
            "ONE_TIME_FINAL_EVALUATION_ONLY"
        ),
        "checkpoint": (
            "Compare predictions with observed transfer and "
            "apply the pre-frozen project stop rule."
        ),
    },
]


prospective_workflow = pd.DataFrame(
    workflow_rows
)


# ============================================================
# 8. Freeze prohibitions
# ============================================================

prohibitions = [
    (
        "Do not access APTOS or IDRiD evaluation labels before "
        "the label-quarantine stage."
    ),
    (
        "Do not change the grade >= 2 endpoint after observing "
        "target transfer performance."
    ),
    (
        "Do not select only the most favourable source-target "
        "edge after evaluation."
    ),
    (
        "Do not describe dataset-domain classification alone "
        "as diagnostic observability prediction."
    ),
    (
        "Do not expand to skin or chest imaging before the "
        "retinal decision is complete."
    ),
    (
        "Do not reuse the retired bilateral severity-direction "
        "endpoint as the principal outcome."
    ),
]


# ============================================================
# 9. Save candidate design artifacts
# ============================================================

ENDPOINT_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Endpoint_Definition_v0.1.json"
)

DATASET_ROLE_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Candidate_Dataset_Roles_v0.1.csv"
)

EDGE_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Candidate_Transfer_Edges_v0.1.csv"
)

WORKFLOW_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Test_Workflow_v0.1.csv"
)

DESIGN_DECISION_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Candidate_Design_Decision_v0.1.json"
)

ENVIRONMENT_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Candidate_Design_Environment_v0.1.json"
)


with open(
    ENDPOINT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        endpoint_definition,
        file,
        indent=2,
    )


candidate_datasets.to_csv(
    DATASET_ROLE_PATH,
    index=False,
)

candidate_edges.to_csv(
    EDGE_PATH,
    index=False,
)

prospective_workflow.to_csv(
    WORKFLOW_PATH,
    index=False,
)


design_decision = {
    "decision": (
        "FREEZE_CANDIDATE_DESIGN_ADVANCE_TO_"
        "TARGET_FEASIBILITY_AUDIT"
    ),
    "status": (
        "CANDIDATE_DATASETS_NOT_YET_ACQUIRED_OR_UNBLINDED"
    ),
    "endpoint": endpoint_definition,
    "candidate_sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
    "candidate_targets": [
        "APTOS_2019",
        "IDRiD",
    ],
    "candidate_edges": (
        candidate_edges.to_dict(
            orient="records"
        )
    ),
    "prohibitions": prohibitions,
    "stage5_stop_rule": (
        stage5_decision["stop_rule"]
    ),
    "important_boundary": (
        "This file freezes candidate roles and workflow only. "
        "Final dataset inclusion remains conditional on a "
        "label-free feasibility and provenance audit."
    ),
}


with open(
    DESIGN_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        design_decision,
        file,
        indent=2,
    )


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "stage5_decision_path": str(
        STAGE5_DECISION_PATH
    ),
    "prospective_design_root": str(
        PROSPECTIVE_DESIGN_ROOT
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 10. Final report
# ============================================================

required_outputs = [
    ENDPOINT_PATH,
    DATASET_ROLE_PATH,
    EDGE_PATH,
    WORKFLOW_PATH,
    DESIGN_DECISION_PATH,
    ENVIRONMENT_PATH,
]

missing_outputs = [
    path
    for path in required_outputs
    if not path.is_file()
]

assert not missing_outputs, (
    "Missing prospective-design outputs:\n"
    +
    "\n".join(
        map(str, missing_outputs)
    )
)


print(
    "================ PROSPECTIVE RETINAL "
    "CANDIDATE DESIGN ================"
)

print("\nImported Stage 5 decision:")
print(
    stage5_decision["decision"]
)

print("\nFrozen endpoint:")
print(
    endpoint_definition["endpoint_name"]
)

print(
    endpoint_definition["positive_class"]
)


print(
    "\n================ CANDIDATE DATASET ROLES "
    "================"
)

display(
    candidate_datasets
)


print(
    "\n================ CANDIDATE TRANSFER EDGES "
    "================"
)

display(
    candidate_edges
)


print(
    "\n================ PROSPECTIVE WORKFLOW "
    "================"
)

display(
    prospective_workflow
)


print(
    "\nCandidate-design decision:"
)

print(
    design_decision["decision"]
)


print(
    "\nOutput root:"
)

print(
    PROSPECTIVE_DESIGN_ROOT
)


print(
    "\nCandidate prospective retinal-test design "
    "frozen successfully."
)

print(
    "No new target dataset or target label was accessed."
)

Mounted at /content/drive
================ PROSPECTIVE RETINAL CANDIDATE DESIGN ================

Imported Stage 5 decision:
RETIRE_BILATERAL_LOCALITY_ENDPOINT_AND_PROCEED_TO_ONE_REDESIGNED_PROSPECTIVE_RETINAL_BLIND_TEST

Frozen endpoint:
Moderate-or-worse diabetic retinopathy
ICDR-equivalent DR grade greater than or equal to 2

================ CANDIDATE DATASET ROLES ================


,dataset,candidate_role,current_label_status,proposed_use,target_evaluation_authorised,reason
0,EyePACS_2015,source,ALREADY_UNBLINDED,Train a source-domain moderate-or-worse DR model,False,Already used and unblinded during the external...
1,DeepDRiD,source,ALREADY_UNBLINDED,Train an independent source-domain moderate-or...,False,Already used for discovery and mechanism analy...
2,APTOS_2019,target_candidate,DO_NOT_ACCESS_FOR_MODEL_EVALUATION,Candidate prospectively sealed target domain,PENDING_FEASIBILITY_AND_LABEL_QUARANTINE,Five-grade DR dataset with a different acquisi...
3,IDRiD,target_candidate,DO_NOT_ACCESS_FOR_MODEL_EVALUATION,Candidate second prospectively sealed target d...,PENDING_FEASIBILITY_AND_LABEL_QUARANTINE,Official image-level DR grading dataset with a...
4,Messidor_2,excluded_candidate,NO_OFFICIAL_DR_GROUND_TRUTH,Not authorised for the primary confirmatory test,False,The official dataset does not provide official...



================ CANDIDATE TRANSFER EDGES ================


,edge_id,source_dataset,target_dataset,endpoint,status
0,E1,EyePACS_2015,APTOS_2019,MODERATE_OR_WORSE_DR_GRADE_GE_2,CANDIDATE_PENDING_FEASIBILITY_AUDIT
1,E2,DeepDRiD,APTOS_2019,MODERATE_OR_WORSE_DR_GRADE_GE_2,CANDIDATE_PENDING_FEASIBILITY_AUDIT
2,E3,EyePACS_2015,IDRiD,MODERATE_OR_WORSE_DR_GRADE_GE_2,CANDIDATE_PENDING_FEASIBILITY_AUDIT
3,E4,DeepDRiD,IDRiD,MODERATE_OR_WORSE_DR_GRADE_GE_2,CANDIDATE_PENDING_FEASIBILITY_AUDIT



================ PROSPECTIVE WORKFLOW ================


,stage,name,allowed_information,target_labels_allowed,checkpoint
0,0,Candidate-design import and endpoint freeze,Stage 5 outputs and public dataset documentation,False,Endpoint and candidate roles must be frozen be...
1,1,Target dataset feasibility and provenance audit,"File inventory, dataset documentation, image c...",False,Confirm that at least two targets are obtainab...
2,2,Label quarantine and deterministic split sealing,Labels may be accessed only by the quarantine ...,QUARANTINE_PROCEDURE_ONLY,Save hashes and remove evaluation labels from ...
3,3,Development-only target recoverability gate,Source labels and target-development labels only,DEVELOPMENT_PARTITION_ONLY,Retain only targets whose endpoint is demonstr...
4,4,Label-free target-support and baseline extraction,"Source labels, target images and unlabelled se...",False,Compute CDO variables and ordinary baselines b...
5,5,Prospective transfer-performance prediction,"Source performance, target-support variables a...",False,"Seal a numerical performance interval, rank or..."
6,6,One-time sealed evaluation,Previously quarantined evaluation labels,ONE_TIME_FINAL_EVALUATION_ONLY,Compare predictions with observed transfer and...



Candidate-design decision:
FREEZE_CANDIDATE_DESIGN_ADVANCE_TO_TARGET_FEASIBILITY_AUDIT

Output root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_Design_v0.1

Candidate prospective retinal-test design frozen successfully.
No new target dataset or target label was accessed.


In [2]:
#@title 01A. Target access, provenance, licensing, and acquisition-feasibility audit — NO DATA DOWNLOAD

from pathlib import Path
from datetime import datetime, timezone
from io import StringIO

import hashlib
import importlib.util
import json
import os
import platform
import re
import subprocess
import sys
import time

import numpy as np
import pandas as pd
import requests


# ============================================================
# 0. Safety boundary
# ============================================================

AUDIT_BOUNDARY = (
    "This stage may inspect public documentation, URL status, "
    "competition file names and sizes, authentication status, "
    "and local file-system inventory. It must not download, "
    "extract, open or parse any target label file."
)

TARGET_LABEL_CONTENT_ACCESSED = False
TARGET_DATA_DOWNLOADED = False


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

try:
    from google.colab import drive
except ImportError:
    drive = None


if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

else:
    assert drive is not None, (
        "Google Drive is not mounted."
    )

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

PROSPECTIVE_DESIGN_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
)

STAGE1_ROOT = (
    PROSPECTIVE_DESIGN_ROOT
    / "Stage1_Target_Feasibility_v0.1"
)

STAGE1_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


DESIGN_DECISION_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Candidate_Design_Decision_v0.1.json"
)

DATASET_ROLE_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Candidate_Dataset_Roles_v0.1.csv"
)

EDGE_PATH = (
    PROSPECTIVE_DESIGN_ROOT
    / "Prospective_Retinal_Candidate_Transfer_Edges_v0.1.csv"
)


required_design_inputs = [
    DESIGN_DECISION_PATH,
    DATASET_ROLE_PATH,
    EDGE_PATH,
]

missing_design_inputs = [
    path
    for path in required_design_inputs
    if not path.is_file()
]

assert not missing_design_inputs, (
    "Missing frozen design files:\n"
    +
    "\n".join(
        map(str, missing_design_inputs)
    )
)


with open(
    DESIGN_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    design_decision = json.load(file)


candidate_datasets = pd.read_csv(
    DATASET_ROLE_PATH
)

candidate_edges = pd.read_csv(
    EDGE_PATH
)


assert (
    design_decision["decision"]
    ==
    "FREEZE_CANDIDATE_DESIGN_ADVANCE_TO_TARGET_FEASIBILITY_AUDIT"
)

assert set(
    design_decision["candidate_targets"]
) == {
    "APTOS_2019",
    "IDRiD",
}


print(
    "================ STAGE 1A BOOTSTRAP ================"
)

print("\nImported design decision:")
print(
    design_decision["decision"]
)

print("\nAudit boundary:")
print(AUDIT_BOUNDARY)


# ============================================================
# 2. Freeze official documentation registry
# ============================================================

documentation_rows = [
    {
        "dataset": "APTOS_2019",
        "resource_id": "APTOS_OVERVIEW",
        "resource_type": (
            "official_competition_overview"
        ),
        "url": (
            "https://www.kaggle.com/competitions/"
            "aptos2019-blindness-detection/overview"
        ),
        "expected_access_mode": (
            "Kaggle account and competition access"
        ),
        "diagnostic_label_content_expected": False,
    },
    {
        "dataset": "APTOS_2019",
        "resource_id": "APTOS_DATA",
        "resource_type": (
            "official_competition_data_page"
        ),
        "url": (
            "https://www.kaggle.com/competitions/"
            "aptos2019-blindness-detection/data"
        ),
        "expected_access_mode": (
            "Kaggle account and accepted competition rules"
        ),
        "diagnostic_label_content_expected": False,
    },
    {
        "dataset": "APTOS_2019",
        "resource_id": "APTOS_RULES",
        "resource_type": (
            "official_competition_rules"
        ),
        "url": (
            "https://www.kaggle.com/competitions/"
            "aptos2019-blindness-detection/rules"
        ),
        "expected_access_mode": (
            "Public page or authenticated Kaggle session"
        ),
        "diagnostic_label_content_expected": False,
    },
    {
        "dataset": "IDRiD",
        "resource_id": "IDRID_DATA",
        "resource_type": (
            "official_dataset_description"
        ),
        "url": (
            "https://idrid.grand-challenge.org/Data/"
        ),
        "expected_access_mode": (
            "Public documentation page"
        ),
        "diagnostic_label_content_expected": False,
    },
    {
        "dataset": "IDRiD",
        "resource_id": "IDRID_RULES",
        "resource_type": (
            "official_challenge_rules"
        ),
        "url": (
            "https://idrid.grand-challenge.org/Rules/"
        ),
        "expected_access_mode": (
            "Public documentation page"
        ),
        "diagnostic_label_content_expected": False,
    },
    {
        "dataset": "IDRiD",
        "resource_id": "IDRID_DATAPORT",
        "resource_type": (
            "official_data_repository"
        ),
        "url": (
            "https://ieee-dataport.org/open-access/"
            "indian-diabetic-retinopathy-image-dataset-idrid"
        ),
        "expected_access_mode": (
            "IEEE DataPort web access; manual authentication "
            "may be required"
        ),
        "diagnostic_label_content_expected": False,
    },
]


documentation_registry = pd.DataFrame(
    documentation_rows
)


# ============================================================
# 3. Probe official URLs without downloading data
# ============================================================

def probe_url(
    url,
    timeout_seconds=20,
):
    record = {
        "url": url,
        "request_method": "GET_STREAM_NO_BODY_ANALYSIS",
        "request_success": False,
        "status_code": None,
        "final_url": None,
        "content_type": None,
        "content_length_header": None,
        "elapsed_seconds": None,
        "access_classification": None,
        "error": None,
    }

    started = time.time()

    try:
        response = requests.get(
            url,
            timeout=timeout_seconds,
            allow_redirects=True,
            stream=True,
            headers={
                "User-Agent": (
                    "Mozilla/5.0 "
                    "CrossModalDiagnosticObservability/"
                    "FeasibilityAudit"
                )
            },
        )

        elapsed = time.time() - started

        record["status_code"] = int(
            response.status_code
        )

        record["final_url"] = str(
            response.url
        )

        record["content_type"] = (
            response.headers.get(
                "content-type"
            )
        )

        record["content_length_header"] = (
            response.headers.get(
                "content-length"
            )
        )

        record["elapsed_seconds"] = float(
            elapsed
        )

        if 200 <= response.status_code < 400:
            record["request_success"] = True
            record["access_classification"] = (
                "ACCESSIBLE"
            )

        elif response.status_code in {
            401,
            403,
        }:
            record["access_classification"] = (
                "REACHABLE_AUTH_OR_AUTOMATION_BLOCKED"
            )

        elif response.status_code == 404:
            record["access_classification"] = (
                "NOT_FOUND"
            )

        else:
            record["access_classification"] = (
                "REMOTE_ERROR"
            )

        response.close()

    except Exception as error:
        record["elapsed_seconds"] = float(
            time.time()
            -
            started
        )

        record["access_classification"] = (
            "REQUEST_FAILED"
        )

        record["error"] = (
            f"{type(error).__name__}: {error}"
        )

    return record


url_probe_rows = []

for _, row in documentation_registry.iterrows():
    probe = probe_url(
        row["url"]
    )

    url_probe_rows.append(
        {
            "dataset": row["dataset"],
            "resource_id": row["resource_id"],
            "resource_type": row[
                "resource_type"
            ],
            **probe,
        }
    )


url_access_audit = pd.DataFrame(
    url_probe_rows
)


# ============================================================
# 4. Detect Kaggle authentication safely
# ============================================================

def detect_kaggle_authentication():
    checks = {
        "environment_token_present": bool(
            os.environ.get(
                "KAGGLE_API_TOKEN"
            )
        ),
        "legacy_username_present": bool(
            os.environ.get(
                "KAGGLE_USERNAME"
            )
        ),
        "legacy_key_present": bool(
            os.environ.get(
                "KAGGLE_KEY"
            )
        ),
        "access_token_file_present": (
            Path.home()
            /
            ".kaggle"
            /
            "access_token"
        ).is_file(),
        "legacy_kaggle_json_present": (
            Path.home()
            /
            ".kaggle"
            /
            "kaggle.json"
        ).is_file(),
    }

    checks["any_authentication_route_present"] = bool(
        any(
            checks.values()
        )
    )

    return checks


kaggle_authentication = (
    detect_kaggle_authentication()
)


# ============================================================
# 5. Ensure official Kaggle CLI is available
# ============================================================

kaggle_install_record = {
    "kaggle_cli_initially_available": (
        importlib.util.find_spec(
            "kaggle"
        )
        is not None
    ),
    "installation_attempted": False,
    "installation_success": False,
    "installation_error": None,
}


if not kaggle_install_record[
    "kaggle_cli_initially_available"
]:
    kaggle_install_record[
        "installation_attempted"
    ] = True

    install_result = subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "kaggle",
        ],
        capture_output=True,
        text=True,
        timeout=180,
    )

    kaggle_install_record[
        "installation_success"
    ] = (
        install_result.returncode == 0
    )

    if install_result.returncode != 0:
        kaggle_install_record[
            "installation_error"
        ] = (
            install_result.stderr[-2000:]
        )

else:
    kaggle_install_record[
        "installation_success"
    ] = True


# ============================================================
# 6. List APTOS official competition files
#    Metadata listing only — no download
# ============================================================

def run_kaggle_file_listing():
    commands = [
        [
            "kaggle",
            "competitions",
            "files",
            "aptos2019-blindness-detection",
            "--page-size",
            "200",
            "-v",
            "-q",
        ],
        [
            "kaggle",
            "competitions",
            "files",
            "-c",
            "aptos2019-blindness-detection",
            "-v",
            "-q",
        ],
    ]

    attempts = []

    for command in commands:
        try:
            result = subprocess.run(
                command,
                capture_output=True,
                text=True,
                timeout=120,
            )

            attempts.append(
                {
                    "command": " ".join(
                        command
                    ),
                    "returncode": int(
                        result.returncode
                    ),
                    "stdout": (
                        result.stdout
                    ),
                    "stderr": (
                        result.stderr
                    ),
                }
            )

            if (
                result.returncode == 0
                and
                result.stdout.strip()
            ):
                return attempts[-1], attempts

        except Exception as error:
            attempts.append(
                {
                    "command": " ".join(
                        command
                    ),
                    "returncode": -1,
                    "stdout": "",
                    "stderr": (
                        f"{type(error).__name__}: "
                        f"{error}"
                    ),
                }
            )

    return attempts[-1], attempts


def parse_kaggle_csv_output(
    stdout_text,
):
    lines = [
        line
        for line in stdout_text.splitlines()
        if line.strip()
    ]

    header_index = None

    for index, line in enumerate(lines):
        normalised = (
            line.lower()
            .replace(" ", "")
        )

        if (
            "name" in normalised
            and
            (
                "size" in normalised
                or
                "totalbytes" in normalised
            )
        ):
            header_index = index
            break

    if header_index is None:
        return pd.DataFrame()

    csv_text = "\n".join(
        lines[
            header_index:
        ]
    )

    try:
        dataframe = pd.read_csv(
            StringIO(
                csv_text
            )
        )

        return dataframe

    except Exception:
        return pd.DataFrame()


if kaggle_install_record[
    "installation_success"
]:
    (
        selected_kaggle_attempt,
        all_kaggle_attempts,
    ) = run_kaggle_file_listing()

else:
    selected_kaggle_attempt = {
        "command": None,
        "returncode": -1,
        "stdout": "",
        "stderr": (
            "Kaggle CLI installation unavailable."
        ),
    }

    all_kaggle_attempts = [
        selected_kaggle_attempt
    ]


aptos_file_listing = parse_kaggle_csv_output(
    selected_kaggle_attempt[
        "stdout"
    ]
)


if not aptos_file_listing.empty:
    aptos_file_listing.columns = [
        str(column).strip()
        for column in aptos_file_listing.columns
    ]


kaggle_error_text = (
    str(
        selected_kaggle_attempt[
            "stderr"
        ]
    )
    +
    "\n"
    +
    str(
        selected_kaggle_attempt[
            "stdout"
        ]
    )
).lower()


if (
    selected_kaggle_attempt[
        "returncode"
    ]
    ==
    0
    and
    not aptos_file_listing.empty
):
    kaggle_access_status = (
        "OFFICIAL_FILE_LISTING_OBTAINED"
    )

elif any(
    token in kaggle_error_text
    for token in [
        "401",
        "unauthorized",
        "authentication",
        "credentials",
        "kaggle.json",
        "api token",
    ]
):
    kaggle_access_status = (
        "KAGGLE_AUTHENTICATION_REQUIRED"
    )

elif any(
    token in kaggle_error_text
    for token in [
        "403",
        "forbidden",
        "accept",
        "rules",
        "permission",
    ]
):
    kaggle_access_status = (
        "COMPETITION_RULE_ACCEPTANCE_OR_PERMISSION_REQUIRED"
    )

else:
    kaggle_access_status = (
        "KAGGLE_ACCESS_UNRESOLVED"
    )


# ============================================================
# 7. Interpret APTOS file-list structure
# ============================================================

def obtain_filename_series(
    dataframe,
):
    if dataframe.empty:
        return pd.Series(
            dtype=str
        )

    candidate_columns = [
        column
        for column in dataframe.columns
        if str(column).strip().lower()
        in {
            "name",
            "filename",
            "file",
        }
    ]

    if not candidate_columns:
        candidate_columns = [
            dataframe.columns[0]
        ]

    return (
        dataframe[
            candidate_columns[0]
        ]
        .astype(str)
        .str.strip()
    )


aptos_filenames = obtain_filename_series(
    aptos_file_listing
)


def contains_file_token(
    filename_series,
    token,
):
    if filename_series.empty:
        return False

    return bool(
        filename_series
        .str.lower()
        .str.contains(
            token.lower(),
            regex=False,
        )
        .any()
    )


aptos_components = {
    "train_image_archive_or_directory_listed": (
        contains_file_token(
            aptos_filenames,
            "train_images",
        )
    ),
    "test_image_archive_or_directory_listed": (
        contains_file_token(
            aptos_filenames,
            "test_images",
        )
    ),
    "training_label_file_listed": (
        contains_file_token(
            aptos_filenames,
            "train.csv",
        )
    ),
    "sample_submission_listed": (
        contains_file_token(
            aptos_filenames,
            "sample_submission",
        )
    ),
}


aptos_quarantine_structure_supported = bool(
    aptos_components[
        "train_image_archive_or_directory_listed"
    ]
    and
    aptos_components[
        "training_label_file_listed"
    ]
)


# ============================================================
# 8. Local artifact inventory
#    Names, paths and sizes only — no contents opened
# ============================================================

LOCAL_SEARCH_ROOTS = [
    PROJECT_ROOT
    / "02_Dataset_Map",
    RETINAL_ROOT,
    Path.home()
    / ".cache"
    / "kagglehub",
]


TARGET_TOKENS = {
    "APTOS_2019": [
        "aptos",
        "blindness-detection",
    ],
    "IDRiD": [
        "idrid",
        "indian-diabetic-retinopathy",
    ],
}


local_inventory_rows = []


for search_root in LOCAL_SEARCH_ROOTS:
    if not search_root.exists():
        continue

    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_path = Path(
            current_root
        )

        current_text = str(
            current_path
        ).lower()

        matching_directory_datasets = [
            dataset_name
            for dataset_name, tokens
            in TARGET_TOKENS.items()
            if any(
                token in current_text
                for token in tokens
            )
        ]

        for filename in filenames:
            file_path = (
                current_path
                /
                filename
            )

            file_text = str(
                file_path
            ).lower()

            matching_datasets = [
                dataset_name
                for dataset_name, tokens
                in TARGET_TOKENS.items()
                if any(
                    token in file_text
                    for token in tokens
                )
            ]

            matching_datasets = sorted(
                set(
                    matching_datasets
                    +
                    matching_directory_datasets
                )
            )

            for dataset_name in matching_datasets:
                try:
                    size_bytes = int(
                        file_path.stat().st_size
                    )
                except OSError:
                    size_bytes = None

                local_inventory_rows.append(
                    {
                        "dataset": dataset_name,
                        "path": str(
                            file_path
                        ),
                        "filename": filename,
                        "suffix": (
                            file_path.suffix.lower()
                        ),
                        "size_bytes": (
                            size_bytes
                        ),
                        "content_opened": False,
                    }
                )


local_target_inventory = pd.DataFrame(
    local_inventory_rows,
    columns=[
        "dataset",
        "path",
        "filename",
        "suffix",
        "size_bytes",
        "content_opened",
    ],
)


# ============================================================
# 9. Frozen official provenance facts
# ============================================================

official_provenance_facts = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "official_host": "Kaggle",
            "official_resource_type": (
                "competition dataset"
            ),
            "nominal_label_scale": (
                "five-grade diabetic-retinopathy severity"
            ),
            "planned_endpoint_compatible": True,
            "license_or_terms": (
                "KAGGLE_COMPETITION_RULES_GOVERNED"
            ),
            "automated_download_authorised_now": False,
            "label_quarantine_requirement": (
                "Training-label CSV must be placed in a "
                "restricted quarantine path before inspection."
            ),
        },
        {
            "dataset": "IDRiD",
            "official_host": (
                "IDRiD Grand Challenge and IEEE DataPort"
            ),
            "official_resource_type": (
                "open research dataset"
            ),
            "nominal_label_scale": (
                "international clinical DR severity grading"
            ),
            "planned_endpoint_compatible": True,
            "license_or_terms": (
                "CC_BY_4_0_REPORTED_BY_OFFICIAL_DATA_PAGE"
            ),
            "automated_download_authorised_now": False,
            "label_quarantine_requirement": (
                "Complete archive must remain sealed until "
                "controlled extraction separates images, "
                "development labels and evaluation labels."
            ),
        },
    ]
)


idrid_public_metadata = {
    "official_image_count": 516,
    "official_image_format": "JPG",
    "official_resolution": "4288x2848",
    "official_field_of_view_degrees": 50,
    "official_license": "CC BY 4.0",
    "official_download_route": "IEEE DataPort",
    "content_accessed_in_this_stage": False,
}


# ============================================================
# 10. Build target feasibility assessment
# ============================================================

aptos_local_count = int(
    (
        local_target_inventory[
            "dataset"
        ]
        ==
        "APTOS_2019"
    ).sum()
) if not local_target_inventory.empty else 0


idrid_local_count = int(
    (
        local_target_inventory[
            "dataset"
        ]
        ==
        "IDRiD"
    ).sum()
) if not local_target_inventory.empty else 0


aptos_documentation_probe = (
    url_access_audit[
        url_access_audit["dataset"]
        ==
        "APTOS_2019"
    ]
)


idrid_documentation_probe = (
    url_access_audit[
        url_access_audit["dataset"]
        ==
        "IDRiD"
    ]
)


aptos_any_remote_reachable = bool(
    aptos_documentation_probe[
        "access_classification"
    ].isin(
        [
            "ACCESSIBLE",
            "REACHABLE_AUTH_OR_AUTOMATION_BLOCKED",
        ]
    ).any()
)


idrid_any_remote_reachable = bool(
    idrid_documentation_probe[
        "access_classification"
    ].isin(
        [
            "ACCESSIBLE",
            "REACHABLE_AUTH_OR_AUTOMATION_BLOCKED",
        ]
    ).any()
)


if (
    kaggle_access_status
    ==
    "OFFICIAL_FILE_LISTING_OBTAINED"
    and
    aptos_quarantine_structure_supported
):
    aptos_status = (
        "FEASIBLE_PENDING_CONTROLLED_DOWNLOAD"
    )

    aptos_blocking_action = (
        "No blocking provenance issue. Perform a controlled "
        "image-and-label acquisition into quarantine."
    )

elif kaggle_access_status in {
    "KAGGLE_AUTHENTICATION_REQUIRED",
    "COMPETITION_RULE_ACCEPTANCE_OR_PERMISSION_REQUIRED",
}:
    aptos_status = (
        "CONDITIONAL_KAGGLE_ACCESS_ACTION_REQUIRED"
    )

    aptos_blocking_action = kaggle_access_status

else:
    aptos_status = (
        "UNRESOLVED_OFFICIAL_FILE_ACCESS"
    )

    aptos_blocking_action = (
        "Resolve Kaggle authentication and competition "
        "file-list access before acquisition."
    )


if idrid_local_count > 0:
    idrid_status = (
        "LOCAL_ARTIFACTS_DETECTED_PENDING_PROVENANCE_CHECK"
    )

    idrid_blocking_action = (
        "Verify that the local files originate from the "
        "official IEEE DataPort release before opening them."
    )

elif idrid_any_remote_reachable:
    idrid_status = (
        "FEASIBLE_PENDING_MANUAL_IEEE_DATAPORT_ACQUISITION"
    )

    idrid_blocking_action = (
        "Acquire the official archive manually or through an "
        "authenticated IEEE DataPort session into quarantine."
    )

else:
    idrid_status = (
        "OFFICIAL_SOURCE_KNOWN_BUT_AUTOMATED_ACCESS_UNRESOLVED"
    )

    idrid_blocking_action = (
        "Open the official IEEE DataPort page manually and "
        "confirm download access before acquisition."
    )


feasibility_assessment = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "official_provenance_verified": True,
            "official_host": "Kaggle competition",
            "remote_resource_reachable": (
                aptos_any_remote_reachable
            ),
            "official_file_listing_status": (
                kaggle_access_status
            ),
            "official_file_rows_listed": int(
                len(
                    aptos_file_listing
                )
            ),
            "local_artifacts_detected": (
                aptos_local_count
            ),
            "label_content_accessed": False,
            "data_downloaded_in_this_stage": False,
            "quarantine_structure_supported": (
                aptos_quarantine_structure_supported
            ),
            "license_or_terms_status": (
                "KAGGLE_COMPETITION_RULES_GOVERNED"
            ),
            "feasibility_status": (
                aptos_status
            ),
            "blocking_or_next_action": (
                aptos_blocking_action
            ),
        },
        {
            "dataset": "IDRiD",
            "official_provenance_verified": True,
            "official_host": (
                "IDRiD Grand Challenge / IEEE DataPort"
            ),
            "remote_resource_reachable": (
                idrid_any_remote_reachable
            ),
            "official_file_listing_status": (
                "NO_AUTOMATED_FILE_LISTING_ATTEMPTED"
            ),
            "official_file_rows_listed": 0,
            "local_artifacts_detected": (
                idrid_local_count
            ),
            "label_content_accessed": False,
            "data_downloaded_in_this_stage": False,
            "quarantine_structure_supported": True,
            "license_or_terms_status": (
                "CC_BY_4_0_OFFICIAL_DOCUMENTATION"
            ),
            "feasibility_status": (
                idrid_status
            ),
            "blocking_or_next_action": (
                idrid_blocking_action
            ),
        },
    ]
)


# ============================================================
# 11. Overall Stage 1A decision
# ============================================================

aptos_not_rejected = (
    aptos_status
    !=
    "UNRESOLVED_OFFICIAL_FILE_ACCESS"
)

idrid_not_rejected = True


if (
    aptos_status
    ==
    "FEASIBLE_PENDING_CONTROLLED_DOWNLOAD"
    and
    idrid_status
    in {
        "FEASIBLE_PENDING_MANUAL_IEEE_DATAPORT_ACQUISITION",
        "LOCAL_ARTIFACTS_DETECTED_PENDING_PROVENANCE_CHECK",
    }
):
    stage1a_decision = (
        "PASS_TWO_TARGETS_ADVANCE_TO_"
        "CONTROLLED_TARGET_ACQUISITION"
    )

    stage1a_interpretation = (
        "Both candidate targets have identifiable official "
        "provenance and a viable controlled-acquisition route. "
        "No target labels were accessed. Acquisition must occur "
        "inside a quarantine structure before any file contents "
        "are inspected."
    )

elif (
    aptos_not_rejected
    and
    idrid_not_rejected
):
    stage1a_decision = (
        "CONDITIONAL_PASS_ACCESS_ACTIONS_REQUIRED_"
        "BEFORE_TARGET_ACQUISITION"
    )

    stage1a_interpretation = (
        "Two candidate targets remain scientifically viable, "
        "but one or more account, rule-acceptance, manual-download "
        "or automated-access actions must be resolved before "
        "controlled acquisition."
    )

else:
    stage1a_decision = (
        "HOLD_TARGET_ACQUISITION_FEASIBILITY_UNRESOLVED"
    )

    stage1a_interpretation = (
        "The current environment does not yet demonstrate a "
        "reliable acquisition route for two candidate targets."
    )


authorised_next_step = (
    "CONTROLLED_TARGET_ACQUISITION_WITH_LABEL_QUARANTINE"
    if stage1a_decision
    ==
    (
        "PASS_TWO_TARGETS_ADVANCE_TO_"
        "CONTROLLED_TARGET_ACQUISITION"
    )
    else
    "RESOLVE_ACCESS_ACTIONS_AND_REPEAT_STAGE_1A"
)


# ============================================================
# 12. Save Stage 1A artifacts
# ============================================================

DOCUMENTATION_REGISTRY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Official_Documentation_Registry_v0.1.csv"
)

URL_ACCESS_PATH = (
    STAGE1_ROOT
    / "Stage1A_Official_URL_Access_Audit_v0.1.csv"
)

APTOS_FILE_LIST_PATH = (
    STAGE1_ROOT
    / "Stage1A_APTOS_Official_File_Listing_Metadata_v0.1.csv"
)

LOCAL_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)

PROVENANCE_FACTS_PATH = (
    STAGE1_ROOT
    / "Stage1A_Frozen_Official_Provenance_Facts_v0.1.csv"
)

FEASIBILITY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Target_Feasibility_Assessment_v0.1.csv"
)

DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1A_Target_Feasibility_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE1_ROOT
    / "Stage1A_Target_Feasibility_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    STAGE1_ROOT
    / "Stage1A_Target_Feasibility_Environment_v0.1.json"
)


documentation_registry.to_csv(
    DOCUMENTATION_REGISTRY_PATH,
    index=False,
)

url_access_audit.to_csv(
    URL_ACCESS_PATH,
    index=False,
)

aptos_file_listing.to_csv(
    APTOS_FILE_LIST_PATH,
    index=False,
)

local_target_inventory.to_csv(
    LOCAL_INVENTORY_PATH,
    index=False,
)

official_provenance_facts.to_csv(
    PROVENANCE_FACTS_PATH,
    index=False,
)

feasibility_assessment.to_csv(
    FEASIBILITY_PATH,
    index=False,
)


decision_payload = {
    "decision": stage1a_decision,
    "interpretation": stage1a_interpretation,
    "authorised_next_step": (
        authorised_next_step
    ),
    "audit_boundary": AUDIT_BOUNDARY,
    "target_label_content_accessed": (
        TARGET_LABEL_CONTENT_ACCESSED
    ),
    "target_data_downloaded": (
        TARGET_DATA_DOWNLOADED
    ),
    "aptos_kaggle_access_status": (
        kaggle_access_status
    ),
    "aptos_components": (
        aptos_components
    ),
    "aptos_quarantine_structure_supported": (
        aptos_quarantine_structure_supported
    ),
    "idrid_public_metadata": (
        idrid_public_metadata
    ),
    "kaggle_authentication_routes": (
        kaggle_authentication
    ),
    "feasibility_assessment": (
        feasibility_assessment.to_dict(
            orient="records"
        )
    ),
    "important_boundary": (
        "This stage establishes provenance and access "
        "feasibility only. It does not establish target "
        "recoverability, transfer performance, or CDO validity."
    ),
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 1A",
    "",
    "## Target access and provenance feasibility audit",
    "",
    f"Decision: `{stage1a_decision}`",
    "",
    stage1a_interpretation,
    "",
    "## Safety result",
    "",
    "- Target label content accessed: `False`",
    "- Target data downloaded: `False`",
    "",
    "## APTOS",
    "",
    (
        f"- Kaggle access status: "
        f"`{kaggle_access_status}`"
    ),
    (
        f"- Official file rows listed: "
        f"{len(aptos_file_listing)}"
    ),
    "",
    "## IDRiD",
    "",
    (
        "- Official source: IDRiD Grand Challenge and "
        "IEEE DataPort"
    ),
    "- Official image count: 516",
    "- Reported license: CC BY 4.0",
    "",
    "## Next step",
    "",
    f"`{authorised_next_step}`",
    "",
]


REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "requests": requests.__version__,
    "kaggle_install_record": (
        kaggle_install_record
    ),
    "selected_kaggle_command": (
        selected_kaggle_attempt[
            "command"
        ]
    ),
    "selected_kaggle_returncode": (
        selected_kaggle_attempt[
            "returncode"
        ]
    ),
    "stage1_root": str(
        STAGE1_ROOT
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 13. Final integrity checks
# ============================================================

required_outputs = [
    DOCUMENTATION_REGISTRY_PATH,
    URL_ACCESS_PATH,
    APTOS_FILE_LIST_PATH,
    LOCAL_INVENTORY_PATH,
    PROVENANCE_FACTS_PATH,
    FEASIBILITY_PATH,
    DECISION_PATH,
    REPORT_PATH,
    ENVIRONMENT_PATH,
]

missing_outputs = [
    path
    for path in required_outputs
    if not path.is_file()
]

assert not missing_outputs, (
    "Missing Stage 1A outputs:\n"
    +
    "\n".join(
        map(str, missing_outputs)
    )
)


assert TARGET_LABEL_CONTENT_ACCESSED is False
assert TARGET_DATA_DOWNLOADED is False

assert not (
    feasibility_assessment[
        "label_content_accessed"
    ]
).any()

assert not (
    feasibility_assessment[
        "data_downloaded_in_this_stage"
    ]
).any()


# ============================================================
# 14. Display results
# ============================================================

print(
    "\n================ OFFICIAL URL ACCESS "
    "AUDIT ================"
)

display(
    url_access_audit[
        [
            "dataset",
            "resource_id",
            "status_code",
            "access_classification",
            "final_url",
            "error",
        ]
    ]
)


print(
    "\n================ KAGGLE AUTHENTICATION "
    "AUDIT ================"
)

for key, value in kaggle_authentication.items():
    print(
        f"{key}: {value}"
    )


print(
    "\nAPTOS Kaggle access status:"
)

print(
    kaggle_access_status
)


print(
    "\n================ APTOS OFFICIAL FILE "
    "LISTING ================"
)

if aptos_file_listing.empty:
    print(
        "No official file listing was obtained."
    )

    print(
        "\nKaggle command error excerpt:"
    )

    print(
        str(
            selected_kaggle_attempt[
                "stderr"
            ]
        )[-2000:]
    )

else:
    display(
        aptos_file_listing
    )


print(
    "\nAPTOS expected components:"
)

for key, value in aptos_components.items():
    print(
        f"  {key}: {value}"
    )


print(
    "\n================ LOCAL TARGET ARTIFACT "
    "INVENTORY ================"
)

if local_target_inventory.empty:
    print(
        "No existing local APTOS or IDRiD artifacts "
        "were detected in the audited roots."
    )

else:
    display(
        local_target_inventory
    )


print(
    "\n================ TARGET FEASIBILITY "
    "ASSESSMENT ================"
)

display(
    feasibility_assessment
)


print(
    "\n================ STAGE 1A DECISION "
    "================"
)

print("Decision:")
print(stage1a_decision)

print("\nInterpretation:")
print(stage1a_interpretation)

print("\nAuthorised next step:")
print(authorised_next_step)

print("\nTarget label content accessed:")
print(TARGET_LABEL_CONTENT_ACCESSED)

print("\nTarget data downloaded:")
print(TARGET_DATA_DOWNLOADED)


print("\nOutput root:")
print(STAGE1_ROOT)


print(
    "\nStage 1A target access and provenance audit "
    "completed and sealed."
)

================ STAGE 1A BOOTSTRAP ================

Imported design decision:
FREEZE_CANDIDATE_DESIGN_ADVANCE_TO_TARGET_FEASIBILITY_AUDIT

Audit boundary:
This stage may inspect public documentation, URL status, competition file names and sizes, authentication status, and local file-system inventory. It must not download, extract, open or parse any target label file.

================ OFFICIAL URL ACCESS AUDIT ================


,dataset,resource_id,status_code,access_classification,final_url,error
0,APTOS_2019,APTOS_OVERVIEW,200,ACCESSIBLE,https://www.kaggle.com/competitions/aptos2019-...,None
1,APTOS_2019,APTOS_DATA,200,ACCESSIBLE,https://www.kaggle.com/competitions/aptos2019-...,None
2,APTOS_2019,APTOS_RULES,200,ACCESSIBLE,https://www.kaggle.com/competitions/aptos2019-...,None
3,IDRiD,IDRID_DATA,200,ACCESSIBLE,https://idrid.grand-challenge.org/Data/,None
4,IDRiD,IDRID_RULES,200,ACCESSIBLE,https://idrid.grand-challenge.org/Rules/,None
5,IDRiD,IDRID_DATAPORT,200,ACCESSIBLE,https://ieee-dataport.org/open-access/indian-d...,None



================ KAGGLE AUTHENTICATION AUDIT ================
environment_token_present: False
legacy_username_present: False
legacy_key_present: False
access_token_file_present: False
legacy_kaggle_json_present: False
any_authentication_route_present: False

APTOS Kaggle access status:
KAGGLE_AUTHENTICATION_REQUIRED

================ APTOS OFFICIAL FILE LISTING ================
No official file listing was obtained.

Kaggle command error excerpt:


APTOS expected components:
  train_image_archive_or_directory_listed: False
  test_image_archive_or_directory_listed: False
  training_label_file_listed: False
  sample_submission_listed: False

================ LOCAL TARGET ARTIFACT INVENTORY ================


,dataset,path,filename,suffix,size_bytes,content_opened
0,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,LICENSE.txt,.txt,306,False
1,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,CC-BY-4.0.txt,.txt,17344,False
2,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,a. IDRiD_Disease Grading_Training Labels.csv,.csv,9975,False
3,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,b. IDRiD_Disease Grading_Testing Labels.csv,.csv,1598,False
4,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,IDRiD_087.jpg,.jpg,747704,False
...,...,...,...,...,...,...
515,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,IDRiD_160.jpg,.jpg,281282,False
516,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,IDRiD_171.jpg,.jpg,373195,False
517,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,IDRiD_396.jpg,.jpg,808559,False
518,IDRiD,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,IDRiD_413.jpg,.jpg,765963,False



================ TARGET FEASIBILITY ASSESSMENT ================


,dataset,official_provenance_verified,official_host,remote_resource_reachable,official_file_listing_status,official_file_rows_listed,local_artifacts_detected,label_content_accessed,data_downloaded_in_this_stage,quarantine_structure_supported,license_or_terms_status,feasibility_status,blocking_or_next_action
0,APTOS_2019,True,Kaggle competition,True,KAGGLE_AUTHENTICATION_REQUIRED,0,0,False,False,False,KAGGLE_COMPETITION_RULES_GOVERNED,CONDITIONAL_KAGGLE_ACCESS_ACTION_REQUIRED,KAGGLE_AUTHENTICATION_REQUIRED
1,IDRiD,True,IDRiD Grand Challenge / IEEE DataPort,True,NO_AUTOMATED_FILE_LISTING_ATTEMPTED,0,520,False,False,True,CC_BY_4_0_OFFICIAL_DOCUMENTATION,LOCAL_ARTIFACTS_DETECTED_PENDING_PROVENANCE_CHECK,Verify that the local files originate from the...



================ STAGE 1A DECISION ================
Decision:
CONDITIONAL_PASS_ACCESS_ACTIONS_REQUIRED_BEFORE_TARGET_ACQUISITION

Interpretation:
Two candidate targets remain scientifically viable, but one or more account, rule-acceptance, manual-download or automated-access actions must be resolved before controlled acquisition.

Authorised next step:
RESOLVE_ACCESS_ACTIONS_AND_REPEAT_STAGE_1A

Target label content accessed:
False

Target data downloaded:
False

Output root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_Design_v0.1/Stage1_Target_Feasibility_v0.1

Stage 1A target access and provenance audit completed and sealed.


In [5]:
#@title 01B. One-click Kaggle credential upload and final target-feasibility audit

from pathlib import Path
from io import StringIO
from datetime import datetime, timezone

import json
import os
import re
import shutil
import subprocess
import sys

import pandas as pd


# ============================================================
# 0. Safety boundary
# ============================================================

COMPETITION = "aptos2019-blindness-detection"

TARGET_DATA_DOWNLOADED = False
TARGET_LABEL_CONTENT_ACCESSED = False

SAFETY_BOUNDARY = (
    "This cell may upload a Kaggle credential into the temporary "
    "Colab runtime, request official competition file-list metadata, "
    "and inspect previously saved file-inventory metadata. It must "
    "not download target data or open target-label contents."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

STAGE1_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
    / "Stage1_Target_Feasibility_v0.1"
)

STAGE1_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE1A_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1A_Target_Feasibility_Decision_v0.1.json"
)

LOCAL_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)


assert STAGE1A_DECISION_PATH.is_file(), (
    STAGE1A_DECISION_PATH
)

assert LOCAL_INVENTORY_PATH.is_file(), (
    LOCAL_INVENTORY_PATH
)


with open(
    STAGE1A_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    previous_decision = json.load(file)


assert (
    previous_decision["decision"]
    ==
    "CONDITIONAL_PASS_ACCESS_ACTIONS_REQUIRED_BEFORE_TARGET_ACQUISITION"
), previous_decision["decision"]


print(
    "================ STAGE 1B AUTOMATED RESOLUTION "
    "================"
)

print("\nPrevious decision:")
print(previous_decision["decision"])

print("\nSafety boundary:")
print(SAFETY_BOUNDARY)


# ============================================================
# 2. Install/update official Kaggle CLI
# ============================================================

install_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "kaggle",
    ],
    capture_output=True,
    text=True,
    timeout=240,
)


if install_result.returncode != 0:
    raise RuntimeError(
        "Kaggle CLI installation failed:\n"
        +
        install_result.stderr[-2000:]
    )


version_result = subprocess.run(
    [
        "kaggle",
        "--version",
    ],
    capture_output=True,
    text=True,
    timeout=60,
)


print("\nKaggle CLI:")
print(
    (
        version_result.stdout
        or
        version_result.stderr
    ).strip()
)


# ============================================================
# 3. Request kaggle.json through one upload window
# ============================================================

from google.colab import files


print(
    "\nPlease select the kaggle.json file downloaded from "
    "Kaggle Settings → API → Create Legacy API Key."
)

print(
    "The credential will remain only in the temporary Colab "
    "runtime and will not be copied to Google Drive."
)


uploaded = files.upload()


json_candidates = [
    filename
    for filename in uploaded.keys()
    if Path(filename).suffix.lower() == ".json"
]


if len(json_candidates) != 1:
    raise RuntimeError(
        "Exactly one JSON credential file must be uploaded. "
        f"Received: {list(uploaded.keys())}"
    )


uploaded_filename = json_candidates[0]
uploaded_path = Path(uploaded_filename)


try:
    credential_payload = json.loads(
        uploaded[
            uploaded_filename
        ].decode("utf-8")
    )

except Exception as error:
    raise RuntimeError(
        "The uploaded file is not valid JSON.\n"
        f"{type(error).__name__}: {error}"
    )


required_legacy_fields = {
    "username",
    "key",
}


if not required_legacy_fields.issubset(
    credential_payload.keys()
):
    raise RuntimeError(
        "The uploaded JSON does not appear to be a legacy "
        "Kaggle API credential. Expected fields: username and key."
    )


# Never print credential values.
KAGGLE_DIRECTORY = (
    Path.home()
    / ".kaggle"
)

KAGGLE_DIRECTORY.mkdir(
    parents=True,
    exist_ok=True,
)

KAGGLE_CREDENTIAL_PATH = (
    KAGGLE_DIRECTORY
    / "kaggle.json"
)


KAGGLE_CREDENTIAL_PATH.write_bytes(
    uploaded[
        uploaded_filename
    ]
)

os.chmod(
    KAGGLE_CREDENTIAL_PATH,
    0o600,
)


# Remove the duplicate file from the notebook working directory.
if uploaded_path.exists():
    uploaded_path.unlink()


print(
    "\nKaggle credential installed in the temporary runtime."
)

print(
    "Credential contents were not displayed and were not saved "
    "to Google Drive."
)


# ============================================================
# 4. Request APTOS official file-list metadata only
# ============================================================

command = [
    "kaggle",
    "competitions",
    "files",
    COMPETITION,
    "--page-size",
    "200",
    "-v",
    "-q",
]


listing_result = subprocess.run(
    command,
    capture_output=True,
    text=True,
    timeout=180,
)


if listing_result.returncode != 0:
    error_text = (
        listing_result.stdout
        +
        "\n"
        +
        listing_result.stderr
    )

    error_lower = error_text.lower()

    if any(
        token in error_lower
        for token in [
            "403",
            "forbidden",
            "accept",
            "rules",
            "permission",
        ]
    ):
        access_status = (
            "COMPETITION_RULE_ACCEPTANCE_REQUIRED"
        )

        explanation = (
            "Kaggle authentication succeeded, but the APTOS "
            "competition rules have not been accepted. Open the "
            "APTOS competition Rules page while logged in, accept "
            "the rules, then rerun this same cell."
        )

    elif any(
        token in error_lower
        for token in [
            "401",
            "unauthorized",
            "authentication",
            "credential",
            "invalid",
        ]
    ):
        access_status = (
            "KAGGLE_CREDENTIAL_REJECTED"
        )

        explanation = (
            "Kaggle rejected the credential. Generate a new "
            "Legacy API Key and rerun this same cell."
        )

    else:
        access_status = (
            "APTOS_FILE_LISTING_FAILED_UNRESOLVED"
        )

        explanation = (
            "Kaggle authentication was configured, but the "
            "official APTOS file listing could not be retrieved."
        )

    raise RuntimeError(
        f"{access_status}\n\n"
        f"{explanation}\n\n"
        "Command output excerpt:\n"
        +
        error_text[-2000:]
    )


# ============================================================
# 5. Parse official CSV file listing
# ============================================================

stdout_lines = [
    line
    for line in listing_result.stdout.splitlines()
    if line.strip()
]


header_index = None


for index, line in enumerate(
    stdout_lines
):
    compact = (
        line.lower()
        .replace(" ", "")
    )

    if (
        "name" in compact
        and
        (
            "size" in compact
            or
            "totalbytes" in compact
        )
    ):
        header_index = index
        break


if header_index is None:
    raise RuntimeError(
        "Kaggle returned a successful response, but the "
        "official file-list header could not be parsed.\n\n"
        +
        listing_result.stdout[:3000]
    )


csv_text = "\n".join(
    stdout_lines[
        header_index:
    ]
)


aptos_file_listing = pd.read_csv(
    StringIO(
        csv_text
    )
)


aptos_file_listing.columns = [
    str(column).strip()
    for column in aptos_file_listing.columns
]


assert not aptos_file_listing.empty, (
    "Parsed APTOS file listing is empty."
)


filename_columns = [
    column
    for column in aptos_file_listing.columns
    if str(column).strip().lower()
    in {
        "name",
        "filename",
        "file",
    }
]


filename_column = (
    filename_columns[0]
    if filename_columns
    else aptos_file_listing.columns[0]
)


aptos_filenames = (
    aptos_file_listing[
        filename_column
    ]
    .astype(str)
    .str.strip()
)


def contains_filename_token(token):
    return bool(
        aptos_filenames
        .str.lower()
        .str.contains(
            token.lower(),
            regex=False,
        )
        .any()
    )


aptos_components = {
    "train_images_listed": contains_filename_token(
        "train_images"
    ),
    "test_images_listed": contains_filename_token(
        "test_images"
    ),
    "train_csv_listed": contains_filename_token(
        "train.csv"
    ),
    "sample_submission_listed": (
        contains_filename_token(
            "sample_submission"
        )
    ),
}


aptos_structure_pass = bool(
    aptos_components[
        "train_images_listed"
    ]
    and
    aptos_components[
        "train_csv_listed"
    ]
)


# ============================================================
# 6. Verify IDRiD using previously saved metadata only
#    No label CSV is opened
# ============================================================

local_inventory = pd.read_csv(
    LOCAL_INVENTORY_PATH
)


idrid_inventory = (
    local_inventory[
        local_inventory["dataset"]
        ==
        "IDRiD"
    ]
    .copy()
    .reset_index(drop=True)
)


idrid_inventory[
    "filename_lower"
] = (
    idrid_inventory[
        "filename"
    ]
    .astype(str)
    .str.lower()
)


idrid_jpgs = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            ".jpg",
            ".jpeg",
        ]
    )
].copy()


idrid_csvs = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    ==
    ".csv"
].copy()


idrid_licences = idrid_inventory[
    idrid_inventory[
        "filename_lower"
    ]
    .str.contains(
        "license|cc-by|cc_by",
        regex=True,
    )
].copy()


training_label_candidates = idrid_csvs[
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "training",
        regex=False,
    )
    &
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


testing_label_candidates = idrid_csvs[
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "testing",
        regex=False,
    )
    &
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


official_name_matches = int(
    idrid_jpgs[
        "filename"
    ]
    .astype(str)
    .map(
        lambda value: bool(
            re.fullmatch(
                r"IDRiD_\d{3}\.(jpg|jpeg)",
                value,
                flags=re.IGNORECASE,
            )
        )
    )
    .sum()
)


idrid_checks = {
    "total_inventory_rows": int(
        len(idrid_inventory)
    ),
    "jpg_count": int(
        len(idrid_jpgs)
    ),
    "expected_jpg_count": 516,
    "official_image_name_matches": (
        official_name_matches
    ),
    "training_label_filename_count": int(
        len(training_label_candidates)
    ),
    "testing_label_filename_count": int(
        len(testing_label_candidates)
    ),
    "licence_filename_count": int(
        len(idrid_licences)
    ),
    "label_content_opened": False,
}


idrid_structure_pass = bool(
    idrid_checks["jpg_count"] == 516
    and
    idrid_checks[
        "official_image_name_matches"
    ] == 516
    and
    idrid_checks[
        "training_label_filename_count"
    ] >= 1
    and
    idrid_checks[
        "testing_label_filename_count"
    ] >= 1
    and
    idrid_checks[
        "licence_filename_count"
    ] >= 1
)


# ============================================================
# 7. Final feasibility decision
# ============================================================

if (
    aptos_structure_pass
    and
    idrid_structure_pass
):
    final_decision = (
        "PASS_TWO_TARGETS_ADVANCE_TO_"
        "CONTROLLED_TARGET_ACQUISITION"
    )

    interpretation = (
        "The authenticated official APTOS competition listing "
        "contains the expected image and training-label files. "
        "The existing IDRiD inventory matches the expected "
        "516-image release structure and includes separate "
        "training/testing grading filenames and licence files. "
        "No target data were downloaded and no label-table "
        "contents were opened during this stage."
    )

    authorised_next_step = (
        "CONTROLLED_TARGET_ACQUISITION_WITH_LABEL_QUARANTINE"
    )

else:
    final_decision = (
        "HOLD_TARGET_ACQUISITION_STRUCTURE_CHECK_FAILED"
    )

    failed_components = []

    if not aptos_structure_pass:
        failed_components.append(
            "APTOS official file structure"
        )

    if not idrid_structure_pass:
        failed_components.append(
            "IDRiD local release structure"
        )

    interpretation = (
        "Acquisition remains on hold because the following "
        "structure checks failed: "
        +
        ", ".join(
            failed_components
        )
        +
        "."
    )

    authorised_next_step = (
        "REVIEW_FAILED_STRUCTURE_CHECKS"
    )


# ============================================================
# 8. Save sealed Stage 1B outputs
# ============================================================

APTOS_LISTING_PATH = (
    STAGE1_ROOT
    / "Stage1B_APTOS_Authenticated_File_Listing_v0.1.csv"
)

IDRID_CHECK_PATH = (
    STAGE1_ROOT
    / "Stage1B_IDRiD_Structure_Check_v0.1.json"
)

FINAL_ASSESSMENT_PATH = (
    STAGE1_ROOT
    / "Stage1B_Final_Target_Assessment_v0.1.csv"
)

FINAL_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1B_Final_Target_Feasibility_Decision_v0.1.json"
)

FINAL_REPORT_PATH = (
    STAGE1_ROOT
    / "Stage1B_Final_Target_Feasibility_Report_v0.1.md"
)


aptos_file_listing.to_csv(
    APTOS_LISTING_PATH,
    index=False,
)


with open(
    IDRID_CHECK_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        idrid_checks,
        file,
        indent=2,
    )


final_assessment = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "structure_pass": (
                aptos_structure_pass
            ),
            "official_rows_listed": int(
                len(aptos_file_listing)
            ),
            "data_downloaded": False,
            "label_content_opened": False,
            "status": (
                "PASS_PENDING_CONTROLLED_ACQUISITION"
                if aptos_structure_pass
                else
                "FAIL_STRUCTURE_CHECK"
            ),
        },
        {
            "dataset": "IDRiD",
            "structure_pass": (
                idrid_structure_pass
            ),
            "official_rows_listed": int(
                len(idrid_inventory)
            ),
            "data_downloaded": False,
            "label_content_opened": False,
            "status": (
                "PASS_LOCAL_RELEASE_STRUCTURE"
                if idrid_structure_pass
                else
                "FAIL_STRUCTURE_CHECK"
            ),
        },
    ]
)


final_assessment.to_csv(
    FINAL_ASSESSMENT_PATH,
    index=False,
)


decision_payload = {
    "decision": final_decision,
    "interpretation": interpretation,
    "authorised_next_step": (
        authorised_next_step
    ),
    "aptos_components": (
        aptos_components
    ),
    "idrid_structure_checks": (
        idrid_checks
    ),
    "target_data_downloaded": False,
    "target_label_content_accessed": False,
    "credential_storage": (
        "TEMPORARY_COLAB_RUNTIME_ONLY_NOT_GOOGLE_DRIVE"
    ),
    "important_boundary": (
        "This stage establishes access and file-structure "
        "feasibility only. It does not evaluate target labels, "
        "target recoverability, transfer performance, or CDO."
    ),
}


with open(
    FINAL_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 1B",
    "",
    "## Final target access and structure audit",
    "",
    f"Decision: `{final_decision}`",
    "",
    interpretation,
    "",
    "## Safety",
    "",
    "- Target data downloaded: `False`",
    "- Target-label contents opened: `False`",
    (
        "- Kaggle credential stored only in the temporary "
        "Colab runtime"
    ),
    "",
    "## Authorised next step",
    "",
    f"`{authorised_next_step}`",
    "",
]


FINAL_REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


# ============================================================
# 9. Integrity checks and display
# ============================================================

assert TARGET_DATA_DOWNLOADED is False
assert TARGET_LABEL_CONTENT_ACCESSED is False


required_outputs = [
    APTOS_LISTING_PATH,
    IDRID_CHECK_PATH,
    FINAL_ASSESSMENT_PATH,
    FINAL_DECISION_PATH,
    FINAL_REPORT_PATH,
]


for path in required_outputs:
    assert path.is_file(), path


print(
    "\n================ APTOS OFFICIAL FILE LISTING "
    "================"
)

display(
    aptos_file_listing
)


print(
    "\n================ APTOS COMPONENT CHECK "
    "================"
)

for key, value in aptos_components.items():
    print(
        f"{key}: {value}"
    )


print(
    "\n================ IDRiD STRUCTURE CHECK "
    "================"
)

for key, value in idrid_checks.items():
    print(
        f"{key}: {value}"
    )


print(
    "\n================ FINAL TARGET ASSESSMENT "
    "================"
)

display(
    final_assessment
)


print(
    "\n================ FINAL STAGE 1B DECISION "
    "================"
)

print("Decision:")
print(final_decision)

print("\nInterpretation:")
print(interpretation)

print("\nAuthorised next step:")
print(authorised_next_step)

print("\nTarget data downloaded:")
print(False)

print("\nTarget-label content accessed:")
print(False)


print(
    "\nStage 1B automated target-feasibility audit "
    "completed and sealed."
)

================ STAGE 1B AUTOMATED RESOLUTION ================

Previous decision:
CONDITIONAL_PASS_ACCESS_ACTIONS_REQUIRED_BEFORE_TARGET_ACQUISITION

Safety boundary:
This cell may upload a Kaggle credential into the temporary Colab runtime, request official competition file-list metadata, and inspect previously saved file-inventory metadata. It must not download target data or open target-label contents.

Kaggle CLI:
Kaggle CLI 2.2.3

Please select the kaggle.json file downloaded from Kaggle Settings → API → Create Legacy API Key.
The credential will remain only in the temporary Colab runtime and will not be copied to Google Drive.


Saving kaggle.json to kaggle.json

Kaggle credential installed in the temporary runtime.
Credential contents were not displayed and were not saved to Google Drive.

================ APTOS OFFICIAL FILE LISTING ================


,name,size,creationDate
0,sample_submission.csv,28938,2019-12-18 03:20:17.427000
1,test.csv,25072,2019-12-18 03:20:17.452000
2,test_images/0005cfc8afb6.png,236005,2019-12-18 03:20:46.144000
3,test_images/003f0afdcd15.png,212295,2019-12-18 03:20:50.190000
4,test_images/006efc72b638.png,202491,2019-12-18 03:20:49.451000
...,...,...,...
195,test_images/1d5a1d9fe2cb.png,5133867,2019-12-18 03:20:44.172000
196,test_images/1d7c4c12769e.png,224952,2019-12-18 03:20:42.441000
197,test_images/1de193623108.png,216514,2019-12-18 03:20:47.708000
198,test_images/1e0309115a25.png,208479,2019-12-18 03:20:50.487000



================ APTOS COMPONENT CHECK ================
train_images_listed: False
test_images_listed: True
train_csv_listed: False
sample_submission_listed: True

================ IDRiD STRUCTURE CHECK ================
total_inventory_rows: 520
jpg_count: 516
expected_jpg_count: 516
official_image_name_matches: 516
training_label_filename_count: 1
testing_label_filename_count: 1
licence_filename_count: 2
label_content_opened: False

================ FINAL TARGET ASSESSMENT ================


,dataset,structure_pass,official_rows_listed,data_downloaded,label_content_opened,status
0,APTOS_2019,False,200,False,False,FAIL_STRUCTURE_CHECK
1,IDRiD,True,520,False,False,PASS_LOCAL_RELEASE_STRUCTURE



================ FINAL STAGE 1B DECISION ================
Decision:
HOLD_TARGET_ACQUISITION_STRUCTURE_CHECK_FAILED

Interpretation:
Acquisition remains on hold because the following structure checks failed: APTOS official file structure.

Authorised next step:
REVIEW_FAILED_STRUCTURE_CHECKS

Target data downloaded:
False

Target-label content accessed:
False

Stage 1B automated target-feasibility audit completed and sealed.


In [6]:
#@title 01C. Repair APTOS pagination and finalise target feasibility — NO DATA DOWNLOAD

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import re
import sys

import pandas as pd


# ============================================================
# 0. Safety boundary
# ============================================================

COMPETITION = "aptos2019-blindness-detection"
PAGE_SIZE = 200
MAX_PAGES = 100

TARGET_DATA_DOWNLOADED = False
TARGET_LABEL_CONTENT_ACCESSED = False

SAFETY_BOUNDARY = (
    "This repair stage retrieves every page of official Kaggle "
    "competition file metadata and reuses the previously sealed "
    "IDRiD filename inventory. It must not download target data "
    "or open any target-label table."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

STAGE1_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
    / "Stage1_Target_Feasibility_v0.1"
)

STAGE1_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE1A_LOCAL_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)

STAGE1B_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1B_Final_Target_Feasibility_Decision_v0.1.json"
)


assert STAGE1A_LOCAL_INVENTORY_PATH.is_file(), (
    STAGE1A_LOCAL_INVENTORY_PATH
)

assert STAGE1B_DECISION_PATH.is_file(), (
    STAGE1B_DECISION_PATH
)


with open(
    STAGE1B_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage1b_decision = json.load(
        file
    )


assert (
    stage1b_decision["decision"]
    ==
    "HOLD_TARGET_ACQUISITION_STRUCTURE_CHECK_FAILED"
), stage1b_decision["decision"]


print(
    "================ STAGE 1C PAGINATION REPAIR "
    "================"
)

print("\nPrevious Stage 1B decision:")
print(
    stage1b_decision["decision"]
)

print("\nRepair reason:")
print(
    "Stage 1B inspected only the first 200 APTOS "
    "metadata records and therefore produced an "
    "incomplete structure assessment."
)

print("\nSafety boundary:")
print(SAFETY_BOUNDARY)


# ============================================================
# 2. Confirm Kaggle credential
# ============================================================

credential_candidates = [
    Path.home()
    / ".kaggle"
    / "kaggle.json",

    Path.home()
    / ".config"
    / "kaggle"
    / "kaggle.json",
]


credential_path = next(
    (
        path
        for path in credential_candidates
        if path.is_file()
    ),
    None,
)


if credential_path is None:
    from google.colab import files

    print(
        "\nThe temporary Kaggle credential is no longer "
        "present, probably because the Colab runtime restarted."
    )

    print(
        "Select the same kaggle.json file once. "
        "Its contents will not be displayed or saved to Drive."
    )

    uploaded = files.upload()

    json_candidates = [
        filename
        for filename in uploaded
        if Path(filename).suffix.lower()
        ==
        ".json"
    ]

    if len(json_candidates) != 1:
        raise RuntimeError(
            "Exactly one kaggle.json file is required. "
            f"Received: {list(uploaded.keys())}"
        )

    uploaded_name = json_candidates[0]

    try:
        credential_payload = json.loads(
            uploaded[
                uploaded_name
            ].decode("utf-8")
        )

    except Exception as error:
        raise RuntimeError(
            "The uploaded credential is not valid JSON.\n"
            f"{type(error).__name__}: {error}"
        )

    if not {
        "username",
        "key",
    }.issubset(
        credential_payload.keys()
    ):
        raise RuntimeError(
            "The uploaded file is not a legacy Kaggle "
            "API credential."
        )

    credential_path = (
        Path.home()
        / ".kaggle"
        / "kaggle.json"
    )

    credential_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    credential_path.write_bytes(
        uploaded[
            uploaded_name
        ]
    )

    os.chmod(
        credential_path,
        0o600,
    )

    temporary_uploaded_path = Path(
        uploaded_name
    )

    if temporary_uploaded_path.exists():
        temporary_uploaded_path.unlink()


assert credential_path.is_file()

print(
    "\nKaggle credential found in the temporary runtime."
)

print(
    "Credential value will not be displayed or copied "
    "to Google Drive."
)


# ============================================================
# 3. Authenticate official Kaggle Python API
# ============================================================

from kaggle.api.kaggle_api_extended import KaggleApi


api = KaggleApi()
api.authenticate()


print(
    "\nKaggle API authentication succeeded."
)


# ============================================================
# 4. Retrieve every APTOS metadata page
# ============================================================

all_file_rows = []
page_audit_rows = []

page_token = None
seen_tokens = set()


for page_number in range(
    1,
    MAX_PAGES + 1,
):
    response = api.competition_list_files(
        competition=COMPETITION,
        page_token=page_token,
        page_size=PAGE_SIZE,
    )

    response_files = (
        response.files
        or
        []
    )

    for file_record in response_files:
        if file_record is None:
            continue

        creation_date = getattr(
            file_record,
            "creation_date",
            None,
        )

        all_file_rows.append(
            {
                "name": str(
                    getattr(
                        file_record,
                        "name",
                        "",
                    )
                ),
                "size": int(
                    getattr(
                        file_record,
                        "total_bytes",
                        0,
                    )
                    or
                    0
                ),
                "creation_date": (
                    None
                    if creation_date is None
                    else str(
                        creation_date
                    )
                ),
                "retrieved_page": int(
                    page_number
                ),
            }
        )

    next_page_token = str(
        getattr(
            response,
            "next_page_token",
            "",
        )
        or
        ""
    )


    page_audit_rows.append(
        {
            "page": int(
                page_number
            ),
            "rows_returned": int(
                len(
                    response_files
                )
            ),
            "cumulative_rows": int(
                len(
                    all_file_rows
                )
            ),
            "next_page_token_present": bool(
                next_page_token
            ),
            "next_page_token_sha256": (
                hashlib.sha256(
                    next_page_token.encode(
                        "utf-8"
                    )
                ).hexdigest()
                if next_page_token
                else
                None
            ),
        }
    )


    print(
        f"Page {page_number:02d}: "
        f"{len(response_files)} rows; "
        f"cumulative {len(all_file_rows)}"
    )


    if not next_page_token:
        break


    if next_page_token in seen_tokens:
        raise RuntimeError(
            "Kaggle returned a repeated page token. "
            "Pagination was stopped to prevent an "
            "infinite loop."
        )


    seen_tokens.add(
        next_page_token
    )

    page_token = (
        next_page_token
    )

else:
    raise RuntimeError(
        f"Pagination exceeded {MAX_PAGES} pages. "
        "The full APTOS listing was not sealed."
    )


aptos_full_listing = pd.DataFrame(
    all_file_rows
)


if aptos_full_listing.empty:
    raise RuntimeError(
        "The full APTOS file listing is empty."
    )


aptos_full_listing["name"] = (
    aptos_full_listing["name"]
    .astype(str)
    .str.strip()
)


duplicate_name_count = int(
    aptos_full_listing[
        "name"
    ].duplicated().sum()
)


aptos_full_listing = (
    aptos_full_listing
    .drop_duplicates(
        subset=[
            "name",
        ],
        keep="first",
    )
    .sort_values(
        "name"
    )
    .reset_index(
        drop=True
    )
)


page_audit = pd.DataFrame(
    page_audit_rows
)


# ============================================================
# 5. Verify complete APTOS structure
# ============================================================

aptos_names = (
    aptos_full_listing[
        "name"
    ]
    .str.lower()
)


train_image_mask = (
    aptos_names
    .str.startswith(
        "train_images/"
    )
    &
    aptos_names
    .str.endswith(
        (
            ".png",
            ".jpg",
            ".jpeg",
        )
    )
)


test_image_mask = (
    aptos_names
    .str.startswith(
        "test_images/"
    )
    &
    aptos_names
    .str.endswith(
        (
            ".png",
            ".jpg",
            ".jpeg",
        )
    )
)


train_image_count = int(
    train_image_mask.sum()
)

test_image_count = int(
    test_image_mask.sum()
)


aptos_components = {
    "all_paginated_pages_retrieved": bool(
        len(page_audit) >= 2
        and
        page_audit.iloc[-1][
            "next_page_token_present"
        ]
        is False
    ),
    "unique_official_file_rows": int(
        len(
            aptos_full_listing
        )
    ),
    "duplicate_file_names_removed": (
        duplicate_name_count
    ),
    "train_image_count": (
        train_image_count
    ),
    "test_image_count": (
        test_image_count
    ),
    "train_csv_present": bool(
        (
            aptos_names
            ==
            "train.csv"
        ).any()
    ),
    "test_csv_present": bool(
        (
            aptos_names
            ==
            "test.csv"
        ).any()
    ),
    "sample_submission_present": bool(
        (
            aptos_names
            ==
            "sample_submission.csv"
        ).any()
    ),
    "label_content_opened": False,
    "target_data_downloaded": False,
}


aptos_structure_pass = bool(
    aptos_components[
        "all_paginated_pages_retrieved"
    ]
    and
    aptos_components[
        "train_image_count"
    ]
    >
    0
    and
    aptos_components[
        "test_image_count"
    ]
    >
    0
    and
    aptos_components[
        "train_csv_present"
    ]
    and
    aptos_components[
        "test_csv_present"
    ]
    and
    aptos_components[
        "sample_submission_present"
    ]
)


# ============================================================
# 6. Re-verify IDRiD from sealed filename inventory
# ============================================================

local_inventory = pd.read_csv(
    STAGE1A_LOCAL_INVENTORY_PATH
)


idrid_inventory = (
    local_inventory[
        local_inventory["dataset"]
        ==
        "IDRiD"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


idrid_inventory[
    "filename_lower"
] = (
    idrid_inventory[
        "filename"
    ]
    .astype(str)
    .str.lower()
)


idrid_jpgs = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            ".jpg",
            ".jpeg",
        ]
    )
].copy()


idrid_csvs = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    ==
    ".csv"
].copy()


idrid_licences = idrid_inventory[
    idrid_inventory[
        "filename_lower"
    ]
    .str.contains(
        "license|cc-by|cc_by",
        regex=True,
    )
].copy()


idrid_training_labels = idrid_csvs[
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "training",
        regex=False,
    )
    &
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


idrid_testing_labels = idrid_csvs[
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "testing",
        regex=False,
    )
    &
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


idrid_official_name_matches = int(
    idrid_jpgs[
        "filename"
    ]
    .astype(str)
    .map(
        lambda value: bool(
            re.fullmatch(
                r"IDRiD_\d{3}\.(jpg|jpeg)",
                value,
                flags=re.IGNORECASE,
            )
        )
    )
    .sum()
)


idrid_checks = {
    "total_inventory_rows": int(
        len(
            idrid_inventory
        )
    ),
    "jpg_count": int(
        len(
            idrid_jpgs
        )
    ),
    "expected_jpg_count": 516,
    "official_image_name_matches": (
        idrid_official_name_matches
    ),
    "training_label_filename_count": int(
        len(
            idrid_training_labels
        )
    ),
    "testing_label_filename_count": int(
        len(
            idrid_testing_labels
        )
    ),
    "licence_filename_count": int(
        len(
            idrid_licences
        )
    ),
    "label_content_opened": False,
}


idrid_structure_pass = bool(
    idrid_checks[
        "jpg_count"
    ]
    ==
    516
    and
    idrid_checks[
        "official_image_name_matches"
    ]
    ==
    516
    and
    idrid_checks[
        "training_label_filename_count"
    ]
    >=
    1
    and
    idrid_checks[
        "testing_label_filename_count"
    ]
    >=
    1
    and
    idrid_checks[
        "licence_filename_count"
    ]
    >=
    1
)


# ============================================================
# 7. Corrected Stage 1C decision
# ============================================================

if (
    aptos_structure_pass
    and
    idrid_structure_pass
):
    final_decision = (
        "PASS_TWO_TARGETS_ADVANCE_TO_"
        "CONTROLLED_TARGET_ACQUISITION"
    )

    interpretation = (
        "The previous APTOS failure was caused by incomplete "
        "single-page metadata retrieval. Full token-based "
        "pagination now confirms the expected APTOS training "
        "images, test images and CSV filenames. IDRiD retains "
        "its valid 516-image local release structure. No target "
        "data were downloaded and no target-label contents "
        "were opened."
    )

    authorised_next_step = (
        "CONTROLLED_TARGET_ACQUISITION_WITH_"
        "LABEL_QUARANTINE"
    )

else:
    final_decision = (
        "HOLD_TARGET_ACQUISITION_AFTER_"
        "COMPLETE_PAGINATION_CHECK"
    )

    failed_checks = []

    if not aptos_structure_pass:
        failed_checks.append(
            "APTOS complete paginated structure"
        )

    if not idrid_structure_pass:
        failed_checks.append(
            "IDRiD local release structure"
        )

    interpretation = (
        "At least one target still failed after complete "
        "pagination: "
        +
        ", ".join(
            failed_checks
        )
        +
        "."
    )

    authorised_next_step = (
        "REVIEW_COMPLETE_STRUCTURE_CHECKS"
    )


# ============================================================
# 8. Save Stage 1C repair artifacts
# ============================================================

APTOS_FULL_LISTING_PATH = (
    STAGE1_ROOT
    / "Stage1C_APTOS_Full_Paginated_File_Listing_v0.1.csv"
)

PAGINATION_AUDIT_PATH = (
    STAGE1_ROOT
    / "Stage1C_APTOS_Pagination_Audit_v0.1.csv"
)

FINAL_ASSESSMENT_PATH = (
    STAGE1_ROOT
    / "Stage1C_Corrected_Target_Assessment_v0.1.csv"
)

SUPERSESSION_PATH = (
    STAGE1_ROOT
    / "Stage1C_Stage1B_Supersession_Manifest_v0.1.json"
)

FINAL_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1C_Final_Target_Feasibility_Decision_v0.1.json"
)

FINAL_REPORT_PATH = (
    STAGE1_ROOT
    / "Stage1C_Final_Target_Feasibility_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    STAGE1_ROOT
    / "Stage1C_Target_Feasibility_Environment_v0.1.json"
)


aptos_full_listing.to_csv(
    APTOS_FULL_LISTING_PATH,
    index=False,
)

page_audit.to_csv(
    PAGINATION_AUDIT_PATH,
    index=False,
)


final_assessment = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "complete_pagination": (
                aptos_components[
                    "all_paginated_pages_retrieved"
                ]
            ),
            "official_metadata_rows": int(
                len(
                    aptos_full_listing
                )
            ),
            "train_images": (
                train_image_count
            ),
            "test_images": (
                test_image_count
            ),
            "structure_pass": (
                aptos_structure_pass
            ),
            "data_downloaded": False,
            "label_content_opened": False,
            "status": (
                "PASS_PENDING_CONTROLLED_ACQUISITION"
                if aptos_structure_pass
                else
                "FAIL_AFTER_COMPLETE_PAGINATION"
            ),
        },
        {
            "dataset": "IDRiD",
            "complete_pagination": (
                "NOT_APPLICABLE_LOCAL_RELEASE"
            ),
            "official_metadata_rows": int(
                len(
                    idrid_inventory
                )
            ),
            "train_images": (
                "LOCAL_RELEASE_SPLIT_NOT_OPENED"
            ),
            "test_images": (
                "LOCAL_RELEASE_SPLIT_NOT_OPENED"
            ),
            "structure_pass": (
                idrid_structure_pass
            ),
            "data_downloaded": False,
            "label_content_opened": False,
            "status": (
                "PASS_LOCAL_RELEASE_STRUCTURE"
                if idrid_structure_pass
                else
                "FAIL_LOCAL_RELEASE_STRUCTURE"
            ),
        },
    ]
)


final_assessment.to_csv(
    FINAL_ASSESSMENT_PATH,
    index=False,
)


supersession_payload = {
    "superseded_artifact": str(
        STAGE1B_DECISION_PATH
    ),
    "superseded_decision": (
        stage1b_decision[
            "decision"
        ]
    ),
    "supersession_status": (
        "SUPERSEDED_DUE_TO_INCOMPLETE_KAGGLE_PAGINATION"
    ),
    "reason": (
        "Stage 1B requested the maximum single page of "
        "200 records but did not follow next_page_token. "
        "Its APTOS structure failure was therefore based "
        "on an incomplete official file listing."
    ),
    "replacement_artifact": str(
        FINAL_DECISION_PATH
    ),
    "old_artifact_deleted": False,
}


with open(
    SUPERSESSION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        supersession_payload,
        file,
        indent=2,
    )


decision_payload = {
    "decision": final_decision,
    "interpretation": interpretation,
    "authorised_next_step": (
        authorised_next_step
    ),
    "aptos_structure_pass": (
        aptos_structure_pass
    ),
    "aptos_components": (
        aptos_components
    ),
    "idrid_structure_pass": (
        idrid_structure_pass
    ),
    "idrid_structure_checks": (
        idrid_checks
    ),
    "target_data_downloaded": False,
    "target_label_content_accessed": False,
    "supersedes_stage1b": True,
    "supersession_manifest": str(
        SUPERSESSION_PATH
    ),
    "important_boundary": (
        "This stage establishes complete metadata and "
        "file-structure feasibility only. It does not inspect "
        "target labels, evaluate target recoverability, reveal "
        "transfer performance, or validate CDO."
    ),
}


with open(
    FINAL_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 1C",
    "",
    "## Pagination-repaired target-feasibility audit",
    "",
    f"Decision: `{final_decision}`",
    "",
    interpretation,
    "",
    "## APTOS",
    "",
    (
        f"- Paginated metadata pages: "
        f"{len(page_audit)}"
    ),
    (
        f"- Unique official file records: "
        f"{len(aptos_full_listing)}"
    ),
    (
        f"- Training images listed: "
        f"{train_image_count}"
    ),
    (
        f"- Test images listed: "
        f"{test_image_count}"
    ),
    "",
    "## IDRiD",
    "",
    (
        f"- Local JPG images: "
        f"{idrid_checks['jpg_count']}"
    ),
    "",
    "## Safety",
    "",
    "- Target data downloaded: `False`",
    "- Target-label contents opened: `False`",
    "",
    "## Supersession",
    "",
    (
        "The Stage 1B hold decision is retained for audit "
        "history but superseded because it used only the "
        "first Kaggle metadata page."
    ),
    "",
    "## Authorised next step",
    "",
    f"`{authorised_next_step}`",
    "",
]


FINAL_REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "page_size": PAGE_SIZE,
    "pages_retrieved": int(
        len(
            page_audit
        )
    ),
    "credential_storage": (
        "TEMPORARY_COLAB_RUNTIME_ONLY"
    ),
    "stage1_root": str(
        STAGE1_ROOT
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 9. Final integrity checks
# ============================================================

assert TARGET_DATA_DOWNLOADED is False
assert TARGET_LABEL_CONTENT_ACCESSED is False

required_outputs = [
    APTOS_FULL_LISTING_PATH,
    PAGINATION_AUDIT_PATH,
    FINAL_ASSESSMENT_PATH,
    SUPERSESSION_PATH,
    FINAL_DECISION_PATH,
    FINAL_REPORT_PATH,
    ENVIRONMENT_PATH,
]


for output_path in required_outputs:
    assert output_path.is_file(), (
        output_path
    )


# ============================================================
# 10. Display concise results
# ============================================================

print(
    "\n================ PAGINATION AUDIT "
    "================"
)

display(
    page_audit
)


print(
    "\n================ APTOS COMPONENT CHECK "
    "================"
)

for key, value in aptos_components.items():
    print(
        f"{key}: {value}"
    )


print(
    "\nRepresentative required APTOS records:"
)

required_record_mask = (
    aptos_names.isin(
        [
            "train.csv",
            "test.csv",
            "sample_submission.csv",
        ]
    )
    |
    aptos_names.str.startswith(
        "train_images/"
    )
    |
    aptos_names.str.startswith(
        "test_images/"
    )
)


display(
    pd.concat(
        [
            aptos_full_listing[
                aptos_names
                ==
                "train.csv"
            ],
            aptos_full_listing[
                aptos_names
                ==
                "test.csv"
            ],
            aptos_full_listing[
                aptos_names
                ==
                "sample_submission.csv"
            ],
            aptos_full_listing[
                train_image_mask
            ].head(3),
            aptos_full_listing[
                test_image_mask
            ].head(3),
        ],
        ignore_index=True,
    )
)


print(
    "\n================ IDRiD STRUCTURE CHECK "
    "================"
)

for key, value in idrid_checks.items():
    print(
        f"{key}: {value}"
    )


print(
    "\n================ CORRECTED TARGET ASSESSMENT "
    "================"
)

display(
    final_assessment
)


print(
    "\n================ FINAL STAGE 1C DECISION "
    "================"
)

print("Decision:")
print(final_decision)

print("\nInterpretation:")
print(interpretation)

print("\nAuthorised next step:")
print(authorised_next_step)

print("\nStage 1B decision superseded:")
print(True)

print("\nTarget data downloaded:")
print(False)

print("\nTarget-label content accessed:")
print(False)


print(
    "\nStage 1C complete-pagination repair "
    "completed and sealed."
)

================ STAGE 1C PAGINATION REPAIR ================

Previous Stage 1B decision:
HOLD_TARGET_ACQUISITION_STRUCTURE_CHECK_FAILED

Repair reason:
Stage 1B inspected only the first 200 APTOS metadata records and therefore produced an incomplete structure assessment.

Safety boundary:
This repair stage retrieves every page of official Kaggle competition file metadata and reuses the previously sealed IDRiD filename inventory. It must not download target data or open any target-label table.

Kaggle credential found in the temporary runtime.
Credential value will not be displayed or copied to Google Drive.

Kaggle API authentication succeeded.
Page 01: 200 rows; cumulative 200
Page 02: 200 rows; cumulative 400
Page 03: 200 rows; cumulative 600
Page 04: 200 rows; cumulative 800
Page 05: 200 rows; cumulative 1000
Page 06: 200 rows; cumulative 1200
Page 07: 200 rows; cumulative 1400
Page 08: 200 rows; cumulative 1600
Page 09: 200 rows; cumulative 1800
Page 10: 200 rows; cumulative 2000


,page,rows_returned,cumulative_rows,next_page_token_present,next_page_token_sha256
0,1,200,200,True,6f2c79e7c11515723b8b449acedba7ea33fbc308cabdd5...
1,2,200,400,True,cdd2a08e6c936333d1e52fe867b4a5c7f94ea345faded9...
2,3,200,600,True,e6c49e50aca612e04f1d4b81f3d2fc6003ec361dd85b77...
3,4,200,800,True,1f6c9de4a73bf8e171b2a34bb049a5ece9bf0c724da790...
4,5,200,1000,True,e45e6a60d4cf5b6da13e05b07c1c7ecba85bd0a45e7f49...
5,6,200,1200,True,4032b5a99696834832560c1a15441e17cb6b4259726627...
6,7,200,1400,True,434992427e512bd6fea6791606107293c5a4454dc32e38...
7,8,200,1600,True,c74234fc8385f350b5c417aa4a046d00452c10bf056068...
8,9,200,1800,True,3a8afb07aba255d4237e1e23651fa2a166d996737c95ad...
9,10,200,2000,True,96eb1dbf1899ce67627be60816dd8c3d7042004c5929fb...



================ APTOS COMPONENT CHECK ================
all_paginated_pages_retrieved: False
unique_official_file_rows: 5593
duplicate_file_names_removed: 0
train_image_count: 3662
test_image_count: 1928
train_csv_present: True
test_csv_present: True
sample_submission_present: True
label_content_opened: False
target_data_downloaded: False

Representative required APTOS records:


,name,size,creation_date,retrieved_page
0,train.csv,54948,2019-12-18 03:20:39.731000,10
1,test.csv,25072,2019-12-18 03:20:17.452000,1
2,sample_submission.csv,28938,2019-12-18 03:20:17.427000,1
3,train_images/000c1434d8d7.png,3218676,2019-12-18 03:22:30.228000,10
4,train_images/001639a390f0.png,2261129,2019-12-18 03:22:43.817000,10
5,train_images/0024cdab0c1e.png,1882172,2019-12-18 03:22:38.982000,10
6,test_images/0005cfc8afb6.png,236005,2019-12-18 03:20:46.144000,1
7,test_images/003f0afdcd15.png,212295,2019-12-18 03:20:50.190000,1
8,test_images/006efc72b638.png,202491,2019-12-18 03:20:49.451000,1



================ IDRiD STRUCTURE CHECK ================
total_inventory_rows: 520
jpg_count: 516
expected_jpg_count: 516
official_image_name_matches: 516
training_label_filename_count: 1
testing_label_filename_count: 1
licence_filename_count: 2
label_content_opened: False

================ CORRECTED TARGET ASSESSMENT ================


,dataset,complete_pagination,official_metadata_rows,train_images,test_images,structure_pass,data_downloaded,label_content_opened,status
0,APTOS_2019,False,5593,3662,1928,False,False,False,FAIL_AFTER_COMPLETE_PAGINATION
1,IDRiD,NOT_APPLICABLE_LOCAL_RELEASE,520,LOCAL_RELEASE_SPLIT_NOT_OPENED,LOCAL_RELEASE_SPLIT_NOT_OPENED,True,False,False,PASS_LOCAL_RELEASE_STRUCTURE



================ FINAL STAGE 1C DECISION ================
Decision:
HOLD_TARGET_ACQUISITION_AFTER_COMPLETE_PAGINATION_CHECK

Interpretation:
At least one target still failed after complete pagination: APTOS complete paginated structure.

Authorised next step:
REVIEW_COMPLETE_STRUCTURE_CHECKS

Stage 1B decision superseded:
True

Target data downloaded:
False

Target-label content accessed:
False

Stage 1C complete-pagination repair completed and sealed.


In [7]:
#@title 01D. Correct pandas boolean identity bug and finalise target feasibility

from pathlib import Path
from datetime import datetime, timezone

import json
import platform
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Safety boundary
# ============================================================

TARGET_DATA_DOWNLOADED = False
TARGET_LABEL_CONTENT_ACCESSED = False

SAFETY_BOUNDARY = (
    "This administrative correction re-evaluates already sealed "
    "Stage 1C metadata files only. It does not contact Kaggle, "
    "download target data, open images, or read target-label contents."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

STAGE1_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
    / "Stage1_Target_Feasibility_v0.1"
)

STAGE1_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 2. Resolve sealed Stage 1C inputs
# ============================================================

APTOS_FULL_LISTING_PATH = (
    STAGE1_ROOT
    / "Stage1C_APTOS_Full_Paginated_File_Listing_v0.1.csv"
)

PAGINATION_AUDIT_PATH = (
    STAGE1_ROOT
    / "Stage1C_APTOS_Pagination_Audit_v0.1.csv"
)

LOCAL_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)

STAGE1C_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1C_Final_Target_Feasibility_Decision_v0.1.json"
)


required_inputs = [
    APTOS_FULL_LISTING_PATH,
    PAGINATION_AUDIT_PATH,
    LOCAL_INVENTORY_PATH,
    STAGE1C_DECISION_PATH,
]


missing_inputs = [
    path
    for path in required_inputs
    if not path.is_file()
]


assert not missing_inputs, (
    "Missing Stage 1D input artifacts:\n"
    +
    "\n".join(
        map(str, missing_inputs)
    )
)


aptos_listing = pd.read_csv(
    APTOS_FULL_LISTING_PATH
)

page_audit = pd.read_csv(
    PAGINATION_AUDIT_PATH
)

local_inventory = pd.read_csv(
    LOCAL_INVENTORY_PATH
)


with open(
    STAGE1C_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage1c_decision = json.load(
        file
    )


assert (
    stage1c_decision["decision"]
    ==
    "HOLD_TARGET_ACQUISITION_AFTER_COMPLETE_PAGINATION_CHECK"
), stage1c_decision["decision"]


print(
    "================ STAGE 1D BOOLEAN CORRECTION "
    "================"
)

print("\nPrevious Stage 1C decision:")
print(
    stage1c_decision["decision"]
)

print("\nCorrection reason:")
print(
    "Stage 1C used Python object identity `is False` on a "
    "pandas/numpy boolean. The saved final-page value is False, "
    "but numpy.bool_(False) is not the same object as Python False."
)

print("\nSafety boundary:")
print(SAFETY_BOUNDARY)


# ============================================================
# 3. Verify pagination completeness correctly
# ============================================================

required_pagination_columns = [
    "page",
    "rows_returned",
    "cumulative_rows",
    "next_page_token_present",
]


missing_pagination_columns = [
    column
    for column in required_pagination_columns
    if column not in page_audit.columns
]


assert not missing_pagination_columns, (
    "Missing pagination columns: "
    f"{missing_pagination_columns}"
)


page_audit = (
    page_audit
    .sort_values("page")
    .reset_index(drop=True)
)


last_page = page_audit.iloc[-1]


last_token_present = bool(
    last_page[
        "next_page_token_present"
    ]
)


pagination_complete = bool(
    len(page_audit) >= 2
    and
    int(
        last_page["rows_returned"]
    ) > 0
    and
    not last_token_present
    and
    int(
        last_page["cumulative_rows"]
    )
    ==
    len(aptos_listing)
)


assert pagination_complete, (
    "Corrected pagination check still failed.\n"
    f"Pages: {len(page_audit)}\n"
    f"Last rows returned: {last_page['rows_returned']}\n"
    f"Last cumulative rows: {last_page['cumulative_rows']}\n"
    f"Saved listing rows: {len(aptos_listing)}\n"
    f"Last next-page token present: {last_token_present}"
)


# ============================================================
# 4. Verify complete APTOS structure
# ============================================================

assert "name" in aptos_listing.columns


aptos_listing["name"] = (
    aptos_listing["name"]
    .astype(str)
    .str.strip()
)


duplicate_name_count = int(
    aptos_listing[
        "name"
    ].duplicated().sum()
)


assert duplicate_name_count == 0, (
    f"Duplicate APTOS filenames detected: "
    f"{duplicate_name_count}"
)


aptos_names = (
    aptos_listing[
        "name"
    ]
    .str.lower()
)


train_image_mask = (
    aptos_names
    .str.startswith(
        "train_images/"
    )
    &
    aptos_names
    .str.endswith(
        (
            ".png",
            ".jpg",
            ".jpeg",
        )
    )
)


test_image_mask = (
    aptos_names
    .str.startswith(
        "test_images/"
    )
    &
    aptos_names
    .str.endswith(
        (
            ".png",
            ".jpg",
            ".jpeg",
        )
    )
)


train_image_count = int(
    train_image_mask.sum()
)

test_image_count = int(
    test_image_mask.sum()
)


train_csv_present = bool(
    (
        aptos_names
        ==
        "train.csv"
    ).any()
)

test_csv_present = bool(
    (
        aptos_names
        ==
        "test.csv"
    ).any()
)

sample_submission_present = bool(
    (
        aptos_names
        ==
        "sample_submission.csv"
    ).any()
)


non_image_required_file_count = int(
    train_csv_present
    +
    test_csv_present
    +
    sample_submission_present
)


expected_total_from_components = int(
    train_image_count
    +
    test_image_count
    +
    non_image_required_file_count
)


aptos_components = {
    "pagination_complete": (
        pagination_complete
    ),
    "pages_retrieved": int(
        len(page_audit)
    ),
    "official_metadata_rows": int(
        len(aptos_listing)
    ),
    "duplicate_file_names": (
        duplicate_name_count
    ),
    "train_image_count": (
        train_image_count
    ),
    "test_image_count": (
        test_image_count
    ),
    "train_csv_present": (
        train_csv_present
    ),
    "test_csv_present": (
        test_csv_present
    ),
    "sample_submission_present": (
        sample_submission_present
    ),
    "component_count_sum": (
        expected_total_from_components
    ),
    "component_count_matches_listing": bool(
        expected_total_from_components
        ==
        len(aptos_listing)
    ),
    "target_data_downloaded": False,
    "label_content_opened": False,
}


aptos_structure_pass = bool(
    pagination_complete
    and
    train_image_count == 3662
    and
    test_image_count == 1928
    and
    train_csv_present
    and
    test_csv_present
    and
    sample_submission_present
    and
    expected_total_from_components
    ==
    len(aptos_listing)
)


assert aptos_structure_pass, (
    "APTOS corrected structure verification failed:\n"
    +
    json.dumps(
        aptos_components,
        indent=2,
    )
)


# ============================================================
# 5. Re-verify IDRiD structure from sealed filename inventory
# ============================================================

idrid_inventory = (
    local_inventory[
        local_inventory["dataset"]
        ==
        "IDRiD"
    ]
    .copy()
    .reset_index(drop=True)
)


assert len(idrid_inventory) > 0


idrid_inventory[
    "filename_lower"
] = (
    idrid_inventory[
        "filename"
    ]
    .astype(str)
    .str.lower()
)


idrid_jpgs = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            ".jpg",
            ".jpeg",
        ]
    )
].copy()


idrid_csvs = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    ==
    ".csv"
].copy()


idrid_licences = idrid_inventory[
    idrid_inventory[
        "filename_lower"
    ]
    .str.contains(
        "license|cc-by|cc_by",
        regex=True,
    )
].copy()


idrid_training_labels = idrid_csvs[
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "training",
        regex=False,
    )
    &
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


idrid_testing_labels = idrid_csvs[
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "testing",
        regex=False,
    )
    &
    idrid_csvs[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


idrid_name_match_count = int(
    idrid_jpgs[
        "filename"
    ]
    .astype(str)
    .str.match(
        r"(?i)^IDRiD_\d{3}\.(jpg|jpeg)$",
        na=False,
    )
    .sum()
)


idrid_components = {
    "total_inventory_rows": int(
        len(idrid_inventory)
    ),
    "jpg_count": int(
        len(idrid_jpgs)
    ),
    "expected_jpg_count": 516,
    "official_image_name_matches": (
        idrid_name_match_count
    ),
    "training_label_filename_count": int(
        len(idrid_training_labels)
    ),
    "testing_label_filename_count": int(
        len(idrid_testing_labels)
    ),
    "licence_filename_count": int(
        len(idrid_licences)
    ),
    "target_data_downloaded_in_this_stage": False,
    "label_content_opened": False,
}


idrid_structure_pass = bool(
    idrid_components[
        "jpg_count"
    ]
    ==
    516
    and
    idrid_components[
        "official_image_name_matches"
    ]
    ==
    516
    and
    idrid_components[
        "training_label_filename_count"
    ]
    >=
    1
    and
    idrid_components[
        "testing_label_filename_count"
    ]
    >=
    1
    and
    idrid_components[
        "licence_filename_count"
    ]
    >=
    1
)


assert idrid_structure_pass, (
    "IDRiD structure verification failed:\n"
    +
    json.dumps(
        idrid_components,
        indent=2,
    )
)


# ============================================================
# 6. Final corrected feasibility decision
# ============================================================

final_decision = (
    "PASS_TWO_TARGETS_ADVANCE_TO_"
    "CONTROLLED_TARGET_ACQUISITION"
)


final_interpretation = (
    "APTOS complete pagination is confirmed across 28 pages "
    "and 5,593 unique official records: 3,662 training images, "
    "1,928 test images, and the three expected CSV files. "
    "The prior Stage 1C hold resulted solely from a pandas/numpy "
    "boolean identity-comparison bug. IDRiD retains its verified "
    "516-image local release structure. No target data were "
    "downloaded and no target-label contents were opened during "
    "this correction."
)


authorised_next_step = (
    "CONTROLLED_TARGET_ACQUISITION_WITH_"
    "LABEL_QUARANTINE"
)


# ============================================================
# 7. Save Stage 1D corrected artifacts
# ============================================================

CORRECTED_ASSESSMENT_PATH = (
    STAGE1_ROOT
    / "Stage1D_Corrected_Target_Assessment_v0.1.csv"
)

SUPERSESSION_PATH = (
    STAGE1_ROOT
    / "Stage1D_Stage1C_Supersession_Manifest_v0.1.json"
)

FINAL_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1D_Final_Target_Feasibility_Decision_v0.1.json"
)

FINAL_REPORT_PATH = (
    STAGE1_ROOT
    / "Stage1D_Final_Target_Feasibility_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    STAGE1_ROOT
    / "Stage1D_Target_Feasibility_Environment_v0.1.json"
)


corrected_assessment = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "complete_pagination": True,
            "pages_retrieved": int(
                len(page_audit)
            ),
            "official_metadata_rows": int(
                len(aptos_listing)
            ),
            "train_images": (
                train_image_count
            ),
            "test_images": (
                test_image_count
            ),
            "structure_pass": True,
            "data_downloaded_in_this_stage": False,
            "label_content_opened": False,
            "status": (
                "PASS_PENDING_CONTROLLED_ACQUISITION"
            ),
        },
        {
            "dataset": "IDRiD",
            "complete_pagination": (
                "NOT_APPLICABLE_LOCAL_RELEASE"
            ),
            "pages_retrieved": (
                "NOT_APPLICABLE"
            ),
            "official_metadata_rows": int(
                len(idrid_inventory)
            ),
            "train_images": (
                "LOCAL_RELEASE_SPLIT_NOT_OPENED"
            ),
            "test_images": (
                "LOCAL_RELEASE_SPLIT_NOT_OPENED"
            ),
            "structure_pass": True,
            "data_downloaded_in_this_stage": False,
            "label_content_opened": False,
            "status": (
                "PASS_LOCAL_RELEASE_STRUCTURE"
            ),
        },
    ]
)


corrected_assessment.to_csv(
    CORRECTED_ASSESSMENT_PATH,
    index=False,
)


supersession_payload = {
    "superseded_artifact": str(
        STAGE1C_DECISION_PATH
    ),
    "superseded_decision": (
        stage1c_decision["decision"]
    ),
    "supersession_status": (
        "SUPERSEDED_DUE_TO_NUMPY_BOOLEAN_IDENTITY_BUG"
    ),
    "technical_cause": (
        "The expression "
        "`page_audit.iloc[-1]['next_page_token_present'] is False` "
        "used object identity instead of boolean value comparison. "
        "The saved value was numpy.bool_(False), which displays as "
        "False but is not the same object as Python False."
    ),
    "evidence_of_complete_pagination": {
        "pages": int(
            len(page_audit)
        ),
        "last_page_rows": int(
            last_page["rows_returned"]
        ),
        "last_page_token_present": (
            last_token_present
        ),
        "cumulative_rows": int(
            last_page["cumulative_rows"]
        ),
        "saved_listing_rows": int(
            len(aptos_listing)
        ),
    },
    "replacement_artifact": str(
        FINAL_DECISION_PATH
    ),
    "old_artifact_deleted": False,
}


with open(
    SUPERSESSION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        supersession_payload,
        file,
        indent=2,
    )


decision_payload = {
    "decision": final_decision,
    "interpretation": (
        final_interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "aptos_structure_pass": True,
    "aptos_components": (
        aptos_components
    ),
    "idrid_structure_pass": True,
    "idrid_components": (
        idrid_components
    ),
    "target_data_downloaded": False,
    "target_label_content_accessed": False,
    "supersedes_stage1c": True,
    "supersession_manifest": str(
        SUPERSESSION_PATH
    ),
    "important_boundary": (
        "This correction establishes target access and "
        "file-structure feasibility only. It does not inspect "
        "target-label values, evaluate recoverability, reveal "
        "transfer performance, or validate CDO."
    ),
}


with open(
    FINAL_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 1D",
    "",
    "## Corrected final target-feasibility decision",
    "",
    f"Decision: `{final_decision}`",
    "",
    final_interpretation,
    "",
    "## APTOS verification",
    "",
    f"- Metadata pages retrieved: {len(page_audit)}",
    f"- Unique official records: {len(aptos_listing)}",
    f"- Training images: {train_image_count}",
    f"- Test images: {test_image_count}",
    "- train.csv present: `True`",
    "- test.csv present: `True`",
    "- sample_submission.csv present: `True`",
    "",
    "## IDRiD verification",
    "",
    f"- Local JPG count: {len(idrid_jpgs)}",
    (
        f"- Official image-name matches: "
        f"{idrid_name_match_count}"
    ),
    "",
    "## Supersession",
    "",
    (
        "Stage 1C is retained for audit history but superseded "
        "because it used object identity on a numpy boolean."
    ),
    "",
    "## Safety",
    "",
    "- Target data downloaded: `False`",
    "- Target-label contents opened: `False`",
    "",
    "## Authorised next step",
    "",
    f"`{authorised_next_step}`",
    "",
]


FINAL_REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "stage1_root": str(
        STAGE1_ROOT
    ),
    "correction_type": (
        "NUMPY_BOOLEAN_IDENTITY_TO_VALUE_COMPARISON"
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 8. Final integrity checks and display
# ============================================================

assert TARGET_DATA_DOWNLOADED is False
assert TARGET_LABEL_CONTENT_ACCESSED is False


required_outputs = [
    CORRECTED_ASSESSMENT_PATH,
    SUPERSESSION_PATH,
    FINAL_DECISION_PATH,
    FINAL_REPORT_PATH,
    ENVIRONMENT_PATH,
]


for path in required_outputs:
    assert path.is_file(), path


print(
    "\n================ CORRECTED PAGINATION CHECK "
    "================"
)

print(
    "Pages retrieved:",
    len(page_audit),
)

print(
    "Last page rows:",
    int(
        last_page["rows_returned"]
    ),
)

print(
    "Last next-page token present:",
    last_token_present,
)

print(
    "Cumulative rows:",
    int(
        last_page["cumulative_rows"]
    ),
)

print(
    "Saved full-listing rows:",
    len(aptos_listing),
)

print(
    "Corrected pagination complete:",
    pagination_complete,
)


print(
    "\n================ APTOS COUNT CONSISTENCY "
    "================"
)

print(
    "Training images:",
    train_image_count,
)

print(
    "Test images:",
    test_image_count,
)

print(
    "Required CSV files:",
    non_image_required_file_count,
)

print(
    "Component total:",
    expected_total_from_components,
)

print(
    "Official listing total:",
    len(aptos_listing),
)

print(
    "Counts match:",
    (
        expected_total_from_components
        ==
        len(aptos_listing)
    ),
)


print(
    "\n================ CORRECTED TARGET ASSESSMENT "
    "================"
)

display(
    corrected_assessment
)


print(
    "\n================ FINAL STAGE 1D DECISION "
    "================"
)

print("Decision:")
print(final_decision)

print("\nInterpretation:")
print(final_interpretation)

print("\nAuthorised next step:")
print(authorised_next_step)

print("\nStage 1C decision superseded:")
print(True)

print("\nTarget data downloaded:")
print(False)

print("\nTarget-label content accessed:")
print(False)


print(
    "\nStage 1D administrative correction "
    "completed and sealed."
)

================ STAGE 1D BOOLEAN CORRECTION ================

Previous Stage 1C decision:
HOLD_TARGET_ACQUISITION_AFTER_COMPLETE_PAGINATION_CHECK

Correction reason:
Stage 1C used Python object identity `is False` on a pandas/numpy boolean. The saved final-page value is False, but numpy.bool_(False) is not the same object as Python False.

Safety boundary:
This administrative correction re-evaluates already sealed Stage 1C metadata files only. It does not contact Kaggle, download target data, open images, or read target-label contents.

================ CORRECTED PAGINATION CHECK ================
Pages retrieved: 28
Last page rows: 193
Last next-page token present: False
Cumulative rows: 5593
Saved full-listing rows: 5593
Corrected pagination complete: True

================ APTOS COUNT CONSISTENCY ================
Training images: 3662
Test images: 1928
Required CSV files: 3
Component total: 5593
Official listing total: 5593
Counts match: True

================ CORRECTED TARGET ASSES

,dataset,complete_pagination,pages_retrieved,official_metadata_rows,train_images,test_images,structure_pass,data_downloaded_in_this_stage,label_content_opened,status
0,APTOS_2019,True,28,5593,3662,1928,True,False,False,PASS_PENDING_CONTROLLED_ACQUISITION
1,IDRiD,NOT_APPLICABLE_LOCAL_RELEASE,NOT_APPLICABLE,520,LOCAL_RELEASE_SPLIT_NOT_OPENED,LOCAL_RELEASE_SPLIT_NOT_OPENED,True,False,False,PASS_LOCAL_RELEASE_STRUCTURE



================ FINAL STAGE 1D DECISION ================
Decision:
PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION

Interpretation:
APTOS complete pagination is confirmed across 28 pages and 5,593 unique official records: 3,662 training images, 1,928 test images, and the three expected CSV files. The prior Stage 1C hold resulted solely from a pandas/numpy boolean identity-comparison bug. IDRiD retains its verified 516-image local release structure. No target data were downloaded and no target-label contents were opened during this correction.

Authorised next step:
CONTROLLED_TARGET_ACQUISITION_WITH_LABEL_QUARANTINE

Stage 1C decision superseded:
True

Target data downloaded:
False

Target-label content accessed:
False

Stage 1D administrative correction completed and sealed.


In [12]:
#@title 02R. Storage-efficient controlled acquisition, canonicalisation, and label quarantine

from pathlib import Path
from datetime import datetime, timezone

from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen protocol
# ============================================================

RANDOM_SEED = 20260720
APTOS_EVALUATION_FRACTION = 0.30

APTOS_EXPECTED_IMAGES = 3662

IDRID_EXPECTED_TOTAL = 516
IDRID_EXPECTED_DEVELOPMENT = 413
IDRID_EXPECTED_EVALUATION = 103

COMPETITION_NAME = (
    "aptos2019-blindness-detection"
)

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

CANONICAL_SIZE = 768
CANONICAL_JPEG_QUALITY = 92
CANONICAL_JPEG_SUBSAMPLING = 0
BLACK_BORDER_THRESHOLD = 10

TARGET_DATA_DOWNLOADED_THIS_RUN = False
TARGET_LABEL_CONTENT_ACCESSED_BY_QUARANTINE = False


PROTOCOL_BOUNDARY = (
    "Target originals may exist only in temporary Colab storage. "
    "Every image is deterministically converted into the frozen "
    "768x768 canonical JPEG representation before being copied "
    "to Google Drive. Target labels may be read only by the "
    "quarantine procedure to construct development and sealed "
    "evaluation partitions. Sealed-evaluation label values must "
    "not be displayed or inserted into modelling manifests."
)


CANONICALISATION_DESCRIPTION = (
    "Convert to RGB; identify non-black retinal content using "
    f"grayscale intensity > {BLACK_BORDER_THRESHOLD}; crop to "
    "that bounding box when valid; pad centrally to a square "
    "with black pixels; resize with LANCZOS to "
    f"{CANONICAL_SIZE}x{CANONICAL_SIZE}; save as JPEG quality "
    f"{CANONICAL_JPEG_QUALITY}, subsampling "
    f"{CANONICAL_JPEG_SUBSAMPLING}. The same procedure must be "
    "applied to all future source and target datasets."
)


# ============================================================
# 1. Resolve Google Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

DESIGN_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
)

STAGE1_ROOT = (
    DESIGN_ROOT
    / "Stage1_Target_Feasibility_v0.1"
)

TEST_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

PROTOCOL_ROOT = (
    TEST_ROOT
    / "00_Protocol"
)

CANONICAL_ROOT = (
    TEST_ROOT
    / "01_Canonical_Target_Images"
)

DEVELOPMENT_ROOT = (
    TEST_ROOT
    / "02_Development"
)

SEALED_EVALUATION_ROOT = (
    TEST_ROOT
    / "03_Sealed_Evaluation"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

ORIGINAL_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "00_Original_Source_Label_Files"
)

SEALED_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "01_Sealed_Evaluation_Labels"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)


for directory in [
    PROTOCOL_ROOT,
    CANONICAL_ROOT,
    DEVELOPMENT_ROOT,
    SEALED_EVALUATION_ROOT,
    ORIGINAL_LABEL_ROOT,
    SEALED_LABEL_ROOT,
    STAGE2_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


APTOS_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019"
)

IDRID_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "IDRiD"
)


# The old failed cell may have created only empty folders.
OLD_RAW_ROOT = (
    TEST_ROOT
    / "01_Raw_Target_Images"
)

if (
    OLD_RAW_ROOT.is_dir()
    and
    not any(
        OLD_RAW_ROOT.rglob("*")
    )
):
    OLD_RAW_ROOT.rmdir()


# ============================================================
# 2. Import Stage 1D authorisation
# ============================================================

STAGE1D_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1D_Final_Target_Feasibility_Decision_v0.1.json"
)

IDRID_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)


assert STAGE1D_DECISION_PATH.is_file(), (
    STAGE1D_DECISION_PATH
)

assert IDRID_INVENTORY_PATH.is_file(), (
    IDRID_INVENTORY_PATH
)


with open(
    STAGE1D_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage1d_decision = json.load(
        file
    )


assert (
    stage1d_decision["decision"]
    ==
    "PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION"
)

assert (
    stage1d_decision["authorised_next_step"]
    ==
    "CONTROLLED_TARGET_ACQUISITION_WITH_LABEL_QUARANTINE"
)


idrid_stage1_inventory = pd.read_csv(
    IDRID_INVENTORY_PATH
)


print(
    "================ STAGE 2R BOOTSTRAP ================"
)

print("\nImported Stage 1D decision:")
print(
    stage1d_decision["decision"]
)

print("\nProtocol boundary:")
print(PROTOCOL_BOUNDARY)

print("\nFrozen canonicalisation:")
print(CANONICALISATION_DESCRIPTION)


# ============================================================
# 3. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with open(
        path,
        "rb",
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def directory_size_bytes(path):
    path = Path(path)

    return int(
        sum(
            file_path.stat().st_size
            for file_path in path.rglob("*")
            if file_path.is_file()
        )
    )


def bytes_to_gib(value):
    return float(
        value
        /
        (1024 ** 3)
    )


def normalise_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def resolve_column(
    dataframe,
    candidates,
    description,
):
    lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        key = normalise_column_name(
            candidate
        )

        if key in lookup:
            return lookup[key]

    raise KeyError(
        f"Could not resolve {description}.\n"
        f"Candidates: {candidates}\n"
        f"Columns: {list(dataframe.columns)}"
    )


def normalise_image_id(value):
    return Path(
        str(value).strip()
    ).stem


def make_read_only(path):
    try:
        os.chmod(
            path,
            0o400,
        )

        return True

    except Exception:
        return False


def verified_copy(
    source,
    destination,
):
    source = Path(source)
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    source_hash = sha256_file(
        source
    )

    shutil.copy2(
        source,
        destination,
    )

    destination_hash = sha256_file(
        destination
    )

    assert source_hash == destination_hash

    return {
        "source_path": str(source),
        "destination_path": str(destination),
        "sha256": source_hash,
        "hash_verified": True,
    }


def locate_kaggle_credential():
    candidates = [
        Path.home()
        / ".kaggle"
        / "kaggle.json",

        Path.home()
        / ".config"
        / "kaggle"
        / "kaggle.json",
    ]

    for candidate in candidates:
        if candidate.is_file():
            return candidate

    from google.colab import files

    print(
        "\nThe Kaggle credential is no longer present "
        "in this Colab runtime."
    )

    print(
        "Select the same kaggle.json file once."
    )

    uploaded = files.upload()

    candidates_uploaded = [
        name
        for name in uploaded
        if Path(name).suffix.lower()
        ==
        ".json"
    ]

    if len(candidates_uploaded) != 1:
        raise RuntimeError(
            "Exactly one kaggle.json file is required."
        )

    uploaded_name = candidates_uploaded[0]

    payload = json.loads(
        uploaded[
            uploaded_name
        ].decode("utf-8")
    )

    if not {
        "username",
        "key",
    }.issubset(
        payload.keys()
    ):
        raise RuntimeError(
            "The selected file is not a legacy "
            "Kaggle credential."
        )

    credential_path = (
        Path.home()
        / ".kaggle"
        / "kaggle.json"
    )

    credential_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    credential_path.write_bytes(
        uploaded[
            uploaded_name
        ]
    )

    os.chmod(
        credential_path,
        0o600,
    )

    temporary_upload = Path(
        uploaded_name
    )

    if temporary_upload.exists():
        temporary_upload.unlink()

    return credential_path


def crop_and_canonicalise(
    source_path,
    destination_path,
):
    source_path = Path(
        source_path
    )

    destination_path = Path(
        destination_path
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    original_sha256 = sha256_file(
        source_path
    )

    with Image.open(
        source_path
    ) as image:
        image.load()

        original_width = int(
            image.width
        )

        original_height = int(
            image.height
        )

        rgb = image.convert(
            "RGB"
        )

        grayscale = rgb.convert(
            "L"
        )

        binary_mask = grayscale.point(
            lambda pixel: (
                255
                if pixel
                >
                BLACK_BORDER_THRESHOLD
                else
                0
            )
        )

        crop_box = binary_mask.getbbox()

        if crop_box is None:
            crop_box = (
                0,
                0,
                original_width,
                original_height,
            )

        cropped = rgb.crop(
            crop_box
        )

        side_length = max(
            cropped.width,
            cropped.height,
        )

        square = Image.new(
            "RGB",
            (
                side_length,
                side_length,
            ),
            (
                0,
                0,
                0,
            ),
        )

        paste_x = (
            side_length
            -
            cropped.width
        ) // 2

        paste_y = (
            side_length
            -
            cropped.height
        ) // 2

        square.paste(
            cropped,
            (
                paste_x,
                paste_y,
            ),
        )

        canonical = square.resize(
            (
                CANONICAL_SIZE,
                CANONICAL_SIZE,
            ),
            resample=(
                Image.Resampling.LANCZOS
            ),
        )

        canonical.save(
            destination_path,
            format="JPEG",
            quality=(
                CANONICAL_JPEG_QUALITY
            ),
            subsampling=(
                CANONICAL_JPEG_SUBSAMPLING
            ),
            optimize=True,
            progressive=False,
        )

    canonical_sha256 = sha256_file(
        destination_path
    )

    with Image.open(
        destination_path
    ) as verification_image:
        verification_image.verify()

    return {
        "original_sha256": (
            original_sha256
        ),
        "canonical_sha256": (
            canonical_sha256
        ),
        "original_width": (
            original_width
        ),
        "original_height": (
            original_height
        ),
        "crop_left": int(
            crop_box[0]
        ),
        "crop_top": int(
            crop_box[1]
        ),
        "crop_right": int(
            crop_box[2]
        ),
        "crop_bottom": int(
            crop_box[3]
        ),
        "canonical_width": (
            CANONICAL_SIZE
        ),
        "canonical_height": (
            CANONICAL_SIZE
        ),
        "canonical_size_bytes": int(
            destination_path.stat().st_size
        ),
    }


def json_default(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    raise TypeError(
        f"Unsupported JSON value: "
        f"{type(value)}"
    )


# ============================================================
# 4. Storage preflight
# ============================================================

local_disk = shutil.disk_usage(
    "/content"
)

drive_disk = shutil.disk_usage(
    DRIVE_ROOT
)


print(
    "\n================ STORAGE PREFLIGHT "
    "================"
)

print(
    "Colab local free space:",
    f"{bytes_to_gib(local_disk.free):.2f} GiB",
)

print(
    "Google Drive reported free space:",
    f"{bytes_to_gib(drive_disk.free):.2f} GiB",
)

print(
    "Minimum Drive free space required by "
    "storage-efficient protocol: 3.00 GiB"
)


assert (
    local_disk.free
    >
    25
    *
    1024 ** 3
), (
    "At least 25 GiB of local Colab space is "
    "required for temporary download, extraction "
    "and canonicalisation."
)


assert (
    drive_disk.free
    >
    3
    *
    1024 ** 3
), (
    "At least 3 GiB of Google Drive space is required "
    "for the canonical 768x768 target images."
)


# ============================================================
# 5. Temporary local workspace
# ============================================================

TEMP_ROOT = Path(
    "/content"
    /
    "cdo_stage2r_storage_efficient"
)

DOWNLOAD_ROOT = (
    TEMP_ROOT
    / "download"
)

EXTRACT_ROOT = (
    TEMP_ROOT
    / "extract"
)

LOCAL_CANONICAL_ROOT = (
    TEMP_ROOT
    / "canonical"
)

LOCAL_APTOS_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "APTOS_2019"
)

LOCAL_IDRID_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "IDRiD"
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


for directory in [
    DOWNLOAD_ROOT,
    EXTRACT_ROOT,
    LOCAL_APTOS_CANONICAL,
    LOCAL_IDRID_CANONICAL,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 6. Download APTOS into temporary local storage
# ============================================================

locate_kaggle_credential()


print(
    "\n================ APTOS TEMPORARY DOWNLOAD "
    "================"
)

download_result = subprocess.run(
    [
        "kaggle",
        "competitions",
        "download",
        "-c",
        COMPETITION_NAME,
        "-p",
        str(
            DOWNLOAD_ROOT
        ),
        "--force",
    ],
    capture_output=True,
    text=True,
    timeout=14400,
)


if download_result.returncode != 0:
    raise RuntimeError(
        "APTOS download failed:\n"
        +
        (
            download_result.stdout
            +
            "\n"
            +
            download_result.stderr
        )[-3000:]
    )


TARGET_DATA_DOWNLOADED_THIS_RUN = True


zip_candidates = sorted(
    DOWNLOAD_ROOT.glob(
        "*.zip"
    ),
    key=lambda path: (
        path.stat().st_mtime
    ),
    reverse=True,
)


assert zip_candidates, (
    "Kaggle did not produce a ZIP archive."
)


aptos_zip_path = zip_candidates[0]


print(
    "Downloaded archive:",
    aptos_zip_path,
)

print(
    "Archive size:",
    f"{bytes_to_gib(aptos_zip_path.stat().st_size):.2f} GiB",
)


# ============================================================
# 7. Extract only labelled APTOS training material
# ============================================================

print(
    "\nExtracting only train.csv and train_images/..."
)


with zipfile.ZipFile(
    aptos_zip_path,
    "r",
) as archive:
    archive_names = (
        archive.namelist()
    )

    train_image_members = [
        name
        for name in archive_names
        if (
            name.startswith(
                "train_images/"
            )
            and
            Path(name).suffix.lower()
            in {
                ".png",
                ".jpg",
                ".jpeg",
            }
        )
    ]

    assert (
        len(
            train_image_members
        )
        ==
        APTOS_EXPECTED_IMAGES
    ), (
        f"Expected {APTOS_EXPECTED_IMAGES} "
        f"APTOS training images, found "
        f"{len(train_image_members)}."
    )

    assert (
        "train.csv"
        in archive_names
    )

    archive.extract(
        "train.csv",
        path=EXTRACT_ROOT,
    )

    for member in tqdm(
        train_image_members,
        desc="Extracting APTOS training images",
    ):
        member_path = Path(
            member
        )

        if (
            member_path.is_absolute()
            or
            ".."
            in member_path.parts
        ):
            raise RuntimeError(
                f"Unsafe ZIP member: {member}"
            )

        archive.extract(
            member,
            path=EXTRACT_ROOT,
        )


APTOS_EXTRACTED_IMAGE_ROOT = (
    EXTRACT_ROOT
    / "train_images"
)

APTOS_EXTRACTED_LABEL_PATH = (
    EXTRACT_ROOT
    / "train.csv"
)


assert APTOS_EXTRACTED_IMAGE_ROOT.is_dir()
assert APTOS_EXTRACTED_LABEL_PATH.is_file()


# ============================================================
# 8. Copy original APTOS label table into quarantine
# ============================================================

APTOS_ORIGINAL_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "APTOS_2019_Original_train.csv"
)


verified_copy(
    APTOS_EXTRACTED_LABEL_PATH,
    APTOS_ORIGINAL_LABEL_PATH,
)


# ============================================================
# 9. Locate and quarantine IDRiD grading files
# ============================================================

idrid_inventory = (
    idrid_stage1_inventory[
        idrid_stage1_inventory[
            "dataset"
        ]
        ==
        "IDRiD"
    ]
    .copy()
    .reset_index(drop=True)
)


idrid_inventory[
    "filename_lower"
] = (
    idrid_inventory[
        "filename"
    ]
    .astype(str)
    .str.lower()
)


idrid_image_rows = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            ".jpg",
            ".jpeg",
        ]
    )
].copy()


assert (
    len(idrid_image_rows)
    ==
    IDRID_EXPECTED_TOTAL
)


idrid_csv_rows = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    ==
    ".csv"
].copy()


idrid_training_rows = idrid_csv_rows[
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "training",
        regex=False,
    )
    &
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


idrid_testing_rows = idrid_csv_rows[
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "testing",
        regex=False,
    )
    &
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


assert len(
    idrid_training_rows
) == 1

assert len(
    idrid_testing_rows
) == 1


IDRID_ORIGINAL_TRAIN_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Training_Labels.csv"
)

IDRID_ORIGINAL_TEST_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Testing_Labels.csv"
)


def quarantine_existing_label(
    inventory_row,
    destination_path,
):
    source_path = Path(
        inventory_row["path"]
    )

    if source_path.is_file():
        record = verified_copy(
            source_path,
            destination_path,
        )

        source_path.unlink()

        record[
            "source_removed_after_hash_verification"
        ] = True

        return record

    if destination_path.is_file():
        return {
            "source_path": str(
                source_path
            ),
            "destination_path": str(
                destination_path
            ),
            "sha256": sha256_file(
                destination_path
            ),
            "hash_verified": True,
            "source_removed_after_hash_verification": (
                "ALREADY_RELOCATED"
            ),
        }

    raise FileNotFoundError(
        "IDRiD label file is missing from both "
        "its original location and quarantine:\n"
        f"{source_path}\n"
        f"{destination_path}"
    )


idrid_relocation_records = [
    quarantine_existing_label(
        idrid_training_rows.iloc[0],
        IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    ),
    quarantine_existing_label(
        idrid_testing_rows.iloc[0],
        IDRID_ORIGINAL_TEST_LABEL_PATH,
    ),
]


# ============================================================
# 10. Quarantine-only label reading
# ============================================================

TARGET_LABEL_CONTENT_ACCESSED_BY_QUARANTINE = True


aptos_labels_raw = pd.read_csv(
    APTOS_ORIGINAL_LABEL_PATH
)

idrid_train_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TRAIN_LABEL_PATH
)

idrid_test_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TEST_LABEL_PATH
)


aptos_id_column = resolve_column(
    aptos_labels_raw,
    [
        "id_code",
        "image_id",
        "image",
    ],
    "APTOS image ID",
)

aptos_grade_column = resolve_column(
    aptos_labels_raw,
    [
        "diagnosis",
        "grade",
        "dr_grade",
    ],
    "APTOS DR grade",
)


aptos_labels = pd.DataFrame(
    {
        "dataset": (
            "APTOS_2019"
        ),
        "image_id": (
            aptos_labels_raw[
                aptos_id_column
            ]
            .map(
                normalise_image_id
            )
        ),
        "original_dr_grade": (
            pd.to_numeric(
                aptos_labels_raw[
                    aptos_grade_column
                ],
                errors="raise",
            )
            .astype(int)
        ),
    }
)


assert len(
    aptos_labels
) == APTOS_EXPECTED_IMAGES

assert aptos_labels[
    "image_id"
].is_unique


def standardise_idrid_labels(
    dataframe,
    official_partition,
):
    image_column = resolve_column(
        dataframe,
        [
            "Image name",
            "image_name",
            "image",
        ],
        "IDRiD image name",
    )

    grade_column = resolve_column(
        dataframe,
        [
            "Retinopathy grade",
            "retinopathy_grade",
            "DR grade",
            "dr_grade",
            "grade",
        ],
        "IDRiD retinopathy grade",
    )

    return pd.DataFrame(
        {
            "dataset": "IDRiD",
            "image_id": (
                dataframe[
                    image_column
                ]
                .map(
                    normalise_image_id
                )
            ),
            "original_dr_grade": (
                pd.to_numeric(
                    dataframe[
                        grade_column
                    ],
                    errors="raise",
                )
                .astype(int)
            ),
            "official_partition": (
                official_partition
            ),
        }
    )


idrid_train_labels = (
    standardise_idrid_labels(
        idrid_train_labels_raw,
        "development",
    )
)

idrid_test_labels = (
    standardise_idrid_labels(
        idrid_test_labels_raw,
        "sealed_evaluation",
    )
)


assert (
    len(idrid_train_labels)
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(idrid_test_labels)
    ==
    IDRID_EXPECTED_EVALUATION
)


idrid_labels = pd.concat(
    [
        idrid_train_labels,
        idrid_test_labels,
    ],
    ignore_index=True,
)


assert (
    len(idrid_labels)
    ==
    IDRID_EXPECTED_TOTAL
)

assert idrid_labels[
    "image_id"
].is_unique


# ============================================================
# 11. Canonicalise APTOS images locally
# ============================================================

aptos_processing_records = []


aptos_source_paths = sorted(
    [
        path
        for path
        in APTOS_EXTRACTED_IMAGE_ROOT.iterdir()
        if (
            path.is_file()
            and
            path.suffix.lower()
            in {
                ".png",
                ".jpg",
                ".jpeg",
            }
        )
    ]
)


assert (
    len(aptos_source_paths)
    ==
    APTOS_EXPECTED_IMAGES
)


print(
    "\n================ APTOS CANONICALISATION "
    "================"
)


for source_path in tqdm(
    aptos_source_paths,
    desc="Canonicalising APTOS",
):
    image_id = source_path.stem

    destination_path = (
        LOCAL_APTOS_CANONICAL
        /
        f"{image_id}.jpg"
    )

    processing = crop_and_canonicalise(
        source_path,
        destination_path,
    )

    aptos_processing_records.append(
        {
            "dataset": (
                "APTOS_2019"
            ),
            "image_id": (
                image_id
            ),
            "original_filename": (
                source_path.name
            ),
            "canonical_filename": (
                destination_path.name
            ),
            **processing,
        }
    )


aptos_integrity = pd.DataFrame(
    aptos_processing_records
)


assert (
    len(aptos_integrity)
    ==
    APTOS_EXPECTED_IMAGES
)

assert aptos_integrity[
    "image_id"
].is_unique


# ============================================================
# 12. Canonicalise IDRiD images locally
# ============================================================

idrid_processing_records = []


print(
    "\n================ IDRiD CANONICALISATION "
    "================"
)


for _, inventory_row in tqdm(
    idrid_image_rows.iterrows(),
    total=len(
        idrid_image_rows
    ),
    desc="Canonicalising IDRiD",
):
    source_path = Path(
        inventory_row["path"]
    )

    assert source_path.is_file(), (
        source_path
    )

    image_id = source_path.stem

    destination_path = (
        LOCAL_IDRID_CANONICAL
        /
        f"{image_id}.jpg"
    )

    processing = crop_and_canonicalise(
        source_path,
        destination_path,
    )

    idrid_processing_records.append(
        {
            "dataset": "IDRiD",
            "image_id": image_id,
            "original_filename": (
                source_path.name
            ),
            "canonical_filename": (
                destination_path.name
            ),
            **processing,
        }
    )


idrid_integrity = pd.DataFrame(
    idrid_processing_records
)


assert (
    len(idrid_integrity)
    ==
    IDRID_EXPECTED_TOTAL
)

assert idrid_integrity[
    "image_id"
].is_unique


# ============================================================
# 13. Verify canonical output size before Drive copy
# ============================================================

aptos_canonical_bytes = (
    directory_size_bytes(
        LOCAL_APTOS_CANONICAL
    )
)

idrid_canonical_bytes = (
    directory_size_bytes(
        LOCAL_IDRID_CANONICAL
    )
)

canonical_total_bytes = (
    aptos_canonical_bytes
    +
    idrid_canonical_bytes
)


current_drive_free = (
    shutil.disk_usage(
        DRIVE_ROOT
    ).free
)


print(
    "\n================ CANONICAL STORAGE AUDIT "
    "================"
)

print(
    "APTOS canonical size:",
    f"{bytes_to_gib(aptos_canonical_bytes):.2f} GiB",
)

print(
    "IDRiD canonical size:",
    f"{bytes_to_gib(idrid_canonical_bytes):.2f} GiB",
)

print(
    "Total canonical size:",
    f"{bytes_to_gib(canonical_total_bytes):.2f} GiB",
)

print(
    "Current Drive free space:",
    f"{bytes_to_gib(current_drive_free):.2f} GiB",
)


assert (
    current_drive_free
    >
    canonical_total_bytes
    +
    512
    *
    1024 ** 2
), (
    "Canonical files are still too large for the "
    "remaining Drive space. No canonical images "
    "have yet been copied to Drive."
)


# ============================================================
# 14. Copy completed canonical sets to Drive
# ============================================================

APTOS_STAGING_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019__STAGING"
)

IDRID_STAGING_ROOT = (
    CANONICAL_ROOT
    / "IDRiD__STAGING"
)


for staging_root in [
    APTOS_STAGING_ROOT,
    IDRID_STAGING_ROOT,
]:
    if staging_root.exists():
        shutil.rmtree(
            staging_root
        )


print(
    "\nCopying canonical images to Google Drive..."
)


shutil.copytree(
    LOCAL_APTOS_CANONICAL,
    APTOS_STAGING_ROOT,
)

shutil.copytree(
    LOCAL_IDRID_CANONICAL,
    IDRID_STAGING_ROOT,
)


assert (
    len(
        list(
            APTOS_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    APTOS_EXPECTED_IMAGES
)

assert (
    len(
        list(
            IDRID_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_EXPECTED_TOTAL
)


if APTOS_CANONICAL_ROOT.exists():
    shutil.rmtree(
        APTOS_CANONICAL_ROOT
    )

if IDRID_CANONICAL_ROOT.exists():
    shutil.rmtree(
        IDRID_CANONICAL_ROOT
    )


APTOS_STAGING_ROOT.rename(
    APTOS_CANONICAL_ROOT
)

IDRID_STAGING_ROOT.rename(
    IDRID_CANONICAL_ROOT
)


# ============================================================
# 15. Join labels, hashes and final Drive paths
# ============================================================

aptos_integrity[
    "canonical_image_path"
] = (
    aptos_integrity[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            APTOS_CANONICAL_ROOT
            /
            f"{image_id}.jpg"
        )
    )
)


idrid_integrity[
    "canonical_image_path"
] = (
    idrid_integrity[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            IDRID_CANONICAL_ROOT
            /
            f"{image_id}.jpg"
        )
    )
)


aptos_verified = aptos_labels.merge(
    aptos_integrity,
    on=[
        "dataset",
        "image_id",
    ],
    how="inner",
    validate="one_to_one",
)


idrid_verified = idrid_labels.merge(
    idrid_integrity,
    on=[
        "dataset",
        "image_id",
    ],
    how="inner",
    validate="one_to_one",
)


assert (
    len(aptos_verified)
    ==
    APTOS_EXPECTED_IMAGES
)

assert (
    len(idrid_verified)
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 16. Deterministic APTOS partition
# ============================================================

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=(
        APTOS_EVALUATION_FRACTION
    ),
    random_state=RANDOM_SEED,
)


development_indices, evaluation_indices = next(
    splitter.split(
        np.zeros(
            len(aptos_verified)
        ),
        aptos_verified[
            "original_dr_grade"
        ].to_numpy(),
    )
)


aptos_verified[
    "prospective_partition"
] = "development"

aptos_verified.loc[
    evaluation_indices,
    "prospective_partition",
] = "sealed_evaluation"


# Exact original duplicates cannot cross partitions.
hash_partition_count = (
    aptos_verified
    .groupby(
        "original_sha256"
    )[
        "prospective_partition"
    ]
    .nunique()
)


crossing_duplicate_hashes = (
    hash_partition_count[
        hash_partition_count
        >
        1
    ]
    .index
    .tolist()
)


if crossing_duplicate_hashes:
    aptos_verified.loc[
        aptos_verified[
            "original_sha256"
        ].isin(
            crossing_duplicate_hashes
        ),
        "prospective_partition",
    ] = "sealed_evaluation"


assert not (
    aptos_verified
    .groupby(
        "original_sha256"
    )[
        "prospective_partition"
    ]
    .nunique()
    >
    1
).any()


# ============================================================
# 17. Preserve official IDRiD partition
# ============================================================

idrid_verified[
    "prospective_partition"
] = (
    idrid_verified[
        "official_partition"
    ]
)


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "development"
).sum() == IDRID_EXPECTED_DEVELOPMENT


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "sealed_evaluation"
).sum() == IDRID_EXPECTED_EVALUATION


assert not (
    idrid_verified
    .groupby(
        "original_sha256"
    )[
        "prospective_partition"
    ]
    .nunique()
    >
    1
).any()


cross_dataset_overlap = set(
    aptos_verified[
        "original_sha256"
    ]
).intersection(
    set(
        idrid_verified[
            "original_sha256"
        ]
    )
)


assert not cross_dataset_overlap


# ============================================================
# 18. Harmonised endpoint
# ============================================================

for dataframe in [
    aptos_verified,
    idrid_verified,
]:
    dataframe[
        "moderate_or_worse_dr"
    ] = (
        dataframe[
            "original_dr_grade"
        ]
        >=
        2
    ).astype(int)


# ============================================================
# 19. Partition tables
# ============================================================

aptos_development = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


aptos_evaluation = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_development = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_evaluation = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


# ============================================================
# 20. Save development labels and sealed outputs
# ============================================================

APTOS_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "APTOS_2019_Development_Labels_v0.1.csv"
)

IDRID_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "IDRiD_Development_Labels_v0.1.csv"
)

APTOS_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv"
)

IDRID_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv"
)

APTOS_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "APTOS_2019_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

IDRID_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "IDRiD_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)


development_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_dr_grade",
    "moderate_or_worse_dr",
    "original_sha256",
    "canonical_sha256",
]


evaluation_manifest_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_sha256",
    "canonical_sha256",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


sealed_label_columns = [
    "dataset",
    "image_id",
    "original_dr_grade",
    "moderate_or_worse_dr",
]


aptos_development[
    development_columns
].to_csv(
    APTOS_DEVELOPMENT_LABEL_PATH,
    index=False,
)

idrid_development[
    development_columns
].to_csv(
    IDRID_DEVELOPMENT_LABEL_PATH,
    index=False,
)


aptos_evaluation[
    evaluation_manifest_columns
].to_csv(
    APTOS_EVALUATION_MANIFEST_PATH,
    index=False,
)

idrid_evaluation[
    evaluation_manifest_columns
].to_csv(
    IDRID_EVALUATION_MANIFEST_PATH,
    index=False,
)


aptos_evaluation[
    sealed_label_columns
].to_csv(
    APTOS_SEALED_LABEL_PATH,
    index=False,
)

idrid_evaluation[
    sealed_label_columns
].to_csv(
    IDRID_SEALED_LABEL_PATH,
    index=False,
)


for protected_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_read_only(
        protected_path
    )


# ============================================================
# 21. Label-free master manifest
# ============================================================

label_free_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "prospective_partition",
    "original_sha256",
    "canonical_sha256",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


label_free_master_manifest = pd.concat(
    [
        aptos_verified[
            label_free_columns
        ],
        idrid_verified[
            label_free_columns
        ],
    ],
    ignore_index=True,
)


assert (
    "original_dr_grade"
    not in label_free_master_manifest.columns
)

assert (
    "moderate_or_worse_dr"
    not in label_free_master_manifest.columns
)


# ============================================================
# 22. Save protocol and audit artifacts
# ============================================================

CANONICAL_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Canonical_Fundus_Preprocessing_Protocol_v0.1.json"
)

QUARANTINE_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Target_Label_Quarantine_Protocol_v0.1.json"
)

MASTER_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_LabelFree_Master_Target_Manifest_v0.1.csv"
)

INTEGRITY_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

PARTITION_SUMMARY_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Partition_Summary_v0.1.csv"
)

LABEL_COMMITMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Sealed_Label_Commitments_v0.1.json"
)

RELOCATION_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_IDRiD_Label_Relocation_Manifest_v0.1.csv"
)

STAGE2_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)

STAGE2_REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Report_v0.1.md"
)

STAGE2_ENVIRONMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Environment_v0.1.json"
)


canonical_protocol = {
    "protocol_version": "v0.1",
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "black_border_threshold": (
        BLACK_BORDER_THRESHOLD
    ),
    "jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "jpeg_subsampling": (
        CANONICAL_JPEG_SUBSAMPLING
    ),
    "resize_filter": "LANCZOS",
    "description": (
        CANONICALISATION_DESCRIPTION
    ),
    "frozen_before_target_performance": True,
    "must_be_applied_to_sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
}


with open(
    CANONICAL_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        canonical_protocol,
        file,
        indent=2,
    )


quarantine_protocol = {
    "protocol_version": "v0.1",
    "endpoint_id": ENDPOINT_ID,
    "random_seed": RANDOM_SEED,
    "aptos_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "development_labels_authorised_from_stage": 3,
    "sealed_evaluation_labels_authorised_stage": 6,
    "sealed_evaluation_manifests_label_free": True,
    "prohibitions": [
        (
            "Do not open, display, summarise or model "
            "sealed-evaluation labels before Stage 6."
        ),
        (
            "Do not change the target partitions after "
            "recoverability or transfer results are observed."
        ),
        (
            "Do not change the canonicalisation protocol "
            "after target performance is observed."
        ),
        (
            "Do not insert evaluation labels into any "
            "label-free manifest."
        ),
    ],
}


with open(
    QUARANTINE_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quarantine_protocol,
        file,
        indent=2,
    )


label_free_master_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False,
)


integrity_manifest = pd.concat(
    [
        aptos_integrity,
        idrid_integrity,
    ],
    ignore_index=True,
)


integrity_manifest.to_csv(
    INTEGRITY_MANIFEST_PATH,
    index=False,
)


partition_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "total_images": int(
                len(aptos_verified)
            ),
            "development_images": int(
                len(aptos_development)
            ),
            "sealed_evaluation_images": int(
                len(aptos_evaluation)
            ),
            "partition_source": (
                "deterministic grade-stratified "
                "70/30 split"
            ),
            "exact_duplicate_groups_forced_to_one_partition": int(
                len(
                    crossing_duplicate_hashes
                )
            ),
            "sealed_evaluation_labels_displayed": False,
        },
        {
            "dataset": "IDRiD",
            "total_images": int(
                len(idrid_verified)
            ),
            "development_images": int(
                len(idrid_development)
            ),
            "sealed_evaluation_images": int(
                len(idrid_evaluation)
            ),
            "partition_source": (
                "official IDRiD training/testing split"
            ),
            "exact_duplicate_groups_forced_to_one_partition": 0,
            "sealed_evaluation_labels_displayed": False,
        },
    ]
)


partition_summary.to_csv(
    PARTITION_SUMMARY_PATH,
    index=False,
)


label_commitments = {
    "endpoint_id": ENDPOINT_ID,
    "aptos_sealed_evaluation": {
        "rows": int(
            len(aptos_evaluation)
        ),
        "path": str(
            APTOS_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            APTOS_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "idrid_sealed_evaluation": {
        "rows": int(
            len(idrid_evaluation)
        ),
        "path": str(
            IDRID_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "original_label_files": {
        "aptos_train_sha256": (
            sha256_file(
                APTOS_ORIGINAL_LABEL_PATH
            )
        ),
        "idrid_train_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TRAIN_LABEL_PATH
            )
        ),
        "idrid_test_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TEST_LABEL_PATH
            )
        ),
    },
}


with open(
    LABEL_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_commitments,
        file,
        indent=2,
    )


pd.DataFrame(
    idrid_relocation_records
).to_csv(
    RELOCATION_MANIFEST_PATH,
    index=False,
)


# ============================================================
# 23. Leakage and label-boundary checks
# ============================================================

aptos_cross_split_duplicates = int(
    (
        aptos_verified
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_cross_split_duplicates = int(
    (
        idrid_verified
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


evaluation_manifest_has_no_labels = bool(
    all(
        label_column
        not in pd.read_csv(
            manifest_path,
            nrows=1,
        ).columns
        for manifest_path in [
            APTOS_EVALUATION_MANIFEST_PATH,
            IDRID_EVALUATION_MANIFEST_PATH,
        ]
        for label_column in [
            "original_dr_grade",
            "moderate_or_worse_dr",
        ]
    )
)


stage2_pass = bool(
    len(aptos_verified)
    ==
    APTOS_EXPECTED_IMAGES
    and
    len(idrid_verified)
    ==
    IDRID_EXPECTED_TOTAL
    and
    len(idrid_development)
    ==
    IDRID_EXPECTED_DEVELOPMENT
    and
    len(idrid_evaluation)
    ==
    IDRID_EXPECTED_EVALUATION
    and
    aptos_cross_split_duplicates
    ==
    0
    and
    idrid_cross_split_duplicates
    ==
    0
    and
    len(cross_dataset_overlap)
    ==
    0
    and
    evaluation_manifest_has_no_labels
)


assert stage2_pass


# ============================================================
# 24. Final decision
# ============================================================

stage2_decision = (
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_"
    "AND_LABEL_QUARANTINE_ADVANCE_TO_"
    "DEVELOPMENT_RECOVERABILITY_GATE"
)


stage2_interpretation = (
    "APTOS originals were temporarily acquired in Colab, "
    "verified and converted into the frozen 768x768 canonical "
    "fundus representation before storage on Google Drive. "
    "IDRiD was processed with the identical canonicalisation. "
    "Original-image and canonical-image SHA-256 commitments "
    "were retained. APTOS was deterministically divided into "
    "development and sealed evaluation partitions; IDRiD "
    "retained its official split. Sealed labels were excluded "
    "from modelling manifests and committed for one-time "
    "Stage 6 evaluation."
)


decision_payload = {
    "decision": stage2_decision,
    "interpretation": (
        stage2_interpretation
    ),
    "authorised_next_step": (
        "DEVELOPMENT_ONLY_TARGET_RECOVERABILITY_GATE"
    ),
    "endpoint_id": ENDPOINT_ID,
    "canonicalisation_protocol": str(
        CANONICAL_PROTOCOL_PATH
    ),
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "target_data_downloaded_this_run": (
        TARGET_DATA_DOWNLOADED_THIS_RUN
    ),
    "target_label_content_accessed_by_quarantine": (
        TARGET_LABEL_CONTENT_ACCESSED_BY_QUARANTINE
    ),
    "aptos": {
        "total": int(
            len(aptos_verified)
        ),
        "development": int(
            len(aptos_development)
        ),
        "sealed_evaluation": int(
            len(aptos_evaluation)
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                aptos_canonical_bytes
            )
        ),
        "cross_split_exact_duplicates": (
            aptos_cross_split_duplicates
        ),
    },
    "idrid": {
        "total": int(
            len(idrid_verified)
        ),
        "development": int(
            len(idrid_development)
        ),
        "sealed_evaluation": int(
            len(idrid_evaluation)
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                idrid_canonical_bytes
            )
        ),
        "cross_split_exact_duplicates": (
            idrid_cross_split_duplicates
        ),
    },
    "cross_dataset_exact_overlap": int(
        len(cross_dataset_overlap)
    ),
    "evaluation_manifests_label_free": (
        evaluation_manifest_has_no_labels
    ),
    "sealed_label_commitment_path": str(
        LABEL_COMMITMENT_PATH
    ),
    "important_boundary": (
        "This stage establishes acquisition, frozen "
        "canonicalisation and procedural label quarantine. "
        "It does not establish target recoverability, "
        "source-target transfer or CDO validity."
    ),
}


with open(
    STAGE2_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
        default=json_default,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 2R",
    "",
    "## Storage-efficient controlled acquisition",
    "",
    f"Decision: `{stage2_decision}`",
    "",
    stage2_interpretation,
    "",
    "## Canonical representation",
    "",
    f"- Size: {CANONICAL_SIZE} x {CANONICAL_SIZE}",
    (
        f"- JPEG quality: "
        f"{CANONICAL_JPEG_QUALITY}"
    ),
    (
        f"- APTOS storage: "
        f"{bytes_to_gib(aptos_canonical_bytes):.3f} GiB"
    ),
    (
        f"- IDRiD storage: "
        f"{bytes_to_gib(idrid_canonical_bytes):.3f} GiB"
    ),
    "",
    "## Label boundary",
    "",
    "- Sealed-evaluation labels displayed: `False`",
    "- Evaluation manifests are label-free: `True`",
    "",
]


STAGE2_REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "random_seed": RANDOM_SEED,
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "canonical_jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "test_root": str(
        TEST_ROOT
    ),
}


with open(
    STAGE2_ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 25. Delete all temporary originals
# ============================================================

print(
    "\nDeleting temporary APTOS archive, extracted "
    "originals and local canonical staging files..."
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


# ============================================================
# 26. Final display
# ============================================================

print(
    "\n================ CANONICAL IMAGE SUMMARY "
    "================"
)

canonical_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "images": int(
                len(aptos_integrity)
            ),
            "unique_original_sha256": int(
                aptos_integrity[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                aptos_integrity[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    aptos_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
        {
            "dataset": "IDRiD",
            "images": int(
                len(idrid_integrity)
            ),
            "unique_original_sha256": int(
                idrid_integrity[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                idrid_integrity[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    idrid_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
    ]
)


display(
    canonical_summary
)


print(
    "\n================ PARTITION SUMMARY "
    "================"
)

display(
    partition_summary
)


print(
    "\n================ LEAKAGE CHECK "
    "================"
)

print(
    "APTOS cross-split exact duplicates:",
    aptos_cross_split_duplicates,
)

print(
    "IDRiD cross-split exact duplicates:",
    idrid_cross_split_duplicates,
)

print(
    "Cross-dataset exact overlap:",
    len(cross_dataset_overlap),
)

print(
    "Evaluation manifests contain labels:",
    not evaluation_manifest_has_no_labels,
)


print(
    "\n================ LABEL QUARANTINE CHECK "
    "================"
)

print(
    "APTOS sealed-label SHA-256:",
    label_commitments[
        "aptos_sealed_evaluation"
    ]["sha256"],
)

print(
    "IDRiD sealed-label SHA-256:",
    label_commitments[
        "idrid_sealed_evaluation"
    ]["sha256"],
)

print(
    "Sealed-evaluation label values displayed:",
    False,
)


print(
    "\n================ STAGE 2R DECISION "
    "================"
)

print("Decision:")
print(stage2_decision)

print("\nInterpretation:")
print(stage2_interpretation)

print("\nAuthorised next step:")
print(
    decision_payload[
        "authorised_next_step"
    ]
)

print(
    "\nTemporary original APTOS images retained:"
)

print(False)

print(
    "\nStage 2R storage-efficient controlled "
    "acquisition completed and sealed."
)

================ STAGE 2R BOOTSTRAP ================

Imported Stage 1D decision:
PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION

Protocol boundary:
Target originals may exist only in temporary Colab storage. Every image is deterministically converted into the frozen 768x768 canonical JPEG representation before being copied to Google Drive. Target labels may be read only by the quarantine procedure to construct development and sealed evaluation partitions. Sealed-evaluation label values must not be displayed or inserted into modelling manifests.

Frozen canonicalisation:
Convert to RGB; identify non-black retinal content using grayscale intensity > 10; crop to that bounding box when valid; pad centrally to a square with black pixels; resize with LANCZOS to 768x768; save as JPEG quality 92, subsampling 0. The same procedure must be applied to all future source and target datasets.

================ STORAGE PREFLIGHT ================
Colab local free space: 87.70 GiB
Google Dr

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [13]:
#@title 02R. Storage-efficient acquisition, canonicalisation, split sealing, and label quarantine — corrected

from pathlib import Path
from datetime import datetime, timezone

from PIL import Image
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen protocol
# ============================================================

RANDOM_SEED = 20260720
APTOS_EVALUATION_FRACTION = 0.30

APTOS_EXPECTED_IMAGES = 3662

IDRID_EXPECTED_TOTAL = 516
IDRID_EXPECTED_DEVELOPMENT = 413
IDRID_EXPECTED_EVALUATION = 103

COMPETITION_NAME = "aptos2019-blindness-detection"

ENDPOINT_ID = "MODERATE_OR_WORSE_DR_GRADE_GE_2"

CANONICAL_SIZE = 768
CANONICAL_JPEG_QUALITY = 92
CANONICAL_JPEG_SUBSAMPLING = 0
BLACK_BORDER_THRESHOLD = 10

TARGET_DATA_DOWNLOADED_THIS_RUN = False
TARGET_LABEL_CONTENT_ACCESSED_BY_QUARANTINE = False


PROTOCOL_BOUNDARY = (
    "Target originals may exist only in temporary Colab storage. "
    "Every image is deterministically converted into the frozen "
    "768x768 canonical JPEG representation before being copied "
    "to Google Drive. Target labels may be read only by the "
    "quarantine procedure to construct development and sealed "
    "evaluation partitions. Sealed-evaluation label values must "
    "not be displayed or inserted into modelling manifests."
)


CANONICALISATION_DESCRIPTION = (
    "Convert to RGB; identify non-black retinal content using "
    f"grayscale intensity > {BLACK_BORDER_THRESHOLD}; crop to "
    "the detected bounding box when valid; pad centrally to a "
    "square with black pixels; resize with LANCZOS to "
    f"{CANONICAL_SIZE}x{CANONICAL_SIZE}; save as JPEG quality "
    f"{CANONICAL_JPEG_QUALITY}, subsampling "
    f"{CANONICAL_JPEG_SUBSAMPLING}. The identical procedure must "
    "later be applied to EyePACS and DeepDRiD."
)


# ============================================================
# 1. Resolve Google Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

DESIGN_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
)

STAGE1_ROOT = (
    DESIGN_ROOT
    / "Stage1_Target_Feasibility_v0.1"
)

TEST_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

PROTOCOL_ROOT = (
    TEST_ROOT
    / "00_Protocol"
)

CANONICAL_ROOT = (
    TEST_ROOT
    / "01_Canonical_Target_Images"
)

DEVELOPMENT_ROOT = (
    TEST_ROOT
    / "02_Development"
)

SEALED_EVALUATION_ROOT = (
    TEST_ROOT
    / "03_Sealed_Evaluation"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

ORIGINAL_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "00_Original_Source_Label_Files"
)

SEALED_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "01_Sealed_Evaluation_Labels"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)


for directory in [
    PROTOCOL_ROOT,
    CANONICAL_ROOT,
    DEVELOPMENT_ROOT,
    SEALED_EVALUATION_ROOT,
    ORIGINAL_LABEL_ROOT,
    SEALED_LABEL_ROOT,
    STAGE2_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


APTOS_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019"
)

IDRID_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "IDRiD"
)


# ============================================================
# 2. Import Stage 1D authorisation
# ============================================================

STAGE1D_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1D_Final_Target_Feasibility_Decision_v0.1.json"
)

IDRID_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)


assert STAGE1D_DECISION_PATH.is_file(), (
    STAGE1D_DECISION_PATH
)

assert IDRID_INVENTORY_PATH.is_file(), (
    IDRID_INVENTORY_PATH
)


with open(
    STAGE1D_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage1d_decision = json.load(file)


assert (
    stage1d_decision["decision"]
    ==
    "PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION"
)

assert (
    stage1d_decision["authorised_next_step"]
    ==
    "CONTROLLED_TARGET_ACQUISITION_WITH_LABEL_QUARANTINE"
)


idrid_stage1_inventory = pd.read_csv(
    IDRID_INVENTORY_PATH
)


print(
    "================ STAGE 2R CORRECTED BOOTSTRAP "
    "================"
)

print("\nImported Stage 1D decision:")
print(stage1d_decision["decision"])

print("\nProtocol boundary:")
print(PROTOCOL_BOUNDARY)

print("\nFrozen canonicalisation:")
print(CANONICALISATION_DESCRIPTION)


# ============================================================
# 3. General helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


def directory_size_bytes(path):
    path = Path(path)

    return int(
        sum(
            item.stat().st_size
            for item in path.rglob("*")
            if item.is_file()
        )
    )


def bytes_to_gib(value):
    return float(
        value
        /
        (1024 ** 3)
    )


def normalise_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def resolve_column(
    dataframe,
    candidates,
    description,
):
    lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        candidate_key = normalise_column_name(
            candidate
        )

        if candidate_key in lookup:
            return lookup[candidate_key]

    raise KeyError(
        f"Could not resolve {description}.\n"
        f"Candidates: {candidates}\n"
        f"Available columns: {list(dataframe.columns)}"
    )


def normalise_image_id(value):
    return Path(
        str(value).strip()
    ).stem


def make_read_only(path):
    path = Path(path)

    try:
        os.chmod(
            path,
            0o400,
        )

        return True

    except Exception:
        return False


def verified_copy(
    source,
    destination,
):
    source = Path(source)
    destination = Path(destination)

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if destination.exists():
        try:
            os.chmod(
                destination,
                0o600,
            )
        except Exception:
            pass

    source_hash = sha256_file(
        source
    )

    shutil.copy2(
        source,
        destination,
    )

    destination_hash = sha256_file(
        destination
    )

    assert source_hash == destination_hash, (
        "Hash mismatch after copying:\n"
        f"{source}\n{destination}"
    )

    return {
        "source_path": str(source),
        "destination_path": str(destination),
        "sha256": source_hash,
        "hash_verified": True,
        "source_deleted": False,
    }


def locate_kaggle_credential():
    credential_candidates = [
        (
            Path.home()
            / ".kaggle"
            / "kaggle.json"
        ),
        (
            Path.home()
            / ".config"
            / "kaggle"
            / "kaggle.json"
        ),
    ]

    for credential_path in credential_candidates:
        if credential_path.is_file():
            return credential_path

    from google.colab import files

    print(
        "\nThe current runtime no longer contains "
        "the Kaggle credential."
    )

    print(
        "Select the same kaggle.json file once."
    )

    uploaded = files.upload()

    uploaded_jsons = [
        filename
        for filename in uploaded
        if Path(filename).suffix.lower()
        ==
        ".json"
    ]

    if len(uploaded_jsons) != 1:
        raise RuntimeError(
            "Exactly one kaggle.json file is required."
        )

    uploaded_name = uploaded_jsons[0]

    credential_payload = json.loads(
        uploaded[
            uploaded_name
        ].decode("utf-8")
    )

    if not {
        "username",
        "key",
    }.issubset(
        credential_payload.keys()
    ):
        raise RuntimeError(
            "The selected JSON file is not a "
            "legacy Kaggle API credential."
        )

    credential_path = (
        Path.home()
        / ".kaggle"
        / "kaggle.json"
    )

    credential_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    credential_path.write_bytes(
        uploaded[
            uploaded_name
        ]
    )

    os.chmod(
        credential_path,
        0o600,
    )

    uploaded_runtime_path = Path(
        uploaded_name
    )

    if uploaded_runtime_path.exists():
        uploaded_runtime_path.unlink()

    return credential_path


def crop_and_canonicalise(
    source_path,
    destination_path,
):
    source_path = Path(source_path)
    destination_path = Path(
        destination_path
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    original_sha256 = sha256_file(
        source_path
    )

    with Image.open(source_path) as image:
        image.load()

        original_width = int(
            image.width
        )

        original_height = int(
            image.height
        )

        rgb = image.convert("RGB")
        grayscale = rgb.convert("L")

        binary_mask = grayscale.point(
            lambda pixel: (
                255
                if pixel > BLACK_BORDER_THRESHOLD
                else 0
            )
        )

        crop_box = binary_mask.getbbox()

        if crop_box is None:
            crop_box = (
                0,
                0,
                original_width,
                original_height,
            )

        cropped = rgb.crop(
            crop_box
        )

        side_length = max(
            cropped.width,
            cropped.height,
        )

        square = Image.new(
            "RGB",
            (
                side_length,
                side_length,
            ),
            (
                0,
                0,
                0,
            ),
        )

        paste_x = (
            side_length
            -
            cropped.width
        ) // 2

        paste_y = (
            side_length
            -
            cropped.height
        ) // 2

        square.paste(
            cropped,
            (
                paste_x,
                paste_y,
            ),
        )

        canonical = square.resize(
            (
                CANONICAL_SIZE,
                CANONICAL_SIZE,
            ),
            resample=Image.Resampling.LANCZOS,
        )

        canonical.save(
            destination_path,
            format="JPEG",
            quality=CANONICAL_JPEG_QUALITY,
            subsampling=CANONICAL_JPEG_SUBSAMPLING,
            optimize=True,
            progressive=False,
        )

    canonical_sha256 = sha256_file(
        destination_path
    )

    with Image.open(
        destination_path
    ) as verification_image:
        verification_image.verify()

    return {
        "original_sha256": (
            original_sha256
        ),
        "canonical_sha256": (
            canonical_sha256
        ),
        "original_width": (
            original_width
        ),
        "original_height": (
            original_height
        ),
        "crop_left": int(
            crop_box[0]
        ),
        "crop_top": int(
            crop_box[1]
        ),
        "crop_right": int(
            crop_box[2]
        ),
        "crop_bottom": int(
            crop_box[3]
        ),
        "canonical_width": (
            CANONICAL_SIZE
        ),
        "canonical_height": (
            CANONICAL_SIZE
        ),
        "canonical_size_bytes": int(
            destination_path.stat().st_size
        ),
    }


def json_default(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    raise TypeError(
        f"Unsupported JSON value: "
        f"{type(value)}"
    )


# ============================================================
# 4. Storage preflight
# ============================================================

local_disk = shutil.disk_usage(
    "/content"
)

drive_disk = shutil.disk_usage(
    DRIVE_ROOT
)


print(
    "\n================ STORAGE PREFLIGHT "
    "================"
)

print(
    "Colab local free space:",
    f"{bytes_to_gib(local_disk.free):.2f} GiB",
)

print(
    "Google Drive reported free space:",
    f"{bytes_to_gib(drive_disk.free):.2f} GiB",
)

print(
    "Minimum Drive free space required by "
    "the canonical-image protocol: 3.00 GiB"
)


assert (
    local_disk.free
    >
    25
    *
    1024 ** 3
), (
    "At least 25 GiB of local Colab space is "
    "required for the temporary archive, extraction "
    "and canonicalisation."
)


assert (
    drive_disk.free
    >
    3
    *
    1024 ** 3
), (
    "At least 3 GiB of Google Drive space is "
    "required for the canonical target images."
)


# ============================================================
# 5. Temporary local workspace
#    Corrected Path construction
# ============================================================

TEMP_ROOT = (
    Path("/content")
    / "cdo_stage2r_storage_efficient"
)

DOWNLOAD_ROOT = (
    TEMP_ROOT
    / "download"
)

EXTRACT_ROOT = (
    TEMP_ROOT
    / "extract"
)

LOCAL_CANONICAL_ROOT = (
    TEMP_ROOT
    / "canonical"
)

LOCAL_APTOS_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "APTOS_2019"
)

LOCAL_IDRID_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "IDRiD"
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


for directory in [
    DOWNLOAD_ROOT,
    EXTRACT_ROOT,
    LOCAL_APTOS_CANONICAL,
    LOCAL_IDRID_CANONICAL,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


# ============================================================
# 6. Download APTOS to temporary local storage
# ============================================================

locate_kaggle_credential()


print(
    "\n================ APTOS TEMPORARY DOWNLOAD "
    "================"
)

print(
    "Downloading the official APTOS competition "
    "archive to temporary Colab storage..."
)


download_command = [
    "kaggle",
    "competitions",
    "download",
    "-c",
    COMPETITION_NAME,
    "-p",
    str(DOWNLOAD_ROOT),
    "--force",
]


download_result = subprocess.run(
    download_command,
    text=True,
    timeout=14400,
)


if download_result.returncode != 0:
    raise RuntimeError(
        "APTOS competition download failed. "
        "No target labels were processed."
    )


TARGET_DATA_DOWNLOADED_THIS_RUN = True


zip_candidates = sorted(
    DOWNLOAD_ROOT.glob("*.zip"),
    key=lambda path: (
        path.stat().st_mtime
    ),
    reverse=True,
)


assert zip_candidates, (
    "Kaggle completed without producing "
    "a ZIP archive."
)


aptos_zip_path = zip_candidates[0]


print(
    "\nDownloaded archive:"
)

print(aptos_zip_path)

print(
    "Archive size:",
    f"{bytes_to_gib(aptos_zip_path.stat().st_size):.2f} GiB",
)


# ============================================================
# 7. Extract only labelled APTOS training material
# ============================================================

print(
    "\nExtracting only train.csv and train_images/..."
)


with zipfile.ZipFile(
    aptos_zip_path,
    "r",
) as archive:
    archive_names = archive.namelist()

    train_image_members = [
        member
        for member in archive_names
        if (
            member.startswith(
                "train_images/"
            )
            and
            Path(member).suffix.lower()
            in {
                ".png",
                ".jpg",
                ".jpeg",
            }
        )
    ]

    assert (
        len(train_image_members)
        ==
        APTOS_EXPECTED_IMAGES
    ), (
        f"Expected {APTOS_EXPECTED_IMAGES} "
        f"APTOS training images, found "
        f"{len(train_image_members)}."
    )

    assert "train.csv" in archive_names

    archive.extract(
        "train.csv",
        path=EXTRACT_ROOT,
    )

    for member in tqdm(
        train_image_members,
        desc="Extracting APTOS training images",
    ):
        member_path = Path(member)

        if (
            member_path.is_absolute()
            or
            ".." in member_path.parts
        ):
            raise RuntimeError(
                f"Unsafe ZIP member: {member}"
            )

        archive.extract(
            member,
            path=EXTRACT_ROOT,
        )


APTOS_EXTRACTED_IMAGE_ROOT = (
    EXTRACT_ROOT
    / "train_images"
)

APTOS_EXTRACTED_LABEL_PATH = (
    EXTRACT_ROOT
    / "train.csv"
)


assert APTOS_EXTRACTED_IMAGE_ROOT.is_dir()
assert APTOS_EXTRACTED_LABEL_PATH.is_file()


# ============================================================
# 8. Place original label files in quarantine
# ============================================================

APTOS_ORIGINAL_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "APTOS_2019_Original_train.csv"
)


aptos_label_copy_record = verified_copy(
    APTOS_EXTRACTED_LABEL_PATH,
    APTOS_ORIGINAL_LABEL_PATH,
)


idrid_inventory = (
    idrid_stage1_inventory[
        idrid_stage1_inventory["dataset"]
        ==
        "IDRiD"
    ]
    .copy()
    .reset_index(drop=True)
)


idrid_inventory[
    "filename_lower"
] = (
    idrid_inventory[
        "filename"
    ]
    .astype(str)
    .str.lower()
)


idrid_image_rows = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            ".jpg",
            ".jpeg",
        ]
    )
].copy()


assert (
    len(idrid_image_rows)
    ==
    IDRID_EXPECTED_TOTAL
)


idrid_csv_rows = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    ==
    ".csv"
].copy()


idrid_training_rows = idrid_csv_rows[
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "training",
        regex=False,
    )
    &
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


idrid_testing_rows = idrid_csv_rows[
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "testing",
        regex=False,
    )
    &
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


assert len(idrid_training_rows) == 1
assert len(idrid_testing_rows) == 1


IDRID_ORIGINAL_TRAIN_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Training_Labels.csv"
)

IDRID_ORIGINAL_TEST_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Testing_Labels.csv"
)


idrid_train_source_path = Path(
    idrid_training_rows.iloc[0][
        "path"
    ]
)

idrid_test_source_path = Path(
    idrid_testing_rows.iloc[0][
        "path"
    ]
)


if idrid_train_source_path.is_file():
    idrid_train_copy_record = verified_copy(
        idrid_train_source_path,
        IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    )

elif IDRID_ORIGINAL_TRAIN_LABEL_PATH.is_file():
    idrid_train_copy_record = {
        "source_path": str(
            idrid_train_source_path
        ),
        "destination_path": str(
            IDRID_ORIGINAL_TRAIN_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_ORIGINAL_TRAIN_LABEL_PATH
        ),
        "hash_verified": True,
        "source_deleted": False,
    }

else:
    raise FileNotFoundError(
        "IDRiD training labels are missing."
    )


if idrid_test_source_path.is_file():
    idrid_test_copy_record = verified_copy(
        idrid_test_source_path,
        IDRID_ORIGINAL_TEST_LABEL_PATH,
    )

elif IDRID_ORIGINAL_TEST_LABEL_PATH.is_file():
    idrid_test_copy_record = {
        "source_path": str(
            idrid_test_source_path
        ),
        "destination_path": str(
            IDRID_ORIGINAL_TEST_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_ORIGINAL_TEST_LABEL_PATH
        ),
        "hash_verified": True,
        "source_deleted": False,
    }

else:
    raise FileNotFoundError(
        "IDRiD testing labels are missing."
    )


idrid_label_copy_records = [
    idrid_train_copy_record,
    idrid_test_copy_record,
]


# ============================================================
# 9. Quarantine-only label parsing
# ============================================================

TARGET_LABEL_CONTENT_ACCESSED_BY_QUARANTINE = True


aptos_labels_raw = pd.read_csv(
    APTOS_ORIGINAL_LABEL_PATH
)

idrid_train_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TRAIN_LABEL_PATH
)

idrid_test_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TEST_LABEL_PATH
)


aptos_id_column = resolve_column(
    aptos_labels_raw,
    [
        "id_code",
        "image_id",
        "image",
    ],
    "APTOS image ID",
)

aptos_grade_column = resolve_column(
    aptos_labels_raw,
    [
        "diagnosis",
        "grade",
        "dr_grade",
    ],
    "APTOS DR grade",
)


aptos_labels = pd.DataFrame(
    {
        "dataset": "APTOS_2019",
        "image_id": (
            aptos_labels_raw[
                aptos_id_column
            ]
            .map(
                normalise_image_id
            )
        ),
        "original_dr_grade": (
            pd.to_numeric(
                aptos_labels_raw[
                    aptos_grade_column
                ],
                errors="raise",
            )
            .astype(int)
        ),
    }
)


assert len(aptos_labels) == APTOS_EXPECTED_IMAGES
assert aptos_labels["image_id"].is_unique


def standardise_idrid_labels(
    dataframe,
    official_partition,
):
    image_column = resolve_column(
        dataframe,
        [
            "Image name",
            "image_name",
            "image",
        ],
        "IDRiD image name",
    )

    grade_column = resolve_column(
        dataframe,
        [
            "Retinopathy grade",
            "retinopathy_grade",
            "DR grade",
            "dr_grade",
            "grade",
        ],
        "IDRiD retinopathy grade",
    )

    return pd.DataFrame(
        {
            "dataset": "IDRiD",
            "image_id": (
                dataframe[
                    image_column
                ]
                .map(
                    normalise_image_id
                )
            ),
            "original_dr_grade": (
                pd.to_numeric(
                    dataframe[
                        grade_column
                    ],
                    errors="raise",
                )
                .astype(int)
            ),
            "official_partition": (
                official_partition
            ),
        }
    )


idrid_train_labels = standardise_idrid_labels(
    idrid_train_labels_raw,
    "development",
)

idrid_test_labels = standardise_idrid_labels(
    idrid_test_labels_raw,
    "sealed_evaluation",
)


assert (
    len(idrid_train_labels)
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(idrid_test_labels)
    ==
    IDRID_EXPECTED_EVALUATION
)


idrid_labels = pd.concat(
    [
        idrid_train_labels,
        idrid_test_labels,
    ],
    ignore_index=True,
)


assert len(idrid_labels) == IDRID_EXPECTED_TOTAL
assert idrid_labels["image_id"].is_unique


# ============================================================
# 10. Canonicalise APTOS locally
# ============================================================

aptos_source_paths = sorted(
    [
        path
        for path in APTOS_EXTRACTED_IMAGE_ROOT.iterdir()
        if (
            path.is_file()
            and
            path.suffix.lower()
            in {
                ".png",
                ".jpg",
                ".jpeg",
            }
        )
    ]
)


assert (
    len(aptos_source_paths)
    ==
    APTOS_EXPECTED_IMAGES
)


print(
    "\n================ APTOS CANONICALISATION "
    "================"
)


aptos_processing_records = []


for source_path in tqdm(
    aptos_source_paths,
    desc="Canonicalising APTOS",
):
    image_id = source_path.stem

    destination_path = (
        LOCAL_APTOS_CANONICAL
        / f"{image_id}.jpg"
    )

    processing_record = crop_and_canonicalise(
        source_path,
        destination_path,
    )

    aptos_processing_records.append(
        {
            "dataset": "APTOS_2019",
            "image_id": image_id,
            "original_filename": (
                source_path.name
            ),
            "canonical_filename": (
                destination_path.name
            ),
            **processing_record,
        }
    )


aptos_integrity = pd.DataFrame(
    aptos_processing_records
)


assert len(aptos_integrity) == APTOS_EXPECTED_IMAGES
assert aptos_integrity["image_id"].is_unique


# ============================================================
# 11. Canonicalise IDRiD locally
# ============================================================

print(
    "\n================ IDRiD CANONICALISATION "
    "================"
)


idrid_processing_records = []


for _, inventory_row in tqdm(
    idrid_image_rows.iterrows(),
    total=len(idrid_image_rows),
    desc="Canonicalising IDRiD",
):
    source_path = Path(
        inventory_row["path"]
    )

    assert source_path.is_file(), (
        source_path
    )

    image_id = source_path.stem

    destination_path = (
        LOCAL_IDRID_CANONICAL
        / f"{image_id}.jpg"
    )

    processing_record = crop_and_canonicalise(
        source_path,
        destination_path,
    )

    idrid_processing_records.append(
        {
            "dataset": "IDRiD",
            "image_id": image_id,
            "original_filename": (
                source_path.name
            ),
            "canonical_filename": (
                destination_path.name
            ),
            **processing_record,
        }
    )


idrid_integrity = pd.DataFrame(
    idrid_processing_records
)


assert len(idrid_integrity) == IDRID_EXPECTED_TOTAL
assert idrid_integrity["image_id"].is_unique


# ============================================================
# 12. Actual canonical storage audit
# ============================================================

aptos_canonical_bytes = directory_size_bytes(
    LOCAL_APTOS_CANONICAL
)

idrid_canonical_bytes = directory_size_bytes(
    LOCAL_IDRID_CANONICAL
)

canonical_total_bytes = (
    aptos_canonical_bytes
    +
    idrid_canonical_bytes
)

current_drive_free = shutil.disk_usage(
    DRIVE_ROOT
).free


print(
    "\n================ CANONICAL STORAGE AUDIT "
    "================"
)

print(
    "APTOS canonical size:",
    f"{bytes_to_gib(aptos_canonical_bytes):.2f} GiB",
)

print(
    "IDRiD canonical size:",
    f"{bytes_to_gib(idrid_canonical_bytes):.2f} GiB",
)

print(
    "Total canonical size:",
    f"{bytes_to_gib(canonical_total_bytes):.2f} GiB",
)

print(
    "Current Drive free space:",
    f"{bytes_to_gib(current_drive_free):.2f} GiB",
)


assert (
    current_drive_free
    >
    canonical_total_bytes
    +
    512
    *
    1024 ** 2
), (
    "The generated canonical images still exceed "
    "the available Drive space. No canonical images "
    "have been copied to Drive."
)


# ============================================================
# 13. Copy complete canonical sets to Drive staging
# ============================================================

APTOS_STAGING_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019__STAGING"
)

IDRID_STAGING_ROOT = (
    CANONICAL_ROOT
    / "IDRiD__STAGING"
)


for staging_root in [
    APTOS_STAGING_ROOT,
    IDRID_STAGING_ROOT,
]:
    if staging_root.exists():
        shutil.rmtree(
            staging_root
        )


print(
    "\nCopying canonical image sets to "
    "Google Drive staging directories..."
)


shutil.copytree(
    LOCAL_APTOS_CANONICAL,
    APTOS_STAGING_ROOT,
)

shutil.copytree(
    LOCAL_IDRID_CANONICAL,
    IDRID_STAGING_ROOT,
)


assert (
    len(
        list(
            APTOS_STAGING_ROOT.glob("*.jpg")
        )
    )
    ==
    APTOS_EXPECTED_IMAGES
)

assert (
    len(
        list(
            IDRID_STAGING_ROOT.glob("*.jpg")
        )
    )
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 14. Verify Drive copies by SHA-256
# ============================================================

print(
    "\n================ DRIVE COPY VERIFICATION "
    "================"
)


for _, row in tqdm(
    aptos_integrity.iterrows(),
    total=len(aptos_integrity),
    desc="Verifying APTOS Drive hashes",
):
    copied_path = (
        APTOS_STAGING_ROOT
        / row["canonical_filename"]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(copied_path)
        ==
        row["canonical_sha256"]
    )


for _, row in tqdm(
    idrid_integrity.iterrows(),
    total=len(idrid_integrity),
    desc="Verifying IDRiD Drive hashes",
):
    copied_path = (
        IDRID_STAGING_ROOT
        / row["canonical_filename"]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(copied_path)
        ==
        row["canonical_sha256"]
    )


if APTOS_CANONICAL_ROOT.exists():
    shutil.rmtree(
        APTOS_CANONICAL_ROOT
    )

if IDRID_CANONICAL_ROOT.exists():
    shutil.rmtree(
        IDRID_CANONICAL_ROOT
    )


APTOS_STAGING_ROOT.rename(
    APTOS_CANONICAL_ROOT
)

IDRID_STAGING_ROOT.rename(
    IDRID_CANONICAL_ROOT
)


# ============================================================
# 15. Join labels and image-integrity records
# ============================================================

aptos_integrity[
    "canonical_image_path"
] = (
    aptos_integrity["image_id"]
    .map(
        lambda image_id: str(
            APTOS_CANONICAL_ROOT
            / f"{image_id}.jpg"
        )
    )
)


idrid_integrity[
    "canonical_image_path"
] = (
    idrid_integrity["image_id"]
    .map(
        lambda image_id: str(
            IDRID_CANONICAL_ROOT
            / f"{image_id}.jpg"
        )
    )
)


aptos_verified = aptos_labels.merge(
    aptos_integrity,
    on=[
        "dataset",
        "image_id",
    ],
    how="inner",
    validate="one_to_one",
)


idrid_verified = idrid_labels.merge(
    idrid_integrity,
    on=[
        "dataset",
        "image_id",
    ],
    how="inner",
    validate="one_to_one",
)


assert len(aptos_verified) == APTOS_EXPECTED_IMAGES
assert len(idrid_verified) == IDRID_EXPECTED_TOTAL


# ============================================================
# 16. Deterministic APTOS development/evaluation split
# ============================================================

splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=APTOS_EVALUATION_FRACTION,
    random_state=RANDOM_SEED,
)


development_indices, evaluation_indices = next(
    splitter.split(
        np.zeros(
            len(aptos_verified)
        ),
        aptos_verified[
            "original_dr_grade"
        ].to_numpy(),
    )
)


aptos_verified[
    "prospective_partition"
] = "development"

aptos_verified.loc[
    evaluation_indices,
    "prospective_partition",
] = "sealed_evaluation"


# Exact duplicate groups may not cross partitions.
duplicate_hash_columns = [
    "original_sha256",
    "canonical_sha256",
]


crossing_duplicate_values = {
    "original_sha256": [],
    "canonical_sha256": [],
}


for hash_column in duplicate_hash_columns:
    crossing_hashes = (
        aptos_verified
        .groupby(hash_column)[
            "prospective_partition"
        ]
        .nunique()
    )

    crossing_hashes = (
        crossing_hashes[
            crossing_hashes > 1
        ]
        .index
        .tolist()
    )

    crossing_duplicate_values[
        hash_column
    ] = crossing_hashes

    if crossing_hashes:
        aptos_verified.loc[
            aptos_verified[
                hash_column
            ].isin(
                crossing_hashes
            ),
            "prospective_partition",
        ] = "sealed_evaluation"


for hash_column in duplicate_hash_columns:
    assert not (
        aptos_verified
        .groupby(hash_column)[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).any()


# ============================================================
# 17. Preserve official IDRiD split
# ============================================================

idrid_verified[
    "prospective_partition"
] = (
    idrid_verified[
        "official_partition"
    ]
)


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "development"
).sum() == IDRID_EXPECTED_DEVELOPMENT


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "sealed_evaluation"
).sum() == IDRID_EXPECTED_EVALUATION


for hash_column in duplicate_hash_columns:
    assert not (
        idrid_verified
        .groupby(hash_column)[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).any()


original_cross_dataset_overlap = set(
    aptos_verified[
        "original_sha256"
    ]
).intersection(
    set(
        idrid_verified[
            "original_sha256"
        ]
    )
)


canonical_cross_dataset_overlap = set(
    aptos_verified[
        "canonical_sha256"
    ]
).intersection(
    set(
        idrid_verified[
            "canonical_sha256"
        ]
    )
)


assert not original_cross_dataset_overlap
assert not canonical_cross_dataset_overlap


# ============================================================
# 18. Harmonised endpoint
# ============================================================

for dataframe in [
    aptos_verified,
    idrid_verified,
]:
    dataframe[
        "moderate_or_worse_dr"
    ] = (
        dataframe[
            "original_dr_grade"
        ]
        >=
        2
    ).astype(int)


# ============================================================
# 19. Construct partition tables
# ============================================================

aptos_development = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values("image_id")
)


aptos_evaluation = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values("image_id")
)


idrid_development = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values("image_id")
)


idrid_evaluation = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values("image_id")
)


assert len(aptos_development) > 0
assert len(aptos_evaluation) > 0

assert (
    len(idrid_development)
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(idrid_evaluation)
    ==
    IDRID_EXPECTED_EVALUATION
)


# ============================================================
# 20. Save development and sealed label artifacts
# ============================================================

APTOS_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "APTOS_2019_Development_Labels_v0.1.csv"
)

IDRID_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "IDRiD_Development_Labels_v0.1.csv"
)

APTOS_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv"
)

IDRID_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv"
)

APTOS_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "APTOS_2019_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

IDRID_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "IDRiD_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)


development_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_dr_grade",
    "moderate_or_worse_dr",
    "original_sha256",
    "canonical_sha256",
]


evaluation_manifest_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_sha256",
    "canonical_sha256",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


sealed_label_columns = [
    "dataset",
    "image_id",
    "original_dr_grade",
    "moderate_or_worse_dr",
]


aptos_development[
    development_columns
].to_csv(
    APTOS_DEVELOPMENT_LABEL_PATH,
    index=False,
)


idrid_development[
    development_columns
].to_csv(
    IDRID_DEVELOPMENT_LABEL_PATH,
    index=False,
)


aptos_evaluation[
    evaluation_manifest_columns
].to_csv(
    APTOS_EVALUATION_MANIFEST_PATH,
    index=False,
)


idrid_evaluation[
    evaluation_manifest_columns
].to_csv(
    IDRID_EVALUATION_MANIFEST_PATH,
    index=False,
)


aptos_evaluation[
    sealed_label_columns
].to_csv(
    APTOS_SEALED_LABEL_PATH,
    index=False,
)


idrid_evaluation[
    sealed_label_columns
].to_csv(
    IDRID_SEALED_LABEL_PATH,
    index=False,
)


for protected_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_read_only(
        protected_path
    )


# ============================================================
# 21. Create label-free master target manifest
# ============================================================

label_free_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "prospective_partition",
    "original_sha256",
    "canonical_sha256",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


label_free_master_manifest = pd.concat(
    [
        aptos_verified[
            label_free_columns
        ],
        idrid_verified[
            label_free_columns
        ],
    ],
    ignore_index=True,
)


assert (
    "original_dr_grade"
    not in label_free_master_manifest.columns
)

assert (
    "moderate_or_worse_dr"
    not in label_free_master_manifest.columns
)


# ============================================================
# 22. Protocol and audit output paths
# ============================================================

CANONICAL_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Canonical_Fundus_Preprocessing_Protocol_v0.1.json"
)

QUARANTINE_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Target_Label_Quarantine_Protocol_v0.1.json"
)

MASTER_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_LabelFree_Master_Target_Manifest_v0.1.csv"
)

INTEGRITY_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

PARTITION_SUMMARY_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Partition_Summary_v0.1.csv"
)

LABEL_COMMITMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Sealed_Label_Commitments_v0.1.json"
)

IDRID_LABEL_COPY_PATH = (
    STAGE2_ROOT
    / "Stage2_IDRiD_Label_Copy_Manifest_v0.1.csv"
)

STAGE2_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)

STAGE2_REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Report_v0.1.md"
)

STAGE2_ENVIRONMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Environment_v0.1.json"
)


# ============================================================
# 23. Save protocol and integrity records
# ============================================================

canonical_protocol = {
    "protocol_version": "v0.1",
    "canonical_size": CANONICAL_SIZE,
    "black_border_threshold": (
        BLACK_BORDER_THRESHOLD
    ),
    "jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "jpeg_subsampling": (
        CANONICAL_JPEG_SUBSAMPLING
    ),
    "resize_filter": "LANCZOS",
    "description": (
        CANONICALISATION_DESCRIPTION
    ),
    "frozen_before_target_performance": True,
    "must_be_applied_to_sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
}


with open(
    CANONICAL_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        canonical_protocol,
        file,
        indent=2,
    )


quarantine_protocol = {
    "protocol_version": "v0.1",
    "endpoint_id": ENDPOINT_ID,
    "random_seed": RANDOM_SEED,
    "aptos_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "development_labels_authorised_from_stage": 3,
    "sealed_evaluation_labels_authorised_stage": 6,
    "sealed_evaluation_manifests_label_free": True,
    "idrid_source_label_files_deleted": False,
    "prohibitions": [
        (
            "Do not open, display, summarise or model "
            "sealed-evaluation labels before Stage 6."
        ),
        (
            "Do not change target partitions after "
            "recoverability or transfer results are observed."
        ),
        (
            "Do not change the canonicalisation protocol "
            "after target performance is observed."
        ),
        (
            "Do not insert evaluation labels into any "
            "label-free modelling manifest."
        ),
    ],
}


with open(
    QUARANTINE_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quarantine_protocol,
        file,
        indent=2,
    )


label_free_master_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False,
)


integrity_manifest = pd.concat(
    [
        aptos_integrity,
        idrid_integrity,
    ],
    ignore_index=True,
)


integrity_manifest.to_csv(
    INTEGRITY_MANIFEST_PATH,
    index=False,
)


partition_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "total_images": int(
                len(aptos_verified)
            ),
            "development_images": int(
                len(aptos_development)
            ),
            "sealed_evaluation_images": int(
                len(aptos_evaluation)
            ),
            "partition_source": (
                "deterministic grade-stratified "
                "70/30 split"
            ),
            "original_duplicate_groups_reassigned": int(
                len(
                    crossing_duplicate_values[
                        "original_sha256"
                    ]
                )
            ),
            "canonical_duplicate_groups_reassigned": int(
                len(
                    crossing_duplicate_values[
                        "canonical_sha256"
                    ]
                )
            ),
            "sealed_evaluation_labels_displayed": False,
        },
        {
            "dataset": "IDRiD",
            "total_images": int(
                len(idrid_verified)
            ),
            "development_images": int(
                len(idrid_development)
            ),
            "sealed_evaluation_images": int(
                len(idrid_evaluation)
            ),
            "partition_source": (
                "official IDRiD training/testing split"
            ),
            "original_duplicate_groups_reassigned": 0,
            "canonical_duplicate_groups_reassigned": 0,
            "sealed_evaluation_labels_displayed": False,
        },
    ]
)


partition_summary.to_csv(
    PARTITION_SUMMARY_PATH,
    index=False,
)


label_commitments = {
    "endpoint_id": ENDPOINT_ID,
    "aptos_sealed_evaluation": {
        "rows": int(
            len(aptos_evaluation)
        ),
        "path": str(
            APTOS_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            APTOS_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "idrid_sealed_evaluation": {
        "rows": int(
            len(idrid_evaluation)
        ),
        "path": str(
            IDRID_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "original_label_files": {
        "aptos_train_sha256": (
            sha256_file(
                APTOS_ORIGINAL_LABEL_PATH
            )
        ),
        "idrid_train_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TRAIN_LABEL_PATH
            )
        ),
        "idrid_test_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TEST_LABEL_PATH
            )
        ),
    },
}


with open(
    LABEL_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_commitments,
        file,
        indent=2,
    )


pd.DataFrame(
    idrid_label_copy_records
).to_csv(
    IDRID_LABEL_COPY_PATH,
    index=False,
)


# ============================================================
# 24. Leakage and label-boundary checks
# ============================================================

aptos_original_cross_split_duplicates = int(
    (
        aptos_verified
        .groupby("original_sha256")[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


aptos_canonical_cross_split_duplicates = int(
    (
        aptos_verified
        .groupby("canonical_sha256")[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_original_cross_split_duplicates = int(
    (
        idrid_verified
        .groupby("original_sha256")[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_canonical_cross_split_duplicates = int(
    (
        idrid_verified
        .groupby("canonical_sha256")[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


evaluation_manifest_has_no_labels = bool(
    all(
        label_column
        not in pd.read_csv(
            manifest_path,
            nrows=1,
        ).columns
        for manifest_path in [
            APTOS_EVALUATION_MANIFEST_PATH,
            IDRID_EVALUATION_MANIFEST_PATH,
        ]
        for label_column in [
            "original_dr_grade",
            "moderate_or_worse_dr",
        ]
    )
)


stage2_pass = bool(
    len(aptos_verified) == APTOS_EXPECTED_IMAGES
    and
    len(idrid_verified) == IDRID_EXPECTED_TOTAL
    and
    len(idrid_development) == IDRID_EXPECTED_DEVELOPMENT
    and
    len(idrid_evaluation) == IDRID_EXPECTED_EVALUATION
    and
    aptos_original_cross_split_duplicates == 0
    and
    aptos_canonical_cross_split_duplicates == 0
    and
    idrid_original_cross_split_duplicates == 0
    and
    idrid_canonical_cross_split_duplicates == 0
    and
    len(original_cross_dataset_overlap) == 0
    and
    len(canonical_cross_dataset_overlap) == 0
    and
    evaluation_manifest_has_no_labels
)


assert stage2_pass


# ============================================================
# 25. Final Stage 2R decision
# ============================================================

stage2_decision = (
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_"
    "AND_LABEL_QUARANTINE_ADVANCE_TO_"
    "DEVELOPMENT_RECOVERABILITY_GATE"
)


stage2_interpretation = (
    "APTOS originals were temporarily acquired in Colab and "
    "converted into the frozen 768x768 canonical fundus "
    "representation before storage on Google Drive. IDRiD was "
    "processed with the identical canonicalisation. Original "
    "and canonical SHA-256 commitments were retained and the "
    "Drive copies were hash-verified. APTOS was deterministically "
    "partitioned into development and sealed evaluation data; "
    "IDRiD retained its official split. Sealed labels are absent "
    "from all modelling manifests and committed for one-time "
    "Stage 6 evaluation."
)


decision_payload = {
    "decision": stage2_decision,
    "interpretation": (
        stage2_interpretation
    ),
    "authorised_next_step": (
        "DEVELOPMENT_ONLY_TARGET_RECOVERABILITY_GATE"
    ),
    "endpoint_id": ENDPOINT_ID,
    "canonicalisation_protocol": str(
        CANONICAL_PROTOCOL_PATH
    ),
    "canonical_size": CANONICAL_SIZE,
    "target_data_downloaded_this_run": (
        TARGET_DATA_DOWNLOADED_THIS_RUN
    ),
    "target_label_content_accessed_by_quarantine": (
        TARGET_LABEL_CONTENT_ACCESSED_BY_QUARANTINE
    ),
    "aptos": {
        "total": int(
            len(aptos_verified)
        ),
        "development": int(
            len(aptos_development)
        ),
        "sealed_evaluation": int(
            len(aptos_evaluation)
        ),
        "canonical_size_gib": bytes_to_gib(
            aptos_canonical_bytes
        ),
        "original_cross_split_duplicates": (
            aptos_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            aptos_canonical_cross_split_duplicates
        ),
    },
    "idrid": {
        "total": int(
            len(idrid_verified)
        ),
        "development": int(
            len(idrid_development)
        ),
        "sealed_evaluation": int(
            len(idrid_evaluation)
        ),
        "canonical_size_gib": bytes_to_gib(
            idrid_canonical_bytes
        ),
        "original_cross_split_duplicates": (
            idrid_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            idrid_canonical_cross_split_duplicates
        ),
    },
    "original_cross_dataset_overlap": int(
        len(original_cross_dataset_overlap)
    ),
    "canonical_cross_dataset_overlap": int(
        len(canonical_cross_dataset_overlap)
    ),
    "evaluation_manifests_label_free": (
        evaluation_manifest_has_no_labels
    ),
    "sealed_label_commitment_path": str(
        LABEL_COMMITMENT_PATH
    ),
    "important_boundary": (
        "This stage establishes acquisition, frozen "
        "canonicalisation and procedural label quarantine. "
        "It does not establish target recoverability, "
        "source-target transfer or CDO validity."
    ),
}


with open(
    STAGE2_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
        default=json_default,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 2R",
    "",
    "## Storage-efficient controlled acquisition",
    "",
    f"Decision: `{stage2_decision}`",
    "",
    stage2_interpretation,
    "",
    "## Canonical representation",
    "",
    f"- Size: {CANONICAL_SIZE} x {CANONICAL_SIZE}",
    f"- JPEG quality: {CANONICAL_JPEG_QUALITY}",
    (
        f"- APTOS storage: "
        f"{bytes_to_gib(aptos_canonical_bytes):.3f} GiB"
    ),
    (
        f"- IDRiD storage: "
        f"{bytes_to_gib(idrid_canonical_bytes):.3f} GiB"
    ),
    "",
    "## Label boundary",
    "",
    "- Sealed-evaluation labels displayed: `False`",
    "- Evaluation manifests are label-free: `True`",
    "",
]


STAGE2_REPORT_PATH.write_text(
    "\n".join(report_lines),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "random_seed": RANDOM_SEED,
    "canonical_size": CANONICAL_SIZE,
    "canonical_jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "test_root": str(TEST_ROOT),
}


with open(
    STAGE2_ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 26. Delete temporary APTOS originals and local staging
# ============================================================

print(
    "\nDeleting the temporary APTOS archive, extracted "
    "original images and local canonical staging files..."
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


# ============================================================
# 27. Final output display
# ============================================================

canonical_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "images": int(
                len(aptos_integrity)
            ),
            "unique_original_sha256": int(
                aptos_integrity[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                aptos_integrity[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    aptos_canonical_bytes
                )
            ),
            "canonical_width": CANONICAL_SIZE,
            "canonical_height": CANONICAL_SIZE,
        },
        {
            "dataset": "IDRiD",
            "images": int(
                len(idrid_integrity)
            ),
            "unique_original_sha256": int(
                idrid_integrity[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                idrid_integrity[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    idrid_canonical_bytes
                )
            ),
            "canonical_width": CANONICAL_SIZE,
            "canonical_height": CANONICAL_SIZE,
        },
    ]
)


print(
    "\n================ CANONICAL IMAGE SUMMARY "
    "================"
)

display(canonical_summary)


print(
    "\n================ PARTITION SUMMARY "
    "================"
)

display(partition_summary)


print(
    "\n================ LEAKAGE CHECK "
    "================"
)

print(
    "APTOS original-hash cross-split duplicates:",
    aptos_original_cross_split_duplicates,
)

print(
    "APTOS canonical-hash cross-split duplicates:",
    aptos_canonical_cross_split_duplicates,
)

print(
    "IDRiD original-hash cross-split duplicates:",
    idrid_original_cross_split_duplicates,
)

print(
    "IDRiD canonical-hash cross-split duplicates:",
    idrid_canonical_cross_split_duplicates,
)

print(
    "Original cross-dataset overlap:",
    len(original_cross_dataset_overlap),
)

print(
    "Canonical cross-dataset overlap:",
    len(canonical_cross_dataset_overlap),
)

print(
    "Evaluation manifests contain labels:",
    not evaluation_manifest_has_no_labels,
)


print(
    "\n================ LABEL QUARANTINE CHECK "
    "================"
)

print(
    "APTOS sealed-label SHA-256:",
    label_commitments[
        "aptos_sealed_evaluation"
    ]["sha256"],
)

print(
    "IDRiD sealed-label SHA-256:",
    label_commitments[
        "idrid_sealed_evaluation"
    ]["sha256"],
)

print(
    "Sealed-evaluation label values displayed:",
    False,
)


print(
    "\n================ STAGE 2R DECISION "
    "================"
)

print("Decision:")
print(stage2_decision)

print("\nInterpretation:")
print(stage2_interpretation)

print("\nAuthorised next step:")
print(
    decision_payload[
        "authorised_next_step"
    ]
)

print(
    "\nTemporary original APTOS images retained:"
)

print(False)

print(
    "\nOriginal IDRiD label files deleted:"
)

print(False)

print(
    "\nStage 2R corrected storage-efficient "
    "acquisition completed and sealed."
)

================ STAGE 2R CORRECTED BOOTSTRAP ================

Imported Stage 1D decision:
PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION

Protocol boundary:
Target originals may exist only in temporary Colab storage. Every image is deterministically converted into the frozen 768x768 canonical JPEG representation before being copied to Google Drive. Target labels may be read only by the quarantine procedure to construct development and sealed evaluation partitions. Sealed-evaluation label values must not be displayed or inserted into modelling manifests.

Frozen canonicalisation:
Convert to RGB; identify non-black retinal content using grayscale intensity > 10; crop to the detected bounding box when valid; pad centrally to a square with black pixels; resize with LANCZOS to 768x768; save as JPEG quality 92, subsampling 0. The identical procedure must later be applied to EyePACS and DeepDRiD.

================ STORAGE PREFLIGHT ================
Colab local free space: 87.70 G

RuntimeError: APTOS competition download failed. No target labels were processed.

In [15]:
#@title 02R-D. Robust APTOS download gate with visible diagnostics — NO LABEL READING

from pathlib import Path
from datetime import datetime, timezone
from IPython.display import Javascript, display

import json
import os
import subprocess
import sys
import time


# ============================================================
# 0. Fixed safety boundary
# ============================================================

COMPETITION = "aptos2019-blindness-detection"

TARGET_LABEL_CONTENT_ACCESSED = False
TARGET_FILES_EXTRACTED = False

SAFETY_BOUNDARY = (
    "This cell verifies Kaggle competition-download permission "
    "and downloads the official APTOS competition archive into "
    "temporary Colab storage. It does not extract the archive, "
    "open train.csv, inspect labels, or write target images to Drive."
)


# ============================================================
# 1. Resolve Drive and audit paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

STAGE2_RECORD_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_v0.1"
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)

STAGE2_RECORD_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


TEMP_ROOT = (
    Path("/content")
    / "cdo_stage2r_storage_efficient"
)

DOWNLOAD_ROOT = (
    TEMP_ROOT
    / "download"
)

PERMISSION_TEST_ROOT = (
    TEMP_ROOT
    / "permission_test"
)


DOWNLOAD_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

PERMISSION_TEST_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


AUDIT_PATH = (
    STAGE2_RECORD_ROOT
    / "Stage2R_APTOS_Download_Gate_v0.1.json"
)


print(
    "================ APTOS DOWNLOAD GATE "
    "================"
)

print("\nSafety boundary:")
print(SAFETY_BOUNDARY)


# ============================================================
# 2. Confirm Kaggle credential
# ============================================================

credential_candidates = [
    (
        Path.home()
        / ".kaggle"
        / "kaggle.json"
    ),
    (
        Path.home()
        / ".config"
        / "kaggle"
        / "kaggle.json"
    ),
]


credential_path = next(
    (
        path
        for path in credential_candidates
        if path.is_file()
    ),
    None,
)


if credential_path is None:
    from google.colab import files

    print(
        "\nThe temporary Kaggle credential is no longer "
        "available, probably because the runtime restarted."
    )

    print(
        "Select the same kaggle.json file once."
    )

    uploaded = files.upload()

    json_files = [
        filename
        for filename in uploaded
        if Path(filename).suffix.lower() == ".json"
    ]

    if len(json_files) != 1:
        raise RuntimeError(
            "Exactly one kaggle.json file is required."
        )

    uploaded_name = json_files[0]

    credential_payload = json.loads(
        uploaded[
            uploaded_name
        ].decode("utf-8")
    )

    if not {
        "username",
        "key",
    }.issubset(
        credential_payload.keys()
    ):
        raise RuntimeError(
            "The selected file is not a legacy "
            "Kaggle API credential."
        )

    credential_path = (
        Path.home()
        / ".kaggle"
        / "kaggle.json"
    )

    credential_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    credential_path.write_bytes(
        uploaded[
            uploaded_name
        ]
    )

    os.chmod(
        credential_path,
        0o600,
    )

    runtime_upload = Path(
        uploaded_name
    )

    if runtime_upload.exists():
        runtime_upload.unlink()


print("\nKaggle credential found.")
print(
    "Credential contents will not be displayed "
    "or copied to Drive."
)


# ============================================================
# 3. Install current official Kaggle CLI
# ============================================================

install_result = subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--quiet",
        "--upgrade",
        "kaggle",
    ],
    capture_output=True,
    text=True,
    timeout=240,
)


if install_result.returncode != 0:
    raise RuntimeError(
        "Kaggle CLI installation failed:\n"
        +
        install_result.stderr[-2000:]
    )


version_result = subprocess.run(
    [
        "kaggle",
        "--version",
    ],
    capture_output=True,
    text=True,
    timeout=60,
)


kaggle_version = (
    version_result.stdout
    or
    version_result.stderr
).strip()


print("\nKaggle CLI:")
print(kaggle_version)


# ============================================================
# 4. Command runner with retained diagnostics
# ============================================================

def run_command_with_visible_output(
    command,
    timeout_seconds,
):
    print(
        "\nRunning:"
    )

    safe_command_text = " ".join(
        command
    )

    print(
        safe_command_text
    )

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )

    output_lines = []
    started = time.time()

    try:
        assert process.stdout is not None

        while True:
            line = process.stdout.readline()

            if line:
                print(
                    line,
                    end="",
                    flush=True,
                )

                output_lines.append(
                    line.rstrip("\n")
                )

                if len(output_lines) > 500:
                    output_lines = (
                        output_lines[-500:]
                    )

            if process.poll() is not None:
                remaining_output = (
                    process.stdout.read()
                )

                if remaining_output:
                    print(
                        remaining_output,
                        end="",
                        flush=True,
                    )

                    output_lines.extend(
                        remaining_output.splitlines()
                    )

                    output_lines = (
                        output_lines[-500:]
                    )

                break

            if (
                time.time()
                -
                started
                >
                timeout_seconds
            ):
                process.kill()

                raise TimeoutError(
                    "Kaggle command exceeded the "
                    f"{timeout_seconds}-second timeout."
                )

    finally:
        if process.poll() is None:
            process.kill()

    return {
        "command": safe_command_text,
        "returncode": int(
            process.returncode
        ),
        "elapsed_seconds": float(
            time.time()
            -
            started
        ),
        "output_tail": output_lines[-200:],
    }


def classify_failure(
    output_lines,
):
    text = "\n".join(
        output_lines
    ).lower()

    if any(
        token in text
        for token in [
            "403",
            "forbidden",
            "accept the rules",
            "competition rules",
            "must accept",
            "join the competition",
            "permission",
        ]
    ):
        return (
            "COMPETITION_RULE_ACCEPTANCE_REQUIRED"
        )

    if any(
        token in text
        for token in [
            "401",
            "unauthorized",
            "authentication",
            "invalid credential",
            "invalid token",
        ]
    ):
        return (
            "KAGGLE_AUTHENTICATION_FAILED"
        )

    if any(
        token in text
        for token in [
            "unrecognized arguments: -c",
            "no such option: -c",
            "unknown option -c",
        ]
    ):
        return (
            "LEGACY_COMMAND_SYNTAX_REJECTED"
        )

    if any(
        token in text
        for token in [
            "no space left",
            "disk quota",
            "insufficient space",
        ]
    ):
        return (
            "LOCAL_STORAGE_FAILURE"
        )

    if any(
        token in text
        for token in [
            "connection reset",
            "connection aborted",
            "timed out",
            "temporary failure",
            "service unavailable",
            "502",
            "503",
            "504",
        ]
    ):
        return (
            "TRANSIENT_NETWORK_OR_SERVER_FAILURE"
        )

    return (
        "UNRESOLVED_KAGGLE_DOWNLOAD_FAILURE"
    )


# ============================================================
# 5. Tiny permission test using current positional syntax
# ============================================================

for existing_file in (
    PERMISSION_TEST_ROOT.glob("*")
):
    if existing_file.is_file():
        existing_file.unlink()


permission_command = [
    "kaggle",
    "competitions",
    "download",
    COMPETITION,
    "-f",
    "train.csv",
    "-p",
    str(
        PERMISSION_TEST_ROOT
    ),
    "-o",
    "-q",
]


permission_result = (
    run_command_with_visible_output(
        permission_command,
        timeout_seconds=300,
    )
)


if permission_result["returncode"] != 0:
    failure_status = classify_failure(
        permission_result[
            "output_tail"
        ]
    )

    audit_payload = {
        "status": failure_status,
        "stage": "permission_test",
        "kaggle_version": kaggle_version,
        "permission_test": (
            permission_result
        ),
        "target_label_content_accessed": False,
        "target_files_extracted": False,
        "archive_downloaded": False,
    }

    with open(
        AUDIT_PATH,
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            audit_payload,
            file,
            indent=2,
        )

    if (
        failure_status
        ==
        "COMPETITION_RULE_ACCEPTANCE_REQUIRED"
    ):
        display(
            Javascript(
                """
                window.open(
                    'https://www.kaggle.com/competitions/'
                    + 'aptos2019-blindness-detection/rules',
                    '_blank'
                );
                """
            )
        )

        raise RuntimeError(
            "Kaggle credential is valid, but the APTOS "
            "competition rules have not been accepted.\n\n"
            "The official Rules page has been opened in a "
            "new tab. Log in, click Join Competition or "
            "I Understand and Accept, then rerun this same cell.\n\n"
            "No target data or label content was accessed."
        )

    raise RuntimeError(
        "APTOS permission test failed.\n"
        f"Classified status: {failure_status}\n\n"
        "The detailed command output is shown immediately "
        "above and saved in the Stage 2 audit directory.\n\n"
        "No target label content was accessed."
    )


permission_files = [
    path
    for path in PERMISSION_TEST_ROOT.iterdir()
    if path.is_file()
]


if not permission_files:
    raise RuntimeError(
        "Kaggle returned success for the permission test, "
        "but no train.csv file was produced."
    )


print(
    "\nAPTOS download permission test passed."
)

print(
    "The small test file was not opened."
)


# Delete the tiny test copy without reading it.
for permission_file in permission_files:
    permission_file.unlink()


# ============================================================
# 6. Full archive download using positional syntax
# ============================================================

existing_zip_candidates = sorted(
    DOWNLOAD_ROOT.glob(
        "*.zip"
    ),
    key=lambda path: (
        path.stat().st_mtime
    ),
    reverse=True,
)


usable_existing_zip = next(
    (
        path
        for path in existing_zip_candidates
        if path.stat().st_size
        >
        1024 ** 3
    ),
    None,
)


if usable_existing_zip is not None:
    aptos_zip_path = (
        usable_existing_zip
    )

    full_download_result = {
        "command": (
            "REUSED_EXISTING_LARGE_ZIP"
        ),
        "returncode": 0,
        "elapsed_seconds": 0.0,
        "output_tail": [],
    }

    print(
        "\nA previously downloaded APTOS archive "
        "was found and will be reused:"
    )

    print(
        aptos_zip_path
    )

else:
    full_download_command = [
        "kaggle",
        "competitions",
        "download",
        COMPETITION,
        "-p",
        str(
            DOWNLOAD_ROOT
        ),
        "-o",
    ]

    print(
        "\nThe full archive is about 9.5 GiB."
    )

    print(
        "The command may show little output for several "
        "minutes while data are transferred."
    )

    full_download_result = (
        run_command_with_visible_output(
            full_download_command,
            timeout_seconds=14400,
        )
    )

    if (
        full_download_result[
            "returncode"
        ]
        !=
        0
    ):
        failure_status = classify_failure(
            full_download_result[
                "output_tail"
            ]
        )

        audit_payload = {
            "status": failure_status,
            "stage": "full_archive_download",
            "kaggle_version": kaggle_version,
            "permission_test": (
                permission_result
            ),
            "full_download": (
                full_download_result
            ),
            "target_label_content_accessed": False,
            "target_files_extracted": False,
            "archive_downloaded": False,
        }

        with open(
            AUDIT_PATH,
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                audit_payload,
                file,
                indent=2,
            )

        raise RuntimeError(
            "APTOS full archive download failed.\n"
            f"Classified status: {failure_status}\n\n"
            "The detailed Kaggle output is shown above and "
            "saved in the Stage 2 audit directory.\n\n"
            "No target labels were opened or processed."
        )

    produced_zip_candidates = sorted(
        DOWNLOAD_ROOT.glob(
            "*.zip"
        ),
        key=lambda path: (
            path.stat().st_mtime
        ),
        reverse=True,
    )

    if not produced_zip_candidates:
        raise RuntimeError(
            "Kaggle returned success, but no ZIP archive "
            "was found in the download directory."
        )

    aptos_zip_path = (
        produced_zip_candidates[0]
    )


archive_size_bytes = int(
    aptos_zip_path.stat().st_size
)

archive_size_gib = float(
    archive_size_bytes
    /
    (1024 ** 3)
)


if archive_size_bytes < 1024 ** 3:
    raise RuntimeError(
        "The resulting APTOS ZIP is unexpectedly small: "
        f"{archive_size_gib:.3f} GiB."
    )


# ============================================================
# 7. Seal successful download-gate record
# ============================================================

audit_payload = {
    "status": (
        "PASS_APTOS_ARCHIVE_DOWNLOAD_"
        "ADVANCE_TO_LOCAL_CANONICALISATION"
    ),
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "kaggle_version": kaggle_version,
    "competition": COMPETITION,
    "permission_test": (
        permission_result
    ),
    "full_download": (
        full_download_result
    ),
    "archive_path": str(
        aptos_zip_path
    ),
    "archive_size_bytes": (
        archive_size_bytes
    ),
    "archive_size_gib": (
        archive_size_gib
    ),
    "target_label_content_accessed": False,
    "target_files_extracted": False,
    "archive_retained_in_temporary_colab_storage": True,
    "authorised_next_step": (
        "EXTRACT_TRAINING_MATERIAL_AND_CANONICALISE"
    ),
}


with open(
    AUDIT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        audit_payload,
        file,
        indent=2,
    )


# ============================================================
# 8. Final display
# ============================================================

print(
    "\n================ APTOS DOWNLOAD GATE RESULT "
    "================"
)

print("Decision:")
print(
    audit_payload["status"]
)

print("\nArchive path:")
print(
    aptos_zip_path
)

print("\nArchive size:")
print(
    f"{archive_size_gib:.2f} GiB"
)

print("\nTarget files extracted:")
print(False)

print("\nTarget-label content accessed:")
print(False)

print("\nAuthorised next step:")
print(
    audit_payload[
        "authorised_next_step"
    ]
)

print(
    "\nAPTOS download gate completed and sealed."
)

================ APTOS DOWNLOAD GATE ================

Safety boundary:
This cell verifies Kaggle competition-download permission and downloads the official APTOS competition archive into temporary Colab storage. It does not extract the archive, open train.csv, inspect labels, or write target images to Drive.

Kaggle credential found.
Credential contents will not be displayed or copied to Drive.

Kaggle CLI:
Kaggle CLI 2.2.3

Running:
kaggle competitions download aptos2019-blindness-detection -f train.csv -p /content/cdo_stage2r_storage_efficient/permission_test -o -q

APTOS download permission test passed.
The small test file was not opened.

The full archive is about 9.5 GiB.
The command may show little output for several minutes while data are transferred.

Running:
kaggle competitions download aptos2019-blindness-detection -p /content/cdo_stage2r_storage_efficient/download -o

  0%|          | 0.00/9.51G [00:00<?, ?B/s]
  0%|          | 13.0M/9.51G [00:00<01:25, 120MB/s]
  0%|     

In [16]:
#@title 02R-C. Continue from downloaded APTOS archive: canonicalisation, split sealing, and label quarantine

from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO

from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import re
import shutil
import sys
import zipfile

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen protocol
# ============================================================

RANDOM_SEED = 20260720
APTOS_EVALUATION_FRACTION = 0.30

APTOS_EXPECTED_IMAGES = 3662

IDRID_EXPECTED_TOTAL = 516
IDRID_EXPECTED_DEVELOPMENT = 413
IDRID_EXPECTED_EVALUATION = 103

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

CANONICAL_SIZE = 768
CANONICAL_JPEG_QUALITY = 92
CANONICAL_JPEG_SUBSAMPLING = 0
BLACK_BORDER_THRESHOLD = 10

ImageFile.LOAD_TRUNCATED_IMAGES = False


PROTOCOL_BOUNDARY = (
    "This continuation reuses the already downloaded APTOS "
    "archive. Images are read directly from the ZIP and converted "
    "into the frozen canonical representation; persistent original "
    "APTOS images are never created. Target labels may be read only "
    "by the quarantine procedure. Sealed-evaluation label values "
    "must not be displayed, summarised, or inserted into modelling "
    "manifests."
)


CANONICALISATION_DESCRIPTION = (
    "Convert to RGB; identify non-black retinal content using "
    f"grayscale intensity > {BLACK_BORDER_THRESHOLD}; crop to the "
    "detected content bounding box when valid; pad centrally to a "
    "square using black pixels; resize with LANCZOS to "
    f"{CANONICAL_SIZE}x{CANONICAL_SIZE}; save as JPEG quality "
    f"{CANONICAL_JPEG_QUALITY}, subsampling "
    f"{CANONICAL_JPEG_SUBSAMPLING}. The identical procedure must "
    "later be applied to EyePACS and DeepDRiD."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

DESIGN_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
)

STAGE1_ROOT = (
    DESIGN_ROOT
    / "Stage1_Target_Feasibility_v0.1"
)

TEST_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

PROTOCOL_ROOT = (
    TEST_ROOT
    / "00_Protocol"
)

CANONICAL_ROOT = (
    TEST_ROOT
    / "01_Canonical_Target_Images"
)

DEVELOPMENT_ROOT = (
    TEST_ROOT
    / "02_Development"
)

SEALED_EVALUATION_ROOT = (
    TEST_ROOT
    / "03_Sealed_Evaluation"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

ORIGINAL_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "00_Original_Source_Label_Files"
)

SEALED_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "01_Sealed_Evaluation_Labels"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)


for directory in [
    PROTOCOL_ROOT,
    CANONICAL_ROOT,
    DEVELOPMENT_ROOT,
    SEALED_EVALUATION_ROOT,
    ORIGINAL_LABEL_ROOT,
    SEALED_LABEL_ROOT,
    STAGE2_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


APTOS_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019"
)

IDRID_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "IDRiD"
)


# ============================================================
# 2. Import sealed authorisations and download gate
# ============================================================

STAGE1D_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1D_Final_Target_Feasibility_Decision_v0.1.json"
)

IDRID_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)

DOWNLOAD_GATE_PATH = (
    STAGE2_ROOT
    / "Stage2R_APTOS_Download_Gate_v0.1.json"
)


for required_path in [
    STAGE1D_DECISION_PATH,
    IDRID_INVENTORY_PATH,
    DOWNLOAD_GATE_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with open(
    STAGE1D_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage1d_decision = json.load(
        file
    )


with open(
    DOWNLOAD_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    download_gate = json.load(
        file
    )


assert (
    stage1d_decision["decision"]
    ==
    "PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION"
)


assert (
    download_gate["status"]
    ==
    "PASS_APTOS_ARCHIVE_DOWNLOAD_ADVANCE_TO_LOCAL_CANONICALISATION"
)


idrid_inventory_raw = pd.read_csv(
    IDRID_INVENTORY_PATH
)


archive_path = Path(
    download_gate["archive_path"]
)


if not archive_path.is_file():
    fallback_archives = sorted(
        (
            Path("/content")
            / "cdo_stage2r_storage_efficient"
            / "download"
        ).glob("*.zip"),
        key=lambda path: (
            path.stat().st_mtime
        ),
        reverse=True,
    )

    assert fallback_archives, (
        "The downloaded APTOS archive is no longer present. "
        "The Colab runtime may have restarted. In that case, "
        "rerun only the 02R-D download-gate cell."
    )

    archive_path = fallback_archives[0]


assert archive_path.stat().st_size > (
    9
    *
    1024 ** 3
)


print(
    "================ STAGE 2R CONTINUATION "
    "================"
)

print("\nImported Stage 1D decision:")
print(
    stage1d_decision["decision"]
)

print("\nImported download-gate decision:")
print(
    download_gate["status"]
)

print("\nAPTOS archive:")
print(
    archive_path
)

print(
    "\nArchive size:",
    f"{archive_path.stat().st_size / (1024 ** 3):.2f} GiB",
)

print("\nProtocol boundary:")
print(PROTOCOL_BOUNDARY)

print("\nFrozen canonicalisation:")
print(CANONICALISATION_DESCRIPTION)


# ============================================================
# 3. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    path = Path(path)
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_bytes(payload):
    return hashlib.sha256(
        payload
    ).hexdigest()


def directory_size_bytes(path):
    path = Path(path)

    return int(
        sum(
            item.stat().st_size
            for item in path.rglob("*")
            if item.is_file()
        )
    )


def bytes_to_gib(value):
    return float(
        value
        /
        (1024 ** 3)
    )


def normalise_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def resolve_column(
    dataframe,
    candidates,
    description,
):
    lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        candidate_key = (
            normalise_column_name(
                candidate
            )
        )

        if candidate_key in lookup:
            return lookup[
                candidate_key
            ]

    raise KeyError(
        f"Could not resolve {description}.\n"
        f"Candidates: {candidates}\n"
        f"Available columns: "
        f"{list(dataframe.columns)}"
    )


def normalise_image_id(value):
    return Path(
        str(value).strip()
    ).stem


def make_writable(path):
    path = Path(path)

    if path.exists():
        try:
            os.chmod(
                path,
                0o600,
            )
        except Exception:
            pass


def make_read_only(path):
    path = Path(path)

    try:
        os.chmod(
            path,
            0o400,
        )

        return True

    except Exception:
        return False


def write_bytes_verified(
    destination,
    payload,
):
    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    make_writable(
        destination
    )

    temporary_path = destination.with_name(
        destination.name
        +
        ".tmp"
    )

    if temporary_path.exists():
        temporary_path.unlink()

    temporary_path.write_bytes(
        payload
    )

    os.replace(
        temporary_path,
        destination,
    )

    expected_hash = sha256_bytes(
        payload
    )

    observed_hash = sha256_file(
        destination
    )

    assert expected_hash == observed_hash

    return {
        "destination_path": str(
            destination
        ),
        "sha256": (
            expected_hash
        ),
        "size_bytes": int(
            len(payload)
        ),
        "hash_verified": True,
    }


def copy_or_reuse_label(
    source,
    destination,
):
    source = Path(source)
    destination = Path(
        destination
    )

    destination.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    if source.is_file():
        source_hash = sha256_file(
            source
        )

        if destination.is_file():
            destination_hash = sha256_file(
                destination
            )

            assert (
                source_hash
                ==
                destination_hash
            ), (
                "The existing quarantined copy differs "
                "from the original source label file:\n"
                f"{source}\n{destination}"
            )

            return {
                "source_path": str(
                    source
                ),
                "destination_path": str(
                    destination
                ),
                "sha256": (
                    source_hash
                ),
                "copied_now": False,
                "source_deleted": False,
            }

        make_writable(
            destination
        )

        shutil.copy2(
            source,
            destination,
        )

        destination_hash = sha256_file(
            destination
        )

        assert (
            source_hash
            ==
            destination_hash
        )

        return {
            "source_path": str(
                source
            ),
            "destination_path": str(
                destination
            ),
            "sha256": (
                source_hash
            ),
            "copied_now": True,
            "source_deleted": False,
        }

    if destination.is_file():
        return {
            "source_path": str(
                source
            ),
            "destination_path": str(
                destination
            ),
            "sha256": sha256_file(
                destination
            ),
            "copied_now": False,
            "source_deleted": False,
        }

    raise FileNotFoundError(
        "Label file is absent from both its original "
        "location and the quarantine directory:\n"
        f"{source}\n{destination}"
    )


def canonicalise_loaded_image(
    image,
    destination_path,
    original_sha256,
    original_size_bytes,
    original_format,
):
    destination_path = Path(
        destination_path
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    image.load()

    original_width = int(
        image.width
    )

    original_height = int(
        image.height
    )

    rgb = image.convert(
        "RGB"
    )

    grayscale = rgb.convert(
        "L"
    )

    binary_mask = grayscale.point(
        lambda pixel: (
            255
            if pixel
            >
            BLACK_BORDER_THRESHOLD
            else 0
        )
    )

    crop_box = binary_mask.getbbox()

    if crop_box is None:
        crop_box = (
            0,
            0,
            original_width,
            original_height,
        )

    cropped = rgb.crop(
        crop_box
    )

    side_length = max(
        cropped.width,
        cropped.height,
    )

    square = Image.new(
        "RGB",
        (
            side_length,
            side_length,
        ),
        (
            0,
            0,
            0,
        ),
    )

    paste_x = (
        side_length
        -
        cropped.width
    ) // 2

    paste_y = (
        side_length
        -
        cropped.height
    ) // 2

    square.paste(
        cropped,
        (
            paste_x,
            paste_y,
        ),
    )

    canonical = square.resize(
        (
            CANONICAL_SIZE,
            CANONICAL_SIZE,
        ),
        resample=(
            Image.Resampling.LANCZOS
        ),
    )

    canonical.save(
        destination_path,
        format="JPEG",
        quality=(
            CANONICAL_JPEG_QUALITY
        ),
        subsampling=(
            CANONICAL_JPEG_SUBSAMPLING
        ),
        optimize=True,
        progressive=False,
    )

    canonical_sha256 = sha256_file(
        destination_path
    )

    with Image.open(
        destination_path
    ) as verification_image:
        verification_image.verify()

    return {
        "original_sha256": (
            original_sha256
        ),
        "original_size_bytes": int(
            original_size_bytes
        ),
        "original_format": str(
            original_format
        ),
        "original_width": (
            original_width
        ),
        "original_height": (
            original_height
        ),
        "crop_left": int(
            crop_box[0]
        ),
        "crop_top": int(
            crop_box[1]
        ),
        "crop_right": int(
            crop_box[2]
        ),
        "crop_bottom": int(
            crop_box[3]
        ),
        "canonical_sha256": (
            canonical_sha256
        ),
        "canonical_width": (
            CANONICAL_SIZE
        ),
        "canonical_height": (
            CANONICAL_SIZE
        ),
        "canonical_size_bytes": int(
            destination_path.stat().st_size
        ),
    }


def canonicalise_bytes(
    payload,
    destination_path,
):
    original_sha256 = (
        sha256_bytes(
            payload
        )
    )

    with Image.open(
        BytesIO(
            payload
        )
    ) as image:
        original_format = (
            image.format
        )

        return canonicalise_loaded_image(
            image=image,
            destination_path=(
                destination_path
            ),
            original_sha256=(
                original_sha256
            ),
            original_size_bytes=(
                len(payload)
            ),
            original_format=(
                original_format
            ),
        )


def canonicalise_file(
    source_path,
    destination_path,
):
    source_path = Path(
        source_path
    )

    original_sha256 = sha256_file(
        source_path
    )

    with Image.open(
        source_path
    ) as image:
        original_format = (
            image.format
        )

        return canonicalise_loaded_image(
            image=image,
            destination_path=(
                destination_path
            ),
            original_sha256=(
                original_sha256
            ),
            original_size_bytes=(
                source_path.stat().st_size
            ),
            original_format=(
                original_format
            ),
        )


def json_default(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    raise TypeError(
        f"Unsupported JSON value: "
        f"{type(value)}"
    )


# ============================================================
# 4. Storage preflight and local workspace
# ============================================================

local_disk = shutil.disk_usage(
    "/content"
)

drive_disk = shutil.disk_usage(
    DRIVE_ROOT
)


print(
    "\n================ STORAGE PREFLIGHT "
    "================"
)

print(
    "Colab local free space:",
    f"{bytes_to_gib(local_disk.free):.2f} GiB",
)

print(
    "Google Drive reported free space:",
    f"{bytes_to_gib(drive_disk.free):.2f} GiB",
)

print(
    "APTOS archive already present:",
    f"{bytes_to_gib(archive_path.stat().st_size):.2f} GiB",
)


assert (
    local_disk.free
    >
    3
    *
    1024 ** 3
), (
    "At least 3 GiB of local Colab space is required "
    "for canonical image staging."
)


assert (
    drive_disk.free
    >
    2
    *
    1024 ** 3
), (
    "At least 2 GiB of Drive space is required before "
    "canonicalisation begins."
)


TEMP_ROOT = (
    archive_path.parent.parent
)

LOCAL_WORK_ROOT = (
    TEMP_ROOT
    / "stage2r_continuation_work"
)

LOCAL_CANONICAL_ROOT = (
    LOCAL_WORK_ROOT
    / "canonical"
)

LOCAL_APTOS_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "APTOS_2019"
)

LOCAL_IDRID_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "IDRiD"
)


if LOCAL_WORK_ROOT.exists():
    shutil.rmtree(
        LOCAL_WORK_ROOT
    )


LOCAL_APTOS_CANONICAL.mkdir(
    parents=True,
    exist_ok=True,
)

LOCAL_IDRID_CANONICAL.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 5. Read APTOS directly from ZIP
# ============================================================

APTOS_ORIGINAL_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "APTOS_2019_Original_train.csv"
)


with zipfile.ZipFile(
    archive_path,
    "r",
) as archive:
    archive_names = archive.namelist()

    train_image_members = sorted(
        [
            member
            for member in archive_names
            if (
                member.startswith(
                    "train_images/"
                )
                and
                Path(member).suffix.lower()
                in {
                    ".png",
                    ".jpg",
                    ".jpeg",
                }
            )
        ]
    )

    assert (
        len(train_image_members)
        ==
        APTOS_EXPECTED_IMAGES
    ), (
        f"Expected {APTOS_EXPECTED_IMAGES} "
        f"APTOS training images, found "
        f"{len(train_image_members)}."
    )

    assert (
        "train.csv"
        in archive_names
    )

    train_csv_bytes = archive.read(
        "train.csv"
    )

    aptos_label_copy_record = (
        write_bytes_verified(
            APTOS_ORIGINAL_LABEL_PATH,
            train_csv_bytes,
        )
    )

    print(
        "\n================ APTOS CANONICALISATION "
        "================"
    )

    aptos_processing_records = []

    for member in tqdm(
        train_image_members,
        desc=(
            "Canonicalising APTOS directly from ZIP"
        ),
    ):
        member_path = Path(
            member
        )

        if (
            member_path.is_absolute()
            or
            ".."
            in member_path.parts
        ):
            raise RuntimeError(
                f"Unsafe ZIP member: "
                f"{member}"
            )

        payload = archive.read(
            member
        )

        image_id = (
            member_path.stem
        )

        destination_path = (
            LOCAL_APTOS_CANONICAL
            /
            f"{image_id}.jpg"
        )

        processing = (
            canonicalise_bytes(
                payload,
                destination_path,
            )
        )

        aptos_processing_records.append(
            {
                "dataset": (
                    "APTOS_2019"
                ),
                "image_id": (
                    image_id
                ),
                "original_filename": (
                    member_path.name
                ),
                "archive_member": (
                    member
                ),
                "canonical_filename": (
                    destination_path.name
                ),
                **processing,
            }
        )


aptos_integrity = pd.DataFrame(
    aptos_processing_records
)


assert (
    len(aptos_integrity)
    ==
    APTOS_EXPECTED_IMAGES
)

assert aptos_integrity[
    "image_id"
].is_unique


# ============================================================
# 6. Resolve and canonicalise IDRiD
# ============================================================

idrid_inventory = (
    idrid_inventory_raw[
        idrid_inventory_raw[
            "dataset"
        ]
        ==
        "IDRiD"
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


required_inventory_columns = {
    "path",
    "filename",
    "suffix",
}


assert (
    required_inventory_columns
    .issubset(
        idrid_inventory.columns
    )
)


idrid_inventory[
    "filename_lower"
] = (
    idrid_inventory[
        "filename"
    ]
    .astype(str)
    .str.lower()
)


idrid_image_rows = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            ".jpg",
            ".jpeg",
        ]
    )
].copy()


assert (
    len(idrid_image_rows)
    ==
    IDRID_EXPECTED_TOTAL
)


idrid_csv_rows = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    ==
    ".csv"
].copy()


idrid_training_rows = idrid_csv_rows[
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "training",
        regex=False,
    )
    &
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


idrid_testing_rows = idrid_csv_rows[
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "testing",
        regex=False,
    )
    &
    idrid_csv_rows[
        "filename_lower"
    ]
    .str.contains(
        "grading",
        regex=False,
    )
]


assert (
    len(idrid_training_rows)
    ==
    1
)

assert (
    len(idrid_testing_rows)
    ==
    1
)


IDRID_ORIGINAL_TRAIN_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Training_Labels.csv"
)

IDRID_ORIGINAL_TEST_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Testing_Labels.csv"
)


idrid_train_copy_record = (
    copy_or_reuse_label(
        Path(
            idrid_training_rows
            .iloc[0]["path"]
        ),
        IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    )
)


idrid_test_copy_record = (
    copy_or_reuse_label(
        Path(
            idrid_testing_rows
            .iloc[0]["path"]
        ),
        IDRID_ORIGINAL_TEST_LABEL_PATH,
    )
)


print(
    "\n================ IDRiD CANONICALISATION "
    "================"
)


idrid_processing_records = []


for _, inventory_row in tqdm(
    idrid_image_rows.iterrows(),
    total=len(
        idrid_image_rows
    ),
    desc="Canonicalising IDRiD",
):
    source_path = Path(
        inventory_row["path"]
    )

    assert source_path.is_file(), (
        source_path
    )

    image_id = (
        source_path.stem
    )

    destination_path = (
        LOCAL_IDRID_CANONICAL
        /
        f"{image_id}.jpg"
    )

    processing = canonicalise_file(
        source_path,
        destination_path,
    )

    idrid_processing_records.append(
        {
            "dataset": "IDRiD",
            "image_id": image_id,
            "original_filename": (
                source_path.name
            ),
            "original_source_path": str(
                source_path
            ),
            "canonical_filename": (
                destination_path.name
            ),
            **processing,
        }
    )


idrid_integrity = pd.DataFrame(
    idrid_processing_records
)


assert (
    len(idrid_integrity)
    ==
    IDRID_EXPECTED_TOTAL
)

assert idrid_integrity[
    "image_id"
].is_unique


# ============================================================
# 7. Quarantine-only label parsing
# ============================================================

aptos_labels_raw = pd.read_csv(
    APTOS_ORIGINAL_LABEL_PATH
)

idrid_train_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TRAIN_LABEL_PATH
)

idrid_test_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TEST_LABEL_PATH
)


aptos_id_column = resolve_column(
    aptos_labels_raw,
    [
        "id_code",
        "image_id",
        "image",
    ],
    "APTOS image ID",
)

aptos_grade_column = resolve_column(
    aptos_labels_raw,
    [
        "diagnosis",
        "grade",
        "dr_grade",
    ],
    "APTOS DR grade",
)


aptos_labels = pd.DataFrame(
    {
        "dataset": (
            "APTOS_2019"
        ),
        "image_id": (
            aptos_labels_raw[
                aptos_id_column
            ]
            .map(
                normalise_image_id
            )
        ),
        "original_dr_grade": (
            pd.to_numeric(
                aptos_labels_raw[
                    aptos_grade_column
                ],
                errors="raise",
            )
            .astype(int)
        ),
    }
)


assert (
    len(aptos_labels)
    ==
    APTOS_EXPECTED_IMAGES
)

assert aptos_labels[
    "image_id"
].is_unique

assert set(
    aptos_labels[
        "original_dr_grade"
    ].unique()
).issubset(
    {
        0,
        1,
        2,
        3,
        4,
    }
)


def standardise_idrid_labels(
    dataframe,
    official_partition,
):
    image_column = resolve_column(
        dataframe,
        [
            "Image name",
            "image_name",
            "image",
        ],
        "IDRiD image name",
    )

    grade_column = resolve_column(
        dataframe,
        [
            "Retinopathy grade",
            "retinopathy_grade",
            "DR grade",
            "dr_grade",
            "grade",
        ],
        "IDRiD retinopathy grade",
    )

    return pd.DataFrame(
        {
            "dataset": "IDRiD",
            "image_id": (
                dataframe[
                    image_column
                ]
                .map(
                    normalise_image_id
                )
            ),
            "original_dr_grade": (
                pd.to_numeric(
                    dataframe[
                        grade_column
                    ],
                    errors="raise",
                )
                .astype(int)
            ),
            "official_partition": (
                official_partition
            ),
        }
    )


idrid_train_labels = (
    standardise_idrid_labels(
        idrid_train_labels_raw,
        "development",
    )
)

idrid_test_labels = (
    standardise_idrid_labels(
        idrid_test_labels_raw,
        "sealed_evaluation",
    )
)


assert (
    len(idrid_train_labels)
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(idrid_test_labels)
    ==
    IDRID_EXPECTED_EVALUATION
)


idrid_labels = pd.concat(
    [
        idrid_train_labels,
        idrid_test_labels,
    ],
    ignore_index=True,
)


assert (
    len(idrid_labels)
    ==
    IDRID_EXPECTED_TOTAL
)

assert idrid_labels[
    "image_id"
].is_unique

assert set(
    idrid_labels[
        "original_dr_grade"
    ].unique()
).issubset(
    {
        0,
        1,
        2,
        3,
        4,
    }
)


# ============================================================
# 8. Join labels and integrity records
# ============================================================

aptos_verified = aptos_labels.merge(
    aptos_integrity,
    on=[
        "dataset",
        "image_id",
    ],
    how="inner",
    validate="one_to_one",
)


idrid_verified = idrid_labels.merge(
    idrid_integrity,
    on=[
        "dataset",
        "image_id",
    ],
    how="inner",
    validate="one_to_one",
)


assert (
    len(aptos_verified)
    ==
    APTOS_EXPECTED_IMAGES
)

assert (
    len(idrid_verified)
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 9. Deterministic, duplicate-group-aware APTOS split
# ============================================================

original_hash_grade_counts = (
    aptos_verified
    .groupby(
        "original_sha256"
    )[
        "original_dr_grade"
    ]
    .nunique()
)


canonical_hash_grade_counts = (
    aptos_verified
    .groupby(
        "canonical_sha256"
    )[
        "original_dr_grade"
    ]
    .nunique()
)


assert not (
    original_hash_grade_counts
    >
    1
).any(), (
    "Exact original-image duplicates have "
    "conflicting DR grades."
)


assert not (
    canonical_hash_grade_counts
    >
    1
).any(), (
    "Identical canonical images have "
    "conflicting DR grades."
)


aptos_groups = (
    aptos_verified
    .groupby(
        "original_sha256",
        as_index=False,
    )
    .agg(
        original_dr_grade=(
            "original_dr_grade",
            "first",
        ),
        group_size=(
            "image_id",
            "size",
        ),
    )
    .sort_values(
        "original_sha256"
    )
    .reset_index(
        drop=True
    )
)


splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=(
        APTOS_EVALUATION_FRACTION
    ),
    random_state=(
        RANDOM_SEED
    ),
)


development_group_indices, evaluation_group_indices = next(
    splitter.split(
        np.zeros(
            len(aptos_groups)
        ),
        aptos_groups[
            "original_dr_grade"
        ].to_numpy(),
    )
)


evaluation_original_hashes = set(
    aptos_groups.iloc[
        evaluation_group_indices
    ][
        "original_sha256"
    ]
)


aptos_verified[
    "prospective_partition"
] = np.where(
    aptos_verified[
        "original_sha256"
    ].isin(
        evaluation_original_hashes
    ),
    "sealed_evaluation",
    "development",
)


canonical_crossing_counts = (
    aptos_verified
    .groupby(
        "canonical_sha256"
    )[
        "prospective_partition"
    ]
    .nunique()
)


canonical_crossing_hashes = set(
    canonical_crossing_counts[
        canonical_crossing_counts
        >
        1
    ].index
)


if canonical_crossing_hashes:
    aptos_verified.loc[
        aptos_verified[
            "canonical_sha256"
        ].isin(
            canonical_crossing_hashes
        ),
        "prospective_partition",
    ] = "sealed_evaluation"


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    assert not (
        aptos_verified
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).any()


# ============================================================
# 10. Preserve official IDRiD split and audit leakage
# ============================================================

idrid_verified[
    "prospective_partition"
] = (
    idrid_verified[
        "official_partition"
    ]
)


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "development"
).sum() == IDRID_EXPECTED_DEVELOPMENT


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "sealed_evaluation"
).sum() == IDRID_EXPECTED_EVALUATION


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grade_conflicts = (
        idrid_verified
        .groupby(
            hash_column
        )[
            "original_dr_grade"
        ]
        .nunique()
    )

    assert not (
        grade_conflicts
        >
        1
    ).any(), (
        "IDRiD contains identical images "
        "with conflicting grades."
    )

    assert not (
        idrid_verified
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).any(), (
        "IDRiD exact duplicate leakage was "
        "detected across its official split."
    )


original_cross_dataset_overlap = set(
    aptos_verified[
        "original_sha256"
    ]
).intersection(
    set(
        idrid_verified[
            "original_sha256"
        ]
    )
)


canonical_cross_dataset_overlap = set(
    aptos_verified[
        "canonical_sha256"
    ]
).intersection(
    set(
        idrid_verified[
            "canonical_sha256"
        ]
    )
)


assert not (
    original_cross_dataset_overlap
)

assert not (
    canonical_cross_dataset_overlap
)


# ============================================================
# 11. Harmonised endpoint
# ============================================================

for dataframe in [
    aptos_verified,
    idrid_verified,
]:
    dataframe[
        "moderate_or_worse_dr"
    ] = (
        dataframe[
            "original_dr_grade"
        ]
        >=
        2
    ).astype(int)


# ============================================================
# 12. Canonical storage audit
# ============================================================

aptos_canonical_bytes = (
    directory_size_bytes(
        LOCAL_APTOS_CANONICAL
    )
)

idrid_canonical_bytes = (
    directory_size_bytes(
        LOCAL_IDRID_CANONICAL
    )
)

canonical_total_bytes = (
    aptos_canonical_bytes
    +
    idrid_canonical_bytes
)


current_drive_free = (
    shutil.disk_usage(
        DRIVE_ROOT
    ).free
)


print(
    "\n================ CANONICAL STORAGE AUDIT "
    "================"
)

print(
    "APTOS canonical size:",
    f"{bytes_to_gib(aptos_canonical_bytes):.2f} GiB",
)

print(
    "IDRiD canonical size:",
    f"{bytes_to_gib(idrid_canonical_bytes):.2f} GiB",
)

print(
    "Total canonical size:",
    f"{bytes_to_gib(canonical_total_bytes):.2f} GiB",
)

print(
    "Current Drive free space:",
    f"{bytes_to_gib(current_drive_free):.2f} GiB",
)


assert (
    current_drive_free
    >
    canonical_total_bytes
    +
    512
    *
    1024 ** 2
), (
    "The generated canonical images exceed the "
    "remaining Drive capacity. No canonical images "
    "have yet been copied to Drive."
)


# ============================================================
# 13. Copy complete canonical sets to Drive staging
# ============================================================

APTOS_STAGING_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019__STAGING"
)

IDRID_STAGING_ROOT = (
    CANONICAL_ROOT
    / "IDRiD__STAGING"
)


for staging_root in [
    APTOS_STAGING_ROOT,
    IDRID_STAGING_ROOT,
]:
    if staging_root.exists():
        shutil.rmtree(
            staging_root
        )


print(
    "\nCopying canonical image sets to "
    "Google Drive staging directories..."
)


shutil.copytree(
    LOCAL_APTOS_CANONICAL,
    APTOS_STAGING_ROOT,
)


shutil.copytree(
    LOCAL_IDRID_CANONICAL,
    IDRID_STAGING_ROOT,
)


assert (
    len(
        list(
            APTOS_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    APTOS_EXPECTED_IMAGES
)


assert (
    len(
        list(
            IDRID_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 14. Verify Drive copies by SHA-256
# ============================================================

print(
    "\n================ DRIVE COPY VERIFICATION "
    "================"
)


for _, row in tqdm(
    aptos_integrity.iterrows(),
    total=len(
        aptos_integrity
    ),
    desc="Verifying APTOS Drive hashes",
):
    copied_path = (
        APTOS_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


for _, row in tqdm(
    idrid_integrity.iterrows(),
    total=len(
        idrid_integrity
    ),
    desc="Verifying IDRiD Drive hashes",
):
    copied_path = (
        IDRID_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


if APTOS_CANONICAL_ROOT.exists():
    shutil.rmtree(
        APTOS_CANONICAL_ROOT
    )


if IDRID_CANONICAL_ROOT.exists():
    shutil.rmtree(
        IDRID_CANONICAL_ROOT
    )


APTOS_STAGING_ROOT.rename(
    APTOS_CANONICAL_ROOT
)

IDRID_STAGING_ROOT.rename(
    IDRID_CANONICAL_ROOT
)


# ============================================================
# 15. Attach final Drive image paths
# ============================================================

aptos_verified[
    "canonical_image_path"
] = (
    aptos_verified[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            APTOS_CANONICAL_ROOT
            /
            f"{image_id}.jpg"
        )
    )
)


idrid_verified[
    "canonical_image_path"
] = (
    idrid_verified[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            IDRID_CANONICAL_ROOT
            /
            f"{image_id}.jpg"
        )
    )
)


for dataframe in [
    aptos_verified,
    idrid_verified,
]:
    assert dataframe[
        "canonical_image_path"
    ].map(
        lambda value: Path(
            value
        ).is_file()
    ).all()


# ============================================================
# 16. Construct partition tables
# ============================================================

aptos_development = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


aptos_evaluation = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_development = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_evaluation = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


assert (
    len(aptos_development)
    >
    0
)

assert (
    len(aptos_evaluation)
    >
    0
)

assert (
    len(idrid_development)
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(idrid_evaluation)
    ==
    IDRID_EXPECTED_EVALUATION
)


# ============================================================
# 17. Save development labels and sealed artifacts
# ============================================================

APTOS_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "APTOS_2019_Development_Labels_v0.1.csv"
)

IDRID_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "IDRiD_Development_Labels_v0.1.csv"
)

APTOS_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv"
)

IDRID_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv"
)

APTOS_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "APTOS_2019_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

IDRID_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "IDRiD_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)


for output_path in [
    APTOS_DEVELOPMENT_LABEL_PATH,
    IDRID_DEVELOPMENT_LABEL_PATH,
    APTOS_EVALUATION_MANIFEST_PATH,
    IDRID_EVALUATION_MANIFEST_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_writable(
        output_path
    )


development_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_dr_grade",
    "moderate_or_worse_dr",
    "original_sha256",
    "canonical_sha256",
]


evaluation_manifest_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_sha256",
    "canonical_sha256",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


sealed_label_columns = [
    "dataset",
    "image_id",
    "original_dr_grade",
    "moderate_or_worse_dr",
]


aptos_development[
    development_columns
].to_csv(
    APTOS_DEVELOPMENT_LABEL_PATH,
    index=False,
)


idrid_development[
    development_columns
].to_csv(
    IDRID_DEVELOPMENT_LABEL_PATH,
    index=False,
)


aptos_evaluation[
    evaluation_manifest_columns
].to_csv(
    APTOS_EVALUATION_MANIFEST_PATH,
    index=False,
)


idrid_evaluation[
    evaluation_manifest_columns
].to_csv(
    IDRID_EVALUATION_MANIFEST_PATH,
    index=False,
)


aptos_evaluation[
    sealed_label_columns
].to_csv(
    APTOS_SEALED_LABEL_PATH,
    index=False,
)


idrid_evaluation[
    sealed_label_columns
].to_csv(
    IDRID_SEALED_LABEL_PATH,
    index=False,
)


for protected_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_read_only(
        protected_path
    )


# ============================================================
# 18. Label-free master manifest
# ============================================================

label_free_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "prospective_partition",
    "original_sha256",
    "canonical_sha256",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


label_free_master_manifest = pd.concat(
    [
        aptos_verified[
            label_free_columns
        ],
        idrid_verified[
            label_free_columns
        ],
    ],
    ignore_index=True,
)


assert (
    "original_dr_grade"
    not in
    label_free_master_manifest.columns
)

assert (
    "moderate_or_worse_dr"
    not in
    label_free_master_manifest.columns
)


# ============================================================
# 19. Save protocol and audit artifacts
# ============================================================

CANONICAL_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Canonical_Fundus_Preprocessing_Protocol_v0.1.json"
)

QUARANTINE_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Target_Label_Quarantine_Protocol_v0.1.json"
)

MASTER_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_LabelFree_Master_Target_Manifest_v0.1.csv"
)

INTEGRITY_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

PARTITION_SUMMARY_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Partition_Summary_v0.1.csv"
)

LABEL_COMMITMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Sealed_Label_Commitments_v0.1.json"
)

SOURCE_LABEL_COPY_PATH = (
    STAGE2_ROOT
    / "Stage2_Source_Label_Quarantine_Copy_Manifest_v0.1.csv"
)

STAGE2_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)

STAGE2_REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Report_v0.1.md"
)

STAGE2_ENVIRONMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Environment_v0.1.json"
)


canonical_protocol = {
    "protocol_version": "v0.1",
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "black_border_threshold": (
        BLACK_BORDER_THRESHOLD
    ),
    "jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "jpeg_subsampling": (
        CANONICAL_JPEG_SUBSAMPLING
    ),
    "resize_filter": (
        "LANCZOS"
    ),
    "description": (
        CANONICALISATION_DESCRIPTION
    ),
    "frozen_before_target_performance": True,
    "must_be_applied_to_sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
}


with open(
    CANONICAL_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        canonical_protocol,
        file,
        indent=2,
    )


quarantine_protocol = {
    "protocol_version": "v0.1",
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "random_seed": (
        RANDOM_SEED
    ),
    "aptos_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "aptos_split_method": (
        "exact-original-hash-grouped "
        "grade-stratified split"
    ),
    "development_labels_authorised_from_stage": 3,
    "sealed_evaluation_labels_authorised_stage": 6,
    "sealed_evaluation_manifests_label_free": True,
    "sealed_label_values_displayed": False,
    "prohibitions": [
        (
            "Do not open, display, summarise or model "
            "sealed-evaluation labels before Stage 6."
        ),
        (
            "Do not change target partitions after "
            "target recoverability or transfer results "
            "are observed."
        ),
        (
            "Do not change the canonicalisation protocol "
            "after target performance is observed."
        ),
        (
            "Do not insert evaluation labels into any "
            "label-free manifest."
        ),
    ],
}


with open(
    QUARANTINE_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quarantine_protocol,
        file,
        indent=2,
    )


label_free_master_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False,
)


integrity_manifest = pd.concat(
    [
        aptos_integrity,
        idrid_integrity,
    ],
    ignore_index=True,
)


integrity_manifest.to_csv(
    INTEGRITY_MANIFEST_PATH,
    index=False,
)


partition_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "total_images": int(
                len(aptos_verified)
            ),
            "development_images": int(
                len(aptos_development)
            ),
            "sealed_evaluation_images": int(
                len(aptos_evaluation)
            ),
            "partition_source": (
                "original-hash-grouped deterministic "
                "grade-stratified split"
            ),
            "original_duplicate_groups": int(
                (
                    aptos_groups[
                        "group_size"
                    ]
                    >
                    1
                ).sum()
            ),
            "canonical_crossing_groups_reassigned": int(
                len(
                    canonical_crossing_hashes
                )
            ),
            "sealed_evaluation_labels_displayed": False,
        },
        {
            "dataset": "IDRiD",
            "total_images": int(
                len(idrid_verified)
            ),
            "development_images": int(
                len(idrid_development)
            ),
            "sealed_evaluation_images": int(
                len(idrid_evaluation)
            ),
            "partition_source": (
                "official IDRiD training/testing split"
            ),
            "original_duplicate_groups": int(
                (
                    idrid_verified
                    .groupby(
                        "original_sha256"
                    )
                    .size()
                    >
                    1
                ).sum()
            ),
            "canonical_crossing_groups_reassigned": 0,
            "sealed_evaluation_labels_displayed": False,
        },
    ]
)


partition_summary.to_csv(
    PARTITION_SUMMARY_PATH,
    index=False,
)


label_commitments = {
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "aptos_sealed_evaluation": {
        "rows": int(
            len(
                aptos_evaluation
            )
        ),
        "path": str(
            APTOS_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            APTOS_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "idrid_sealed_evaluation": {
        "rows": int(
            len(
                idrid_evaluation
            )
        ),
        "path": str(
            IDRID_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "original_label_files": {
        "aptos_train_sha256": (
            sha256_file(
                APTOS_ORIGINAL_LABEL_PATH
            )
        ),
        "idrid_train_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TRAIN_LABEL_PATH
            )
        ),
        "idrid_test_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TEST_LABEL_PATH
            )
        ),
    },
}


with open(
    LABEL_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_commitments,
        file,
        indent=2,
    )


source_label_copy_manifest = pd.DataFrame(
    [
        {
            "dataset": (
                "APTOS_2019"
            ),
            **aptos_label_copy_record,
            "source_deleted": False,
        },
        {
            "dataset": (
                "IDRiD_training"
            ),
            **idrid_train_copy_record,
        },
        {
            "dataset": (
                "IDRiD_testing"
            ),
            **idrid_test_copy_record,
        },
    ]
)


source_label_copy_manifest.to_csv(
    SOURCE_LABEL_COPY_PATH,
    index=False,
)


# ============================================================
# 20. Leakage and label-boundary checks
# ============================================================

aptos_original_cross_split_duplicates = int(
    (
        aptos_verified
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


aptos_canonical_cross_split_duplicates = int(
    (
        aptos_verified
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_original_cross_split_duplicates = int(
    (
        idrid_verified
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_canonical_cross_split_duplicates = int(
    (
        idrid_verified
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


evaluation_manifest_has_no_labels = bool(
    all(
        label_column
        not in pd.read_csv(
            manifest_path,
            nrows=1,
        ).columns
        for manifest_path in [
            APTOS_EVALUATION_MANIFEST_PATH,
            IDRID_EVALUATION_MANIFEST_PATH,
        ]
        for label_column in [
            "original_dr_grade",
            "moderate_or_worse_dr",
        ]
    )
)


stage2_pass = bool(
    len(aptos_verified)
    ==
    APTOS_EXPECTED_IMAGES
    and
    len(idrid_verified)
    ==
    IDRID_EXPECTED_TOTAL
    and
    len(idrid_development)
    ==
    IDRID_EXPECTED_DEVELOPMENT
    and
    len(idrid_evaluation)
    ==
    IDRID_EXPECTED_EVALUATION
    and
    aptos_original_cross_split_duplicates
    ==
    0
    and
    aptos_canonical_cross_split_duplicates
    ==
    0
    and
    idrid_original_cross_split_duplicates
    ==
    0
    and
    idrid_canonical_cross_split_duplicates
    ==
    0
    and
    len(
        original_cross_dataset_overlap
    )
    ==
    0
    and
    len(
        canonical_cross_dataset_overlap
    )
    ==
    0
    and
    evaluation_manifest_has_no_labels
)


assert stage2_pass


# ============================================================
# 21. Final Stage 2R decision
# ============================================================

stage2_decision = (
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_"
    "AND_LABEL_QUARANTINE_ADVANCE_TO_"
    "DEVELOPMENT_RECOVERABILITY_GATE"
)


stage2_interpretation = (
    "The downloaded APTOS archive was processed directly "
    "without creating persistent original-image copies. "
    "APTOS and IDRiD were converted with the identical frozen "
    "768x768 canonicalisation, copied to Google Drive, and "
    "hash-verified. APTOS was divided using a deterministic "
    "grade-stratified split with exact original-image groups "
    "kept together; IDRiD retained its official split. Sealed "
    "labels are absent from modelling manifests and committed "
    "for one-time Stage 6 evaluation."
)


decision_payload = {
    "decision": (
        stage2_decision
    ),
    "interpretation": (
        stage2_interpretation
    ),
    "authorised_next_step": (
        "DEVELOPMENT_ONLY_TARGET_RECOVERABILITY_GATE"
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "download_gate_path": str(
        DOWNLOAD_GATE_PATH
    ),
    "archive_reused": True,
    "persistent_aptos_original_images_created": False,
    "canonicalisation_protocol": str(
        CANONICAL_PROTOCOL_PATH
    ),
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "aptos": {
        "total": int(
            len(
                aptos_verified
            )
        ),
        "development": int(
            len(
                aptos_development
            )
        ),
        "sealed_evaluation": int(
            len(
                aptos_evaluation
            )
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                aptos_canonical_bytes
            )
        ),
        "original_cross_split_duplicates": (
            aptos_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            aptos_canonical_cross_split_duplicates
        ),
    },
    "idrid": {
        "total": int(
            len(
                idrid_verified
            )
        ),
        "development": int(
            len(
                idrid_development
            )
        ),
        "sealed_evaluation": int(
            len(
                idrid_evaluation
            )
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                idrid_canonical_bytes
            )
        ),
        "original_cross_split_duplicates": (
            idrid_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            idrid_canonical_cross_split_duplicates
        ),
    },
    "original_cross_dataset_overlap": int(
        len(
            original_cross_dataset_overlap
        )
    ),
    "canonical_cross_dataset_overlap": int(
        len(
            canonical_cross_dataset_overlap
        )
    ),
    "evaluation_manifests_label_free": (
        evaluation_manifest_has_no_labels
    ),
    "sealed_label_values_displayed": False,
    "sealed_label_commitment_path": str(
        LABEL_COMMITMENT_PATH
    ),
    "important_boundary": (
        "Stage 2 establishes acquisition, frozen "
        "canonicalisation, deterministic partitioning, and "
        "procedural label quarantine only. It does not establish "
        "target recoverability, source-target transfer, or CDO "
        "validity."
    ),
}


with open(
    STAGE2_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
        default=json_default,
    )


report_lines = [
    (
        "# Prospective Retinal Blind Test — Stage 2R"
    ),
    "",
    (
        "## Storage-efficient controlled acquisition "
        "and label quarantine"
    ),
    "",
    f"Decision: `{stage2_decision}`",
    "",
    stage2_interpretation,
    "",
    "## Canonical storage",
    "",
    (
        f"- APTOS: "
        f"{bytes_to_gib(aptos_canonical_bytes):.3f} GiB"
    ),
    (
        f"- IDRiD: "
        f"{bytes_to_gib(idrid_canonical_bytes):.3f} GiB"
    ),
    "",
    "## Label boundary",
    "",
    (
        "- Sealed-evaluation label values "
        "displayed: `False`"
    ),
    (
        "- Evaluation manifests are label-free: `True`"
    ),
    "",
]


STAGE2_REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "random_seed": (
        RANDOM_SEED
    ),
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "canonical_jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "archive_path_before_cleanup": str(
        archive_path
    ),
    "test_root": str(
        TEST_ROOT
    ),
}


with open(
    STAGE2_ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 22. Final integrity checks
# ============================================================

required_outputs = [
    CANONICAL_PROTOCOL_PATH,
    QUARANTINE_PROTOCOL_PATH,
    MASTER_MANIFEST_PATH,
    INTEGRITY_MANIFEST_PATH,
    PARTITION_SUMMARY_PATH,
    LABEL_COMMITMENT_PATH,
    SOURCE_LABEL_COPY_PATH,
    STAGE2_DECISION_PATH,
    STAGE2_REPORT_PATH,
    STAGE2_ENVIRONMENT_PATH,
    APTOS_DEVELOPMENT_LABEL_PATH,
    IDRID_DEVELOPMENT_LABEL_PATH,
    APTOS_EVALUATION_MANIFEST_PATH,
    IDRID_EVALUATION_MANIFEST_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]


for output_path in required_outputs:
    assert output_path.is_file(), (
        output_path
    )


canonical_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "images": int(
                len(
                    aptos_integrity
                )
            ),
            "unique_original_sha256": int(
                aptos_integrity[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                aptos_integrity[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    aptos_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
        {
            "dataset": "IDRiD",
            "images": int(
                len(
                    idrid_integrity
                )
            ),
            "unique_original_sha256": int(
                idrid_integrity[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                idrid_integrity[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    idrid_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
    ]
)


# ============================================================
# 23. Delete temporary ZIP and local canonical staging
# ============================================================

print(
    "\nDeleting the temporary APTOS archive and "
    "local canonical staging files..."
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


# ============================================================
# 24. Final display
# ============================================================

print(
    "\n================ CANONICAL IMAGE SUMMARY "
    "================"
)

display(
    canonical_summary
)


print(
    "\n================ PARTITION SUMMARY "
    "================"
)

display(
    partition_summary
)


print(
    "\n================ LEAKAGE CHECK "
    "================"
)

print(
    "APTOS original-hash cross-split duplicates:",
    aptos_original_cross_split_duplicates,
)

print(
    "APTOS canonical-hash cross-split duplicates:",
    aptos_canonical_cross_split_duplicates,
)

print(
    "IDRiD original-hash cross-split duplicates:",
    idrid_original_cross_split_duplicates,
)

print(
    "IDRiD canonical-hash cross-split duplicates:",
    idrid_canonical_cross_split_duplicates,
)

print(
    "Original cross-dataset overlap:",
    len(
        original_cross_dataset_overlap
    ),
)

print(
    "Canonical cross-dataset overlap:",
    len(
        canonical_cross_dataset_overlap
    ),
)

print(
    "Evaluation manifests contain labels:",
    not evaluation_manifest_has_no_labels,
)


print(
    "\n================ LABEL QUARANTINE CHECK "
    "================"
)

print(
    "APTOS sealed-label SHA-256:",
    label_commitments[
        "aptos_sealed_evaluation"
    ][
        "sha256"
    ],
)

print(
    "IDRiD sealed-label SHA-256:",
    label_commitments[
        "idrid_sealed_evaluation"
    ][
        "sha256"
    ],
)

print(
    "Sealed-evaluation label values displayed:",
    False,
)


print(
    "\n================ STAGE 2R DECISION "
    "================"
)

print("Decision:")
print(
    stage2_decision
)

print("\nInterpretation:")
print(
    stage2_interpretation
)

print("\nAuthorised next step:")
print(
    decision_payload[
        "authorised_next_step"
    ]
)

print(
    "\nPersistent original APTOS images retained:"
)

print(False)

print(
    "\nOriginal IDRiD source files deleted:"
)

print(False)

print(
    "\nTemporary APTOS archive retained:"
)

print(False)

print(
    "\nStage 2R continuation completed and sealed."
)

================ STAGE 2R CONTINUATION ================

Imported Stage 1D decision:
PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION

Imported download-gate decision:
PASS_APTOS_ARCHIVE_DOWNLOAD_ADVANCE_TO_LOCAL_CANONICALISATION

APTOS archive:
/content/cdo_stage2r_storage_efficient/download/aptos2019-blindness-detection.zip

Archive size: 9.51 GiB

Protocol boundary:
This continuation reuses the already downloaded APTOS archive. Images are read directly from the ZIP and converted into the frozen canonical representation; persistent original APTOS images are never created. Target labels may be read only by the quarantine procedure. Sealed-evaluation label values must not be displayed, summarised, or inserted into modelling manifests.

Frozen canonicalisation:
Convert to RGB; identify non-black retinal content using grayscale intensity > 10; crop to the detected content bounding box when valid; pad centrally to a square using black pixels; resize with LANCZOS to 768x768; save 

Canonicalising APTOS directly from ZIP:   0%|          | 0/3662 [00:00<?, ?it/s]


================ IDRiD CANONICALISATION ================


Canonicalising IDRiD:   0%|          | 0/516 [00:00<?, ?it/s]

AssertionError: 

In [17]:
#@title 02R-E. Repair overlapping IDRiD filenames and complete Stage 2R — reuse finished APTOS

from pathlib import Path
from datetime import datetime, timezone
from io import BytesIO

from PIL import Image, ImageFile
from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import re
import shutil
import sys
import zipfile

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen protocol
# ============================================================

RANDOM_SEED = 20260720
APTOS_EVALUATION_FRACTION = 0.30

APTOS_EXPECTED_IMAGES = 3662

IDRID_EXPECTED_TOTAL = 516
IDRID_EXPECTED_DEVELOPMENT = 413
IDRID_EXPECTED_EVALUATION = 103

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

CANONICAL_SIZE = 768
CANONICAL_JPEG_QUALITY = 92
CANONICAL_JPEG_SUBSAMPLING = 0
BLACK_BORDER_THRESHOLD = 10

ImageFile.LOAD_TRUNCATED_IMAGES = False


REPAIR_BOUNDARY = (
    "APTOS canonical images already completed in the current "
    "runtime are reused. Only IDRiD is canonicalised again, "
    "using partition-qualified unique image identifiers. "
    "Sealed-evaluation label values must not be displayed, "
    "summarised, or inserted into modelling manifests."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount("/content/drive")

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

DESIGN_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_Design_v0.1"
)

STAGE1_ROOT = (
    DESIGN_ROOT
    / "Stage1_Target_Feasibility_v0.1"
)

TEST_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

PROTOCOL_ROOT = (
    TEST_ROOT
    / "00_Protocol"
)

CANONICAL_ROOT = (
    TEST_ROOT
    / "01_Canonical_Target_Images"
)

DEVELOPMENT_ROOT = (
    TEST_ROOT
    / "02_Development"
)

SEALED_EVALUATION_ROOT = (
    TEST_ROOT
    / "03_Sealed_Evaluation"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

ORIGINAL_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "00_Original_Source_Label_Files"
)

SEALED_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "01_Sealed_Evaluation_Labels"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)


for directory in [
    PROTOCOL_ROOT,
    CANONICAL_ROOT,
    DEVELOPMENT_ROOT,
    SEALED_EVALUATION_ROOT,
    ORIGINAL_LABEL_ROOT,
    SEALED_LABEL_ROOT,
    STAGE2_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


APTOS_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019"
)

IDRID_CANONICAL_ROOT = (
    CANONICAL_ROOT
    / "IDRiD"
)


# ============================================================
# 2. Resolve sealed inputs and local completed work
# ============================================================

STAGE1D_DECISION_PATH = (
    STAGE1_ROOT
    / "Stage1D_Final_Target_Feasibility_Decision_v0.1.json"
)

IDRID_INVENTORY_PATH = (
    STAGE1_ROOT
    / "Stage1A_Local_Target_Artifact_Inventory_v0.1.csv"
)

DOWNLOAD_GATE_PATH = (
    STAGE2_ROOT
    / "Stage2R_APTOS_Download_Gate_v0.1.json"
)


for required_path in [
    STAGE1D_DECISION_PATH,
    IDRID_INVENTORY_PATH,
    DOWNLOAD_GATE_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with open(
    STAGE1D_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage1d_decision = json.load(
        file
    )


with open(
    DOWNLOAD_GATE_PATH,
    "r",
    encoding="utf-8",
) as file:
    download_gate = json.load(
        file
    )


assert (
    stage1d_decision["decision"]
    ==
    "PASS_TWO_TARGETS_ADVANCE_TO_CONTROLLED_TARGET_ACQUISITION"
)

assert (
    download_gate["status"]
    ==
    "PASS_APTOS_ARCHIVE_DOWNLOAD_ADVANCE_TO_LOCAL_CANONICALISATION"
)


archive_path = Path(
    download_gate["archive_path"]
)


assert archive_path.is_file(), (
    "The APTOS ZIP is no longer present. "
    "The Colab runtime may have restarted."
)


TEMP_ROOT = (
    archive_path.parent.parent
)

LOCAL_WORK_ROOT = (
    TEMP_ROOT
    / "stage2r_continuation_work"
)

LOCAL_CANONICAL_ROOT = (
    LOCAL_WORK_ROOT
    / "canonical"
)

LOCAL_APTOS_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "APTOS_2019"
)

LOCAL_IDRID_CANONICAL = (
    LOCAL_CANONICAL_ROOT
    / "IDRiD"
)


assert LOCAL_APTOS_CANONICAL.is_dir(), (
    "The completed local APTOS canonical directory "
    "is missing. Do not restart the runtime."
)


aptos_local_files = sorted(
    LOCAL_APTOS_CANONICAL.glob(
        "*.jpg"
    )
)


assert (
    len(aptos_local_files)
    ==
    APTOS_EXPECTED_IMAGES
), (
    f"Expected {APTOS_EXPECTED_IMAGES} completed "
    f"APTOS canonical images, found "
    f"{len(aptos_local_files)}."
)


if LOCAL_IDRID_CANONICAL.exists():
    shutil.rmtree(
        LOCAL_IDRID_CANONICAL
    )


LOCAL_IDRID_CANONICAL.mkdir(
    parents=True,
    exist_ok=True,
)


idrid_inventory_raw = pd.read_csv(
    IDRID_INVENTORY_PATH
)


print(
    "================ STAGE 2R IDRiD-ID REPAIR "
    "================"
)

print("\nRepair boundary:")
print(REPAIR_BOUNDARY)

print("\nCompleted APTOS canonical images reused:")
print(
    len(
        aptos_local_files
    )
)

print("\nAPTOS ZIP retained:")
print(
    archive_path
)


# ============================================================
# 3. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def sha256_bytes(payload):
    return hashlib.sha256(
        payload
    ).hexdigest()


def bytes_to_gib(value):
    return float(
        value
        /
        (1024 ** 3)
    )


def directory_size_bytes(path):
    path = Path(path)

    return int(
        sum(
            item.stat().st_size
            for item in path.rglob("*")
            if item.is_file()
        )
    )


def normalise_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def resolve_column(
    dataframe,
    candidates,
    description,
):
    lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        key = normalise_column_name(
            candidate
        )

        if key in lookup:
            return lookup[key]

    raise KeyError(
        f"Could not resolve {description}.\n"
        f"Candidates: {candidates}\n"
        f"Available columns: "
        f"{list(dataframe.columns)}"
    )


def normalise_image_id(value):
    return Path(
        str(value).strip()
    ).stem


def make_writable(path):
    path = Path(path)

    if path.exists():
        try:
            os.chmod(
                path,
                0o600,
            )
        except Exception:
            pass


def make_read_only(path):
    path = Path(path)

    try:
        os.chmod(
            path,
            0o400,
        )

        return True

    except Exception:
        return False


def infer_idrid_partition(path):
    path_text = str(
        path
    ).lower()

    if "training" in path_text:
        return "development"

    if "testing" in path_text:
        return "sealed_evaluation"

    raise ValueError(
        "Could not infer whether this IDRiD image "
        "belongs to training or testing:\n"
        f"{path}"
    )


def canonicalise_file(
    source_path,
    destination_path,
):
    source_path = Path(
        source_path
    )

    destination_path = Path(
        destination_path
    )

    destination_path.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    original_sha256 = sha256_file(
        source_path
    )

    original_size_bytes = int(
        source_path.stat().st_size
    )

    with Image.open(
        source_path
    ) as image:
        original_format = str(
            image.format
        )

        image.load()

        original_width = int(
            image.width
        )

        original_height = int(
            image.height
        )

        rgb = image.convert(
            "RGB"
        )

        grayscale = rgb.convert(
            "L"
        )

        binary_mask = grayscale.point(
            lambda pixel: (
                255
                if pixel
                >
                BLACK_BORDER_THRESHOLD
                else 0
            )
        )

        crop_box = binary_mask.getbbox()

        if crop_box is None:
            crop_box = (
                0,
                0,
                original_width,
                original_height,
            )

        cropped = rgb.crop(
            crop_box
        )

        side_length = max(
            cropped.width,
            cropped.height,
        )

        square = Image.new(
            "RGB",
            (
                side_length,
                side_length,
            ),
            (
                0,
                0,
                0,
            ),
        )

        paste_x = (
            side_length
            -
            cropped.width
        ) // 2

        paste_y = (
            side_length
            -
            cropped.height
        ) // 2

        square.paste(
            cropped,
            (
                paste_x,
                paste_y,
            ),
        )

        canonical = square.resize(
            (
                CANONICAL_SIZE,
                CANONICAL_SIZE,
            ),
            resample=(
                Image.Resampling.LANCZOS
            ),
        )

        canonical.save(
            destination_path,
            format="JPEG",
            quality=(
                CANONICAL_JPEG_QUALITY
            ),
            subsampling=(
                CANONICAL_JPEG_SUBSAMPLING
            ),
            optimize=True,
            progressive=False,
        )

    canonical_sha256 = sha256_file(
        destination_path
    )

    with Image.open(
        destination_path
    ) as verification_image:
        verification_image.verify()

    return {
        "original_sha256": (
            original_sha256
        ),
        "original_size_bytes": (
            original_size_bytes
        ),
        "original_format": (
            original_format
        ),
        "original_width": (
            original_width
        ),
        "original_height": (
            original_height
        ),
        "crop_left": int(
            crop_box[0]
        ),
        "crop_top": int(
            crop_box[1]
        ),
        "crop_right": int(
            crop_box[2]
        ),
        "crop_bottom": int(
            crop_box[3]
        ),
        "canonical_sha256": (
            canonical_sha256
        ),
        "canonical_width": (
            CANONICAL_SIZE
        ),
        "canonical_height": (
            CANONICAL_SIZE
        ),
        "canonical_size_bytes": int(
            destination_path.stat().st_size
        ),
    }


def json_default(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    raise TypeError(
        f"Unsupported JSON type: "
        f"{type(value)}"
    )


# ============================================================
# 4. Recover or rebuild APTOS integrity table
# ============================================================

existing_aptos_integrity = globals().get(
    "aptos_integrity"
)


use_memory_table = bool(
    isinstance(
        existing_aptos_integrity,
        pd.DataFrame,
    )
    and
    len(
        existing_aptos_integrity
    )
    ==
    APTOS_EXPECTED_IMAGES
    and
    {
        "image_id",
        "original_sha256",
        "canonical_sha256",
        "canonical_filename",
    }.issubset(
        existing_aptos_integrity.columns
    )
)


if use_memory_table:
    aptos_integrity_fixed = (
        existing_aptos_integrity
        .copy()
        .reset_index(drop=True)
    )

    print(
        "\nAPTOS integrity table recovered from "
        "the current notebook memory."
    )

else:
    print(
        "\nAPTOS integrity table was not found in memory."
    )

    print(
        "Reconstructing hashes and metadata from the "
        "existing ZIP and completed canonical images. "
        "APTOS will not be resized or saved again."
    )

    aptos_rebuild_rows = []

    with zipfile.ZipFile(
        archive_path,
        "r",
    ) as archive:
        train_members = sorted(
            [
                member
                for member in archive.namelist()
                if (
                    member.startswith(
                        "train_images/"
                    )
                    and
                    Path(
                        member
                    ).suffix.lower()
                    in {
                        ".png",
                        ".jpg",
                        ".jpeg",
                    }
                )
            ]
        )

        assert (
            len(train_members)
            ==
            APTOS_EXPECTED_IMAGES
        )

        for member in tqdm(
            train_members,
            desc=(
                "Reconstructing APTOS integrity"
            ),
        ):
            payload = archive.read(
                member
            )

            image_id = Path(
                member
            ).stem

            canonical_path = (
                LOCAL_APTOS_CANONICAL
                /
                f"{image_id}.jpg"
            )

            assert canonical_path.is_file()

            with Image.open(
                BytesIO(
                    payload
                )
            ) as image:
                original_format = str(
                    image.format
                )

                image.load()

                original_width = int(
                    image.width
                )

                original_height = int(
                    image.height
                )

                grayscale = (
                    image
                    .convert("RGB")
                    .convert("L")
                )

                binary_mask = (
                    grayscale.point(
                        lambda pixel: (
                            255
                            if pixel
                            >
                            BLACK_BORDER_THRESHOLD
                            else 0
                        )
                    )
                )

                crop_box = (
                    binary_mask.getbbox()
                )

                if crop_box is None:
                    crop_box = (
                        0,
                        0,
                        original_width,
                        original_height,
                    )

            with Image.open(
                canonical_path
            ) as canonical_image:
                assert (
                    canonical_image.size
                    ==
                    (
                        CANONICAL_SIZE,
                        CANONICAL_SIZE,
                    )
                )

                canonical_image.verify()

            aptos_rebuild_rows.append(
                {
                    "dataset": (
                        "APTOS_2019"
                    ),
                    "image_id": (
                        image_id
                    ),
                    "original_filename": (
                        Path(
                            member
                        ).name
                    ),
                    "archive_member": (
                        member
                    ),
                    "canonical_filename": (
                        canonical_path.name
                    ),
                    "original_sha256": (
                        sha256_bytes(
                            payload
                        )
                    ),
                    "original_size_bytes": int(
                        len(payload)
                    ),
                    "original_format": (
                        original_format
                    ),
                    "original_width": (
                        original_width
                    ),
                    "original_height": (
                        original_height
                    ),
                    "crop_left": int(
                        crop_box[0]
                    ),
                    "crop_top": int(
                        crop_box[1]
                    ),
                    "crop_right": int(
                        crop_box[2]
                    ),
                    "crop_bottom": int(
                        crop_box[3]
                    ),
                    "canonical_sha256": (
                        sha256_file(
                            canonical_path
                        )
                    ),
                    "canonical_width": (
                        CANONICAL_SIZE
                    ),
                    "canonical_height": (
                        CANONICAL_SIZE
                    ),
                    "canonical_size_bytes": int(
                        canonical_path
                        .stat()
                        .st_size
                    ),
                }
            )

    aptos_integrity_fixed = pd.DataFrame(
        aptos_rebuild_rows
    )


assert (
    len(
        aptos_integrity_fixed
    )
    ==
    APTOS_EXPECTED_IMAGES
)

assert aptos_integrity_fixed[
    "image_id"
].is_unique


# ============================================================
# 5. Diagnose IDRiD filename overlap
# ============================================================

idrid_inventory = (
    idrid_inventory_raw[
        idrid_inventory_raw[
            "dataset"
        ]
        ==
        "IDRiD"
    ]
    .copy()
    .reset_index(drop=True)
)


idrid_image_rows = idrid_inventory[
    idrid_inventory[
        "suffix"
    ]
    .astype(str)
    .str.lower()
    .isin(
        [
            ".jpg",
            ".jpeg",
        ]
    )
].copy()


assert (
    len(idrid_image_rows)
    ==
    IDRID_EXPECTED_TOTAL
)


idrid_image_rows[
    "source_image_id"
] = (
    idrid_image_rows[
        "filename"
    ]
    .astype(str)
    .map(
        normalise_image_id
    )
)


idrid_image_rows[
    "official_partition"
] = (
    idrid_image_rows[
        "path"
    ]
    .map(
        infer_idrid_partition
    )
)


idrid_image_rows[
    "image_id"
] = (
    idrid_image_rows[
        "official_partition"
    ]
    +
    "__"
    +
    idrid_image_rows[
        "source_image_id"
    ]
)


overlap_diagnostic = (
    idrid_image_rows
    .groupby(
        "source_image_id"
    )
    .agg(
        files=(
            "path",
            "size",
        ),
        partitions=(
            "official_partition",
            lambda values: (
                ", ".join(
                    sorted(
                        set(
                            values
                        )
                    )
                )
            ),
        ),
    )
    .reset_index()
)


duplicated_source_ids = (
    overlap_diagnostic[
        overlap_diagnostic[
            "files"
        ]
        >
        1
    ]
    .copy()
)


print(
    "\n================ IDRiD DUPLICATE-ID "
    "DIAGNOSIS ================"
)

print(
    "Total IDRiD image files:",
    len(
        idrid_image_rows
    ),
)

print(
    "Unique bare filename stems:",
    idrid_image_rows[
        "source_image_id"
    ].nunique(),
)

print(
    "Bare filename stems reused across folders:",
    len(
        duplicated_source_ids
    ),
)

print(
    "Partition-qualified IDs unique:",
    idrid_image_rows[
        "image_id"
    ].is_unique,
)


assert len(
    duplicated_source_ids
) > 0, (
    "No filename overlap was found; the previous "
    "failure requires a different diagnosis."
)


assert idrid_image_rows[
    "image_id"
].is_unique


display(
    duplicated_source_ids.head(
        10
    )
)


# ============================================================
# 6. Re-canonicalise only IDRiD with unique filenames
# ============================================================

print(
    "\n================ IDRiD RE-CANONICALISATION "
    "================"
)


idrid_processing_rows = []


for _, row in tqdm(
    idrid_image_rows.iterrows(),
    total=len(
        idrid_image_rows
    ),
    desc=(
        "Re-canonicalising IDRiD with partition IDs"
    ),
):
    source_path = Path(
        row["path"]
    )

    assert source_path.is_file(), (
        source_path
    )

    unique_image_id = row[
        "image_id"
    ]

    destination_path = (
        LOCAL_IDRID_CANONICAL
        /
        f"{unique_image_id}.jpg"
    )

    processing = canonicalise_file(
        source_path,
        destination_path,
    )

    idrid_processing_rows.append(
        {
            "dataset": "IDRiD",
            "image_id": (
                unique_image_id
            ),
            "source_image_id": (
                row[
                    "source_image_id"
                ]
            ),
            "official_partition": (
                row[
                    "official_partition"
                ]
            ),
            "original_filename": (
                source_path.name
            ),
            "original_source_path": str(
                source_path
            ),
            "canonical_filename": (
                destination_path.name
            ),
            **processing,
        }
    )


idrid_integrity_fixed = pd.DataFrame(
    idrid_processing_rows
)


assert (
    len(
        idrid_integrity_fixed
    )
    ==
    IDRID_EXPECTED_TOTAL
)

assert idrid_integrity_fixed[
    "image_id"
].is_unique


assert (
    len(
        list(
            LOCAL_IDRID_CANONICAL.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 7. Resolve quarantined label files
# ============================================================

APTOS_ORIGINAL_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "APTOS_2019_Original_train.csv"
)

IDRID_ORIGINAL_TRAIN_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Training_Labels.csv"
)

IDRID_ORIGINAL_TEST_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Testing_Labels.csv"
)


assert APTOS_ORIGINAL_LABEL_PATH.is_file(), (
    APTOS_ORIGINAL_LABEL_PATH
)

assert IDRID_ORIGINAL_TRAIN_LABEL_PATH.is_file(), (
    IDRID_ORIGINAL_TRAIN_LABEL_PATH
)

assert IDRID_ORIGINAL_TEST_LABEL_PATH.is_file(), (
    IDRID_ORIGINAL_TEST_LABEL_PATH
)


# ============================================================
# 8. Quarantine-only label parsing
# ============================================================

aptos_labels_raw = pd.read_csv(
    APTOS_ORIGINAL_LABEL_PATH
)

idrid_train_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TRAIN_LABEL_PATH
)

idrid_test_labels_raw = pd.read_csv(
    IDRID_ORIGINAL_TEST_LABEL_PATH
)


aptos_id_column = resolve_column(
    aptos_labels_raw,
    [
        "id_code",
        "image_id",
        "image",
    ],
    "APTOS image ID",
)

aptos_grade_column = resolve_column(
    aptos_labels_raw,
    [
        "diagnosis",
        "grade",
        "dr_grade",
    ],
    "APTOS DR grade",
)


aptos_labels = pd.DataFrame(
    {
        "dataset": (
            "APTOS_2019"
        ),
        "image_id": (
            aptos_labels_raw[
                aptos_id_column
            ]
            .map(
                normalise_image_id
            )
        ),
        "original_dr_grade": (
            pd.to_numeric(
                aptos_labels_raw[
                    aptos_grade_column
                ],
                errors="raise",
            )
            .astype(int)
        ),
    }
)


assert (
    len(aptos_labels)
    ==
    APTOS_EXPECTED_IMAGES
)


def standardise_idrid_labels(
    dataframe,
    official_partition,
):
    image_column = resolve_column(
        dataframe,
        [
            "Image name",
            "image_name",
            "image",
        ],
        "IDRiD image name",
    )

    grade_column = resolve_column(
        dataframe,
        [
            "Retinopathy grade",
            "retinopathy_grade",
            "DR grade",
            "dr_grade",
            "grade",
        ],
        "IDRiD retinopathy grade",
    )

    source_image_ids = (
        dataframe[
            image_column
        ]
        .map(
            normalise_image_id
        )
    )

    return pd.DataFrame(
        {
            "dataset": "IDRiD",
            "source_image_id": (
                source_image_ids
            ),
            "image_id": (
                official_partition
                +
                "__"
                +
                source_image_ids
            ),
            "original_dr_grade": (
                pd.to_numeric(
                    dataframe[
                        grade_column
                    ],
                    errors="raise",
                )
                .astype(int)
            ),
            "official_partition": (
                official_partition
            ),
        }
    )


idrid_train_labels = (
    standardise_idrid_labels(
        idrid_train_labels_raw,
        "development",
    )
)


idrid_test_labels = (
    standardise_idrid_labels(
        idrid_test_labels_raw,
        "sealed_evaluation",
    )
)


assert (
    len(
        idrid_train_labels
    )
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(
        idrid_test_labels
    )
    ==
    IDRID_EXPECTED_EVALUATION
)


idrid_labels = pd.concat(
    [
        idrid_train_labels,
        idrid_test_labels,
    ],
    ignore_index=True,
)


assert (
    len(idrid_labels)
    ==
    IDRID_EXPECTED_TOTAL
)

assert idrid_labels[
    "image_id"
].is_unique


# ============================================================
# 9. Join labels and image-integrity records
# ============================================================

aptos_verified = aptos_labels.merge(
    aptos_integrity_fixed,
    on=[
        "dataset",
        "image_id",
    ],
    how="inner",
    validate="one_to_one",
)


idrid_verified = idrid_labels.merge(
    idrid_integrity_fixed,
    on=[
        "dataset",
        "image_id",
        "source_image_id",
        "official_partition",
    ],
    how="inner",
    validate="one_to_one",
)


assert (
    len(aptos_verified)
    ==
    APTOS_EXPECTED_IMAGES
)

assert (
    len(idrid_verified)
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 10. Deterministic duplicate-aware APTOS split
# ============================================================

for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grade_conflicts = (
        aptos_verified
        .groupby(
            hash_column
        )[
            "original_dr_grade"
        ]
        .nunique()
    )

    assert not (
        grade_conflicts
        >
        1
    ).any(), (
        f"APTOS {hash_column} duplicates "
        "have conflicting labels."
    )


aptos_groups = (
    aptos_verified
    .groupby(
        "original_sha256",
        as_index=False,
    )
    .agg(
        original_dr_grade=(
            "original_dr_grade",
            "first",
        ),
        group_size=(
            "image_id",
            "size",
        ),
    )
    .sort_values(
        "original_sha256"
    )
    .reset_index(drop=True)
)


splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=(
        APTOS_EVALUATION_FRACTION
    ),
    random_state=(
        RANDOM_SEED
    ),
)


development_group_indices, evaluation_group_indices = next(
    splitter.split(
        np.zeros(
            len(
                aptos_groups
            )
        ),
        aptos_groups[
            "original_dr_grade"
        ].to_numpy(),
    )
)


evaluation_original_hashes = set(
    aptos_groups.iloc[
        evaluation_group_indices
    ][
        "original_sha256"
    ]
)


aptos_verified[
    "prospective_partition"
] = np.where(
    aptos_verified[
        "original_sha256"
    ].isin(
        evaluation_original_hashes
    ),
    "sealed_evaluation",
    "development",
)


canonical_partition_counts = (
    aptos_verified
    .groupby(
        "canonical_sha256"
    )[
        "prospective_partition"
    ]
    .nunique()
)


canonical_crossing_hashes = set(
    canonical_partition_counts[
        canonical_partition_counts
        >
        1
    ].index
)


if canonical_crossing_hashes:
    aptos_verified.loc[
        aptos_verified[
            "canonical_sha256"
        ].isin(
            canonical_crossing_hashes
        ),
        "prospective_partition",
    ] = "sealed_evaluation"


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    assert not (
        aptos_verified
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).any()


# ============================================================
# 11. Preserve official IDRiD split
# ============================================================

idrid_verified[
    "prospective_partition"
] = (
    idrid_verified[
        "official_partition"
    ]
)


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "development"
).sum() == IDRID_EXPECTED_DEVELOPMENT


assert (
    idrid_verified[
        "prospective_partition"
    ]
    ==
    "sealed_evaluation"
).sum() == IDRID_EXPECTED_EVALUATION


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grade_conflicts = (
        idrid_verified
        .groupby(
            hash_column
        )[
            "original_dr_grade"
        ]
        .nunique()
    )

    assert not (
        grade_conflicts
        >
        1
    ).any(), (
        f"IDRiD {hash_column} duplicates "
        "have conflicting grades."
    )

    partition_counts = (
        idrid_verified
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
    )

    assert not (
        partition_counts
        >
        1
    ).any(), (
        f"IDRiD {hash_column} duplicate leakage "
        "was detected across the official split."
    )


original_cross_dataset_overlap = set(
    aptos_verified[
        "original_sha256"
    ]
).intersection(
    set(
        idrid_verified[
            "original_sha256"
        ]
    )
)


canonical_cross_dataset_overlap = set(
    aptos_verified[
        "canonical_sha256"
    ]
).intersection(
    set(
        idrid_verified[
            "canonical_sha256"
        ]
    )
)


assert not (
    original_cross_dataset_overlap
)

assert not (
    canonical_cross_dataset_overlap
)


# ============================================================
# 12. Harmonised endpoint
# ============================================================

for dataframe in [
    aptos_verified,
    idrid_verified,
]:
    dataframe[
        "moderate_or_worse_dr"
    ] = (
        dataframe[
            "original_dr_grade"
        ]
        >=
        2
    ).astype(int)


# ============================================================
# 13. Canonical storage audit
# ============================================================

aptos_canonical_bytes = (
    directory_size_bytes(
        LOCAL_APTOS_CANONICAL
    )
)

idrid_canonical_bytes = (
    directory_size_bytes(
        LOCAL_IDRID_CANONICAL
    )
)

canonical_total_bytes = (
    aptos_canonical_bytes
    +
    idrid_canonical_bytes
)


current_drive_free = (
    shutil.disk_usage(
        DRIVE_ROOT
    ).free
)


print(
    "\n================ CANONICAL STORAGE AUDIT "
    "================"
)

print(
    "APTOS canonical size:",
    f"{bytes_to_gib(aptos_canonical_bytes):.2f} GiB",
)

print(
    "IDRiD canonical size:",
    f"{bytes_to_gib(idrid_canonical_bytes):.2f} GiB",
)

print(
    "Total canonical size:",
    f"{bytes_to_gib(canonical_total_bytes):.2f} GiB",
)

print(
    "Current Drive free space:",
    f"{bytes_to_gib(current_drive_free):.2f} GiB",
)


assert (
    current_drive_free
    >
    canonical_total_bytes
    +
    512
    *
    1024 ** 2
), (
    "Canonical images still exceed the remaining "
    "Google Drive capacity. Nothing has yet been "
    "copied into the final canonical directories."
)


# ============================================================
# 14. Copy canonical sets into Drive staging
# ============================================================

APTOS_STAGING_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019__STAGING"
)

IDRID_STAGING_ROOT = (
    CANONICAL_ROOT
    / "IDRiD__STAGING"
)


for staging_root in [
    APTOS_STAGING_ROOT,
    IDRID_STAGING_ROOT,
]:
    if staging_root.exists():
        shutil.rmtree(
            staging_root
        )


print(
    "\nCopying canonical images to "
    "Google Drive staging directories..."
)


shutil.copytree(
    LOCAL_APTOS_CANONICAL,
    APTOS_STAGING_ROOT,
)


shutil.copytree(
    LOCAL_IDRID_CANONICAL,
    IDRID_STAGING_ROOT,
)


assert (
    len(
        list(
            APTOS_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    APTOS_EXPECTED_IMAGES
)


assert (
    len(
        list(
            IDRID_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 15. Verify Drive copies
# ============================================================

print(
    "\n================ DRIVE COPY VERIFICATION "
    "================"
)


for _, row in tqdm(
    aptos_integrity_fixed.iterrows(),
    total=len(
        aptos_integrity_fixed
    ),
    desc="Verifying APTOS Drive hashes",
):
    copied_path = (
        APTOS_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


for _, row in tqdm(
    idrid_integrity_fixed.iterrows(),
    total=len(
        idrid_integrity_fixed
    ),
    desc="Verifying IDRiD Drive hashes",
):
    copied_path = (
        IDRID_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


if APTOS_CANONICAL_ROOT.exists():
    shutil.rmtree(
        APTOS_CANONICAL_ROOT
    )


if IDRID_CANONICAL_ROOT.exists():
    shutil.rmtree(
        IDRID_CANONICAL_ROOT
    )


APTOS_STAGING_ROOT.rename(
    APTOS_CANONICAL_ROOT
)

IDRID_STAGING_ROOT.rename(
    IDRID_CANONICAL_ROOT
)


# ============================================================
# 16. Attach final Drive paths
# ============================================================

aptos_verified[
    "canonical_image_path"
] = (
    aptos_verified[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            APTOS_CANONICAL_ROOT
            /
            f"{image_id}.jpg"
        )
    )
)


idrid_verified[
    "canonical_image_path"
] = (
    idrid_verified[
        "image_id"
    ]
    .map(
        lambda image_id: str(
            IDRID_CANONICAL_ROOT
            /
            f"{image_id}.jpg"
        )
    )
)


assert aptos_verified[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


assert idrid_verified[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


# ============================================================
# 17. Construct partition tables
# ============================================================

aptos_development = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


aptos_evaluation = (
    aptos_verified[
        aptos_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_development = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_evaluation = (
    idrid_verified[
        idrid_verified[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


assert (
    len(idrid_development)
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(idrid_evaluation)
    ==
    IDRID_EXPECTED_EVALUATION
)


# ============================================================
# 18. Save development and sealed artifacts
# ============================================================

APTOS_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "APTOS_2019_Development_Labels_v0.1.csv"
)

IDRID_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "IDRiD_Development_Labels_v0.1.csv"
)

APTOS_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv"
)

IDRID_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv"
)

APTOS_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "APTOS_2019_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

IDRID_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "IDRiD_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)


for output_path in [
    APTOS_DEVELOPMENT_LABEL_PATH,
    IDRID_DEVELOPMENT_LABEL_PATH,
    APTOS_EVALUATION_MANIFEST_PATH,
    IDRID_EVALUATION_MANIFEST_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_writable(
        output_path
    )


development_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_dr_grade",
    "moderate_or_worse_dr",
    "original_sha256",
    "canonical_sha256",
]


evaluation_manifest_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_sha256",
    "canonical_sha256",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


sealed_label_columns = [
    "dataset",
    "image_id",
    "original_dr_grade",
    "moderate_or_worse_dr",
]


aptos_development[
    development_columns
].to_csv(
    APTOS_DEVELOPMENT_LABEL_PATH,
    index=False,
)


idrid_development[
    development_columns
].to_csv(
    IDRID_DEVELOPMENT_LABEL_PATH,
    index=False,
)


aptos_evaluation[
    evaluation_manifest_columns
].to_csv(
    APTOS_EVALUATION_MANIFEST_PATH,
    index=False,
)


idrid_evaluation[
    evaluation_manifest_columns
].to_csv(
    IDRID_EVALUATION_MANIFEST_PATH,
    index=False,
)


aptos_evaluation[
    sealed_label_columns
].to_csv(
    APTOS_SEALED_LABEL_PATH,
    index=False,
)


idrid_evaluation[
    sealed_label_columns
].to_csv(
    IDRID_SEALED_LABEL_PATH,
    index=False,
)


for protected_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_read_only(
        protected_path
    )


# ============================================================
# 19. Label-free master manifest
# ============================================================

label_free_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "prospective_partition",
    "original_sha256",
    "canonical_sha256",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


label_free_master_manifest = pd.concat(
    [
        aptos_verified[
            label_free_columns
        ],
        idrid_verified[
            label_free_columns
        ],
    ],
    ignore_index=True,
)


assert (
    "original_dr_grade"
    not in
    label_free_master_manifest.columns
)

assert (
    "moderate_or_worse_dr"
    not in
    label_free_master_manifest.columns
)


# ============================================================
# 20. Save protocol, integrity and decision artifacts
# ============================================================

CANONICAL_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Canonical_Fundus_Preprocessing_Protocol_v0.1.json"
)

QUARANTINE_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Target_Label_Quarantine_Protocol_v0.1.json"
)

MASTER_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_LabelFree_Master_Target_Manifest_v0.1.csv"
)

INTEGRITY_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

PARTITION_SUMMARY_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Partition_Summary_v0.1.csv"
)

LABEL_COMMITMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Sealed_Label_Commitments_v0.1.json"
)

ID_REPAIR_PATH = (
    STAGE2_ROOT
    / "Stage2_IDRiD_PartitionQualified_ID_Repair_v0.1.json"
)

STAGE2_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)

STAGE2_REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Report_v0.1.md"
)

STAGE2_ENVIRONMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Environment_v0.1.json"
)


canonical_protocol = {
    "protocol_version": "v0.1",
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "black_border_threshold": (
        BLACK_BORDER_THRESHOLD
    ),
    "jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "jpeg_subsampling": (
        CANONICAL_JPEG_SUBSAMPLING
    ),
    "resize_filter": "LANCZOS",
    "frozen_before_target_performance": True,
    "must_be_applied_to_sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
}


with open(
    CANONICAL_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        canonical_protocol,
        file,
        indent=2,
    )


quarantine_protocol = {
    "protocol_version": "v0.1",
    "endpoint_id": ENDPOINT_ID,
    "random_seed": RANDOM_SEED,
    "aptos_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "development_labels_authorised_from_stage": 3,
    "sealed_evaluation_labels_authorised_stage": 6,
    "sealed_label_values_displayed": False,
    "idrid_unique_id_rule": (
        "<official_partition>__<official_image_stem>"
    ),
}


with open(
    QUARANTINE_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quarantine_protocol,
        file,
        indent=2,
    )


label_free_master_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False,
)


integrity_manifest = pd.concat(
    [
        aptos_integrity_fixed,
        idrid_integrity_fixed,
    ],
    ignore_index=True,
)


integrity_manifest.to_csv(
    INTEGRITY_MANIFEST_PATH,
    index=False,
)


partition_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "total_images": int(
                len(
                    aptos_verified
                )
            ),
            "development_images": int(
                len(
                    aptos_development
                )
            ),
            "sealed_evaluation_images": int(
                len(
                    aptos_evaluation
                )
            ),
            "partition_source": (
                "original-hash-grouped deterministic "
                "grade-stratified split"
            ),
            "sealed_evaluation_labels_displayed": False,
        },
        {
            "dataset": "IDRiD",
            "total_images": int(
                len(
                    idrid_verified
                )
            ),
            "development_images": int(
                len(
                    idrid_development
                )
            ),
            "sealed_evaluation_images": int(
                len(
                    idrid_evaluation
                )
            ),
            "partition_source": (
                "official IDRiD training/testing split"
            ),
            "sealed_evaluation_labels_displayed": False,
        },
    ]
)


partition_summary.to_csv(
    PARTITION_SUMMARY_PATH,
    index=False,
)


label_commitments = {
    "endpoint_id": ENDPOINT_ID,
    "aptos_sealed_evaluation": {
        "rows": int(
            len(
                aptos_evaluation
            )
        ),
        "path": str(
            APTOS_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            APTOS_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "idrid_sealed_evaluation": {
        "rows": int(
            len(
                idrid_evaluation
            )
        ),
        "path": str(
            IDRID_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
}


with open(
    LABEL_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_commitments,
        file,
        indent=2,
    )


id_repair_payload = {
    "previous_failure": (
        "IDRID_IMAGE_ID_NOT_UNIQUE"
    ),
    "technical_cause": (
        "Official IDRiD training and testing folders "
        "reuse bare image stems such as IDRiD_001."
    ),
    "bare_unique_stems": int(
        idrid_image_rows[
            "source_image_id"
        ].nunique()
    ),
    "total_files": int(
        len(
            idrid_image_rows
        )
    ),
    "reused_bare_stems": int(
        len(
            duplicated_source_ids
        )
    ),
    "corrected_unique_id_rule": (
        "<official_partition>__<official_image_stem>"
    ),
    "corrected_ids_unique": bool(
        idrid_image_rows[
            "image_id"
        ].is_unique
    ),
    "computer_sleep_or_lock_was_cause": False,
}


with open(
    ID_REPAIR_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        id_repair_payload,
        file,
        indent=2,
    )


# ============================================================
# 21. Leakage checks
# ============================================================

aptos_original_cross_split_duplicates = int(
    (
        aptos_verified
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


aptos_canonical_cross_split_duplicates = int(
    (
        aptos_verified
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_original_cross_split_duplicates = int(
    (
        idrid_verified
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_canonical_cross_split_duplicates = int(
    (
        idrid_verified
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


evaluation_manifest_has_no_labels = bool(
    all(
        label_column
        not in pd.read_csv(
            manifest_path,
            nrows=1,
        ).columns
        for manifest_path in [
            APTOS_EVALUATION_MANIFEST_PATH,
            IDRID_EVALUATION_MANIFEST_PATH,
        ]
        for label_column in [
            "original_dr_grade",
            "moderate_or_worse_dr",
        ]
    )
)


assert (
    aptos_original_cross_split_duplicates
    ==
    0
)

assert (
    aptos_canonical_cross_split_duplicates
    ==
    0
)

assert (
    idrid_original_cross_split_duplicates
    ==
    0
)

assert (
    idrid_canonical_cross_split_duplicates
    ==
    0
)

assert not (
    original_cross_dataset_overlap
)

assert not (
    canonical_cross_dataset_overlap
)

assert (
    evaluation_manifest_has_no_labels
)


# ============================================================
# 22. Final Stage 2R decision
# ============================================================

stage2_decision = (
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_"
    "AND_LABEL_QUARANTINE_ADVANCE_TO_"
    "DEVELOPMENT_RECOVERABILITY_GATE"
)


stage2_interpretation = (
    "APTOS and IDRiD were converted using the identical "
    "frozen 768x768 canonicalisation and hash-verified after "
    "copying to Google Drive. The earlier IDRiD failure was "
    "caused by official training and testing folders reusing "
    "bare image filenames; partition-qualified identifiers "
    "now preserve all 516 images without collision. APTOS was "
    "deterministically partitioned while IDRiD retained its "
    "official split. Sealed labels remain absent from modelling "
    "manifests and committed for one-time Stage 6 evaluation."
)


decision_payload = {
    "decision": stage2_decision,
    "interpretation": (
        stage2_interpretation
    ),
    "authorised_next_step": (
        "DEVELOPMENT_ONLY_TARGET_RECOVERABILITY_GATE"
    ),
    "endpoint_id": ENDPOINT_ID,
    "idrid_identifier_repair": str(
        ID_REPAIR_PATH
    ),
    "aptos": {
        "total": int(
            len(
                aptos_verified
            )
        ),
        "development": int(
            len(
                aptos_development
            )
        ),
        "sealed_evaluation": int(
            len(
                aptos_evaluation
            )
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                aptos_canonical_bytes
            )
        ),
    },
    "idrid": {
        "total": int(
            len(
                idrid_verified
            )
        ),
        "development": int(
            len(
                idrid_development
            )
        ),
        "sealed_evaluation": int(
            len(
                idrid_evaluation
            )
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                idrid_canonical_bytes
            )
        ),
        "partition_qualified_ids": True,
    },
    "evaluation_manifests_label_free": True,
    "sealed_label_values_displayed": False,
    "important_boundary": (
        "Stage 2 establishes controlled acquisition, "
        "canonicalisation, partitioning, and procedural "
        "label quarantine only. It does not establish target "
        "recoverability, transfer performance, or CDO validity."
    ),
}


with open(
    STAGE2_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
        default=json_default,
    )


STAGE2_REPORT_PATH.write_text(
    "\n".join(
        [
            (
                "# Prospective Retinal Blind Test — "
                "Stage 2R"
            ),
            "",
            f"Decision: `{stage2_decision}`",
            "",
            stage2_interpretation,
            "",
            (
                "Sealed-evaluation label values "
                "displayed: `False`"
            ),
            (
                "Evaluation manifests are "
                "label-free: `True`"
            ),
        ]
    ),
    encoding="utf-8",
)


with open(
    STAGE2_ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "timestamp_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "python": sys.version,
            "platform": platform.platform(),
            "numpy": np.__version__,
            "pandas": pd.__version__,
            "repair": (
                "PARTITION_QUALIFIED_IDRID_IDENTIFIERS"
            ),
        },
        file,
        indent=2,
    )


# ============================================================
# 23. Prepare final summaries before cleanup
# ============================================================

canonical_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "images": int(
                len(
                    aptos_integrity_fixed
                )
            ),
            "unique_image_ids": int(
                aptos_integrity_fixed[
                    "image_id"
                ].nunique()
            ),
            "unique_original_sha256": int(
                aptos_integrity_fixed[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                aptos_integrity_fixed[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    aptos_canonical_bytes
                )
            ),
        },
        {
            "dataset": "IDRiD",
            "images": int(
                len(
                    idrid_integrity_fixed
                )
            ),
            "unique_image_ids": int(
                idrid_integrity_fixed[
                    "image_id"
                ].nunique()
            ),
            "unique_original_sha256": int(
                idrid_integrity_fixed[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                idrid_integrity_fixed[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    idrid_canonical_bytes
                )
            ),
        },
    ]
)


# ============================================================
# 24. Cleanup temporary ZIP and local staging
# ============================================================

print(
    "\nDeleting temporary APTOS ZIP and local "
    "canonical working files..."
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


# ============================================================
# 25. Final display
# ============================================================

print(
    "\n================ CANONICAL IMAGE SUMMARY "
    "================"
)

display(
    canonical_summary
)


print(
    "\n================ PARTITION SUMMARY "
    "================"
)

display(
    partition_summary
)


print(
    "\n================ LEAKAGE CHECK "
    "================"
)

print(
    "APTOS original-hash cross-split duplicates:",
    aptos_original_cross_split_duplicates,
)

print(
    "APTOS canonical-hash cross-split duplicates:",
    aptos_canonical_cross_split_duplicates,
)

print(
    "IDRiD original-hash cross-split duplicates:",
    idrid_original_cross_split_duplicates,
)

print(
    "IDRiD canonical-hash cross-split duplicates:",
    idrid_canonical_cross_split_duplicates,
)

print(
    "Original cross-dataset overlap:",
    len(
        original_cross_dataset_overlap
    ),
)

print(
    "Canonical cross-dataset overlap:",
    len(
        canonical_cross_dataset_overlap
    ),
)

print(
    "Evaluation manifests contain labels:",
    not evaluation_manifest_has_no_labels,
)


print(
    "\n================ LABEL QUARANTINE CHECK "
    "================"
)

print(
    "APTOS sealed-label SHA-256:",
    label_commitments[
        "aptos_sealed_evaluation"
    ]["sha256"],
)

print(
    "IDRiD sealed-label SHA-256:",
    label_commitments[
        "idrid_sealed_evaluation"
    ]["sha256"],
)

print(
    "Sealed-evaluation label values displayed:",
    False,
)


print(
    "\n================ STAGE 2R DECISION "
    "================"
)

print("Decision:")
print(
    stage2_decision
)

print("\nInterpretation:")
print(
    stage2_interpretation
)

print("\nAuthorised next step:")
print(
    decision_payload[
        "authorised_next_step"
    ]
)

print(
    "\nComputer sleep or automatic lock "
    "caused the earlier failure:"
)

print(False)

print(
    "\nTemporary APTOS archive retained:"
)

print(False)

print(
    "\nStage 2R ID repair and continuation "
    "completed and sealed."
)

================ STAGE 2R IDRiD-ID REPAIR ================

Repair boundary:
APTOS canonical images already completed in the current runtime are reused. Only IDRiD is canonicalised again, using partition-qualified unique image identifiers. Sealed-evaluation label values must not be displayed, summarised, or inserted into modelling manifests.

Completed APTOS canonical images reused:
3662

APTOS ZIP retained:
/content/cdo_stage2r_storage_efficient/download/aptos2019-blindness-detection.zip

APTOS integrity table recovered from the current notebook memory.

================ IDRiD DUPLICATE-ID DIAGNOSIS ================
Total IDRiD image files: 516
Unique bare filename stems: 413
Bare filename stems reused across folders: 103
Partition-qualified IDs unique: True


,source_image_id,files,partitions
0,IDRiD_001,2,"development, sealed_evaluation"
1,IDRiD_002,2,"development, sealed_evaluation"
2,IDRiD_003,2,"development, sealed_evaluation"
3,IDRiD_004,2,"development, sealed_evaluation"
4,IDRiD_005,2,"development, sealed_evaluation"
5,IDRiD_006,2,"development, sealed_evaluation"
6,IDRiD_007,2,"development, sealed_evaluation"
7,IDRiD_008,2,"development, sealed_evaluation"
8,IDRiD_009,2,"development, sealed_evaluation"
9,IDRiD_010,2,"development, sealed_evaluation"



================ IDRiD RE-CANONICALISATION ================


Re-canonicalising IDRiD with partition IDs:   0%|          | 0/516 [00:00<?, ?it/s]

AssertionError: APTOS original_sha256 duplicates have conflicting labels.

In [18]:
#@title 02R-F. Audit APTOS duplicate-label conflicts and freeze deterministic exclusion rule

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen data-integrity rule
# ============================================================

MINIMUM_RETAINED_FRACTION = 0.95
MINIMUM_IMAGES_PER_ORIGINAL_GRADE = 50

INTEGRITY_RULE = (
    "Construct connected components using exact original-image "
    "SHA-256 equality and exact canonical-image SHA-256 equality. "
    "If any component contains more than one official DR grade, "
    "exclude the entire component from development and sealed "
    "evaluation. Label-consistent duplicate components remain "
    "eligible but must subsequently be assigned wholly to one "
    "partition."
)

GATE_RULE = (
    "Proceed only if at least 95% of APTOS images remain after "
    "deterministic exclusion and every original grade retains "
    "at least 50 images."
)


# ============================================================
# 1. Verify current runtime state
# ============================================================

required_runtime_variables = [
    "aptos_verified",
    "aptos_integrity_fixed",
    "idrid_verified",
    "idrid_integrity_fixed",
    "LOCAL_APTOS_CANONICAL",
    "LOCAL_IDRID_CANONICAL",
    "archive_path",
]


missing_runtime_variables = [
    variable_name
    for variable_name in required_runtime_variables
    if variable_name not in globals()
]


if missing_runtime_variables:
    raise RuntimeError(
        "The required in-memory Stage 2R objects are missing:\n"
        +
        "\n".join(
            missing_runtime_variables
        )
        +
        "\n\nThe Colab runtime may have restarted. "
        "Do not run an earlier partial cell blindly."
    )


assert isinstance(
    aptos_verified,
    pd.DataFrame,
)

assert len(
    aptos_verified
) == 3662


required_aptos_columns = {
    "dataset",
    "image_id",
    "original_dr_grade",
    "original_sha256",
    "canonical_sha256",
}


assert required_aptos_columns.issubset(
    aptos_verified.columns
), (
    "Missing APTOS columns: "
    f"{sorted(required_aptos_columns - set(aptos_verified.columns))}"
)


assert aptos_verified[
    "image_id"
].is_unique


assert aptos_verified[
    "original_dr_grade"
].isin(
    [
        0,
        1,
        2,
        3,
        4,
    ]
).all()


print(
    "================ STAGE 2R-F "
    "APTOS CONFLICT AUDIT ================"
)

print("\nFrozen integrity rule:")
print(INTEGRITY_RULE)

print("\nFrozen continuation gate:")
print(GATE_RULE)

print("\nAPTOS rows available:")
print(
    len(
        aptos_verified
    )
)

print("\nNo model or transfer performance has been evaluated.")


# ============================================================
# 2. Resolve project output paths
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

TEST_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

EXCLUSION_QUARANTINE_ROOT = (
    QUARANTINE_ROOT
    / "02_Excluded_Ambiguous_Duplicate_Components"
)


STAGE2_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

EXCLUSION_QUARANTINE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. Freeze working table
# ============================================================

audit_table = (
    aptos_verified[
        [
            "dataset",
            "image_id",
            "original_dr_grade",
            "original_sha256",
            "canonical_sha256",
        ]
    ]
    .copy()
    .sort_values(
        "image_id"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 4. Direct duplicate-group diagnostics
# ============================================================

original_group_summary = (
    audit_table
    .groupby(
        "original_sha256",
        as_index=False,
    )
    .agg(
        image_count=(
            "image_id",
            "size",
        ),
        grade_count=(
            "original_dr_grade",
            "nunique",
        ),
        grade_set=(
            "original_dr_grade",
            lambda values: (
                "|".join(
                    map(
                        str,
                        sorted(
                            set(
                                map(
                                    int,
                                    values,
                                )
                            )
                        ),
                    )
                )
            ),
        ),
    )
)


canonical_group_summary = (
    audit_table
    .groupby(
        "canonical_sha256",
        as_index=False,
    )
    .agg(
        image_count=(
            "image_id",
            "size",
        ),
        grade_count=(
            "original_dr_grade",
            "nunique",
        ),
        grade_set=(
            "original_dr_grade",
            lambda values: (
                "|".join(
                    map(
                        str,
                        sorted(
                            set(
                                map(
                                    int,
                                    values,
                                )
                            )
                        ),
                    )
                )
            ),
        ),
    )
)


original_duplicate_groups = (
    original_group_summary[
        original_group_summary[
            "image_count"
        ]
        >
        1
    ]
)


canonical_duplicate_groups = (
    canonical_group_summary[
        canonical_group_summary[
            "image_count"
        ]
        >
        1
    ]
)


original_conflicting_groups = (
    original_group_summary[
        original_group_summary[
            "grade_count"
        ]
        >
        1
    ]
)


canonical_conflicting_groups = (
    canonical_group_summary[
        canonical_group_summary[
            "grade_count"
        ]
        >
        1
    ]
)


# ============================================================
# 5. Build connected components across both hash types
# ============================================================

number_of_rows = len(
    audit_table
)

parent = np.arange(
    number_of_rows,
    dtype=np.int64,
)

rank = np.zeros(
    number_of_rows,
    dtype=np.int64,
)


def find_root(index):
    index = int(index)

    while parent[index] != index:
        parent[index] = parent[
            parent[index]
        ]

        index = int(
            parent[index]
        )

    return index


def union_indices(
    left_index,
    right_index,
):
    left_root = find_root(
        left_index
    )

    right_root = find_root(
        right_index
    )

    if left_root == right_root:
        return

    if rank[left_root] < rank[right_root]:
        parent[left_root] = right_root

    elif rank[left_root] > rank[right_root]:
        parent[right_root] = left_root

    else:
        parent[right_root] = left_root
        rank[left_root] += 1


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grouped_indices = (
        audit_table
        .groupby(
            hash_column,
            sort=False,
        )
        .indices
    )

    for row_indices in grouped_indices.values():
        row_indices = list(
            map(
                int,
                row_indices,
            )
        )

        if len(row_indices) <= 1:
            continue

        anchor = row_indices[0]

        for other_index in row_indices[1:]:
            union_indices(
                anchor,
                other_index,
            )


component_roots = [
    find_root(index)
    for index in range(
        number_of_rows
    )
]


root_to_row_indices = {}


for row_index, root in enumerate(
    component_roots
):
    root_to_row_indices.setdefault(
        int(root),
        [],
    ).append(
        int(row_index)
    )


root_to_component_id = {}


for root, row_indices in root_to_row_indices.items():
    sorted_image_ids = sorted(
        audit_table.iloc[
            row_indices
        ][
            "image_id"
        ].astype(str)
    )

    component_digest = hashlib.sha256(
        "|".join(
            sorted_image_ids
        ).encode(
            "utf-8"
        )
    ).hexdigest()[:16]

    root_to_component_id[root] = (
        "APTOS_COMPONENT_"
        +
        component_digest
    )


audit_table[
    "duplicate_component_id"
] = [
    root_to_component_id[
        int(root)
    ]
    for root in component_roots
]


component_summary = (
    audit_table
    .groupby(
        "duplicate_component_id",
        as_index=False,
    )
    .agg(
        component_images=(
            "image_id",
            "size",
        ),
        original_hashes=(
            "original_sha256",
            "nunique",
        ),
        canonical_hashes=(
            "canonical_sha256",
            "nunique",
        ),
        grade_count=(
            "original_dr_grade",
            "nunique",
        ),
        grade_set=(
            "original_dr_grade",
            lambda values: (
                "|".join(
                    map(
                        str,
                        sorted(
                            set(
                                map(
                                    int,
                                    values,
                                )
                            )
                        ),
                    )
                )
            ),
        ),
    )
)


component_summary[
    "ambiguous_label_component"
] = (
    component_summary[
        "grade_count"
    ]
    >
    1
)


ambiguous_component_ids = set(
    component_summary.loc[
        component_summary[
            "ambiguous_label_component"
        ],
        "duplicate_component_id",
    ]
)


audit_table[
    "excluded_due_to_ambiguous_duplicate_labels"
] = (
    audit_table[
        "duplicate_component_id"
    ].isin(
        ambiguous_component_ids
    )
)


audit_table[
    "eligible_after_integrity_audit"
] = ~(
    audit_table[
        "excluded_due_to_ambiguous_duplicate_labels"
    ]
)


# ============================================================
# 6. Apply frozen continuation gate
# ============================================================

excluded_rows = audit_table[
    audit_table[
        "excluded_due_to_ambiguous_duplicate_labels"
    ]
].copy()


eligible_rows = audit_table[
    audit_table[
        "eligible_after_integrity_audit"
    ]
].copy()


excluded_image_count = int(
    len(
        excluded_rows
    )
)

retained_image_count = int(
    len(
        eligible_rows
    )
)

retained_fraction = float(
    retained_image_count
    /
    len(
        audit_table
    )
)


grade_retention = (
    audit_table
    .groupby(
        "original_dr_grade"
    )
    .agg(
        original_images=(
            "image_id",
            "size",
        ),
        excluded_images=(
            "excluded_due_to_ambiguous_duplicate_labels",
            "sum",
        ),
        retained_images=(
            "eligible_after_integrity_audit",
            "sum",
        ),
    )
    .reset_index()
)


minimum_retained_grade_count = int(
    grade_retention[
        "retained_images"
    ].min()
)


retention_gate_pass = bool(
    retained_fraction
    >=
    MINIMUM_RETAINED_FRACTION
)


grade_count_gate_pass = bool(
    minimum_retained_grade_count
    >=
    MINIMUM_IMAGES_PER_ORIGINAL_GRADE
)


if (
    retention_gate_pass
    and
    grade_count_gate_pass
):
    stage2f_decision = (
        "PASS_EXCLUDE_AMBIGUOUS_DUPLICATE_COMPONENTS_"
        "AND_COMPLETE_STAGE2R"
    )

    authorised_next_step = (
        "COMPLETE_STAGE2R_USING_ONLY_"
        "LABEL_CONSISTENT_COMPONENTS"
    )

    interpretation = (
        "APTOS contains exact duplicate-image components with "
        "discordant official grades. The frozen deterministic "
        "rule excludes every connected component containing "
        "more than one grade. The remaining cohort satisfies "
        "the pre-frozen retention and per-grade sample-size "
        "gates, so Stage 2R may continue without using the "
        "ambiguous images."
    )

else:
    stage2f_decision = (
        "HOLD_APTOS_LABEL_INCONSISTENCY_EXCEEDS_"
        "FROZEN_INTEGRITY_GATE"
    )

    authorised_next_step = (
        "DO_NOT_COMPLETE_STAGE2R_REVIEW_DATASET_INTEGRITY"
    )

    interpretation = (
        "The deterministic exclusion required for discordant "
        "duplicate-image components would leave too little of "
        "the cohort or too few images in at least one grade. "
        "Stage 2R remains on hold."
    )


# ============================================================
# 7. Save audit artifacts
# ============================================================

FULL_LABELLED_CONFLICT_PATH = (
    EXCLUSION_QUARANTINE_ROOT
    / "APTOS_Ambiguous_Duplicate_Components_"
      "WITH_LABELS_DO_NOT_USE_v0.1.csv"
)

LABEL_FREE_EXCLUSION_PATH = (
    STAGE2_ROOT
    / "Stage2F_APTOS_Ambiguous_Duplicate_"
      "Exclusion_Manifest_v0.1.csv"
)

COMPONENT_SUMMARY_PATH = (
    EXCLUSION_QUARANTINE_ROOT
    / "APTOS_Duplicate_Component_Summary_"
      "WITH_GRADE_SETS_v0.1.csv"
)

GRADE_RETENTION_PATH = (
    STAGE2_ROOT
    / "Stage2F_APTOS_Grade_Retention_After_"
      "Duplicate_Exclusion_v0.1.csv"
)

DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2F_APTOS_Duplicate_Label_"
      "Conflict_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2F_APTOS_Duplicate_Label_"
      "Conflict_Report_v0.1.md"
)


excluded_rows.to_csv(
    FULL_LABELLED_CONFLICT_PATH,
    index=False,
)


excluded_rows[
    [
        "dataset",
        "image_id",
        "original_sha256",
        "canonical_sha256",
        "duplicate_component_id",
        "excluded_due_to_ambiguous_duplicate_labels",
    ]
].to_csv(
    LABEL_FREE_EXCLUSION_PATH,
    index=False,
)


component_summary.to_csv(
    COMPONENT_SUMMARY_PATH,
    index=False,
)


grade_retention.to_csv(
    GRADE_RETENTION_PATH,
    index=False,
)


decision_payload = {
    "decision": stage2f_decision,
    "interpretation": interpretation,
    "authorised_next_step": (
        authorised_next_step
    ),
    "integrity_rule": INTEGRITY_RULE,
    "gate_rule": GATE_RULE,
    "total_aptos_images": int(
        len(
            audit_table
        )
    ),
    "original_duplicate_groups": int(
        len(
            original_duplicate_groups
        )
    ),
    "original_conflicting_groups": int(
        len(
            original_conflicting_groups
        )
    ),
    "canonical_duplicate_groups": int(
        len(
            canonical_duplicate_groups
        )
    ),
    "canonical_conflicting_groups": int(
        len(
            canonical_conflicting_groups
        )
    ),
    "connected_components": int(
        len(
            component_summary
        )
    ),
    "ambiguous_connected_components": int(
        len(
            ambiguous_component_ids
        )
    ),
    "excluded_images": (
        excluded_image_count
    ),
    "retained_images": (
        retained_image_count
    ),
    "retained_fraction": (
        retained_fraction
    ),
    "minimum_retained_grade_count": (
        minimum_retained_grade_count
    ),
    "retention_gate_pass": (
        retention_gate_pass
    ),
    "grade_count_gate_pass": (
        grade_count_gate_pass
    ),
    "model_performance_observed": False,
    "sealed_evaluation_performance_observed": False,
    "individual_conflicting_labels_displayed": False,
    "full_labelled_conflict_table": str(
        FULL_LABELLED_CONFLICT_PATH
    ),
    "label_free_exclusion_manifest": str(
        LABEL_FREE_EXCLUSION_PATH
    ),
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# APTOS Duplicate-Label Conflict Audit",
    "",
    f"Decision: `{stage2f_decision}`",
    "",
    interpretation,
    "",
    "## Frozen rule",
    "",
    INTEGRITY_RULE,
    "",
    "## Cohort impact",
    "",
    (
        f"- Original images: "
        f"{len(audit_table)}"
    ),
    (
        f"- Ambiguous connected components: "
        f"{len(ambiguous_component_ids)}"
    ),
    (
        f"- Excluded images: "
        f"{excluded_image_count}"
    ),
    (
        f"- Retained images: "
        f"{retained_image_count}"
    ),
    (
        f"- Retained fraction: "
        f"{retained_fraction:.6f}"
    ),
    "",
    "## Performance boundary",
    "",
    "- Model performance observed: `False`",
    (
        "- Sealed-evaluation performance "
        "observed: `False`"
    ),
    "",
]


REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


for protected_path in [
    FULL_LABELLED_CONFLICT_PATH,
    COMPONENT_SUMMARY_PATH,
]:
    try:
        os.chmod(
            protected_path,
            0o400,
        )
    except Exception:
        pass


# ============================================================
# 8. Keep repaired tables available for the next cell
# ============================================================

aptos_duplicate_component_audit = (
    audit_table
)

aptos_ambiguous_component_ids = (
    ambiguous_component_ids
)

aptos_eligible_image_ids = set(
    eligible_rows[
        "image_id"
    ]
)

aptos_excluded_image_ids = set(
    excluded_rows[
        "image_id"
    ]
)


# ============================================================
# 9. Display aggregate results only
# ============================================================

summary_table = pd.DataFrame(
    [
        {
            "metric": (
                "Original-hash duplicate groups"
            ),
            "value": int(
                len(
                    original_duplicate_groups
                )
            ),
        },
        {
            "metric": (
                "Original-hash conflicting groups"
            ),
            "value": int(
                len(
                    original_conflicting_groups
                )
            ),
        },
        {
            "metric": (
                "Canonical-hash duplicate groups"
            ),
            "value": int(
                len(
                    canonical_duplicate_groups
                )
            ),
        },
        {
            "metric": (
                "Canonical-hash conflicting groups"
            ),
            "value": int(
                len(
                    canonical_conflicting_groups
                )
            ),
        },
        {
            "metric": (
                "Ambiguous connected components"
            ),
            "value": int(
                len(
                    ambiguous_component_ids
                )
            ),
        },
        {
            "metric": (
                "Images excluded"
            ),
            "value": (
                excluded_image_count
            ),
        },
        {
            "metric": (
                "Images retained"
            ),
            "value": (
                retained_image_count
            ),
        },
        {
            "metric": (
                "Retained fraction"
            ),
            "value": (
                retained_fraction
            ),
        },
    ]
)


print(
    "\n================ APTOS DUPLICATE "
    "CONFLICT SUMMARY ================"
)

display(
    summary_table
)


print(
    "\n================ GRADE RETENTION AFTER "
    "DETERMINISTIC EXCLUSION ================"
)

display(
    grade_retention
)


ambiguous_grade_set_summary = (
    component_summary[
        component_summary[
            "ambiguous_label_component"
        ]
    ]
    .groupby(
        "grade_set"
    )
    .agg(
        components=(
            "duplicate_component_id",
            "size",
        ),
        images=(
            "component_images",
            "sum",
        ),
    )
    .reset_index()
    .sort_values(
        [
            "images",
            "components",
        ],
        ascending=False,
    )
)


print(
    "\n================ AMBIGUOUS GRADE-SET "
    "SUMMARY ================"
)

display(
    ambiguous_grade_set_summary
)


print(
    "\n================ STAGE 2R-F DECISION "
    "================"
)

print("Decision:")
print(
    stage2f_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nIndividual conflicting rows displayed:")
print(False)

print("\nModel performance observed:")
print(False)

print("\nSealed evaluation performance observed:")
print(False)

print(
    "\nAPTOS duplicate-label conflict audit "
    "completed and sealed."
)

================ STAGE 2R-F APTOS CONFLICT AUDIT ================

Frozen integrity rule:
Construct connected components using exact original-image SHA-256 equality and exact canonical-image SHA-256 equality. If any component contains more than one official DR grade, exclude the entire component from development and sealed evaluation. Label-consistent duplicate components remain eligible but must subsequently be assigned wholly to one partition.

Frozen continuation gate:
Proceed only if at least 95% of APTOS images remain after deterministic exclusion and every original grade retains at least 50 images.

APTOS rows available:
3662

No model or transfer performance has been evaluated.

================ APTOS DUPLICATE CONFLICT SUMMARY ================


,metric,value
0,Original-hash duplicate groups,123.000000
1,Original-hash conflicting groups,30.000000
2,Canonical-hash duplicate groups,123.000000
3,Canonical-hash conflicting groups,30.000000
4,Ambiguous connected components,30.000000
5,Images excluded,62.000000
6,Images retained,3600.000000
7,Retained fraction,0.983069



================ GRADE RETENTION AFTER DETERMINISTIC EXCLUSION ================


,original_dr_grade,original_images,excluded_images,retained_images
0,0,1805,4,1801
1,1,370,10,360
2,2,999,23,976
3,3,193,13,180
4,4,295,12,283



================ AMBIGUOUS GRADE-SET SUMMARY ================


,grade_set,components,images
3,2|3,9,19
4,2|4,8,16
1,1|2,5,11
0,0|1,4,8
5,3|4,3,6
2,1|4,1,2



================ STAGE 2R-F DECISION ================
Decision:
PASS_EXCLUDE_AMBIGUOUS_DUPLICATE_COMPONENTS_AND_COMPLETE_STAGE2R

Interpretation:
APTOS contains exact duplicate-image components with discordant official grades. The frozen deterministic rule excludes every connected component containing more than one grade. The remaining cohort satisfies the pre-frozen retention and per-grade sample-size gates, so Stage 2R may continue without using the ambiguous images.

Authorised next step:
COMPLETE_STAGE2R_USING_ONLY_LABEL_CONSISTENT_COMPONENTS

Individual conflicting rows displayed:
False

Model performance observed:
False

Sealed evaluation performance observed:
False

APTOS duplicate-label conflict audit completed and sealed.


In [20]:
#@title 02R-G. Complete and seal Stage 2R after frozen APTOS conflict exclusion

from pathlib import Path
from datetime import datetime, timezone

from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import shutil
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen finalisation protocol
# ============================================================

RANDOM_SEED = 20260720
APTOS_EVALUATION_FRACTION = 0.30

APTOS_EXPECTED_ORIGINAL = 3662
APTOS_EXPECTED_EXCLUDED = 62
APTOS_EXPECTED_ELIGIBLE = 3600

IDRID_EXPECTED_TOTAL = 516
IDRID_EXPECTED_DEVELOPMENT = 413
IDRID_EXPECTED_EVALUATION = 103

CANONICAL_SIZE = 768
CANONICAL_JPEG_QUALITY = 92
CANONICAL_JPEG_SUBSAMPLING = 0
BLACK_BORDER_THRESHOLD = 10

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

POSITIVE_CLASS_RULE = (
    "original DR grade >= 2"
)

FINALISATION_BOUNDARY = (
    "Stage 2R-G uses only the label-consistent APTOS duplicate "
    "components authorised by Stage 2R-F. Partitioning is performed "
    "at connected-component level so no exact original-image or "
    "canonical-image duplicate can cross development and sealed "
    "evaluation. IDRiD retains its official training/testing split. "
    "Sealed-evaluation label values must not be displayed, summarised "
    "or inserted into modelling manifests."
)


# ============================================================
# 1. Verify required in-memory state
# ============================================================

required_runtime_variables = [
    "aptos_verified",
    "idrid_verified",
    "aptos_integrity_fixed",
    "idrid_integrity_fixed",
    "aptos_duplicate_component_audit",
    "LOCAL_APTOS_CANONICAL",
    "LOCAL_IDRID_CANONICAL",
    "archive_path",
]


missing_runtime_variables = [
    variable_name
    for variable_name in required_runtime_variables
    if variable_name not in globals()
]


if missing_runtime_variables:
    raise RuntimeError(
        "Required current-runtime objects are missing:\n"
        +
        "\n".join(
            missing_runtime_variables
        )
        +
        "\n\nThe Colab runtime may have restarted. "
        "Do not rerun earlier download or processing cells blindly."
    )


assert isinstance(
    aptos_verified,
    pd.DataFrame,
)

assert isinstance(
    idrid_verified,
    pd.DataFrame,
)

assert isinstance(
    aptos_duplicate_component_audit,
    pd.DataFrame,
)


assert (
    len(aptos_verified)
    ==
    APTOS_EXPECTED_ORIGINAL
)

assert (
    len(idrid_verified)
    ==
    IDRID_EXPECTED_TOTAL
)


LOCAL_APTOS_CANONICAL = Path(
    LOCAL_APTOS_CANONICAL
)

LOCAL_IDRID_CANONICAL = Path(
    LOCAL_IDRID_CANONICAL
)

archive_path = Path(
    archive_path
)


assert LOCAL_APTOS_CANONICAL.is_dir()
assert LOCAL_IDRID_CANONICAL.is_dir()
assert archive_path.is_file()


print(
    "================ STAGE 2R-G FINALISATION "
    "================"
)

print("\nFinalisation boundary:")
print(FINALISATION_BOUNDARY)

print("\nAPTOS rows before exclusion:")
print(len(aptos_verified))

print("\nIDRiD rows:")
print(len(idrid_verified))

print("\nModel performance observed:")
print(False)

print("\nSealed evaluation performance observed:")
print(False)


# ============================================================
# 2. Resolve Drive and project paths
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

TEST_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_v0.1"
)

PROTOCOL_ROOT = (
    TEST_ROOT
    / "00_Protocol"
)

CANONICAL_ROOT = (
    TEST_ROOT
    / "01_Canonical_Target_Images"
)

DEVELOPMENT_ROOT = (
    TEST_ROOT
    / "02_Development"
)

SEALED_EVALUATION_ROOT = (
    TEST_ROOT
    / "03_Sealed_Evaluation"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

ORIGINAL_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "00_Original_Source_Label_Files"
)

SEALED_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "01_Sealed_Evaluation_Labels"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)


for directory in [
    PROTOCOL_ROOT,
    CANONICAL_ROOT,
    DEVELOPMENT_ROOT,
    SEALED_EVALUATION_ROOT,
    ORIGINAL_LABEL_ROOT,
    SEALED_LABEL_ROOT,
    STAGE2_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


APTOS_FINAL_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019"
)

IDRID_FINAL_ROOT = (
    CANONICAL_ROOT
    / "IDRiD"
)

APTOS_STAGING_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019__STAGING"
)

IDRID_STAGING_ROOT = (
    CANONICAL_ROOT
    / "IDRiD__STAGING"
)


# ============================================================
# 3. Import sealed Stage 2R-F decision
# ============================================================

STAGE2F_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2F_APTOS_Duplicate_Label_Conflict_Decision_v0.1.json"
)


assert STAGE2F_DECISION_PATH.is_file(), (
    STAGE2F_DECISION_PATH
)


with open(
    STAGE2F_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage2f_decision = json.load(
        file
    )


assert (
    stage2f_decision["decision"]
    ==
    "PASS_EXCLUDE_AMBIGUOUS_DUPLICATE_COMPONENTS_AND_COMPLETE_STAGE2R"
)

assert (
    stage2f_decision["authorised_next_step"]
    ==
    "COMPLETE_STAGE2R_USING_ONLY_LABEL_CONSISTENT_COMPONENTS"
)

assert (
    int(
        stage2f_decision[
            "excluded_images"
        ]
    )
    ==
    APTOS_EXPECTED_EXCLUDED
)

assert (
    int(
        stage2f_decision[
            "retained_images"
        ]
    )
    ==
    APTOS_EXPECTED_ELIGIBLE
)


print("\nImported Stage 2R-F decision:")
print(
    stage2f_decision[
        "decision"
    ]
)


# ============================================================
# 4. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    path = Path(path)

    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def directory_size_bytes(path):
    path = Path(path)

    if not path.exists():
        return 0

    return int(
        sum(
            item.stat().st_size
            for item in path.rglob("*")
            if item.is_file()
        )
    )


def bytes_to_gib(value):
    return float(
        value
        /
        (1024 ** 3)
    )


def make_writable(path):
    path = Path(path)

    if path.exists():
        try:
            os.chmod(
                path,
                0o600,
            )
        except Exception:
            pass


def make_read_only(path):
    path = Path(path)

    try:
        os.chmod(
            path,
            0o400,
        )

        return True

    except Exception:
        return False


def json_default(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(value)

    raise TypeError(
        f"Unsupported JSON type: "
        f"{type(value)}"
    )


# ============================================================
# 5. Apply the frozen APTOS exclusion manifest
# ============================================================

required_component_columns = {
    "image_id",
    "duplicate_component_id",
    "excluded_due_to_ambiguous_duplicate_labels",
    "eligible_after_integrity_audit",
}


assert required_component_columns.issubset(
    aptos_duplicate_component_audit.columns
), (
    "Missing component-audit columns: "
    f"{sorted(required_component_columns - set(aptos_duplicate_component_audit.columns))}"
)


component_mapping = (
    aptos_duplicate_component_audit[
        [
            "image_id",
            "duplicate_component_id",
            "excluded_due_to_ambiguous_duplicate_labels",
            "eligible_after_integrity_audit",
        ]
    ]
    .copy()
)


assert component_mapping[
    "image_id"
].is_unique


aptos_with_components = aptos_verified.merge(
    component_mapping,
    on="image_id",
    how="inner",
    validate="one_to_one",
)


assert (
    len(aptos_with_components)
    ==
    APTOS_EXPECTED_ORIGINAL
)


aptos_eligible = (
    aptos_with_components[
        aptos_with_components[
            "eligible_after_integrity_audit"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


aptos_excluded = (
    aptos_with_components[
        ~aptos_with_components[
            "eligible_after_integrity_audit"
        ]
    ]
    .copy()
    .reset_index(drop=True)
)


assert (
    len(aptos_eligible)
    ==
    APTOS_EXPECTED_ELIGIBLE
)

assert (
    len(aptos_excluded)
    ==
    APTOS_EXPECTED_EXCLUDED
)


assert not aptos_eligible[
    "excluded_due_to_ambiguous_duplicate_labels"
].any()


# ============================================================
# 6. Component-level APTOS split
# ============================================================

component_grade_counts = (
    aptos_eligible
    .groupby(
        "duplicate_component_id"
    )[
        "original_dr_grade"
    ]
    .nunique()
)


assert not (
    component_grade_counts
    >
    1
).any(), (
    "An eligible APTOS component still contains "
    "more than one official grade."
)


aptos_component_table = (
    aptos_eligible
    .groupby(
        "duplicate_component_id",
        as_index=False,
    )
    .agg(
        original_dr_grade=(
            "original_dr_grade",
            "first",
        ),
        component_images=(
            "image_id",
            "size",
        ),
    )
    .sort_values(
        "duplicate_component_id"
    )
    .reset_index(drop=True)
)


components_per_grade = (
    aptos_component_table[
        "original_dr_grade"
    ]
    .value_counts()
)


assert (
    components_per_grade.min()
    >=
    2
), (
    "At least one grade has too few connected "
    "components for a stratified split."
)


splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=(
        APTOS_EVALUATION_FRACTION
    ),
    random_state=(
        RANDOM_SEED
    ),
)


development_component_indices, evaluation_component_indices = next(
    splitter.split(
        X=np.zeros(
            len(
                aptos_component_table
            )
        ),
        y=aptos_component_table[
            "original_dr_grade"
        ].to_numpy(),
    )
)


evaluation_component_ids = set(
    aptos_component_table.iloc[
        evaluation_component_indices
    ][
        "duplicate_component_id"
    ]
)


aptos_eligible[
    "prospective_partition"
] = np.where(
    aptos_eligible[
        "duplicate_component_id"
    ].isin(
        evaluation_component_ids
    ),
    "sealed_evaluation",
    "development",
)


component_partition_counts = (
    aptos_eligible
    .groupby(
        "duplicate_component_id"
    )[
        "prospective_partition"
    ]
    .nunique()
)


assert not (
    component_partition_counts
    >
    1
).any()


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    crossing_hashes = (
        aptos_eligible
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
    )

    assert not (
        crossing_hashes
        >
        1
    ).any(), (
        f"APTOS {hash_column} leakage "
        "detected across partitions."
    )


# ============================================================
# 7. Preserve and audit the IDRiD official split
# ============================================================

assert idrid_verified[
    "image_id"
].is_unique


idrid_final = (
    idrid_verified
    .copy()
    .reset_index(drop=True)
)


idrid_final[
    "prospective_partition"
] = (
    idrid_final[
        "official_partition"
    ]
)


assert (
    idrid_final[
        "prospective_partition"
    ]
    ==
    "development"
).sum() == IDRID_EXPECTED_DEVELOPMENT


assert (
    idrid_final[
        "prospective_partition"
    ]
    ==
    "sealed_evaluation"
).sum() == IDRID_EXPECTED_EVALUATION


idrid_integrity_audit = {}


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grade_conflicts = (
        idrid_final
        .groupby(
            hash_column
        )[
            "original_dr_grade"
        ]
        .nunique()
    )

    cross_partition = (
        idrid_final
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
    )

    idrid_integrity_audit[
        f"{hash_column}_conflicting_grade_groups"
    ] = int(
        (
            grade_conflicts
            >
            1
        ).sum()
    )

    idrid_integrity_audit[
        f"{hash_column}_cross_partition_groups"
    ] = int(
        (
            cross_partition
            >
            1
        ).sum()
    )


assert all(
    value == 0
    for value in idrid_integrity_audit.values()
), (
    "IDRiD exact-image integrity audit failed:\n"
    +
    json.dumps(
        idrid_integrity_audit,
        indent=2,
    )
)


# ============================================================
# 8. Cross-dataset exact-overlap audit
# ============================================================

original_cross_dataset_overlap = set(
    aptos_eligible[
        "original_sha256"
    ]
).intersection(
    set(
        idrid_final[
            "original_sha256"
        ]
    )
)


canonical_cross_dataset_overlap = set(
    aptos_eligible[
        "canonical_sha256"
    ]
).intersection(
    set(
        idrid_final[
            "canonical_sha256"
        ]
    )
)


assert not original_cross_dataset_overlap, (
    "Exact original-image overlap exists "
    "between APTOS and IDRiD."
)

assert not canonical_cross_dataset_overlap, (
    "Exact canonical-image overlap exists "
    "between APTOS and IDRiD."
)


# ============================================================
# 9. Harmonised binary endpoint
# ============================================================

for dataframe in [
    aptos_eligible,
    idrid_final,
]:
    dataframe[
        "moderate_or_worse_dr"
    ] = (
        dataframe[
            "original_dr_grade"
        ]
        >=
        2
    ).astype(int)


# ============================================================
# 10. Verify local canonical files and storage
# ============================================================

aptos_eligible[
    "local_canonical_path"
] = (
    aptos_eligible[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            LOCAL_APTOS_CANONICAL
            /
            filename
        )
    )
)


idrid_final[
    "local_canonical_path"
] = (
    idrid_final[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            LOCAL_IDRID_CANONICAL
            /
            filename
        )
    )
)


assert aptos_eligible[
    "local_canonical_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


assert idrid_final[
    "local_canonical_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


aptos_canonical_bytes = int(
    aptos_eligible[
        "local_canonical_path"
    ].map(
        lambda value: Path(
            value
        ).stat().st_size
    ).sum()
)


idrid_canonical_bytes = int(
    idrid_final[
        "local_canonical_path"
    ].map(
        lambda value: Path(
            value
        ).stat().st_size
    ).sum()
)


canonical_total_bytes = (
    aptos_canonical_bytes
    +
    idrid_canonical_bytes
)


# Remove stale partial outputs from failed attempts.
for staging_root in [
    APTOS_STAGING_ROOT,
    IDRID_STAGING_ROOT,
]:
    if staging_root.exists():
        shutil.rmtree(
            staging_root
        )


FINAL_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)


existing_successful_decision = False


if FINAL_DECISION_PATH.is_file():
    try:
        with open(
            FINAL_DECISION_PATH,
            "r",
            encoding="utf-8",
        ) as file:
            previous_final_decision = json.load(
                file
            )

        existing_successful_decision = (
            previous_final_decision.get(
                "decision"
            )
            ==
            "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_AND_LABEL_QUARANTINE_ADVANCE_TO_DEVELOPMENT_RECOVERABILITY_GATE"
        )

    except Exception:
        existing_successful_decision = False


if not existing_successful_decision:
    for partial_final_root in [
        APTOS_FINAL_ROOT,
        IDRID_FINAL_ROOT,
    ]:
        if partial_final_root.exists():
            shutil.rmtree(
                partial_final_root
            )


current_drive_free = (
    shutil.disk_usage(
        DRIVE_ROOT
    ).free
)


print(
    "\n================ CANONICAL STORAGE AUDIT "
    "================"
)

print(
    "Eligible APTOS canonical size:",
    f"{bytes_to_gib(aptos_canonical_bytes):.2f} GiB",
)

print(
    "IDRiD canonical size:",
    f"{bytes_to_gib(idrid_canonical_bytes):.2f} GiB",
)

print(
    "Total canonical size:",
    f"{bytes_to_gib(canonical_total_bytes):.2f} GiB",
)

print(
    "Current Drive free space:",
    f"{bytes_to_gib(current_drive_free):.2f} GiB",
)


assert (
    current_drive_free
    >
    canonical_total_bytes
    +
    512
    *
    1024 ** 2
), (
    "The final canonical cohort exceeds the "
    "remaining Google Drive capacity."
)


# ============================================================
# 11. Copy only eligible images to Drive staging
# ============================================================

APTOS_STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IDRID_STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "\n================ DRIVE STAGING COPY "
    "================"
)


for _, row in tqdm(
    aptos_eligible.iterrows(),
    total=len(
        aptos_eligible
    ),
    desc="Copying eligible APTOS images",
):
    source_path = Path(
        row[
            "local_canonical_path"
        ]
    )

    destination_path = (
        APTOS_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    shutil.copy2(
        source_path,
        destination_path,
    )


for _, row in tqdm(
    idrid_final.iterrows(),
    total=len(
        idrid_final
    ),
    desc="Copying IDRiD images",
):
    source_path = Path(
        row[
            "local_canonical_path"
        ]
    )

    destination_path = (
        IDRID_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    shutil.copy2(
        source_path,
        destination_path,
    )


assert (
    len(
        list(
            APTOS_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    APTOS_EXPECTED_ELIGIBLE
)


assert (
    len(
        list(
            IDRID_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_EXPECTED_TOTAL
)


# ============================================================
# 12. Verify every Drive copy by SHA-256
# ============================================================

print(
    "\n================ DRIVE COPY VERIFICATION "
    "================"
)


for _, row in tqdm(
    aptos_eligible.iterrows(),
    total=len(
        aptos_eligible
    ),
    desc="Verifying APTOS Drive hashes",
):
    copied_path = (
        APTOS_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


for _, row in tqdm(
    idrid_final.iterrows(),
    total=len(
        idrid_final
    ),
    desc="Verifying IDRiD Drive hashes",
):
    copied_path = (
        IDRID_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


if APTOS_FINAL_ROOT.exists():
    shutil.rmtree(
        APTOS_FINAL_ROOT
    )

if IDRID_FINAL_ROOT.exists():
    shutil.rmtree(
        IDRID_FINAL_ROOT
    )


APTOS_STAGING_ROOT.rename(
    APTOS_FINAL_ROOT
)

IDRID_STAGING_ROOT.rename(
    IDRID_FINAL_ROOT
)


# ============================================================
# 13. Attach final persistent image paths
# ============================================================

aptos_eligible[
    "canonical_image_path"
] = (
    aptos_eligible[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            APTOS_FINAL_ROOT
            /
            filename
        )
    )
)


idrid_final[
    "canonical_image_path"
] = (
    idrid_final[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            IDRID_FINAL_ROOT
            /
            filename
        )
    )
)


assert aptos_eligible[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


assert idrid_final[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


# ============================================================
# 14. Construct final partitions
# ============================================================

aptos_development = (
    aptos_eligible[
        aptos_eligible[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


aptos_evaluation = (
    aptos_eligible[
        aptos_eligible[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_development = (
    idrid_final[
        idrid_final[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_evaluation = (
    idrid_final[
        idrid_final[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


assert (
    len(aptos_development)
    +
    len(aptos_evaluation)
    ==
    APTOS_EXPECTED_ELIGIBLE
)


assert (
    len(idrid_development)
    ==
    IDRID_EXPECTED_DEVELOPMENT
)

assert (
    len(idrid_evaluation)
    ==
    IDRID_EXPECTED_EVALUATION
)


aptos_actual_evaluation_fraction = float(
    len(
        aptos_evaluation
    )
    /
    APTOS_EXPECTED_ELIGIBLE
)


# ============================================================
# 15. Output paths
# ============================================================

APTOS_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "APTOS_2019_Development_Labels_v0.1.csv"
)

IDRID_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "IDRiD_Development_Labels_v0.1.csv"
)

APTOS_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv"
)

IDRID_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv"
)

APTOS_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "APTOS_2019_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

IDRID_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "IDRiD_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

APTOS_ORIGINAL_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "APTOS_2019_Original_train.csv"
)

IDRID_ORIGINAL_TRAIN_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Training_Labels.csv"
)

IDRID_ORIGINAL_TEST_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Testing_Labels.csv"
)


for required_label_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
]:
    assert required_label_path.is_file(), (
        required_label_path
    )


for output_path in [
    APTOS_DEVELOPMENT_LABEL_PATH,
    IDRID_DEVELOPMENT_LABEL_PATH,
    APTOS_EVALUATION_MANIFEST_PATH,
    IDRID_EVALUATION_MANIFEST_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_writable(
        output_path
    )


# ============================================================
# 16. Save development labels and sealed evaluation artifacts
# ============================================================

development_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_dr_grade",
    "moderate_or_worse_dr",
    "original_sha256",
    "canonical_sha256",
]


evaluation_manifest_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_sha256",
    "canonical_sha256",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


sealed_label_columns = [
    "dataset",
    "image_id",
    "original_dr_grade",
    "moderate_or_worse_dr",
]


aptos_development[
    development_columns
].to_csv(
    APTOS_DEVELOPMENT_LABEL_PATH,
    index=False,
)


idrid_development[
    development_columns
].to_csv(
    IDRID_DEVELOPMENT_LABEL_PATH,
    index=False,
)


aptos_evaluation[
    evaluation_manifest_columns
].to_csv(
    APTOS_EVALUATION_MANIFEST_PATH,
    index=False,
)


idrid_evaluation[
    evaluation_manifest_columns
].to_csv(
    IDRID_EVALUATION_MANIFEST_PATH,
    index=False,
)


aptos_evaluation[
    sealed_label_columns
].to_csv(
    APTOS_SEALED_LABEL_PATH,
    index=False,
)


idrid_evaluation[
    sealed_label_columns
].to_csv(
    IDRID_SEALED_LABEL_PATH,
    index=False,
)


for protected_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_read_only(
        protected_path
    )


# ============================================================
# 17. Create label-free manifests
# ============================================================

label_free_master_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "prospective_partition",
    "original_sha256",
    "canonical_sha256",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


aptos_label_free = (
    aptos_eligible[
        label_free_master_columns
        +
        [
            "duplicate_component_id",
        ]
    ]
    .copy()
)


idrid_label_free = (
    idrid_final[
        label_free_master_columns
    ]
    .copy()
)


idrid_label_free[
    "duplicate_component_id"
] = (
    "NOT_APPLICABLE_IDRID_OFFICIAL_SPLIT"
)


label_free_master_manifest = pd.concat(
    [
        aptos_label_free,
        idrid_label_free,
    ],
    ignore_index=True,
)


assert (
    "original_dr_grade"
    not in
    label_free_master_manifest.columns
)

assert (
    "moderate_or_worse_dr"
    not in
    label_free_master_manifest.columns
)


integrity_columns = [
    "dataset",
    "image_id",
    "original_filename",
    "canonical_filename",
    "original_sha256",
    "canonical_sha256",
    "original_size_bytes",
    "canonical_size_bytes",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
]


final_integrity_manifest = pd.concat(
    [
        aptos_eligible[
            integrity_columns
        ],
        idrid_final[
            integrity_columns
        ],
    ],
    ignore_index=True,
)


# ============================================================
# 18. Save protocol and audit artifacts
# ============================================================

CANONICAL_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Canonical_Fundus_Preprocessing_Protocol_v0.1.json"
)

QUARANTINE_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Target_Label_Quarantine_Protocol_v0.1.json"
)

MASTER_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_LabelFree_Master_Target_Manifest_v0.1.csv"
)

PARTITION_ASSIGNMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Final_LabelFree_Partition_Assignment_v0.1.csv"
)

INTEGRITY_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

PARTITION_SUMMARY_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Partition_Summary_v0.1.csv"
)

LABEL_COMMITMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Sealed_Label_Commitments_v0.1.json"
)

FINAL_REPAIR_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2R_Final_Repair_And_Exclusion_Manifest_v0.1.json"
)

STAGE2_REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Report_v0.1.md"
)

STAGE2_ENVIRONMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Environment_v0.1.json"
)


canonical_protocol = {
    "protocol_version": "v0.1",
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "black_border_threshold": (
        BLACK_BORDER_THRESHOLD
    ),
    "jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "jpeg_subsampling": (
        CANONICAL_JPEG_SUBSAMPLING
    ),
    "resize_filter": "LANCZOS",
    "frozen_before_target_performance": True,
    "must_be_applied_identically_to_sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
}


with open(
    CANONICAL_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        canonical_protocol,
        file,
        indent=2,
    )


quarantine_protocol = {
    "protocol_version": "v0.1",
    "endpoint_id": ENDPOINT_ID,
    "positive_class_rule": (
        POSITIVE_CLASS_RULE
    ),
    "random_seed": (
        RANDOM_SEED
    ),
    "aptos_requested_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "aptos_split_unit": (
        "label-consistent connected duplicate component"
    ),
    "aptos_ambiguous_images_excluded": (
        APTOS_EXPECTED_EXCLUDED
    ),
    "development_labels_authorised_from_stage": 3,
    "sealed_evaluation_labels_authorised_stage": 6,
    "sealed_evaluation_manifests_label_free": True,
    "sealed_label_values_displayed": False,
    "idrid_unique_identifier_rule": (
        "<official_partition>__<official_image_stem>"
    ),
    "prohibitions": [
        (
            "Do not open, display, summarise or model "
            "sealed-evaluation labels before Stage 6."
        ),
        (
            "Do not change the final partitions after "
            "recoverability or transfer results are observed."
        ),
        (
            "Do not change the canonicalisation protocol "
            "after target performance is observed."
        ),
        (
            "Do not restore the 62 ambiguous APTOS images "
            "to development or evaluation analyses."
        ),
    ],
}


with open(
    QUARANTINE_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quarantine_protocol,
        file,
        indent=2,
    )


label_free_master_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False,
)


label_free_master_manifest[
    [
        "dataset",
        "image_id",
        "prospective_partition",
        "duplicate_component_id",
        "original_sha256",
        "canonical_sha256",
    ]
].to_csv(
    PARTITION_ASSIGNMENT_PATH,
    index=False,
)


final_integrity_manifest.to_csv(
    INTEGRITY_MANIFEST_PATH,
    index=False,
)


partition_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "original_images": (
                APTOS_EXPECTED_ORIGINAL
            ),
            "excluded_ambiguous_images": (
                APTOS_EXPECTED_EXCLUDED
            ),
            "eligible_images": (
                APTOS_EXPECTED_ELIGIBLE
            ),
            "development_images": int(
                len(
                    aptos_development
                )
            ),
            "sealed_evaluation_images": int(
                len(
                    aptos_evaluation
                )
            ),
            "actual_evaluation_fraction": (
                aptos_actual_evaluation_fraction
            ),
            "partition_source": (
                "component-grouped deterministic "
                "grade-stratified split"
            ),
            "sealed_evaluation_labels_displayed": False,
        },
        {
            "dataset": "IDRiD",
            "original_images": (
                IDRID_EXPECTED_TOTAL
            ),
            "excluded_ambiguous_images": 0,
            "eligible_images": (
                IDRID_EXPECTED_TOTAL
            ),
            "development_images": int(
                len(
                    idrid_development
                )
            ),
            "sealed_evaluation_images": int(
                len(
                    idrid_evaluation
                )
            ),
            "actual_evaluation_fraction": float(
                len(
                    idrid_evaluation
                )
                /
                IDRID_EXPECTED_TOTAL
            ),
            "partition_source": (
                "official IDRiD training/testing split"
            ),
            "sealed_evaluation_labels_displayed": False,
        },
    ]
)


partition_summary.to_csv(
    PARTITION_SUMMARY_PATH,
    index=False,
)


label_commitments = {
    "endpoint_id": ENDPOINT_ID,
    "aptos_sealed_evaluation": {
        "rows": int(
            len(
                aptos_evaluation
            )
        ),
        "path": str(
            APTOS_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            APTOS_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "idrid_sealed_evaluation": {
        "rows": int(
            len(
                idrid_evaluation
            )
        ),
        "path": str(
            IDRID_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "original_label_files": {
        "aptos_train_sha256": sha256_file(
            APTOS_ORIGINAL_LABEL_PATH
        ),
        "idrid_train_sha256": sha256_file(
            IDRID_ORIGINAL_TRAIN_LABEL_PATH
        ),
        "idrid_test_sha256": sha256_file(
            IDRID_ORIGINAL_TEST_LABEL_PATH
        ),
    },
}


with open(
    LABEL_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_commitments,
        file,
        indent=2,
    )


repair_manifest = {
    "idrid_identifier_repair": {
        "problem": (
            "Official training and testing folders reused "
            "103 bare filename stems."
        ),
        "resolution": (
            "Partition-qualified image identifiers."
        ),
        "final_ids_unique": True,
    },
    "aptos_duplicate_label_resolution": {
        "original_images": (
            APTOS_EXPECTED_ORIGINAL
        ),
        "ambiguous_components": int(
            stage2f_decision[
                "ambiguous_connected_components"
            ]
        ),
        "excluded_images": (
            APTOS_EXPECTED_EXCLUDED
        ),
        "retained_images": (
            APTOS_EXPECTED_ELIGIBLE
        ),
        "retained_fraction": float(
            stage2f_decision[
                "retained_fraction"
            ]
        ),
        "resolution": (
            "Entire label-discordant connected components "
            "excluded before final partitioning."
        ),
    },
    "model_performance_observed_before_resolution": False,
    "sealed_evaluation_performance_observed_before_resolution": False,
    "computer_sleep_or_lock_caused_failures": False,
}


with open(
    FINAL_REPAIR_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_manifest,
        file,
        indent=2,
    )


# ============================================================
# 19. Final leakage and label-boundary checks
# ============================================================

aptos_original_cross_split_duplicates = int(
    (
        aptos_eligible
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


aptos_canonical_cross_split_duplicates = int(
    (
        aptos_eligible
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_original_cross_split_duplicates = int(
    (
        idrid_final
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_canonical_cross_split_duplicates = int(
    (
        idrid_final
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


evaluation_manifest_has_no_labels = bool(
    all(
        label_column
        not in pd.read_csv(
            manifest_path,
            nrows=1,
        ).columns
        for manifest_path in [
            APTOS_EVALUATION_MANIFEST_PATH,
            IDRID_EVALUATION_MANIFEST_PATH,
        ]
        for label_column in [
            "original_dr_grade",
            "moderate_or_worse_dr",
        ]
    )
)


assert (
    aptos_original_cross_split_duplicates
    ==
    0
)

assert (
    aptos_canonical_cross_split_duplicates
    ==
    0
)

assert (
    idrid_original_cross_split_duplicates
    ==
    0
)

assert (
    idrid_canonical_cross_split_duplicates
    ==
    0
)

assert (
    len(
        original_cross_dataset_overlap
    )
    ==
    0
)

assert (
    len(
        canonical_cross_dataset_overlap
    )
    ==
    0
)

assert evaluation_manifest_has_no_labels


# ============================================================
# 20. Final Stage 2R decision
# ============================================================

stage2_decision = (
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_"
    "AND_LABEL_QUARANTINE_ADVANCE_TO_"
    "DEVELOPMENT_RECOVERABILITY_GATE"
)


stage2_interpretation = (
    "APTOS and IDRiD were converted using the identical frozen "
    "768x768 canonicalisation and hash-verified after copying to "
    "Google Drive. Thirty APTOS duplicate components containing "
    "discordant official grades were deterministically excluded, "
    "removing 62 of 3,662 images and retaining 3,600 images. "
    "Eligible APTOS duplicate components were kept intact during "
    "the final development/evaluation split. IDRiD retained its "
    "official training/testing split with partition-qualified "
    "identifiers. Sealed labels are absent from every modelling "
    "manifest and committed for one-time Stage 6 evaluation."
)


decision_payload = {
    "decision": (
        stage2_decision
    ),
    "interpretation": (
        stage2_interpretation
    ),
    "authorised_next_step": (
        "DEVELOPMENT_ONLY_TARGET_RECOVERABILITY_GATE"
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "positive_class_rule": (
        POSITIVE_CLASS_RULE
    ),
    "canonicalisation_protocol": str(
        CANONICAL_PROTOCOL_PATH
    ),
    "quarantine_protocol": str(
        QUARANTINE_PROTOCOL_PATH
    ),
    "stage2f_decision": str(
        STAGE2F_DECISION_PATH
    ),
    "aptos": {
        "original_images": (
            APTOS_EXPECTED_ORIGINAL
        ),
        "excluded_ambiguous_images": (
            APTOS_EXPECTED_EXCLUDED
        ),
        "eligible_images": (
            APTOS_EXPECTED_ELIGIBLE
        ),
        "development_images": int(
            len(
                aptos_development
            )
        ),
        "sealed_evaluation_images": int(
            len(
                aptos_evaluation
            )
        ),
        "actual_evaluation_fraction": (
            aptos_actual_evaluation_fraction
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                aptos_canonical_bytes
            )
        ),
        "original_cross_split_duplicates": (
            aptos_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            aptos_canonical_cross_split_duplicates
        ),
    },
    "idrid": {
        "eligible_images": (
            IDRID_EXPECTED_TOTAL
        ),
        "development_images": (
            IDRID_EXPECTED_DEVELOPMENT
        ),
        "sealed_evaluation_images": (
            IDRID_EXPECTED_EVALUATION
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                idrid_canonical_bytes
            )
        ),
        "partition_qualified_ids": True,
        "original_cross_split_duplicates": (
            idrid_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            idrid_canonical_cross_split_duplicates
        ),
    },
    "original_cross_dataset_overlap": int(
        len(
            original_cross_dataset_overlap
        )
    ),
    "canonical_cross_dataset_overlap": int(
        len(
            canonical_cross_dataset_overlap
        )
    ),
    "evaluation_manifests_label_free": (
        evaluation_manifest_has_no_labels
    ),
    "sealed_label_values_displayed": False,
    "model_performance_observed": False,
    "sealed_evaluation_performance_observed": False,
    "sealed_label_commitment_path": str(
        LABEL_COMMITMENT_PATH
    ),
    "important_boundary": (
        "Stage 2 establishes controlled acquisition, frozen "
        "canonicalisation, integrity filtering, deterministic "
        "partitioning and procedural label quarantine only. "
        "It does not establish target recoverability, transfer "
        "performance or CDO validity."
    ),
}


make_writable(
    FINAL_DECISION_PATH
)


with open(
    FINAL_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
        default=json_default,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 2R",
    "",
    "## Final controlled acquisition and quarantine",
    "",
    f"Decision: `{stage2_decision}`",
    "",
    stage2_interpretation,
    "",
    "## Final cohorts",
    "",
    (
        f"- APTOS eligible images: "
        f"{APTOS_EXPECTED_ELIGIBLE}"
    ),
    (
        f"- APTOS development images: "
        f"{len(aptos_development)}"
    ),
    (
        f"- APTOS sealed-evaluation images: "
        f"{len(aptos_evaluation)}"
    ),
    (
        f"- IDRiD development images: "
        f"{len(idrid_development)}"
    ),
    (
        f"- IDRiD sealed-evaluation images: "
        f"{len(idrid_evaluation)}"
    ),
    "",
    "## Safety boundary",
    "",
    "- Model performance observed: `False`",
    (
        "- Sealed-evaluation performance "
        "observed: `False`"
    ),
    (
        "- Sealed-evaluation label values "
        "displayed: `False`"
    ),
    (
        "- Evaluation manifests are "
        "label-free: `True`"
    ),
    "",
]


STAGE2_REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "random_seed": (
        RANDOM_SEED
    ),
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "aptos_requested_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "aptos_actual_evaluation_fraction": (
        aptos_actual_evaluation_fraction
    ),
    "test_root": str(
        TEST_ROOT
    ),
}


with open(
    STAGE2_ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 21. Final integrity checks
# ============================================================

required_outputs = [
    CANONICAL_PROTOCOL_PATH,
    QUARANTINE_PROTOCOL_PATH,
    MASTER_MANIFEST_PATH,
    PARTITION_ASSIGNMENT_PATH,
    INTEGRITY_MANIFEST_PATH,
    PARTITION_SUMMARY_PATH,
    LABEL_COMMITMENT_PATH,
    FINAL_REPAIR_MANIFEST_PATH,
    FINAL_DECISION_PATH,
    STAGE2_REPORT_PATH,
    STAGE2_ENVIRONMENT_PATH,
    APTOS_DEVELOPMENT_LABEL_PATH,
    IDRID_DEVELOPMENT_LABEL_PATH,
    APTOS_EVALUATION_MANIFEST_PATH,
    IDRID_EVALUATION_MANIFEST_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]


for output_path in required_outputs:
    assert output_path.is_file(), (
        output_path
    )


assert (
    len(
        list(
            APTOS_FINAL_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    APTOS_EXPECTED_ELIGIBLE
)


assert (
    len(
        list(
            IDRID_FINAL_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_EXPECTED_TOTAL
)


canonical_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "images": int(
                len(
                    aptos_eligible
                )
            ),
            "unique_image_ids": int(
                aptos_eligible[
                    "image_id"
                ].nunique()
            ),
            "unique_original_sha256": int(
                aptos_eligible[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                aptos_eligible[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    aptos_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
        {
            "dataset": "IDRiD",
            "images": int(
                len(
                    idrid_final
                )
            ),
            "unique_image_ids": int(
                idrid_final[
                    "image_id"
                ].nunique()
            ),
            "unique_original_sha256": int(
                idrid_final[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                idrid_final[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    idrid_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
    ]
)


# ============================================================
# 22. Delete temporary ZIP and local working files
# ============================================================

TEMP_ROOT = (
    archive_path.parent.parent
)


print(
    "\nDeleting temporary APTOS ZIP and "
    "local canonical working files..."
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


# ============================================================
# 23. Final display
# ============================================================

print(
    "\n================ CANONICAL IMAGE SUMMARY "
    "================"
)

display(
    canonical_summary
)


print(
    "\n================ PARTITION SUMMARY "
    "================"
)

display(
    partition_summary
)


print(
    "\n================ LEAKAGE CHECK "
    "================"
)

print(
    "APTOS original-hash cross-split duplicates:",
    aptos_original_cross_split_duplicates,
)

print(
    "APTOS canonical-hash cross-split duplicates:",
    aptos_canonical_cross_split_duplicates,
)

print(
    "IDRiD original-hash cross-split duplicates:",
    idrid_original_cross_split_duplicates,
)

print(
    "IDRiD canonical-hash cross-split duplicates:",
    idrid_canonical_cross_split_duplicates,
)

print(
    "Original cross-dataset overlap:",
    len(
        original_cross_dataset_overlap
    ),
)

print(
    "Canonical cross-dataset overlap:",
    len(
        canonical_cross_dataset_overlap
    ),
)

print(
    "Evaluation manifests contain labels:",
    not evaluation_manifest_has_no_labels,
)


print(
    "\n================ LABEL QUARANTINE CHECK "
    "================"
)

print(
    "APTOS sealed-label SHA-256:",
    label_commitments[
        "aptos_sealed_evaluation"
    ][
        "sha256"
    ],
)

print(
    "IDRiD sealed-label SHA-256:",
    label_commitments[
        "idrid_sealed_evaluation"
    ][
        "sha256"
    ],
)

print(
    "Sealed-evaluation label values displayed:",
    False,
)


print(
    "\n================ STAGE 2R-G DECISION "
    "================"
)

print("Decision:")
print(
    stage2_decision
)

print("\nInterpretation:")
print(
    stage2_interpretation
)

print("\nAuthorised next step:")
print(
    decision_payload[
        "authorised_next_step"
    ]
)

print(
    "\nTemporary APTOS archive retained:"
)

print(False)

print(
    "\nModel performance observed:"
)

print(False)

print(
    "\nSealed evaluation performance observed:"
)

print(False)

print(
    "\nStage 2R finalisation completed and sealed."
)

================ STAGE 2R-G FINALISATION ================

Finalisation boundary:
Stage 2R-G uses only the label-consistent APTOS duplicate components authorised by Stage 2R-F. Partitioning is performed at connected-component level so no exact original-image or canonical-image duplicate can cross development and sealed evaluation. IDRiD retains its official training/testing split. Sealed-evaluation label values must not be displayed, summarised or inserted into modelling manifests.

APTOS rows before exclusion:
3662

IDRiD rows:
516

Model performance observed:
False

Sealed evaluation performance observed:
False

Imported Stage 2R-F decision:
PASS_EXCLUDE_AMBIGUOUS_DUPLICATE_COMPONENTS_AND_COMPLETE_STAGE2R


AssertionError: IDRiD exact-image integrity audit failed:
{
  "original_sha256_conflicting_grade_groups": 3,
  "original_sha256_cross_partition_groups": 1,
  "canonical_sha256_conflicting_grade_groups": 3,
  "canonical_sha256_cross_partition_groups": 1
}

In [21]:
#@title 02R-H. Audit IDRiD exact-duplicate label conflicts and official-split leakage

from pathlib import Path

import hashlib
import json
import os

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen integrity rule
# ============================================================

MINIMUM_OVERALL_RETAINED_FRACTION = 0.95
MINIMUM_PARTITION_RETAINED_FRACTION = 0.95

INTEGRITY_RULE = (
    "Construct connected components using exact original-image "
    "SHA-256 equality and exact canonical-image SHA-256 equality. "
    "Exclude an entire component if it contains more than one "
    "official DR grade or if it spans both the official development "
    "and sealed-evaluation partitions. Label-consistent duplicate "
    "components contained wholly within one partition remain eligible."
)

CONTINUATION_GATE = (
    "Proceed only if at least 95% of the entire IDRiD cohort remains "
    "and at least 95% of each official partition remains after the "
    "deterministic exclusion."
)

MODEL_PERFORMANCE_OBSERVED = False
SEALED_EVALUATION_PERFORMANCE_OBSERVED = False
INDIVIDUAL_SEALED_LABELS_DISPLAYED = False


# ============================================================
# 1. Verify current runtime objects
# ============================================================

required_runtime_variables = [
    "idrid_verified",
    "idrid_integrity_fixed",
    "LOCAL_IDRID_CANONICAL",
    "aptos_verified",
    "LOCAL_APTOS_CANONICAL",
    "archive_path",
]


missing_runtime_variables = [
    variable_name
    for variable_name in required_runtime_variables
    if variable_name not in globals()
]


if missing_runtime_variables:
    raise RuntimeError(
        "Required Stage 2R runtime objects are missing:\n"
        +
        "\n".join(
            missing_runtime_variables
        )
        +
        "\n\nThe Colab runtime may have restarted. "
        "Do not rerun the download or image-processing cells blindly."
    )


assert isinstance(
    idrid_verified,
    pd.DataFrame,
)

assert len(
    idrid_verified
) == 516


required_columns = {
    "dataset",
    "image_id",
    "official_partition",
    "original_dr_grade",
    "original_sha256",
    "canonical_sha256",
}


missing_columns = (
    required_columns
    -
    set(
        idrid_verified.columns
    )
)


assert not missing_columns, (
    "Missing IDRiD columns: "
    f"{sorted(missing_columns)}"
)


assert idrid_verified[
    "image_id"
].is_unique


assert set(
    idrid_verified[
        "official_partition"
    ].unique()
) == {
    "development",
    "sealed_evaluation",
}


assert idrid_verified[
    "original_dr_grade"
].isin(
    [
        0,
        1,
        2,
        3,
        4,
    ]
).all()


print(
    "================ STAGE 2R-H "
    "IDRiD INTEGRITY AUDIT ================"
)

print("\nFrozen integrity rule:")
print(INTEGRITY_RULE)

print("\nFrozen continuation gate:")
print(CONTINUATION_GATE)

print("\nIDRiD rows available:")
print(
    len(
        idrid_verified
    )
)

print("\nModel performance observed:")
print(False)

print("\nSealed evaluation performance observed:")
print(False)


# ============================================================
# 2. Resolve Drive output paths
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

TEST_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

IDRID_INTEGRITY_QUARANTINE_ROOT = (
    QUARANTINE_ROOT
    / "03_IDRiD_Exact_Duplicate_Integrity"
)


STAGE2_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IDRID_INTEGRITY_QUARANTINE_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 3. Freeze the audit table
# ============================================================

audit_table = (
    idrid_verified[
        [
            "dataset",
            "image_id",
            "official_partition",
            "original_dr_grade",
            "original_sha256",
            "canonical_sha256",
        ]
    ]
    .copy()
    .sort_values(
        "image_id"
    )
    .reset_index(
        drop=True
    )
)


# ============================================================
# 4. Direct duplicate-group summaries
# ============================================================

def summarise_hash_groups(
    dataframe,
    hash_column,
):
    return (
        dataframe
        .groupby(
            hash_column,
            as_index=False,
        )
        .agg(
            image_count=(
                "image_id",
                "size",
            ),
            grade_count=(
                "original_dr_grade",
                "nunique",
            ),
            partition_count=(
                "official_partition",
                "nunique",
            ),
            development_images=(
                "official_partition",
                lambda values: int(
                    (
                        values
                        ==
                        "development"
                    ).sum()
                ),
            ),
            sealed_evaluation_images=(
                "official_partition",
                lambda values: int(
                    (
                        values
                        ==
                        "sealed_evaluation"
                    ).sum()
                ),
            ),
        )
    )


original_group_summary = (
    summarise_hash_groups(
        audit_table,
        "original_sha256",
    )
)

canonical_group_summary = (
    summarise_hash_groups(
        audit_table,
        "canonical_sha256",
    )
)


original_duplicate_groups = (
    original_group_summary[
        original_group_summary[
            "image_count"
        ]
        >
        1
    ]
)


canonical_duplicate_groups = (
    canonical_group_summary[
        canonical_group_summary[
            "image_count"
        ]
        >
        1
    ]
)


original_conflicting_groups = (
    original_group_summary[
        original_group_summary[
            "grade_count"
        ]
        >
        1
    ]
)


canonical_conflicting_groups = (
    canonical_group_summary[
        canonical_group_summary[
            "grade_count"
        ]
        >
        1
    ]
)


original_cross_partition_groups = (
    original_group_summary[
        original_group_summary[
            "partition_count"
        ]
        >
        1
    ]
)


canonical_cross_partition_groups = (
    canonical_group_summary[
        canonical_group_summary[
            "partition_count"
        ]
        >
        1
    ]
)


# ============================================================
# 5. Connected components across both hash types
# ============================================================

number_of_rows = len(
    audit_table
)

parent = np.arange(
    number_of_rows,
    dtype=np.int64,
)

rank = np.zeros(
    number_of_rows,
    dtype=np.int64,
)


def find_root(index):
    index = int(index)

    while parent[index] != index:
        parent[index] = parent[
            parent[index]
        ]

        index = int(
            parent[index]
        )

    return index


def union_indices(
    left_index,
    right_index,
):
    left_root = find_root(
        left_index
    )

    right_root = find_root(
        right_index
    )

    if left_root == right_root:
        return

    if rank[left_root] < rank[right_root]:
        parent[left_root] = right_root

    elif rank[left_root] > rank[right_root]:
        parent[right_root] = left_root

    else:
        parent[right_root] = left_root
        rank[left_root] += 1


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grouped_indices = (
        audit_table
        .groupby(
            hash_column,
            sort=False,
        )
        .indices
    )

    for row_indices in grouped_indices.values():
        row_indices = list(
            map(
                int,
                row_indices,
            )
        )

        if len(row_indices) <= 1:
            continue

        anchor_index = row_indices[0]

        for other_index in row_indices[1:]:
            union_indices(
                anchor_index,
                other_index,
            )


component_roots = [
    find_root(index)
    for index in range(
        number_of_rows
    )
]


root_to_indices = {}


for row_index, root in enumerate(
    component_roots
):
    root_to_indices.setdefault(
        int(root),
        [],
    ).append(
        int(row_index)
    )


root_to_component_id = {}


for root, row_indices in root_to_indices.items():
    component_image_ids = sorted(
        audit_table.iloc[
            row_indices
        ][
            "image_id"
        ].astype(str)
    )

    component_digest = hashlib.sha256(
        "|".join(
            component_image_ids
        ).encode(
            "utf-8"
        )
    ).hexdigest()[:16]

    root_to_component_id[
        root
    ] = (
        "IDRID_COMPONENT_"
        +
        component_digest
    )


audit_table[
    "duplicate_component_id"
] = [
    root_to_component_id[
        int(root)
    ]
    for root in component_roots
]


component_summary = (
    audit_table
    .groupby(
        "duplicate_component_id",
        as_index=False,
    )
    .agg(
        component_images=(
            "image_id",
            "size",
        ),
        original_hashes=(
            "original_sha256",
            "nunique",
        ),
        canonical_hashes=(
            "canonical_sha256",
            "nunique",
        ),
        grade_count=(
            "original_dr_grade",
            "nunique",
        ),
        partition_count=(
            "official_partition",
            "nunique",
        ),
        development_images=(
            "official_partition",
            lambda values: int(
                (
                    values
                    ==
                    "development"
                ).sum()
            ),
        ),
        sealed_evaluation_images=(
            "official_partition",
            lambda values: int(
                (
                    values
                    ==
                    "sealed_evaluation"
                ).sum()
            ),
        ),
    )
)


component_summary[
    "ambiguous_label_component"
] = (
    component_summary[
        "grade_count"
    ]
    >
    1
)


component_summary[
    "cross_partition_component"
] = (
    component_summary[
        "partition_count"
    ]
    >
    1
)


component_summary[
    "excluded_component"
] = (
    component_summary[
        "ambiguous_label_component"
    ]
    |
    component_summary[
        "cross_partition_component"
    ]
)


excluded_component_ids = set(
    component_summary.loc[
        component_summary[
            "excluded_component"
        ],
        "duplicate_component_id",
    ]
)


ambiguous_component_ids = set(
    component_summary.loc[
        component_summary[
            "ambiguous_label_component"
        ],
        "duplicate_component_id",
    ]
)


cross_partition_component_ids = set(
    component_summary.loc[
        component_summary[
            "cross_partition_component"
        ],
        "duplicate_component_id",
    ]
)


audit_table[
    "excluded_due_to_label_conflict"
] = (
    audit_table[
        "duplicate_component_id"
    ].isin(
        ambiguous_component_ids
    )
)


audit_table[
    "excluded_due_to_cross_partition_leakage"
] = (
    audit_table[
        "duplicate_component_id"
    ].isin(
        cross_partition_component_ids
    )
)


audit_table[
    "excluded_by_integrity_rule"
] = (
    audit_table[
        "duplicate_component_id"
    ].isin(
        excluded_component_ids
    )
)


audit_table[
    "eligible_after_integrity_audit"
] = ~(
    audit_table[
        "excluded_by_integrity_rule"
    ]
)


# ============================================================
# 6. Retention assessment
# ============================================================

excluded_rows = (
    audit_table[
        audit_table[
            "excluded_by_integrity_rule"
        ]
    ]
    .copy()
)


eligible_rows = (
    audit_table[
        audit_table[
            "eligible_after_integrity_audit"
        ]
    ]
    .copy()
)


overall_retained_fraction = float(
    len(
        eligible_rows
    )
    /
    len(
        audit_table
    )
)


partition_retention = (
    audit_table
    .groupby(
        "official_partition"
    )
    .agg(
        original_images=(
            "image_id",
            "size",
        ),
        excluded_images=(
            "excluded_by_integrity_rule",
            "sum",
        ),
        retained_images=(
            "eligible_after_integrity_audit",
            "sum",
        ),
    )
    .reset_index()
)


partition_retention[
    "retained_fraction"
] = (
    partition_retention[
        "retained_images"
    ]
    /
    partition_retention[
        "original_images"
    ]
)


minimum_partition_retained_fraction = float(
    partition_retention[
        "retained_fraction"
    ].min()
)


overall_gate_pass = bool(
    overall_retained_fraction
    >=
    MINIMUM_OVERALL_RETAINED_FRACTION
)


partition_gate_pass = bool(
    minimum_partition_retained_fraction
    >=
    MINIMUM_PARTITION_RETAINED_FRACTION
)


# ============================================================
# 7. Decision
# ============================================================

if (
    overall_gate_pass
    and
    partition_gate_pass
):
    stage2h_decision = (
        "PASS_EXCLUDE_IDRID_CONFLICT_AND_LEAKAGE_"
        "COMPONENTS_COMPLETE_STAGE2R"
    )

    authorised_next_step = (
        "COMPLETE_STAGE2R_USING_IDRID_"
        "INTEGRITY_ELIGIBLE_IMAGES"
    )

    interpretation = (
        "IDRiD contains exact duplicate-image components with "
        "discordant official grades and at least one component "
        "crossing the official training/testing boundary. The "
        "frozen deterministic rule excludes every component with "
        "either label ambiguity or cross-partition leakage. The "
        "remaining overall cohort and both official partitions "
        "satisfy the pre-frozen retention gates."
    )

else:
    stage2h_decision = (
        "HOLD_IDRID_INTEGRITY_EXCLUSION_EXCEEDS_"
        "FROZEN_RETENTION_GATE"
    )

    authorised_next_step = (
        "DO_NOT_COMPLETE_STAGE2R_REVIEW_IDRID_INTEGRITY"
    )

    interpretation = (
        "The deterministic exclusion required to eliminate "
        "IDRiD label conflicts and official-split leakage would "
        "remove more data than permitted by the frozen retention "
        "gate. Stage 2R remains on hold."
    )


# ============================================================
# 8. Save audit artifacts
# ============================================================

FULL_LABELLED_AUDIT_PATH = (
    IDRID_INTEGRITY_QUARANTINE_ROOT
    / "IDRiD_Exact_Duplicate_Integrity_"
      "WITH_LABELS_DO_NOT_OPEN_v0.1.csv"
)

COMPONENT_SUMMARY_PATH = (
    IDRID_INTEGRITY_QUARANTINE_ROOT
    / "IDRiD_Exact_Duplicate_Component_"
      "Summary_DO_NOT_OPEN_v0.1.csv"
)

LABEL_FREE_EXCLUSION_PATH = (
    STAGE2_ROOT
    / "Stage2H_IDRiD_Exact_Duplicate_"
      "Exclusion_Manifest_v0.1.csv"
)

PARTITION_RETENTION_PATH = (
    STAGE2_ROOT
    / "Stage2H_IDRiD_Partition_Retention_"
      "After_Integrity_Exclusion_v0.1.csv"
)

DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2H_IDRiD_Exact_Duplicate_"
      "Integrity_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2H_IDRiD_Exact_Duplicate_"
      "Integrity_Report_v0.1.md"
)


audit_table.to_csv(
    FULL_LABELLED_AUDIT_PATH,
    index=False,
)


component_summary.to_csv(
    COMPONENT_SUMMARY_PATH,
    index=False,
)


excluded_rows[
    [
        "dataset",
        "image_id",
        "official_partition",
        "original_sha256",
        "canonical_sha256",
        "duplicate_component_id",
        "excluded_due_to_label_conflict",
        "excluded_due_to_cross_partition_leakage",
        "excluded_by_integrity_rule",
    ]
].to_csv(
    LABEL_FREE_EXCLUSION_PATH,
    index=False,
)


partition_retention.to_csv(
    PARTITION_RETENTION_PATH,
    index=False,
)


decision_payload = {
    "decision": (
        stage2h_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "integrity_rule": (
        INTEGRITY_RULE
    ),
    "continuation_gate": (
        CONTINUATION_GATE
    ),
    "total_idrid_images": int(
        len(
            audit_table
        )
    ),
    "original_duplicate_groups": int(
        len(
            original_duplicate_groups
        )
    ),
    "original_conflicting_groups": int(
        len(
            original_conflicting_groups
        )
    ),
    "original_cross_partition_groups": int(
        len(
            original_cross_partition_groups
        )
    ),
    "canonical_duplicate_groups": int(
        len(
            canonical_duplicate_groups
        )
    ),
    "canonical_conflicting_groups": int(
        len(
            canonical_conflicting_groups
        )
    ),
    "canonical_cross_partition_groups": int(
        len(
            canonical_cross_partition_groups
        )
    ),
    "connected_components": int(
        len(
            component_summary
        )
    ),
    "ambiguous_label_components": int(
        len(
            ambiguous_component_ids
        )
    ),
    "cross_partition_components": int(
        len(
            cross_partition_component_ids
        )
    ),
    "excluded_components": int(
        len(
            excluded_component_ids
        )
    ),
    "excluded_images": int(
        len(
            excluded_rows
        )
    ),
    "retained_images": int(
        len(
            eligible_rows
        )
    ),
    "overall_retained_fraction": (
        overall_retained_fraction
    ),
    "minimum_partition_retained_fraction": (
        minimum_partition_retained_fraction
    ),
    "overall_gate_pass": (
        overall_gate_pass
    ),
    "partition_gate_pass": (
        partition_gate_pass
    ),
    "model_performance_observed": False,
    "sealed_evaluation_performance_observed": False,
    "individual_sealed_labels_displayed": False,
    "full_labelled_audit_path": str(
        FULL_LABELLED_AUDIT_PATH
    ),
    "label_free_exclusion_manifest_path": str(
        LABEL_FREE_EXCLUSION_PATH
    ),
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# IDRiD Exact-Duplicate Integrity Audit",
    "",
    f"Decision: `{stage2h_decision}`",
    "",
    interpretation,
    "",
    "## Frozen rule",
    "",
    INTEGRITY_RULE,
    "",
    "## Aggregate impact",
    "",
    (
        f"- Original images: "
        f"{len(audit_table)}"
    ),
    (
        f"- Excluded components: "
        f"{len(excluded_component_ids)}"
    ),
    (
        f"- Excluded images: "
        f"{len(excluded_rows)}"
    ),
    (
        f"- Retained images: "
        f"{len(eligible_rows)}"
    ),
    (
        f"- Overall retained fraction: "
        f"{overall_retained_fraction:.6f}"
    ),
    "",
    "## Performance boundary",
    "",
    "- Model performance observed: `False`",
    (
        "- Sealed-evaluation performance "
        "observed: `False`"
    ),
    (
        "- Individual sealed labels "
        "displayed: `False`"
    ),
    "",
]


REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


for protected_path in [
    FULL_LABELLED_AUDIT_PATH,
    COMPONENT_SUMMARY_PATH,
]:
    try:
        os.chmod(
            protected_path,
            0o400,
        )
    except Exception:
        pass


# ============================================================
# 9. Keep repaired objects available for finalisation
# ============================================================

idrid_duplicate_component_audit = (
    audit_table
)

idrid_integrity_component_summary = (
    component_summary
)

idrid_integrity_eligible_image_ids = set(
    eligible_rows[
        "image_id"
    ]
)

idrid_integrity_excluded_image_ids = set(
    excluded_rows[
        "image_id"
    ]
)

idrid_integrity_excluded_component_ids = (
    excluded_component_ids
)


# ============================================================
# 10. Display aggregate information only
# ============================================================

summary_table = pd.DataFrame(
    [
        {
            "metric": (
                "Original-hash duplicate groups"
            ),
            "value": int(
                len(
                    original_duplicate_groups
                )
            ),
        },
        {
            "metric": (
                "Original-hash conflicting groups"
            ),
            "value": int(
                len(
                    original_conflicting_groups
                )
            ),
        },
        {
            "metric": (
                "Original-hash cross-partition groups"
            ),
            "value": int(
                len(
                    original_cross_partition_groups
                )
            ),
        },
        {
            "metric": (
                "Canonical-hash duplicate groups"
            ),
            "value": int(
                len(
                    canonical_duplicate_groups
                )
            ),
        },
        {
            "metric": (
                "Canonical-hash conflicting groups"
            ),
            "value": int(
                len(
                    canonical_conflicting_groups
                )
            ),
        },
        {
            "metric": (
                "Canonical-hash cross-partition groups"
            ),
            "value": int(
                len(
                    canonical_cross_partition_groups
                )
            ),
        },
        {
            "metric": (
                "Excluded connected components"
            ),
            "value": int(
                len(
                    excluded_component_ids
                )
            ),
        },
        {
            "metric": (
                "Images excluded"
            ),
            "value": int(
                len(
                    excluded_rows
                )
            ),
        },
        {
            "metric": (
                "Images retained"
            ),
            "value": int(
                len(
                    eligible_rows
                )
            ),
        },
        {
            "metric": (
                "Overall retained fraction"
            ),
            "value": (
                overall_retained_fraction
            ),
        },
    ]
)


print(
    "\n================ IDRiD INTEGRITY "
    "SUMMARY ================"
)

display(
    summary_table
)


print(
    "\n================ IDRiD PARTITION "
    "RETENTION ================"
)

display(
    partition_retention
)


component_reason_summary = pd.DataFrame(
    [
        {
            "exclusion_reason": (
                "Label conflict only"
            ),
            "components": int(
                (
                    component_summary[
                        "ambiguous_label_component"
                    ]
                    &
                    ~component_summary[
                        "cross_partition_component"
                    ]
                ).sum()
            ),
        },
        {
            "exclusion_reason": (
                "Cross-partition leakage only"
            ),
            "components": int(
                (
                    ~component_summary[
                        "ambiguous_label_component"
                    ]
                    &
                    component_summary[
                        "cross_partition_component"
                    ]
                ).sum()
            ),
        },
        {
            "exclusion_reason": (
                "Both label conflict and leakage"
            ),
            "components": int(
                (
                    component_summary[
                        "ambiguous_label_component"
                    ]
                    &
                    component_summary[
                        "cross_partition_component"
                    ]
                ).sum()
            ),
        },
    ]
)


print(
    "\n================ EXCLUSION REASON "
    "SUMMARY ================"
)

display(
    component_reason_summary
)


print(
    "\n================ STAGE 2R-H DECISION "
    "================"
)

print("Decision:")
print(
    stage2h_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nIndividual sealed labels displayed:")
print(False)

print("\nModel performance observed:")
print(False)

print("\nSealed evaluation performance observed:")
print(False)

print(
    "\nIDRiD exact-duplicate integrity audit "
    "completed and sealed."
)

================ STAGE 2R-H IDRiD INTEGRITY AUDIT ================

Frozen integrity rule:
Construct connected components using exact original-image SHA-256 equality and exact canonical-image SHA-256 equality. Exclude an entire component if it contains more than one official DR grade or if it spans both the official development and sealed-evaluation partitions. Label-consistent duplicate components contained wholly within one partition remain eligible.

Frozen continuation gate:
Proceed only if at least 95% of the entire IDRiD cohort remains and at least 95% of each official partition remains after the deterministic exclusion.

IDRiD rows available:
516

Model performance observed:
False

Sealed evaluation performance observed:
False

================ IDRiD INTEGRITY SUMMARY ================


,metric,value
0,Original-hash duplicate groups,6.000000
1,Original-hash conflicting groups,3.000000
2,Original-hash cross-partition groups,1.000000
3,Canonical-hash duplicate groups,6.000000
4,Canonical-hash conflicting groups,3.000000
5,Canonical-hash cross-partition groups,1.000000
6,Excluded connected components,3.000000
7,Images excluded,6.000000
8,Images retained,510.000000
9,Overall retained fraction,0.988372



================ IDRiD PARTITION RETENTION ================


,official_partition,original_images,excluded_images,retained_images,retained_fraction
0,development,413,5,408,0.987893
1,sealed_evaluation,103,1,102,0.990291



================ EXCLUSION REASON SUMMARY ================


,exclusion_reason,components
0,Label conflict only,2
1,Cross-partition leakage only,0
2,Both label conflict and leakage,1



================ STAGE 2R-H DECISION ================
Decision:
PASS_EXCLUDE_IDRID_CONFLICT_AND_LEAKAGE_COMPONENTS_COMPLETE_STAGE2R

Interpretation:
IDRiD contains exact duplicate-image components with discordant official grades and at least one component crossing the official training/testing boundary. The frozen deterministic rule excludes every component with either label ambiguity or cross-partition leakage. The remaining overall cohort and both official partitions satisfy the pre-frozen retention gates.

Authorised next step:
COMPLETE_STAGE2R_USING_IDRID_INTEGRITY_ELIGIBLE_IMAGES

Individual sealed labels displayed:
False

Model performance observed:
False

Sealed evaluation performance observed:
False

IDRiD exact-duplicate integrity audit completed and sealed.


In [22]:
#@title 02R-I. Finalise and seal Stage 2R after APTOS and IDRiD integrity exclusions

from pathlib import Path
from datetime import datetime, timezone

from sklearn.model_selection import StratifiedShuffleSplit
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import shutil
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen finalisation protocol
# ============================================================

RANDOM_SEED = 20260720
APTOS_EVALUATION_FRACTION = 0.30

APTOS_ORIGINAL_IMAGES = 3662
APTOS_EXCLUDED_IMAGES = 62
APTOS_ELIGIBLE_IMAGES = 3600

IDRID_ORIGINAL_IMAGES = 516
IDRID_EXCLUDED_IMAGES = 6
IDRID_ELIGIBLE_IMAGES = 510
IDRID_DEVELOPMENT_IMAGES = 408
IDRID_SEALED_EVALUATION_IMAGES = 102

CANONICAL_SIZE = 768
CANONICAL_JPEG_QUALITY = 92
CANONICAL_JPEG_SUBSAMPLING = 0
BLACK_BORDER_THRESHOLD = 10

ENDPOINT_ID = "MODERATE_OR_WORSE_DR_GRADE_GE_2"
POSITIVE_CLASS_RULE = "original DR grade >= 2"

FINALISATION_BOUNDARY = (
    "Use only the APTOS and IDRiD images authorised by the two "
    "pre-performance integrity audits. APTOS is split at connected-"
    "component level so exact original-image and canonical-image "
    "duplicates cannot cross development and sealed evaluation. "
    "IDRiD retains its official development/evaluation assignment "
    "after excluding every component with label conflict or official-"
    "split leakage. Sealed-evaluation label values must not be "
    "displayed, summarised, or inserted into modelling manifests."
)


# ============================================================
# 1. Verify required current-runtime objects
# ============================================================

required_runtime_variables = [
    "aptos_verified",
    "idrid_verified",
    "aptos_duplicate_component_audit",
    "idrid_duplicate_component_audit",
    "LOCAL_APTOS_CANONICAL",
    "LOCAL_IDRID_CANONICAL",
    "archive_path",
]

missing_runtime_variables = [
    name
    for name in required_runtime_variables
    if name not in globals()
]

if missing_runtime_variables:
    raise RuntimeError(
        "Required Stage 2R in-memory objects are missing:\n"
        + "\n".join(missing_runtime_variables)
        + "\n\nThe Colab runtime may have restarted. Do not rerun "
          "the download or image-processing cells blindly."
    )

assert isinstance(
    aptos_verified,
    pd.DataFrame,
)

assert isinstance(
    idrid_verified,
    pd.DataFrame,
)

assert isinstance(
    aptos_duplicate_component_audit,
    pd.DataFrame,
)

assert isinstance(
    idrid_duplicate_component_audit,
    pd.DataFrame,
)

assert (
    len(aptos_verified)
    ==
    APTOS_ORIGINAL_IMAGES
)

assert (
    len(idrid_verified)
    ==
    IDRID_ORIGINAL_IMAGES
)

LOCAL_APTOS_CANONICAL = Path(
    LOCAL_APTOS_CANONICAL
)

LOCAL_IDRID_CANONICAL = Path(
    LOCAL_IDRID_CANONICAL
)

archive_path = Path(
    archive_path
)

assert LOCAL_APTOS_CANONICAL.is_dir(), (
    LOCAL_APTOS_CANONICAL
)

assert LOCAL_IDRID_CANONICAL.is_dir(), (
    LOCAL_IDRID_CANONICAL
)

assert archive_path.is_file(), (
    archive_path
)

required_verified_columns = {
    "dataset",
    "image_id",
    "original_dr_grade",
    "original_sha256",
    "canonical_sha256",
    "canonical_filename",
    "original_filename",
    "original_size_bytes",
    "canonical_size_bytes",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
}

assert required_verified_columns.issubset(
    aptos_verified.columns
), (
    "APTOS verified table is missing columns: "
    f"{sorted(required_verified_columns - set(aptos_verified.columns))}"
)

assert required_verified_columns.union(
    {
        "official_partition",
    }
).issubset(
    idrid_verified.columns
), (
    "IDRiD verified table is missing columns: "
    f"{sorted(required_verified_columns.union({'official_partition'}) - set(idrid_verified.columns))}"
)

print(
    "================ STAGE 2R-I FINALISATION "
    "================"
)

print("\nFinalisation boundary:")
print(FINALISATION_BOUNDARY)

print("\nAPTOS rows before exclusion:")
print(
    len(aptos_verified)
)

print("\nIDRiD rows before exclusion:")
print(
    len(idrid_verified)
)

print("\nModel performance observed:")
print(False)

print("\nSealed evaluation performance observed:")
print(False)


# ============================================================
# 2. Resolve Drive and project paths
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

TEST_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
    / "Prospective_Retinal_Blind_Test_v0.1"
)

PROTOCOL_ROOT = (
    TEST_ROOT
    / "00_Protocol"
)

CANONICAL_ROOT = (
    TEST_ROOT
    / "01_Canonical_Target_Images"
)

DEVELOPMENT_ROOT = (
    TEST_ROOT
    / "02_Development"
)

SEALED_EVALUATION_ROOT = (
    TEST_ROOT
    / "03_Sealed_Evaluation"
)

QUARANTINE_ROOT = (
    TEST_ROOT
    / "99_Label_Quarantine_DO_NOT_OPEN_BEFORE_STAGE6"
)

ORIGINAL_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "00_Original_Source_Label_Files"
)

SEALED_LABEL_ROOT = (
    QUARANTINE_ROOT
    / "01_Sealed_Evaluation_Labels"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)


for directory in [
    PROTOCOL_ROOT,
    CANONICAL_ROOT,
    DEVELOPMENT_ROOT,
    SEALED_EVALUATION_ROOT,
    ORIGINAL_LABEL_ROOT,
    SEALED_LABEL_ROOT,
    STAGE2_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


APTOS_FINAL_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019"
)

IDRID_FINAL_ROOT = (
    CANONICAL_ROOT
    / "IDRiD"
)

APTOS_STAGING_ROOT = (
    CANONICAL_ROOT
    / "APTOS_2019__STAGING"
)

IDRID_STAGING_ROOT = (
    CANONICAL_ROOT
    / "IDRiD__STAGING"
)

STAGE2F_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2F_APTOS_Duplicate_Label_Conflict_Decision_v0.1.json"
)

STAGE2H_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2H_IDRiD_Exact_Duplicate_Integrity_Decision_v0.1.json"
)

FINAL_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)


for required_path in [
    STAGE2F_DECISION_PATH,
    STAGE2H_DECISION_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with open(
    STAGE2F_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage2f_decision = json.load(
        file
    )


with open(
    STAGE2H_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage2h_decision = json.load(
        file
    )


assert (
    stage2f_decision[
        "decision"
    ]
    ==
    "PASS_EXCLUDE_AMBIGUOUS_DUPLICATE_COMPONENTS_AND_COMPLETE_STAGE2R"
)

assert (
    stage2h_decision[
        "decision"
    ]
    ==
    "PASS_EXCLUDE_IDRID_CONFLICT_AND_LEAKAGE_COMPONENTS_COMPLETE_STAGE2R"
)

assert (
    int(
        stage2f_decision[
            "excluded_images"
        ]
    )
    ==
    APTOS_EXCLUDED_IMAGES
)

assert (
    int(
        stage2f_decision[
            "retained_images"
        ]
    )
    ==
    APTOS_ELIGIBLE_IMAGES
)

assert (
    int(
        stage2h_decision[
            "excluded_images"
        ]
    )
    ==
    IDRID_EXCLUDED_IMAGES
)

assert (
    int(
        stage2h_decision[
            "retained_images"
        ]
    )
    ==
    IDRID_ELIGIBLE_IMAGES
)


print(
    "\nImported APTOS integrity decision:"
)

print(
    stage2f_decision[
        "decision"
    ]
)

print(
    "\nImported IDRiD integrity decision:"
)

print(
    stage2h_decision[
        "decision"
    ]
)


# ============================================================
# 3. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def bytes_to_gib(value):
    return float(
        value
        /
        (1024 ** 3)
    )


def make_writable(path):
    path = Path(
        path
    )

    if path.exists():
        try:
            os.chmod(
                path,
                0o600,
            )

        except Exception:
            pass


def make_read_only(path):
    path = Path(
        path
    )

    try:
        os.chmod(
            path,
            0o400,
        )

        return True

    except Exception:
        return False


def json_default(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    raise TypeError(
        f"Unsupported JSON type: "
        f"{type(value)}"
    )


# ============================================================
# 4. Apply both frozen integrity audits
# ============================================================

aptos_audit_columns = {
    "image_id",
    "duplicate_component_id",
    "excluded_due_to_ambiguous_duplicate_labels",
    "eligible_after_integrity_audit",
}

idrid_audit_columns = {
    "image_id",
    "duplicate_component_id",
    "excluded_due_to_label_conflict",
    "excluded_due_to_cross_partition_leakage",
    "excluded_by_integrity_rule",
    "eligible_after_integrity_audit",
}


assert aptos_audit_columns.issubset(
    aptos_duplicate_component_audit.columns
)

assert idrid_audit_columns.issubset(
    idrid_duplicate_component_audit.columns
)


aptos_component_map = (
    aptos_duplicate_component_audit[
        [
            "image_id",
            "duplicate_component_id",
            "excluded_due_to_ambiguous_duplicate_labels",
            "eligible_after_integrity_audit",
        ]
    ]
    .copy()
)


idrid_component_map = (
    idrid_duplicate_component_audit[
        [
            "image_id",
            "duplicate_component_id",
            "excluded_due_to_label_conflict",
            "excluded_due_to_cross_partition_leakage",
            "excluded_by_integrity_rule",
            "eligible_after_integrity_audit",
        ]
    ]
    .copy()
)


assert aptos_component_map[
    "image_id"
].is_unique

assert idrid_component_map[
    "image_id"
].is_unique


aptos_base = aptos_verified.drop(
    columns=[
        "prospective_partition",
    ],
    errors="ignore",
).copy()


idrid_base = idrid_verified.drop(
    columns=[
        "prospective_partition",
    ],
    errors="ignore",
).copy()


aptos_all = aptos_base.merge(
    aptos_component_map,
    on="image_id",
    how="inner",
    validate="one_to_one",
)


idrid_all = idrid_base.merge(
    idrid_component_map,
    on="image_id",
    how="inner",
    validate="one_to_one",
)


assert (
    len(aptos_all)
    ==
    APTOS_ORIGINAL_IMAGES
)

assert (
    len(idrid_all)
    ==
    IDRID_ORIGINAL_IMAGES
)


aptos_eligible = (
    aptos_all[
        aptos_all[
            "eligible_after_integrity_audit"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


aptos_excluded = (
    aptos_all[
        ~aptos_all[
            "eligible_after_integrity_audit"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


idrid_eligible = (
    idrid_all[
        idrid_all[
            "eligible_after_integrity_audit"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


idrid_excluded = (
    idrid_all[
        ~idrid_all[
            "eligible_after_integrity_audit"
        ]
    ]
    .copy()
    .reset_index(
        drop=True
    )
)


assert (
    len(aptos_eligible)
    ==
    APTOS_ELIGIBLE_IMAGES
)

assert (
    len(aptos_excluded)
    ==
    APTOS_EXCLUDED_IMAGES
)

assert (
    len(idrid_eligible)
    ==
    IDRID_ELIGIBLE_IMAGES
)

assert (
    len(idrid_excluded)
    ==
    IDRID_EXCLUDED_IMAGES
)


assert not aptos_eligible[
    "excluded_due_to_ambiguous_duplicate_labels"
].any()

assert not idrid_eligible[
    "excluded_by_integrity_rule"
].any()


# ============================================================
# 5. Final APTOS component-level deterministic split
# ============================================================

aptos_component_grade_counts = (
    aptos_eligible
    .groupby(
        "duplicate_component_id"
    )[
        "original_dr_grade"
    ]
    .nunique()
)


assert not (
    aptos_component_grade_counts
    >
    1
).any(), (
    "An eligible APTOS component still "
    "contains conflicting grades."
)


aptos_component_table = (
    aptos_eligible
    .groupby(
        "duplicate_component_id",
        as_index=False,
    )
    .agg(
        original_dr_grade=(
            "original_dr_grade",
            "first",
        ),
        component_images=(
            "image_id",
            "size",
        ),
    )
    .sort_values(
        "duplicate_component_id"
    )
    .reset_index(
        drop=True
    )
)


components_per_grade = (
    aptos_component_table[
        "original_dr_grade"
    ]
    .value_counts()
)


assert (
    components_per_grade.min()
    >=
    2
)


splitter = StratifiedShuffleSplit(
    n_splits=1,
    test_size=(
        APTOS_EVALUATION_FRACTION
    ),
    random_state=(
        RANDOM_SEED
    ),
)


_, evaluation_component_indices = next(
    splitter.split(
        X=np.zeros(
            len(
                aptos_component_table
            )
        ),
        y=aptos_component_table[
            "original_dr_grade"
        ].to_numpy(),
    )
)


evaluation_component_ids = set(
    aptos_component_table.iloc[
        evaluation_component_indices
    ][
        "duplicate_component_id"
    ]
)


aptos_eligible[
    "prospective_partition"
] = np.where(
    aptos_eligible[
        "duplicate_component_id"
    ].isin(
        evaluation_component_ids
    ),
    "sealed_evaluation",
    "development",
)


assert not (
    aptos_eligible
    .groupby(
        "duplicate_component_id"
    )[
        "prospective_partition"
    ]
    .nunique()
    >
    1
).any()


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grade_conflicts = (
        aptos_eligible
        .groupby(
            hash_column
        )[
            "original_dr_grade"
        ]
        .nunique()
    )

    split_crossings = (
        aptos_eligible
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
    )

    assert not (
        grade_conflicts
        >
        1
    ).any()

    assert not (
        split_crossings
        >
        1
    ).any()


# ============================================================
# 6. Final IDRiD cohort with official partitions retained
# ============================================================

idrid_eligible[
    "prospective_partition"
] = (
    idrid_eligible[
        "official_partition"
    ]
)


assert (
    (
        idrid_eligible[
            "prospective_partition"
        ]
        ==
        "development"
    ).sum()
    ==
    IDRID_DEVELOPMENT_IMAGES
)

assert (
    (
        idrid_eligible[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ).sum()
    ==
    IDRID_SEALED_EVALUATION_IMAGES
)


for hash_column in [
    "original_sha256",
    "canonical_sha256",
]:
    grade_conflicts = (
        idrid_eligible
        .groupby(
            hash_column
        )[
            "original_dr_grade"
        ]
        .nunique()
    )

    split_crossings = (
        idrid_eligible
        .groupby(
            hash_column
        )[
            "prospective_partition"
        ]
        .nunique()
    )

    assert not (
        grade_conflicts
        >
        1
    ).any()

    assert not (
        split_crossings
        >
        1
    ).any()


# ============================================================
# 7. Cross-dataset overlap and endpoint construction
# ============================================================

original_cross_dataset_overlap = set(
    aptos_eligible[
        "original_sha256"
    ]
).intersection(
    set(
        idrid_eligible[
            "original_sha256"
        ]
    )
)


canonical_cross_dataset_overlap = set(
    aptos_eligible[
        "canonical_sha256"
    ]
).intersection(
    set(
        idrid_eligible[
            "canonical_sha256"
        ]
    )
)


assert not original_cross_dataset_overlap
assert not canonical_cross_dataset_overlap


for dataframe in [
    aptos_eligible,
    idrid_eligible,
]:
    dataframe[
        "moderate_or_worse_dr"
    ] = (
        dataframe[
            "original_dr_grade"
        ]
        >=
        2
    ).astype(int)


# ============================================================
# 8. Verify local canonical files and Drive capacity
# ============================================================

aptos_eligible[
    "local_canonical_path"
] = (
    aptos_eligible[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            LOCAL_APTOS_CANONICAL
            /
            filename
        )
    )
)


idrid_eligible[
    "local_canonical_path"
] = (
    idrid_eligible[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            LOCAL_IDRID_CANONICAL
            /
            filename
        )
    )
)


assert aptos_eligible[
    "local_canonical_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


assert idrid_eligible[
    "local_canonical_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


aptos_canonical_bytes = int(
    aptos_eligible[
        "local_canonical_path"
    ]
    .map(
        lambda value: Path(
            value
        ).stat().st_size
    )
    .sum()
)


idrid_canonical_bytes = int(
    idrid_eligible[
        "local_canonical_path"
    ]
    .map(
        lambda value: Path(
            value
        ).stat().st_size
    )
    .sum()
)


canonical_total_bytes = (
    aptos_canonical_bytes
    +
    idrid_canonical_bytes
)


existing_successful_decision = False


if FINAL_DECISION_PATH.is_file():
    try:
        with open(
            FINAL_DECISION_PATH,
            "r",
            encoding="utf-8",
        ) as file:
            previous_final_decision = json.load(
                file
            )

        existing_successful_decision = (
            previous_final_decision.get(
                "decision"
            )
            ==
            "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_AND_LABEL_QUARANTINE_ADVANCE_TO_DEVELOPMENT_RECOVERABILITY_GATE"
        )

    except Exception:
        existing_successful_decision = False


if existing_successful_decision:
    raise RuntimeError(
        "Stage 2R already has a sealed successful "
        "final decision. Do not overwrite it without "
        "a versioned protocol amendment."
    )


for stale_root in [
    APTOS_STAGING_ROOT,
    IDRID_STAGING_ROOT,
]:
    if stale_root.exists():
        shutil.rmtree(
            stale_root
        )


# No successful final decision exists, so these are stale partial outputs.
for stale_final_root in [
    APTOS_FINAL_ROOT,
    IDRID_FINAL_ROOT,
]:
    if stale_final_root.exists():
        shutil.rmtree(
            stale_final_root
        )


current_drive_free = (
    shutil.disk_usage(
        DRIVE_ROOT
    ).free
)


print(
    "\n================ CANONICAL STORAGE AUDIT "
    "================"
)

print(
    "Eligible APTOS canonical size:",
    f"{bytes_to_gib(aptos_canonical_bytes):.2f} GiB",
)

print(
    "Eligible IDRiD canonical size:",
    f"{bytes_to_gib(idrid_canonical_bytes):.2f} GiB",
)

print(
    "Total canonical size:",
    f"{bytes_to_gib(canonical_total_bytes):.2f} GiB",
)

print(
    "Current Drive free space:",
    f"{bytes_to_gib(current_drive_free):.2f} GiB",
)


assert (
    current_drive_free
    >
    canonical_total_bytes
    +
    512
    *
    1024 ** 2
), (
    "The final eligible canonical cohort "
    "exceeds the remaining Drive capacity."
)


# ============================================================
# 9. Copy only eligible images to Drive staging
# ============================================================

APTOS_STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

IDRID_STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


print(
    "\n================ DRIVE STAGING COPY "
    "================"
)


for _, row in tqdm(
    aptos_eligible.iterrows(),
    total=len(
        aptos_eligible
    ),
    desc="Copying eligible APTOS images",
):
    shutil.copy2(
        Path(
            row[
                "local_canonical_path"
            ]
        ),
        (
            APTOS_STAGING_ROOT
            /
            row[
                "canonical_filename"
            ]
        ),
    )


for _, row in tqdm(
    idrid_eligible.iterrows(),
    total=len(
        idrid_eligible
    ),
    desc="Copying eligible IDRiD images",
):
    shutil.copy2(
        Path(
            row[
                "local_canonical_path"
            ]
        ),
        (
            IDRID_STAGING_ROOT
            /
            row[
                "canonical_filename"
            ]
        ),
    )


assert (
    len(
        list(
            APTOS_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    APTOS_ELIGIBLE_IMAGES
)


assert (
    len(
        list(
            IDRID_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_ELIGIBLE_IMAGES
)


# ============================================================
# 10. Verify every Drive copy by SHA-256
# ============================================================

print(
    "\n================ DRIVE COPY VERIFICATION "
    "================"
)


for _, row in tqdm(
    aptos_eligible.iterrows(),
    total=len(
        aptos_eligible
    ),
    desc="Verifying APTOS Drive hashes",
):
    copied_path = (
        APTOS_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


for _, row in tqdm(
    idrid_eligible.iterrows(),
    total=len(
        idrid_eligible
    ),
    desc="Verifying IDRiD Drive hashes",
):
    copied_path = (
        IDRID_STAGING_ROOT
        /
        row[
            "canonical_filename"
        ]
    )

    assert copied_path.is_file()

    assert (
        sha256_file(
            copied_path
        )
        ==
        row[
            "canonical_sha256"
        ]
    )


APTOS_STAGING_ROOT.rename(
    APTOS_FINAL_ROOT
)

IDRID_STAGING_ROOT.rename(
    IDRID_FINAL_ROOT
)


# ============================================================
# 11. Attach final persistent image paths
# ============================================================

aptos_eligible[
    "canonical_image_path"
] = (
    aptos_eligible[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            APTOS_FINAL_ROOT
            /
            filename
        )
    )
)


idrid_eligible[
    "canonical_image_path"
] = (
    idrid_eligible[
        "canonical_filename"
    ]
    .map(
        lambda filename: str(
            IDRID_FINAL_ROOT
            /
            filename
        )
    )
)


assert aptos_eligible[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


assert idrid_eligible[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


# ============================================================
# 12. Construct final partitions
# ============================================================

aptos_development = (
    aptos_eligible[
        aptos_eligible[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


aptos_evaluation = (
    aptos_eligible[
        aptos_eligible[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_development = (
    idrid_eligible[
        idrid_eligible[
            "prospective_partition"
        ]
        ==
        "development"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


idrid_evaluation = (
    idrid_eligible[
        idrid_eligible[
            "prospective_partition"
        ]
        ==
        "sealed_evaluation"
    ]
    .copy()
    .sort_values(
        "image_id"
    )
)


assert (
    len(aptos_development)
    +
    len(aptos_evaluation)
    ==
    APTOS_ELIGIBLE_IMAGES
)

assert (
    len(idrid_development)
    ==
    IDRID_DEVELOPMENT_IMAGES
)

assert (
    len(idrid_evaluation)
    ==
    IDRID_SEALED_EVALUATION_IMAGES
)


aptos_actual_evaluation_fraction = float(
    len(
        aptos_evaluation
    )
    /
    APTOS_ELIGIBLE_IMAGES
)


# ============================================================
# 13. Output paths
# ============================================================

APTOS_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "APTOS_2019_Development_Labels_v0.1.csv"
)

IDRID_DEVELOPMENT_LABEL_PATH = (
    DEVELOPMENT_ROOT
    / "IDRiD_Development_Labels_v0.1.csv"
)

APTOS_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "APTOS_2019_Sealed_Evaluation_Manifest_v0.1.csv"
)

IDRID_EVALUATION_MANIFEST_PATH = (
    SEALED_EVALUATION_ROOT
    / "IDRiD_Sealed_Evaluation_Manifest_v0.1.csv"
)

APTOS_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "APTOS_2019_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

IDRID_SEALED_LABEL_PATH = (
    SEALED_LABEL_ROOT
    / "IDRiD_Sealed_Evaluation_Labels_DO_NOT_OPEN_v0.1.csv"
)

APTOS_ORIGINAL_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "APTOS_2019_Original_train.csv"
)

IDRID_ORIGINAL_TRAIN_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Training_Labels.csv"
)

IDRID_ORIGINAL_TEST_LABEL_PATH = (
    ORIGINAL_LABEL_ROOT
    / "IDRiD_Original_Disease_Grading_Testing_Labels.csv"
)


for required_label_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
]:
    assert required_label_path.is_file(), (
        required_label_path
    )


for output_path in [
    APTOS_DEVELOPMENT_LABEL_PATH,
    IDRID_DEVELOPMENT_LABEL_PATH,
    APTOS_EVALUATION_MANIFEST_PATH,
    IDRID_EVALUATION_MANIFEST_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_writable(
        output_path
    )


# ============================================================
# 14. Save development labels and sealed evaluation artifacts
# ============================================================

development_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_dr_grade",
    "moderate_or_worse_dr",
    "duplicate_component_id",
    "original_sha256",
    "canonical_sha256",
]


evaluation_manifest_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "duplicate_component_id",
    "original_sha256",
    "canonical_sha256",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


sealed_label_columns = [
    "dataset",
    "image_id",
    "original_dr_grade",
    "moderate_or_worse_dr",
]


aptos_development[
    development_columns
].to_csv(
    APTOS_DEVELOPMENT_LABEL_PATH,
    index=False,
)


idrid_development[
    development_columns
].to_csv(
    IDRID_DEVELOPMENT_LABEL_PATH,
    index=False,
)


aptos_evaluation[
    evaluation_manifest_columns
].to_csv(
    APTOS_EVALUATION_MANIFEST_PATH,
    index=False,
)


idrid_evaluation[
    evaluation_manifest_columns
].to_csv(
    IDRID_EVALUATION_MANIFEST_PATH,
    index=False,
)


aptos_evaluation[
    sealed_label_columns
].to_csv(
    APTOS_SEALED_LABEL_PATH,
    index=False,
)


idrid_evaluation[
    sealed_label_columns
].to_csv(
    IDRID_SEALED_LABEL_PATH,
    index=False,
)


for protected_path in [
    APTOS_ORIGINAL_LABEL_PATH,
    IDRID_ORIGINAL_TRAIN_LABEL_PATH,
    IDRID_ORIGINAL_TEST_LABEL_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]:
    make_read_only(
        protected_path
    )


# ============================================================
# 15. Create label-free manifests and integrity records
# ============================================================

label_free_columns = [
    "dataset",
    "image_id",
    "canonical_image_path",
    "prospective_partition",
    "duplicate_component_id",
    "original_sha256",
    "canonical_sha256",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "canonical_size_bytes",
]


label_free_master_manifest = pd.concat(
    [
        aptos_eligible[
            label_free_columns
        ],
        idrid_eligible[
            label_free_columns
        ],
    ],
    ignore_index=True,
)


assert (
    "original_dr_grade"
    not in
    label_free_master_manifest.columns
)

assert (
    "moderate_or_worse_dr"
    not in
    label_free_master_manifest.columns
)


integrity_columns = [
    "dataset",
    "image_id",
    "original_filename",
    "canonical_filename",
    "original_sha256",
    "canonical_sha256",
    "original_size_bytes",
    "canonical_size_bytes",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
]


final_integrity_manifest = pd.concat(
    [
        aptos_eligible[
            integrity_columns
        ],
        idrid_eligible[
            integrity_columns
        ],
    ],
    ignore_index=True,
)


CANONICAL_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Canonical_Fundus_Preprocessing_Protocol_v0.1.json"
)

QUARANTINE_PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Target_Label_Quarantine_Protocol_v0.1.json"
)

MASTER_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_LabelFree_Master_Target_Manifest_v0.1.csv"
)

PARTITION_ASSIGNMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Final_LabelFree_Partition_Assignment_v0.1.csv"
)

INTEGRITY_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

PARTITION_SUMMARY_PATH = (
    STAGE2_ROOT
    / "Stage2_Target_Partition_Summary_v0.1.csv"
)

LABEL_COMMITMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Sealed_Label_Commitments_v0.1.json"
)

FINAL_REPAIR_MANIFEST_PATH = (
    STAGE2_ROOT
    / "Stage2R_Final_Integrity_Exclusion_Manifest_v0.1.json"
)

STAGE2_REPORT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Report_v0.1.md"
)

STAGE2_ENVIRONMENT_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Environment_v0.1.json"
)


canonical_protocol = {
    "protocol_version": "v0.1",
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "black_border_threshold": (
        BLACK_BORDER_THRESHOLD
    ),
    "jpeg_quality": (
        CANONICAL_JPEG_QUALITY
    ),
    "jpeg_subsampling": (
        CANONICAL_JPEG_SUBSAMPLING
    ),
    "resize_filter": "LANCZOS",
    "frozen_before_target_performance": True,
    "must_be_applied_identically_to_sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
}


with open(
    CANONICAL_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        canonical_protocol,
        file,
        indent=2,
    )


quarantine_protocol = {
    "protocol_version": "v0.1",
    "endpoint_id": ENDPOINT_ID,
    "positive_class_rule": (
        POSITIVE_CLASS_RULE
    ),
    "random_seed": (
        RANDOM_SEED
    ),
    "aptos_requested_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "aptos_split_unit": (
        "label-consistent connected duplicate component"
    ),
    "aptos_ambiguous_images_excluded": (
        APTOS_EXCLUDED_IMAGES
    ),
    "idrid_integrity_images_excluded": (
        IDRID_EXCLUDED_IMAGES
    ),
    "development_labels_authorised_from_stage": 3,
    "sealed_evaluation_labels_authorised_stage": 6,
    "sealed_evaluation_manifests_label_free": True,
    "sealed_label_values_displayed": False,
    "idrid_unique_identifier_rule": (
        "<official_partition>__<official_image_stem>"
    ),
    "prohibitions": [
        (
            "Do not open, display, summarise or model "
            "sealed-evaluation labels before Stage 6."
        ),
        (
            "Do not change final partitions after "
            "recoverability or transfer results are observed."
        ),
        (
            "Do not change the canonicalisation protocol "
            "after target performance is observed."
        ),
        (
            "Do not restore excluded APTOS or IDRiD "
            "integrity-failed images to analysis."
        ),
    ],
}


with open(
    QUARANTINE_PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        quarantine_protocol,
        file,
        indent=2,
    )


label_free_master_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False,
)


label_free_master_manifest[
    [
        "dataset",
        "image_id",
        "prospective_partition",
        "duplicate_component_id",
        "original_sha256",
        "canonical_sha256",
    ]
].to_csv(
    PARTITION_ASSIGNMENT_PATH,
    index=False,
)


final_integrity_manifest.to_csv(
    INTEGRITY_MANIFEST_PATH,
    index=False,
)


partition_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "original_images": (
                APTOS_ORIGINAL_IMAGES
            ),
            "excluded_integrity_images": (
                APTOS_EXCLUDED_IMAGES
            ),
            "eligible_images": (
                APTOS_ELIGIBLE_IMAGES
            ),
            "development_images": int(
                len(
                    aptos_development
                )
            ),
            "sealed_evaluation_images": int(
                len(
                    aptos_evaluation
                )
            ),
            "actual_evaluation_fraction": (
                aptos_actual_evaluation_fraction
            ),
            "partition_source": (
                "component-grouped deterministic "
                "grade-stratified split"
            ),
            "sealed_evaluation_labels_displayed": False,
        },
        {
            "dataset": "IDRiD",
            "original_images": (
                IDRID_ORIGINAL_IMAGES
            ),
            "excluded_integrity_images": (
                IDRID_EXCLUDED_IMAGES
            ),
            "eligible_images": (
                IDRID_ELIGIBLE_IMAGES
            ),
            "development_images": int(
                len(
                    idrid_development
                )
            ),
            "sealed_evaluation_images": int(
                len(
                    idrid_evaluation
                )
            ),
            "actual_evaluation_fraction": float(
                len(
                    idrid_evaluation
                )
                /
                IDRID_ELIGIBLE_IMAGES
            ),
            "partition_source": (
                "official IDRiD split after "
                "deterministic integrity exclusion"
            ),
            "sealed_evaluation_labels_displayed": False,
        },
    ]
)


partition_summary.to_csv(
    PARTITION_SUMMARY_PATH,
    index=False,
)


label_commitments = {
    "endpoint_id": ENDPOINT_ID,
    "aptos_sealed_evaluation": {
        "rows": int(
            len(
                aptos_evaluation
            )
        ),
        "path": str(
            APTOS_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            APTOS_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "idrid_sealed_evaluation": {
        "rows": int(
            len(
                idrid_evaluation
            )
        ),
        "path": str(
            IDRID_SEALED_LABEL_PATH
        ),
        "sha256": sha256_file(
            IDRID_SEALED_LABEL_PATH
        ),
        "displayed": False,
        "authorised_open_stage": 6,
    },
    "original_label_files": {
        "aptos_train_sha256": (
            sha256_file(
                APTOS_ORIGINAL_LABEL_PATH
            )
        ),
        "idrid_train_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TRAIN_LABEL_PATH
            )
        ),
        "idrid_test_sha256": (
            sha256_file(
                IDRID_ORIGINAL_TEST_LABEL_PATH
            )
        ),
    },
}


with open(
    LABEL_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        label_commitments,
        file,
        indent=2,
    )


repair_manifest = {
    "aptos": {
        "audit_decision": str(
            STAGE2F_DECISION_PATH
        ),
        "original_images": (
            APTOS_ORIGINAL_IMAGES
        ),
        "excluded_components": int(
            stage2f_decision[
                "ambiguous_connected_components"
            ]
        ),
        "excluded_images": (
            APTOS_EXCLUDED_IMAGES
        ),
        "eligible_images": (
            APTOS_ELIGIBLE_IMAGES
        ),
        "resolution": (
            "Exclude complete connected components "
            "containing discordant grades."
        ),
    },
    "idrid": {
        "audit_decision": str(
            STAGE2H_DECISION_PATH
        ),
        "original_images": (
            IDRID_ORIGINAL_IMAGES
        ),
        "excluded_components": int(
            stage2h_decision[
                "excluded_components"
            ]
        ),
        "excluded_images": (
            IDRID_EXCLUDED_IMAGES
        ),
        "eligible_images": (
            IDRID_ELIGIBLE_IMAGES
        ),
        "development_images": (
            IDRID_DEVELOPMENT_IMAGES
        ),
        "sealed_evaluation_images": (
            IDRID_SEALED_EVALUATION_IMAGES
        ),
        "resolution": (
            "Exclude complete components with grade "
            "conflict or official-split leakage."
        ),
    },
    "model_performance_observed_before_resolution": False,
    "sealed_evaluation_performance_observed_before_resolution": False,
    "computer_sleep_or_lock_caused_failures": False,
}


with open(
    FINAL_REPAIR_MANIFEST_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_manifest,
        file,
        indent=2,
    )


# ============================================================
# 16. Final leakage and label-boundary checks
# ============================================================

aptos_original_cross_split_duplicates = int(
    (
        aptos_eligible
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


aptos_canonical_cross_split_duplicates = int(
    (
        aptos_eligible
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_original_cross_split_duplicates = int(
    (
        idrid_eligible
        .groupby(
            "original_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


idrid_canonical_cross_split_duplicates = int(
    (
        idrid_eligible
        .groupby(
            "canonical_sha256"
        )[
            "prospective_partition"
        ]
        .nunique()
        >
        1
    ).sum()
)


evaluation_manifest_has_no_labels = bool(
    all(
        label_column
        not in pd.read_csv(
            manifest_path,
            nrows=1,
        ).columns
        for manifest_path in [
            APTOS_EVALUATION_MANIFEST_PATH,
            IDRID_EVALUATION_MANIFEST_PATH,
        ]
        for label_column in [
            "original_dr_grade",
            "moderate_or_worse_dr",
        ]
    )
)


assert (
    aptos_original_cross_split_duplicates
    ==
    0
)

assert (
    aptos_canonical_cross_split_duplicates
    ==
    0
)

assert (
    idrid_original_cross_split_duplicates
    ==
    0
)

assert (
    idrid_canonical_cross_split_duplicates
    ==
    0
)

assert (
    len(
        original_cross_dataset_overlap
    )
    ==
    0
)

assert (
    len(
        canonical_cross_dataset_overlap
    )
    ==
    0
)

assert (
    evaluation_manifest_has_no_labels
)


# ============================================================
# 17. Final Stage 2R decision and report
# ============================================================

stage2_decision = (
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_"
    "AND_LABEL_QUARANTINE_ADVANCE_TO_"
    "DEVELOPMENT_RECOVERABILITY_GATE"
)


stage2_interpretation = (
    "APTOS and IDRiD were converted using the identical frozen "
    "768x768 canonicalisation and hash-verified after copying to "
    "Google Drive. APTOS excluded 62 images from 30 connected "
    "duplicate components containing discordant official grades, "
    "retaining 3,600 images. IDRiD excluded six images from three "
    "components with label conflict and/or official-split leakage, "
    "retaining 510 images: 408 development and 102 sealed evaluation. "
    "No exact duplicate crosses a final partition or dataset. Sealed "
    "labels are absent from modelling manifests and committed for "
    "one-time Stage 6 evaluation."
)


decision_payload = {
    "decision": (
        stage2_decision
    ),
    "interpretation": (
        stage2_interpretation
    ),
    "authorised_next_step": (
        "DEVELOPMENT_ONLY_TARGET_RECOVERABILITY_GATE"
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "positive_class_rule": (
        POSITIVE_CLASS_RULE
    ),
    "canonicalisation_protocol": str(
        CANONICAL_PROTOCOL_PATH
    ),
    "quarantine_protocol": str(
        QUARANTINE_PROTOCOL_PATH
    ),
    "aptos_integrity_decision": str(
        STAGE2F_DECISION_PATH
    ),
    "idrid_integrity_decision": str(
        STAGE2H_DECISION_PATH
    ),
    "aptos": {
        "original_images": (
            APTOS_ORIGINAL_IMAGES
        ),
        "excluded_images": (
            APTOS_EXCLUDED_IMAGES
        ),
        "eligible_images": (
            APTOS_ELIGIBLE_IMAGES
        ),
        "development_images": int(
            len(
                aptos_development
            )
        ),
        "sealed_evaluation_images": int(
            len(
                aptos_evaluation
            )
        ),
        "actual_evaluation_fraction": (
            aptos_actual_evaluation_fraction
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                aptos_canonical_bytes
            )
        ),
        "original_cross_split_duplicates": (
            aptos_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            aptos_canonical_cross_split_duplicates
        ),
    },
    "idrid": {
        "original_images": (
            IDRID_ORIGINAL_IMAGES
        ),
        "excluded_images": (
            IDRID_EXCLUDED_IMAGES
        ),
        "eligible_images": (
            IDRID_ELIGIBLE_IMAGES
        ),
        "development_images": (
            IDRID_DEVELOPMENT_IMAGES
        ),
        "sealed_evaluation_images": (
            IDRID_SEALED_EVALUATION_IMAGES
        ),
        "canonical_size_gib": (
            bytes_to_gib(
                idrid_canonical_bytes
            )
        ),
        "partition_qualified_ids": True,
        "original_cross_split_duplicates": (
            idrid_original_cross_split_duplicates
        ),
        "canonical_cross_split_duplicates": (
            idrid_canonical_cross_split_duplicates
        ),
    },
    "original_cross_dataset_overlap": int(
        len(
            original_cross_dataset_overlap
        )
    ),
    "canonical_cross_dataset_overlap": int(
        len(
            canonical_cross_dataset_overlap
        )
    ),
    "evaluation_manifests_label_free": (
        evaluation_manifest_has_no_labels
    ),
    "sealed_label_values_displayed": False,
    "model_performance_observed": False,
    "sealed_evaluation_performance_observed": False,
    "sealed_label_commitment_path": str(
        LABEL_COMMITMENT_PATH
    ),
    "important_boundary": (
        "Stage 2 establishes controlled acquisition, frozen "
        "canonicalisation, deterministic integrity filtering, "
        "final partitioning, and procedural label quarantine only. "
        "It does not establish target recoverability, source-target "
        "transfer, or CDO validity."
    ),
}


make_writable(
    FINAL_DECISION_PATH
)


with open(
    FINAL_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
        default=json_default,
    )


report_lines = [
    "# Prospective Retinal Blind Test — Stage 2R",
    "",
    "## Final controlled acquisition and label quarantine",
    "",
    f"Decision: `{stage2_decision}`",
    "",
    stage2_interpretation,
    "",
    "## Final cohorts",
    "",
    (
        f"- APTOS eligible images: "
        f"{APTOS_ELIGIBLE_IMAGES}"
    ),
    (
        f"- APTOS development images: "
        f"{len(aptos_development)}"
    ),
    (
        f"- APTOS sealed-evaluation images: "
        f"{len(aptos_evaluation)}"
    ),
    (
        f"- IDRiD eligible images: "
        f"{IDRID_ELIGIBLE_IMAGES}"
    ),
    (
        f"- IDRiD development images: "
        f"{len(idrid_development)}"
    ),
    (
        f"- IDRiD sealed-evaluation images: "
        f"{len(idrid_evaluation)}"
    ),
    "",
    "## Safety boundary",
    "",
    "- Model performance observed: `False`",
    "- Sealed-evaluation performance observed: `False`",
    "- Sealed-evaluation label values displayed: `False`",
    "- Evaluation manifests are label-free: `True`",
    "",
]


STAGE2_REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "random_seed": (
        RANDOM_SEED
    ),
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "aptos_requested_evaluation_fraction": (
        APTOS_EVALUATION_FRACTION
    ),
    "aptos_actual_evaluation_fraction": (
        aptos_actual_evaluation_fraction
    ),
    "test_root": str(
        TEST_ROOT
    ),
}


with open(
    STAGE2_ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 18. Final integrity checks
# ============================================================

required_outputs = [
    CANONICAL_PROTOCOL_PATH,
    QUARANTINE_PROTOCOL_PATH,
    MASTER_MANIFEST_PATH,
    PARTITION_ASSIGNMENT_PATH,
    INTEGRITY_MANIFEST_PATH,
    PARTITION_SUMMARY_PATH,
    LABEL_COMMITMENT_PATH,
    FINAL_REPAIR_MANIFEST_PATH,
    FINAL_DECISION_PATH,
    STAGE2_REPORT_PATH,
    STAGE2_ENVIRONMENT_PATH,
    APTOS_DEVELOPMENT_LABEL_PATH,
    IDRID_DEVELOPMENT_LABEL_PATH,
    APTOS_EVALUATION_MANIFEST_PATH,
    IDRID_EVALUATION_MANIFEST_PATH,
    APTOS_SEALED_LABEL_PATH,
    IDRID_SEALED_LABEL_PATH,
]


for output_path in required_outputs:
    assert output_path.is_file(), (
        output_path
    )


assert (
    len(
        list(
            APTOS_FINAL_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    APTOS_ELIGIBLE_IMAGES
)


assert (
    len(
        list(
            IDRID_FINAL_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    IDRID_ELIGIBLE_IMAGES
)


canonical_summary = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "images": int(
                len(
                    aptos_eligible
                )
            ),
            "unique_image_ids": int(
                aptos_eligible[
                    "image_id"
                ].nunique()
            ),
            "unique_original_sha256": int(
                aptos_eligible[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                aptos_eligible[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    aptos_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
        {
            "dataset": "IDRiD",
            "images": int(
                len(
                    idrid_eligible
                )
            ),
            "unique_image_ids": int(
                idrid_eligible[
                    "image_id"
                ].nunique()
            ),
            "unique_original_sha256": int(
                idrid_eligible[
                    "original_sha256"
                ].nunique()
            ),
            "unique_canonical_sha256": int(
                idrid_eligible[
                    "canonical_sha256"
                ].nunique()
            ),
            "canonical_size_gib": (
                bytes_to_gib(
                    idrid_canonical_bytes
                )
            ),
            "canonical_width": (
                CANONICAL_SIZE
            ),
            "canonical_height": (
                CANONICAL_SIZE
            ),
        },
    ]
)


# ============================================================
# 19. Delete temporary ZIP and local working files
# ============================================================

TEMP_ROOT = (
    archive_path.parent.parent
)


print(
    "\nDeleting temporary APTOS ZIP and "
    "local canonical working files..."
)


if TEMP_ROOT.exists():
    shutil.rmtree(
        TEMP_ROOT
    )


# ============================================================
# 20. Final display
# ============================================================

print(
    "\n================ CANONICAL IMAGE SUMMARY "
    "================"
)

display(
    canonical_summary
)


print(
    "\n================ PARTITION SUMMARY "
    "================"
)

display(
    partition_summary
)


print(
    "\n================ LEAKAGE CHECK "
    "================"
)

print(
    "APTOS original-hash cross-split duplicates:",
    aptos_original_cross_split_duplicates,
)

print(
    "APTOS canonical-hash cross-split duplicates:",
    aptos_canonical_cross_split_duplicates,
)

print(
    "IDRiD original-hash cross-split duplicates:",
    idrid_original_cross_split_duplicates,
)

print(
    "IDRiD canonical-hash cross-split duplicates:",
    idrid_canonical_cross_split_duplicates,
)

print(
    "Original cross-dataset overlap:",
    len(
        original_cross_dataset_overlap
    ),
)

print(
    "Canonical cross-dataset overlap:",
    len(
        canonical_cross_dataset_overlap
    ),
)

print(
    "Evaluation manifests contain labels:",
    not evaluation_manifest_has_no_labels,
)


print(
    "\n================ LABEL QUARANTINE CHECK "
    "================"
)

print(
    "APTOS sealed-label SHA-256:",
    label_commitments[
        "aptos_sealed_evaluation"
    ][
        "sha256"
    ],
)

print(
    "IDRiD sealed-label SHA-256:",
    label_commitments[
        "idrid_sealed_evaluation"
    ][
        "sha256"
    ],
)

print(
    "Sealed-evaluation label values displayed:",
    False,
)


print(
    "\n================ STAGE 2R-I DECISION "
    "================"
)

print("Decision:")
print(
    stage2_decision
)

print("\nInterpretation:")
print(
    stage2_interpretation
)

print("\nAuthorised next step:")
print(
    decision_payload[
        "authorised_next_step"
    ]
)

print(
    "\nTemporary APTOS archive retained:"
)

print(False)

print(
    "\nModel performance observed:"
)

print(False)

print(
    "\nSealed evaluation performance observed:"
)

print(False)

print(
    "\nStage 2R finalisation completed and sealed."
)

================ STAGE 2R-I FINALISATION ================

Finalisation boundary:
Use only the APTOS and IDRiD images authorised by the two pre-performance integrity audits. APTOS is split at connected-component level so exact original-image and canonical-image duplicates cannot cross development and sealed evaluation. IDRiD retains its official development/evaluation assignment after excluding every component with label conflict or official-split leakage. Sealed-evaluation label values must not be displayed, summarised, or inserted into modelling manifests.

APTOS rows before exclusion:
3662

IDRiD rows before exclusion:
516

Model performance observed:
False

Sealed evaluation performance observed:
False

Imported APTOS integrity decision:
PASS_EXCLUDE_AMBIGUOUS_DUPLICATE_COMPONENTS_AND_COMPLETE_STAGE2R

Imported IDRiD integrity decision:
PASS_EXCLUDE_IDRID_CONFLICT_AND_LEAKAGE_COMPONENTS_COMPLETE_STAGE2R

================ CANONICAL STORAGE AUDIT ================
Eligible APTOS canon

Copying eligible APTOS images:   0%|          | 0/3600 [00:00<?, ?it/s]

Copying eligible IDRiD images:   0%|          | 0/510 [00:00<?, ?it/s]


================ DRIVE COPY VERIFICATION ================


Verifying APTOS Drive hashes:   0%|          | 0/3600 [00:00<?, ?it/s]

...


Deleting temporary APTOS ZIP and local canonical working files...

================ CANONICAL IMAGE SUMMARY ================


,dataset,images,unique_image_ids,unique_original_sha256,unique_canonical_sha256,canonical_size_gib,canonical_width,canonical_height
0,APTOS_2019,3600,3600,3504,3504,0.344107,768,768
1,IDRiD,510,510,507,507,0.049590,768,768



================ PARTITION SUMMARY ================


,dataset,original_images,excluded_integrity_images,eligible_images,development_images,sealed_evaluation_images,actual_evaluation_fraction,partition_source,sealed_evaluation_labels_displayed
0,APTOS_2019,3662,62,3600,2520,1080,0.3,component-grouped deterministic grade-stratifi...,False
1,IDRiD,516,6,510,408,102,0.2,official IDRiD split after deterministic integ...,False



================ LEAKAGE CHECK ================
APTOS original-hash cross-split duplicates: 0
APTOS canonical-hash cross-split duplicates: 0
IDRiD original-hash cross-split duplicates: 0
IDRiD canonical-hash cross-split duplicates: 0
Original cross-dataset overlap: 0
Canonical cross-dataset overlap: 0
Evaluation manifests contain labels: False

================ LABEL QUARANTINE CHECK ================
APTOS sealed-label SHA-256: cc4fdbdd0d17e0696bcd569e082a99b266dd177d6c721344dbb7a51d19687c13
IDRiD sealed-label SHA-256: 361c2dfc7b362d60b6ce374922c2373889a8286ce904ada8515641262c8734ae
Sealed-evaluation label values displayed: False

================ STAGE 2R-I DECISION ================
Decision:
PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_AND_LABEL_QUARANTINE_ADVANCE_TO_DEVELOPMENT_RECOVERABILITY_GATE

Interpretation:
APTOS and IDRiD were converted using the identical frozen 768x768 canonicalisation and hash-verified after copying to Google Drive. APTOS excluded 62 images from 30 conn

In [23]:
#@title 03A. Development-only target recoverability gate — sealed evaluation remains untouched

from pathlib import Path
from datetime import datetime, timezone
from torch.utils.data import Dataset, DataLoader
from torchvision.models import (
    resnet50,
    ResNet50_Weights,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
)
from sklearn.exceptions import ConvergenceWarning
from tqdm.auto import tqdm
from PIL import Image, ImageFile

import hashlib
import json
import os
import platform
import random
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


# ============================================================
# 0. Frozen Stage 3A protocol
# ============================================================

RANDOM_SEED = 20260720

N_FOLDS = 5
N_BOOTSTRAP = 2000

PRIMARY_AUC_THRESHOLD = 0.70
PRIMARY_CI_LOWER_THRESHOLD = 0.55

FROZEN_BACKBONE = "ResNet50_IMAGENET1K_V2"
FROZEN_FEATURE_DIMENSION = 2048

LOGISTIC_C = 1.0
LOGISTIC_CLASS_WEIGHT = "balanced"
LOGISTIC_SOLVER = "liblinear"
LOGISTIC_MAX_ITER = 5000

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

POSITIVE_CLASS_RULE = (
    "original DR grade >= 2"
)

ImageFile.LOAD_TRUNCATED_IMAGES = False


SAFETY_BOUNDARY = (
    "Stage 3A reads only the two authorised development-label "
    "files created by Stage 2R. It must not read, list, open, "
    "summarise, hash again, or evaluate either sealed-evaluation "
    "label file. The frozen ResNet50 representation and fixed "
    "linear probe are evaluated using duplicate-component-grouped "
    "out-of-fold prediction only."
)


INTERPRETATION_BOUNDARY = (
    "Passing this gate establishes only that the frozen representation "
    "contains a linearly recoverable signal for the moderate-or-worse "
    "DR endpoint inside the target development cohort. It does not "
    "establish source-to-target transfer, external generalisation, "
    "sealed-evaluation performance, causal validity, or CDO validity."
)


# ============================================================
# 1. Reproducibility
# ============================================================

os.environ[
    "PYTHONHASHSEED"
] = str(
    RANDOM_SEED
)

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(
        RANDOM_SEED
    )

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)


# ============================================================
# 2. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    path = Path(
        path
    )

    digest = hashlib.sha256()

    with path.open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def json_default(value):
    if isinstance(
        value,
        np.generic,
    ):
        return value.item()

    if isinstance(
        value,
        Path,
    ):
        return str(
            value
        )

    raise TypeError(
        f"Unsupported JSON type: "
        f"{type(value)}"
    )


def bootstrap_auc(
    labels,
    probabilities,
    n_bootstrap,
    seed,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    assert (
        len(labels)
        ==
        len(probabilities)
    )

    rng = np.random.default_rng(
        seed
    )

    bootstrap_values = []

    number_of_samples = len(
        labels
    )

    for _ in range(
        n_bootstrap
    ):
        indices = rng.integers(
            low=0,
            high=number_of_samples,
            size=number_of_samples,
        )

        sampled_labels = labels[
            indices
        ]

        if np.unique(
            sampled_labels
        ).size < 2:
            continue

        sampled_probabilities = probabilities[
            indices
        ]

        bootstrap_values.append(
            roc_auc_score(
                sampled_labels,
                sampled_probabilities,
            )
        )

    assert len(
        bootstrap_values
    ) >= int(
        n_bootstrap
        *
        0.90
    )

    lower, upper = np.quantile(
        bootstrap_values,
        [
            0.025,
            0.975,
        ],
    )

    return (
        float(lower),
        float(upper),
        int(
            len(
                bootstrap_values
            )
        ),
    )


# ============================================================
# 3. Resolve Drive and Stage 2 inputs
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

TEST_ROOT = (
    RETINAL_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)

DEVELOPMENT_ROOT = (
    TEST_ROOT
    / "02_Development"
)

STAGE2_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)

APTOS_DEVELOPMENT_PATH = (
    DEVELOPMENT_ROOT
    / "APTOS_2019_Development_Labels_v0.1.csv"
)

IDRID_DEVELOPMENT_PATH = (
    DEVELOPMENT_ROOT
    / "IDRiD_Development_Labels_v0.1.csv"
)


for required_path in [
    STAGE2_DECISION_PATH,
    APTOS_DEVELOPMENT_PATH,
    IDRID_DEVELOPMENT_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with open(
    STAGE2_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage2_decision = json.load(
        file
    )


assert (
    stage2_decision[
        "decision"
    ]
    ==
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_AND_LABEL_QUARANTINE_ADVANCE_TO_DEVELOPMENT_RECOVERABILITY_GATE"
)


EXPECTED_APTOS_DEVELOPMENT = int(
    stage2_decision[
        "aptos"
    ][
        "development_images"
    ]
)

EXPECTED_IDRID_DEVELOPMENT = int(
    stage2_decision[
        "idrid"
    ][
        "development_images"
    ]
)


assert (
    EXPECTED_APTOS_DEVELOPMENT
    ==
    2520
)

assert (
    EXPECTED_IDRID_DEVELOPMENT
    ==
    408
)


# ============================================================
# 4. Stage 3A output paths
# ============================================================

STAGE3_ROOT = (
    TEST_ROOT
    / "Stage3_Development_Recoverability_Gate_v0.1"
)

STAGE3_PROTOCOL_ROOT = (
    STAGE3_ROOT
    / "00_Protocol"
)

STAGE3_EMBEDDING_ROOT = (
    STAGE3_ROOT
    / "01_Frozen_Embeddings"
)

STAGE3_RESULT_ROOT = (
    STAGE3_ROOT
    / "02_Results"
)


for directory in [
    STAGE3_PROTOCOL_ROOT,
    STAGE3_EMBEDDING_ROOT,
    STAGE3_RESULT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


PROTOCOL_PATH = (
    STAGE3_PROTOCOL_ROOT
    / "Stage3A_Target_Recoverability_Protocol_v0.1.json"
)

INPUT_COMMITMENT_PATH = (
    STAGE3_PROTOCOL_ROOT
    / "Stage3A_Development_Input_Commitments_v0.1.json"
)

SUMMARY_PATH = (
    STAGE3_RESULT_ROOT
    / "Stage3A_Target_Recoverability_Summary_v0.1.csv"
)

FOLD_METRICS_PATH = (
    STAGE3_RESULT_ROOT
    / "Stage3A_Fold_Metrics_v0.1.csv"
)

DECISION_PATH = (
    STAGE3_RESULT_ROOT
    / "Stage3A_Target_Recoverability_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE3_RESULT_ROOT
    / "Stage3A_Target_Recoverability_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    STAGE3_RESULT_ROOT
    / "Stage3A_Environment_v0.1.json"
)


# ============================================================
# 5. Freeze and hash protocol before image or model access
# ============================================================

protocol_payload = {
    "protocol_version": "v0.1",
    "stage": (
        "DEVELOPMENT_ONLY_TARGET_RECOVERABILITY_GATE"
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "positive_class_rule": (
        POSITIVE_CLASS_RULE
    ),
    "safety_boundary": (
        SAFETY_BOUNDARY
    ),
    "interpretation_boundary": (
        INTERPRETATION_BOUNDARY
    ),
    "datasets": [
        "APTOS_2019_development",
        "IDRiD_development",
    ],
    "sealed_evaluation_labels_access_authorised": False,
    "backbone": (
        FROZEN_BACKBONE
    ),
    "backbone_trainable": False,
    "embedding_dimension": (
        FROZEN_FEATURE_DIMENSION
    ),
    "embedding_normalisation": (
        "L2"
    ),
    "input_transform": (
        "ResNet50 IMAGENET1K_V2 official inference transform"
    ),
    "probe": {
        "type": (
            "logistic_regression"
        ),
        "C": (
            LOGISTIC_C
        ),
        "class_weight": (
            LOGISTIC_CLASS_WEIGHT
        ),
        "solver": (
            LOGISTIC_SOLVER
        ),
        "max_iter": (
            LOGISTIC_MAX_ITER
        ),
        "hyperparameter_tuning": False,
    },
    "validation": {
        "folds": (
            N_FOLDS
        ),
        "grouping_unit": (
            "duplicate_component_id"
        ),
        "group_weighting": (
            "each duplicate component receives total training weight 1"
        ),
        "prediction_type": (
            "out_of_fold"
        ),
        "primary_metric_unit": (
            "duplicate_component"
        ),
        "component_prediction_aggregation": (
            "mean_probability"
        ),
        "bootstrap_iterations": (
            N_BOOTSTRAP
        ),
    },
    "dataset_pass_rule": {
        "component_oof_auc_minimum": (
            PRIMARY_AUC_THRESHOLD
        ),
        "component_bootstrap_95ci_lower_strictly_greater_than": (
            PRIMARY_CI_LOWER_THRESHOLD
        ),
    },
    "decision_map": {
        "both_targets_pass": (
            "retain all four precommitted source-target edges"
        ),
        "one_target_passes": (
            "retain only the two edges ending in the passing target"
        ),
        "zero_targets_pass": (
            "retire the redesigned prospective retinal test"
        ),
    },
    "random_seed": (
        RANDOM_SEED
    ),
    "protocol_frozen_before_embedding_extraction": True,
    "protocol_frozen_before_model_performance": True,
}


with open(
    PROTOCOL_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_payload,
        file,
        indent=2,
    )


protocol_sha256 = sha256_file(
    PROTOCOL_PATH
)


input_commitments = {
    "stage2_decision_path": str(
        STAGE2_DECISION_PATH
    ),
    "stage2_decision_sha256": sha256_file(
        STAGE2_DECISION_PATH
    ),
    "aptos_development_path": str(
        APTOS_DEVELOPMENT_PATH
    ),
    "aptos_development_sha256": sha256_file(
        APTOS_DEVELOPMENT_PATH
    ),
    "idrid_development_path": str(
        IDRID_DEVELOPMENT_PATH
    ),
    "idrid_development_sha256": sha256_file(
        IDRID_DEVELOPMENT_PATH
    ),
    "protocol_path": str(
        PROTOCOL_PATH
    ),
    "protocol_sha256": (
        protocol_sha256
    ),
    "sealed_evaluation_label_files_accessed": False,
}


with open(
    INPUT_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        input_commitments,
        file,
        indent=2,
    )


print(
    "================ STAGE 3A TARGET "
    "RECOVERABILITY GATE ================"
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)

print("\nInterpretation boundary:")
print(
    INTERPRETATION_BOUNDARY
)

print("\nProtocol frozen before embeddings:")
print(True)

print("\nProtocol SHA-256:")
print(
    protocol_sha256
)


# ============================================================
# 6. Read development-only labels
# ============================================================

aptos_development = pd.read_csv(
    APTOS_DEVELOPMENT_PATH
)

idrid_development = pd.read_csv(
    IDRID_DEVELOPMENT_PATH
)


required_columns = {
    "dataset",
    "image_id",
    "canonical_image_path",
    "original_dr_grade",
    "moderate_or_worse_dr",
    "duplicate_component_id",
    "original_sha256",
    "canonical_sha256",
}


for dataset_name, dataframe, expected_rows in [
    (
        "APTOS_2019",
        aptos_development,
        EXPECTED_APTOS_DEVELOPMENT,
    ),
    (
        "IDRiD",
        idrid_development,
        EXPECTED_IDRID_DEVELOPMENT,
    ),
]:
    missing_columns = (
        required_columns
        -
        set(
            dataframe.columns
        )
    )

    assert not missing_columns, (
        f"{dataset_name} missing columns: "
        f"{sorted(missing_columns)}"
    )

    assert (
        len(
            dataframe
        )
        ==
        expected_rows
    )

    assert dataframe[
        "image_id"
    ].is_unique

    assert dataframe[
        "moderate_or_worse_dr"
    ].isin(
        [
            0,
            1,
        ]
    ).all()

    assert (
        dataframe[
            "moderate_or_worse_dr"
        ].nunique()
        ==
        2
    )

    assert dataframe[
        "canonical_image_path"
    ].map(
        lambda value: Path(
            value
        ).is_file()
    ).all()

    component_label_counts = (
        dataframe
        .groupby(
            "duplicate_component_id"
        )[
            "moderate_or_worse_dr"
        ]
        .nunique()
    )

    assert not (
        component_label_counts
        >
        1
    ).any()


development_inventory = pd.DataFrame(
    [
        {
            "dataset": "APTOS_2019",
            "images": int(
                len(
                    aptos_development
                )
            ),
            "duplicate_components": int(
                aptos_development[
                    "duplicate_component_id"
                ].nunique()
            ),
            "negative_images": int(
                (
                    aptos_development[
                        "moderate_or_worse_dr"
                    ]
                    ==
                    0
                ).sum()
            ),
            "positive_images": int(
                (
                    aptos_development[
                        "moderate_or_worse_dr"
                    ]
                    ==
                    1
                ).sum()
            ),
        },
        {
            "dataset": "IDRiD",
            "images": int(
                len(
                    idrid_development
                )
            ),
            "duplicate_components": int(
                idrid_development[
                    "duplicate_component_id"
                ].nunique()
            ),
            "negative_images": int(
                (
                    idrid_development[
                        "moderate_or_worse_dr"
                    ]
                    ==
                    0
                ).sum()
            ),
            "positive_images": int(
                (
                    idrid_development[
                        "moderate_or_worse_dr"
                    ]
                    ==
                    1
                ).sum()
            ),
        },
    ]
)


print(
    "\n================ DEVELOPMENT INVENTORY "
    "================"
)

display(
    development_inventory
)

print(
    "\nSealed-evaluation label files accessed:"
)

print(False)


# ============================================================
# 7. Frozen image dataset
# ============================================================

weights = (
    ResNet50_Weights
    .IMAGENET1K_V2
)

inference_transform = (
    weights.transforms()
)


class CanonicalFundusDataset(
    Dataset
):
    def __init__(
        self,
        dataframe,
        transform,
    ):
        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.transform = (
            transform
        )

    def __len__(
        self,
    ):
        return len(
            self.dataframe
        )

    def __getitem__(
        self,
        index,
    ):
        row = self.dataframe.iloc[
            index
        ]

        image_path = Path(
            row[
                "canonical_image_path"
            ]
        )

        with Image.open(
            image_path
        ) as image:
            image.load()

            assert (
                image.size
                ==
                (
                    CANONICAL_SIZE,
                    CANONICAL_SIZE,
                )
            )

            image = image.convert(
                "RGB"
            )

            tensor = self.transform(
                image
            )

        return (
            tensor,
            int(
                index
            ),
        )


# ============================================================
# 8. Frozen ResNet50 embedding extraction
# ============================================================

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else
    "cpu"
)


if device.type == "cuda":
    batch_size = 64
    number_of_workers = 2

else:
    batch_size = 16
    number_of_workers = 2


print(
    "\n================ FROZEN BACKBONE "
    "================"
)

print("Device:")
print(
    device
)

if device.type == "cuda":
    print("\nGPU:")
    print(
        torch.cuda.get_device_name(
            0
        )
    )

print("\nBackbone:")
print(
    FROZEN_BACKBONE
)

print("\nBackbone trainable:")
print(False)

print("\nBatch size:")
print(
    batch_size
)


model = resnet50(
    weights=weights
)

model.fc = nn.Identity()

model.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

model = model.to(
    device
)


def extract_embeddings(
    dataframe,
    dataset_name,
):
    dataset = CanonicalFundusDataset(
        dataframe=dataframe,
        transform=inference_transform,
    )

    loader = DataLoader(
        dataset,
        batch_size=(
            batch_size
        ),
        shuffle=False,
        num_workers=(
            number_of_workers
        ),
        pin_memory=(
            device.type
            ==
            "cuda"
        ),
        drop_last=False,
        persistent_workers=(
            number_of_workers
            >
            0
        ),
    )

    embeddings = np.zeros(
        (
            len(
                dataframe
            ),
            FROZEN_FEATURE_DIMENSION,
        ),
        dtype=np.float32,
    )

    with torch.inference_mode():
        for image_batch, index_batch in tqdm(
            loader,
            desc=(
                f"Extracting {dataset_name} embeddings"
            ),
        ):
            image_batch = image_batch.to(
                device,
                non_blocking=True,
            )

            feature_batch = model(
                image_batch
            )

            feature_batch = F.normalize(
                feature_batch,
                p=2,
                dim=1,
            )

            embeddings[
                index_batch.numpy()
            ] = (
                feature_batch
                .detach()
                .cpu()
                .numpy()
                .astype(
                    np.float32
                )
            )

    assert np.isfinite(
        embeddings
    ).all()

    embedding_norms = np.linalg.norm(
        embeddings,
        axis=1,
    )

    assert np.allclose(
        embedding_norms,
        1.0,
        atol=1e-5,
    )

    embedding_path = (
        STAGE3_EMBEDDING_ROOT
        /
        f"{dataset_name}_Development_"
        f"ResNet50V2_L2_Embeddings_v0.1.npz"
    )

    np.savez_compressed(
        embedding_path,
        image_id=(
            dataframe[
                "image_id"
            ]
            .astype(str)
            .to_numpy()
        ),
        duplicate_component_id=(
            dataframe[
                "duplicate_component_id"
            ]
            .astype(str)
            .to_numpy()
        ),
        embedding=(
            embeddings
        ),
    )

    return (
        embeddings,
        embedding_path,
    )


aptos_embeddings, aptos_embedding_path = (
    extract_embeddings(
        aptos_development,
        "APTOS_2019",
    )
)


idrid_embeddings, idrid_embedding_path = (
    extract_embeddings(
        idrid_development,
        "IDRiD",
    )
)


# ============================================================
# 9. Fixed grouped out-of-fold linear probe
# ============================================================

def run_recoverability_probe(
    dataframe,
    embeddings,
    dataset_name,
    bootstrap_seed,
):
    dataframe = (
        dataframe
        .reset_index(
            drop=True
        )
        .copy()
    )

    labels = (
        dataframe[
            "moderate_or_worse_dr"
        ]
        .astype(int)
        .to_numpy()
    )

    groups = (
        dataframe[
            "duplicate_component_id"
        ]
        .astype(str)
        .to_numpy()
    )

    group_table = (
        dataframe
        .groupby(
            "duplicate_component_id",
            as_index=False,
        )
        .agg(
            group_label=(
                "moderate_or_worse_dr",
                "first",
            ),
            group_images=(
                "image_id",
                "size",
            ),
        )
        .sort_values(
            "duplicate_component_id"
        )
        .reset_index(
            drop=True
        )
    )

    assert (
        group_table[
            "group_label"
        ].nunique()
        ==
        2
    )

    assert (
        group_table[
            "group_label"
        ]
        .value_counts()
        .min()
        >=
        N_FOLDS
    )

    group_splitter = StratifiedKFold(
        n_splits=(
            N_FOLDS
        ),
        shuffle=True,
        random_state=(
            RANDOM_SEED
        ),
    )

    out_of_fold_probability = np.full(
        len(
            dataframe
        ),
        np.nan,
        dtype=np.float64,
    )

    row_fold_assignment = np.full(
        len(
            dataframe
        ),
        -1,
        dtype=np.int64,
    )

    fold_records = []

    for fold_index, (
        training_group_indices,
        validation_group_indices,
    ) in enumerate(
        group_splitter.split(
            np.zeros(
                len(
                    group_table
                )
            ),
            group_table[
                "group_label"
            ].to_numpy(),
        ),
        start=1,
    ):
        training_groups = set(
            group_table.iloc[
                training_group_indices
            ][
                "duplicate_component_id"
            ].astype(str)
        )

        validation_groups = set(
            group_table.iloc[
                validation_group_indices
            ][
                "duplicate_component_id"
            ].astype(str)
        )

        assert not (
            training_groups
            .intersection(
                validation_groups
            )
        )

        training_rows = np.flatnonzero(
            np.isin(
                groups,
                list(
                    training_groups
                ),
            )
        )

        validation_rows = np.flatnonzero(
            np.isin(
                groups,
                list(
                    validation_groups
                ),
            )
        )

        assert (
            np.unique(
                labels[
                    training_rows
                ]
            ).size
            ==
            2
        )

        assert (
            np.unique(
                labels[
                    validation_rows
                ]
            ).size
            ==
            2
        )

        training_group_sizes = (
            dataframe.iloc[
                training_rows
            ]
            .groupby(
                "duplicate_component_id"
            )[
                "image_id"
            ]
            .transform(
                "size"
            )
            .to_numpy(
                dtype=np.float64
            )
        )

        training_sample_weight = (
            1.0
            /
            training_group_sizes
        )

        fixed_probe = Pipeline(
            steps=[
                (
                    "scaler",
                    StandardScaler(),
                ),
                (
                    "logisticregression",
                    LogisticRegression(
                        C=(
                            LOGISTIC_C
                        ),
                        class_weight=(
                            LOGISTIC_CLASS_WEIGHT
                        ),
                        solver=(
                            LOGISTIC_SOLVER
                        ),
                        max_iter=(
                            LOGISTIC_MAX_ITER
                        ),
                        random_state=(
                            RANDOM_SEED
                        ),
                    ),
                ),
            ]
        )

        fixed_probe.fit(
            embeddings[
                training_rows
            ],
            labels[
                training_rows
            ],
            logisticregression__sample_weight=(
                training_sample_weight
            ),
        )

        validation_probability = (
            fixed_probe.predict_proba(
                embeddings[
                    validation_rows
                ]
            )[
                :,
                1,
            ]
        )

        out_of_fold_probability[
            validation_rows
        ] = validation_probability

        row_fold_assignment[
            validation_rows
        ] = fold_index

        validation_frame = pd.DataFrame(
            {
                "duplicate_component_id": (
                    groups[
                        validation_rows
                    ]
                ),
                "label": (
                    labels[
                        validation_rows
                    ]
                ),
                "probability": (
                    validation_probability
                ),
            }
        )

        validation_component_frame = (
            validation_frame
            .groupby(
                "duplicate_component_id",
                as_index=False,
            )
            .agg(
                label=(
                    "label",
                    "first",
                ),
                probability=(
                    "probability",
                    "mean",
                ),
                images=(
                    "label",
                    "size",
                ),
            )
        )

        fold_records.append(
            {
                "dataset": (
                    dataset_name
                ),
                "fold": int(
                    fold_index
                ),
                "training_images": int(
                    len(
                        training_rows
                    )
                ),
                "validation_images": int(
                    len(
                        validation_rows
                    )
                ),
                "training_components": int(
                    len(
                        training_groups
                    )
                ),
                "validation_components": int(
                    len(
                        validation_groups
                    )
                ),
                "image_auc": float(
                    roc_auc_score(
                        labels[
                            validation_rows
                        ],
                        validation_probability,
                    )
                ),
                "component_auc": float(
                    roc_auc_score(
                        validation_component_frame[
                            "label"
                        ],
                        validation_component_frame[
                            "probability"
                        ],
                    )
                ),
            }
        )

    assert np.isfinite(
        out_of_fold_probability
    ).all()

    assert (
        row_fold_assignment
        >
        0
    ).all()

    prediction_table = dataframe[
        [
            "dataset",
            "image_id",
            "duplicate_component_id",
            "original_dr_grade",
            "moderate_or_worse_dr",
        ]
    ].copy()

    prediction_table[
        "fold"
    ] = (
        row_fold_assignment
    )

    prediction_table[
        "oof_probability"
    ] = (
        out_of_fold_probability
    )

    component_prediction_table = (
        prediction_table
        .groupby(
            "duplicate_component_id",
            as_index=False,
        )
        .agg(
            dataset=(
                "dataset",
                "first",
            ),
            moderate_or_worse_dr=(
                "moderate_or_worse_dr",
                "first",
            ),
            oof_probability=(
                "oof_probability",
                "mean",
            ),
            images=(
                "image_id",
                "size",
            ),
            fold=(
                "fold",
                "first",
            ),
        )
    )

    assert (
        component_prediction_table[
            "moderate_or_worse_dr"
        ].nunique()
        ==
        2
    )

    image_auc = float(
        roc_auc_score(
            labels,
            out_of_fold_probability,
        )
    )

    component_auc = float(
        roc_auc_score(
            component_prediction_table[
                "moderate_or_worse_dr"
            ],
            component_prediction_table[
                "oof_probability"
            ],
        )
    )

    component_average_precision = float(
        average_precision_score(
            component_prediction_table[
                "moderate_or_worse_dr"
            ],
            component_prediction_table[
                "oof_probability"
            ],
        )
    )

    component_balanced_accuracy = float(
        balanced_accuracy_score(
            component_prediction_table[
                "moderate_or_worse_dr"
            ],
            (
                component_prediction_table[
                    "oof_probability"
                ]
                >=
                0.5
            ).astype(int),
        )
    )

    component_brier = float(
        brier_score_loss(
            component_prediction_table[
                "moderate_or_worse_dr"
            ],
            component_prediction_table[
                "oof_probability"
            ],
        )
    )

    ci_lower, ci_upper, valid_bootstraps = (
        bootstrap_auc(
            labels=(
                component_prediction_table[
                    "moderate_or_worse_dr"
                ].to_numpy()
            ),
            probabilities=(
                component_prediction_table[
                    "oof_probability"
                ].to_numpy()
            ),
            n_bootstrap=(
                N_BOOTSTRAP
            ),
            seed=(
                bootstrap_seed
            ),
        )
    )

    dataset_pass = bool(
        component_auc
        >=
        PRIMARY_AUC_THRESHOLD
        and
        ci_lower
        >
        PRIMARY_CI_LOWER_THRESHOLD
    )

    prediction_path = (
        STAGE3_RESULT_ROOT
        /
        f"{dataset_name}_Development_OOF_"
        f"Predictions_v0.1.csv"
    )

    component_prediction_path = (
        STAGE3_RESULT_ROOT
        /
        f"{dataset_name}_Development_Component_"
        f"OOF_Predictions_v0.1.csv"
    )

    prediction_table.to_csv(
        prediction_path,
        index=False,
    )

    component_prediction_table.to_csv(
        component_prediction_path,
        index=False,
    )

    summary_record = {
        "dataset": (
            dataset_name
        ),
        "development_images": int(
            len(
                dataframe
            )
        ),
        "development_components": int(
            len(
                component_prediction_table
            )
        ),
        "negative_components": int(
            (
                component_prediction_table[
                    "moderate_or_worse_dr"
                ]
                ==
                0
            ).sum()
        ),
        "positive_components": int(
            (
                component_prediction_table[
                    "moderate_or_worse_dr"
                ]
                ==
                1
            ).sum()
        ),
        "image_oof_auc": (
            image_auc
        ),
        "component_oof_auc": (
            component_auc
        ),
        "component_auc_ci_lower_95": (
            ci_lower
        ),
        "component_auc_ci_upper_95": (
            ci_upper
        ),
        "component_average_precision": (
            component_average_precision
        ),
        "component_balanced_accuracy_at_0_5": (
            component_balanced_accuracy
        ),
        "component_brier_score": (
            component_brier
        ),
        "valid_bootstrap_samples": (
            valid_bootstraps
        ),
        "auc_threshold": (
            PRIMARY_AUC_THRESHOLD
        ),
        "ci_lower_threshold": (
            PRIMARY_CI_LOWER_THRESHOLD
        ),
        "dataset_pass": (
            dataset_pass
        ),
        "prediction_path": str(
            prediction_path
        ),
        "component_prediction_path": str(
            component_prediction_path
        ),
    }

    return (
        summary_record,
        pd.DataFrame(
            fold_records
        ),
    )


aptos_result, aptos_fold_metrics = (
    run_recoverability_probe(
        dataframe=(
            aptos_development
        ),
        embeddings=(
            aptos_embeddings
        ),
        dataset_name=(
            "APTOS_2019"
        ),
        bootstrap_seed=(
            RANDOM_SEED
            +
            101
        ),
    )
)


idrid_result, idrid_fold_metrics = (
    run_recoverability_probe(
        dataframe=(
            idrid_development
        ),
        embeddings=(
            idrid_embeddings
        ),
        dataset_name=(
            "IDRiD"
        ),
        bootstrap_seed=(
            RANDOM_SEED
            +
            202
        ),
    )
)


# ============================================================
# 10. Save metric tables
# ============================================================

recoverability_summary = pd.DataFrame(
    [
        aptos_result,
        idrid_result,
    ]
)


fold_metrics = pd.concat(
    [
        aptos_fold_metrics,
        idrid_fold_metrics,
    ],
    ignore_index=True,
)


recoverability_summary.to_csv(
    SUMMARY_PATH,
    index=False,
)

fold_metrics.to_csv(
    FOLD_METRICS_PATH,
    index=False,
)


# ============================================================
# 11. Frozen decision map
# ============================================================

passing_targets = (
    recoverability_summary.loc[
        recoverability_summary[
            "dataset_pass"
        ],
        "dataset",
    ]
    .astype(str)
    .tolist()
)


candidate_sources = [
    "EyePACS_2015",
    "DeepDRiD",
]


candidate_targets = [
    "APTOS_2019",
    "IDRiD",
]


authorised_edges = [
    {
        "source": source,
        "target": target,
    }
    for source in candidate_sources
    for target in passing_targets
]


retired_targets = [
    target
    for target in candidate_targets
    if target not in passing_targets
]


if len(
    passing_targets
) == 2:
    stage3_decision = (
        "PASS_BOTH_TARGETS_RECOVERABLE_"
        "ADVANCE_ALL_FOUR_EDGES"
    )

    authorised_next_step = (
        "CANONICALISE_EYEPACS_AND_DEEPDRID_WITH_"
        "THE_FROZEN_TARGET_PROTOCOL"
    )

    interpretation = (
        "Both prospective targets passed the pre-frozen "
        "development-only recoverability gate. All four "
        "precommitted source-target edges remain authorised."
    )

elif len(
    passing_targets
) == 1:
    stage3_decision = (
        "PARTIAL_PASS_RETAIN_ONLY_EDGES_TO_"
        "RECOVERABLE_TARGET"
    )

    authorised_next_step = (
        "CANONICALISE_SOURCES_AND_RETAIN_ONLY_"
        "THE_TWO_AUTHORISED_EDGES"
    )

    interpretation = (
        "Only one prospective target passed the pre-frozen "
        "development-only recoverability gate. The two edges "
        "ending in that target remain authorised; both edges "
        "ending in the non-recoverable target are retired before "
        "any source-target transfer result is observed."
    )

else:
    stage3_decision = (
        "FAIL_TARGET_RECOVERABILITY_"
        "RETIRE_PROSPECTIVE_RETINAL_TEST"
    )

    authorised_next_step = (
        "DO_NOT_RUN_SOURCE_TARGET_TRANSFER"
    )

    interpretation = (
        "Neither target passed the pre-frozen recoverability "
        "gate under the frozen representation. Source-target "
        "transfer would not be interpretable, so the redesigned "
        "prospective retinal test is retired."
    )


decision_payload = {
    "decision": (
        stage3_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "protocol_path": str(
        PROTOCOL_PATH
    ),
    "protocol_sha256": (
        protocol_sha256
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "backbone": (
        FROZEN_BACKBONE
    ),
    "passing_targets": (
        passing_targets
    ),
    "retired_targets": (
        retired_targets
    ),
    "authorised_edges": (
        authorised_edges
    ),
    "results": (
        recoverability_summary
        .to_dict(
            orient="records"
        )
    ),
    "sealed_evaluation_labels_accessed": False,
    "sealed_evaluation_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "important_boundary": (
        INTERPRETATION_BOUNDARY
    ),
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
        default=json_default,
    )


# ============================================================
# 12. Report and environment
# ============================================================

report_lines = [
    "# Stage 3A — Development-Only Target Recoverability Gate",
    "",
    f"Decision: `{stage3_decision}`",
    "",
    interpretation,
    "",
    "## Frozen method",
    "",
    f"- Backbone: `{FROZEN_BACKBONE}`",
    (
        "- Probe: fixed class-balanced logistic regression, "
        "no hyperparameter tuning"
    ),
    (
        "- Validation: duplicate-component-grouped "
        f"{N_FOLDS}-fold out-of-fold prediction"
    ),
    (
        "- Dataset pass threshold: component AUC "
        f">= {PRIMARY_AUC_THRESHOLD:.2f} and bootstrap "
        f"95% CI lower > {PRIMARY_CI_LOWER_THRESHOLD:.2f}"
    ),
    "",
    "## Safety boundary",
    "",
    "- Sealed-evaluation labels accessed: `False`",
    (
        "- Sealed-evaluation performance observed: `False`"
    ),
    (
        "- Source-target transfer performance observed: `False`"
    ),
    "",
    "## Interpretation boundary",
    "",
    INTERPRETATION_BOUNDARY,
    "",
]


REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": str(
        device
    ),
    "gpu_name": (
        torch.cuda.get_device_name(
            0
        )
        if device.type
        ==
        "cuda"
        else
        None
    ),
    "random_seed": (
        RANDOM_SEED
    ),
    "protocol_sha256": (
        protocol_sha256
    ),
    "aptos_embedding_path": str(
        aptos_embedding_path
    ),
    "aptos_embedding_sha256": sha256_file(
        aptos_embedding_path
    ),
    "idrid_embedding_path": str(
        idrid_embedding_path
    ),
    "idrid_embedding_sha256": sha256_file(
        idrid_embedding_path
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 13. Final display
# ============================================================

display_columns = [
    "dataset",
    "development_images",
    "development_components",
    "negative_components",
    "positive_components",
    "component_oof_auc",
    "component_auc_ci_lower_95",
    "component_auc_ci_upper_95",
    "component_average_precision",
    "dataset_pass",
]


print(
    "\n================ TARGET RECOVERABILITY "
    "SUMMARY ================"
)

display(
    recoverability_summary[
        display_columns
    ]
)


print(
    "\n================ FOLD METRIC SUMMARY "
    "================"
)

display(
    fold_metrics
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        folds=(
            "fold",
            "size",
        ),
        minimum_component_auc=(
            "component_auc",
            "min",
        ),
        median_component_auc=(
            "component_auc",
            "median",
        ),
        maximum_component_auc=(
            "component_auc",
            "max",
        ),
    )
)


print(
    "\n================ STAGE 3A DECISION "
    "================"
)

print("Decision:")
print(
    stage3_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nPassing targets:")
print(
    passing_targets
)

print("\nRetired targets:")
print(
    retired_targets
)

print("\nAuthorised edges:")
print(
    json.dumps(
        authorised_edges,
        indent=2,
    )
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print(
    "\nSealed-evaluation labels accessed:"
)

print(False)

print(
    "\nSealed-evaluation performance observed:"
)

print(False)

print(
    "\nSource-target transfer performance observed:"
)

print(False)

print(
    "\nStage 3A development-only target "
    "recoverability gate completed and sealed."
)

================ STAGE 3A TARGET RECOVERABILITY GATE ================

Safety boundary:
Stage 3A reads only the two authorised development-label files created by Stage 2R. It must not read, list, open, summarise, hash again, or evaluate either sealed-evaluation label file. The frozen ResNet50 representation and fixed linear probe are evaluated using duplicate-component-grouped out-of-fold prediction only.

Interpretation boundary:
Passing this gate establishes only that the frozen representation contains a linearly recoverable signal for the moderate-or-worse DR endpoint inside the target development cohort. It does not establish source-to-target transfer, external generalisation, sealed-evaluation performance, causal validity, or CDO validity.

Protocol frozen before embeddings:
True

Protocol SHA-256:
a6a6783ed2b8af1a1f1748c5851789207de718f8156f701cc160afdb73e7f849

================ DEVELOPMENT INVENTORY ================


,dataset,images,duplicate_components,negative_images,positive_images
0,APTOS_2019,2520,2452,1513,1007
1,IDRiD,408,405,152,256



Sealed-evaluation label files accessed:
False

================ FROZEN BACKBONE ================
Device:
cpu

Backbone:
ResNet50_IMAGENET1K_V2

Backbone trainable:
False

Batch size:
16
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 135MB/s]


Extracting APTOS_2019 embeddings:   0%|          | 0/158 [00:00<?, ?it/s]

...


================ TARGET RECOVERABILITY SUMMARY ================


,dataset,development_images,development_components,negative_components,positive_components,component_oof_auc,component_auc_ci_lower_95,component_auc_ci_upper_95,component_average_precision,dataset_pass
0,APTOS_2019,2520,2452,1493,959,0.966688,0.960361,0.972849,0.950147,True
1,IDRiD,408,405,151,254,0.929681,0.903033,0.952624,0.940430,True



================ FOLD METRIC SUMMARY ================


,dataset,folds,minimum_component_auc,median_component_auc,maximum_component_auc
0,APTOS_2019,5,0.962375,0.966294,0.973504
1,IDRiD,5,0.898039,0.942484,0.964706



================ STAGE 3A DECISION ================
Decision:
PASS_BOTH_TARGETS_RECOVERABLE_ADVANCE_ALL_FOUR_EDGES

Interpretation:
Both prospective targets passed the pre-frozen development-only recoverability gate. All four precommitted source-target edges remain authorised.

Passing targets:
['APTOS_2019', 'IDRiD']

Retired targets:
[]

Authorised edges:
[
  {
    "source": "EyePACS_2015",
    "target": "APTOS_2019"
  },
  {
    "source": "EyePACS_2015",
    "target": "IDRiD"
  },
  {
    "source": "DeepDRiD",
    "target": "APTOS_2019"
  },
  {
    "source": "DeepDRiD",
    "target": "IDRiD"
  }
]

Authorised next step:
CANONICALISE_EYEPACS_AND_DEEPDRID_WITH_THE_FROZEN_TARGET_PROTOCOL

Sealed-evaluation labels accessed:
False

Sealed-evaluation performance observed:
False

Source-target transfer performance observed:
False

Stage 3A development-only target recoverability gate completed and sealed.


In [24]:
#@title 04A. Source artifact discovery and canonicalisation readiness audit

from pathlib import Path
from datetime import datetime, timezone

import json
import os
import re

import pandas as pd


# ============================================================
# 0. Frozen Stage 4A boundary
# ============================================================

SOURCE_EXPECTATIONS = {
    "EyePACS_2015": {
        "historical_selected_images": 4468,
        "historical_development_eyes": 1675,
        "historical_validation_eyes": 559,
        "keywords": [
            "eyepacs",
            "eye_pacs",
            "eye-pacs",
        ],
    },
    "DeepDRiD": {
        "historical_selected_images": 1600,
        "historical_training_eyes": 600,
        "historical_validation_eyes": 200,
        "keywords": [
            "deepdrid",
            "deep_drid",
            "deep-drid",
            "deep drid",
        ],
    },
}

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".bmp",
}

ARTIFACT_EXTENSIONS = {
    ".csv",
    ".json",
    ".npz",
    ".npy",
    ".parquet",
    ".pkl",
    ".pickle",
    ".txt",
}

SAFETY_BOUNDARY = (
    "Stage 4A performs read-only source artifact discovery. "
    "It does not access target sealed-label files, compute model "
    "performance, modify source or target data, or begin source "
    "canonicalisation."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

RETINAL_CODE_ROOT = (
    PROJECT_ROOT
    / "05_Code"
    / "Retinal_DR"
)

PROSPECTIVE_TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE3_ROOT = (
    PROSPECTIVE_TEST_ROOT
    / "Stage3_Development_Recoverability_Gate_v0.1"
)

STAGE3_DECISION_PATH = (
    STAGE3_ROOT
    / "02_Results"
    / "Stage3A_Target_Recoverability_Decision_v0.1.json"
)

STAGE4_ROOT = (
    PROSPECTIVE_TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4_AUDIT_ROOT = (
    STAGE4_ROOT
    / "00_Source_Artifact_Audit"
)

STAGE4_AUDIT_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


assert STAGE3_DECISION_PATH.is_file(), (
    STAGE3_DECISION_PATH
)


with open(
    STAGE3_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage3_decision = json.load(
        file
    )


assert (
    stage3_decision[
        "decision"
    ]
    ==
    "PASS_BOTH_TARGETS_RECOVERABLE_ADVANCE_ALL_FOUR_EDGES"
)


print(
    "================ STAGE 4A SOURCE "
    "ARTIFACT AUDIT ================"
)

print("\nImported Stage 3A decision:")
print(
    stage3_decision[
        "decision"
    ]
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)


# ============================================================
# 2. Path helpers
# ============================================================

def normalise_text(
    value,
):
    return str(
        value
    ).lower()


def path_is_inside(
    path,
    parent,
):
    path = Path(
        path
    ).resolve()

    parent = Path(
        parent
    ).resolve()

    try:
        path.relative_to(
            parent
        )

        return True

    except ValueError:
        return False


def classify_source_dataset(
    path,
):
    path_text = normalise_text(
        path
    )

    for dataset_name, specification in (
        SOURCE_EXPECTATIONS.items()
    ):
        if any(
            keyword in path_text
            for keyword in specification[
                "keywords"
            ]
        ):
            return dataset_name

    return None


def resolve_keyword_root(
    path,
    dataset_name,
):
    path = Path(
        path
    )

    keywords = SOURCE_EXPECTATIONS[
        dataset_name
    ][
        "keywords"
    ]

    matching_ancestors = []

    for ancestor in [
        path.parent,
        *path.parents,
    ]:
        ancestor_name = normalise_text(
            ancestor.name
        )

        if any(
            keyword in ancestor_name
            for keyword in keywords
        ):
            matching_ancestors.append(
                ancestor
            )

    if matching_ancestors:
        return matching_ancestors[0]

    return path.parent


def safe_csv_header(
    path,
):
    path = Path(
        path
    )

    try:
        if path.stat().st_size > (
            100
            *
            1024 ** 2
        ):
            return (
                "SKIPPED_HEADER_FILE_OVER_100MB"
            )

        header = pd.read_csv(
            path,
            nrows=0,
        )

        return " | ".join(
            map(
                str,
                header.columns,
            )
        )

    except Exception as error:
        return (
            "HEADER_READ_ERROR: "
            +
            type(
                error
            ).__name__
        )


def safe_json_keys(
    path,
):
    path = Path(
        path
    )

    try:
        if path.stat().st_size > (
            20
            *
            1024 ** 2
        ):
            return (
                "SKIPPED_KEYS_FILE_OVER_20MB"
            )

        with path.open(
            "r",
            encoding="utf-8",
        ) as file:
            payload = json.load(
                file
            )

        if isinstance(
            payload,
            dict,
        ):
            return " | ".join(
                map(
                    str,
                    list(
                        payload.keys()
                    )[:50],
                )
            )

        return (
            "TOP_LEVEL_TYPE_"
            +
            type(
                payload
            ).__name__
        )

    except Exception as error:
        return (
            "JSON_READ_ERROR: "
            +
            type(
                error
            ).__name__
        )


# ============================================================
# 3. Define search roots
# ============================================================

search_roots = [
    root
    for root in [
        RETINAL_DATA_ROOT,
        RETINAL_CODE_ROOT,
    ]
    if root.is_dir()
]


assert search_roots, (
    "No retinal project search roots were found."
)


print("\nSearch roots:")

for root in search_roots:
    print(
        root
    )


# ============================================================
# 4. Read-only file scan
# ============================================================

image_records = []
artifact_records = []
other_keyword_records = []

excluded_directory_names = {
    ".ipynb_checkpoints",
    "__pycache__",
}


for search_root in search_roots:
    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_root_path = Path(
            current_root
        )

        directory_names[:] = [
            directory_name
            for directory_name in directory_names
            if directory_name
            not in excluded_directory_names
        ]

        # Never inspect the prospective target test directory
        # during source discovery.
        if path_is_inside(
            current_root_path,
            PROSPECTIVE_TEST_ROOT,
        ):
            directory_names[:] = []
            continue

        for filename in filenames:
            file_path = (
                current_root_path
                /
                filename
            )

            dataset_name = (
                classify_source_dataset(
                    file_path
                )
            )

            if dataset_name is None:
                continue

            try:
                file_size = int(
                    file_path
                    .stat()
                    .st_size
                )

            except FileNotFoundError:
                continue

            suffix = (
                file_path
                .suffix
                .lower()
            )

            common_record = {
                "dataset": (
                    dataset_name
                ),
                "path": str(
                    file_path
                ),
                "filename": (
                    file_path.name
                ),
                "suffix": (
                    suffix
                ),
                "size_bytes": (
                    file_size
                ),
                "size_mib": float(
                    file_size
                    /
                    (1024 ** 2)
                ),
            }

            if suffix in IMAGE_EXTENSIONS:
                keyword_root = (
                    resolve_keyword_root(
                        file_path,
                        dataset_name,
                    )
                )

                image_records.append(
                    {
                        **common_record,
                        "candidate_keyword_root": str(
                            keyword_root
                        ),
                        "immediate_parent": str(
                            file_path.parent
                        ),
                    }
                )

            elif suffix in ARTIFACT_EXTENSIONS:
                artifact_type = (
                    "other"
                )

                structure_preview = ""

                if suffix == ".csv":
                    artifact_type = "csv"
                    structure_preview = (
                        safe_csv_header(
                            file_path
                        )
                    )

                elif suffix == ".json":
                    artifact_type = "json"
                    structure_preview = (
                        safe_json_keys(
                            file_path
                        )
                    )

                elif suffix == ".npz":
                    artifact_type = "npz"

                elif suffix == ".npy":
                    artifact_type = "npy"

                elif suffix == ".parquet":
                    artifact_type = "parquet"

                artifact_records.append(
                    {
                        **common_record,
                        "artifact_type": (
                            artifact_type
                        ),
                        "structure_preview": (
                            structure_preview
                        ),
                    }
                )

            else:
                other_keyword_records.append(
                    common_record
                )


image_inventory = pd.DataFrame(
    image_records
)

artifact_inventory = pd.DataFrame(
    artifact_records
)

other_keyword_inventory = pd.DataFrame(
    other_keyword_records
)


# ============================================================
# 5. Summarise candidate image roots
# ============================================================

if len(
    image_inventory
) > 0:
    image_root_summary = (
        image_inventory
        .groupby(
            [
                "dataset",
                "candidate_keyword_root",
            ],
            as_index=False,
        )
        .agg(
            image_files=(
                "path",
                "size",
            ),
            unique_filenames=(
                "filename",
                "nunique",
            ),
            total_size_gib=(
                "size_bytes",
                lambda values: float(
                    values.sum()
                    /
                    (1024 ** 3)
                ),
            ),
            image_extensions=(
                "suffix",
                lambda values: (
                    " | ".join(
                        sorted(
                            set(
                                values
                            )
                        )
                    )
                ),
            ),
        )
    )

else:
    image_root_summary = pd.DataFrame(
        columns=[
            "dataset",
            "candidate_keyword_root",
            "image_files",
            "unique_filenames",
            "total_size_gib",
            "image_extensions",
        ]
    )


def add_expected_count_distance(
    row,
):
    expected_count = (
        SOURCE_EXPECTATIONS[
            row[
                "dataset"
            ]
        ][
            "historical_selected_images"
        ]
    )

    return int(
        abs(
            int(
                row[
                    "image_files"
                ]
            )
            -
            expected_count
        )
    )


if len(
    image_root_summary
) > 0:
    image_root_summary[
        "distance_from_historical_selected_count"
    ] = image_root_summary.apply(
        add_expected_count_distance,
        axis=1,
    )

    image_root_summary = (
        image_root_summary
        .sort_values(
            [
                "dataset",
                "distance_from_historical_selected_count",
                "image_files",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 6. Rank source artifact candidates
# ============================================================

def artifact_relevance_score(
    row,
):
    filename_text = normalise_text(
        row[
            "filename"
        ]
    )

    structure_text = normalise_text(
        row[
            "structure_preview"
        ]
    )

    score = 0

    for keyword in [
        "label",
        "grade",
        "diagnosis",
        "manifest",
        "split",
        "train",
        "validation",
        "val",
        "selected",
        "inventory",
        "metadata",
    ]:
        if keyword in filename_text:
            score += 3

        if keyword in structure_text:
            score += 1

    for keyword in [
        "embedding",
        "prediction",
        "result",
        "auc",
    ]:
        if keyword in filename_text:
            score -= 1

    return int(
        score
    )


if len(
    artifact_inventory
) > 0:
    artifact_inventory[
        "relevance_score"
    ] = artifact_inventory.apply(
        artifact_relevance_score,
        axis=1,
    )

    artifact_inventory = (
        artifact_inventory
        .sort_values(
            [
                "dataset",
                "relevance_score",
                "size_bytes",
            ],
            ascending=[
                True,
                False,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 7. Dataset-level readiness decision
# ============================================================

readiness_records = []


for dataset_name, specification in (
    SOURCE_EXPECTATIONS.items()
):
    if len(
        image_root_summary
    ) > 0:
        dataset_image_roots = (
            image_root_summary[
                image_root_summary[
                    "dataset"
                ]
                ==
                dataset_name
            ]
            .copy()
        )

    else:
        dataset_image_roots = (
            image_root_summary.copy()
        )

    if len(
        artifact_inventory
    ) > 0:
        dataset_artifacts = (
            artifact_inventory[
                artifact_inventory[
                    "dataset"
                ]
                ==
                dataset_name
            ]
            .copy()
        )

    else:
        dataset_artifacts = (
            artifact_inventory.copy()
        )

    image_files_found = int(
        dataset_image_roots[
            "image_files"
        ].max()
    ) if len(
        dataset_image_roots
    ) > 0 else 0

    candidate_image_roots = int(
        len(
            dataset_image_roots
        )
    )

    candidate_artifacts = int(
        len(
            dataset_artifacts
        )
    )

    expected_selected_images = int(
        specification[
            "historical_selected_images"
        ]
    )

    exact_historical_count_found = bool(
        (
            dataset_image_roots[
                "image_files"
            ]
            ==
            expected_selected_images
        ).any()
    ) if len(
        dataset_image_roots
    ) > 0 else False

    image_material_found = bool(
        image_files_found
        >
        0
    )

    metadata_material_found = bool(
        candidate_artifacts
        >
        0
    )

    dataset_ready_for_path_resolution = bool(
        image_material_found
        and
        metadata_material_found
    )

    readiness_records.append(
        {
            "dataset": (
                dataset_name
            ),
            "historical_selected_image_count": (
                expected_selected_images
            ),
            "largest_candidate_image_count": (
                image_files_found
            ),
            "candidate_image_roots": (
                candidate_image_roots
            ),
            "candidate_metadata_artifacts": (
                candidate_artifacts
            ),
            "exact_historical_count_found": (
                exact_historical_count_found
            ),
            "image_material_found": (
                image_material_found
            ),
            "metadata_material_found": (
                metadata_material_found
            ),
            "ready_for_exact_path_resolution": (
                dataset_ready_for_path_resolution
            ),
        }
    )


readiness_summary = pd.DataFrame(
    readiness_records
)


both_sources_have_material = bool(
    readiness_summary[
        "ready_for_exact_path_resolution"
    ].all()
)


if both_sources_have_material:
    stage4a_decision = (
        "PASS_BOTH_SOURCE_ARTIFACT_SETS_LOCATED_"
        "ADVANCE_TO_EXACT_COHORT_RESOLUTION"
    )

    authorised_next_step = (
        "RESOLVE_EXACT_SOURCE_IMAGE_AND_LABEL_MANIFESTS"
    )

    interpretation = (
        "Candidate source image material and source metadata "
        "artifacts were found for both EyePACS and DeepDRiD. "
        "The exact historical cohorts must now be resolved "
        "before any canonicalisation begins."
    )

else:
    missing_sources = (
        readiness_summary.loc[
            ~readiness_summary[
                "ready_for_exact_path_resolution"
            ],
            "dataset",
        ]
        .astype(str)
        .tolist()
    )

    stage4a_decision = (
        "HOLD_SOURCE_CANONICALISATION_"
        "MISSING_OR_AMBIGUOUS_SOURCE_ARTIFACTS"
    )

    authorised_next_step = (
        "REACQUIRE_OR_LOCATE_MISSING_SOURCE_MATERIAL"
    )

    interpretation = (
        "At least one source lacks either discoverable image "
        "material or discoverable metadata artifacts. Source "
        "canonicalisation must not begin until the missing "
        "material is located or reacquired."
    )


# ============================================================
# 8. Save audit artifacts
# ============================================================

IMAGE_INVENTORY_PATH = (
    STAGE4_AUDIT_ROOT
    / "Stage4A_Source_Image_File_Inventory_v0.1.csv"
)

IMAGE_ROOT_SUMMARY_PATH = (
    STAGE4_AUDIT_ROOT
    / "Stage4A_Source_Image_Root_Summary_v0.1.csv"
)

ARTIFACT_INVENTORY_PATH = (
    STAGE4_AUDIT_ROOT
    / "Stage4A_Source_Metadata_Artifact_Inventory_v0.1.csv"
)

READINESS_SUMMARY_PATH = (
    STAGE4_AUDIT_ROOT
    / "Stage4A_Source_Readiness_Summary_v0.1.csv"
)

DECISION_PATH = (
    STAGE4_AUDIT_ROOT
    / "Stage4A_Source_Artifact_Audit_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE4_AUDIT_ROOT
    / "Stage4A_Source_Artifact_Audit_Report_v0.1.md"
)


image_inventory.to_csv(
    IMAGE_INVENTORY_PATH,
    index=False,
)

image_root_summary.to_csv(
    IMAGE_ROOT_SUMMARY_PATH,
    index=False,
)

artifact_inventory.to_csv(
    ARTIFACT_INVENTORY_PATH,
    index=False,
)

readiness_summary.to_csv(
    READINESS_SUMMARY_PATH,
    index=False,
)


decision_payload = {
    "decision": (
        stage4a_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "stage3_decision_path": str(
        STAGE3_DECISION_PATH
    ),
    "sources": (
        readiness_summary
        .to_dict(
            orient="records"
        )
    ),
    "target_sealed_label_files_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "files_modified_outside_stage4_audit_directory": False,
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# Stage 4A — Source Artifact Discovery",
    "",
    f"Decision: `{stage4a_decision}`",
    "",
    interpretation,
    "",
    "## Safety boundary",
    "",
    "- Target sealed-label files accessed: `False`",
    "- Source performance observed: `False`",
    "- Source-target transfer performance observed: `False`",
    "- Source files modified: `False`",
    "",
]


REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


# ============================================================
# 9. Display concise audit outputs
# ============================================================

print(
    "\n================ SOURCE READINESS SUMMARY "
    "================"
)

display(
    readiness_summary
)


print(
    "\n================ SOURCE IMAGE ROOT "
    "CANDIDATES ================"
)

if len(
    image_root_summary
) > 0:
    display(
        image_root_summary.head(
            20
        )
    )

else:
    print(
        "No keyword-classified source images were found."
    )


print(
    "\n================ SOURCE METADATA "
    "CANDIDATES ================"
)

if len(
    artifact_inventory
) > 0:
    display(
        artifact_inventory[
            [
                "dataset",
                "filename",
                "suffix",
                "size_mib",
                "relevance_score",
                "structure_preview",
                "path",
            ]
        ].head(
            30
        )
    )

else:
    print(
        "No keyword-classified source metadata "
        "artifacts were found."
    )


print(
    "\n================ STAGE 4A DECISION "
    "================"
)

print("Decision:")
print(
    stage4a_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print(
    "\nTarget sealed-label files accessed:"
)

print(False)

print(
    "\nSource performance observed:"
)

print(False)

print(
    "\nSource-target transfer performance observed:"
)

print(False)

print(
    "\nStage 4A source artifact audit "
    "completed and sealed."
)

================ STAGE 4A SOURCE ARTIFACT AUDIT ================

Imported Stage 3A decision:
PASS_BOTH_TARGETS_RECOVERABLE_ADVANCE_ALL_FOUR_EDGES

Safety boundary:
Stage 4A performs read-only source artifact discovery. It does not access target sealed-label files, compute model performance, modify source or target data, or begin source canonicalisation.

Search roots:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/05_Code/Retinal_DR

================ SOURCE READINESS SUMMARY ================


,dataset,historical_selected_image_count,largest_candidate_image_count,candidate_image_roots,candidate_metadata_artifacts,exact_historical_count_found,image_material_found,metadata_material_found,ready_for_exact_path_resolution
0,EyePACS_2015,4468,0,0,39,False,False,True,False
1,DeepDRiD,1600,0,0,1,False,False,True,False



================ SOURCE IMAGE ROOT CANDIDATES ================
No keyword-classified source images were found.

================ SOURCE METADATA CANDIDATES ================


,dataset,filename,suffix,size_mib,relevance_score,structure_preview,path
0,DeepDRiD,DeepDRiD_Image_Size_Audit.csv,.csv,0.037125,0,embedding_row | width | height | resolution,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,EyePACS_2015,EyePACS_Clean_Validation_GradeGap_Subgroups_v0...,.csv,0.000267,9,subgroup | patients | correct_pairs | accuracy...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,EyePACS_2015,EyePACS_2015_Acquired_Image_Manifest_v0.1.csv,.csv,1.589820,8,patient_id | split | eye_side | image_id | eye...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
3,EyePACS_2015,EyePACS_2015_Split_Grade_Gap_Audit_v0.1.csv,.csv,0.000160,8,split | absolute_grade_gap | patients,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
4,EyePACS_2015,EyePACS_2015_Filtered_Image_Acquisition_Manife...,.csv,1.204515,7,patient_id | split | eye_side | image_id | eye...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
5,EyePACS_2015,EyePACS_2015_Cropped_Availability_Manifest_v0....,.csv,1.187369,7,patient_id | split | eye_side | image_id | eye...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
6,EyePACS_2015,EyePACS_Clean_Validation_PairedDifference_Pred...,.csv,0.046663,7,patient_id | split | left_image_id | right_ima...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
7,EyePACS_2015,EyePACS_2015_Image_Acquisition_Manifest_v0.1.csv,.csv,1.148883,6,patient_id | split | eye_side | image_id | eye...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
8,EyePACS_2015,EyePACS_2015_Split_Decision_v0.1.json,.json,0.000454,6,decision | random_seed | total_unequal_grade_p...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
9,EyePACS_2015,EyePACS_2015_Patient_Split_v0.1.csv,.csv,0.176107,5,patient_id | left | right | absolute_grade_gap...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...



================ STAGE 4A DECISION ================
Decision:
HOLD_SOURCE_CANONICALISATION_MISSING_OR_AMBIGUOUS_SOURCE_ARTIFACTS

Interpretation:
At least one source lacks either discoverable image material or discoverable metadata artifacts. Source canonicalisation must not begin until the missing material is located or reacquired.

Authorised next step:
REACQUIRE_OR_LOCATE_MISSING_SOURCE_MATERIAL

Target sealed-label files accessed:
False

Source performance observed:
False

Source-target transfer performance observed:
False

Stage 4A source artifact audit completed and sealed.


In [25]:
#@title 04B. Exact source-cohort recovery and project-wide image path resolution

from pathlib import Path
from datetime import datetime, timezone

import json
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen audit boundary
# ============================================================

EYEPACS_EXPECTED_IMAGES = 4468
DEEPDRID_EXPECTED_IMAGES = 1600

IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".tif",
    ".tiff",
    ".bmp",
}

TABULAR_EXTENSIONS = {
    ".csv",
    ".parquet",
}

ARRAY_EXTENSIONS = {
    ".npz",
    ".npy",
}

NOTEBOOK_EXTENSIONS = {
    ".ipynb",
    ".py",
}

SAFETY_BOUNDARY = (
    "Stage 4B performs read-only source-cohort recovery and "
    "source-image path resolution. It may read source metadata, "
    "source labels, array structure, and notebook text. It must "
    "not access target sealed-label files, compute source or "
    "transfer performance, alter source files, or begin source "
    "canonicalisation."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

RETINAL_CODE_ROOT = (
    PROJECT_ROOT
    / "05_Code"
    / "Retinal_DR"
)

PROSPECTIVE_TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE4_ROOT = (
    PROSPECTIVE_TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4A_ROOT = (
    STAGE4_ROOT
    / "00_Source_Artifact_Audit"
)

STAGE4B_ROOT = (
    STAGE4_ROOT
    / "01_Exact_Source_Cohort_Resolution"
)

STAGE4B_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE4A_DECISION_PATH = (
    STAGE4A_ROOT
    / "Stage4A_Source_Artifact_Audit_Decision_v0.1.json"
)

STAGE4A_METADATA_INVENTORY_PATH = (
    STAGE4A_ROOT
    / "Stage4A_Source_Metadata_Artifact_Inventory_v0.1.csv"
)


for required_path in [
    STAGE4A_DECISION_PATH,
    STAGE4A_METADATA_INVENTORY_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with open(
    STAGE4A_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage4a_decision = json.load(
        file
    )


assert (
    stage4a_decision[
        "decision"
    ]
    ==
    "HOLD_SOURCE_CANONICALISATION_MISSING_OR_AMBIGUOUS_SOURCE_ARTIFACTS"
)


metadata_inventory = pd.read_csv(
    STAGE4A_METADATA_INVENTORY_PATH
)


print(
    "================ STAGE 4B EXACT SOURCE "
    "COHORT RESOLUTION ================"
)

print("\nImported Stage 4A decision:")
print(
    stage4a_decision[
        "decision"
    ]
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)


# ============================================================
# 2. Helpers
# ============================================================

def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def normalise_image_id(
    value,
):
    value = str(
        value
    ).strip()

    if not value:
        return ""

    return Path(
        value
    ).stem


def first_matching_column(
    dataframe,
    candidate_names,
):
    normalised_lookup = {
        normalise_column_name(
            column
        ): column
        for column in dataframe.columns
    }

    for candidate in candidate_names:
        key = normalise_column_name(
            candidate
        )

        if key in normalised_lookup:
            return normalised_lookup[
                key
            ]

    return None


def first_pattern_column(
    dataframe,
    patterns,
):
    for column in dataframe.columns:
        column_key = normalise_column_name(
            column
        )

        if any(
            pattern in column_key
            for pattern in patterns
        ):
            return column

    return None


def path_is_inside(
    path,
    parent,
):
    path = Path(
        path
    ).resolve()

    parent = Path(
        parent
    ).resolve()

    try:
        path.relative_to(
            parent
        )

        return True

    except ValueError:
        return False


def safe_read_table(
    path,
):
    path = Path(
        path
    )

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(
            path,
            low_memory=False,
        )

    if suffix == ".parquet":
        return pd.read_parquet(
            path
        )

    raise ValueError(
        f"Unsupported table: {path}"
    )


def table_summary_record(
    path,
    dataframe,
):
    image_id_column = first_matching_column(
        dataframe,
        [
            "image_id",
            "imageid",
            "id_code",
            "image",
            "image_name",
            "filename",
            "file_name",
            "eye_id",
        ],
    )

    if image_id_column is None:
        image_id_column = first_pattern_column(
            dataframe,
            [
                "imageid",
                "imagename",
                "filename",
            ],
        )

    grade_column = first_matching_column(
        dataframe,
        [
            "dr_grade",
            "grade",
            "diagnosis",
            "level",
            "retinopathy_grade",
            "retinopathy grade",
            "label",
        ],
    )

    if grade_column is None:
        grade_column = first_pattern_column(
            dataframe,
            [
                "drgrade",
                "retinopathygrade",
                "diagnosis",
            ],
        )

    split_column = first_matching_column(
        dataframe,
        [
            "split",
            "partition",
            "subset",
            "fold",
            "set",
        ],
    )

    patient_column = first_matching_column(
        dataframe,
        [
            "patient_id",
            "patientid",
            "patient",
            "subject_id",
            "subject",
        ],
    )

    eye_side_column = first_matching_column(
        dataframe,
        [
            "eye_side",
            "eyeside",
            "laterality",
            "side",
            "eye",
        ],
    )

    row_key_column = first_matching_column(
        dataframe,
        [
            "embedding_row",
            "row_index",
            "row",
            "index",
        ],
    )

    unique_image_ids = None

    if image_id_column is not None:
        unique_image_ids = int(
            dataframe[
                image_id_column
            ]
            .map(
                normalise_image_id
            )
            .nunique()
        )

    return {
        "path": str(
            path
        ),
        "filename": (
            path.name
        ),
        "rows": int(
            len(
                dataframe
            )
        ),
        "columns": int(
            len(
                dataframe.columns
            )
        ),
        "image_id_column": (
            image_id_column
        ),
        "unique_image_ids": (
            unique_image_ids
        ),
        "grade_column": (
            grade_column
        ),
        "split_column": (
            split_column
        ),
        "patient_column": (
            patient_column
        ),
        "eye_side_column": (
            eye_side_column
        ),
        "row_key_column": (
            row_key_column
        ),
        "column_names": (
            " | ".join(
                map(
                    str,
                    dataframe.columns,
                )
            )
        ),
    }


def manifest_score(
    record,
    expected_rows,
    preferred_name_tokens,
):
    score = 0

    if record[
        "rows"
    ] == expected_rows:
        score += 100

    if record[
        "unique_image_ids"
    ] == expected_rows:
        score += 80

    if record[
        "image_id_column"
    ] is not None:
        score += 20

    if record[
        "grade_column"
    ] is not None:
        score += 20

    if record[
        "split_column"
    ] is not None:
        score += 10

    if record[
        "patient_column"
    ] is not None:
        score += 5

    filename_lower = record[
        "filename"
    ].lower()

    for token, points in preferred_name_tokens:
        if token in filename_lower:
            score += points

    return int(
        score
    )


def standardise_source_manifest(
    dataframe,
    dataset_name,
    source_manifest_path,
):
    image_id_column = first_matching_column(
        dataframe,
        [
            "image_id",
            "imageid",
            "id_code",
            "image",
            "image_name",
            "filename",
            "file_name",
            "eye_id",
        ],
    )

    if image_id_column is None:
        image_id_column = first_pattern_column(
            dataframe,
            [
                "imageid",
                "imagename",
                "filename",
            ],
        )

    assert image_id_column is not None

    grade_column = first_matching_column(
        dataframe,
        [
            "dr_grade",
            "grade",
            "diagnosis",
            "level",
            "retinopathy_grade",
            "retinopathy grade",
            "label",
        ],
    )

    if grade_column is None:
        grade_column = first_pattern_column(
            dataframe,
            [
                "drgrade",
                "retinopathygrade",
                "diagnosis",
            ],
        )

    split_column = first_matching_column(
        dataframe,
        [
            "split",
            "partition",
            "subset",
            "fold",
            "set",
        ],
    )

    patient_column = first_matching_column(
        dataframe,
        [
            "patient_id",
            "patientid",
            "patient",
            "subject_id",
            "subject",
        ],
    )

    eye_side_column = first_matching_column(
        dataframe,
        [
            "eye_side",
            "eyeside",
            "laterality",
            "side",
            "eye",
        ],
    )

    output = pd.DataFrame(
        {
            "dataset": dataset_name,
            "image_id": (
                dataframe[
                    image_id_column
                ]
                .map(
                    normalise_image_id
                )
            ),
            "source_manifest_path": str(
                source_manifest_path
            ),
        }
    )

    output[
        "dr_grade"
    ] = (
        dataframe[
            grade_column
        ]
        if grade_column is not None
        else pd.NA
    )

    output[
        "split"
    ] = (
        dataframe[
            split_column
        ].astype(str)
        if split_column is not None
        else pd.NA
    )

    output[
        "patient_id"
    ] = (
        dataframe[
            patient_column
        ].astype(str)
        if patient_column is not None
        else pd.NA
    )

    output[
        "eye_side"
    ] = (
        dataframe[
            eye_side_column
        ].astype(str)
        if eye_side_column is not None
        else pd.NA
    )

    path_like_columns = [
        column
        for column in dataframe.columns
        if any(
            token in normalise_column_name(
                column
            )
            for token in [
                "path",
                "filepath",
                "imagefile",
                "localfile",
                "croppedfile",
            ]
        )
    ]

    resolved_embedded_paths = []

    for row_index in range(
        len(
            dataframe
        )
    ):
        resolved_path = ""

        for column in path_like_columns:
            raw_value = dataframe.iloc[
                row_index
            ][
                column
            ]

            if pd.isna(
                raw_value
            ):
                continue

            candidate_path = Path(
                str(
                    raw_value
                )
            )

            if candidate_path.is_file():
                resolved_path = str(
                    candidate_path
                )
                break

        resolved_embedded_paths.append(
            resolved_path
        )

    output[
        "existing_embedded_image_path"
    ] = resolved_embedded_paths

    return output


# ============================================================
# 3. Resolve EyePACS exact historical cohort
# ============================================================

eyepacs_inventory = metadata_inventory[
    metadata_inventory[
        "dataset"
    ]
    ==
    "EyePACS_2015"
].copy()


eyepacs_table_records = []
eyepacs_table_objects = {}


for _, row in eyepacs_inventory.iterrows():
    candidate_path = Path(
        row[
            "path"
        ]
    )

    if (
        not candidate_path.is_file()
        or
        candidate_path.suffix.lower()
        not in TABULAR_EXTENSIONS
    ):
        continue

    if candidate_path.stat().st_size > (
        100
        *
        1024 ** 2
    ):
        continue

    try:
        dataframe = safe_read_table(
            candidate_path
        )

    except Exception:
        continue

    record = table_summary_record(
        candidate_path,
        dataframe,
    )

    record[
        "manifest_score"
    ] = manifest_score(
        record,
        expected_rows=(
            EYEPACS_EXPECTED_IMAGES
        ),
        preferred_name_tokens=[
            (
                "canonical_metadata",
                30,
            ),
            (
                "acquired_image_manifest",
                25,
            ),
            (
                "filtered_image_acquisition",
                20,
            ),
            (
                "image_acquisition_manifest",
                15,
            ),
            (
                "embedding",
                -10,
            ),
            (
                "prediction",
                -20,
            ),
        ],
    )

    eyepacs_table_records.append(
        record
    )

    eyepacs_table_objects[
        str(
            candidate_path
        )
    ] = dataframe


eyepacs_candidate_summary = pd.DataFrame(
    eyepacs_table_records
)


if len(
    eyepacs_candidate_summary
) > 0:
    eyepacs_candidate_summary = (
        eyepacs_candidate_summary
        .sort_values(
            [
                "manifest_score",
                "rows",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


eyepacs_exact_manifest_recovered = False
eyepacs_exact_manifest = None
eyepacs_selected_manifest_path = None


if len(
    eyepacs_candidate_summary
) > 0:
    exact_candidates = (
        eyepacs_candidate_summary[
            (
                eyepacs_candidate_summary[
                    "rows"
                ]
                ==
                EYEPACS_EXPECTED_IMAGES
            )
            &
            (
                eyepacs_candidate_summary[
                    "unique_image_ids"
                ]
                ==
                EYEPACS_EXPECTED_IMAGES
            )
            &
            (
                eyepacs_candidate_summary[
                    "grade_column"
                ]
                .notna()
            )
        ]
    )

    if len(
        exact_candidates
    ) > 0:
        selected_record = (
            exact_candidates.iloc[0]
        )

        eyepacs_selected_manifest_path = Path(
            selected_record[
                "path"
            ]
        )

        selected_dataframe = (
            eyepacs_table_objects[
                str(
                    eyepacs_selected_manifest_path
                )
            ]
        )

        eyepacs_exact_manifest = (
            standardise_source_manifest(
                dataframe=(
                    selected_dataframe
                ),
                dataset_name=(
                    "EyePACS_2015"
                ),
                source_manifest_path=(
                    eyepacs_selected_manifest_path
                ),
            )
        )

        assert (
            len(
                eyepacs_exact_manifest
            )
            ==
            EYEPACS_EXPECTED_IMAGES
        )

        assert eyepacs_exact_manifest[
            "image_id"
        ].is_unique

        eyepacs_exact_manifest_recovered = True


# ============================================================
# 4. Broad DeepDRiD artifact search
# ============================================================

deepdrid_search_roots = [
    root
    for root in [
        RETINAL_DATA_ROOT,
        RETINAL_CODE_ROOT,
    ]
    if root.is_dir()
]


deepdrid_table_records = []
deepdrid_table_objects = {}

array_records = []
notebook_records = []


for search_root in deepdrid_search_roots:
    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_root_path = Path(
            current_root
        )

        if path_is_inside(
            current_root_path,
            PROSPECTIVE_TEST_ROOT,
        ):
            directory_names[:] = []
            continue

        directory_names[:] = [
            name
            for name in directory_names
            if name not in {
                ".ipynb_checkpoints",
                "__pycache__",
            }
        ]

        for filename in filenames:
            file_path = (
                current_root_path
                /
                filename
            )

            suffix = file_path.suffix.lower()

            # ----------------------------------------------
            # Tables
            # ----------------------------------------------
            if suffix in TABULAR_EXTENSIONS:
                if file_path.stat().st_size > (
                    100
                    *
                    1024 ** 2
                ):
                    continue

                try:
                    dataframe = safe_read_table(
                        file_path
                    )

                except Exception:
                    continue

                record = table_summary_record(
                    file_path,
                    dataframe,
                )

                filename_lower = (
                    file_path.name.lower()
                )

                column_text = (
                    record[
                        "column_names"
                    ].lower()
                )

                deepdrid_relevance = int(
                    "deepdrid"
                    in str(
                        file_path
                    ).lower()
                    or
                    "deep_drid"
                    in str(
                        file_path
                    ).lower()
                    or
                    "deep-drid"
                    in str(
                        file_path
                    ).lower()
                    or
                    (
                        record[
                            "rows"
                        ]
                        ==
                        DEEPDRID_EXPECTED_IMAGES
                    )
                    or
                    "embedding_row"
                    in column_text
                )

                if deepdrid_relevance:
                    record[
                        "exact_1600_rows"
                    ] = bool(
                        record[
                            "rows"
                        ]
                        ==
                        DEEPDRID_EXPECTED_IMAGES
                    )

                    record[
                        "direct_exact_cohort_candidate"
                    ] = bool(
                        record[
                            "rows"
                        ]
                        ==
                        DEEPDRID_EXPECTED_IMAGES
                        and
                        record[
                            "unique_image_ids"
                        ]
                        ==
                        DEEPDRID_EXPECTED_IMAGES
                        and
                        record[
                            "grade_column"
                        ]
                        is not None
                    )

                    record[
                        "candidate_score"
                    ] = (
                        manifest_score(
                            record,
                            expected_rows=(
                                DEEPDRID_EXPECTED_IMAGES
                            ),
                            preferred_name_tokens=[
                                (
                                    "manifest",
                                    20,
                                ),
                                (
                                    "metadata",
                                    20,
                                ),
                                (
                                    "label",
                                    15,
                                ),
                                (
                                    "grade",
                                    15,
                                ),
                                (
                                    "image_size",
                                    2,
                                ),
                                (
                                    "embedding",
                                    -5,
                                ),
                            ],
                        )
                    )

                    deepdrid_table_records.append(
                        record
                    )

                    deepdrid_table_objects[
                        str(
                            file_path
                        )
                    ] = dataframe

            # ----------------------------------------------
            # NPZ / NPY arrays
            # ----------------------------------------------
            elif suffix in ARRAY_EXTENSIONS:
                if file_path.stat().st_size > (
                    1000
                    *
                    1024 ** 2
                ):
                    continue

                try:
                    if suffix == ".npz":
                        with np.load(
                            file_path,
                            allow_pickle=False,
                        ) as archive:
                            key_records = []

                            has_first_dimension_1600 = False
                            candidate_id_keys = []
                            candidate_grade_keys = []

                            for key in archive.files:
                                try:
                                    array = archive[
                                        key
                                    ]

                                except Exception:
                                    continue

                                shape = tuple(
                                    int(
                                        value
                                    )
                                    for value in array.shape
                                )

                                first_dimension = (
                                    shape[0]
                                    if len(shape) > 0
                                    else None
                                )

                                if (
                                    first_dimension
                                    ==
                                    DEEPDRID_EXPECTED_IMAGES
                                ):
                                    has_first_dimension_1600 = True

                                    key_lower = key.lower()

                                    if any(
                                        token in key_lower
                                        for token in [
                                            "image_id",
                                            "imageid",
                                            "filename",
                                            "name",
                                            "path",
                                        ]
                                    ):
                                        candidate_id_keys.append(
                                            key
                                        )

                                    if any(
                                        token in key_lower
                                        for token in [
                                            "grade",
                                            "label",
                                            "diagnosis",
                                            "target",
                                        ]
                                    ):
                                        candidate_grade_keys.append(
                                            key
                                        )

                                key_records.append(
                                    {
                                        "key": key,
                                        "shape": str(
                                            shape
                                        ),
                                        "dtype": str(
                                            array.dtype
                                        ),
                                    }
                                )

                            if (
                                has_first_dimension_1600
                                or
                                "deepdrid"
                                in str(
                                    file_path
                                ).lower()
                            ):
                                array_records.append(
                                    {
                                        "path": str(
                                            file_path
                                        ),
                                        "filename": (
                                            file_path.name
                                        ),
                                        "suffix": suffix,
                                        "size_mib": float(
                                            file_path
                                            .stat()
                                            .st_size
                                            /
                                            (1024 ** 2)
                                        ),
                                        "has_first_dimension_1600": (
                                            has_first_dimension_1600
                                        ),
                                        "candidate_id_keys": (
                                            " | ".join(
                                                candidate_id_keys
                                            )
                                        ),
                                        "candidate_grade_keys": (
                                            " | ".join(
                                                candidate_grade_keys
                                            )
                                        ),
                                        "array_structure": json.dumps(
                                            key_records
                                        ),
                                    }
                                )

                    else:
                        array = np.load(
                            file_path,
                            mmap_mode="r",
                            allow_pickle=False,
                        )

                        shape = tuple(
                            int(
                                value
                            )
                            for value in array.shape
                        )

                        if (
                            len(shape) > 0
                            and
                            shape[0]
                            ==
                            DEEPDRID_EXPECTED_IMAGES
                        ):
                            array_records.append(
                                {
                                    "path": str(
                                        file_path
                                    ),
                                    "filename": (
                                        file_path.name
                                    ),
                                    "suffix": suffix,
                                    "size_mib": float(
                                        file_path
                                        .stat()
                                        .st_size
                                        /
                                        (1024 ** 2)
                                    ),
                                    "has_first_dimension_1600": True,
                                    "candidate_id_keys": "",
                                    "candidate_grade_keys": "",
                                    "array_structure": json.dumps(
                                        [
                                            {
                                                "key": (
                                                    "single_array"
                                                ),
                                                "shape": str(
                                                    shape
                                                ),
                                                "dtype": str(
                                                    array.dtype
                                                ),
                                            }
                                        ]
                                    ),
                                }
                            )

                except Exception:
                    pass

            # ----------------------------------------------
            # Notebook and Python source references
            # ----------------------------------------------
            elif suffix in NOTEBOOK_EXTENSIONS:
                if file_path.stat().st_size > (
                    30
                    *
                    1024 ** 2
                ):
                    continue

                try:
                    text = file_path.read_text(
                        encoding="utf-8",
                        errors="ignore",
                    )

                except Exception:
                    continue

                lower_text = text.lower()

                if (
                    "deepdrid"
                    not in lower_text
                    and
                    "deep_drid"
                    not in lower_text
                    and
                    "deep-drid"
                    not in lower_text
                ):
                    continue

                lines = text.splitlines()

                matching_lines = []

                for line_number, line in enumerate(
                    lines,
                    start=1,
                ):
                    line_lower = line.lower()

                    if any(
                        token in line_lower
                        for token in [
                            "deepdrid",
                            "deep_drid",
                            "deep-drid",
                        ]
                    ):
                        clean_line = re.sub(
                            r"\s+",
                            " ",
                            line,
                        ).strip()

                        if clean_line:
                            matching_lines.append(
                                {
                                    "line_number": int(
                                        line_number
                                    ),
                                    "text": clean_line[
                                        :1000
                                    ],
                                }
                            )

                    if len(
                        matching_lines
                    ) >= 50:
                        break

                notebook_records.append(
                    {
                        "path": str(
                            file_path
                        ),
                        "filename": (
                            file_path.name
                        ),
                        "matching_lines": int(
                            len(
                                matching_lines
                            )
                        ),
                        "reference_preview": json.dumps(
                            matching_lines
                        ),
                    }
                )


deepdrid_candidate_summary = pd.DataFrame(
    deepdrid_table_records
)

array_candidate_summary = pd.DataFrame(
    array_records
)

notebook_reference_summary = pd.DataFrame(
    notebook_records
)


if len(
    deepdrid_candidate_summary
) > 0:
    deepdrid_candidate_summary = (
        deepdrid_candidate_summary
        .sort_values(
            [
                "candidate_score",
                "exact_1600_rows",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 5. Attempt direct DeepDRiD exact-cohort recovery
# ============================================================

deepdrid_exact_manifest_recovered = False
deepdrid_exact_manifest = None
deepdrid_selected_manifest_path = None


if len(
    deepdrid_candidate_summary
) > 0:
    direct_candidates = (
        deepdrid_candidate_summary[
            deepdrid_candidate_summary[
                "direct_exact_cohort_candidate"
            ]
        ]
    )

    if len(
        direct_candidates
    ) > 0:
        selected_record = (
            direct_candidates.iloc[0]
        )

        deepdrid_selected_manifest_path = Path(
            selected_record[
                "path"
            ]
        )

        selected_dataframe = (
            deepdrid_table_objects[
                str(
                    deepdrid_selected_manifest_path
                )
            ]
        )

        deepdrid_exact_manifest = (
            standardise_source_manifest(
                dataframe=(
                    selected_dataframe
                ),
                dataset_name=(
                    "DeepDRiD"
                ),
                source_manifest_path=(
                    deepdrid_selected_manifest_path
                ),
            )
        )

        assert (
            len(
                deepdrid_exact_manifest
            )
            ==
            DEEPDRID_EXPECTED_IMAGES
        )

        assert deepdrid_exact_manifest[
            "image_id"
        ].is_unique

        deepdrid_exact_manifest_recovered = True


# ============================================================
# 6. Project-wide image filename scan
# ============================================================

eyepacs_id_set = (
    set(
        eyepacs_exact_manifest[
            "image_id"
        ].astype(str)
    )
    if eyepacs_exact_manifest_recovered
    else set()
)

deepdrid_id_set = (
    set(
        deepdrid_exact_manifest[
            "image_id"
        ].astype(str)
    )
    if deepdrid_exact_manifest_recovered
    else set()
)


eyepacs_image_matches = []
deepdrid_image_matches = []

image_parent_counts = {}


for current_root, directory_names, filenames in os.walk(
    PROJECT_ROOT
):
    current_root_path = Path(
        current_root
    )

    if path_is_inside(
        current_root_path,
        PROSPECTIVE_TEST_ROOT,
    ):
        directory_names[:] = []
        continue

    directory_names[:] = [
        name
        for name in directory_names
        if name not in {
            ".ipynb_checkpoints",
            "__pycache__",
        }
    ]

    current_image_count = 0

    for filename in filenames:
        file_path = (
            current_root_path
            /
            filename
        )

        if (
            file_path
            .suffix
            .lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        current_image_count += 1

        image_id = normalise_image_id(
            filename
        )

        if image_id in eyepacs_id_set:
            eyepacs_image_matches.append(
                {
                    "image_id": (
                        image_id
                    ),
                    "path": str(
                        file_path
                    ),
                    "filename": (
                        file_path.name
                    ),
                    "size_bytes": int(
                        file_path
                        .stat()
                        .st_size
                    ),
                }
            )

        if image_id in deepdrid_id_set:
            deepdrid_image_matches.append(
                {
                    "image_id": (
                        image_id
                    ),
                    "path": str(
                        file_path
                    ),
                    "filename": (
                        file_path.name
                    ),
                    "size_bytes": int(
                        file_path
                        .stat()
                        .st_size
                    ),
                }
            )

    if current_image_count > 0:
        image_parent_counts[
            str(
                current_root_path
            )
        ] = int(
            current_image_count
        )


eyepacs_image_match_table = pd.DataFrame(
    eyepacs_image_matches
)

deepdrid_image_match_table = pd.DataFrame(
    deepdrid_image_matches
)


large_image_parent_summary = pd.DataFrame(
    [
        {
            "parent_directory": path,
            "direct_image_files": count,
            "distance_from_4468": int(
                abs(
                    count
                    -
                    EYEPACS_EXPECTED_IMAGES
                )
            ),
            "distance_from_1600": int(
                abs(
                    count
                    -
                    DEEPDRID_EXPECTED_IMAGES
                )
            ),
        }
        for path, count in (
            image_parent_counts.items()
        )
        if count >= 100
    ]
)


if len(
    large_image_parent_summary
) > 0:
    large_image_parent_summary = (
        large_image_parent_summary
        .sort_values(
            [
                "distance_from_1600",
                "distance_from_4468",
                "direct_image_files",
            ],
            ascending=[
                True,
                True,
                False,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 7. Attach resolved image paths to exact manifests
# ============================================================

if eyepacs_exact_manifest_recovered:
    if len(
        eyepacs_image_match_table
    ) > 0:
        eyepacs_match_counts = (
            eyepacs_image_match_table
            .groupby(
                "image_id"
            )
            .size()
        )

        eyepacs_unique_match_map = (
            eyepacs_image_match_table
            .drop_duplicates(
                "image_id",
                keep="first",
            )
            .set_index(
                "image_id"
            )[
                "path"
            ]
            .to_dict()
        )

    else:
        eyepacs_match_counts = pd.Series(
            dtype=int
        )

        eyepacs_unique_match_map = {}

    eyepacs_exact_manifest[
        "project_image_match_count"
    ] = (
        eyepacs_exact_manifest[
            "image_id"
        ]
        .map(
            eyepacs_match_counts
        )
        .fillna(0)
        .astype(int)
    )

    eyepacs_exact_manifest[
        "project_resolved_image_path"
    ] = (
        eyepacs_exact_manifest[
            "image_id"
        ]
        .map(
            eyepacs_unique_match_map
        )
        .fillna("")
    )


if deepdrid_exact_manifest_recovered:
    if len(
        deepdrid_image_match_table
    ) > 0:
        deepdrid_match_counts = (
            deepdrid_image_match_table
            .groupby(
                "image_id"
            )
            .size()
        )

        deepdrid_unique_match_map = (
            deepdrid_image_match_table
            .drop_duplicates(
                "image_id",
                keep="first",
            )
            .set_index(
                "image_id"
            )[
                "path"
            ]
            .to_dict()
        )

    else:
        deepdrid_match_counts = pd.Series(
            dtype=int
        )

        deepdrid_unique_match_map = {}

    deepdrid_exact_manifest[
        "project_image_match_count"
    ] = (
        deepdrid_exact_manifest[
            "image_id"
        ]
        .map(
            deepdrid_match_counts
        )
        .fillna(0)
        .astype(int)
    )

    deepdrid_exact_manifest[
        "project_resolved_image_path"
    ] = (
        deepdrid_exact_manifest[
            "image_id"
        ]
        .map(
            deepdrid_unique_match_map
        )
        .fillna("")
    )


# ============================================================
# 8. Aggregate resolution summary
# ============================================================

eyepacs_project_matched_ids = int(
    eyepacs_image_match_table[
        "image_id"
    ].nunique()
) if len(
    eyepacs_image_match_table
) > 0 else 0


deepdrid_project_matched_ids = int(
    deepdrid_image_match_table[
        "image_id"
    ].nunique()
) if len(
    deepdrid_image_match_table
) > 0 else 0


eyepacs_embedded_existing_paths = int(
    (
        eyepacs_exact_manifest[
            "existing_embedded_image_path"
        ]
        !=
        ""
    ).sum()
) if eyepacs_exact_manifest_recovered else 0


deepdrid_embedded_existing_paths = int(
    (
        deepdrid_exact_manifest[
            "existing_embedded_image_path"
        ]
        !=
        ""
    ).sum()
) if deepdrid_exact_manifest_recovered else 0


resolution_summary = pd.DataFrame(
    [
        {
            "dataset": (
                "EyePACS_2015"
            ),
            "expected_images": (
                EYEPACS_EXPECTED_IMAGES
            ),
            "exact_manifest_recovered": (
                eyepacs_exact_manifest_recovered
            ),
            "selected_manifest": (
                str(
                    eyepacs_selected_manifest_path
                )
                if
                eyepacs_selected_manifest_path
                is not None
                else
                ""
            ),
            "manifest_rows": (
                len(
                    eyepacs_exact_manifest
                )
                if
                eyepacs_exact_manifest_recovered
                else
                0
            ),
            "existing_embedded_paths": (
                eyepacs_embedded_existing_paths
            ),
            "project_exact_id_matches": (
                eyepacs_project_matched_ids
            ),
            "all_source_images_resolved": bool(
                eyepacs_project_matched_ids
                ==
                EYEPACS_EXPECTED_IMAGES
                or
                eyepacs_embedded_existing_paths
                ==
                EYEPACS_EXPECTED_IMAGES
            ),
        },
        {
            "dataset": (
                "DeepDRiD"
            ),
            "expected_images": (
                DEEPDRID_EXPECTED_IMAGES
            ),
            "exact_manifest_recovered": (
                deepdrid_exact_manifest_recovered
            ),
            "selected_manifest": (
                str(
                    deepdrid_selected_manifest_path
                )
                if
                deepdrid_selected_manifest_path
                is not None
                else
                ""
            ),
            "manifest_rows": (
                len(
                    deepdrid_exact_manifest
                )
                if
                deepdrid_exact_manifest_recovered
                else
                0
            ),
            "existing_embedded_paths": (
                deepdrid_embedded_existing_paths
            ),
            "project_exact_id_matches": (
                deepdrid_project_matched_ids
            ),
            "all_source_images_resolved": bool(
                deepdrid_project_matched_ids
                ==
                DEEPDRID_EXPECTED_IMAGES
                or
                deepdrid_embedded_existing_paths
                ==
                DEEPDRID_EXPECTED_IMAGES
            ),
        },
    ]
)


both_manifests_recovered = bool(
    resolution_summary[
        "exact_manifest_recovered"
    ].all()
)


both_image_sets_resolved = bool(
    resolution_summary[
        "all_source_images_resolved"
    ].all()
)


if (
    both_manifests_recovered
    and
    both_image_sets_resolved
):
    stage4b_decision = (
        "PASS_BOTH_EXACT_SOURCE_COHORTS_AND_IMAGES_"
        "RESOLVED_ADVANCE_TO_CANONICALISATION"
    )

    authorised_next_step = (
        "CANONICALISE_BOTH_SOURCE_COHORTS"
    )

    interpretation = (
        "Exact historical source manifests and all corresponding "
        "source images were resolved for both EyePACS and DeepDRiD. "
        "Source canonicalisation may begin using the frozen target "
        "preprocessing protocol."
    )

elif both_manifests_recovered:
    stage4b_decision = (
        "PASS_EXACT_SOURCE_COHORTS_RECOVERED_"
        "REACQUIRE_MISSING_IMAGES_ONLY"
    )

    authorised_next_step = (
        "REACQUIRE_ONLY_THE_UNRESOLVED_SOURCE_IMAGES"
    )

    interpretation = (
        "The exact historical source cohorts were recovered for "
        "both datasets, but at least one source image set is absent "
        "from the current project storage. Reacquisition may be "
        "limited to the missing image material; the historical "
        "cohort identities and labels do not need to be redesigned."
    )

elif (
    eyepacs_exact_manifest_recovered
    and
    not deepdrid_exact_manifest_recovered
):
    stage4b_decision = (
        "HOLD_DEEPDRID_EXACT_COHORT_IDENTITY_UNRESOLVED"
    )

    authorised_next_step = (
        "RECOVER_DEEPDRID_IMAGE_IDENTITIES_AND_LABEL_ORDER"
    )

    interpretation = (
        "The exact 4,468-image EyePACS historical cohort was "
        "recovered, but the current artifacts do not yet establish "
        "the exact 1,600 DeepDRiD image identities and labels. "
        "DeepDRiD must be reconstructed from a reliable manifest, "
        "array metadata, or the original deterministic loading order "
        "before source canonicalisation begins."
    )

else:
    stage4b_decision = (
        "HOLD_SOURCE_COHORT_IDENTITIES_UNRESOLVED"
    )

    authorised_next_step = (
        "RECOVER_EXACT_SOURCE_MANIFESTS_BEFORE_REACQUISITION"
    )

    interpretation = (
        "At least one historical source cohort cannot yet be "
        "reconstructed exactly from the retained artifacts. "
        "Reacquiring images before resolving cohort identity could "
        "silently change the precommitted source data and is not "
        "authorised."
    )


# ============================================================
# 9. Save Stage 4B audit artifacts
# ============================================================

EYEPACS_CANDIDATE_PATH = (
    STAGE4B_ROOT
    / "Stage4B_EyePACS_Manifest_Candidate_Summary_v0.1.csv"
)

DEEPDRID_TABLE_CANDIDATE_PATH = (
    STAGE4B_ROOT
    / "Stage4B_DeepDRiD_Table_Candidate_Summary_v0.1.csv"
)

DEEPDRID_ARRAY_CANDIDATE_PATH = (
    STAGE4B_ROOT
    / "Stage4B_DeepDRiD_Array_Candidate_Summary_v0.1.csv"
)

DEEPDRID_NOTEBOOK_REFERENCE_PATH = (
    STAGE4B_ROOT
    / "Stage4B_DeepDRiD_Notebook_Reference_Summary_v0.1.csv"
)

IMAGE_PARENT_SUMMARY_PATH = (
    STAGE4B_ROOT
    / "Stage4B_Project_Image_Parent_Summary_v0.1.csv"
)

EYEPACS_MATCH_PATH = (
    STAGE4B_ROOT
    / "Stage4B_EyePACS_Project_Image_Matches_v0.1.csv"
)

DEEPDRID_MATCH_PATH = (
    STAGE4B_ROOT
    / "Stage4B_DeepDRiD_Project_Image_Matches_v0.1.csv"
)

RESOLUTION_SUMMARY_PATH = (
    STAGE4B_ROOT
    / "Stage4B_Source_Cohort_Resolution_Summary_v0.1.csv"
)

DECISION_PATH = (
    STAGE4B_ROOT
    / "Stage4B_Source_Cohort_Resolution_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE4B_ROOT
    / "Stage4B_Source_Cohort_Resolution_Report_v0.1.md"
)


eyepacs_candidate_summary.to_csv(
    EYEPACS_CANDIDATE_PATH,
    index=False,
)

deepdrid_candidate_summary.to_csv(
    DEEPDRID_TABLE_CANDIDATE_PATH,
    index=False,
)

array_candidate_summary.to_csv(
    DEEPDRID_ARRAY_CANDIDATE_PATH,
    index=False,
)

notebook_reference_summary.to_csv(
    DEEPDRID_NOTEBOOK_REFERENCE_PATH,
    index=False,
)

large_image_parent_summary.to_csv(
    IMAGE_PARENT_SUMMARY_PATH,
    index=False,
)

eyepacs_image_match_table.to_csv(
    EYEPACS_MATCH_PATH,
    index=False,
)

deepdrid_image_match_table.to_csv(
    DEEPDRID_MATCH_PATH,
    index=False,
)

resolution_summary.to_csv(
    RESOLUTION_SUMMARY_PATH,
    index=False,
)


EYEPACS_EXACT_MANIFEST_PATH = (
    STAGE4B_ROOT
    / "Stage4B_EyePACS_Exact_Historical_Source_Cohort_v0.1.csv"
)

DEEPDRID_EXACT_MANIFEST_PATH = (
    STAGE4B_ROOT
    / "Stage4B_DeepDRiD_Exact_Historical_Source_Cohort_v0.1.csv"
)


if eyepacs_exact_manifest_recovered:
    eyepacs_exact_manifest.to_csv(
        EYEPACS_EXACT_MANIFEST_PATH,
        index=False,
    )


if deepdrid_exact_manifest_recovered:
    deepdrid_exact_manifest.to_csv(
        DEEPDRID_EXACT_MANIFEST_PATH,
        index=False,
    )


decision_payload = {
    "decision": (
        stage4b_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "stage4a_decision_path": str(
        STAGE4A_DECISION_PATH
    ),
    "resolution_summary": (
        resolution_summary
        .to_dict(
            orient="records"
        )
    ),
    "eyepacs_exact_manifest_path": (
        str(
            EYEPACS_EXACT_MANIFEST_PATH
        )
        if
        eyepacs_exact_manifest_recovered
        else
        None
    ),
    "deepdrid_exact_manifest_path": (
        str(
            DEEPDRID_EXACT_MANIFEST_PATH
        )
        if
        deepdrid_exact_manifest_recovered
        else
        None
    ),
    "deepdrid_exact_1600_row_table_candidates": int(
        (
            deepdrid_candidate_summary[
                "exact_1600_rows"
            ]
        ).sum()
    ) if len(
        deepdrid_candidate_summary
    ) > 0 else 0,
    "deepdrid_arrays_with_first_dimension_1600": int(
        (
            array_candidate_summary[
                "has_first_dimension_1600"
            ]
        ).sum()
    ) if len(
        array_candidate_summary
    ) > 0 else 0,
    "deepdrid_notebooks_with_references": int(
        len(
            notebook_reference_summary
        )
    ),
    "target_sealed_label_files_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_files_modified": False,
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


report_lines = [
    "# Stage 4B — Exact Source-Cohort Resolution",
    "",
    f"Decision: `{stage4b_decision}`",
    "",
    interpretation,
    "",
    "## Safety boundary",
    "",
    "- Target sealed-label files accessed: `False`",
    "- Source performance observed: `False`",
    "- Source-target transfer performance observed: `False`",
    "- Existing source files modified: `False`",
    "",
]


REPORT_PATH.write_text(
    "\n".join(
        report_lines
    ),
    encoding="utf-8",
)


# ============================================================
# 10. Display concise outputs
# ============================================================

print(
    "\n================ SOURCE COHORT "
    "RESOLUTION SUMMARY ================"
)

display(
    resolution_summary
)


print(
    "\n================ EYEPACS EXACT MANIFEST "
    "CANDIDATES ================"
)

if len(
    eyepacs_candidate_summary
) > 0:
    display(
        eyepacs_candidate_summary[
            [
                "filename",
                "rows",
                "unique_image_ids",
                "image_id_column",
                "grade_column",
                "split_column",
                "manifest_score",
                "path",
            ]
        ].head(
            15
        )
    )

else:
    print(
        "No readable EyePACS table candidates found."
    )


print(
    "\n================ DEEPDRID 1600-ROW "
    "TABLE CANDIDATES ================"
)

if len(
    deepdrid_candidate_summary
) > 0:
    display(
        deepdrid_candidate_summary[
            [
                "filename",
                "rows",
                "unique_image_ids",
                "image_id_column",
                "grade_column",
                "split_column",
                "row_key_column",
                "direct_exact_cohort_candidate",
                "path",
            ]
        ].head(
            20
        )
    )

else:
    print(
        "No DeepDRiD-relevant table candidates found."
    )


print(
    "\n================ DEEPDRID 1600-DIMENSION "
    "ARRAY CANDIDATES ================"
)

if len(
    array_candidate_summary
) > 0:
    display(
        array_candidate_summary[
            [
                "filename",
                "size_mib",
                "has_first_dimension_1600",
                "candidate_id_keys",
                "candidate_grade_keys",
                "path",
            ]
        ].head(
            20
        )
    )

else:
    print(
        "No arrays with a first dimension of 1600 were found."
    )


print(
    "\n================ PROJECT IMAGE FOLDER "
    "CANDIDATES ================"
)

if len(
    large_image_parent_summary
) > 0:
    display(
        large_image_parent_summary.head(
            20
        )
    )

else:
    print(
        "No non-target project folder contains "
        "100 or more directly stored images."
    )


print(
    "\n================ STAGE 4B DECISION "
    "================"
)

print("Decision:")
print(
    stage4b_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print(
    "\nTarget sealed-label files accessed:"
)

print(False)

print(
    "\nSource performance observed:"
)

print(False)

print(
    "\nSource-target transfer performance observed:"
)

print(False)

print(
    "\nStage 4B exact source-cohort resolution "
    "completed and sealed."
)

================ STAGE 4B EXACT SOURCE COHORT RESOLUTION ================

Imported Stage 4A decision:
HOLD_SOURCE_CANONICALISATION_MISSING_OR_AMBIGUOUS_SOURCE_ARTIFACTS

Safety boundary:
Stage 4B performs read-only source-cohort recovery and source-image path resolution. It may read source metadata, source labels, array structure, and notebook text. It must not access target sealed-label files, compute source or transfer performance, alter source files, or begin source canonicalisation.

================ SOURCE COHORT RESOLUTION SUMMARY ================


,dataset,expected_images,exact_manifest_recovered,selected_manifest,manifest_rows,existing_embedded_paths,project_exact_id_matches,all_source_images_resolved
0,EyePACS_2015,4468,False,,0,0,0,False
1,DeepDRiD,1600,False,,0,0,0,False



================ EYEPACS EXACT MANIFEST CANDIDATES ================


,filename,rows,unique_image_ids,image_id_column,grade_column,split_column,manifest_score,path
0,EyePACS_2015_Filtered_Image_Acquisition_Manife...,4468,4468.0,image_id,None,split,250,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,EyePACS_2015_Acquired_Image_Manifest_v0.1.csv,4468,4468.0,image_id,None,split,240,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,EyePACS_2015_Image_Integrity_Audit_v0.1.csv,4468,4468.0,image_id,None,split,215,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
3,EyePACS_2015_ResNet50_ImageNet1K_V2_Clean_Embe...,4468,4468.0,image_id,None,split,205,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
4,EyePACS_2015_Canonical_Metadata_v0.1.csv,35126,35126.0,image_id,dr_grade,None,75,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
5,EyePACS_2015_Image_Acquisition_Manifest_v0.1.csv,4480,4480.0,image_id,None,split,50,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
6,EyePACS_2015_Cropped_Availability_Manifest_v0....,4480,4480.0,image_id,None,split,35,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
7,EyePACS_2015_Patient_Split_v0.1.csv,2240,2240.0,left_image_id,None,split,35,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
8,EyePACS_2015_Missing_Cropped_Target_Images_v0....,7,7.0,image_id,None,split,35,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
9,EyePACS_2015_Cropped_Pair_Availability_v0.1.csv,2240,NaN,None,None,split,15,/content/drive/MyDrive/Cross-Modal_Diagnostic_...



================ DEEPDRID 1600-ROW TABLE CANDIDATES ================


...


================ DEEPDRID 1600-DIMENSION ARRAY CANDIDATES ================


,filename,size_mib,has_first_dimension_1600,candidate_id_keys,candidate_grade_keys,path
0,ResNet50_ImageNet1K_V2_Embeddings_float32.npy,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,ResNet50_ImageNet1K_V2_grayscale_ImageEmbeddin...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,ResNet50_ImageNet1K_V2_strong_blur_ImageEmbedd...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
3,ResNet50_ImageNet1K_V2_patch_shuffle_4x4_Image...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
4,ResNet50_ImageNet1K_V2_blur_sigma_1_ImageEmbed...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
5,ResNet50_ImageNet1K_V2_blur_sigma_2_ImageEmbed...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
6,ResNet50_ImageNet1K_V2_blur_sigma_4_ImageEmbed...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
7,ResNet50_ImageNet1K_V2_blur_sigma_6_ImageEmbed...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
8,ResNet50_ImageNet1K_V2_blur_sigma_8_ImageEmbed...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
9,ResNet50_ImageNet1K_V2_sobel_edges_ImageEmbedd...,12.500122,True,,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...



================ PROJECT IMAGE FOLDER CANDIDATES ================


,parent_directory,direct_image_files,distance_from_4468,distance_from_1600
0,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,413,4055,1187
1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,103,4365,1497
2,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4468,0,2868



================ STAGE 4B DECISION ================
Decision:
HOLD_SOURCE_COHORT_IDENTITIES_UNRESOLVED

Interpretation:
At least one historical source cohort cannot yet be reconstructed exactly from the retained artifacts. Reacquiring images before resolving cohort identity could silently change the precommitted source data and is not authorised.

Authorised next step:
RECOVER_EXACT_SOURCE_MANIFESTS_BEFORE_REACQUISITION

Target sealed-label files accessed:
False

Source performance observed:
False

Source-target transfer performance observed:
False

Stage 4B exact source-cohort resolution completed and sealed.


In [26]:
#@title 04C. Recover EyePACS exactly and resolve DeepDRiD grade/image provenance

from pathlib import Path
from datetime import datetime, timezone

import json
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen audit boundary
# ============================================================

EYEPACS_EXPECTED_IMAGES = 4468
DEEPDRID_EXPECTED_IMAGES = 1600

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"
}

TABLE_EXTENSIONS = {
    ".csv", ".parquet"
}

ARRAY_EXTENSIONS = {
    ".npy", ".npz"
}

NOTEBOOK_EXTENSIONS = {
    ".ipynb", ".py"
}

SAFETY_BOUNDARY = (
    "Stage 4C reconstructs source cohort identities and source-label "
    "provenance using retained source artifacts. It does not access "
    "target sealed-label files, calculate source or transfer "
    "performance, alter source artifacts, or canonicalise images."
)


# ============================================================
# 1. Resolve project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/drive/MyDrive")

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path("/content/gdrive/MyDrive")

else:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive")


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

RETINAL_CODE_ROOT = (
    PROJECT_ROOT
    / "05_Code"
    / "Retinal_DR"
)

TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE4_ROOT = (
    TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4B_ROOT = (
    STAGE4_ROOT
    / "01_Exact_Source_Cohort_Resolution"
)

STAGE4C_ROOT = (
    STAGE4_ROOT
    / "02_Source_Manifest_And_Provenance_Recovery"
)

STAGE4C_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE4B_DECISION_PATH = (
    STAGE4B_ROOT
    / "Stage4B_Source_Cohort_Resolution_Decision_v0.1.json"
)

EYEPACS_CANDIDATE_PATH = (
    STAGE4B_ROOT
    / "Stage4B_EyePACS_Manifest_Candidate_Summary_v0.1.csv"
)

DEEPDRID_TABLE_CANDIDATE_PATH = (
    STAGE4B_ROOT
    / "Stage4B_DeepDRiD_Table_Candidate_Summary_v0.1.csv"
)

IMAGE_PARENT_SUMMARY_PATH = (
    STAGE4B_ROOT
    / "Stage4B_Project_Image_Parent_Summary_v0.1.csv"
)


for required_path in [
    STAGE4B_DECISION_PATH,
    EYEPACS_CANDIDATE_PATH,
    DEEPDRID_TABLE_CANDIDATE_PATH,
    IMAGE_PARENT_SUMMARY_PATH,
]:
    assert required_path.is_file(), required_path


with open(
    STAGE4B_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage4b_decision = json.load(file)


assert (
    stage4b_decision["decision"]
    ==
    "HOLD_SOURCE_COHORT_IDENTITIES_UNRESOLVED"
)


eyepacs_candidates = pd.read_csv(
    EYEPACS_CANDIDATE_PATH
)

deepdrid_candidates = pd.read_csv(
    DEEPDRID_TABLE_CANDIDATE_PATH
)

image_parent_summary = pd.read_csv(
    IMAGE_PARENT_SUMMARY_PATH
)


print(
    "================ STAGE 4C SOURCE "
    "PROVENANCE RECOVERY ================"
)

print("\nImported Stage 4B decision:")
print(stage4b_decision["decision"])

print("\nSafety boundary:")
print(SAFETY_BOUNDARY)


# ============================================================
# 2. Helpers
# ============================================================

def normalise_column_name(value):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def normalise_image_id(value):
    if pd.isna(value):
        return ""

    value = str(value).strip()

    if not value:
        return ""

    return Path(value).stem


def path_is_inside(path, parent):
    try:
        Path(path).resolve().relative_to(
            Path(parent).resolve()
        )
        return True

    except ValueError:
        return False


def read_table(path, usecols=None, nrows=None):
    path = Path(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(
            path,
            usecols=usecols,
            nrows=nrows,
            low_memory=False,
        )

    if path.suffix.lower() == ".parquet":
        frame = pd.read_parquet(path)

        if usecols is not None:
            frame = frame[usecols]

        if nrows is not None:
            frame = frame.head(nrows)

        return frame

    raise ValueError(path)


def find_exact_column(dataframe, candidates):
    lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        key = normalise_column_name(candidate)

        if key in lookup:
            return lookup[key]

    return None


def find_columns_containing(dataframe, tokens):
    output = []

    for column in dataframe.columns:
        key = normalise_column_name(column)

        if any(token in key for token in tokens):
            output.append(column)

    return output


def side_token(column_name):
    key = normalise_column_name(column_name)

    if "left" in key:
        return "left"

    if "right" in key:
        return "right"

    return None


def compatible_id_grade_columns(id_column, grade_column):
    id_side = side_token(id_column)
    grade_side = side_token(grade_column)

    if id_side is None and grade_side is None:
        return True

    if id_side is not None and grade_side == id_side:
        return True

    return False


def dataframe_optional_column(dataframe, candidates):
    column = find_exact_column(
        dataframe,
        candidates,
    )

    if column is None:
        return pd.Series(
            [pd.NA] * len(dataframe)
        )

    return dataframe[column]


def stable_grade_mapping(frame):
    if len(frame) == 0:
        return (
            pd.DataFrame(
                columns=[
                    "image_id",
                    "dr_grade",
                ]
            ),
            0,
        )

    frame = frame[
        frame["image_id"] != ""
    ].copy()

    frame["dr_grade"] = pd.to_numeric(
        frame["dr_grade"],
        errors="coerce",
    )

    frame = frame[
        frame["dr_grade"].notna()
    ].copy()

    frame["dr_grade"] = frame[
        "dr_grade"
    ].astype(int)

    frame = frame[
        frame["dr_grade"].isin(
            [0, 1, 2, 3, 4]
        )
    ].copy()

    conflicts = int(
        (
            frame.groupby("image_id")[
                "dr_grade"
            ].nunique()
            >
            1
        ).sum()
    )

    stable = (
        frame
        .sort_values("image_id")
        .drop_duplicates(
            "image_id",
            keep="first",
        )
        .reset_index(drop=True)
    )

    return stable, conflicts


# ============================================================
# 3. Recover exact EyePACS cohort
# ============================================================

eyepacs_manifest_rows = eyepacs_candidates[
    (
        eyepacs_candidates["rows"]
        ==
        EYEPACS_EXPECTED_IMAGES
    )
    &
    (
        eyepacs_candidates[
            "unique_image_ids"
        ]
        ==
        EYEPACS_EXPECTED_IMAGES
    )
    &
    eyepacs_candidates[
        "image_id_column"
    ].notna()
].copy()


assert len(eyepacs_manifest_rows) > 0, (
    "No exact 4468-ID EyePACS manifest candidate remains."
)


eyepacs_manifest_rows = (
    eyepacs_manifest_rows
    .sort_values(
        "manifest_score",
        ascending=False,
    )
    .reset_index(drop=True)
)


eyepacs_cohort_path = Path(
    eyepacs_manifest_rows.iloc[0][
        "path"
    ]
)


eyepacs_metadata_rows = eyepacs_candidates[
    (
        eyepacs_candidates[
            "grade_column"
        ].notna()
    )
    &
    (
        eyepacs_candidates[
            "filename"
        ]
        .astype(str)
        .str.lower()
        .str.contains(
            "canonical_metadata",
            regex=False,
        )
    )
].copy()


if len(eyepacs_metadata_rows) == 0:
    eyepacs_metadata_rows = (
        eyepacs_candidates[
            eyepacs_candidates[
                "grade_column"
            ].notna()
        ]
        .copy()
    )


assert len(eyepacs_metadata_rows) > 0, (
    "No EyePACS metadata table with DR grades was found."
)


eyepacs_metadata_rows = (
    eyepacs_metadata_rows
    .sort_values(
        "rows",
        ascending=False,
    )
    .reset_index(drop=True)
)


eyepacs_metadata_path = Path(
    eyepacs_metadata_rows.iloc[0][
        "path"
    ]
)


eyepacs_cohort_raw = read_table(
    eyepacs_cohort_path
)

eyepacs_metadata_raw = read_table(
    eyepacs_metadata_path
)


eyepacs_cohort_id_column = find_exact_column(
    eyepacs_cohort_raw,
    [
        "image_id",
        "imageid",
        "id_code",
        "image",
        "filename",
    ],
)

eyepacs_metadata_id_column = find_exact_column(
    eyepacs_metadata_raw,
    [
        "image_id",
        "imageid",
        "id_code",
        "image",
        "filename",
    ],
)

eyepacs_grade_column = find_exact_column(
    eyepacs_metadata_raw,
    [
        "dr_grade",
        "grade",
        "diagnosis",
        "level",
    ],
)


assert eyepacs_cohort_id_column is not None
assert eyepacs_metadata_id_column is not None
assert eyepacs_grade_column is not None


eyepacs_cohort = pd.DataFrame(
    {
        "dataset": "EyePACS_2015",
        "image_id": (
            eyepacs_cohort_raw[
                eyepacs_cohort_id_column
            ].map(normalise_image_id)
        ),
        "split": dataframe_optional_column(
            eyepacs_cohort_raw,
            [
                "split",
                "partition",
                "subset",
            ],
        ),
        "patient_id": dataframe_optional_column(
            eyepacs_cohort_raw,
            [
                "patient_id",
                "patientid",
                "patient",
            ],
        ),
        "eye_side": dataframe_optional_column(
            eyepacs_cohort_raw,
            [
                "eye_side",
                "eyeside",
                "laterality",
                "side",
            ],
        ),
    }
)


assert (
    len(eyepacs_cohort)
    ==
    EYEPACS_EXPECTED_IMAGES
)

assert eyepacs_cohort[
    "image_id"
].is_unique


eyepacs_grade_map_raw = pd.DataFrame(
    {
        "image_id": (
            eyepacs_metadata_raw[
                eyepacs_metadata_id_column
            ].map(normalise_image_id)
        ),
        "dr_grade": (
            eyepacs_metadata_raw[
                eyepacs_grade_column
            ]
        ),
    }
)


eyepacs_grade_map, eyepacs_grade_conflicts = (
    stable_grade_mapping(
        eyepacs_grade_map_raw
    )
)


assert eyepacs_grade_conflicts == 0


eyepacs_exact_manifest = eyepacs_cohort.merge(
    eyepacs_grade_map,
    on="image_id",
    how="left",
    validate="one_to_one",
)


assert eyepacs_exact_manifest[
    "dr_grade"
].notna().all(), (
    "At least one selected EyePACS image has no DR grade."
)


eyepacs_exact_manifest[
    "dr_grade"
] = eyepacs_exact_manifest[
    "dr_grade"
].astype(int)


# ============================================================
# 4. Resolve exact EyePACS image directory
# ============================================================

eyepacs_folder_candidates = (
    image_parent_summary[
        image_parent_summary[
            "direct_image_files"
        ]
        ==
        EYEPACS_EXPECTED_IMAGES
    ]
    .copy()
)


eyepacs_folder_records = []
eyepacs_folder_maps = {}


for _, row in eyepacs_folder_candidates.iterrows():
    folder = Path(
        row["parent_directory"]
    )

    if not folder.is_dir():
        continue

    image_paths = [
        path
        for path in folder.iterdir()
        if (
            path.is_file()
            and
            path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    ]

    stem_to_paths = {}

    for image_path in image_paths:
        stem = normalise_image_id(
            image_path.name
        )

        stem_to_paths.setdefault(
            stem,
            [],
        ).append(
            image_path
        )

    cohort_ids = set(
        eyepacs_exact_manifest[
            "image_id"
        ]
    )

    matched_ids = cohort_ids.intersection(
        set(stem_to_paths)
    )

    ambiguous_ids = {
        image_id
        for image_id in matched_ids
        if len(
            stem_to_paths[image_id]
        )
        >
        1
    }

    eyepacs_folder_records.append(
        {
            "folder": str(folder),
            "direct_image_files": int(
                len(image_paths)
            ),
            "matched_cohort_ids": int(
                len(matched_ids)
            ),
            "missing_cohort_ids": int(
                len(cohort_ids - matched_ids)
            ),
            "extra_image_ids": int(
                len(
                    set(stem_to_paths)
                    -
                    cohort_ids
                )
            ),
            "ambiguous_matched_ids": int(
                len(ambiguous_ids)
            ),
            "exact_match": bool(
                len(matched_ids)
                ==
                EYEPACS_EXPECTED_IMAGES
                and
                len(ambiguous_ids)
                ==
                0
            ),
        }
    )

    eyepacs_folder_maps[
        str(folder)
    ] = stem_to_paths


eyepacs_folder_audit = pd.DataFrame(
    eyepacs_folder_records
)


eyepacs_images_resolved = False
eyepacs_selected_image_folder = None


if len(eyepacs_folder_audit) > 0:
    eyepacs_folder_audit = (
        eyepacs_folder_audit
        .sort_values(
            [
                "exact_match",
                "matched_cohort_ids",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .reset_index(drop=True)
    )

    exact_eye_folders = (
        eyepacs_folder_audit[
            eyepacs_folder_audit[
                "exact_match"
            ]
        ]
    )

    if len(exact_eye_folders) > 0:
        eyepacs_selected_image_folder = Path(
            exact_eye_folders.iloc[0][
                "folder"
            ]
        )

        selected_map = eyepacs_folder_maps[
            str(
                eyepacs_selected_image_folder
            )
        ]

        eyepacs_exact_manifest[
            "source_image_path"
        ] = (
            eyepacs_exact_manifest[
                "image_id"
            ].map(
                lambda image_id: str(
                    selected_map[
                        image_id
                    ][0]
                )
            )
        )

        eyepacs_images_resolved = bool(
            eyepacs_exact_manifest[
                "source_image_path"
            ].map(
                lambda value: Path(
                    value
                ).is_file()
            ).all()
        )


# ============================================================
# 5. Recover exact DeepDRiD image identity/order
# ============================================================

deepdrid_exact_id_rows = deepdrid_candidates[
    (
        deepdrid_candidates[
            "rows"
        ]
        ==
        DEEPDRID_EXPECTED_IMAGES
    )
    &
    (
        deepdrid_candidates[
            "unique_image_ids"
        ]
        ==
        DEEPDRID_EXPECTED_IMAGES
    )
    &
    deepdrid_candidates[
        "image_id_column"
    ].notna()
].copy()


assert len(deepdrid_exact_id_rows) > 0, (
    "No exact 1600-ID DeepDRiD index table was found."
)


deepdrid_exact_id_rows[
    "preferred_index"
] = (
    deepdrid_exact_id_rows[
        "filename"
    ]
    .astype(str)
    .str.lower()
    .str.contains(
        "embedding_index",
        regex=False,
    )
)


deepdrid_exact_id_rows = (
    deepdrid_exact_id_rows
    .sort_values(
        [
            "preferred_index",
            "candidate_score",
        ],
        ascending=[
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)


deepdrid_index_path = Path(
    deepdrid_exact_id_rows.iloc[0][
        "path"
    ]
)


deepdrid_index_raw = read_table(
    deepdrid_index_path
)


deepdrid_index_id_column = find_exact_column(
    deepdrid_index_raw,
    [
        "image_id",
        "imageid",
        "id_code",
        "image",
        "filename",
    ],
)


assert deepdrid_index_id_column is not None


deepdrid_identity_manifest = pd.DataFrame(
    {
        "dataset": "DeepDRiD",
        "image_id": (
            deepdrid_index_raw[
                deepdrid_index_id_column
            ].map(normalise_image_id)
        ),
        "split": dataframe_optional_column(
            deepdrid_index_raw,
            [
                "split",
                "partition",
                "subset",
            ],
        ),
        "embedding_row": dataframe_optional_column(
            deepdrid_index_raw,
            [
                "embedding_row",
                "row_index",
                "row",
                "index",
            ],
        ),
        "patient_id": dataframe_optional_column(
            deepdrid_index_raw,
            [
                "patient_id",
                "patientid",
                "patient",
                "subject_id",
            ],
        ),
        "eye_side": dataframe_optional_column(
            deepdrid_index_raw,
            [
                "eye_side",
                "eyeside",
                "laterality",
                "side",
            ],
        ),
    }
)


assert (
    len(deepdrid_identity_manifest)
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert deepdrid_identity_manifest[
    "image_id"
].is_unique


deepdrid_id_set = set(
    deepdrid_identity_manifest[
        "image_id"
    ]
)


# ============================================================
# 6. Search all retained source tables for DeepDRiD grades
# ============================================================

source_search_roots = [
    root
    for root in [
        RETINAL_DATA_ROOT,
        RETINAL_CODE_ROOT,
    ]
    if root.is_dir()
]


grade_mapping_records = []
full_grade_mappings = []


for search_root in source_search_roots:
    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_root_path = Path(
            current_root
        )

        if path_is_inside(
            current_root_path,
            TEST_ROOT,
        ):
            directory_names[:] = []
            continue

        directory_names[:] = [
            name
            for name in directory_names
            if name not in {
                ".ipynb_checkpoints",
                "__pycache__",
            }
        ]

        for filename in filenames:
            table_path = (
                current_root_path
                /
                filename
            )

            if (
                table_path.suffix.lower()
                not in TABLE_EXTENSIONS
            ):
                continue

            if table_path.stat().st_size > (
                150
                *
                1024 ** 2
            ):
                continue

            try:
                header = read_table(
                    table_path,
                    nrows=0,
                )

            except Exception:
                continue

            image_id_columns = find_columns_containing(
                header,
                [
                    "imageid",
                    "imagename",
                    "filename",
                    "leftimage",
                    "rightimage",
                    "eyeid",
                ],
            )

            grade_columns = find_columns_containing(
                header,
                [
                    "drgrade",
                    "retinopathygrade",
                    "diagnosis",
                    "diseasegrade",
                    "leftgrade",
                    "rightgrade",
                ],
            )

            if (
                not image_id_columns
                or
                not grade_columns
            ):
                continue

            columns_to_read = list(
                dict.fromkeys(
                    image_id_columns
                    +
                    grade_columns
                )
            )

            try:
                frame = read_table(
                    table_path,
                    usecols=columns_to_read,
                )

            except Exception:
                continue

            for image_column in image_id_columns:
                for grade_column in grade_columns:
                    if not compatible_id_grade_columns(
                        image_column,
                        grade_column,
                    ):
                        continue

                    candidate_raw = pd.DataFrame(
                        {
                            "image_id": (
                                frame[
                                    image_column
                                ].map(
                                    normalise_image_id
                                )
                            ),
                            "dr_grade": (
                                frame[
                                    grade_column
                                ]
                            ),
                        }
                    )

                    candidate_mapping, conflicts = (
                        stable_grade_mapping(
                            candidate_raw
                        )
                    )

                    matched_mapping = (
                        candidate_mapping[
                            candidate_mapping[
                                "image_id"
                            ].isin(
                                deepdrid_id_set
                            )
                        ]
                        .copy()
                    )

                    matched_unique_ids = int(
                        matched_mapping[
                            "image_id"
                        ].nunique()
                    )

                    unique_grades = sorted(
                        matched_mapping[
                            "dr_grade"
                        ].unique()
                        .astype(int)
                        .tolist()
                    ) if len(
                        matched_mapping
                    ) > 0 else []

                    full_exact_mapping = bool(
                        matched_unique_ids
                        ==
                        DEEPDRID_EXPECTED_IMAGES
                        and
                        conflicts
                        ==
                        0
                        and
                        len(unique_grades)
                        >=
                        3
                    )

                    record = {
                        "path": str(
                            table_path
                        ),
                        "filename": (
                            table_path.name
                        ),
                        "image_id_column": (
                            image_column
                        ),
                        "grade_column": (
                            grade_column
                        ),
                        "matched_unique_ids": (
                            matched_unique_ids
                        ),
                        "coverage_fraction": float(
                            matched_unique_ids
                            /
                            DEEPDRID_EXPECTED_IMAGES
                        ),
                        "conflicting_image_ids": int(
                            conflicts
                        ),
                        "unique_grade_values": (
                            "|".join(
                                map(
                                    str,
                                    unique_grades,
                                )
                            )
                        ),
                        "full_exact_mapping": (
                            full_exact_mapping
                        ),
                    }

                    grade_mapping_records.append(
                        record
                    )

                    if full_exact_mapping:
                        full_grade_mappings.append(
                            {
                                "record": record,
                                "mapping": (
                                    matched_mapping
                                    .sort_values(
                                        "image_id"
                                    )
                                    .reset_index(
                                        drop=True
                                    )
                                ),
                            }
                        )


deepdrid_grade_candidate_summary = pd.DataFrame(
    grade_mapping_records
)


if len(
    deepdrid_grade_candidate_summary
) > 0:
    deepdrid_grade_candidate_summary = (
        deepdrid_grade_candidate_summary
        .sort_values(
            [
                "full_exact_mapping",
                "matched_unique_ids",
                "conflicting_image_ids",
            ],
            ascending=[
                False,
                False,
                True,
            ],
        )
        .reset_index(drop=True)
    )


# ============================================================
# 7. Resolve whether full grade mappings agree
# ============================================================

deepdrid_grade_mapping_resolved = False
deepdrid_grade_mapping_ambiguous = False
deepdrid_selected_grade_source = None
deepdrid_grade_map = None


if len(full_grade_mappings) > 0:
    reference_mapping = (
        full_grade_mappings[0][
            "mapping"
        ]
        .set_index(
            "image_id"
        )[
            "dr_grade"
        ]
        .sort_index()
    )

    all_identical = True

    for candidate in full_grade_mappings[1:]:
        comparison_mapping = (
            candidate[
                "mapping"
            ]
            .set_index(
                "image_id"
            )[
                "dr_grade"
            ]
            .sort_index()
        )

        if not reference_mapping.equals(
            comparison_mapping
        ):
            all_identical = False
            break

    if all_identical:
        deepdrid_grade_mapping_resolved = True
        deepdrid_selected_grade_source = (
            full_grade_mappings[0][
                "record"
            ]
        )

        deepdrid_grade_map = (
            reference_mapping
            .rename("dr_grade")
            .reset_index()
        )

    else:
        deepdrid_grade_mapping_ambiguous = True


# ============================================================
# 8. Search for 1600-element grade-like arrays
#    Audit only: arrays are not automatically accepted
# ============================================================

grade_like_array_records = []


for search_root in source_search_roots:
    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_root_path = Path(
            current_root
        )

        if path_is_inside(
            current_root_path,
            TEST_ROOT,
        ):
            directory_names[:] = []
            continue

        directory_names[:] = [
            name
            for name in directory_names
            if name not in {
                ".ipynb_checkpoints",
                "__pycache__",
            }
        ]

        for filename in filenames:
            array_path = (
                current_root_path
                /
                filename
            )

            if (
                array_path.suffix.lower()
                not in ARRAY_EXTENSIONS
            ):
                continue

            if array_path.stat().st_size > (
                200
                *
                1024 ** 2
            ):
                continue

            try:
                if array_path.suffix.lower() == ".npy":
                    arrays = {
                        "single_array": np.load(
                            array_path,
                            mmap_mode="r",
                            allow_pickle=False,
                        )
                    }

                else:
                    archive = np.load(
                        array_path,
                        allow_pickle=False,
                    )

                    arrays = {
                        key: archive[key]
                        for key in archive.files
                    }

                for key, array in arrays.items():
                    if (
                        array.ndim
                        !=
                        1
                        or
                        len(array)
                        !=
                        DEEPDRID_EXPECTED_IMAGES
                    ):
                        continue

                    if not np.issubdtype(
                        array.dtype,
                        np.number,
                    ):
                        continue

                    finite_values = np.asarray(
                        array
                    )[
                        np.isfinite(
                            np.asarray(array)
                        )
                    ]

                    if len(finite_values) == 0:
                        continue

                    rounded = np.rint(
                        finite_values
                    )

                    integer_like = bool(
                        np.allclose(
                            finite_values,
                            rounded,
                            atol=1e-8,
                        )
                    )

                    unique_values = sorted(
                        set(
                            rounded.astype(int)
                            .tolist()
                        )
                    ) if integer_like else []

                    grade_like = bool(
                        integer_like
                        and
                        set(unique_values).issubset(
                            {0, 1, 2, 3, 4}
                        )
                        and
                        len(unique_values)
                        >=
                        3
                    )

                    if grade_like:
                        grade_like_array_records.append(
                            {
                                "path": str(
                                    array_path
                                ),
                                "filename": (
                                    array_path.name
                                ),
                                "array_key": key,
                                "shape": str(
                                    tuple(
                                        array.shape
                                    )
                                ),
                                "dtype": str(
                                    array.dtype
                                ),
                                "unique_values": (
                                    "|".join(
                                        map(
                                            str,
                                            unique_values,
                                        )
                                    )
                                ),
                                "automatically_accepted": False,
                                "reason_not_automatically_accepted": (
                                    "Row-order provenance must be "
                                    "demonstrated before use."
                                ),
                            }
                        )

                if array_path.suffix.lower() == ".npz":
                    archive.close()

            except Exception:
                continue


grade_like_array_summary = pd.DataFrame(
    grade_like_array_records
)


# ============================================================
# 9. Search notebooks for DeepDRiD label provenance
# ============================================================

notebook_reference_records = []


for search_root in source_search_roots:
    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_root_path = Path(
            current_root
        )

        if path_is_inside(
            current_root_path,
            TEST_ROOT,
        ):
            directory_names[:] = []
            continue

        directory_names[:] = [
            name
            for name in directory_names
            if name not in {
                ".ipynb_checkpoints",
                "__pycache__",
            }
        ]

        for filename in filenames:
            code_path = (
                current_root_path
                /
                filename
            )

            if (
                code_path.suffix.lower()
                not in NOTEBOOK_EXTENSIONS
            ):
                continue

            if code_path.stat().st_size > (
                40
                *
                1024 ** 2
            ):
                continue

            try:
                text = code_path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )

            except Exception:
                continue

            lower_text = text.lower()

            if (
                "deepdrid"
                not in lower_text
                and
                "deep_drid"
                not in lower_text
                and
                "deep-drid"
                not in lower_text
            ):
                continue

            lines = text.splitlines()

            relevant_lines = []

            for line_number, line in enumerate(
                lines,
                start=1,
            ):
                line_lower = line.lower()

                if (
                    any(
                        token in line_lower
                        for token in [
                            "deepdrid",
                            "deep_drid",
                            "deep-drid",
                        ]
                    )
                    and
                    any(
                        token in line_lower
                        for token in [
                            "grade",
                            "label",
                            "diagnosis",
                            "csv",
                            "embedding_index",
                            "image_id",
                            "np.load",
                            "read_csv",
                        ]
                    )
                ):
                    relevant_lines.append(
                        {
                            "line_number": int(
                                line_number
                            ),
                            "text": re.sub(
                                r"\s+",
                                " ",
                                line,
                            ).strip()[:1200],
                        }
                    )

                if len(relevant_lines) >= 50:
                    break

            if relevant_lines:
                notebook_reference_records.append(
                    {
                        "path": str(
                            code_path
                        ),
                        "filename": (
                            code_path.name
                        ),
                        "relevant_lines": int(
                            len(
                                relevant_lines
                            )
                        ),
                        "reference_preview": json.dumps(
                            relevant_lines
                        ),
                    }
                )


deepdrid_notebook_reference_summary = pd.DataFrame(
    notebook_reference_records
)


# ============================================================
# 10. Search all MyDrive for exact DeepDRiD image filenames
# ============================================================

deepdrid_image_match_records = []


print(
    "\nSearching MyDrive for exact DeepDRiD image IDs..."
)


for current_root, directory_names, filenames in os.walk(
    DRIVE_ROOT
):
    current_root_path = Path(
        current_root
    )

    if path_is_inside(
        current_root_path,
        TEST_ROOT,
    ):
        directory_names[:] = []
        continue

    directory_names[:] = [
        name
        for name in directory_names
        if name not in {
            ".ipynb_checkpoints",
            "__pycache__",
        }
    ]

    for filename in filenames:
        path = (
            current_root_path
            /
            filename
        )

        if (
            path.suffix.lower()
            not in IMAGE_EXTENSIONS
        ):
            continue

        image_id = normalise_image_id(
            filename
        )

        if image_id in deepdrid_id_set:
            deepdrid_image_match_records.append(
                {
                    "image_id": image_id,
                    "path": str(path),
                    "parent_directory": str(
                        path.parent
                    ),
                    "size_bytes": int(
                        path.stat().st_size
                    ),
                }
            )


deepdrid_image_matches = pd.DataFrame(
    deepdrid_image_match_records
)


if len(deepdrid_image_matches) > 0:
    deepdrid_image_match_counts = (
        deepdrid_image_matches
        .groupby("image_id")
        .size()
    )

    deepdrid_unique_match_ids = int(
        deepdrid_image_match_counts.index.nunique()
    )

    deepdrid_ambiguous_match_ids = int(
        (
            deepdrid_image_match_counts
            >
            1
        ).sum()
    )

    deepdrid_image_parent_summary = (
        deepdrid_image_matches
        .groupby(
            "parent_directory",
            as_index=False,
        )
        .agg(
            matched_files=(
                "path",
                "size",
            ),
            matched_unique_ids=(
                "image_id",
                "nunique",
            ),
        )
        .sort_values(
            "matched_unique_ids",
            ascending=False,
        )
        .reset_index(drop=True)
    )

else:
    deepdrid_unique_match_ids = 0
    deepdrid_ambiguous_match_ids = 0

    deepdrid_image_parent_summary = pd.DataFrame(
        columns=[
            "parent_directory",
            "matched_files",
            "matched_unique_ids",
        ]
    )


deepdrid_images_resolved = bool(
    deepdrid_unique_match_ids
    ==
    DEEPDRID_EXPECTED_IMAGES
    and
    deepdrid_ambiguous_match_ids
    ==
    0
)


# ============================================================
# 11. Construct exact DeepDRiD manifest when grades resolve
# ============================================================

deepdrid_exact_manifest = None


if deepdrid_grade_mapping_resolved:
    deepdrid_exact_manifest = (
        deepdrid_identity_manifest
        .merge(
            deepdrid_grade_map,
            on="image_id",
            how="left",
            validate="one_to_one",
        )
    )

    assert deepdrid_exact_manifest[
        "dr_grade"
    ].notna().all()

    deepdrid_exact_manifest[
        "dr_grade"
    ] = deepdrid_exact_manifest[
        "dr_grade"
    ].astype(int)

    if deepdrid_images_resolved:
        unique_path_map = (
            deepdrid_image_matches
            .drop_duplicates(
                "image_id",
                keep="first",
            )
            .set_index(
                "image_id"
            )[
                "path"
            ]
            .to_dict()
        )

        deepdrid_exact_manifest[
            "source_image_path"
        ] = (
            deepdrid_exact_manifest[
                "image_id"
            ].map(
                unique_path_map
            )
        )


# ============================================================
# 12. Final readiness decision
# ============================================================

eyepacs_manifest_resolved = bool(
    len(eyepacs_exact_manifest)
    ==
    EYEPACS_EXPECTED_IMAGES
    and
    eyepacs_exact_manifest[
        "dr_grade"
    ].notna().all()
)


deepdrid_identity_resolved = bool(
    len(deepdrid_identity_manifest)
    ==
    DEEPDRID_EXPECTED_IMAGES
    and
    deepdrid_identity_manifest[
        "image_id"
    ].is_unique
)


resolution_summary = pd.DataFrame(
    [
        {
            "dataset": "EyePACS_2015",
            "expected_images": (
                EYEPACS_EXPECTED_IMAGES
            ),
            "exact_identity_manifest_resolved": (
                eyepacs_manifest_resolved
            ),
            "exact_grade_mapping_resolved": (
                eyepacs_manifest_resolved
            ),
            "exact_images_resolved": (
                eyepacs_images_resolved
            ),
            "resolved_image_count": (
                EYEPACS_EXPECTED_IMAGES
                if eyepacs_images_resolved
                else 0
            ),
            "selected_identity_manifest": str(
                eyepacs_cohort_path
            ),
            "selected_grade_source": str(
                eyepacs_metadata_path
            ),
            "selected_image_folder": (
                str(
                    eyepacs_selected_image_folder
                )
                if
                eyepacs_selected_image_folder
                is not None
                else
                ""
            ),
        },
        {
            "dataset": "DeepDRiD",
            "expected_images": (
                DEEPDRID_EXPECTED_IMAGES
            ),
            "exact_identity_manifest_resolved": (
                deepdrid_identity_resolved
            ),
            "exact_grade_mapping_resolved": (
                deepdrid_grade_mapping_resolved
            ),
            "exact_images_resolved": (
                deepdrid_images_resolved
            ),
            "resolved_image_count": (
                deepdrid_unique_match_ids
            ),
            "selected_identity_manifest": str(
                deepdrid_index_path
            ),
            "selected_grade_source": (
                deepdrid_selected_grade_source[
                    "path"
                ]
                if
                deepdrid_selected_grade_source
                is not None
                else
                ""
            ),
            "selected_image_folder": "",
        },
    ]
)


if (
    eyepacs_manifest_resolved
    and
    eyepacs_images_resolved
    and
    deepdrid_identity_resolved
    and
    deepdrid_grade_mapping_resolved
    and
    deepdrid_images_resolved
):
    stage4c_decision = (
        "PASS_BOTH_SOURCE_COHORTS_GRADES_AND_IMAGES_"
        "RESOLVED_ADVANCE_TO_CANONICALISATION"
    )

    authorised_next_step = (
        "CANONICALISE_EYEPACS_AND_DEEPDRID"
    )

    interpretation = (
        "Exact identities, grades, and source images were resolved "
        "for both EyePACS and DeepDRiD."
    )

elif (
    eyepacs_manifest_resolved
    and
    eyepacs_images_resolved
    and
    deepdrid_identity_resolved
    and
    deepdrid_grade_mapping_resolved
    and
    not deepdrid_images_resolved
):
    stage4c_decision = (
        "PASS_EXACT_SOURCE_MANIFESTS_REACQUIRE_"
        "DEEPDRID_IMAGES_ONLY"
    )

    authorised_next_step = (
        "REACQUIRE_ONLY_THE_1600_PRECOMMITTED_"
        "DEEPDRID_SOURCE_IMAGES"
    )

    interpretation = (
        "EyePACS is fully resolved. The exact DeepDRiD identities "
        "and grades are also resolved, but its original image files "
        "are not present in current Drive storage. Only the fixed "
        "1,600-image DeepDRiD image material must be reacquired."
    )

elif (
    eyepacs_manifest_resolved
    and
    eyepacs_images_resolved
    and
    deepdrid_identity_resolved
    and
    not deepdrid_grade_mapping_resolved
):
    stage4c_decision = (
        "PASS_EYEPACS_FULLY_RESOLVED_HOLD_"
        "DEEPDRID_GRADE_PROVENANCE"
    )

    authorised_next_step = (
        "RESOLVE_DEEPDRID_GRADE_MAPPING_FROM_"
        "RETAINED_CODE_OR_ORIGINAL_LABEL_FILES"
    )

    interpretation = (
        "EyePACS is fully resolved and DeepDRiD's exact 1,600-image "
        "identity/order is recovered. A reliable grade mapping for "
        "those 1,600 IDs has not yet been established, so reacquiring "
        "images is premature."
    )

elif deepdrid_grade_mapping_ambiguous:
    stage4c_decision = (
        "HOLD_DEEPDRID_MULTIPLE_DISAGREEING_"
        "FULL_GRADE_MAPPINGS"
    )

    authorised_next_step = (
        "AUDIT_DEEPDRID_GRADE_SOURCE_PROVENANCE"
    )

    interpretation = (
        "Multiple retained artifacts produce complete but "
        "non-identical DeepDRiD grade mappings. No mapping is "
        "authorised until provenance is resolved."
    )

else:
    stage4c_decision = (
        "HOLD_SOURCE_PROVENANCE_RECOVERY_INCOMPLETE"
    )

    authorised_next_step = (
        "REVIEW_SAVED_STAGE4C_DIAGNOSTIC_ARTIFACTS"
    )

    interpretation = (
        "At least one exact source identity, grade mapping, or "
        "image set remains unresolved."
    )


# ============================================================
# 13. Save Stage 4C artifacts
# ============================================================

EYEPACS_EXACT_MANIFEST_PATH = (
    STAGE4C_ROOT
    / "Stage4C_EyePACS_Exact_Source_Manifest_v0.1.csv"
)

EYEPACS_FOLDER_AUDIT_PATH = (
    STAGE4C_ROOT
    / "Stage4C_EyePACS_Image_Folder_Audit_v0.1.csv"
)

DEEPDRID_IDENTITY_MANIFEST_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_Exact_Identity_Manifest_v0.1.csv"
)

DEEPDRID_EXACT_MANIFEST_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_Exact_Source_Manifest_v0.1.csv"
)

DEEPDRID_GRADE_CANDIDATE_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_Grade_Mapping_Candidates_v0.1.csv"
)

DEEPDRID_ARRAY_CANDIDATE_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_GradeLike_Array_Candidates_v0.1.csv"
)

DEEPDRID_NOTEBOOK_REFERENCE_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_Label_Provenance_Notebook_References_v0.1.csv"
)

DEEPDRID_IMAGE_MATCH_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_MyDrive_Image_Matches_v0.1.csv"
)

DEEPDRID_IMAGE_PARENT_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_Image_Parent_Summary_v0.1.csv"
)

RESOLUTION_SUMMARY_PATH = (
    STAGE4C_ROOT
    / "Stage4C_Source_Resolution_Summary_v0.1.csv"
)

DECISION_PATH = (
    STAGE4C_ROOT
    / "Stage4C_Source_Provenance_Recovery_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE4C_ROOT
    / "Stage4C_Source_Provenance_Recovery_Report_v0.1.md"
)


eyepacs_exact_manifest.to_csv(
    EYEPACS_EXACT_MANIFEST_PATH,
    index=False,
)

eyepacs_folder_audit.to_csv(
    EYEPACS_FOLDER_AUDIT_PATH,
    index=False,
)

deepdrid_identity_manifest.to_csv(
    DEEPDRID_IDENTITY_MANIFEST_PATH,
    index=False,
)

if deepdrid_exact_manifest is not None:
    deepdrid_exact_manifest.to_csv(
        DEEPDRID_EXACT_MANIFEST_PATH,
        index=False,
    )

deepdrid_grade_candidate_summary.to_csv(
    DEEPDRID_GRADE_CANDIDATE_PATH,
    index=False,
)

grade_like_array_summary.to_csv(
    DEEPDRID_ARRAY_CANDIDATE_PATH,
    index=False,
)

deepdrid_notebook_reference_summary.to_csv(
    DEEPDRID_NOTEBOOK_REFERENCE_PATH,
    index=False,
)

deepdrid_image_matches.to_csv(
    DEEPDRID_IMAGE_MATCH_PATH,
    index=False,
)

deepdrid_image_parent_summary.to_csv(
    DEEPDRID_IMAGE_PARENT_PATH,
    index=False,
)

resolution_summary.to_csv(
    RESOLUTION_SUMMARY_PATH,
    index=False,
)


decision_payload = {
    "decision": stage4c_decision,
    "interpretation": interpretation,
    "authorised_next_step": (
        authorised_next_step
    ),
    "eyepacs": {
        "identity_manifest_resolved": (
            eyepacs_manifest_resolved
        ),
        "grade_mapping_resolved": (
            eyepacs_manifest_resolved
        ),
        "images_resolved": (
            eyepacs_images_resolved
        ),
        "exact_manifest_path": str(
            EYEPACS_EXACT_MANIFEST_PATH
        ),
        "selected_identity_source": str(
            eyepacs_cohort_path
        ),
        "selected_grade_source": str(
            eyepacs_metadata_path
        ),
        "selected_image_folder": (
            str(
                eyepacs_selected_image_folder
            )
            if
            eyepacs_selected_image_folder
            is not None
            else
            None
        ),
    },
    "deepdrid": {
        "identity_manifest_resolved": (
            deepdrid_identity_resolved
        ),
        "grade_mapping_resolved": (
            deepdrid_grade_mapping_resolved
        ),
        "grade_mapping_ambiguous": (
            deepdrid_grade_mapping_ambiguous
        ),
        "images_resolved": (
            deepdrid_images_resolved
        ),
        "resolved_image_ids": int(
            deepdrid_unique_match_ids
        ),
        "ambiguous_image_matches": int(
            deepdrid_ambiguous_match_ids
        ),
        "identity_manifest_path": str(
            DEEPDRID_IDENTITY_MANIFEST_PATH
        ),
        "exact_manifest_path": (
            str(
                DEEPDRID_EXACT_MANIFEST_PATH
            )
            if
            deepdrid_exact_manifest
            is not None
            else
            None
        ),
        "selected_identity_source": str(
            deepdrid_index_path
        ),
        "selected_grade_source": (
            deepdrid_selected_grade_source[
                "path"
            ]
            if
            deepdrid_selected_grade_source
            is not None
            else
            None
        ),
        "complete_grade_mapping_candidates": int(
            len(full_grade_mappings)
        ),
        "grade_like_arrays_not_automatically_used": int(
            len(
                grade_like_array_summary
            )
        ),
    },
    "target_sealed_label_files_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_files_modified": False,
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


REPORT_PATH.write_text(
    "\n".join(
        [
            "# Stage 4C — Source Provenance Recovery",
            "",
            f"Decision: `{stage4c_decision}`",
            "",
            interpretation,
            "",
            "## Safety boundary",
            "",
            "- Target sealed labels accessed: `False`",
            "- Source performance observed: `False`",
            "- Transfer performance observed: `False`",
            "- Existing source files modified: `False`",
        ]
    ),
    encoding="utf-8",
)


# ============================================================
# 14. Display concise results
# ============================================================

print(
    "\n================ SOURCE RESOLUTION "
    "SUMMARY ================"
)

display(
    resolution_summary
)


print(
    "\n================ EYEPACS FOLDER AUDIT "
    "================"
)

display(
    eyepacs_folder_audit
)


print(
    "\n================ DEEPDRID GRADE "
    "MAPPING CANDIDATES ================"
)

if len(
    deepdrid_grade_candidate_summary
) > 0:
    display(
        deepdrid_grade_candidate_summary[
            [
                "filename",
                "image_id_column",
                "grade_column",
                "matched_unique_ids",
                "coverage_fraction",
                "conflicting_image_ids",
                "unique_grade_values",
                "full_exact_mapping",
                "path",
            ]
        ].head(20)
    )

else:
    print(
        "No retained source table supplied a "
        "DeepDRiD grade mapping."
    )


print(
    "\n================ DEEPDRID GRADE-LIKE "
    "ARRAY CANDIDATES ================"
)

if len(
    grade_like_array_summary
) > 0:
    display(
        grade_like_array_summary.head(20)
    )

else:
    print(
        "No 1600-element grade-like arrays were found."
    )


print(
    "\n================ DEEPDRID IMAGE "
    "RESOLUTION ================"
)

print(
    "Exact unique image IDs found in MyDrive:",
    deepdrid_unique_match_ids,
)

print(
    "Image IDs with multiple exact matches:",
    deepdrid_ambiguous_match_ids,
)

if len(
    deepdrid_image_parent_summary
) > 0:
    display(
        deepdrid_image_parent_summary.head(20)
    )


print(
    "\n================ STAGE 4C DECISION "
    "================"
)

print("Decision:")
print(stage4c_decision)

print("\nInterpretation:")
print(interpretation)

print("\nAuthorised next step:")
print(authorised_next_step)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nSource performance observed:")
print(False)

print("\nSource-target transfer performance observed:")
print(False)

print(
    "\nStage 4C source provenance recovery "
    "completed and sealed."
)

================ STAGE 4C SOURCE PROVENANCE RECOVERY ================

Imported Stage 4B decision:
HOLD_SOURCE_COHORT_IDENTITIES_UNRESOLVED

Safety boundary:
Stage 4C reconstructs source cohort identities and source-label provenance using retained source artifacts. It does not access target sealed-label files, calculate source or transfer performance, alter source artifacts, or canonicalise images.

Searching MyDrive for exact DeepDRiD image IDs...

================ SOURCE RESOLUTION SUMMARY ================


,dataset,expected_images,exact_identity_manifest_resolved,exact_grade_mapping_resolved,exact_images_resolved,resolved_image_count,selected_identity_manifest,selected_grade_source,selected_image_folder
0,EyePACS_2015,4468,True,True,True,4468,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,DeepDRiD,1600,True,False,True,1600,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,,



================ EYEPACS FOLDER AUDIT ================


,folder,direct_image_files,matched_cohort_ids,missing_cohort_ids,extra_image_ids,ambiguous_matched_ids,exact_match
0,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4468,4468,0,0,0,True



================ DEEPDRID GRADE MAPPING CANDIDATES ================


,filename,image_id_column,grade_column,matched_unique_ids,coverage_fraction,conflicting_image_ids,unique_grade_values,full_exact_mapping,path
0,EyePACS_2015_Canonical_Metadata_v0.1.csv,image_id,dr_grade,0,0.0,0,,False,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,EyePACS_Clean_Validation_PairedDifference_Pred...,left_image_id,left_grade,0,0.0,0,,False,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,EyePACS_Clean_Validation_PairedDifference_Pred...,right_image_id,right_grade,0,0.0,0,,False,/content/drive/MyDrive/Cross-Modal_Diagnostic_...



================ DEEPDRID GRADE-LIKE ARRAY CANDIDATES ================
No 1600-element grade-like arrays were found.

================ DEEPDRID IMAGE RESOLUTION ================
Exact unique image IDs found in MyDrive: 1600
Image IDs with multiple exact matches: 0


,parent_directory,matched_files,matched_unique_ids
0,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
2,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
3,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
4,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
5,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
6,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
7,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
8,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4
9,/content/drive/MyDrive/Cross-Modal_Diagnostic_...,4,4



================ STAGE 4C DECISION ================
Decision:
PASS_EYEPACS_FULLY_RESOLVED_HOLD_DEEPDRID_GRADE_PROVENANCE

Interpretation:
EyePACS is fully resolved and DeepDRiD's exact 1,600-image identity/order is recovered. A reliable grade mapping for those 1,600 IDs has not yet been established, so reacquiring images is premature.

Authorised next step:
RESOLVE_DEEPDRID_GRADE_MAPPING_FROM_RETAINED_CODE_OR_ORIGINAL_LABEL_FILES

Target sealed-label files accessed:
False

Source performance observed:
False

Source-target transfer performance observed:
False

Stage 4C source provenance recovery completed and sealed.


In [27]:
#@title 04D. Resolve DeepDRiD label provenance from retained source artifacts

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen Stage 4D boundary
# ============================================================

DEEPDRID_EXPECTED_IMAGES = 1600

TABLE_EXTENSIONS = {
    ".csv",
    ".parquet",
    ".xlsx",
    ".xls",
}

CODE_EXTENSIONS = {
    ".ipynb",
    ".py",
}

SAFETY_BOUNDARY = (
    "Stage 4D performs read-only recovery of the DeepDRiD source-label "
    "mapping. It may inspect retained source tables and code, but it "
    "must not access target sealed-label files, modify source images, "
    "canonicalise source images, calculate source performance, or "
    "calculate source-target transfer performance."
)

AUTOMATIC_ACCEPTANCE_RULE = (
    "A candidate is automatically acceptable only when it maps all "
    "1,600 precommitted DeepDRiD image IDs without conflicts and either "
    "provides a trusted ordinal DR grade in {0,1,2,3,4}, or provides an "
    "explicitly named moderate-or-worse / referable-DR binary endpoint. "
    "Generic binary columns without explicit endpoint semantics are "
    "reported but not automatically accepted."
)


# ============================================================
# 1. Resolve project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

RETINAL_CODE_ROOT = (
    PROJECT_ROOT
    / "05_Code"
    / "Retinal_DR"
)

TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE4_ROOT = (
    TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4C_ROOT = (
    STAGE4_ROOT
    / "02_Source_Manifest_And_Provenance_Recovery"
)

STAGE4D_ROOT = (
    STAGE4_ROOT
    / "03_DeepDRiD_Label_Provenance_Resolution"
)

STAGE4D_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE4C_DECISION_PATH = (
    STAGE4C_ROOT
    / "Stage4C_Source_Provenance_Recovery_Decision_v0.1.json"
)

DEEPDRID_IDENTITY_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_Exact_Identity_Manifest_v0.1.csv"
)

DEEPDRID_IMAGE_MATCH_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_MyDrive_Image_Matches_v0.1.csv"
)

EYEPACS_EXACT_MANIFEST_PATH = (
    STAGE4C_ROOT
    / "Stage4C_EyePACS_Exact_Source_Manifest_v0.1.csv"
)


for required_path in [
    STAGE4C_DECISION_PATH,
    DEEPDRID_IDENTITY_PATH,
    DEEPDRID_IMAGE_MATCH_PATH,
    EYEPACS_EXACT_MANIFEST_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with open(
    STAGE4C_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage4c_decision = json.load(
        file
    )


assert (
    stage4c_decision["decision"]
    ==
    "PASS_EYEPACS_FULLY_RESOLVED_HOLD_DEEPDRID_GRADE_PROVENANCE"
)


print(
    "================ STAGE 4D DEEPDRID "
    "LABEL PROVENANCE ================"
)

print("\nImported Stage 4C decision:")
print(
    stage4c_decision["decision"]
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)

print("\nAutomatic acceptance rule:")
print(
    AUTOMATIC_ACCEPTANCE_RULE
)


# ============================================================
# 2. Helpers
# ============================================================

def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def normalise_image_id(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip()

    if not text:
        return ""

    return Path(
        text
    ).stem


def path_is_inside(
    path,
    parent,
):
    try:
        Path(
            path
        ).resolve().relative_to(
            Path(
                parent
            ).resolve()
        )

        return True

    except ValueError:
        return False


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def mapping_sha256(
    mapping,
    label_column,
):
    ordered = (
        mapping[
            [
                "image_id",
                label_column,
            ]
        ]
        .sort_values(
            "image_id"
        )
        .reset_index(
            drop=True
        )
    )

    payload = "\n".join(
        (
            ordered[
                "image_id"
            ].astype(str)
            +
            "|"
            +
            ordered[
                label_column
            ].astype(str)
        ).tolist()
    )

    return hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()


def read_table(
    path,
):
    path = Path(
        path
    )

    suffix = path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(
            path,
            low_memory=False,
        )

    if suffix == ".parquet":
        return pd.read_parquet(
            path
        )

    if suffix in {
        ".xlsx",
        ".xls",
    }:
        return pd.read_excel(
            path
        )

    raise ValueError(
        f"Unsupported table: {path}"
    )


def detect_embedding_row_column(
    dataframe,
):
    exact_candidates = {
        "embeddingrow",
        "rowindex",
        "embeddingindex",
    }

    for column in dataframe.columns:
        key = normalise_column_name(
            column
        )

        if key in exact_candidates:
            return column

    return None


def detect_image_id_columns(
    dataframe,
):
    output = []

    disallowed_exact = {
        "patientid",
        "subjectid",
        "embeddingrow",
        "rowindex",
        "index",
    }

    for column in dataframe.columns:
        key = normalise_column_name(
            column
        )

        if key in disallowed_exact:
            continue

        exact_match = key in {
            "imageid",
            "image",
            "imagename",
            "filename",
            "filepath",
            "idcode",
            "eyeid",
            "leftimageid",
            "rightimageid",
            "leftimage",
            "rightimage",
        }

        semantic_match = bool(
            any(
                token in key
                for token in [
                    "imageid",
                    "imagename",
                    "imagefile",
                    "filepath",
                    "filename",
                    "fundusimage",
                    "leftimage",
                    "rightimage",
                ]
            )
        )

        if (
            exact_match
            or
            semantic_match
        ):
            output.append(
                column
            )

    return list(
        dict.fromkeys(
            output
        )
    )


def detect_label_columns(
    dataframe,
):
    output = []

    disallowed_tokens = {
        "prediction",
        "probability",
        "confidence",
        "score",
        "auc",
        "accuracy",
        "correct",
        "gap",
        "difference",
        "delta",
        "embedding",
        "fold",
        "split",
    }

    positive_tokens = {
        "grade",
        "label",
        "target",
        "diagnosis",
        "severity",
        "retinopathy",
        "diseaseclass",
        "diseaselevel",
        "drlevel",
        "drclass",
        "groundtruth",
        "groundtruthlabel",
        "ytrue",
        "referable",
        "moderateorworse",
        "moderatedr",
    }

    for column in dataframe.columns:
        key = normalise_column_name(
            column
        )

        if any(
            token in key
            for token in disallowed_tokens
        ):
            continue

        if any(
            token in key
            for token in positive_tokens
        ):
            output.append(
                column
            )

    return list(
        dict.fromkeys(
            output
        )
    )


def column_side(
    column_name,
):
    key = normalise_column_name(
        column_name
    )

    if "left" in key:
        return "left"

    if "right" in key:
        return "right"

    return None


def columns_are_compatible(
    id_column,
    label_column,
):
    id_side = column_side(
        id_column
    )

    label_side = column_side(
        label_column
    )

    if (
        id_side is None
        and
        label_side is None
    ):
        return True

    if (
        id_side is not None
        and
        label_side == id_side
    ):
        return True

    return False


def label_semantics(
    source_names,
    values,
):
    source_text = " ".join(
        map(
            str,
            source_names,
        )
    ).lower()

    source_key = normalise_column_name(
        source_text
    )

    numeric = pd.to_numeric(
        values,
        errors="coerce",
    )

    numeric = numeric[
        numeric.notna()
    ]

    if len(
        numeric
    ) == 0:
        return {
            "domain": (
                "non_numeric"
            ),
            "trusted": False,
            "integer_like": False,
            "unique_values": [],
        }

    integer_like = bool(
        np.allclose(
            numeric.to_numpy(
                dtype=float
            ),
            np.rint(
                numeric.to_numpy(
                    dtype=float
                )
            ),
            atol=1e-8,
        )
    )

    if not integer_like:
        return {
            "domain": (
                "non_integer_numeric"
            ),
            "trusted": False,
            "integer_like": False,
            "unique_values": [],
        }

    unique_values = sorted(
        set(
            np.rint(
                numeric.to_numpy(
                    dtype=float
                )
            )
            .astype(int)
            .tolist()
        )
    )

    value_set = set(
        unique_values
    )

    disqualifying_name = any(
        token in source_key
        for token in [
            "gap",
            "difference",
            "delta",
            "prediction",
            "probability",
            "score",
        ]
    )

    explicit_endpoint_name = any(
        token in source_key
        for token in [
            "moderateorworse",
            "gradege2",
            "drge2",
            "referable",
            "binarydr",
            "moderatedr",
        ]
    )

    ordinal_name = any(
        token in source_key
        for token in [
            "grade",
            "diagnosis",
            "retinopathy",
            "severity",
            "drlevel",
            "drclass",
            "label",
            "target",
        ]
    )

    if (
        value_set.issubset(
            {
                0,
                1,
                2,
                3,
                4,
            }
        )
        and
        len(
            value_set
        )
        >=
        3
    ):
        return {
            "domain": (
                "ordinal_dr_grade"
            ),
            "trusted": bool(
                ordinal_name
                and
                not disqualifying_name
            ),
            "integer_like": True,
            "unique_values": (
                unique_values
            ),
        }

    if value_set.issubset(
        {
            0,
            1,
        }
    ):
        return {
            "domain": (
                "explicit_binary_endpoint"
                if explicit_endpoint_name
                else
                "ambiguous_binary"
            ),
            "trusted": bool(
                explicit_endpoint_name
                and
                not disqualifying_name
            ),
            "integer_like": True,
            "unique_values": (
                unique_values
            ),
        }

    return {
        "domain": (
            "unsupported_integer_domain"
        ),
        "trusted": False,
        "integer_like": True,
        "unique_values": (
            unique_values
        ),
    }


def evaluate_mapping(
    raw_mapping,
    source_path,
    strategy,
    id_columns,
    label_columns,
):
    mapping = raw_mapping[
        [
            "image_id",
            "raw_label",
        ]
    ].copy()

    mapping[
        "image_id"
    ] = mapping[
        "image_id"
    ].map(
        normalise_image_id
    )

    mapping[
        "numeric_label"
    ] = pd.to_numeric(
        mapping[
            "raw_label"
        ],
        errors="coerce",
    )

    mapping = mapping[
        (
            mapping[
                "image_id"
            ]
            !=
            ""
        )
        &
        mapping[
            "numeric_label"
        ].notna()
    ].copy()

    mapping = mapping[
        mapping[
            "image_id"
        ].isin(
            deepdrid_id_set
        )
    ].copy()

    semantic = label_semantics(
        source_names=(
            [
                str(
                    source_path
                ),
                strategy,
                *id_columns,
                *label_columns,
            ]
        ),
        values=(
            mapping[
                "numeric_label"
            ]
        ),
    )

    if (
        semantic[
            "integer_like"
        ]
    ):
        mapping[
            "numeric_label"
        ] = np.rint(
            mapping[
                "numeric_label"
            ].to_numpy(
                dtype=float
            )
        ).astype(int)

    conflict_counts = (
        mapping
        .groupby(
            "image_id"
        )[
            "numeric_label"
        ]
        .nunique()
    )

    conflicting_image_ids = int(
        (
            conflict_counts
            >
            1
        ).sum()
    )

    stable_mapping = (
        mapping
        .sort_values(
            "image_id"
        )
        .drop_duplicates(
            "image_id",
            keep="first",
        )
        .reset_index(
            drop=True
        )
    )

    matched_unique_ids = int(
        stable_mapping[
            "image_id"
        ].nunique()
    )

    full_coverage = bool(
        matched_unique_ids
        ==
        DEEPDRID_EXPECTED_IMAGES
    )

    automatically_acceptable = bool(
        full_coverage
        and
        conflicting_image_ids
        ==
        0
        and
        semantic[
            "trusted"
        ]
        and
        semantic[
            "domain"
        ]
        in {
            "ordinal_dr_grade",
            "explicit_binary_endpoint",
        }
    )

    if semantic[
        "domain"
    ] == "ordinal_dr_grade":
        stable_mapping[
            "dr_grade"
        ] = stable_mapping[
            "numeric_label"
        ].astype(int)

        stable_mapping[
            "moderate_or_worse_dr"
        ] = (
            stable_mapping[
                "dr_grade"
            ]
            >=
            2
        ).astype(int)

    elif semantic[
        "domain"
    ] == "explicit_binary_endpoint":
        stable_mapping[
            "dr_grade"
        ] = pd.NA

        stable_mapping[
            "moderate_or_worse_dr"
        ] = stable_mapping[
            "numeric_label"
        ].astype(int)

    else:
        stable_mapping[
            "dr_grade"
        ] = pd.NA

        stable_mapping[
            "moderate_or_worse_dr"
        ] = pd.NA

    score = 0

    if full_coverage:
        score += 100

    if conflicting_image_ids == 0:
        score += 30

    if semantic[
        "domain"
    ] == "ordinal_dr_grade":
        score += 30

    elif semantic[
        "domain"
    ] == "explicit_binary_endpoint":
        score += 20

    if semantic[
        "trusted"
    ]:
        score += 30

    source_text = str(
        source_path
    ).lower()

    if "deepdrid" in source_text:
        score += 10

    if any(
        token in source_text
        for token in [
            "label",
            "grade",
            "ground",
            "metadata",
            "manifest",
        ]
    ):
        score += 10

    record = {
        "source_path": str(
            source_path
        ),
        "filename": Path(
            source_path
        ).name,
        "strategy": (
            strategy
        ),
        "id_columns": (
            " | ".join(
                map(
                    str,
                    id_columns,
                )
            )
        ),
        "label_columns": (
            " | ".join(
                map(
                    str,
                    label_columns,
                )
            )
        ),
        "matched_unique_ids": (
            matched_unique_ids
        ),
        "coverage_fraction": float(
            matched_unique_ids
            /
            DEEPDRID_EXPECTED_IMAGES
        ),
        "conflicting_image_ids": (
            conflicting_image_ids
        ),
        "label_domain": (
            semantic[
                "domain"
            ]
        ),
        "unique_values": (
            "|".join(
                map(
                    str,
                    semantic[
                        "unique_values"
                    ],
                )
            )
        ),
        "semantic_name_trusted": bool(
            semantic[
                "trusted"
            ]
        ),
        "full_coverage": (
            full_coverage
        ),
        "automatically_acceptable": (
            automatically_acceptable
        ),
        "candidate_score": int(
            score
        ),
    }

    return (
        record,
        stable_mapping,
    )


# ============================================================
# 3. Load frozen identities and exact source images
# ============================================================

deepdrid_identity = pd.read_csv(
    DEEPDRID_IDENTITY_PATH
)

deepdrid_image_matches = pd.read_csv(
    DEEPDRID_IMAGE_MATCH_PATH
)


assert (
    len(
        deepdrid_identity
    )
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert deepdrid_identity[
    "image_id"
].is_unique


assert (
    deepdrid_image_matches[
        "image_id"
    ].nunique()
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert not (
    deepdrid_image_matches
    .groupby(
        "image_id"
    )
    .size()
    >
    1
).any()


deepdrid_image_map = (
    deepdrid_image_matches
    .set_index(
        "image_id"
    )[
        "path"
    ]
    .to_dict()
)


deepdrid_identity[
    "source_image_path"
] = (
    deepdrid_identity[
        "image_id"
    ].map(
        deepdrid_image_map
    )
)


assert deepdrid_identity[
    "source_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


deepdrid_id_set = set(
    deepdrid_identity[
        "image_id"
    ].astype(str)
)


common_image_root = Path(
    os.path.commonpath(
        deepdrid_identity[
            "source_image_path"
        ].astype(str).tolist()
    )
)


if common_image_root.is_file():
    common_image_root = (
        common_image_root.parent
    )


print("\nRecovered DeepDRiD image identities:")
print(
    len(
        deepdrid_identity
    )
)

print("\nRecovered DeepDRiD image files:")
print(
    deepdrid_identity[
        "source_image_path"
    ].nunique()
)

print("\nDeepest common image root:")
print(
    common_image_root
)


# ============================================================
# 4. Define targeted source search roots
# ============================================================

candidate_search_roots = [
    common_image_root,
    common_image_root.parent,
    RETINAL_DATA_ROOT,
    RETINAL_CODE_ROOT,
]


search_roots = []

seen_search_roots = set()


for root in candidate_search_roots:
    root = Path(
        root
    )

    if not root.is_dir():
        continue

    resolved = str(
        root.resolve()
    )

    if resolved in seen_search_roots:
        continue

    if path_is_inside(
        root,
        TEST_ROOT,
    ):
        continue

    seen_search_roots.add(
        resolved
    )

    search_roots.append(
        root
    )


print("\nTargeted search roots:")

for root in search_roots:
    print(
        root
    )


# ============================================================
# 5. Scan retained tables for label mappings
# ============================================================

candidate_records = []
candidate_mapping_objects = []

scanned_table_paths = set()


for search_root in search_roots:
    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_root_path = Path(
            current_root
        )

        if path_is_inside(
            current_root_path,
            TEST_ROOT,
        ):
            directory_names[:] = []
            continue

        directory_names[:] = [
            name
            for name in directory_names
            if name not in {
                ".ipynb_checkpoints",
                "__pycache__",
            }
        ]

        for filename in filenames:
            table_path = (
                current_root_path
                /
                filename
            )

            if (
                table_path.suffix.lower()
                not in TABLE_EXTENSIONS
            ):
                continue

            resolved_table_path = str(
                table_path.resolve()
            )

            if resolved_table_path in scanned_table_paths:
                continue

            scanned_table_paths.add(
                resolved_table_path
            )

            if table_path.stat().st_size > (
                250
                *
                1024 ** 2
            ):
                continue

            try:
                frame = read_table(
                    table_path
                )

            except Exception:
                continue

            if len(
                frame
            ) == 0:
                continue

            id_columns = detect_image_id_columns(
                frame
            )

            label_columns = detect_label_columns(
                frame
            )

            embedding_row_column = (
                detect_embedding_row_column(
                    frame
                )
            )

            if not label_columns:
                continue

            # ------------------------------------------------
            # A. Individual ID-column / label-column mappings
            # ------------------------------------------------

            for id_column in id_columns:
                for label_column in label_columns:
                    if not columns_are_compatible(
                        id_column,
                        label_column,
                    ):
                        continue

                    raw_mapping = pd.DataFrame(
                        {
                            "image_id": (
                                frame[
                                    id_column
                                ]
                            ),
                            "raw_label": (
                                frame[
                                    label_column
                                ]
                            ),
                        }
                    )

                    record, stable_mapping = (
                        evaluate_mapping(
                            raw_mapping=(
                                raw_mapping
                            ),
                            source_path=(
                                table_path
                            ),
                            strategy=(
                                "single_column_pair"
                            ),
                            id_columns=[
                                id_column
                            ],
                            label_columns=[
                                label_column
                            ],
                        )
                    )

                    if record[
                        "matched_unique_ids"
                    ] > 0:
                        candidate_records.append(
                            record
                        )

                        candidate_mapping_objects.append(
                            {
                                "record": (
                                    record
                                ),
                                "mapping": (
                                    stable_mapping
                                ),
                            }
                        )

            # ------------------------------------------------
            # B. Combine compatible image/label columns
            #    Supports two images per eye and four images
            #    per patient structures.
            # ------------------------------------------------

            union_parts = []
            union_id_columns = []
            union_label_columns = []

            for label_column in label_columns:
                compatible_id_columns = [
                    id_column
                    for id_column in id_columns
                    if columns_are_compatible(
                        id_column,
                        label_column,
                    )
                ]

                for id_column in compatible_id_columns:
                    union_parts.append(
                        pd.DataFrame(
                            {
                                "image_id": (
                                    frame[
                                        id_column
                                    ]
                                ),
                                "raw_label": (
                                    frame[
                                        label_column
                                    ]
                                ),
                            }
                        )
                    )

                    union_id_columns.append(
                        id_column
                    )

                    union_label_columns.append(
                        label_column
                    )

            if union_parts:
                union_mapping = pd.concat(
                    union_parts,
                    ignore_index=True,
                )

                record, stable_mapping = (
                    evaluate_mapping(
                        raw_mapping=(
                            union_mapping
                        ),
                        source_path=(
                            table_path
                        ),
                        strategy=(
                            "side_aware_table_union"
                        ),
                        id_columns=list(
                            dict.fromkeys(
                                union_id_columns
                            )
                        ),
                        label_columns=list(
                            dict.fromkeys(
                                union_label_columns
                            )
                        ),
                    )
                )

                if record[
                    "matched_unique_ids"
                ] > 0:
                    candidate_records.append(
                        record
                    )

                    candidate_mapping_objects.append(
                        {
                            "record": (
                                record
                            ),
                            "mapping": (
                                stable_mapping
                            ),
                        }
                    )

            # ------------------------------------------------
            # C. Explicit 1,600-row embedding-order mapping
            # ------------------------------------------------

            if (
                len(
                    frame
                )
                ==
                DEEPDRID_EXPECTED_IMAGES
                and
                embedding_row_column
                is not None
                and
                "embedding_row"
                in deepdrid_identity.columns
            ):
                frame_order = pd.to_numeric(
                    frame[
                        embedding_row_column
                    ],
                    errors="coerce",
                )

                identity_order = pd.to_numeric(
                    deepdrid_identity[
                        "embedding_row"
                    ],
                    errors="coerce",
                )

                valid_frame_order = bool(
                    frame_order.notna().all()
                    and
                    frame_order.nunique()
                    ==
                    DEEPDRID_EXPECTED_IMAGES
                )

                valid_identity_order = bool(
                    identity_order.notna().all()
                    and
                    identity_order.nunique()
                    ==
                    DEEPDRID_EXPECTED_IMAGES
                )

                if (
                    valid_frame_order
                    and
                    valid_identity_order
                ):
                    order_lookup = (
                        deepdrid_identity.assign(
                            _embedding_row=(
                                identity_order.astype(
                                    int
                                )
                            )
                        )
                        .set_index(
                            "_embedding_row"
                        )[
                            "image_id"
                        ]
                        .to_dict()
                    )

                    for label_column in label_columns:
                        ordered_mapping = pd.DataFrame(
                            {
                                "image_id": (
                                    frame_order
                                    .astype(int)
                                    .map(
                                        order_lookup
                                    )
                                ),
                                "raw_label": (
                                    frame[
                                        label_column
                                    ]
                                ),
                            }
                        )

                        record, stable_mapping = (
                            evaluate_mapping(
                                raw_mapping=(
                                    ordered_mapping
                                ),
                                source_path=(
                                    table_path
                                ),
                                strategy=(
                                    "embedding_row_order_join"
                                ),
                                id_columns=[
                                    embedding_row_column
                                ],
                                label_columns=[
                                    label_column
                                ],
                            )
                        )

                        if record[
                            "matched_unique_ids"
                        ] > 0:
                            candidate_records.append(
                                record
                            )

                            candidate_mapping_objects.append(
                                {
                                    "record": (
                                        record
                                    ),
                                    "mapping": (
                                        stable_mapping
                                    ),
                                }
                            )


candidate_summary = pd.DataFrame(
    candidate_records
)


if len(
    candidate_summary
) > 0:
    candidate_summary = (
        candidate_summary
        .drop_duplicates(
            subset=[
                "source_path",
                "strategy",
                "id_columns",
                "label_columns",
            ]
        )
        .sort_values(
            [
                "automatically_acceptable",
                "candidate_score",
                "matched_unique_ids",
                "conflicting_image_ids",
            ],
            ascending=[
                False,
                False,
                False,
                True,
            ],
        )
        .reset_index(
            drop=True
        )
    )


# ============================================================
# 6. Identify trusted complete candidates and agreement
# ============================================================

trusted_complete_objects = [
    candidate
    for candidate in candidate_mapping_objects
    if candidate[
        "record"
    ][
        "automatically_acceptable"
    ]
]


trusted_candidate_records = []


for candidate_index, candidate in enumerate(
    trusted_complete_objects,
    start=1,
):
    record = dict(
        candidate[
            "record"
        ]
    )

    mapping = (
        candidate[
            "mapping"
        ]
        .copy()
    )

    endpoint_mapping = mapping[
        [
            "image_id",
            "moderate_or_worse_dr",
        ]
    ].copy()

    endpoint_mapping[
        "moderate_or_worse_dr"
    ] = endpoint_mapping[
        "moderate_or_worse_dr"
    ].astype(int)

    record[
        "trusted_candidate_id"
    ] = (
        f"TRUSTED_{candidate_index:03d}"
    )

    record[
        "endpoint_mapping_sha256"
    ] = mapping_sha256(
        endpoint_mapping,
        "moderate_or_worse_dr",
    )

    if record[
        "label_domain"
    ] == "ordinal_dr_grade":
        grade_mapping = mapping[
            [
                "image_id",
                "dr_grade",
            ]
        ].copy()

        grade_mapping[
            "dr_grade"
        ] = grade_mapping[
            "dr_grade"
        ].astype(int)

        record[
            "grade_mapping_sha256"
        ] = mapping_sha256(
            grade_mapping,
            "dr_grade",
        )

    else:
        record[
            "grade_mapping_sha256"
        ] = ""

    trusted_candidate_records.append(
        record
    )


trusted_candidate_summary = pd.DataFrame(
    trusted_candidate_records
)


trusted_mapping_resolved = False
trusted_mapping_disagreement = False
selected_candidate = None
selected_mapping = None


if len(
    trusted_complete_objects
) > 0:
    endpoint_hashes = {
        record[
            "endpoint_mapping_sha256"
        ]
        for record in trusted_candidate_records
    }

    if len(
        endpoint_hashes
    ) == 1:
        trusted_mapping_resolved = True

        # Prefer an ordinal grade source over endpoint-only binary.
        ranked_candidates = sorted(
            trusted_complete_objects,
            key=lambda candidate: (
                candidate[
                    "record"
                ][
                    "label_domain"
                ]
                ==
                "ordinal_dr_grade",
                candidate[
                    "record"
                ][
                    "candidate_score"
                ],
            ),
            reverse=True,
        )

        selected_candidate = (
            ranked_candidates[0]
        )

        selected_mapping = (
            selected_candidate[
                "mapping"
            ]
            .copy()
        )

    else:
        trusted_mapping_disagreement = True


# ============================================================
# 7. Search retained code for provenance references
# ============================================================

code_reference_records = []

scanned_code_paths = set()


for search_root in search_roots:
    for current_root, directory_names, filenames in os.walk(
        search_root
    ):
        current_root_path = Path(
            current_root
        )

        if path_is_inside(
            current_root_path,
            TEST_ROOT,
        ):
            directory_names[:] = []
            continue

        directory_names[:] = [
            name
            for name in directory_names
            if name not in {
                ".ipynb_checkpoints",
                "__pycache__",
            }
        ]

        for filename in filenames:
            code_path = (
                current_root_path
                /
                filename
            )

            if (
                code_path.suffix.lower()
                not in CODE_EXTENSIONS
            ):
                continue

            resolved_code_path = str(
                code_path.resolve()
            )

            if resolved_code_path in scanned_code_paths:
                continue

            scanned_code_paths.add(
                resolved_code_path
            )

            if code_path.stat().st_size > (
                50
                *
                1024 ** 2
            ):
                continue

            try:
                text = code_path.read_text(
                    encoding="utf-8",
                    errors="ignore",
                )

            except Exception:
                continue

            lower_text = text.lower()

            if not any(
                token in lower_text
                for token in [
                    "deepdrid",
                    "deep_drid",
                    "deep-drid",
                ]
            ):
                continue

            lines = text.splitlines()

            relevant_lines = []

            for line_number, line in enumerate(
                lines,
                start=1,
            ):
                lower_line = line.lower()

                deepdrid_context = any(
                    token in lower_line
                    for token in [
                        "deepdrid",
                        "deep_drid",
                        "deep-drid",
                    ]
                )

                label_context = any(
                    token in lower_line
                    for token in [
                        "grade",
                        "label",
                        "diagnosis",
                        "severity",
                        "target",
                        "referable",
                        "moderate",
                        "read_csv",
                        "read_excel",
                        "embedding_index",
                        "image_id",
                    ]
                )

                if (
                    deepdrid_context
                    and
                    label_context
                ):
                    relevant_lines.append(
                        {
                            "line_number": int(
                                line_number
                            ),
                            "text": re.sub(
                                r"\s+",
                                " ",
                                line,
                            ).strip()[:1500],
                        }
                    )

                if len(
                    relevant_lines
                ) >= 100:
                    break

            if relevant_lines:
                code_reference_records.append(
                    {
                        "path": str(
                            code_path
                        ),
                        "filename": (
                            code_path.name
                        ),
                        "relevant_lines": int(
                            len(
                                relevant_lines
                            )
                        ),
                        "reference_preview": json.dumps(
                            relevant_lines
                        ),
                    }
                )


code_reference_summary = pd.DataFrame(
    code_reference_records
)


# ============================================================
# 8. Construct exact source manifest when resolved
# ============================================================

DEEPDRID_EXACT_SOURCE_MANIFEST_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Exact_Source_Manifest_v0.1.csv"
)


deepdrid_exact_source_manifest = None


if trusted_mapping_resolved:
    selected_record = (
        selected_candidate[
            "record"
        ]
    )

    selected_mapping = (
        selected_mapping[
            [
                "image_id",
                "dr_grade",
                "moderate_or_worse_dr",
            ]
        ]
        .copy()
    )

    deepdrid_exact_source_manifest = (
        deepdrid_identity
        .merge(
            selected_mapping,
            on="image_id",
            how="left",
            validate="one_to_one",
        )
    )

    assert (
        len(
            deepdrid_exact_source_manifest
        )
        ==
        DEEPDRID_EXPECTED_IMAGES
    )

    assert deepdrid_exact_source_manifest[
        "moderate_or_worse_dr"
    ].notna().all()

    deepdrid_exact_source_manifest[
        "moderate_or_worse_dr"
    ] = deepdrid_exact_source_manifest[
        "moderate_or_worse_dr"
    ].astype(int)

    deepdrid_exact_source_manifest[
        "source_image_path"
    ] = (
        deepdrid_exact_source_manifest[
            "image_id"
        ].map(
            deepdrid_image_map
        )
    )

    deepdrid_exact_source_manifest[
        "label_source_path"
    ] = (
        selected_record[
            "source_path"
        ]
    )

    deepdrid_exact_source_manifest[
        "label_source_strategy"
    ] = (
        selected_record[
            "strategy"
        ]
    )

    deepdrid_exact_source_manifest[
        "label_source_columns"
    ] = (
        selected_record[
            "label_columns"
        ]
    )

    deepdrid_exact_source_manifest[
        "label_provenance_type"
    ] = (
        selected_record[
            "label_domain"
        ]
    )

    assert deepdrid_exact_source_manifest[
        "source_image_path"
    ].map(
        lambda value: Path(
            value
        ).is_file()
    ).all()

    deepdrid_exact_source_manifest.to_csv(
        DEEPDRID_EXACT_SOURCE_MANIFEST_PATH,
        index=False,
    )


# ============================================================
# 9. Frozen Stage 4D decision
# ============================================================

if trusted_mapping_resolved:
    stage4d_decision = (
        "PASS_DEEPDRID_SOURCE_LABEL_PROVENANCE_"
        "RESOLVED_ADVANCE_TO_SOURCE_CANONICALISATION"
    )

    authorised_next_step = (
        "CANONICALISE_BOTH_SOURCE_COHORTS_WITH_"
        "THE_FROZEN_TARGET_PROTOCOL"
    )

    interpretation = (
        "The exact 1,600-image DeepDRiD source cohort, all source "
        "images, and a complete conflict-free source-label mapping "
        "were resolved. Every automatically trusted complete candidate "
        "produced the same moderate-or-worse DR endpoint assignment."
    )

elif trusted_mapping_disagreement:
    stage4d_decision = (
        "HOLD_MULTIPLE_TRUSTED_DEEPDRID_LABEL_"
        "MAPPINGS_DISAGREE"
    )

    authorised_next_step = (
        "AUDIT_DISAGREEING_DEEPDRID_LABEL_SOURCES"
    )

    interpretation = (
        "Multiple complete, semantically credible DeepDRiD label "
        "mappings were found, but they disagree on at least one "
        "moderate-or-worse DR endpoint assignment. No source "
        "canonicalisation or transfer analysis is authorised."
    )

else:
    ambiguous_full_candidates = (
        candidate_summary[
            (
                candidate_summary[
                    "full_coverage"
                ]
            )
            &
            (
                candidate_summary[
                    "conflicting_image_ids"
                ]
                ==
                0
            )
        ]
        if len(
            candidate_summary
        ) > 0
        else pd.DataFrame()
    )

    stage4d_decision = (
        "HOLD_DEEPDRID_SOURCE_LABEL_PROVENANCE_"
        "UNRESOLVED"
    )

    authorised_next_step = (
        "REVIEW_RETAINED_CODE_AND_ORIGINAL_"
        "DEEPDRID_LABEL_FILES"
    )

    interpretation = (
        "No automatically acceptable complete DeepDRiD source-label "
        "mapping was established. Any generic binary or incomplete "
        "candidate remains unauthorised until its endpoint semantics "
        "and row-order provenance are demonstrated."
    )


# ============================================================
# 10. Save Stage 4D artifacts
# ============================================================

CANDIDATE_SUMMARY_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Label_Candidate_Summary_v0.1.csv"
)

TRUSTED_CANDIDATE_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Trusted_Candidate_Agreement_v0.1.csv"
)

CODE_REFERENCE_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Label_Code_References_v0.1.csv"
)

SCAN_ROOT_PATH = (
    STAGE4D_ROOT
    / "Stage4D_Source_Label_Search_Roots_v0.1.json"
)

DECISION_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Label_Provenance_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Label_Provenance_Report_v0.1.md"
)


candidate_summary.to_csv(
    CANDIDATE_SUMMARY_PATH,
    index=False,
)

trusted_candidate_summary.to_csv(
    TRUSTED_CANDIDATE_PATH,
    index=False,
)

code_reference_summary.to_csv(
    CODE_REFERENCE_PATH,
    index=False,
)


with open(
    SCAN_ROOT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        {
            "search_roots": [
                str(
                    root
                )
                for root in search_roots
            ],
            "deepdrid_common_image_root": str(
                common_image_root
            ),
            "tables_scanned": int(
                len(
                    scanned_table_paths
                )
            ),
            "code_files_scanned": int(
                len(
                    scanned_code_paths
                )
            ),
        },
        file,
        indent=2,
    )


selected_candidate_payload = None


if trusted_mapping_resolved:
    selected_candidate_payload = dict(
        selected_candidate[
            "record"
        ]
    )

    selected_candidate_payload[
        "source_manifest_path"
    ] = str(
        DEEPDRID_EXACT_SOURCE_MANIFEST_PATH
    )

    selected_candidate_payload[
        "source_manifest_sha256"
    ] = sha256_file(
        DEEPDRID_EXACT_SOURCE_MANIFEST_PATH
    )


decision_payload = {
    "decision": (
        stage4d_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "stage4c_decision_path": str(
        STAGE4C_DECISION_PATH
    ),
    "deepdrid_identity_manifest_path": str(
        DEEPDRID_IDENTITY_PATH
    ),
    "deepdrid_image_match_path": str(
        DEEPDRID_IMAGE_MATCH_PATH
    ),
    "exact_image_ids": int(
        deepdrid_identity[
            "image_id"
        ].nunique()
    ),
    "exact_source_images": int(
        deepdrid_identity[
            "source_image_path"
        ].nunique()
    ),
    "candidate_mappings_found": int(
        len(
            candidate_summary
        )
    ),
    "trusted_complete_candidates": int(
        len(
            trusted_complete_objects
        )
    ),
    "trusted_endpoint_mappings_agree": bool(
        trusted_mapping_resolved
    ),
    "trusted_endpoint_mappings_disagree": bool(
        trusted_mapping_disagreement
    ),
    "selected_candidate": (
        selected_candidate_payload
    ),
    "target_sealed_label_files_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_images_modified": False,
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


REPORT_PATH.write_text(
    "\n".join(
        [
            "# Stage 4D — DeepDRiD Label Provenance",
            "",
            f"Decision: `{stage4d_decision}`",
            "",
            interpretation,
            "",
            "## Safety boundary",
            "",
            "- Target sealed-label files accessed: `False`",
            "- Source performance observed: `False`",
            "- Transfer performance observed: `False`",
            "- Source images modified: `False`",
        ]
    ),
    encoding="utf-8",
)


# ============================================================
# 11. Display concise outputs
# ============================================================

print(
    "\n================ DEEPDRID LABEL "
    "CANDIDATE SUMMARY ================"
)

if len(
    candidate_summary
) > 0:
    display(
        candidate_summary[
            [
                "filename",
                "strategy",
                "id_columns",
                "label_columns",
                "matched_unique_ids",
                "coverage_fraction",
                "conflicting_image_ids",
                "label_domain",
                "unique_values",
                "semantic_name_trusted",
                "automatically_acceptable",
                "source_path",
            ]
        ].head(
            40
        )
    )

else:
    print(
        "No candidate label mappings were found."
    )


print(
    "\n================ TRUSTED COMPLETE "
    "CANDIDATE AGREEMENT ================"
)

if len(
    trusted_candidate_summary
) > 0:
    display(
        trusted_candidate_summary[
            [
                "trusted_candidate_id",
                "filename",
                "strategy",
                "label_columns",
                "label_domain",
                "endpoint_mapping_sha256",
                "grade_mapping_sha256",
                "source_path",
            ]
        ]
    )

else:
    print(
        "No automatically trusted complete "
        "candidate was established."
    )


print(
    "\n================ RETAINED CODE "
    "REFERENCE SUMMARY ================"
)

if len(
    code_reference_summary
) > 0:
    display(
        code_reference_summary[
            [
                "filename",
                "relevant_lines",
                "path",
            ]
        ].head(
            20
        )
    )

else:
    print(
        "No additional retained code references "
        "were detected."
    )


if trusted_mapping_resolved:
    source_class_summary = (
        deepdrid_exact_source_manifest[
            "moderate_or_worse_dr"
        ]
        .value_counts()
        .sort_index()
        .rename_axis(
            "moderate_or_worse_dr"
        )
        .reset_index(
            name="images"
        )
    )

    print(
        "\n================ RESOLVED SOURCE "
        "ENDPOINT SUMMARY ================"
    )

    display(
        source_class_summary
    )


print(
    "\n================ STAGE 4D DECISION "
    "================"
)

print("Decision:")
print(
    stage4d_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nSource performance observed:")
print(False)

print("\nSource-target transfer performance observed:")
print(False)

print(
    "\nStage 4D DeepDRiD label provenance "
    "resolution completed and sealed."
)

================ STAGE 4D DEEPDRID LABEL PROVENANCE ================

Imported Stage 4C decision:
PASS_EYEPACS_FULLY_RESOLVED_HOLD_DEEPDRID_GRADE_PROVENANCE

Safety boundary:
Stage 4D performs read-only recovery of the DeepDRiD source-label mapping. It may inspect retained source tables and code, but it must not access target sealed-label files, modify source images, canonicalise source images, calculate source performance, or calculate source-target transfer performance.

Automatic acceptance rule:
A candidate is automatically acceptable only when it maps all 1,600 precommitted DeepDRiD image IDs without conflicts and either provides a trusted ordinal DR grade in {0,1,2,3,4}, or provides an explicitly named moderate-or-worse / referable-DR binary endpoint. Generic binary columns without explicit endpoint semantics are reported but not automatically accepted.

Recovered DeepDRiD image identities:
1600

Recovered DeepDRiD image files:
1600

Deepest common image root:
/content/drive/MyDr

,filename,strategy,id_columns,label_columns,matched_unique_ids,coverage_fraction,conflicting_image_ids,label_domain,unique_values,semantic_name_trusted,automatically_acceptable,source_path
0,ResNet50_ImageNet1K_V2_Embedding_Index.csv,single_column_pair,image_id,patient_DR_Level,1600,1.00,0,ordinal_dr_grade,0|1|2|3|4,True,True,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,ResNet50_ImageNet1K_V2_Embedding_Index.csv,single_column_pair,image_id,eye_DR_Level,1600,1.00,0,ordinal_dr_grade,0|1|2|3|4,True,True,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,ResNet50_ImageNet1K_V2_Embedding_Index.csv,embedding_row_order_join,embedding_row,patient_DR_Level,1600,1.00,0,ordinal_dr_grade,0|1|2|3|4,True,True,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
3,ResNet50_ImageNet1K_V2_Embedding_Index.csv,embedding_row_order_join,embedding_row,eye_DR_Level,1600,1.00,0,ordinal_dr_grade,0|1|2|3|4,True,True,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
4,ResNet50_ImageNet1K_V2_Embedding_Index.csv,single_column_pair,image_id,referable_DR,1600,1.00,0,explicit_binary_endpoint,0|1,True,True,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
5,ResNet50_ImageNet1K_V2_Embedding_Index.csv,embedding_row_order_join,embedding_row,referable_DR,1600,1.00,0,explicit_binary_endpoint,0|1,True,True,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
6,ResNet50_ImageNet1K_V2_Embedding_Index.csv,side_aware_table_union,image_id | eye_from_filename,patient_DR_Level | eye_DR_Level | referable_DR,1600,1.00,1120,ordinal_dr_grade,0|1|2|3|4,True,False,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
7,Stage2_LabelFree_Domain_OOF_Predictions_v0.1.csv,single_column_pair,image_id,domain_label,1600,1.00,0,ambiguous_binary,0,False,False,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
8,Stage2_LabelFree_Domain_OOF_Predictions_v0.1.csv,side_aware_table_union,image_id,domain_label,1600,1.00,0,ambiguous_binary,0,False,False,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
9,Stage2_Image_Header_Audit_v0.1.csv,single_column_pair,image_id,domain_label,1600,1.00,0,ambiguous_binary,0,False,False,/content/drive/MyDrive/Cross-Modal_Diagnostic_...



================ TRUSTED COMPLETE CANDIDATE AGREEMENT ================


,trusted_candidate_id,filename,strategy,label_columns,label_domain,endpoint_mapping_sha256,grade_mapping_sha256,source_path
0,TRUSTED_001,ResNet50_ImageNet1K_V2_Embedding_Index.csv,single_column_pair,patient_DR_Level,ordinal_dr_grade,ee9905ef10398cf9661aaecd5c96b96f0eb86f4074a984...,32dd6f83ab2e90396a5f01529c72f98761a47d09204bfa...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,TRUSTED_002,ResNet50_ImageNet1K_V2_Embedding_Index.csv,single_column_pair,eye_DR_Level,ordinal_dr_grade,e3d942a1a6e7f914ed7e951319ee62b5e65ac0f152628c...,89add2152273714e32d547ce4198b1d66cbbc6f6849f36...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,TRUSTED_003,ResNet50_ImageNet1K_V2_Embedding_Index.csv,single_column_pair,referable_DR,explicit_binary_endpoint,e3d942a1a6e7f914ed7e951319ee62b5e65ac0f152628c...,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
3,TRUSTED_004,ResNet50_ImageNet1K_V2_Embedding_Index.csv,embedding_row_order_join,patient_DR_Level,ordinal_dr_grade,ee9905ef10398cf9661aaecd5c96b96f0eb86f4074a984...,32dd6f83ab2e90396a5f01529c72f98761a47d09204bfa...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
4,TRUSTED_005,ResNet50_ImageNet1K_V2_Embedding_Index.csv,embedding_row_order_join,eye_DR_Level,ordinal_dr_grade,e3d942a1a6e7f914ed7e951319ee62b5e65ac0f152628c...,89add2152273714e32d547ce4198b1d66cbbc6f6849f36...,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
5,TRUSTED_006,ResNet50_ImageNet1K_V2_Embedding_Index.csv,embedding_row_order_join,referable_DR,explicit_binary_endpoint,e3d942a1a6e7f914ed7e951319ee62b5e65ac0f152628c...,,/content/drive/MyDrive/Cross-Modal_Diagnostic_...



================ RETAINED CODE REFERENCE SUMMARY ================


,filename,relevant_lines,path
0,Retinal_DR_Prospective_Blind_Test_Design_v0.1....,1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
1,Retinal_DR_DeepDRiD_Smoke_Test_v0.1.ipynb,1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
2,Retinal_DR_Fine_Structure_Dissection_v0.1.ipynb,1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
3,Retinal_DR_Locality_Kill_Test_v0.1.ipynb,1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
4,Retinal_DR_Frozen_Representation_Baseline_v0.1...,1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
5,Retinal_DR_External_Replication_Protocol_v0.1....,1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...
6,Retinal_DR_External_Replication_Failure_Audit_...,1,/content/drive/MyDrive/Cross-Modal_Diagnostic_...



================ STAGE 4D DECISION ================
Decision:
HOLD_MULTIPLE_TRUSTED_DEEPDRID_LABEL_MAPPINGS_DISAGREE

Interpretation:
Multiple complete, semantically credible DeepDRiD label mappings were found, but they disagree on at least one moderate-or-worse DR endpoint assignment. No source canonicalisation or transfer analysis is authorised.

Authorised next step:
AUDIT_DISAGREEING_DEEPDRID_LABEL_SOURCES

Target sealed-label files accessed:
False

Source performance observed:
False

Source-target transfer performance observed:
False

Stage 4D DeepDRiD label provenance resolution completed and sealed.


In [34]:
#@title 04E-R2. Resolve DeepDRiD eye side from official sparse-grade schema and seal provenance

from pathlib import Path

import hashlib
import json
import os
import re

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen scientific boundary
# ============================================================

DEEPDRID_EXPECTED_IMAGES = 1600
DEEPDRID_EXPECTED_PATIENTS = 400
DEEPDRID_EXPECTED_EYES = 800

EXPECTED_TRAINING_IMAGES = 1200
EXPECTED_VALIDATION_IMAGES = 400

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

PRIMARY_LABEL_UNIT = "eye"

LABEL_SELECTION_RULE = (
    "For each DeepDRiD image, infer the represented eye from the "
    "official sparse left_eye_DR_Level and right_eye_DR_Level fields: "
    "exactly one side-specific grade must be non-missing. Use that "
    "side-specific grade as the image/eye DR grade and define "
    "moderate-or-worse DR as grade >= 2. Retained eye_DR_Level and "
    "referable_DR are independent complete-column cross-checks. "
    "Patient_DR_Level is auxiliary patient-level metadata only."
)

PASS_RULE = (
    "Proceed only if all 1,600 image identities and source files "
    "resolve uniquely; both retained and official label tables have "
    "exactly one non-missing side-specific grade per image; inferred "
    "side and grade agree between retained and official tables; "
    "the inferred grade exactly matches retained eye_DR_Level; "
    "grade >= 2 exactly matches referable_DR; patient IDs, splits and "
    "patient-level grades agree; and patient-level grade equals the "
    "maximum of the two eye-level grades."
)

SAFETY_BOUNDARY = (
    "Stage 4E-R2 resolves DeepDRiD source-label semantics and "
    "provenance only. It does not access target sealed labels, "
    "canonicalise images, train models, calculate source performance, "
    "or calculate source-target transfer performance."
)


# ============================================================
# 1. Resolve project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE4_ROOT = (
    TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4C_ROOT = (
    STAGE4_ROOT
    / "02_Source_Manifest_And_Provenance_Recovery"
)

STAGE4D_ROOT = (
    STAGE4_ROOT
    / "03_DeepDRiD_Label_Provenance_Resolution"
)

STAGE4E_ROOT = (
    STAGE4_ROOT
    / "04_DeepDRiD_Eye_Level_Label_Resolution"
)

STAGE4E_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE4D_DECISION_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Label_Provenance_Decision_v0.1.json"
)

STAGE4D_CANDIDATE_PATH = (
    STAGE4D_ROOT
    / "Stage4D_DeepDRiD_Label_Candidate_Summary_v0.1.csv"
)

DEEPDRID_IDENTITY_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_Exact_Identity_Manifest_v0.1.csv"
)

DEEPDRID_IMAGE_MATCH_PATH = (
    STAGE4C_ROOT
    / "Stage4C_DeepDRiD_MyDrive_Image_Matches_v0.1.csv"
)


for required_path in [
    STAGE4D_DECISION_PATH,
    STAGE4D_CANDIDATE_PATH,
    DEEPDRID_IDENTITY_PATH,
    DEEPDRID_IMAGE_MATCH_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with open(
    STAGE4D_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage4d_decision = json.load(
        file
    )


assert (
    stage4d_decision["decision"]
    ==
    "HOLD_MULTIPLE_TRUSTED_DEEPDRID_LABEL_MAPPINGS_DISAGREE"
)


candidate_summary = pd.read_csv(
    STAGE4D_CANDIDATE_PATH
)


print(
    "================ STAGE 4E-R2 DEEPDRID "
    "EYE-LEVEL PROVENANCE ================"
)

print("\nImported Stage 4D decision:")
print(
    stage4d_decision["decision"]
)

print("\nFrozen label-selection rule:")
print(
    LABEL_SELECTION_RULE
)

print("\nFrozen pass rule:")
print(
    PASS_RULE
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)


# ============================================================
# 2. Helpers
# ============================================================

def normalise_column_name(
    value,
):
    return re.sub(
        r"[^a-z0-9]+",
        "",
        str(value).lower(),
    )


def resolve_column(
    dataframe,
    candidates,
    description,
    required=True,
):
    lookup = {
        normalise_column_name(column): column
        for column in dataframe.columns
    }

    for candidate in candidates:
        key = normalise_column_name(
            candidate
        )

        if key in lookup:
            return lookup[key]

    if required:
        raise KeyError(
            f"Could not resolve {description}.\n"
            f"Candidates: {candidates}\n"
            f"Available columns: "
            f"{list(dataframe.columns)}"
        )

    return None


def normalise_image_id(
    value,
):
    if pd.isna(value):
        return ""

    return Path(
        str(value).strip()
    ).stem


def normalise_patient_id(
    value,
):
    if pd.isna(value):
        return ""

    text = str(value).strip()

    if re.fullmatch(
        r"\d+(?:\.0+)?",
        text,
    ):
        return str(
            int(
                float(text)
            )
        )

    return text


def normalise_split(
    value,
):
    if pd.isna(value):
        return ""

    key = normalise_column_name(
        value
    )

    if key in {
        "train",
        "training",
        "development",
        "dev",
    }:
        return "training"

    if key in {
        "validation",
        "val",
        "test",
        "testing",
    }:
        return "validation"

    return key


def normalise_side_hint(
    value,
):
    if pd.isna(value):
        return "unknown"

    key = normalise_column_name(
        value
    )

    if key in {
        "l",
        "left",
        "lefteye",
        "os",
    }:
        return "left"

    if key in {
        "r",
        "right",
        "righteye",
        "od",
    }:
        return "right"

    return "unknown"


def nullable_grade(
    series,
    description,
):
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    nonmissing = numeric[
        numeric.notna()
    ]

    assert np.isfinite(
        nonmissing.to_numpy(
            dtype=float
        )
    ).all(), (
        f"{description} contains infinity."
    )

    assert np.allclose(
        nonmissing.to_numpy(
            dtype=float
        ),
        np.rint(
            nonmissing.to_numpy(
                dtype=float
            )
        ),
        atol=1e-8,
    ), (
        f"{description} contains "
        "non-integer values."
    )

    rounded = np.rint(
        nonmissing.to_numpy(
            dtype=float
        )
    ).astype(int)

    assert np.isin(
        rounded,
        [
            0,
            1,
            2,
            3,
            4,
        ],
    ).all(), (
        f"{description} contains values "
        "outside 0–4."
    )

    return pd.Series(
        pd.array(
            numeric,
            dtype="Int64",
        ),
        index=series.index,
    )


def required_grade(
    series,
    description,
):
    output = nullable_grade(
        series,
        description,
    )

    assert output.notna().all(), (
        f"{description} contains missing values."
    )

    return output.astype(int)


def required_binary(
    series,
    description,
):
    if series.dtype == bool:
        output = series.astype(int)

    else:
        cleaned = (
            series
            .astype(str)
            .str.strip()
            .str.lower()
            .replace(
                {
                    "true": "1",
                    "false": "0",
                    "yes": "1",
                    "no": "0",
                }
            )
        )

        output = pd.to_numeric(
            cleaned,
            errors="raise",
        ).astype(int)

    assert output.isin(
        [
            0,
            1,
        ]
    ).all(), (
        f"{description} contains values "
        "outside 0/1."
    )

    return output


def infer_side_and_grade(
    left_grade,
    right_grade,
    description,
):
    left_present = left_grade.notna()
    right_present = right_grade.notna()

    both_present = (
        left_present
        &
        right_present
    )

    neither_present = (
        ~left_present
        &
        ~right_present
    )

    assert not both_present.any(), (
        f"{description}: at least one row has "
        "both left and right grades populated."
    )

    assert not neither_present.any(), (
        f"{description}: at least one row has "
        "neither left nor right grade populated."
    )

    inferred_side = pd.Series(
        np.where(
            left_present,
            "left",
            "right",
        ),
        index=left_grade.index,
        dtype="object",
    )

    inferred_grade = pd.Series(
        pd.array(
            np.where(
                left_present,
                left_grade,
                right_grade,
            ),
            dtype="Int64",
        ),
        index=left_grade.index,
    )

    assert inferred_grade.notna().all()

    return (
        inferred_side,
        inferred_grade.astype(int),
        int(left_present.sum()),
        int(right_present.sum()),
    )


def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def mapping_sha256(
    dataframe,
    value_column,
):
    ordered = (
        dataframe[
            [
                "image_id",
                value_column,
            ]
        ]
        .sort_values(
            "image_id"
        )
        .reset_index(
            drop=True
        )
    )

    payload = "\n".join(
        (
            ordered[
                "image_id"
            ].astype(str)
            +
            "|"
            +
            ordered[
                value_column
            ].astype(str)
        ).tolist()
    )

    return hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()


def unique_candidate_path(
    filename,
):
    rows = candidate_summary[
        candidate_summary[
            "filename"
        ].astype(str)
        ==
        filename
    ]

    assert len(rows) > 0, (
        f"No retained path found for "
        f"{filename}."
    )

    existing_paths = sorted(
        {
            str(
                Path(path)
            )
            for path in rows[
                "source_path"
            ].astype(str)
            if Path(path).is_file()
        }
    )

    assert len(
        existing_paths
    ) == 1, (
        f"Expected exactly one existing path "
        f"for {filename}; found:\n"
        +
        "\n".join(
            existing_paths
        )
    )

    return Path(
        existing_paths[0]
    )


# ============================================================
# 3. Resolve retained and official source tables
# ============================================================

EMBEDDING_INDEX_PATH = unique_candidate_path(
    "ResNet50_ImageNet1K_V2_Embedding_Index.csv"
)

OFFICIAL_TRAINING_PATH = unique_candidate_path(
    "regular-fundus-training.csv"
)

OFFICIAL_VALIDATION_PATH = unique_candidate_path(
    "regular-fundus-validation.csv"
)


print("\nResolved retained embedding index:")
print(
    EMBEDDING_INDEX_PATH
)

print("\nResolved official training labels:")
print(
    OFFICIAL_TRAINING_PATH
)

print("\nResolved official validation labels:")
print(
    OFFICIAL_VALIDATION_PATH
)


# ============================================================
# 4. Load identities and image paths
# ============================================================

identity = pd.read_csv(
    DEEPDRID_IDENTITY_PATH
)

image_matches = pd.read_csv(
    DEEPDRID_IMAGE_MATCH_PATH
)


assert (
    len(identity)
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert identity[
    "image_id"
].is_unique


image_matches[
    "image_id"
] = image_matches[
    "image_id"
].map(
    normalise_image_id
)


assert (
    image_matches[
        "image_id"
    ].nunique()
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert not (
    image_matches
    .groupby(
        "image_id"
    )
    .size()
    >
    1
).any()


image_path_map = (
    image_matches
    .set_index(
        "image_id"
    )[
        "path"
    ]
    .to_dict()
)


# ============================================================
# 5. Standardise retained embedding index
# ============================================================

index_raw = pd.read_csv(
    EMBEDDING_INDEX_PATH,
    low_memory=False,
)


assert (
    len(index_raw)
    ==
    DEEPDRID_EXPECTED_IMAGES
)


index_image_column = resolve_column(
    index_raw,
    [
        "image_id",
        "imageid",
    ],
    "index image ID",
)

index_patient_column = resolve_column(
    index_raw,
    [
        "patient_id",
        "patientid",
    ],
    "index patient ID",
)

index_split_column = resolve_column(
    index_raw,
    [
        "split",
        "partition",
        "subset",
    ],
    "index split",
)

index_row_column = resolve_column(
    index_raw,
    [
        "embedding_row",
        "row_index",
    ],
    "embedding row",
)

index_eye_hint_column = resolve_column(
    index_raw,
    [
        "eye_from_filename",
        "eye_side",
        "eyeside",
    ],
    "retained eye-side hint",
    required=False,
)

index_patient_grade_column = resolve_column(
    index_raw,
    [
        "patient_DR_Level",
        "patient_dr_grade",
    ],
    "index patient grade",
)

index_left_grade_column = resolve_column(
    index_raw,
    [
        "left_eye_DR_Level",
        "left_eye_dr_grade",
    ],
    "index left-eye grade",
)

index_right_grade_column = resolve_column(
    index_raw,
    [
        "right_eye_DR_Level",
        "right_eye_dr_grade",
    ],
    "index right-eye grade",
)

index_eye_grade_column = resolve_column(
    index_raw,
    [
        "eye_DR_Level",
        "eye_dr_grade",
    ],
    "index complete eye grade",
)

index_referable_column = resolve_column(
    index_raw,
    [
        "referable_DR",
        "moderate_or_worse_dr",
    ],
    "index referable endpoint",
)


index_left_grade = nullable_grade(
    index_raw[
        index_left_grade_column
    ],
    "index left_eye_DR_Level",
)

index_right_grade = nullable_grade(
    index_raw[
        index_right_grade_column
    ],
    "index right_eye_DR_Level",
)


(
    index_side_from_sparse,
    index_grade_from_sparse,
    index_left_rows,
    index_right_rows,
) = infer_side_and_grade(
    index_left_grade,
    index_right_grade,
    "retained embedding index",
)


if index_eye_hint_column is not None:
    index_side_hint = index_raw[
        index_eye_hint_column
    ].map(
        normalise_side_hint
    )

else:
    index_side_hint = pd.Series(
        ["unknown"] * len(index_raw),
        index=index_raw.index,
    )


index_table = pd.DataFrame(
    {
        "image_id": (
            index_raw[
                index_image_column
            ].map(
                normalise_image_id
            )
        ),
        "patient_id_index": (
            index_raw[
                index_patient_column
            ].map(
                normalise_patient_id
            )
        ),
        "source_split_index": (
            index_raw[
                index_split_column
            ].map(
                normalise_split
            )
        ),
        "embedding_row": (
            pd.to_numeric(
                index_raw[
                    index_row_column
                ],
                errors="raise",
            ).astype(int)
        ),
        "eye_side_hint": (
            index_side_hint
        ),
        "eye_side_from_sparse_index": (
            index_side_from_sparse
        ),
        "eye_grade_from_sparse_index": (
            index_grade_from_sparse
        ),
        "eye_grade_complete_index": (
            required_grade(
                index_raw[
                    index_eye_grade_column
                ],
                "index eye_DR_Level",
            )
        ),
        "referable_dr_index": (
            required_binary(
                index_raw[
                    index_referable_column
                ],
                "index referable_DR",
            )
        ),
        "patient_dr_level_index": (
            required_grade(
                index_raw[
                    index_patient_grade_column
                ],
                "index patient_DR_Level",
            )
        ),
    }
)


assert index_table[
    "image_id"
].is_unique

assert index_table[
    "embedding_row"
].is_unique


# ============================================================
# 6. Standardise official training and validation tables
# ============================================================

official_frames = []


for source_split, label_path, expected_rows in [
    (
        "training",
        OFFICIAL_TRAINING_PATH,
        EXPECTED_TRAINING_IMAGES,
    ),
    (
        "validation",
        OFFICIAL_VALIDATION_PATH,
        EXPECTED_VALIDATION_IMAGES,
    ),
]:
    raw = pd.read_csv(
        label_path,
        low_memory=False,
    )

    assert (
        len(raw)
        ==
        expected_rows
    ), (
        f"{label_path.name}: expected "
        f"{expected_rows} rows, found "
        f"{len(raw)}."
    )

    image_column = resolve_column(
        raw,
        [
            "image_id",
            "imageid",
        ],
        f"{source_split} image ID",
    )

    patient_column = resolve_column(
        raw,
        [
            "patient_id",
            "patientid",
        ],
        f"{source_split} patient ID",
    )

    patient_grade_column = resolve_column(
        raw,
        [
            "patient_DR_Level",
            "patient_dr_grade",
        ],
        f"{source_split} patient grade",
    )

    left_grade_column = resolve_column(
        raw,
        [
            "left_eye_DR_Level",
            "left_eye_dr_grade",
        ],
        f"{source_split} left-eye grade",
    )

    right_grade_column = resolve_column(
        raw,
        [
            "right_eye_DR_Level",
            "right_eye_dr_grade",
        ],
        f"{source_split} right-eye grade",
    )

    left_grade = nullable_grade(
        raw[
            left_grade_column
        ],
        (
            f"{source_split} "
            "left_eye_DR_Level"
        ),
    )

    right_grade = nullable_grade(
        raw[
            right_grade_column
        ],
        (
            f"{source_split} "
            "right_eye_DR_Level"
        ),
    )

    (
        side_from_sparse,
        grade_from_sparse,
        left_rows,
        right_rows,
    ) = infer_side_and_grade(
        left_grade,
        right_grade,
        (
            f"official {source_split} labels"
        ),
    )

    standardised = pd.DataFrame(
        {
            "image_id": (
                raw[
                    image_column
                ].map(
                    normalise_image_id
                )
            ),
            "patient_id_official": (
                raw[
                    patient_column
                ].map(
                    normalise_patient_id
                )
            ),
            "source_split_official": (
                source_split
            ),
            "eye_side_from_sparse_official": (
                side_from_sparse
            ),
            "eye_grade_from_sparse_official": (
                grade_from_sparse
            ),
            "patient_dr_level_official": (
                required_grade(
                    raw[
                        patient_grade_column
                    ],
                    (
                        f"{source_split} "
                        "patient_DR_Level"
                    ),
                )
            ),
            "official_label_file": str(
                label_path
            ),
        }
    )

    standardised[
        "official_left_sparse_rows"
    ] = int(
        left_rows
    )

    standardised[
        "official_right_sparse_rows"
    ] = int(
        right_rows
    )

    official_frames.append(
        standardised
    )


official_table = pd.concat(
    official_frames,
    ignore_index=True,
)


assert (
    len(official_table)
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert official_table[
    "image_id"
].is_unique


# ============================================================
# 7. Exact identity/index/official join
# ============================================================

identity_table = identity[
    [
        "image_id",
    ]
].copy()

identity_table[
    "image_id"
] = identity_table[
    "image_id"
].map(
    normalise_image_id
)


merged = (
    identity_table
    .merge(
        index_table,
        on="image_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        official_table,
        on="image_id",
        how="inner",
        validate="one_to_one",
    )
)


assert (
    len(merged)
    ==
    DEEPDRID_EXPECTED_IMAGES
)


merged[
    "source_image_path"
] = (
    merged[
        "image_id"
    ].map(
        image_path_map
    )
)


assert merged[
    "source_image_path"
].notna().all()

assert merged[
    "source_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


merged[
    "moderate_or_worse_dr"
] = (
    merged[
        "eye_grade_from_sparse_official"
    ]
    >=
    2
).astype(int)


merged[
    "patient_moderate_or_worse_dr"
] = (
    merged[
        "patient_dr_level_official"
    ]
    >=
    2
).astype(int)


# ============================================================
# 8. Patient/eye structure audit
# ============================================================

patient_eye_table = (
    merged
    .groupby(
        [
            "patient_id_official",
            "eye_side_from_sparse_official",
        ],
        as_index=False,
    )
    .agg(
        eye_grade=(
            "eye_grade_from_sparse_official",
            "first",
        ),
        distinct_eye_grades=(
            "eye_grade_from_sparse_official",
            "nunique",
        ),
        images=(
            "image_id",
            "size",
        ),
    )
)


assert (
    patient_eye_table[
        "distinct_eye_grades"
    ]
    ==
    1
).all(), (
    "Two images belonging to one patient eye "
    "have inconsistent grades."
)


patient_max_grade = (
    patient_eye_table
    .groupby(
        "patient_id_official"
    )[
        "eye_grade"
    ]
    .max()
)


merged[
    "expected_patient_grade_from_eyes"
] = (
    merged[
        "patient_id_official"
    ].map(
        patient_max_grade
    ).astype(int)
)


patient_image_counts = (
    merged
    .groupby(
        "patient_id_official"
    )
    .size()
)


patient_eye_image_counts = (
    merged
    .groupby(
        [
            "patient_id_official",
            "eye_side_from_sparse_official",
        ]
    )
    .size()
)


# ============================================================
# 9. Full provenance audit
# ============================================================

known_hint_rows = (
    merged[
        "eye_side_hint"
    ]
    !=
    "unknown"
)


audit_metrics = {
    "image_rows": int(
        len(merged)
    ),
    "unique_image_ids": int(
        merged[
            "image_id"
        ].nunique()
    ),
    "unique_source_image_paths": int(
        merged[
            "source_image_path"
        ].nunique()
    ),
    "unique_patients": int(
        merged[
            "patient_id_official"
        ].nunique()
    ),
    "unique_patient_eye_units": int(
        len(
            patient_eye_table
        )
    ),

    "index_left_sparse_rows": int(
        index_left_rows
    ),
    "index_right_sparse_rows": int(
        index_right_rows
    ),
    "official_left_sparse_rows": int(
        (
            merged[
                "eye_side_from_sparse_official"
            ]
            ==
            "left"
        ).sum()
    ),
    "official_right_sparse_rows": int(
        (
            merged[
                "eye_side_from_sparse_official"
            ]
            ==
            "right"
        ).sum()
    ),

    "eye_side_hint_known_rows": int(
        known_hint_rows.sum()
    ),
    "eye_side_hint_unknown_rows": int(
        (
            ~known_hint_rows
        ).sum()
    ),
    "eye_side_hint_vs_sparse_mismatches": int(
        (
            merged.loc[
                known_hint_rows,
                "eye_side_hint",
            ]
            !=
            merged.loc[
                known_hint_rows,
                "eye_side_from_sparse_official",
            ]
        ).sum()
    ),

    "patient_id_index_official_mismatches": int(
        (
            merged[
                "patient_id_index"
            ]
            !=
            merged[
                "patient_id_official"
            ]
        ).sum()
    ),
    "split_index_official_mismatches": int(
        (
            merged[
                "source_split_index"
            ]
            !=
            merged[
                "source_split_official"
            ]
        ).sum()
    ),
    "patient_grade_index_official_mismatches": int(
        (
            merged[
                "patient_dr_level_index"
            ]
            !=
            merged[
                "patient_dr_level_official"
            ]
        ).sum()
    ),
    "index_sparse_side_vs_official_sparse_side_mismatches": int(
        (
            merged[
                "eye_side_from_sparse_index"
            ]
            !=
            merged[
                "eye_side_from_sparse_official"
            ]
        ).sum()
    ),
    "index_sparse_grade_vs_official_sparse_grade_mismatches": int(
        (
            merged[
                "eye_grade_from_sparse_index"
            ]
            !=
            merged[
                "eye_grade_from_sparse_official"
            ]
        ).sum()
    ),
    "official_sparse_grade_vs_complete_eye_grade_mismatches": int(
        (
            merged[
                "eye_grade_from_sparse_official"
            ]
            !=
            merged[
                "eye_grade_complete_index"
            ]
        ).sum()
    ),
    "derived_endpoint_vs_referable_mismatches": int(
        (
            merged[
                "moderate_or_worse_dr"
            ]
            !=
            merged[
                "referable_dr_index"
            ]
        ).sum()
    ),
    "patient_grade_not_max_eye_grade": int(
        (
            merged[
                "patient_dr_level_official"
            ]
            !=
            merged[
                "expected_patient_grade_from_eyes"
            ]
        ).sum()
    ),
    "patient_endpoint_vs_eye_endpoint_disagreements": int(
        (
            merged[
                "patient_moderate_or_worse_dr"
            ]
            !=
            merged[
                "moderate_or_worse_dr"
            ]
        ).sum()
    ),
    "patients_not_exactly_four_images": int(
        (
            patient_image_counts
            !=
            4
        ).sum()
    ),
    "patient_eye_units_not_exactly_two_images": int(
        (
            patient_eye_image_counts
            !=
            2
        ).sum()
    ),
}


assert (
    audit_metrics[
        "image_rows"
    ]
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert (
    audit_metrics[
        "unique_image_ids"
    ]
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert (
    audit_metrics[
        "unique_source_image_paths"
    ]
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert (
    audit_metrics[
        "unique_patients"
    ]
    ==
    DEEPDRID_EXPECTED_PATIENTS
)

assert (
    audit_metrics[
        "unique_patient_eye_units"
    ]
    ==
    DEEPDRID_EXPECTED_EYES
)


zero_required_metrics = [
    "patient_id_index_official_mismatches",
    "split_index_official_mismatches",
    "patient_grade_index_official_mismatches",
    "index_sparse_side_vs_official_sparse_side_mismatches",
    "index_sparse_grade_vs_official_sparse_grade_mismatches",
    "official_sparse_grade_vs_complete_eye_grade_mismatches",
    "derived_endpoint_vs_referable_mismatches",
    "patient_grade_not_max_eye_grade",
    "patients_not_exactly_four_images",
    "patient_eye_units_not_exactly_two_images",
]


for metric_name in zero_required_metrics:
    assert (
        audit_metrics[
            metric_name
        ]
        ==
        0
    ), (
        f"Stage 4E-R2 provenance check failed: "
        f"{metric_name} = "
        f"{audit_metrics[metric_name]}"
    )


assert (
    audit_metrics[
        "patient_endpoint_vs_eye_endpoint_disagreements"
    ]
    >
    0
), (
    "Expected patient-level and eye-level endpoint "
    "differences were not reproduced."
)


# ============================================================
# 10. Construct authorised exact source manifest
# ============================================================

exact_source_manifest = pd.DataFrame(
    {
        "dataset": "DeepDRiD",
        "image_id": (
            merged[
                "image_id"
            ]
        ),
        "patient_id": (
            merged[
                "patient_id_official"
            ]
        ),
        "eye_side": (
            merged[
                "eye_side_from_sparse_official"
            ]
        ),
        "source_split": (
            merged[
                "source_split_official"
            ]
        ),
        "embedding_row": (
            merged[
                "embedding_row"
            ]
        ),
        "source_image_path": (
            merged[
                "source_image_path"
            ]
        ),
        "dr_grade": (
            merged[
                "eye_grade_from_sparse_official"
            ]
        ),
        "moderate_or_worse_dr": (
            merged[
                "moderate_or_worse_dr"
            ]
        ),
        "patient_dr_level_auxiliary": (
            merged[
                "patient_dr_level_official"
            ]
        ),
        "eye_side_hint_auxiliary": (
            merged[
                "eye_side_hint"
            ]
        ),
        "referable_dr_original_crosscheck": (
            merged[
                "referable_dr_index"
            ]
        ),
        "primary_label_unit": (
            PRIMARY_LABEL_UNIT
        ),
        "label_derivation": (
            "eye side inferred from exclusive "
            "non-missing official side-specific grade"
        ),
        "official_label_file": (
            merged[
                "official_label_file"
            ]
        ),
    }
)


assert (
    len(
        exact_source_manifest
    )
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert exact_source_manifest[
    "image_id"
].is_unique


EXACT_SOURCE_MANIFEST_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_Exact_EyeLevel_Source_Manifest_v0.1.csv"
)


exact_source_manifest.to_csv(
    EXACT_SOURCE_MANIFEST_PATH,
    index=False,
)


# ============================================================
# 11. Summaries and commitments
# ============================================================

audit_summary = pd.DataFrame(
    [
        {
            "metric": key,
            "value": value,
        }
        for key, value in (
            audit_metrics.items()
        )
    ]
)


grade_summary = (
    exact_source_manifest[
        "dr_grade"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "dr_grade"
    )
    .reset_index(
        name="images"
    )
)


endpoint_summary = (
    exact_source_manifest[
        "moderate_or_worse_dr"
    ]
    .value_counts()
    .sort_index()
    .rename_axis(
        "moderate_or_worse_dr"
    )
    .reset_index(
        name="images"
    )
)


split_summary = (
    exact_source_manifest
    .groupby(
        "source_split",
        as_index=False,
    )
    .agg(
        images=(
            "image_id",
            "size",
        ),
        patients=(
            "patient_id",
            "nunique",
        ),
    )
)


eye_counts_by_split = (
    exact_source_manifest[
        [
            "source_split",
            "patient_id",
            "eye_side",
        ]
    ]
    .drop_duplicates()
    .groupby(
        "source_split"
    )
    .size()
)


split_summary[
    "eyes"
] = split_summary[
    "source_split"
].map(
    eye_counts_by_split
).astype(int)


eye_grade_mapping_hash = mapping_sha256(
    exact_source_manifest,
    "dr_grade",
)

eye_endpoint_mapping_hash = mapping_sha256(
    exact_source_manifest,
    "moderate_or_worse_dr",
)


patient_endpoint_frame = (
    exact_source_manifest[
        [
            "image_id",
            "patient_dr_level_auxiliary",
        ]
    ]
    .copy()
)

patient_endpoint_frame[
    "patient_moderate_or_worse_dr"
] = (
    patient_endpoint_frame[
        "patient_dr_level_auxiliary"
    ]
    >=
    2
).astype(int)


patient_endpoint_mapping_hash = mapping_sha256(
    patient_endpoint_frame,
    "patient_moderate_or_worse_dr",
)


# ============================================================
# 12. Save final decision
# ============================================================

stage4e_decision = (
    "PASS_DEEPDRID_EYE_LEVEL_LABEL_PROVENANCE_"
    "RESOLVED_ADVANCE_TO_SOURCE_CANONICALISATION"
)

authorised_next_step = (
    "CANONICALISE_EYEPACS_AND_DEEPDRID_WITH_"
    "THE_FROZEN_TARGET_PROTOCOL"
)

interpretation = (
    "The Stage 4D disagreement was caused by label-unit mismatch. "
    "DeepDRiD's side-specific grade fields are sparse: exactly one "
    "of the left-eye and right-eye grade columns is populated for "
    "each image. That exclusive non-missing pattern provides the "
    "authoritative image-eye assignment. Across all 1,600 images, "
    "the retained and official sparse side/grade assignments agree, "
    "the official eye grade matches complete eye_DR_Level, and "
    "grade >= 2 matches referable_DR. Patient_DR_Level remains "
    "auxiliary patient-level metadata only."
)


AUDIT_SUMMARY_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_Label_Unit_Audit_v0.1.csv"
)

GRADE_SUMMARY_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_EyeLevel_Grade_Summary_v0.1.csv"
)

ENDPOINT_SUMMARY_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_Endpoint_Summary_v0.1.csv"
)

SPLIT_SUMMARY_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_Source_Split_Summary_v0.1.csv"
)

COMMITMENT_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_Label_Commitments_v0.1.json"
)

DECISION_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_EyeLevel_Label_Decision_v0.1.json"
)

REPORT_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_EyeLevel_Label_Report_v0.1.md"
)


audit_summary.to_csv(
    AUDIT_SUMMARY_PATH,
    index=False,
)

grade_summary.to_csv(
    GRADE_SUMMARY_PATH,
    index=False,
)

endpoint_summary.to_csv(
    ENDPOINT_SUMMARY_PATH,
    index=False,
)

split_summary.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False,
)


commitment_payload = {
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "primary_label_unit": (
        PRIMARY_LABEL_UNIT
    ),
    "label_selection_rule": (
        LABEL_SELECTION_RULE
    ),
    "exact_source_manifest_path": str(
        EXACT_SOURCE_MANIFEST_PATH
    ),
    "exact_source_manifest_sha256": sha256_file(
        EXACT_SOURCE_MANIFEST_PATH
    ),
    "official_training_label_path": str(
        OFFICIAL_TRAINING_PATH
    ),
    "official_training_label_sha256": sha256_file(
        OFFICIAL_TRAINING_PATH
    ),
    "official_validation_label_path": str(
        OFFICIAL_VALIDATION_PATH
    ),
    "official_validation_label_sha256": sha256_file(
        OFFICIAL_VALIDATION_PATH
    ),
    "retained_embedding_index_path": str(
        EMBEDDING_INDEX_PATH
    ),
    "retained_embedding_index_sha256": sha256_file(
        EMBEDDING_INDEX_PATH
    ),
    "eye_grade_mapping_sha256": (
        eye_grade_mapping_hash
    ),
    "eye_endpoint_mapping_sha256": (
        eye_endpoint_mapping_hash
    ),
    "patient_endpoint_mapping_sha256": (
        patient_endpoint_mapping_hash
    ),
    "patient_and_eye_endpoint_hashes_differ": bool(
        patient_endpoint_mapping_hash
        !=
        eye_endpoint_mapping_hash
    ),
    "eye_side_hint_used_as_primary_label_source": False,
    "target_sealed_labels_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
}


with open(
    COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        commitment_payload,
        file,
        indent=2,
    )


decision_payload = {
    "decision": (
        stage4e_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "primary_label_unit": (
        PRIMARY_LABEL_UNIT
    ),
    "label_selection_rule": (
        LABEL_SELECTION_RULE
    ),
    "pass_rule": (
        PASS_RULE
    ),
    "exact_source_manifest_path": str(
        EXACT_SOURCE_MANIFEST_PATH
    ),
    "exact_source_manifest_sha256": sha256_file(
        EXACT_SOURCE_MANIFEST_PATH
    ),
    "audit_metrics": (
        audit_metrics
    ),
    "eye_grade_mapping_sha256": (
        eye_grade_mapping_hash
    ),
    "eye_endpoint_mapping_sha256": (
        eye_endpoint_mapping_hash
    ),
    "patient_endpoint_mapping_sha256": (
        patient_endpoint_mapping_hash
    ),
    "eye_side_hint_used_as_primary_label_source": False,
    "target_sealed_label_files_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_images_modified": False,
}


with open(
    DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


REPORT_PATH.write_text(
    "\n".join(
        [
            (
                "# Stage 4E-R2 — DeepDRiD "
                "Eye-Level Label Resolution"
            ),
            "",
            f"Decision: `{stage4e_decision}`",
            "",
            interpretation,
            "",
            "## Authorised primary label",
            "",
            "- Unit: eye/image",
            (
                "- Eye side: exclusive non-missing "
                "official side-specific grade field"
            ),
            (
                "- Grade: corresponding official "
                "eye-specific DR grade"
            ),
            (
                "- Endpoint: official eye grade >= 2"
            ),
            (
                "- Patient_DR_Level: auxiliary "
                "patient-level metadata only"
            ),
            (
                "- eye_from_filename: auxiliary "
                "hint only"
            ),
            "",
            "## Safety boundary",
            "",
            (
                "- Target sealed labels accessed: "
                "`False`"
            ),
            (
                "- Source performance observed: "
                "`False`"
            ),
            (
                "- Transfer performance observed: "
                "`False`"
            ),
            (
                "- Source images modified: `False`"
            ),
        ]
    ),
    encoding="utf-8",
)


# ============================================================
# 13. Final display
# ============================================================

print(
    "\n================ LABEL UNIT AUDIT "
    "================"
)

display(
    audit_summary
)


print(
    "\n================ EYE-LEVEL GRADE SUMMARY "
    "================"
)

display(
    grade_summary
)


print(
    "\n================ EYE-LEVEL ENDPOINT SUMMARY "
    "================"
)

display(
    endpoint_summary
)


print(
    "\n================ SOURCE SPLIT SUMMARY "
    "================"
)

display(
    split_summary
)


print(
    "\n================ LABEL MAPPING COMMITMENTS "
    "================"
)

print(
    "Eye-grade mapping SHA-256:"
)

print(
    eye_grade_mapping_hash
)

print(
    "\nEye-level endpoint SHA-256:"
)

print(
    eye_endpoint_mapping_hash
)

print(
    "\nPatient-level endpoint SHA-256:"
)

print(
    patient_endpoint_mapping_hash
)

print(
    "\nPatient and eye endpoint mappings differ:"
)

print(
    patient_endpoint_mapping_hash
    !=
    eye_endpoint_mapping_hash
)


print(
    "\n================ STAGE 4E-R2 DECISION "
    "================"
)

print("Decision:")
print(
    stage4e_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nSource performance observed:")
print(False)

print("\nSource-target transfer performance observed:")
print(False)

print(
    "\nStage 4E-R2 eye-level provenance "
    "resolution completed and sealed."
)

================ STAGE 4E-R2 DEEPDRID EYE-LEVEL PROVENANCE ================

Imported Stage 4D decision:
HOLD_MULTIPLE_TRUSTED_DEEPDRID_LABEL_MAPPINGS_DISAGREE

Frozen label-selection rule:
For each DeepDRiD image, infer the represented eye from the official sparse left_eye_DR_Level and right_eye_DR_Level fields: exactly one side-specific grade must be non-missing. Use that side-specific grade as the image/eye DR grade and define moderate-or-worse DR as grade >= 2. Retained eye_DR_Level and referable_DR are independent complete-column cross-checks. Patient_DR_Level is auxiliary patient-level metadata only.

Frozen pass rule:
Proceed only if all 1,600 image identities and source files resolve uniquely; both retained and official label tables have exactly one non-missing side-specific grade per image; inferred side and grade agree between retained and official tables; the inferred grade exactly matches retained eye_DR_Level; grade >= 2 exactly matches referable_DR; patient IDs, splits an

,metric,value
0,image_rows,1600
1,unique_image_ids,1600
2,unique_source_image_paths,1600
3,unique_patients,400
4,unique_patient_eye_units,800
5,index_left_sparse_rows,800
6,index_right_sparse_rows,800
7,official_left_sparse_rows,800
8,official_right_sparse_rows,800
9,eye_side_hint_known_rows,1600



================ EYE-LEVEL GRADE SUMMARY ================


,dr_grade,images
0,0,714
1,1,186
2,2,326
3,3,282
4,4,92



================ EYE-LEVEL ENDPOINT SUMMARY ================


,moderate_or_worse_dr,images
0,0,900
1,1,700



================ SOURCE SPLIT SUMMARY ================


,source_split,images,patients,eyes
0,training,1200,300,600
1,validation,400,100,200



================ LABEL MAPPING COMMITMENTS ================
Eye-grade mapping SHA-256:
89add2152273714e32d547ce4198b1d66cbbc6f6849f363b816c4291e2beb5aa

Eye-level endpoint SHA-256:
e3d942a1a6e7f914ed7e951319ee62b5e65ac0f152628c76b2ffd31dfd95b1d3

Patient-level endpoint SHA-256:
ee9905ef10398cf9661aaecd5c96b96f0eb86f4074a9847d5d60419c97cd4cc7

Patient and eye endpoint mappings differ:
True

================ STAGE 4E-R2 DECISION ================
Decision:
PASS_DEEPDRID_EYE_LEVEL_LABEL_PROVENANCE_RESOLVED_ADVANCE_TO_SOURCE_CANONICALISATION

Interpretation:
The Stage 4D disagreement was caused by label-unit mismatch. DeepDRiD's side-specific grade fields are sparse: exactly one of the left-eye and right-eye grade columns is populated for each image. That exclusive non-missing pattern provides the authoritative image-eye assignment. Across all 1,600 images, the retained and official sparse side/grade assignments agree, the official eye grade matches complete eye_DR_Level, and grade >= 2 ma

In [35]:
#@title 04F. Canonicalise both source cohorts and perform pre-performance integrity audit

from pathlib import Path
from io import BytesIO
from datetime import datetime, timezone
from PIL import Image, ImageFile

from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import re
import shutil
import sys

import numpy as np
import pandas as pd
import PIL


# ============================================================
# 0. Frozen Stage 4F protocol
# ============================================================

EYEPACS_EXPECTED_IMAGES = 4468
EYEPACS_EXPECTED_DEVELOPMENT = 3350
EYEPACS_EXPECTED_VALIDATION = 1118

DEEPDRID_EXPECTED_IMAGES = 1600
DEEPDRID_EXPECTED_DEVELOPMENT = 1200
DEEPDRID_EXPECTED_VALIDATION = 400

TOTAL_EXPECTED_IMAGES = (
    EYEPACS_EXPECTED_IMAGES
    +
    DEEPDRID_EXPECTED_IMAGES
)

MINIMUM_DATASET_RETAINED_FRACTION = 0.95
MINIMUM_SPLIT_RETAINED_FRACTION = 0.95
MINIMUM_RETAINED_IMAGES_PER_GRADE = 50

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

INTEGRITY_RULE = (
    "Within each source dataset, construct connected components "
    "using exact original-image SHA-256 equality and exact canonical-"
    "image SHA-256 equality. Exclude an entire component if it "
    "contains more than one official eye-level DR grade or crosses "
    "the frozen source development/validation boundary. Retain "
    "label-consistent duplicate components contained wholly within "
    "one source partition, but preserve their shared component ID. "
    "Any exact image overlap between EyePACS and DeepDRiD places "
    "source finalisation on hold."
)

PASS_RULE = (
    "Proceed only if no exact original or canonical image overlaps "
    "between EyePACS and DeepDRiD; each source retains at least 95% "
    "of its precommitted images; each source partition retains at "
    "least 95%; and every original DR grade retains at least 50 images."
)

SAFETY_BOUNDARY = (
    "Stage 4F applies the target-frozen canonicalisation to source "
    "images and audits source integrity before observing source-model "
    "or source-target transfer performance. It does not access target "
    "sealed labels, train a model, calculate source performance, or "
    "calculate transfer performance."
)

ImageFile.LOAD_TRUNCATED_IMAGES = False


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE2_ROOT = (
    TEST_ROOT
    / "Stage2_Acquisition_And_Quarantine_v0.1"
)

STAGE3_ROOT = (
    TEST_ROOT
    / "Stage3_Development_Recoverability_Gate_v0.1"
)

STAGE4_ROOT = (
    TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4C_ROOT = (
    STAGE4_ROOT
    / "02_Source_Manifest_And_Provenance_Recovery"
)

STAGE4E_ROOT = (
    STAGE4_ROOT
    / "04_DeepDRiD_Eye_Level_Label_Resolution"
)

STAGE4F_ROOT = (
    STAGE4_ROOT
    / "05_Source_Canonicalisation_And_Integrity_Audit"
)

CANONICAL_STAGING_ROOT = (
    STAGE4_ROOT
    / "06_Canonical_Source_Images__STAGING"
)

EYEPACS_STAGING_ROOT = (
    CANONICAL_STAGING_ROOT
    / "EyePACS_2015"
)

DEEPDRID_STAGING_ROOT = (
    CANONICAL_STAGING_ROOT
    / "DeepDRiD"
)


STAGE4F_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


STAGE2_DECISION_PATH = (
    STAGE2_ROOT
    / "Stage2_Acquisition_And_Quarantine_Decision_v0.1.json"
)

STAGE3_DECISION_PATH = (
    STAGE3_ROOT
    / "02_Results"
    / "Stage3A_Target_Recoverability_Decision_v0.1.json"
)

TARGET_CANONICAL_PROTOCOL_PATH = (
    TEST_ROOT
    / "00_Protocol"
    / "Canonical_Fundus_Preprocessing_Protocol_v0.1.json"
)

EYEPACS_EXACT_MANIFEST_PATH = (
    STAGE4C_ROOT
    / "Stage4C_EyePACS_Exact_Source_Manifest_v0.1.csv"
)

DEEPDRID_STAGE4E_DECISION_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_EyeLevel_Label_Decision_v0.1.json"
)

DEEPDRID_EXACT_MANIFEST_PATH = (
    STAGE4E_ROOT
    / "Stage4E_DeepDRiD_Exact_EyeLevel_Source_Manifest_v0.1.csv"
)

STAGE4F_DECISION_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Integrity_Decision_v0.1.json"
)


for required_path in [
    STAGE2_DECISION_PATH,
    STAGE3_DECISION_PATH,
    TARGET_CANONICAL_PROTOCOL_PATH,
    EYEPACS_EXACT_MANIFEST_PATH,
    DEEPDRID_STAGE4E_DECISION_PATH,
    DEEPDRID_EXACT_MANIFEST_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


# ============================================================
# 2. Verify prior decisions
# ============================================================

with open(
    STAGE2_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage2_decision = json.load(
        file
    )


with open(
    STAGE3_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage3_decision = json.load(
        file
    )


with open(
    DEEPDRID_STAGE4E_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage4e_decision = json.load(
        file
    )


with open(
    TARGET_CANONICAL_PROTOCOL_PATH,
    "r",
    encoding="utf-8",
) as file:
    canonical_protocol = json.load(
        file
    )


assert (
    stage2_decision[
        "decision"
    ]
    ==
    "PASS_STORAGE_EFFICIENT_CONTROLLED_ACQUISITION_AND_LABEL_QUARANTINE_ADVANCE_TO_DEVELOPMENT_RECOVERABILITY_GATE"
)

assert (
    stage3_decision[
        "decision"
    ]
    ==
    "PASS_BOTH_TARGETS_RECOVERABLE_ADVANCE_ALL_FOUR_EDGES"
)

assert (
    stage4e_decision[
        "decision"
    ]
    ==
    "PASS_DEEPDRID_EYE_LEVEL_LABEL_PROVENANCE_RESOLVED_ADVANCE_TO_SOURCE_CANONICALISATION"
)


CANONICAL_SIZE = int(
    canonical_protocol[
        "canonical_size"
    ]
)

BLACK_BORDER_THRESHOLD = int(
    canonical_protocol[
        "black_border_threshold"
    ]
)

JPEG_QUALITY = int(
    canonical_protocol[
        "jpeg_quality"
    ]
)

JPEG_SUBSAMPLING = int(
    canonical_protocol[
        "jpeg_subsampling"
    ]
)


assert CANONICAL_SIZE == 768
assert BLACK_BORDER_THRESHOLD == 10
assert JPEG_QUALITY == 92
assert JPEG_SUBSAMPLING == 0


print(
    "================ STAGE 4F SOURCE "
    "CANONICALISATION ================"
)

print("\nImported Stage 3A decision:")
print(
    stage3_decision[
        "decision"
    ]
)

print("\nImported Stage 4E-R2 decision:")
print(
    stage4e_decision[
        "decision"
    ]
)

print("\nFrozen canonical protocol:")
print(
    json.dumps(
        canonical_protocol,
        indent=2,
    )
)

print("\nFrozen integrity rule:")
print(
    INTEGRITY_RULE
)

print("\nFrozen pass rule:")
print(
    PASS_RULE
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)


# ============================================================
# 3. Do not overwrite a successful Stage 4F run
# ============================================================

if STAGE4F_DECISION_PATH.is_file():
    try:
        with open(
            STAGE4F_DECISION_PATH,
            "r",
            encoding="utf-8",
        ) as file:
            previous_stage4f = json.load(
                file
            )

        if previous_stage4f.get(
            "decision"
        ) == (
            "PASS_SOURCE_CANONICALISATION_AND_"
            "INTEGRITY_AUDIT_ADVANCE_TO_"
            "ELIGIBLE_SOURCE_FINALISATION"
        ):
            raise RuntimeError(
                "Stage 4F already has a successful sealed "
                "decision. Do not overwrite it."
            )

    except RuntimeError:
        raise

    except Exception:
        pass


if CANONICAL_STAGING_ROOT.exists():
    shutil.rmtree(
        CANONICAL_STAGING_ROOT
    )


EYEPACS_STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)

DEEPDRID_STAGING_ROOT.mkdir(
    parents=True,
    exist_ok=True,
)


# ============================================================
# 4. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def normalise_patient_id(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    text = str(
        value
    ).strip()

    if re.fullmatch(
        r"\d+(?:\.0+)?",
        text,
    ):
        return str(
            int(
                float(
                    text
                )
            )
        )

    return text


def normalise_eye_side(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    key = re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )

    if key in {
        "left",
        "l",
        "lefteye",
        "os",
    }:
        return "left"

    if key in {
        "right",
        "r",
        "righteye",
        "od",
    }:
        return "right"

    return str(
        value
    ).strip().lower()


def normalise_source_partition(
    value,
):
    key = re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )

    if key in {
        "development",
        "dev",
        "train",
        "training",
    }:
        return "development"

    if key in {
        "validation",
        "val",
        "test",
        "testing",
    }:
        return "validation"

    raise ValueError(
        f"Unsupported source partition: "
        f"{value}"
    )


def deterministic_filename(
    image_id,
):
    image_id = str(
        image_id
    )

    safe_stem = re.sub(
        r"[^A-Za-z0-9._-]+",
        "_",
        image_id,
    ).strip(
        "._"
    )

    if not safe_stem:
        safe_stem = "image"

    safe_stem = safe_stem[:100]

    digest = hashlib.sha256(
        image_id.encode(
            "utf-8"
        )
    ).hexdigest()[:16]

    return (
        safe_stem
        +
        "__"
        +
        digest
        +
        ".jpg"
    )


def canonicalise_bytes(
    original_bytes,
):
    with Image.open(
        BytesIO(
            original_bytes
        )
    ) as image:
        image.load()

        original_mode = str(
            image.mode
        )

        original_width = int(
            image.width
        )

        original_height = int(
            image.height
        )

        rgb = image.convert(
            "RGB"
        )

    grayscale = np.asarray(
        rgb.convert(
            "L"
        )
    )

    foreground_mask = (
        grayscale
        >
        BLACK_BORDER_THRESHOLD
    )

    if not foreground_mask.any():
        raise ValueError(
            "No pixel exceeded the frozen "
            "black-border threshold."
        )

    y_indices, x_indices = np.where(
        foreground_mask
    )

    crop_left = int(
        x_indices.min()
    )

    crop_right = int(
        x_indices.max()
        +
        1
    )

    crop_top = int(
        y_indices.min()
    )

    crop_bottom = int(
        y_indices.max()
        +
        1
    )

    cropped = rgb.crop(
        (
            crop_left,
            crop_top,
            crop_right,
            crop_bottom,
        )
    )

    cropped_width = int(
        cropped.width
    )

    cropped_height = int(
        cropped.height
    )

    square_side = int(
        max(
            cropped_width,
            cropped_height,
        )
    )

    square_canvas = Image.new(
        "RGB",
        (
            square_side,
            square_side,
        ),
        (
            0,
            0,
            0,
        ),
    )

    paste_left = int(
        (
            square_side
            -
            cropped_width
        )
        //
        2
    )

    paste_top = int(
        (
            square_side
            -
            cropped_height
        )
        //
        2
    )

    square_canvas.paste(
        cropped,
        (
            paste_left,
            paste_top,
        ),
    )

    resized = square_canvas.resize(
        (
            CANONICAL_SIZE,
            CANONICAL_SIZE,
        ),
        resample=(
            Image.Resampling.LANCZOS
        ),
    )

    output_buffer = BytesIO()

    resized.save(
        output_buffer,
        format="JPEG",
        quality=(
            JPEG_QUALITY
        ),
        subsampling=(
            JPEG_SUBSAMPLING
        ),
        optimize=False,
        progressive=False,
    )

    canonical_bytes = (
        output_buffer.getvalue()
    )

    return {
        "canonical_bytes": (
            canonical_bytes
        ),
        "original_mode": (
            original_mode
        ),
        "original_width": (
            original_width
        ),
        "original_height": (
            original_height
        ),
        "crop_left": (
            crop_left
        ),
        "crop_top": (
            crop_top
        ),
        "crop_right": (
            crop_right
        ),
        "crop_bottom": (
            crop_bottom
        ),
        "cropped_width": (
            cropped_width
        ),
        "cropped_height": (
            cropped_height
        ),
        "square_side": (
            square_side
        ),
        "paste_left": (
            paste_left
        ),
        "paste_top": (
            paste_top
        ),
    }


def build_connected_component_audit(
    dataframe,
    dataset_name,
):
    working = (
        dataframe
        .reset_index(
            drop=True
        )
        .copy()
    )

    row_count = len(
        working
    )

    parent = np.arange(
        row_count,
        dtype=np.int64,
    )

    rank = np.zeros(
        row_count,
        dtype=np.int64,
    )

    def find_root(
        index,
    ):
        index = int(
            index
        )

        while parent[
            index
        ] != index:
            parent[
                index
            ] = parent[
                parent[
                    index
                ]
            ]

            index = int(
                parent[
                    index
                ]
            )

        return index

    def union_indices(
        left_index,
        right_index,
    ):
        left_root = find_root(
            left_index
        )

        right_root = find_root(
            right_index
        )

        if left_root == right_root:
            return

        if rank[
            left_root
        ] < rank[
            right_root
        ]:
            parent[
                left_root
            ] = right_root

        elif rank[
            left_root
        ] > rank[
            right_root
        ]:
            parent[
                right_root
            ] = left_root

        else:
            parent[
                right_root
            ] = left_root

            rank[
                left_root
            ] += 1

    for hash_column in [
        "original_sha256",
        "canonical_sha256",
    ]:
        grouped_indices = (
            working
            .groupby(
                hash_column,
                sort=False,
            )
            .indices
        )

        for indices in (
            grouped_indices.values()
        ):
            indices = list(
                map(
                    int,
                    indices,
                )
            )

            if len(
                indices
            ) <= 1:
                continue

            anchor = indices[0]

            for other_index in (
                indices[1:]
            ):
                union_indices(
                    anchor,
                    other_index,
                )

    roots = [
        find_root(
            index
        )
        for index in range(
            row_count
        )
    ]

    root_to_rows = {}

    for row_index, root in enumerate(
        roots
    ):
        root_to_rows.setdefault(
            int(
                root
            ),
            [],
        ).append(
            int(
                row_index
            )
        )

    root_to_component_id = {}

    for root, row_indices in (
        root_to_rows.items()
    ):
        image_ids = sorted(
            working.iloc[
                row_indices
            ][
                "image_id"
            ]
            .astype(str)
            .tolist()
        )

        component_digest = (
            hashlib.sha256(
                "|".join(
                    image_ids
                ).encode(
                    "utf-8"
                )
            ).hexdigest()[:16]
        )

        root_to_component_id[
            root
        ] = (
            dataset_name.upper()
            .replace(
                " ",
                "_"
            )
            +
            "_COMPONENT_"
            +
            component_digest
        )

    working[
        "duplicate_component_id"
    ] = [
        root_to_component_id[
            int(
                root
            )
        ]
        for root in roots
    ]

    component_summary = (
        working
        .groupby(
            "duplicate_component_id",
            as_index=False,
        )
        .agg(
            images=(
                "image_id",
                "size",
            ),
            original_hashes=(
                "original_sha256",
                "nunique",
            ),
            canonical_hashes=(
                "canonical_sha256",
                "nunique",
            ),
            grade_count=(
                "dr_grade",
                "nunique",
            ),
            endpoint_count=(
                "moderate_or_worse_dr",
                "nunique",
            ),
            partition_count=(
                "source_partition",
                "nunique",
            ),
            patient_count=(
                "patient_id",
                "nunique",
            ),
        )
    )

    component_summary[
        "label_conflict"
    ] = (
        component_summary[
            "grade_count"
        ]
        >
        1
    )

    component_summary[
        "cross_partition"
    ] = (
        component_summary[
            "partition_count"
        ]
        >
        1
    )

    component_summary[
        "excluded_component"
    ] = (
        component_summary[
            "label_conflict"
        ]
        |
        component_summary[
            "cross_partition"
        ]
    )

    excluded_component_ids = set(
        component_summary.loc[
            component_summary[
                "excluded_component"
            ],
            "duplicate_component_id",
        ]
    )

    working[
        "excluded_due_to_label_conflict"
    ] = working[
        "duplicate_component_id"
    ].isin(
        set(
            component_summary.loc[
                component_summary[
                    "label_conflict"
                ],
                "duplicate_component_id",
            ]
        )
    )

    working[
        "excluded_due_to_cross_partition"
    ] = working[
        "duplicate_component_id"
    ].isin(
        set(
            component_summary.loc[
                component_summary[
                    "cross_partition"
                ],
                "duplicate_component_id",
            ]
        )
    )

    working[
        "excluded_by_integrity_rule"
    ] = working[
        "duplicate_component_id"
    ].isin(
        excluded_component_ids
    )

    working[
        "eligible_after_integrity_audit"
    ] = ~working[
        "excluded_by_integrity_rule"
    ]

    return (
        working,
        component_summary,
    )


# ============================================================
# 5. Load and standardise source manifests
# ============================================================

eyepacs_raw = pd.read_csv(
    EYEPACS_EXACT_MANIFEST_PATH,
    low_memory=False,
)

deepdrid_raw = pd.read_csv(
    DEEPDRID_EXACT_MANIFEST_PATH,
    low_memory=False,
)


eyepacs_required_columns = {
    "image_id",
    "source_image_path",
    "dr_grade",
    "split",
    "patient_id",
}

deepdrid_required_columns = {
    "image_id",
    "source_image_path",
    "dr_grade",
    "moderate_or_worse_dr",
    "source_split",
    "patient_id",
    "eye_side",
}


assert eyepacs_required_columns.issubset(
    eyepacs_raw.columns
), (
    "EyePACS manifest missing columns: "
    f"{sorted(eyepacs_required_columns - set(eyepacs_raw.columns))}"
)

assert deepdrid_required_columns.issubset(
    deepdrid_raw.columns
), (
    "DeepDRiD manifest missing columns: "
    f"{sorted(deepdrid_required_columns - set(deepdrid_raw.columns))}"
)


eyepacs = pd.DataFrame(
    {
        "dataset": (
            "EyePACS_2015"
        ),
        "image_id": (
            eyepacs_raw[
                "image_id"
            ].astype(str)
        ),
        "patient_id": (
            eyepacs_raw[
                "patient_id"
            ].map(
                normalise_patient_id
            )
        ),
        "eye_side": (
            eyepacs_raw[
                "eye_side"
            ].map(
                normalise_eye_side
            )
            if
            "eye_side"
            in
            eyepacs_raw.columns
            else
            ""
        ),
        "source_partition": (
            eyepacs_raw[
                "split"
            ].map(
                normalise_source_partition
            )
        ),
        "source_image_path": (
            eyepacs_raw[
                "source_image_path"
            ].astype(str)
        ),
        "dr_grade": (
            pd.to_numeric(
                eyepacs_raw[
                    "dr_grade"
                ],
                errors="raise",
            ).astype(int)
        ),
    }
)


eyepacs[
    "moderate_or_worse_dr"
] = (
    eyepacs[
        "dr_grade"
    ]
    >=
    2
).astype(int)


deepdrid = pd.DataFrame(
    {
        "dataset": (
            "DeepDRiD"
        ),
        "image_id": (
            deepdrid_raw[
                "image_id"
            ].astype(str)
        ),
        "patient_id": (
            deepdrid_raw[
                "patient_id"
            ].map(
                normalise_patient_id
            )
        ),
        "eye_side": (
            deepdrid_raw[
                "eye_side"
            ].map(
                normalise_eye_side
            )
        ),
        "source_partition": (
            deepdrid_raw[
                "source_split"
            ].map(
                normalise_source_partition
            )
        ),
        "source_image_path": (
            deepdrid_raw[
                "source_image_path"
            ].astype(str)
        ),
        "dr_grade": (
            pd.to_numeric(
                deepdrid_raw[
                    "dr_grade"
                ],
                errors="raise",
            ).astype(int)
        ),
        "moderate_or_worse_dr": (
            pd.to_numeric(
                deepdrid_raw[
                    "moderate_or_worse_dr"
                ],
                errors="raise",
            ).astype(int)
        ),
    }
)


assert (
    len(
        eyepacs
    )
    ==
    EYEPACS_EXPECTED_IMAGES
)

assert (
    len(
        deepdrid
    )
    ==
    DEEPDRID_EXPECTED_IMAGES
)

assert eyepacs[
    "image_id"
].is_unique

assert deepdrid[
    "image_id"
].is_unique


for dataset_name, dataframe in [
    (
        "EyePACS_2015",
        eyepacs,
    ),
    (
        "DeepDRiD",
        deepdrid,
    ),
]:
    assert dataframe[
        "dr_grade"
    ].isin(
        [
            0,
            1,
            2,
            3,
            4,
        ]
    ).all()

    assert dataframe[
        "moderate_or_worse_dr"
    ].isin(
        [
            0,
            1,
        ]
    ).all()

    assert (
        dataframe[
            "moderate_or_worse_dr"
        ]
        ==
        (
            dataframe[
                "dr_grade"
            ]
            >=
            2
        ).astype(int)
    ).all(), (
        f"{dataset_name} endpoint does not "
        "equal grade >= 2."
    )

    assert dataframe[
        "source_image_path"
    ].map(
        lambda value: Path(
            value
        ).is_file()
    ).all(), (
        f"{dataset_name} has missing "
        "source images."
    )


assert (
    (
        eyepacs[
            "source_partition"
        ]
        ==
        "development"
    ).sum()
    ==
    EYEPACS_EXPECTED_DEVELOPMENT
)

assert (
    (
        eyepacs[
            "source_partition"
        ]
        ==
        "validation"
    ).sum()
    ==
    EYEPACS_EXPECTED_VALIDATION
)

assert (
    (
        deepdrid[
            "source_partition"
        ]
        ==
        "development"
    ).sum()
    ==
    DEEPDRID_EXPECTED_DEVELOPMENT
)

assert (
    (
        deepdrid[
            "source_partition"
        ]
        ==
        "validation"
    ).sum()
    ==
    DEEPDRID_EXPECTED_VALIDATION
)


# ============================================================
# 6. Storage estimate from deterministic canonical samples
# ============================================================

def estimate_canonical_size(
    dataframe,
    sample_count,
):
    sample_count = min(
        sample_count,
        len(
            dataframe
        ),
    )

    sampled = dataframe.sample(
        n=sample_count,
        random_state=20260720,
    )

    canonical_sizes = []

    for source_path in sampled[
        "source_image_path"
    ]:
        original_bytes = Path(
            source_path
        ).read_bytes()

        result = canonicalise_bytes(
            original_bytes
        )

        canonical_sizes.append(
            len(
                result[
                    "canonical_bytes"
                ]
            )
        )

    return {
        "sample_images": int(
            sample_count
        ),
        "mean_bytes": float(
            np.mean(
                canonical_sizes
            )
        ),
        "median_bytes": float(
            np.median(
                canonical_sizes
            )
        ),
        "maximum_bytes": int(
            np.max(
                canonical_sizes
            )
        ),
    }


print(
    "\n================ STORAGE ESTIMATE "
    "================"
)


eyepacs_estimate = estimate_canonical_size(
    eyepacs,
    sample_count=40,
)

deepdrid_estimate = estimate_canonical_size(
    deepdrid,
    sample_count=40,
)


estimated_total_bytes = int(
    (
        eyepacs_estimate[
            "mean_bytes"
        ]
        *
        len(
            eyepacs
        )
    )
    +
    (
        deepdrid_estimate[
            "mean_bytes"
        ]
        *
        len(
            deepdrid
        )
    )
)


estimated_with_safety = int(
    estimated_total_bytes
    *
    1.30
)


drive_free_bytes = int(
    shutil.disk_usage(
        DRIVE_ROOT
    ).free
)


storage_estimate_table = pd.DataFrame(
    [
        {
            "dataset": (
                "EyePACS_2015"
            ),
            **eyepacs_estimate,
            "estimated_total_gib": float(
                eyepacs_estimate[
                    "mean_bytes"
                ]
                *
                len(
                    eyepacs
                )
                /
                (1024 ** 3)
            ),
        },
        {
            "dataset": (
                "DeepDRiD"
            ),
            **deepdrid_estimate,
            "estimated_total_gib": float(
                deepdrid_estimate[
                    "mean_bytes"
                ]
                *
                len(
                    deepdrid
                )
                /
                (1024 ** 3)
            ),
        },
    ]
)


display(
    storage_estimate_table
)


print(
    "Estimated source canonical total:",
    f"{estimated_total_bytes / (1024 ** 3):.2f} GiB",
)

print(
    "Estimated total with 30% safety:",
    f"{estimated_with_safety / (1024 ** 3):.2f} GiB",
)

print(
    "Current Drive free space:",
    f"{drive_free_bytes / (1024 ** 3):.2f} GiB",
)


assert (
    drive_free_bytes
    >
    estimated_with_safety
    +
    512
    *
    1024 ** 2
), (
    "Insufficient Drive space for source "
    "canonical staging with safety margin."
)


# ============================================================
# 7. Canonicalise both complete source cohorts
# ============================================================

def canonicalise_dataset(
    dataframe,
    output_root,
    dataset_name,
):
    records = []
    error_records = []

    ordered = (
        dataframe
        .sort_values(
            "image_id"
        )
        .reset_index(
            drop=True
        )
    )

    for _, row in tqdm(
        ordered.iterrows(),
        total=len(
            ordered
        ),
        desc=(
            f"Canonicalising {dataset_name}"
        ),
    ):
        source_path = Path(
            row[
                "source_image_path"
            ]
        )

        canonical_filename = (
            deterministic_filename(
                row[
                    "image_id"
                ]
            )
        )

        output_path = (
            output_root
            /
            canonical_filename
        )

        try:
            original_bytes = (
                source_path.read_bytes()
            )

            original_sha256 = (
                hashlib.sha256(
                    original_bytes
                ).hexdigest()
            )

            result = canonicalise_bytes(
                original_bytes
            )

            canonical_bytes = result[
                "canonical_bytes"
            ]

            canonical_sha256 = (
                hashlib.sha256(
                    canonical_bytes
                ).hexdigest()
            )

            output_path.write_bytes(
                canonical_bytes
            )

            assert (
                sha256_file(
                    output_path
                )
                ==
                canonical_sha256
            )

            record = {
                "dataset": (
                    dataset_name
                ),
                "image_id": str(
                    row[
                        "image_id"
                    ]
                ),
                "patient_id": str(
                    row[
                        "patient_id"
                    ]
                ),
                "eye_side": str(
                    row[
                        "eye_side"
                    ]
                ),
                "source_partition": str(
                    row[
                        "source_partition"
                    ]
                ),
                "dr_grade": int(
                    row[
                        "dr_grade"
                    ]
                ),
                "moderate_or_worse_dr": int(
                    row[
                        "moderate_or_worse_dr"
                    ]
                ),
                "source_image_path": str(
                    source_path
                ),
                "canonical_staging_path": str(
                    output_path
                ),
                "canonical_filename": (
                    canonical_filename
                ),
                "original_sha256": (
                    original_sha256
                ),
                "canonical_sha256": (
                    canonical_sha256
                ),
                "original_size_bytes": int(
                    len(
                        original_bytes
                    )
                ),
                "canonical_size_bytes": int(
                    len(
                        canonical_bytes
                    )
                ),
                "canonical_width": (
                    CANONICAL_SIZE
                ),
                "canonical_height": (
                    CANONICAL_SIZE
                ),
                **{
                    key: value
                    for key, value in (
                        result.items()
                    )
                    if key
                    !=
                    "canonical_bytes"
                },
            }

            records.append(
                record
            )

        except Exception as error:
            error_records.append(
                {
                    "dataset": (
                        dataset_name
                    ),
                    "image_id": str(
                        row[
                            "image_id"
                        ]
                    ),
                    "source_image_path": str(
                        source_path
                    ),
                    "error_type": type(
                        error
                    ).__name__,
                    "error_message": str(
                        error
                    ),
                }
            )

    return (
        pd.DataFrame(
            records
        ),
        pd.DataFrame(
            error_records
        ),
    )


eyepacs_integrity, eyepacs_errors = (
    canonicalise_dataset(
        dataframe=(
            eyepacs
        ),
        output_root=(
            EYEPACS_STAGING_ROOT
        ),
        dataset_name=(
            "EyePACS_2015"
        ),
    )
)


deepdrid_integrity, deepdrid_errors = (
    canonicalise_dataset(
        dataframe=(
            deepdrid
        ),
        output_root=(
            DEEPDRID_STAGING_ROOT
        ),
        dataset_name=(
            "DeepDRiD"
        ),
    )
)


canonicalisation_errors = pd.concat(
    [
        eyepacs_errors,
        deepdrid_errors,
    ],
    ignore_index=True,
)


CANONICALISATION_ERROR_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Errors_v0.1.csv"
)


canonicalisation_errors.to_csv(
    CANONICALISATION_ERROR_PATH,
    index=False,
)


assert len(
    canonicalisation_errors
) == 0, (
    "Source canonicalisation produced errors. "
    f"See: {CANONICALISATION_ERROR_PATH}"
)


assert (
    len(
        eyepacs_integrity
    )
    ==
    EYEPACS_EXPECTED_IMAGES
)

assert (
    len(
        deepdrid_integrity
    )
    ==
    DEEPDRID_EXPECTED_IMAGES
)


assert (
    len(
        list(
            EYEPACS_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    EYEPACS_EXPECTED_IMAGES
)

assert (
    len(
        list(
            DEEPDRID_STAGING_ROOT.glob(
                "*.jpg"
            )
        )
    )
    ==
    DEEPDRID_EXPECTED_IMAGES
)


# ============================================================
# 8. Exact-hash connected-component integrity audits
# ============================================================

(
    eyepacs_component_audit,
    eyepacs_component_summary,
) = build_connected_component_audit(
    dataframe=(
        eyepacs_integrity
    ),
    dataset_name=(
        "EyePACS_2015"
    ),
)


(
    deepdrid_component_audit,
    deepdrid_component_summary,
) = build_connected_component_audit(
    dataframe=(
        deepdrid_integrity
    ),
    dataset_name=(
        "DeepDRiD"
    ),
)


combined_component_audit = pd.concat(
    [
        eyepacs_component_audit,
        deepdrid_component_audit,
    ],
    ignore_index=True,
)


combined_component_summary = pd.concat(
    [
        eyepacs_component_summary.assign(
            dataset="EyePACS_2015"
        ),
        deepdrid_component_summary.assign(
            dataset="DeepDRiD"
        ),
    ],
    ignore_index=True,
)


# ============================================================
# 9. Cross-source exact-overlap audit
# ============================================================

original_cross_source_overlap = set(
    eyepacs_component_audit[
        "original_sha256"
    ]
).intersection(
    set(
        deepdrid_component_audit[
            "original_sha256"
        ]
    )
)


canonical_cross_source_overlap = set(
    eyepacs_component_audit[
        "canonical_sha256"
    ]
).intersection(
    set(
        deepdrid_component_audit[
            "canonical_sha256"
        ]
    )
)


cross_source_overlap_pass = bool(
    len(
        original_cross_source_overlap
    )
    ==
    0
    and
    len(
        canonical_cross_source_overlap
    )
    ==
    0
)


# ============================================================
# 10. Retention gates
# ============================================================

dataset_retention = (
    combined_component_audit
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        original_images=(
            "image_id",
            "size",
        ),
        excluded_images=(
            "excluded_by_integrity_rule",
            "sum",
        ),
        retained_images=(
            "eligible_after_integrity_audit",
            "sum",
        ),
    )
)


dataset_retention[
    "retained_fraction"
] = (
    dataset_retention[
        "retained_images"
    ]
    /
    dataset_retention[
        "original_images"
    ]
)


split_retention = (
    combined_component_audit
    .groupby(
        [
            "dataset",
            "source_partition",
        ],
        as_index=False,
    )
    .agg(
        original_images=(
            "image_id",
            "size",
        ),
        excluded_images=(
            "excluded_by_integrity_rule",
            "sum",
        ),
        retained_images=(
            "eligible_after_integrity_audit",
            "sum",
        ),
    )
)


split_retention[
    "retained_fraction"
] = (
    split_retention[
        "retained_images"
    ]
    /
    split_retention[
        "original_images"
    ]
)


grade_retention = (
    combined_component_audit
    .groupby(
        [
            "dataset",
            "dr_grade",
        ],
        as_index=False,
    )
    .agg(
        original_images=(
            "image_id",
            "size",
        ),
        excluded_images=(
            "excluded_by_integrity_rule",
            "sum",
        ),
        retained_images=(
            "eligible_after_integrity_audit",
            "sum",
        ),
    )
)


dataset_gate_pass = bool(
    (
        dataset_retention[
            "retained_fraction"
        ]
        >=
        MINIMUM_DATASET_RETAINED_FRACTION
    ).all()
)


split_gate_pass = bool(
    (
        split_retention[
            "retained_fraction"
        ]
        >=
        MINIMUM_SPLIT_RETAINED_FRACTION
    ).all()
)


grade_gate_pass = bool(
    (
        grade_retention[
            "retained_images"
        ]
        >=
        MINIMUM_RETAINED_IMAGES_PER_GRADE
    ).all()
)


# ============================================================
# 11. Aggregate audit summaries
# ============================================================

integrity_summary = pd.DataFrame(
    [
        {
            "dataset": (
                "EyePACS_2015"
            ),
            "images": int(
                len(
                    eyepacs_component_audit
                )
            ),
            "connected_components": int(
                eyepacs_component_summary[
                    "duplicate_component_id"
                ].nunique()
            ),
            "multi_image_components": int(
                (
                    eyepacs_component_summary[
                        "images"
                    ]
                    >
                    1
                ).sum()
            ),
            "label_conflict_components": int(
                eyepacs_component_summary[
                    "label_conflict"
                ].sum()
            ),
            "cross_partition_components": int(
                eyepacs_component_summary[
                    "cross_partition"
                ].sum()
            ),
            "excluded_components": int(
                eyepacs_component_summary[
                    "excluded_component"
                ].sum()
            ),
            "excluded_images": int(
                eyepacs_component_audit[
                    "excluded_by_integrity_rule"
                ].sum()
            ),
            "eligible_images": int(
                eyepacs_component_audit[
                    "eligible_after_integrity_audit"
                ].sum()
            ),
            "canonical_size_gib": float(
                eyepacs_component_audit[
                    "canonical_size_bytes"
                ].sum()
                /
                (1024 ** 3)
            ),
        },
        {
            "dataset": (
                "DeepDRiD"
            ),
            "images": int(
                len(
                    deepdrid_component_audit
                )
            ),
            "connected_components": int(
                deepdrid_component_summary[
                    "duplicate_component_id"
                ].nunique()
            ),
            "multi_image_components": int(
                (
                    deepdrid_component_summary[
                        "images"
                    ]
                    >
                    1
                ).sum()
            ),
            "label_conflict_components": int(
                deepdrid_component_summary[
                    "label_conflict"
                ].sum()
            ),
            "cross_partition_components": int(
                deepdrid_component_summary[
                    "cross_partition"
                ].sum()
            ),
            "excluded_components": int(
                deepdrid_component_summary[
                    "excluded_component"
                ].sum()
            ),
            "excluded_images": int(
                deepdrid_component_audit[
                    "excluded_by_integrity_rule"
                ].sum()
            ),
            "eligible_images": int(
                deepdrid_component_audit[
                    "eligible_after_integrity_audit"
                ].sum()
            ),
            "canonical_size_gib": float(
                deepdrid_component_audit[
                    "canonical_size_bytes"
                ].sum()
                /
                (1024 ** 3)
            ),
        },
    ]
)


# ============================================================
# 12. Frozen Stage 4F decision
# ============================================================

all_gates_pass = bool(
    cross_source_overlap_pass
    and
    dataset_gate_pass
    and
    split_gate_pass
    and
    grade_gate_pass
)


if all_gates_pass:
    stage4f_decision = (
        "PASS_SOURCE_CANONICALISATION_AND_"
        "INTEGRITY_AUDIT_ADVANCE_TO_"
        "ELIGIBLE_SOURCE_FINALISATION"
    )

    authorised_next_step = (
        "FINALISE_ELIGIBLE_CANONICAL_SOURCE_COHORTS"
    )

    interpretation = (
        "Both precommitted source cohorts were canonicalised using "
        "the target-frozen 768x768 protocol. Exact original and "
        "canonical duplicate components were audited before source "
        "or transfer performance was observed. Components with "
        "label conflict or source-partition leakage are marked for "
        "deterministic exclusion. All frozen retention gates passed, "
        "and no exact overlap was detected between the two sources."
    )

else:
    stage4f_decision = (
        "HOLD_SOURCE_FINALISATION_INTEGRITY_"
        "OR_RETENTION_GATE_FAILED"
    )

    authorised_next_step = (
        "REVIEW_STAGE4F_SOURCE_INTEGRITY_AUDIT"
    )

    interpretation = (
        "At least one frozen source integrity condition failed: "
        "cross-source overlap, dataset retention, partition retention, "
        "or per-grade retention. Source finalisation and all modelling "
        "remain on hold."
    )


# ============================================================
# 13. Save Stage 4F artifacts
# ============================================================

FULL_INTEGRITY_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

COMPONENT_AUDIT_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Duplicate_Component_Audit_v0.1.csv"
)

COMPONENT_SUMMARY_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Duplicate_Component_Summary_v0.1.csv"
)

EXCLUSION_MANIFEST_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Integrity_Exclusion_Manifest_v0.1.csv"
)

DATASET_RETENTION_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Dataset_Retention_v0.1.csv"
)

SPLIT_RETENTION_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Split_Retention_v0.1.csv"
)

GRADE_RETENTION_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Grade_Retention_v0.1.csv"
)

INTEGRITY_SUMMARY_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Integrity_Summary_v0.1.csv"
)

PROTOCOL_COMMITMENT_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Protocol_Commitment_v0.1.json"
)

REPORT_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Integrity_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Environment_v0.1.json"
)


combined_component_audit.to_csv(
    FULL_INTEGRITY_PATH,
    index=False,
)

combined_component_audit[
    [
        "dataset",
        "image_id",
        "patient_id",
        "source_partition",
        "dr_grade",
        "moderate_or_worse_dr",
        "original_sha256",
        "canonical_sha256",
        "duplicate_component_id",
        "excluded_due_to_label_conflict",
        "excluded_due_to_cross_partition",
        "excluded_by_integrity_rule",
        "eligible_after_integrity_audit",
        "canonical_staging_path",
    ]
].to_csv(
    COMPONENT_AUDIT_PATH,
    index=False,
)

combined_component_summary.to_csv(
    COMPONENT_SUMMARY_PATH,
    index=False,
)

combined_component_audit[
    combined_component_audit[
        "excluded_by_integrity_rule"
    ]
][
    [
        "dataset",
        "image_id",
        "patient_id",
        "source_partition",
        "original_sha256",
        "canonical_sha256",
        "duplicate_component_id",
        "excluded_due_to_label_conflict",
        "excluded_due_to_cross_partition",
    ]
].to_csv(
    EXCLUSION_MANIFEST_PATH,
    index=False,
)

dataset_retention.to_csv(
    DATASET_RETENTION_PATH,
    index=False,
)

split_retention.to_csv(
    SPLIT_RETENTION_PATH,
    index=False,
)

grade_retention.to_csv(
    GRADE_RETENTION_PATH,
    index=False,
)

integrity_summary.to_csv(
    INTEGRITY_SUMMARY_PATH,
    index=False,
)


protocol_commitment = {
    "target_canonical_protocol_path": str(
        TARGET_CANONICAL_PROTOCOL_PATH
    ),
    "target_canonical_protocol_sha256": sha256_file(
        TARGET_CANONICAL_PROTOCOL_PATH
    ),
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "black_border_threshold": (
        BLACK_BORDER_THRESHOLD
    ),
    "jpeg_quality": (
        JPEG_QUALITY
    ),
    "jpeg_subsampling": (
        JPEG_SUBSAMPLING
    ),
    "resize_filter": (
        "PIL.Image.Resampling.LANCZOS"
    ),
    "integrity_rule": (
        INTEGRITY_RULE
    ),
    "pass_rule": (
        PASS_RULE
    ),
    "eyepacs_input_manifest_path": str(
        EYEPACS_EXACT_MANIFEST_PATH
    ),
    "eyepacs_input_manifest_sha256": sha256_file(
        EYEPACS_EXACT_MANIFEST_PATH
    ),
    "deepdrid_input_manifest_path": str(
        DEEPDRID_EXACT_MANIFEST_PATH
    ),
    "deepdrid_input_manifest_sha256": sha256_file(
        DEEPDRID_EXACT_MANIFEST_PATH
    ),
    "protocol_frozen_before_source_performance": True,
    "protocol_frozen_before_transfer_performance": True,
    "target_sealed_labels_accessed": False,
}


with open(
    PROTOCOL_COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_commitment,
        file,
        indent=2,
    )


decision_payload = {
    "decision": (
        stage4f_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "integrity_rule": (
        INTEGRITY_RULE
    ),
    "pass_rule": (
        PASS_RULE
    ),
    "source_canonical_staging_root": str(
        CANONICAL_STAGING_ROOT
    ),
    "integrity_manifest_path": str(
        FULL_INTEGRITY_PATH
    ),
    "component_audit_path": str(
        COMPONENT_AUDIT_PATH
    ),
    "component_summary_path": str(
        COMPONENT_SUMMARY_PATH
    ),
    "dataset_retention": (
        dataset_retention
        .to_dict(
            orient="records"
        )
    ),
    "split_retention": (
        split_retention
        .to_dict(
            orient="records"
        )
    ),
    "grade_retention": (
        grade_retention
        .to_dict(
            orient="records"
        )
    ),
    "original_cross_source_overlap_groups": int(
        len(
            original_cross_source_overlap
        )
    ),
    "canonical_cross_source_overlap_groups": int(
        len(
            canonical_cross_source_overlap
        )
    ),
    "cross_source_overlap_pass": (
        cross_source_overlap_pass
    ),
    "dataset_gate_pass": (
        dataset_gate_pass
    ),
    "split_gate_pass": (
        split_gate_pass
    ),
    "grade_gate_pass": (
        grade_gate_pass
    ),
    "target_sealed_label_files_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_original_images_modified": False,
    "staging_images_finalised": False,
}


with open(
    STAGE4F_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


REPORT_PATH.write_text(
    "\n".join(
        [
            (
                "# Stage 4F — Source Canonicalisation "
                "and Integrity Audit"
            ),
            "",
            f"Decision: `{stage4f_decision}`",
            "",
            interpretation,
            "",
            "## Safety boundary",
            "",
            (
                "- Target sealed labels accessed: "
                "`False`"
            ),
            (
                "- Source performance observed: "
                "`False`"
            ),
            (
                "- Transfer performance observed: "
                "`False`"
            ),
            (
                "- Source originals modified: "
                "`False`"
            ),
            (
                "- Canonical staging finalised: "
                "`False`"
            ),
        ]
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "pillow": PIL.__version__,
    "canonical_size": (
        CANONICAL_SIZE
    ),
    "total_source_images": (
        TOTAL_EXPECTED_IMAGES
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 14. Final display
# ============================================================

print(
    "\n================ SOURCE CANONICAL "
    "SUMMARY ================"
)

display(
    integrity_summary
)


print(
    "\n================ SOURCE DATASET "
    "RETENTION ================"
)

display(
    dataset_retention
)


print(
    "\n================ SOURCE SPLIT "
    "RETENTION ================"
)

display(
    split_retention
)


print(
    "\n================ SOURCE GRADE "
    "RETENTION ================"
)

display(
    grade_retention
)


print(
    "\n================ CROSS-SOURCE "
    "OVERLAP CHECK ================"
)

print(
    "Original-hash overlap groups:",
    len(
        original_cross_source_overlap
    ),
)

print(
    "Canonical-hash overlap groups:",
    len(
        canonical_cross_source_overlap
    ),
)

print(
    "Cross-source overlap gate passed:",
    cross_source_overlap_pass,
)


print(
    "\n================ STAGE 4F DECISION "
    "================"
)

print("Decision:")
print(
    stage4f_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nSource performance observed:")
print(False)

print("\nSource-target transfer performance observed:")
print(False)

print("\nSource originals modified:")
print(False)

print("\nCanonical staging finalised:")
print(False)

print(
    "\nStage 4F source canonicalisation "
    "and integrity audit completed."
)

================ STAGE 4F SOURCE CANONICALISATION ================

Imported Stage 3A decision:
PASS_BOTH_TARGETS_RECOVERABLE_ADVANCE_ALL_FOUR_EDGES

Imported Stage 4E-R2 decision:
PASS_DEEPDRID_EYE_LEVEL_LABEL_PROVENANCE_RESOLVED_ADVANCE_TO_SOURCE_CANONICALISATION

Frozen canonical protocol:
{
  "protocol_version": "v0.1",
  "canonical_size": 768,
  "black_border_threshold": 10,
  "jpeg_quality": 92,
  "jpeg_subsampling": 0,
  "resize_filter": "LANCZOS",
  "frozen_before_target_performance": true,
  "must_be_applied_identically_to_sources": [
    "EyePACS_2015",
    "DeepDRiD"
  ]
}

Frozen integrity rule:
Within each source dataset, construct connected components using exact original-image SHA-256 equality and exact canonical-image SHA-256 equality. Exclude an entire component if it contains more than one official eye-level DR grade or crosses the frozen source development/validation boundary. Retain label-consistent duplicate components contained wholly within one source partition, 

,dataset,sample_images,mean_bytes,median_bytes,maximum_bytes,estimated_total_gib
0,EyePACS_2015,40,105234.975,107943.0,135386,0.437898
1,DeepDRiD,40,88306.950,88678.5,111089,0.131588


Estimated source canonical total: 0.57 GiB
Estimated total with 30% safety: 0.74 GiB
Current Drive free space: 7.58 GiB


Canonicalising EyePACS_2015:   0%|          | 0/4468 [00:00<?, ?it/s]

Canonicalising DeepDRiD:   0%|          | 0/1600 [00:00<?, ?it/s]

...


================ SOURCE SPLIT RETENTION ================


,dataset,source_partition,original_images,excluded_images,retained_images,retained_fraction
0,DeepDRiD,development,1200,0,1200,1.0
1,DeepDRiD,validation,400,0,400,1.0
2,EyePACS_2015,development,3350,0,3350,1.0
3,EyePACS_2015,validation,1118,0,1118,1.0



================ SOURCE GRADE RETENTION ================


,dataset,dr_grade,original_images,excluded_images,retained_images
0,DeepDRiD,0,714,0,714
1,DeepDRiD,1,186,0,186
2,DeepDRiD,2,326,0,326
3,DeepDRiD,3,282,0,282
4,DeepDRiD,4,92,0,92
5,EyePACS_2015,0,1496,0,1496
6,EyePACS_2015,1,1239,0,1239
7,EyePACS_2015,2,1293,0,1293
8,EyePACS_2015,3,258,0,258
9,EyePACS_2015,4,182,0,182



================ CROSS-SOURCE OVERLAP CHECK ================
Original-hash overlap groups: 0
Canonical-hash overlap groups: 0
Cross-source overlap gate passed: True

================ STAGE 4F DECISION ================
Decision:
PASS_SOURCE_CANONICALISATION_AND_INTEGRITY_AUDIT_ADVANCE_TO_ELIGIBLE_SOURCE_FINALISATION

Interpretation:
Both precommitted source cohorts were canonicalised using the target-frozen 768x768 protocol. Exact original and canonical duplicate components were audited before source or transfer performance was observed. Components with label conflict or source-partition leakage are marked for deterministic exclusion. All frozen retention gates passed, and no exact overlap was detected between the two sources.

Authorised next step:
FINALISE_ELIGIBLE_CANONICAL_SOURCE_COHORTS

Target sealed-label files accessed:
False

Source performance observed:
False

Source-target transfer performance observed:
False

Source originals modified:
False

Canonical staging finalised:
Fa

In [36]:
#@title 04G. Finalise canonical source cohorts and seal Stage 4

from pathlib import Path
from datetime import datetime, timezone
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import sys

import numpy as np
import pandas as pd


# ============================================================
# 0. Frozen Stage 4G boundary
# ============================================================

EXPECTED_DATASET_COUNTS = {
    "EyePACS_2015": 4468,
    "DeepDRiD": 1600,
}

EXPECTED_PARTITION_COUNTS = {
    ("EyePACS_2015", "development"): 3350,
    ("EyePACS_2015", "validation"): 1118,
    ("DeepDRiD", "development"): 1200,
    ("DeepDRiD", "validation"): 400,
}

EXPECTED_TOTAL_IMAGES = 6068

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

FINALISATION_RULE = (
    "Finalise only images marked eligible by the sealed Stage 4F "
    "integrity audit. Verify every staging image against its committed "
    "canonical SHA-256 before finalisation, preserve the frozen source "
    "development/validation assignments, rename the verified staging "
    "tree to the final canonical source tree, and verify every final "
    "image again. Do not modify source originals or labels."
)

SAFETY_BOUNDARY = (
    "Stage 4G performs source-cohort finalisation only. It does not "
    "access target sealed-label files, extract model embeddings, train "
    "a model, calculate source performance, or calculate source-target "
    "transfer performance."
)


# ============================================================
# 1. Resolve Drive and project paths
# ============================================================

if Path("/content/drive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path("/content/gdrive/MyDrive").is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE4_ROOT = (
    TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4F_ROOT = (
    STAGE4_ROOT
    / "05_Source_Canonicalisation_And_Integrity_Audit"
)

STAGING_ROOT = (
    STAGE4_ROOT
    / "06_Canonical_Source_Images__STAGING"
)

FINAL_CANONICAL_ROOT = (
    STAGE4_ROOT
    / "06_Canonical_Source_Images"
)

STAGE4G_ROOT = (
    STAGE4_ROOT
    / "07_Source_Finalisation"
)

MANIFEST_ROOT = (
    STAGE4G_ROOT
    / "01_Final_Source_Manifests"
)

COMMITMENT_ROOT = (
    STAGE4G_ROOT
    / "02_Commitments"
)

REPORT_ROOT = (
    STAGE4G_ROOT
    / "03_Report"
)


for directory in [
    STAGE4G_ROOT,
    MANIFEST_ROOT,
    COMMITMENT_ROOT,
    REPORT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


STAGE4F_DECISION_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Integrity_Decision_v0.1.json"
)

STAGE4F_INTEGRITY_PATH = (
    STAGE4F_ROOT
    / "Stage4F_Source_Canonicalisation_Integrity_Manifest_v0.1.csv"
)

STAGE4G_DECISION_PATH = (
    STAGE4G_ROOT
    / "Stage4G_Source_Finalisation_Decision_v0.1.json"
)


for required_path in [
    STAGE4F_DECISION_PATH,
    STAGE4F_INTEGRITY_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


# ============================================================
# 2. Verify Stage 4F decision
# ============================================================

with open(
    STAGE4F_DECISION_PATH,
    "r",
    encoding="utf-8",
) as file:
    stage4f_decision = json.load(
        file
    )


assert (
    stage4f_decision["decision"]
    ==
    "PASS_SOURCE_CANONICALISATION_AND_INTEGRITY_AUDIT_ADVANCE_TO_ELIGIBLE_SOURCE_FINALISATION"
)


if STAGE4G_DECISION_PATH.is_file():
    try:
        with open(
            STAGE4G_DECISION_PATH,
            "r",
            encoding="utf-8",
        ) as file:
            previous_stage4g_decision = json.load(
                file
            )

        if (
            previous_stage4g_decision.get(
                "decision"
            )
            ==
            "PASS_CANONICAL_SOURCE_FINALISATION_ADVANCE_TO_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE"
        ):
            raise RuntimeError(
                "Stage 4G already has a successful sealed decision. "
                "Do not overwrite or repeat finalisation."
            )

    except RuntimeError:
        raise

    except Exception:
        pass


print(
    "================ STAGE 4G SOURCE "
    "FINALISATION ================"
)

print("\nImported Stage 4F decision:")
print(
    stage4f_decision["decision"]
)

print("\nFrozen finalisation rule:")
print(
    FINALISATION_RULE
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)


# ============================================================
# 3. Helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(path).open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def parse_boolean_series(
    series,
    description,
):
    if pd.api.types.is_bool_dtype(
        series
    ):
        return series.astype(bool)

    mapping = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
    }

    parsed = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
        .map(mapping)
    )

    assert parsed.notna().all(), (
        f"Could not parse Boolean values in "
        f"{description}."
    )

    return parsed.astype(bool)


def path_is_inside(
    path,
    parent,
):
    try:
        Path(path).resolve().relative_to(
            Path(parent).resolve()
        )

        return True

    except ValueError:
        return False


def table_sha256(
    dataframe,
    columns,
):
    ordered = (
        dataframe[
            columns
        ]
        .astype(str)
        .sort_values(
            columns
        )
        .reset_index(
            drop=True
        )
    )

    payload = "\n".join(
        ordered.apply(
            lambda row: "|".join(
                row.tolist()
            ),
            axis=1,
        ).tolist()
    )

    return hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()


# ============================================================
# 4. Load and verify Stage 4F integrity manifest
# ============================================================

integrity = pd.read_csv(
    STAGE4F_INTEGRITY_PATH,
    low_memory=False,
)


required_columns = {
    "dataset",
    "image_id",
    "patient_id",
    "eye_side",
    "source_partition",
    "dr_grade",
    "moderate_or_worse_dr",
    "source_image_path",
    "canonical_staging_path",
    "canonical_filename",
    "original_sha256",
    "canonical_sha256",
    "original_size_bytes",
    "canonical_size_bytes",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "duplicate_component_id",
    "excluded_due_to_label_conflict",
    "excluded_due_to_cross_partition",
    "excluded_by_integrity_rule",
    "eligible_after_integrity_audit",
}


missing_columns = (
    required_columns
    -
    set(
        integrity.columns
    )
)

assert not missing_columns, (
    "Stage 4F integrity manifest is missing columns: "
    f"{sorted(missing_columns)}"
)


for boolean_column in [
    "excluded_due_to_label_conflict",
    "excluded_due_to_cross_partition",
    "excluded_by_integrity_rule",
    "eligible_after_integrity_audit",
]:
    integrity[
        boolean_column
    ] = parse_boolean_series(
        integrity[
            boolean_column
        ],
        boolean_column,
    )


assert (
    len(integrity)
    ==
    EXPECTED_TOTAL_IMAGES
)

assert not integrity[
    "excluded_by_integrity_rule"
].any()

assert integrity[
    "eligible_after_integrity_audit"
].all()

assert (
    integrity[
        "duplicate_component_id"
    ].nunique()
    ==
    EXPECTED_TOTAL_IMAGES
), (
    "Stage 4F reported no multi-image components, "
    "but component IDs are not all unique."
)


for dataset_name, expected_count in (
    EXPECTED_DATASET_COUNTS.items()
):
    dataset_rows = integrity[
        integrity[
            "dataset"
        ]
        ==
        dataset_name
    ]

    assert (
        len(dataset_rows)
        ==
        expected_count
    )


for (
    dataset_name,
    partition_name,
), expected_count in (
    EXPECTED_PARTITION_COUNTS.items()
):
    partition_rows = integrity[
        (
            integrity[
                "dataset"
            ]
            ==
            dataset_name
        )
        &
        (
            integrity[
                "source_partition"
            ]
            ==
            partition_name
        )
    ]

    assert (
        len(partition_rows)
        ==
        expected_count
    )


assert integrity[
    "dr_grade"
].isin(
    [0, 1, 2, 3, 4]
).all()

assert integrity[
    "moderate_or_worse_dr"
].isin(
    [0, 1]
).all()

assert (
    integrity[
        "moderate_or_worse_dr"
    ]
    ==
    (
        integrity[
            "dr_grade"
        ]
        >=
        2
    ).astype(int)
).all()


print("\nEligible source images:")
print(
    len(integrity)
)

print("\nImages excluded by Stage 4F:")
print(
    int(
        integrity[
            "excluded_by_integrity_rule"
        ].sum()
    )
)


# ============================================================
# 5. Resolve fresh or partially completed finalisation state
# ============================================================

staging_exists = STAGING_ROOT.is_dir()
final_exists = FINAL_CANONICAL_ROOT.is_dir()


if staging_exists and final_exists:
    raise RuntimeError(
        "Both staging and final canonical roots exist. "
        "Do not delete either automatically; inspect the "
        "partial finalisation state first."
    )


if not staging_exists and not final_exists:
    raise RuntimeError(
        "Neither staging nor final canonical source root exists."
    )


if staging_exists:
    active_image_root = STAGING_ROOT
    finalisation_state = (
        "FRESH_STAGING_READY"
    )

else:
    active_image_root = FINAL_CANONICAL_ROOT
    finalisation_state = (
        "PARTIAL_RESTART_FINAL_ROOT_ALREADY_PRESENT"
    )


print("\nDetected finalisation state:")
print(
    finalisation_state
)

print("\nActive image root:")
print(
    active_image_root
)


# ============================================================
# 6. Verify every pre-finalisation canonical file
# ============================================================

integrity[
    "expected_active_path"
] = integrity.apply(
    lambda row: str(
        active_image_root
        /
        row[
            "dataset"
        ]
        /
        row[
            "canonical_filename"
        ]
    ),
    axis=1,
)


assert integrity[
    "expected_active_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all(), (
    "At least one canonical source image is missing "
    "from the active image root."
)


active_jpg_files = list(
    active_image_root.rglob(
        "*.jpg"
    )
)


assert (
    len(active_jpg_files)
    ==
    EXPECTED_TOTAL_IMAGES
), (
    f"Expected {EXPECTED_TOTAL_IMAGES} canonical JPG files, "
    f"found {len(active_jpg_files)}."
)


pre_finalisation_hash_mismatches = []


for _, row in tqdm(
    integrity.iterrows(),
    total=len(integrity),
    desc="Verifying canonical images before finalisation",
):
    active_path = Path(
        row[
            "expected_active_path"
        ]
    )

    observed_sha256 = sha256_file(
        active_path
    )

    if (
        observed_sha256
        !=
        row[
            "canonical_sha256"
        ]
    ):
        pre_finalisation_hash_mismatches.append(
            {
                "dataset": (
                    row[
                        "dataset"
                    ]
                ),
                "image_id": (
                    row[
                        "image_id"
                    ]
                ),
                "path": str(
                    active_path
                ),
                "expected_sha256": (
                    row[
                        "canonical_sha256"
                    ]
                ),
                "observed_sha256": (
                    observed_sha256
                ),
            }
        )


assert not pre_finalisation_hash_mismatches, (
    "Pre-finalisation SHA-256 verification failed."
)


print(
    "\nPre-finalisation hash mismatches:"
)

print(
    len(
        pre_finalisation_hash_mismatches
    )
)


# ============================================================
# 7. Atomically promote staging tree to final tree
# ============================================================

if staging_exists:
    STAGING_ROOT.rename(
        FINAL_CANONICAL_ROOT
    )


assert FINAL_CANONICAL_ROOT.is_dir()
assert not STAGING_ROOT.exists()


# ============================================================
# 8. Construct final immutable source manifest
# ============================================================

integrity[
    "canonical_image_path"
] = integrity.apply(
    lambda row: str(
        FINAL_CANONICAL_ROOT
        /
        row[
            "dataset"
        ]
        /
        row[
            "canonical_filename"
        ]
    ),
    axis=1,
)


assert integrity[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


final_manifest_columns = [
    "dataset",
    "image_id",
    "patient_id",
    "eye_side",
    "source_partition",
    "dr_grade",
    "moderate_or_worse_dr",
    "source_image_path",
    "canonical_image_path",
    "canonical_filename",
    "original_sha256",
    "canonical_sha256",
    "original_size_bytes",
    "canonical_size_bytes",
    "original_width",
    "original_height",
    "canonical_width",
    "canonical_height",
    "duplicate_component_id",
]


final_manifest = (
    integrity[
        final_manifest_columns
    ]
    .sort_values(
        [
            "dataset",
            "source_partition",
            "image_id",
        ]
    )
    .reset_index(
        drop=True
    )
)


assert (
    len(final_manifest)
    ==
    EXPECTED_TOTAL_IMAGES
)

assert not final_manifest.duplicated(
    subset=[
        "dataset",
        "image_id",
    ]
).any()


# ============================================================
# 9. Verify every final canonical file
# ============================================================

post_finalisation_hash_mismatches = []


for _, row in tqdm(
    final_manifest.iterrows(),
    total=len(final_manifest),
    desc="Verifying final canonical source images",
):
    final_path = Path(
        row[
            "canonical_image_path"
        ]
    )

    observed_sha256 = sha256_file(
        final_path
    )

    if (
        observed_sha256
        !=
        row[
            "canonical_sha256"
        ]
    ):
        post_finalisation_hash_mismatches.append(
            {
                "dataset": (
                    row[
                        "dataset"
                    ]
                ),
                "image_id": (
                    row[
                        "image_id"
                    ]
                ),
                "path": str(
                    final_path
                ),
                "expected_sha256": (
                    row[
                        "canonical_sha256"
                    ]
                ),
                "observed_sha256": (
                    observed_sha256
                ),
            }
        )


assert not post_finalisation_hash_mismatches, (
    "Post-finalisation SHA-256 verification failed."
)


final_jpg_files = list(
    FINAL_CANONICAL_ROOT.rglob(
        "*.jpg"
    )
)


assert (
    len(final_jpg_files)
    ==
    EXPECTED_TOTAL_IMAGES
)


print(
    "\nPost-finalisation hash mismatches:"
)

print(
    len(
        post_finalisation_hash_mismatches
    )
)


# ============================================================
# 10. Save master and split-specific manifests
# ============================================================

MASTER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "Stage4G_Final_Canonical_Source_Manifest_v0.1.csv"
)

LABEL_FREE_MANIFEST_PATH = (
    MANIFEST_ROOT
    / "Stage4G_Final_LabelFree_Source_Image_Manifest_v0.1.csv"
)

EYEPACS_DEVELOPMENT_PATH = (
    MANIFEST_ROOT
    / "Stage4G_EyePACS_Development_Source_Manifest_v0.1.csv"
)

EYEPACS_VALIDATION_PATH = (
    MANIFEST_ROOT
    / "Stage4G_EyePACS_Validation_Source_Manifest_v0.1.csv"
)

DEEPDRID_DEVELOPMENT_PATH = (
    MANIFEST_ROOT
    / "Stage4G_DeepDRiD_Development_Source_Manifest_v0.1.csv"
)

DEEPDRID_VALIDATION_PATH = (
    MANIFEST_ROOT
    / "Stage4G_DeepDRiD_Validation_Source_Manifest_v0.1.csv"
)


final_manifest.to_csv(
    MASTER_MANIFEST_PATH,
    index=False,
)


label_free_columns = [
    "dataset",
    "image_id",
    "patient_id",
    "eye_side",
    "source_partition",
    "canonical_image_path",
    "canonical_filename",
    "original_sha256",
    "canonical_sha256",
    "duplicate_component_id",
]


final_manifest[
    label_free_columns
].to_csv(
    LABEL_FREE_MANIFEST_PATH,
    index=False,
)


split_paths = {
    (
        "EyePACS_2015",
        "development",
    ): EYEPACS_DEVELOPMENT_PATH,

    (
        "EyePACS_2015",
        "validation",
    ): EYEPACS_VALIDATION_PATH,

    (
        "DeepDRiD",
        "development",
    ): DEEPDRID_DEVELOPMENT_PATH,

    (
        "DeepDRiD",
        "validation",
    ): DEEPDRID_VALIDATION_PATH,
}


for (
    dataset_name,
    partition_name,
), output_path in (
    split_paths.items()
):
    split_manifest = final_manifest[
        (
            final_manifest[
                "dataset"
            ]
            ==
            dataset_name
        )
        &
        (
            final_manifest[
                "source_partition"
            ]
            ==
            partition_name
        )
    ].copy()

    expected_count = (
        EXPECTED_PARTITION_COUNTS[
            (
                dataset_name,
                partition_name,
            )
        ]
    )

    assert (
        len(split_manifest)
        ==
        expected_count
    )

    split_manifest.to_csv(
        output_path,
        index=False,
    )


# ============================================================
# 11. Create final cohort summaries
# ============================================================

dataset_summary = (
    final_manifest
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        images=(
            "image_id",
            "size",
        ),
        patients=(
            "patient_id",
            "nunique",
        ),
        development_images=(
            "source_partition",
            lambda values: int(
                (
                    values
                    ==
                    "development"
                ).sum()
            ),
        ),
        validation_images=(
            "source_partition",
            lambda values: int(
                (
                    values
                    ==
                    "validation"
                ).sum()
            ),
        ),
        negative_images=(
            "moderate_or_worse_dr",
            lambda values: int(
                (
                    values
                    ==
                    0
                ).sum()
            ),
        ),
        positive_images=(
            "moderate_or_worse_dr",
            lambda values: int(
                (
                    values
                    ==
                    1
                ).sum()
            ),
        ),
        canonical_size_bytes=(
            "canonical_size_bytes",
            "sum",
        ),
        unique_original_hashes=(
            "original_sha256",
            "nunique",
        ),
        unique_canonical_hashes=(
            "canonical_sha256",
            "nunique",
        ),
        duplicate_components=(
            "duplicate_component_id",
            "nunique",
        ),
    )
)


dataset_summary[
    "canonical_size_gib"
] = (
    dataset_summary[
        "canonical_size_bytes"
    ]
    /
    (1024 ** 3)
)


grade_summary = (
    final_manifest
    .groupby(
        [
            "dataset",
            "dr_grade",
        ],
        as_index=False,
    )
    .agg(
        images=(
            "image_id",
            "size",
        ),
        development_images=(
            "source_partition",
            lambda values: int(
                (
                    values
                    ==
                    "development"
                ).sum()
            ),
        ),
        validation_images=(
            "source_partition",
            lambda values: int(
                (
                    values
                    ==
                    "validation"
                ).sum()
            ),
        ),
    )
)


hash_verification_summary = pd.DataFrame(
    [
        {
            "verification_stage": (
                "before_finalisation"
            ),
            "images_checked": (
                EXPECTED_TOTAL_IMAGES
            ),
            "hash_mismatches": int(
                len(
                    pre_finalisation_hash_mismatches
                )
            ),
        },
        {
            "verification_stage": (
                "after_finalisation"
            ),
            "images_checked": (
                EXPECTED_TOTAL_IMAGES
            ),
            "hash_mismatches": int(
                len(
                    post_finalisation_hash_mismatches
                )
            ),
        },
    ]
)


DATASET_SUMMARY_PATH = (
    REPORT_ROOT
    / "Stage4G_Final_Source_Dataset_Summary_v0.1.csv"
)

GRADE_SUMMARY_PATH = (
    REPORT_ROOT
    / "Stage4G_Final_Source_Grade_Summary_v0.1.csv"
)

HASH_SUMMARY_PATH = (
    REPORT_ROOT
    / "Stage4G_Final_Hash_Verification_Summary_v0.1.csv"
)


dataset_summary.to_csv(
    DATASET_SUMMARY_PATH,
    index=False,
)

grade_summary.to_csv(
    GRADE_SUMMARY_PATH,
    index=False,
)

hash_verification_summary.to_csv(
    HASH_SUMMARY_PATH,
    index=False,
)


# ============================================================
# 12. Seal commitments and decision
# ============================================================

manifest_inventory_sha256 = table_sha256(
    final_manifest,
    [
        "dataset",
        "image_id",
        "canonical_sha256",
    ],
)

label_mapping_sha256 = table_sha256(
    final_manifest,
    [
        "dataset",
        "image_id",
        "dr_grade",
        "moderate_or_worse_dr",
    ],
)


COMMITMENT_PATH = (
    COMMITMENT_ROOT
    / "Stage4G_Final_Source_Cohort_Commitments_v0.1.json"
)

REPORT_PATH = (
    REPORT_ROOT
    / "Stage4G_Source_Finalisation_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    REPORT_ROOT
    / "Stage4G_Source_Finalisation_Environment_v0.1.json"
)


commitment_payload = {
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "final_canonical_root": str(
        FINAL_CANONICAL_ROOT
    ),
    "total_images": (
        EXPECTED_TOTAL_IMAGES
    ),
    "dataset_counts": (
        EXPECTED_DATASET_COUNTS
    ),
    "partition_counts": {
        (
            f"{dataset_name}__"
            f"{partition_name}"
        ): count
        for (
            dataset_name,
            partition_name,
        ), count in (
            EXPECTED_PARTITION_COUNTS.items()
        )
    },
    "master_manifest_path": str(
        MASTER_MANIFEST_PATH
    ),
    "master_manifest_sha256": sha256_file(
        MASTER_MANIFEST_PATH
    ),
    "label_free_manifest_path": str(
        LABEL_FREE_MANIFEST_PATH
    ),
    "label_free_manifest_sha256": sha256_file(
        LABEL_FREE_MANIFEST_PATH
    ),
    "canonical_inventory_sha256": (
        manifest_inventory_sha256
    ),
    "source_label_mapping_sha256": (
        label_mapping_sha256
    ),
    "stage4f_decision_path": str(
        STAGE4F_DECISION_PATH
    ),
    "stage4f_decision_sha256": sha256_file(
        STAGE4F_DECISION_PATH
    ),
    "stage4f_integrity_manifest_path": str(
        STAGE4F_INTEGRITY_PATH
    ),
    "stage4f_integrity_manifest_sha256": sha256_file(
        STAGE4F_INTEGRITY_PATH
    ),
    "pre_finalisation_hash_mismatches": int(
        len(
            pre_finalisation_hash_mismatches
        )
    ),
    "post_finalisation_hash_mismatches": int(
        len(
            post_finalisation_hash_mismatches
        )
    ),
    "target_sealed_labels_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_original_images_modified": False,
}


with open(
    COMMITMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        commitment_payload,
        file,
        indent=2,
    )


stage4g_decision = (
    "PASS_CANONICAL_SOURCE_FINALISATION_"
    "ADVANCE_TO_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE"
)

authorised_next_step = (
    "SOURCE_DEVELOPMENT_RECOVERABILITY_"
    "AND_SOURCE_AXIS_FREEZE"
)

interpretation = (
    "All 6,068 precommitted source images were retained and promoted "
    "from verified staging storage to the final canonical source tree. "
    "Every file matched its committed canonical SHA-256 both before "
    "and after finalisation. The EyePACS and DeepDRiD development and "
    "validation assignments, eye-level DR labels, duplicate-component "
    "identities, and canonical image inventory are now frozen. No "
    "source-model or source-target transfer performance has been "
    "observed."
)


decision_payload = {
    "decision": (
        stage4g_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "finalisation_rule": (
        FINALISATION_RULE
    ),
    "final_canonical_root": str(
        FINAL_CANONICAL_ROOT
    ),
    "master_manifest_path": str(
        MASTER_MANIFEST_PATH
    ),
    "commitment_path": str(
        COMMITMENT_PATH
    ),
    "dataset_summary": (
        dataset_summary
        .to_dict(
            orient="records"
        )
    ),
    "grade_summary": (
        grade_summary
        .to_dict(
            orient="records"
        )
    ),
    "pre_finalisation_hash_mismatches": int(
        len(
            pre_finalisation_hash_mismatches
        )
    ),
    "post_finalisation_hash_mismatches": int(
        len(
            post_finalisation_hash_mismatches
        )
    ),
    "target_sealed_label_files_accessed": False,
    "source_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_original_images_modified": False,
    "canonical_staging_root_exists": bool(
        STAGING_ROOT.exists()
    ),
    "canonical_final_root_exists": bool(
        FINAL_CANONICAL_ROOT.exists()
    ),
}


with open(
    STAGE4G_DECISION_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


REPORT_PATH.write_text(
    "\n".join(
        [
            "# Stage 4G — Canonical Source Finalisation",
            "",
            f"Decision: `{stage4g_decision}`",
            "",
            interpretation,
            "",
            "## Final source cohort",
            "",
            "- EyePACS images: `4468`",
            "- DeepDRiD images: `1600`",
            "- Total images: `6068`",
            (
                "- Pre-finalisation hash "
                "mismatches: `0`"
            ),
            (
                "- Post-finalisation hash "
                "mismatches: `0`"
            ),
            "",
            "## Safety boundary",
            "",
            (
                "- Target sealed labels "
                "accessed: `False`"
            ),
            (
                "- Source performance "
                "observed: `False`"
            ),
            (
                "- Transfer performance "
                "observed: `False`"
            ),
            (
                "- Source originals "
                "modified: `False`"
            ),
        ]
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "final_canonical_root": str(
        FINAL_CANONICAL_ROOT
    ),
    "final_image_files": int(
        len(
            final_jpg_files
        )
    ),
}


with open(
    ENVIRONMENT_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 13. Final output
# ============================================================

print(
    "\n================ FINAL SOURCE "
    "COHORT SUMMARY ================"
)

display(
    dataset_summary[
        [
            "dataset",
            "images",
            "patients",
            "development_images",
            "validation_images",
            "negative_images",
            "positive_images",
            "unique_original_hashes",
            "unique_canonical_hashes",
            "canonical_size_gib",
        ]
    ]
)


print(
    "\n================ FINAL SOURCE "
    "GRADE SUMMARY ================"
)

display(
    grade_summary
)


print(
    "\n================ FINAL HASH "
    "VERIFICATION ================"
)

display(
    hash_verification_summary
)


print(
    "\n================ STAGE 4G "
    "DECISION ================"
)

print("Decision:")
print(
    stage4g_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nFinal canonical root:")
print(
    FINAL_CANONICAL_ROOT
)

print("\nStaging root still exists:")
print(
    STAGING_ROOT.exists()
)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nSource performance observed:")
print(False)

print("\nSource-target transfer performance observed:")
print(False)

print(
    "\nStage 4 canonical source acquisition, "
    "integrity audit and finalisation are complete."
)

================ STAGE 4G SOURCE FINALISATION ================

Imported Stage 4F decision:
PASS_SOURCE_CANONICALISATION_AND_INTEGRITY_AUDIT_ADVANCE_TO_ELIGIBLE_SOURCE_FINALISATION

Frozen finalisation rule:
Finalise only images marked eligible by the sealed Stage 4F integrity audit. Verify every staging image against its committed canonical SHA-256 before finalisation, preserve the frozen source development/validation assignments, rename the verified staging tree to the final canonical source tree, and verify every final image again. Do not modify source originals or labels.

Safety boundary:
Stage 4G performs source-cohort finalisation only. It does not access target sealed-label files, extract model embeddings, train a model, calculate source performance, or calculate source-target transfer performance.

Eligible source images:
6068

Images excluded by Stage 4F:
0

Detected finalisation state:
FRESH_STAGING_READY

Active image root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Obse

Verifying canonical images before finalisation:   0%|          | 0/6068 [00:00<?, ?it/s]


Pre-finalisation hash mismatches:
0


Verifying final canonical source images:   0%|          | 0/6068 [00:00<?, ?it/s]


Post-finalisation hash mismatches:
0

================ FINAL SOURCE COHORT SUMMARY ================


...


================ FINAL SOURCE GRADE SUMMARY ================


,dataset,dr_grade,images,development_images,validation_images
0,DeepDRiD,0,714,540,174
1,DeepDRiD,1,186,140,46
2,DeepDRiD,2,326,234,92
3,DeepDRiD,3,282,214,68
4,DeepDRiD,4,92,72,20
5,EyePACS_2015,0,1496,1123,373
6,EyePACS_2015,1,1239,938,301
7,EyePACS_2015,2,1293,964,329
8,EyePACS_2015,3,258,187,71
9,EyePACS_2015,4,182,138,44



================ FINAL HASH VERIFICATION ================


,verification_stage,images_checked,hash_mismatches
0,before_finalisation,6068,0
1,after_finalisation,6068,0



================ STAGE 4G DECISION ================
Decision:
PASS_CANONICAL_SOURCE_FINALISATION_ADVANCE_TO_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE

Interpretation:
All 6,068 precommitted source images were retained and promoted from verified staging storage to the final canonical source tree. Every file matched its committed canonical SHA-256 both before and after finalisation. The EyePACS and DeepDRiD development and validation assignments, eye-level DR labels, duplicate-component identities, and canonical image inventory are now frozen. No source-model or source-target transfer performance has been observed.

Authorised next step:
SOURCE_DEVELOPMENT_RECOVERABILITY_AND_SOURCE_AXIS_FREEZE

Final canonical root:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage4_Source_Canonicalisation_v0.1/06_Canonical_Source_Images

Staging root still exists:
False

Target sealed-label files accessed:
False

Source performa

In [40]:
#@title 05A-CPU. Source recoverability, held-out validation, and source-axis freeze

from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
)
from sklearn.exceptions import ConvergenceWarning
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import random
import re
import sys
import warnings

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


# ============================================================
# 0. Frozen Stage 5A protocol
# ============================================================

RANDOM_SEED = 20260720

N_FOLDS = 5
N_BOOTSTRAP = 2000

AUC_THRESHOLD = 0.70
CI_LOWER_THRESHOLD = 0.55

FROZEN_BACKBONE = "ResNet50_IMAGENET1K_V2"
FROZEN_FEATURE_DIMENSION = 2048
EMBEDDING_NORMALISATION = "L2"

LOGISTIC_C = 1.0
LOGISTIC_CLASS_WEIGHT = "balanced"
LOGISTIC_SOLVER = "liblinear"
LOGISTIC_MAX_ITER = 5000

ENDPOINT_ID = (
    "MODERATE_OR_WORSE_DR_GRADE_GE_2"
)

EXPECTED_DATASET_COUNTS = {
    "EyePACS_2015": 4468,
    "DeepDRiD": 1600,
}

EXPECTED_PARTITION_COUNTS = {
    ("EyePACS_2015", "development"): 3350,
    ("EyePACS_2015", "validation"): 1118,
    ("DeepDRiD", "development"): 1200,
    ("DeepDRiD", "validation"): 400,
}

EXPECTED_EYE_UNIT_COUNTS = {
    ("EyePACS_2015", "development"): 3350,
    ("EyePACS_2015", "validation"): 1118,
    ("DeepDRiD", "development"): 600,
    ("DeepDRiD", "validation"): 200,
}

SOURCE_PASS_RULE = (
    "A source passes only if its fixed frozen-representation probe "
    "passes both checks: (1) patient-grouped five-fold development "
    "out-of-fold eye-level ROC AUC >= 0.70 with patient-bootstrap "
    "95% CI lower bound > 0.55; and (2) the final axis fitted once "
    "on all development data achieves held-out validation eye-level "
    "ROC AUC >= 0.70 with patient-bootstrap 95% CI lower bound > 0.55. "
    "No architecture, C, class weight, threshold, preprocessing, or "
    "feature selection may be tuned."
)

AXIS_FREEZE_RULE = (
    "For each source, fit one StandardScaler plus class-balanced "
    "L2-regularised logistic regression on all frozen development "
    "embeddings, weighting images so each eye contributes total "
    "training weight one. Save the scaler, standardised coefficient, "
    "raw-embedding coefficient, and intercept before evaluating the "
    "held-out source validation partition. These exact parameters "
    "must later score all authorised target images without refitting."
)

SAFETY_BOUNDARY = (
    "Stage 5A uses only final canonical source images and source labels. "
    "It does not access target sealed-label files, target images, target "
    "performance, or source-target transfer performance. Source "
    "validation is evaluated only after the corresponding source axis "
    "has been written and hash-committed."
)

ImageFile.LOAD_TRUNCATED_IMAGES = False


# ============================================================
# 1. Reproducibility and CPU configuration
# ============================================================

os.environ["PYTHONHASHSEED"] = str(
    RANDOM_SEED
)

random.seed(
    RANDOM_SEED
)

np.random.seed(
    RANDOM_SEED
)

torch.manual_seed(
    RANDOM_SEED
)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

warnings.filterwarnings(
    "ignore",
    category=ConvergenceWarning,
)


CPU_THREADS = max(
    1,
    min(
        8,
        os.cpu_count() or 2,
    ),
)

torch.set_num_threads(
    CPU_THREADS
)

device = torch.device(
    "cpu"
)

BATCH_SIZE = 16
NUM_WORKERS = min(
    2,
    max(
        0,
        (os.cpu_count() or 2) - 1,
    ),
)

CHECKPOINT_EVERY_BATCHES = 10


# ============================================================
# 2. Resolve Drive and project paths
# ============================================================

if Path(
    "/content/drive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )

elif Path(
    "/content/gdrive/MyDrive"
).is_dir():
    DRIVE_ROOT = Path(
        "/content/gdrive/MyDrive"
    )

else:
    from google.colab import drive

    drive.mount(
        "/content/drive"
    )

    DRIVE_ROOT = Path(
        "/content/drive/MyDrive"
    )


PROJECT_ROOT = (
    DRIVE_ROOT
    / "Cross-Modal_Diagnostic_Observability"
)

RETINAL_DATA_ROOT = (
    PROJECT_ROOT
    / "06_Data_Records"
    / "Retinal_DR"
)

TEST_ROOT = (
    RETINAL_DATA_ROOT
    / "Prospective_Retinal_Blind_Test_v0.1"
)

STAGE4_ROOT = (
    TEST_ROOT
    / "Stage4_Source_Canonicalisation_v0.1"
)

STAGE4G_ROOT = (
    STAGE4_ROOT
    / "07_Source_Finalisation"
)

STAGE4G_DECISION_PATH = (
    STAGE4G_ROOT
    / "Stage4G_Source_Finalisation_Decision_v0.1.json"
)

SOURCE_MANIFEST_PATH = (
    STAGE4G_ROOT
    / "01_Final_Source_Manifests"
    / "Stage4G_Final_Canonical_Source_Manifest_v0.1.csv"
)

STAGE5_ROOT = (
    TEST_ROOT
    / "Stage5_Source_Recoverability_And_Axis_Freeze_v0.1"
)

PROTOCOL_ROOT = (
    STAGE5_ROOT
    / "00_Protocol"
)

EMBEDDING_ROOT = (
    STAGE5_ROOT
    / "01_Frozen_Embeddings"
)

AXIS_ROOT = (
    STAGE5_ROOT
    / "02_Frozen_Source_Axes"
)

RESULT_ROOT = (
    STAGE5_ROOT
    / "03_Results"
)


for directory in [
    PROTOCOL_ROOT,
    EMBEDDING_ROOT,
    AXIS_ROOT,
    RESULT_ROOT,
]:
    directory.mkdir(
        parents=True,
        exist_ok=True,
    )


STAGE5_DECISION_PATH = (
    RESULT_ROOT
    / "Stage5A_Source_Recoverability_And_Axis_Freeze_Decision_v0.1.json"
)


for required_path in [
    STAGE4G_DECISION_PATH,
    SOURCE_MANIFEST_PATH,
]:
    assert required_path.is_file(), (
        required_path
    )


with STAGE4G_DECISION_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    stage4g_decision = json.load(
        file
    )


assert (
    stage4g_decision[
        "decision"
    ]
    ==
    "PASS_CANONICAL_SOURCE_FINALISATION_ADVANCE_TO_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE"
)


if STAGE5_DECISION_PATH.is_file():
    try:
        with STAGE5_DECISION_PATH.open(
            "r",
            encoding="utf-8",
        ) as file:
            previous_stage5 = json.load(
                file
            )

        if previous_stage5.get(
            "decision"
        ) in {
            "PASS_BOTH_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE_ADVANCE_ALL_FOUR_EDGES_TO_BLIND_TRANSFER",
            "PARTIAL_PASS_RETAIN_ONLY_EDGES_FROM_RECOVERABLE_SOURCE",
            "FAIL_SOURCE_RECOVERABILITY_RETIRE_PROSPECTIVE_TEST",
        }:
            raise RuntimeError(
                "Stage 5A already has a sealed decision. "
                "Do not overwrite or rerun it."
            )

    except RuntimeError:
        raise

    except Exception:
        pass


print(
    "================ STAGE 5A-CPU SOURCE "
    "RECOVERABILITY AND AXIS FREEZE ================"
)

print("\nImported Stage 4G decision:")
print(
    stage4g_decision[
        "decision"
    ]
)

print("\nExecution device:")
print(
    device
)

print("\nCPU threads:")
print(
    CPU_THREADS
)

print("\nEmbedding batch size:")
print(
    BATCH_SIZE
)

print("\nDataLoader workers:")
print(
    NUM_WORKERS
)

print("\nFrozen source pass rule:")
print(
    SOURCE_PASS_RULE
)

print("\nFrozen axis rule:")
print(
    AXIS_FREEZE_RULE
)

print("\nSafety boundary:")
print(
    SAFETY_BOUNDARY
)


# ============================================================
# 3. General helpers
# ============================================================

def sha256_file(
    path,
    chunk_size=1024 * 1024,
):
    digest = hashlib.sha256()

    with Path(
        path
    ).open(
        "rb"
    ) as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def stable_table_sha256(
    dataframe,
    columns,
):
    ordered = (
        dataframe[
            columns
        ]
        .astype(str)
        .sort_values(
            columns
        )
        .reset_index(
            drop=True
        )
    )

    payload = "\n".join(
        ordered.apply(
            lambda row: "|".join(
                row.tolist()
            ),
            axis=1,
        ).tolist()
    )

    return hashlib.sha256(
        payload.encode(
            "utf-8"
        )
    ).hexdigest()


def normalise_eye_side(
    value,
):
    if pd.isna(
        value
    ):
        return ""

    key = re.sub(
        r"[^a-z0-9]+",
        "",
        str(
            value
        ).lower(),
    )

    if key in {
        "left",
        "l",
        "lefteye",
        "os",
    }:
        return "left"

    if key in {
        "right",
        "r",
        "righteye",
        "od",
    }:
        return "right"

    return ""


def eye_side_from_image_id(
    image_id,
):
    text = str(
        image_id
    ).lower()

    left_patterns = [
        r"(?:^|[_\-\s])left(?:$|[_\-\s])",
        r"(?:^|[_\-\s])l(?:$|[_\-\s])",
    ]

    right_patterns = [
        r"(?:^|[_\-\s])right(?:$|[_\-\s])",
        r"(?:^|[_\-\s])r(?:$|[_\-\s])",
    ]

    if any(
        re.search(
            pattern,
            text,
        )
        for pattern in left_patterns
    ):
        return "left"

    if any(
        re.search(
            pattern,
            text,
        )
        for pattern in right_patterns
    ):
        return "right"

    return ""


def build_eye_unit_id(
    row,
):
    patient_id = str(
        row[
            "patient_id"
        ]
    )

    image_id = str(
        row[
            "image_id"
        ]
    )

    side = normalise_eye_side(
        row[
            "eye_side"
        ]
    )

    if not side:
        side = eye_side_from_image_id(
            image_id
        )

    if side:
        return (
            str(
                row[
                    "dataset"
                ]
            )
            +
            "__PATIENT_"
            +
            patient_id
            +
            "__EYE_"
            +
            side
        )

    return (
        str(
            row[
                "dataset"
            ]
        )
        +
        "__IMAGE_"
        +
        image_id
    )


def create_probe():
    return Pipeline(
        steps=[
            (
                "scaler",
                StandardScaler(),
            ),
            (
                "logisticregression",
                LogisticRegression(
                    C=LOGISTIC_C,
                    class_weight=(
                        LOGISTIC_CLASS_WEIGHT
                    ),
                    solver=(
                        LOGISTIC_SOLVER
                    ),
                    max_iter=(
                        LOGISTIC_MAX_ITER
                    ),
                    random_state=(
                        RANDOM_SEED
                    ),
                ),
            ),
        ]
    )


def eye_balanced_sample_weight(
    eye_unit_ids,
):
    eye_unit_ids = pd.Series(
        eye_unit_ids,
        dtype=str,
    )

    counts = eye_unit_ids.value_counts()

    eye_sizes = (
        eye_unit_ids
        .map(
            counts
        )
        .to_numpy(
            dtype=np.float64
        )
    )

    weights = (
        1.0
        /
        eye_sizes
    )

    assert np.isfinite(
        weights
    ).all()

    assert (
        weights
        >
        0
    ).all()

    return weights


def aggregate_to_eye(
    prediction_table,
):
    eye_table = (
        prediction_table
        .groupby(
            "eye_unit_id",
            as_index=False,
        )
        .agg(
            dataset=(
                "dataset",
                "first",
            ),
            patient_id=(
                "patient_id",
                "first",
            ),
            source_partition=(
                "source_partition",
                "first",
            ),
            moderate_or_worse_dr=(
                "moderate_or_worse_dr",
                "first",
            ),
            label_count=(
                "moderate_or_worse_dr",
                "nunique",
            ),
            probability=(
                "probability",
                "mean",
            ),
            images=(
                "image_id",
                "size",
            ),
            fold_count=(
                "fold",
                "nunique",
            ),
            fold=(
                "fold",
                "first",
            ),
        )
    )

    assert (
        eye_table[
            "label_count"
        ]
        ==
        1
    ).all()

    assert (
        eye_table[
            "fold_count"
        ]
        ==
        1
    ).all()

    return eye_table.drop(
        columns=[
            "label_count",
            "fold_count",
        ]
    )


def patient_cluster_bootstrap_auc(
    eye_table,
    n_bootstrap,
    seed,
):
    eye_table = (
        eye_table
        .reset_index(
            drop=True
        )
        .copy()
    )

    patient_values = (
        eye_table[
            "patient_id"
        ]
        .astype(str)
    )

    patients = (
        patient_values
        .drop_duplicates()
        .to_numpy()
    )

    patient_to_indices = {
        patient_id: eye_table.index[
            patient_values
            ==
            patient_id
        ].to_numpy()
        for patient_id in patients
    }

    rng = np.random.default_rng(
        seed
    )

    bootstrap_values = []

    for _ in range(
        n_bootstrap
    ):
        sampled_patients = rng.choice(
            patients,
            size=len(
                patients
            ),
            replace=True,
        )

        sampled_indices = np.concatenate(
            [
                patient_to_indices[
                    patient_id
                ]
                for patient_id in (
                    sampled_patients
                )
            ]
        )

        sampled = eye_table.iloc[
            sampled_indices
        ]

        labels = sampled[
            "moderate_or_worse_dr"
        ].to_numpy(
            dtype=np.int64
        )

        if np.unique(
            labels
        ).size < 2:
            continue

        probabilities = sampled[
            "probability"
        ].to_numpy(
            dtype=np.float64
        )

        bootstrap_values.append(
            roc_auc_score(
                labels,
                probabilities,
            )
        )

    assert len(
        bootstrap_values
    ) >= int(
        n_bootstrap
        *
        0.90
    )

    lower, upper = np.quantile(
        bootstrap_values,
        [
            0.025,
            0.975,
        ],
    )

    return (
        float(
            lower
        ),
        float(
            upper
        ),
        int(
            len(
                bootstrap_values
            )
        ),
    )


def metric_record(
    eye_table,
    bootstrap_seed,
):
    labels = eye_table[
        "moderate_or_worse_dr"
    ].to_numpy(
        dtype=np.int64
    )

    probabilities = eye_table[
        "probability"
    ].to_numpy(
        dtype=np.float64
    )

    auc = float(
        roc_auc_score(
            labels,
            probabilities,
        )
    )

    (
        ci_lower,
        ci_upper,
        valid_bootstraps,
    ) = patient_cluster_bootstrap_auc(
        eye_table=eye_table,
        n_bootstrap=N_BOOTSTRAP,
        seed=bootstrap_seed,
    )

    return {
        "eye_auc": (
            auc
        ),
        "eye_auc_ci_lower_95": (
            ci_lower
        ),
        "eye_auc_ci_upper_95": (
            ci_upper
        ),
        "eye_average_precision": float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        "eye_balanced_accuracy_at_0_5": float(
            balanced_accuracy_score(
                labels,
                (
                    probabilities
                    >=
                    0.5
                ).astype(
                    int
                ),
            )
        ),
        "eye_brier_score": float(
            brier_score_loss(
                labels,
                probabilities,
            )
        ),
        "valid_bootstrap_samples": (
            valid_bootstraps
        ),
    }


def save_axis(
    probe,
    dataset_name,
    development_manifest_sha256,
):
    scaler = probe.named_steps[
        "scaler"
    ]

    classifier = probe.named_steps[
        "logisticregression"
    ]

    scaler_mean = (
        scaler.mean_
        .astype(
            np.float64
        )
    )

    scaler_scale = (
        scaler.scale_
        .astype(
            np.float64
        )
    )

    coefficient_standardised = (
        classifier.coef_[
            0
        ]
        .astype(
            np.float64
        )
    )

    intercept_standardised = float(
        classifier.intercept_[
            0
        ]
    )

    coefficient_raw = (
        coefficient_standardised
        /
        scaler_scale
    )

    intercept_raw = float(
        intercept_standardised
        -
        np.dot(
            coefficient_standardised,
            scaler_mean
            /
            scaler_scale,
        )
    )

    safe_name = dataset_name.replace(
        " ",
        "_",
    )

    axis_npz_path = (
        AXIS_ROOT
        /
        f"{safe_name}_Frozen_Source_Axis_v0.1.npz"
    )

    axis_json_path = (
        AXIS_ROOT
        /
        f"{safe_name}_Frozen_Source_Axis_Metadata_v0.1.json"
    )

    np.savez_compressed(
        axis_npz_path,
        scaler_mean=(
            scaler_mean
        ),
        scaler_scale=(
            scaler_scale
        ),
        coefficient_standardised=(
            coefficient_standardised
        ),
        intercept_standardised=np.asarray(
            [
                intercept_standardised
            ],
            dtype=np.float64,
        ),
        coefficient_raw=(
            coefficient_raw
        ),
        intercept_raw=np.asarray(
            [
                intercept_raw
            ],
            dtype=np.float64,
        ),
    )

    axis_payload = {
        "dataset": (
            dataset_name
        ),
        "endpoint_id": (
            ENDPOINT_ID
        ),
        "backbone": (
            FROZEN_BACKBONE
        ),
        "embedding_dimension": (
            FROZEN_FEATURE_DIMENSION
        ),
        "embedding_normalisation": (
            EMBEDDING_NORMALISATION
        ),
        "probe": {
            "scaler": (
                "StandardScaler"
            ),
            "classifier": (
                "LogisticRegression"
            ),
            "C": (
                LOGISTIC_C
            ),
            "class_weight": (
                LOGISTIC_CLASS_WEIGHT
            ),
            "solver": (
                LOGISTIC_SOLVER
            ),
            "max_iter": (
                LOGISTIC_MAX_ITER
            ),
        },
        "training_partition": (
            "development"
        ),
        "training_weighting": (
            "each eye contributes "
            "total sample weight one"
        ),
        "development_manifest_sha256": (
            development_manifest_sha256
        ),
        "axis_npz_path": str(
            axis_npz_path
        ),
        "axis_npz_sha256": sha256_file(
            axis_npz_path
        ),
        "raw_axis_l2_norm": float(
            np.linalg.norm(
                coefficient_raw
            )
        ),
        "standardised_axis_l2_norm": float(
            np.linalg.norm(
                coefficient_standardised
            )
        ),
        "intercept_raw": (
            intercept_raw
        ),
        "axis_frozen_before_source_validation_performance": True,
        "axis_frozen_before_target_image_access": True,
        "axis_frozen_before_target_performance": True,
        "execution_device": (
            "cpu"
        ),
    }

    with axis_json_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            axis_payload,
            file,
            indent=2,
        )

    return {
        "axis_npz_path": (
            axis_npz_path
        ),
        "axis_json_path": (
            axis_json_path
        ),
        "axis_npz_sha256": sha256_file(
            axis_npz_path
        ),
        "axis_json_sha256": sha256_file(
            axis_json_path
        ),
        "coefficient_raw": (
            coefficient_raw
        ),
        "intercept_raw": (
            intercept_raw
        ),
        "raw_axis_l2_norm": float(
            np.linalg.norm(
                coefficient_raw
            )
        ),
    }


# ============================================================
# 4. Freeze protocol before embeddings or performance
# ============================================================

PROTOCOL_PATH = (
    PROTOCOL_ROOT
    / "Stage5A_Source_Recoverability_And_Axis_Freeze_Protocol_v0.1.json"
)

INPUT_COMMITMENT_PATH = (
    PROTOCOL_ROOT
    / "Stage5A_Source_Input_Commitments_v0.1.json"
)


protocol_payload = {
    "protocol_version": (
        "v0.1"
    ),
    "stage": (
        "SOURCE_DEVELOPMENT_RECOVERABILITY_"
        "HELDOUT_VALIDATION_AND_AXIS_FREEZE"
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "source_pass_rule": (
        SOURCE_PASS_RULE
    ),
    "axis_freeze_rule": (
        AXIS_FREEZE_RULE
    ),
    "safety_boundary": (
        SAFETY_BOUNDARY
    ),
    "sources": [
        "EyePACS_2015",
        "DeepDRiD",
    ],
    "backbone": (
        FROZEN_BACKBONE
    ),
    "backbone_trainable": (
        False
    ),
    "embedding_dimension": (
        FROZEN_FEATURE_DIMENSION
    ),
    "embedding_normalisation": (
        EMBEDDING_NORMALISATION
    ),
    "inference_transform": (
        "ResNet50 IMAGENET1K_V2 "
        "official inference transform"
    ),
    "execution_device": (
        "cpu"
    ),
    "cpu_threads": (
        CPU_THREADS
    ),
    "batch_size": (
        BATCH_SIZE
    ),
    "embedding_checkpointing": {
        "enabled": True,
        "checkpoint_every_batches": (
            CHECKPOINT_EVERY_BATCHES
        ),
        "resume_after_runtime_interruption": True,
    },
    "development_validation": {
        "method": (
            "StratifiedGroupKFold"
        ),
        "groups": (
            "patient_id"
        ),
        "folds": (
            N_FOLDS
        ),
        "prediction_aggregation": (
            "mean probability per eye"
        ),
        "bootstrap_cluster": (
            "patient_id"
        ),
        "bootstrap_iterations": (
            N_BOOTSTRAP
        ),
    },
    "heldout_source_validation": {
        "training_partition": (
            "development"
        ),
        "evaluation_partition": (
            "validation"
        ),
        "prediction_aggregation": (
            "mean probability per eye"
        ),
        "bootstrap_cluster": (
            "patient_id"
        ),
        "bootstrap_iterations": (
            N_BOOTSTRAP
        ),
    },
    "probe": {
        "scaler": (
            "StandardScaler"
        ),
        "classifier": (
            "LogisticRegression"
        ),
        "C": (
            LOGISTIC_C
        ),
        "class_weight": (
            LOGISTIC_CLASS_WEIGHT
        ),
        "solver": (
            LOGISTIC_SOLVER
        ),
        "max_iter": (
            LOGISTIC_MAX_ITER
        ),
        "hyperparameter_tuning": (
            False
        ),
    },
    "pass_thresholds": {
        "auc_minimum": (
            AUC_THRESHOLD
        ),
        "ci_lower_strictly_greater_than": (
            CI_LOWER_THRESHOLD
        ),
        "development_and_validation_both_required": True,
    },
    "random_seed": (
        RANDOM_SEED
    ),
    "protocol_frozen_before_embeddings": True,
    "protocol_frozen_before_source_performance": True,
    "protocol_frozen_before_target_image_access": True,
    "protocol_frozen_before_target_performance": True,
}


with PROTOCOL_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        protocol_payload,
        file,
        indent=2,
    )


input_commitments = {
    "stage4g_decision_path": str(
        STAGE4G_DECISION_PATH
    ),
    "stage4g_decision_sha256": sha256_file(
        STAGE4G_DECISION_PATH
    ),
    "source_manifest_path": str(
        SOURCE_MANIFEST_PATH
    ),
    "source_manifest_sha256": sha256_file(
        SOURCE_MANIFEST_PATH
    ),
    "protocol_path": str(
        PROTOCOL_PATH
    ),
    "protocol_sha256": sha256_file(
        PROTOCOL_PATH
    ),
    "target_images_accessed": False,
    "target_sealed_label_files_accessed": False,
}


with INPUT_COMMITMENT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        input_commitments,
        file,
        indent=2,
    )


# ============================================================
# 5. Load and audit final source manifest
# ============================================================

source_manifest = pd.read_csv(
    SOURCE_MANIFEST_PATH,
    low_memory=False,
)


required_columns = {
    "dataset",
    "image_id",
    "patient_id",
    "eye_side",
    "source_partition",
    "dr_grade",
    "moderate_or_worse_dr",
    "canonical_image_path",
    "canonical_sha256",
    "duplicate_component_id",
}


missing_columns = (
    required_columns
    -
    set(
        source_manifest.columns
    )
)


assert not missing_columns, (
    "Source manifest missing columns: "
    f"{sorted(missing_columns)}"
)


assert (
    len(
        source_manifest
    )
    ==
    sum(
        EXPECTED_DATASET_COUNTS.values()
    )
)


assert not source_manifest.duplicated(
    subset=[
        "dataset",
        "image_id",
    ]
).any()


source_manifest[
    "patient_id"
] = source_manifest[
    "patient_id"
].astype(str)


source_manifest[
    "eye_unit_id"
] = source_manifest.apply(
    build_eye_unit_id,
    axis=1,
)


for dataset_name, expected_count in (
    EXPECTED_DATASET_COUNTS.items()
):
    dataset_frame = source_manifest[
        source_manifest[
            "dataset"
        ]
        ==
        dataset_name
    ]

    assert (
        len(
            dataset_frame
        )
        ==
        expected_count
    )


for (
    dataset_name,
    partition_name,
), expected_count in (
    EXPECTED_PARTITION_COUNTS.items()
):
    partition_frame = source_manifest[
        (
            source_manifest[
                "dataset"
            ]
            ==
            dataset_name
        )
        &
        (
            source_manifest[
                "source_partition"
            ]
            ==
            partition_name
        )
    ]

    assert (
        len(
            partition_frame
        )
        ==
        expected_count
    )

    expected_eye_units = (
        EXPECTED_EYE_UNIT_COUNTS[
            (
                dataset_name,
                partition_name,
            )
        ]
    )

    assert (
        partition_frame[
            "eye_unit_id"
        ].nunique()
        ==
        expected_eye_units
    ), (
        f"{dataset_name} {partition_name}: "
        "unexpected eye-unit count."
    )


assert source_manifest[
    "canonical_image_path"
].map(
    lambda value: Path(
        value
    ).is_file()
).all()


assert source_manifest[
    "dr_grade"
].isin(
    [
        0,
        1,
        2,
        3,
        4,
    ]
).all()


assert (
    source_manifest[
        "moderate_or_worse_dr"
    ]
    ==
    (
        source_manifest[
            "dr_grade"
        ]
        >=
        2
    ).astype(int)
).all()


eye_label_counts = (
    source_manifest
    .groupby(
        [
            "dataset",
            "eye_unit_id",
        ]
    )[
        "moderate_or_worse_dr"
    ]
    .nunique()
)


assert not (
    eye_label_counts
    >
    1
).any()


for dataset_name in (
    EXPECTED_DATASET_COUNTS
):
    dataset_frame = source_manifest[
        source_manifest[
            "dataset"
        ]
        ==
        dataset_name
    ]

    development_patients = set(
        dataset_frame.loc[
            dataset_frame[
                "source_partition"
            ]
            ==
            "development",
            "patient_id",
        ].astype(str)
    )

    validation_patients = set(
        dataset_frame.loc[
            dataset_frame[
                "source_partition"
            ]
            ==
            "validation",
            "patient_id",
        ].astype(str)
    )

    assert not development_patients.intersection(
        validation_patients
    ), (
        f"{dataset_name}: patient overlap "
        "across development and validation."
    )


representation_inventory = (
    source_manifest
    .groupby(
        [
            "dataset",
            "source_partition",
        ],
        as_index=False,
    )
    .agg(
        images=(
            "image_id",
            "size",
        ),
        eyes=(
            "eye_unit_id",
            "nunique",
        ),
        patients=(
            "patient_id",
            "nunique",
        ),
        negative_images=(
            "moderate_or_worse_dr",
            lambda values: int(
                (
                    values
                    ==
                    0
                ).sum()
            ),
        ),
        positive_images=(
            "moderate_or_worse_dr",
            lambda values: int(
                (
                    values
                    ==
                    1
                ).sum()
            ),
        ),
    )
)


print(
    "\n================ SOURCE REPRESENTATION "
    "INVENTORY ================"
)

display(
    representation_inventory
)


# ============================================================
# 6. Frozen ResNet50 model
# ============================================================

weights = (
    ResNet50_Weights
    .IMAGENET1K_V2
)

inference_transform = weights.transforms()


model = resnet50(
    weights=weights
)

model.fc = nn.Identity()

model.eval()


for parameter in model.parameters():
    parameter.requires_grad = False


model = model.to(
    device
)


# ============================================================
# 7. Resumable CPU embedding extraction
# ============================================================

class CanonicalSourceDataset(
    Dataset
):
    def __init__(
        self,
        dataframe,
        global_indices,
    ):
        self.dataframe = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.global_indices = np.asarray(
            global_indices,
            dtype=np.int64,
        )

    def __len__(
        self,
    ):
        return len(
            self.global_indices
        )

    def __getitem__(
        self,
        local_index,
    ):
        global_index = int(
            self.global_indices[
                local_index
            ]
        )

        row = self.dataframe.iloc[
            global_index
        ]

        image_path = Path(
            row[
                "canonical_image_path"
            ]
        )

        with Image.open(
            image_path
        ) as image:
            image.load()

            assert image.size == (
                768,
                768,
            )

            image = image.convert(
                "RGB"
            )

            tensor = inference_transform(
                image
            )

        return (
            tensor,
            global_index,
        )


def remove_embedding_checkpoint(
    paths,
):
    for path in paths:
        path = Path(
            path
        )

        if path.exists():
            path.unlink()


def load_or_extract_embeddings(
    dataframe,
    dataset_name,
):
    ordered = (
        dataframe
        .sort_values(
            [
                "source_partition",
                "image_id",
            ]
        )
        .reset_index(
            drop=True
        )
        .copy()
    )

    manifest_commitment = stable_table_sha256(
        ordered,
        [
            "dataset",
            "image_id",
            "canonical_sha256",
        ],
    )

    safe_name = dataset_name.replace(
        " ",
        "_",
    )

    embedding_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy"
    )

    completion_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_Completion_v0.1.npy"
    )

    image_id_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy"
    )

    metadata_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_Embeddings_Metadata_v0.1.json"
    )

    checkpoint_paths = [
        embedding_path,
        completion_path,
        image_id_path,
        metadata_path,
    ]

    expected_shape = (
        len(
            ordered
        ),
        FROZEN_FEATURE_DIMENSION,
    )

    resume_ok = False

    if all(
        path.is_file()
        for path in [
            embedding_path,
            completion_path,
            image_id_path,
            metadata_path,
        ]
    ):
        try:
            with metadata_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                metadata = json.load(
                    file
                )

            stored_ids = np.load(
                image_id_path,
                allow_pickle=False,
            ).astype(str)

            completed = np.load(
                completion_path,
                allow_pickle=False,
            ).astype(bool)

            stored_embeddings = np.load(
                embedding_path,
                mmap_mode="r",
                allow_pickle=False,
            )

            resume_ok = bool(
                metadata.get(
                    "manifest_commitment_sha256"
                )
                ==
                manifest_commitment
                and
                metadata.get(
                    "backbone"
                )
                ==
                FROZEN_BACKBONE
                and
                stored_embeddings.shape
                ==
                expected_shape
                and
                completed.shape
                ==
                (
                    len(
                        ordered
                    ),
                )
                and
                len(
                    stored_ids
                )
                ==
                len(
                    ordered
                )
                and
                np.array_equal(
                    stored_ids,
                    ordered[
                        "image_id"
                    ].astype(str).to_numpy(),
                )
            )

            del stored_embeddings

        except Exception:
            resume_ok = False

    if not resume_ok:
        remove_embedding_checkpoint(
            checkpoint_paths
        )

        embedding_memmap = (
            np.lib.format.open_memmap(
                embedding_path,
                mode="w+",
                dtype=np.float32,
                shape=expected_shape,
            )
        )

        embedding_memmap[:] = 0.0
        embedding_memmap.flush()

        completed = np.zeros(
            len(
                ordered
            ),
            dtype=bool,
        )

        np.save(
            completion_path,
            completed,
            allow_pickle=False,
        )

        np.save(
            image_id_path,
            ordered[
                "image_id"
            ].astype(str).to_numpy(),
            allow_pickle=False,
        )

        metadata_payload = {
            "dataset": (
                dataset_name
            ),
            "status": (
                "IN_PROGRESS"
            ),
            "images": int(
                len(
                    ordered
                )
            ),
            "completed_images": (
                0
            ),
            "embedding_shape": list(
                expected_shape
            ),
            "dtype": (
                "float32"
            ),
            "backbone": (
                FROZEN_BACKBONE
            ),
            "embedding_normalisation": (
                EMBEDDING_NORMALISATION
            ),
            "manifest_commitment_sha256": (
                manifest_commitment
            ),
            "embedding_path": str(
                embedding_path
            ),
            "completion_path": str(
                completion_path
            ),
            "image_id_path": str(
                image_id_path
            ),
            "execution_device": (
                "cpu"
            ),
            "target_images_accessed": False,
            "target_sealed_labels_accessed": False,
        }

        with metadata_path.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                metadata_payload,
                file,
                indent=2,
            )

        del embedding_memmap

    completed = np.load(
        completion_path,
        allow_pickle=False,
    ).astype(bool)

    missing_indices = np.flatnonzero(
        ~completed
    )

    print(
        f"\n{dataset_name}: completed embeddings "
        f"{int(completed.sum())}/{len(completed)}"
    )

    if len(
        missing_indices
    ) > 0:
        embedding_memmap = (
            np.lib.format.open_memmap(
                embedding_path,
                mode="r+",
                dtype=np.float32,
                shape=expected_shape,
            )
        )

        dataset = CanonicalSourceDataset(
            dataframe=ordered,
            global_indices=(
                missing_indices
            ),
        )

        loader = DataLoader(
            dataset,
            batch_size=(
                BATCH_SIZE
            ),
            shuffle=False,
            num_workers=(
                NUM_WORKERS
            ),
            pin_memory=False,
            drop_last=False,
            persistent_workers=(
                NUM_WORKERS
                >
                0
            ),
        )

        with torch.inference_mode():
            for batch_number, (
                image_batch,
                global_index_batch,
            ) in enumerate(
                tqdm(
                    loader,
                    desc=(
                        f"CPU extracting "
                        f"{dataset_name} embeddings"
                    ),
                    total=len(
                        loader
                    ),
                ),
                start=1,
            ):
                image_batch = image_batch.to(
                    device
                )

                feature_batch = model(
                    image_batch
                )

                feature_batch = F.normalize(
                    feature_batch,
                    p=2,
                    dim=1,
                )

                global_indices = (
                    global_index_batch
                    .numpy()
                    .astype(
                        np.int64
                    )
                )

                batch_embeddings = (
                    feature_batch
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(
                        np.float32
                    )
                )

                embedding_memmap[
                    global_indices
                ] = batch_embeddings

                completed[
                    global_indices
                ] = True

                if (
                    batch_number
                    %
                    CHECKPOINT_EVERY_BATCHES
                    ==
                    0
                ):
                    embedding_memmap.flush()

                    np.save(
                        completion_path,
                        completed,
                        allow_pickle=False,
                    )

                    with metadata_path.open(
                        "r",
                        encoding="utf-8",
                    ) as file:
                        metadata_payload = json.load(
                            file
                        )

                    metadata_payload[
                        "completed_images"
                    ] = int(
                        completed.sum()
                    )

                    metadata_payload[
                        "last_checkpoint_utc"
                    ] = datetime.now(
                        timezone.utc
                    ).isoformat()

                    with metadata_path.open(
                        "w",
                        encoding="utf-8",
                    ) as file:
                        json.dump(
                            metadata_payload,
                            file,
                            indent=2,
                        )

        embedding_memmap.flush()

        np.save(
            completion_path,
            completed,
            allow_pickle=False,
        )

        del embedding_memmap

    assert completed.all(), (
        f"{dataset_name}: embedding extraction "
        "did not complete."
    )

    embeddings = np.load(
        embedding_path,
        mmap_mode="r",
        allow_pickle=False,
    )

    assert embeddings.shape == (
        len(
            ordered
        ),
        FROZEN_FEATURE_DIMENSION,
    )

    assert np.isfinite(
        embeddings
    ).all()

    row_norms = np.linalg.norm(
        embeddings,
        axis=1,
    )

    assert np.allclose(
        row_norms,
        1.0,
        atol=1e-5,
    )

    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        metadata_payload = json.load(
            file
        )

    metadata_payload.update(
        {
            "status": (
                "COMPLETE"
            ),
            "completed_images": int(
                len(
                    ordered
                )
            ),
            "completed_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "embedding_sha256": sha256_file(
                embedding_path
            ),
            "completion_sha256": sha256_file(
                completion_path
            ),
            "image_id_sha256": sha256_file(
                image_id_path
            ),
            "minimum_l2_norm": float(
                row_norms.min()
            ),
            "maximum_l2_norm": float(
                row_norms.max()
            ),
        }
    )

    with metadata_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata_payload,
            file,
            indent=2,
        )

    print(
        f"{dataset_name}: embedding extraction complete."
    )

    return (
        ordered,
        embeddings,
        embedding_path,
        metadata_path,
    )


dataset_assets = {}


for dataset_name in [
    "EyePACS_2015",
    "DeepDRiD",
]:
    dataset_frame = source_manifest[
        source_manifest[
            "dataset"
        ]
        ==
        dataset_name
    ].copy()

    (
        ordered_frame,
        embeddings,
        embedding_path,
        embedding_metadata_path,
    ) = load_or_extract_embeddings(
        dataframe=dataset_frame,
        dataset_name=dataset_name,
    )

    dataset_assets[
        dataset_name
    ] = {
        "frame": (
            ordered_frame
        ),
        "embeddings": (
            embeddings
        ),
        "embedding_path": (
            embedding_path
        ),
        "embedding_metadata_path": (
            embedding_metadata_path
        ),
    }


model = model.cpu()

del model


# ============================================================
# 8. Development OOF, axis freeze, then validation
# ============================================================

source_summary_records = []
fold_metric_records = []
axis_summary_records = []


for source_index, dataset_name in enumerate(
    [
        "EyePACS_2015",
        "DeepDRiD",
    ],
    start=1,
):
    print(
        f"\n================ MODELLING "
        f"{dataset_name} ================"
    )

    frame = (
        dataset_assets[
            dataset_name
        ][
            "frame"
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    embeddings = np.asarray(
        dataset_assets[
            dataset_name
        ][
            "embeddings"
        ],
        dtype=np.float32,
    )

    development_mask = (
        frame[
            "source_partition"
        ]
        ==
        "development"
    ).to_numpy()

    validation_mask = (
        frame[
            "source_partition"
        ]
        ==
        "validation"
    ).to_numpy()

    development_indices = np.flatnonzero(
        development_mask
    )

    validation_indices = np.flatnonzero(
        validation_mask
    )

    development_frame = (
        frame.iloc[
            development_indices
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    validation_frame = (
        frame.iloc[
            validation_indices
        ]
        .reset_index(
            drop=True
        )
        .copy()
    )

    development_embeddings = embeddings[
        development_indices
    ]

    validation_embeddings = embeddings[
        validation_indices
    ]

    development_labels = development_frame[
        "moderate_or_worse_dr"
    ].to_numpy(
        dtype=np.int64
    )

    development_groups = development_frame[
        "patient_id"
    ].astype(str).to_numpy()

    assert np.unique(
        development_labels
    ).size == 2

    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    development_oof_probability = np.full(
        len(
            development_frame
        ),
        np.nan,
        dtype=np.float64,
    )

    development_fold = np.full(
        len(
            development_frame
        ),
        -1,
        dtype=np.int64,
    )

    for fold_number, (
        training_rows,
        heldout_rows,
    ) in enumerate(
        splitter.split(
            development_embeddings,
            development_labels,
            groups=(
                development_groups
            ),
        ),
        start=1,
    ):
        assert not set(
            development_groups[
                training_rows
            ]
        ).intersection(
            set(
                development_groups[
                    heldout_rows
                ]
            )
        )

        probe = create_probe()

        training_weights = (
            eye_balanced_sample_weight(
                development_frame.iloc[
                    training_rows
                ][
                    "eye_unit_id"
                ].astype(str)
            )
        )

        probe.fit(
            development_embeddings[
                training_rows
            ],
            development_labels[
                training_rows
            ],
            logisticregression__sample_weight=(
                training_weights
            ),
        )

        heldout_probability = (
            probe.predict_proba(
                development_embeddings[
                    heldout_rows
                ]
            )[
                :,
                1,
            ]
        )

        development_oof_probability[
            heldout_rows
        ] = heldout_probability

        development_fold[
            heldout_rows
        ] = fold_number

        fold_prediction_table = (
            development_frame.iloc[
                heldout_rows
            ][
                [
                    "dataset",
                    "image_id",
                    "patient_id",
                    "eye_unit_id",
                    "source_partition",
                    "moderate_or_worse_dr",
                ]
            ]
            .reset_index(
                drop=True
            )
            .copy()
        )

        fold_prediction_table[
            "probability"
        ] = heldout_probability

        fold_prediction_table[
            "fold"
        ] = fold_number

        fold_eye_table = aggregate_to_eye(
            fold_prediction_table
        )

        assert (
            fold_eye_table[
                "moderate_or_worse_dr"
            ].nunique()
            ==
            2
        )

        fold_auc = float(
            roc_auc_score(
                fold_eye_table[
                    "moderate_or_worse_dr"
                ],
                fold_eye_table[
                    "probability"
                ],
            )
        )

        print(
            f"{dataset_name} fold "
            f"{fold_number} eye AUC: "
            f"{fold_auc:.4f}"
        )

        fold_metric_records.append(
            {
                "dataset": (
                    dataset_name
                ),
                "fold": int(
                    fold_number
                ),
                "training_images": int(
                    len(
                        training_rows
                    )
                ),
                "heldout_images": int(
                    len(
                        heldout_rows
                    )
                ),
                "training_patients": int(
                    pd.Series(
                        development_groups[
                            training_rows
                        ]
                    ).nunique()
                ),
                "heldout_patients": int(
                    pd.Series(
                        development_groups[
                            heldout_rows
                        ]
                    ).nunique()
                ),
                "heldout_eyes": int(
                    len(
                        fold_eye_table
                    )
                ),
                "heldout_eye_auc": (
                    fold_auc
                ),
            }
        )

    assert np.isfinite(
        development_oof_probability
    ).all()

    assert (
        development_fold
        >
        0
    ).all()

    development_prediction_table = (
        development_frame[
            [
                "dataset",
                "image_id",
                "patient_id",
                "eye_unit_id",
                "source_partition",
                "dr_grade",
                "moderate_or_worse_dr",
            ]
        ]
        .copy()
    )

    development_prediction_table[
        "probability"
    ] = development_oof_probability

    development_prediction_table[
        "fold"
    ] = development_fold

    development_eye_table = aggregate_to_eye(
        development_prediction_table
    )

    development_metrics = metric_record(
        eye_table=(
            development_eye_table
        ),
        bootstrap_seed=(
            RANDOM_SEED
            +
            source_index
            *
            100
            +
            1
        ),
    )

    print(
        f"{dataset_name} development OOF AUC: "
        f"{development_metrics['eye_auc']:.4f} "
        f"[{development_metrics['eye_auc_ci_lower_95']:.4f}, "
        f"{development_metrics['eye_auc_ci_upper_95']:.4f}]"
    )

    # Fit exactly one final source axis.
    final_probe = create_probe()

    final_training_weights = (
        eye_balanced_sample_weight(
            development_frame[
                "eye_unit_id"
            ].astype(str)
        )
    )

    final_probe.fit(
        development_embeddings,
        development_labels,
        logisticregression__sample_weight=(
            final_training_weights
        ),
    )

    development_manifest_sha256 = (
        stable_table_sha256(
            development_frame,
            [
                "dataset",
                "image_id",
                "canonical_sha256",
                "moderate_or_worse_dr",
            ],
        )
    )

    # Axis is saved before validation scoring.
    axis_artifacts = save_axis(
        probe=(
            final_probe
        ),
        dataset_name=(
            dataset_name
        ),
        development_manifest_sha256=(
            development_manifest_sha256
        ),
    )

    verification_rows = min(
        64,
        len(
            development_embeddings
        ),
    )

    pipeline_decision = (
        final_probe.decision_function(
            development_embeddings[
                :verification_rows
            ]
        )
    )

    raw_decision = (
        development_embeddings[
            :verification_rows
        ]
        @
        axis_artifacts[
            "coefficient_raw"
        ]
        +
        axis_artifacts[
            "intercept_raw"
        ]
    )

    maximum_axis_equivalence_error = float(
        np.max(
            np.abs(
                pipeline_decision
                -
                raw_decision
            )
        )
    )

    assert (
        maximum_axis_equivalence_error
        <
        1e-6
    )

    # Validation is observed only after axis write and hash.
    validation_probability = (
        final_probe.predict_proba(
            validation_embeddings
        )[
            :,
            1,
        ]
    )

    validation_prediction_table = (
        validation_frame[
            [
                "dataset",
                "image_id",
                "patient_id",
                "eye_unit_id",
                "source_partition",
                "dr_grade",
                "moderate_or_worse_dr",
            ]
        ]
        .copy()
    )

    validation_prediction_table[
        "probability"
    ] = validation_probability

    validation_prediction_table[
        "fold"
    ] = 0

    validation_eye_table = aggregate_to_eye(
        validation_prediction_table
    )

    validation_metrics = metric_record(
        eye_table=(
            validation_eye_table
        ),
        bootstrap_seed=(
            RANDOM_SEED
            +
            source_index
            *
            100
            +
            2
        ),
    )

    print(
        f"{dataset_name} validation AUC: "
        f"{validation_metrics['eye_auc']:.4f} "
        f"[{validation_metrics['eye_auc_ci_lower_95']:.4f}, "
        f"{validation_metrics['eye_auc_ci_upper_95']:.4f}]"
    )

    development_pass = bool(
        development_metrics[
            "eye_auc"
        ]
        >=
        AUC_THRESHOLD
        and
        development_metrics[
            "eye_auc_ci_lower_95"
        ]
        >
        CI_LOWER_THRESHOLD
    )

    validation_pass = bool(
        validation_metrics[
            "eye_auc"
        ]
        >=
        AUC_THRESHOLD
        and
        validation_metrics[
            "eye_auc_ci_lower_95"
        ]
        >
        CI_LOWER_THRESHOLD
    )

    source_pass = bool(
        development_pass
        and
        validation_pass
    )

    safe_name = dataset_name.replace(
        " ",
        "_",
    )

    development_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Development_OOF_Image_Predictions_v0.1.csv"
    )

    development_eye_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Development_OOF_Eye_Predictions_v0.1.csv"
    )

    validation_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Validation_Image_Predictions_v0.1.csv"
    )

    validation_eye_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Validation_Eye_Predictions_v0.1.csv"
    )

    development_prediction_table.to_csv(
        development_prediction_path,
        index=False,
    )

    development_eye_table.to_csv(
        development_eye_prediction_path,
        index=False,
    )

    validation_prediction_table.to_csv(
        validation_prediction_path,
        index=False,
    )

    validation_eye_table.to_csv(
        validation_eye_prediction_path,
        index=False,
    )

    source_summary_records.append(
        {
            "dataset": (
                dataset_name
            ),
            "development_images": int(
                len(
                    development_frame
                )
            ),
            "development_eyes": int(
                len(
                    development_eye_table
                )
            ),
            "development_patients": int(
                development_frame[
                    "patient_id"
                ].nunique()
            ),
            "development_eye_auc": (
                development_metrics[
                    "eye_auc"
                ]
            ),
            "development_eye_auc_ci_lower_95": (
                development_metrics[
                    "eye_auc_ci_lower_95"
                ]
            ),
            "development_eye_auc_ci_upper_95": (
                development_metrics[
                    "eye_auc_ci_upper_95"
                ]
            ),
            "development_pass": (
                development_pass
            ),
            "validation_images": int(
                len(
                    validation_frame
                )
            ),
            "validation_eyes": int(
                len(
                    validation_eye_table
                )
            ),
            "validation_patients": int(
                validation_frame[
                    "patient_id"
                ].nunique()
            ),
            "validation_eye_auc": (
                validation_metrics[
                    "eye_auc"
                ]
            ),
            "validation_eye_auc_ci_lower_95": (
                validation_metrics[
                    "eye_auc_ci_lower_95"
                ]
            ),
            "validation_eye_auc_ci_upper_95": (
                validation_metrics[
                    "eye_auc_ci_upper_95"
                ]
            ),
            "validation_eye_average_precision": (
                validation_metrics[
                    "eye_average_precision"
                ]
            ),
            "validation_eye_balanced_accuracy_at_0_5": (
                validation_metrics[
                    "eye_balanced_accuracy_at_0_5"
                ]
            ),
            "validation_eye_brier_score": (
                validation_metrics[
                    "eye_brier_score"
                ]
            ),
            "validation_pass": (
                validation_pass
            ),
            "source_pass": (
                source_pass
            ),
            "axis_npz_path": str(
                axis_artifacts[
                    "axis_npz_path"
                ]
            ),
            "axis_npz_sha256": (
                axis_artifacts[
                    "axis_npz_sha256"
                ]
            ),
            "axis_metadata_path": str(
                axis_artifacts[
                    "axis_json_path"
                ]
            ),
            "axis_metadata_sha256": (
                axis_artifacts[
                    "axis_json_sha256"
                ]
            ),
            "maximum_axis_equivalence_error": (
                maximum_axis_equivalence_error
            ),
        }
    )

    axis_summary_records.append(
        {
            "dataset": (
                dataset_name
            ),
            "development_images": int(
                len(
                    development_frame
                )
            ),
            "development_eyes": int(
                development_frame[
                    "eye_unit_id"
                ].nunique()
            ),
            "development_patients": int(
                development_frame[
                    "patient_id"
                ].nunique()
            ),
            "raw_axis_l2_norm": (
                axis_artifacts[
                    "raw_axis_l2_norm"
                ]
            ),
            "intercept_raw": (
                axis_artifacts[
                    "intercept_raw"
                ]
            ),
            "maximum_pipeline_raw_axis_error": (
                maximum_axis_equivalence_error
            ),
            "axis_npz_sha256": (
                axis_artifacts[
                    "axis_npz_sha256"
                ]
            ),
            "axis_frozen_before_validation": True,
            "axis_frozen_before_target_access": True,
        }
    )


# ============================================================
# 9. Save summaries and final Stage 5A decision
# ============================================================

source_summary = pd.DataFrame(
    source_summary_records
)

fold_summary = pd.DataFrame(
    fold_metric_records
)

axis_summary = pd.DataFrame(
    axis_summary_records
)


SOURCE_SUMMARY_PATH = (
    RESULT_ROOT
    / "Stage5A_Source_Recoverability_Summary_v0.1.csv"
)

FOLD_SUMMARY_PATH = (
    RESULT_ROOT
    / "Stage5A_Source_Development_Fold_Metrics_v0.1.csv"
)

AXIS_SUMMARY_PATH = (
    RESULT_ROOT
    / "Stage5A_Frozen_Source_Axis_Summary_v0.1.csv"
)

REPORT_PATH = (
    RESULT_ROOT
    / "Stage5A_Source_Recoverability_And_Axis_Freeze_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    RESULT_ROOT
    / "Stage5A_Environment_v0.1.json"
)


source_summary.to_csv(
    SOURCE_SUMMARY_PATH,
    index=False,
)

fold_summary.to_csv(
    FOLD_SUMMARY_PATH,
    index=False,
)

axis_summary.to_csv(
    AXIS_SUMMARY_PATH,
    index=False,
)


passing_sources = (
    source_summary.loc[
        source_summary[
            "source_pass"
        ],
        "dataset",
    ]
    .astype(str)
    .tolist()
)


retired_sources = [
    source
    for source in [
        "EyePACS_2015",
        "DeepDRiD",
    ]
    if source not in (
        passing_sources
    )
]


targets = [
    "APTOS_2019",
    "IDRiD",
]


authorised_edges = [
    {
        "source": (
            source
        ),
        "target": (
            target
        ),
    }
    for source in (
        passing_sources
    )
    for target in (
        targets
    )
]


if len(
    passing_sources
) == 2:
    stage5_decision = (
        "PASS_BOTH_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE_"
        "ADVANCE_ALL_FOUR_EDGES_TO_BLIND_TRANSFER"
    )

    authorised_next_step = (
        "PRECOMMIT_BLIND_SOURCE_TO_TARGET_TRANSFER_"
        "AND_SCORE_LABEL_FREE_TARGETS"
    )

    interpretation = (
        "Both source datasets passed the frozen development OOF and "
        "held-out source-validation recoverability gates. Their exact "
        "development-fitted diagnostic axes were hash-committed before "
        "validation scoring. All four precommitted source-target edges "
        "remain authorised for the prospective blind transfer test."
    )

elif len(
    passing_sources
) == 1:
    stage5_decision = (
        "PARTIAL_PASS_RETAIN_ONLY_EDGES_FROM_RECOVERABLE_SOURCE"
    )

    authorised_next_step = (
        "PRECOMMIT_AND_SCORE_ONLY_THE_TWO_EDGES_"
        "FROM_THE_PASSING_SOURCE"
    )

    interpretation = (
        "Only one source passed both frozen recoverability checks. "
        "Only the two precommitted edges originating from that source "
        "remain authorised. The other source and its two edges are "
        "retired before target transfer performance is observed."
    )

else:
    stage5_decision = (
        "FAIL_SOURCE_RECOVERABILITY_RETIRE_PROSPECTIVE_TEST"
    )

    authorised_next_step = (
        "DO_NOT_RUN_SOURCE_TO_TARGET_TRANSFER"
    )

    interpretation = (
        "Neither source passed both frozen recoverability checks. "
        "The source axes are not sufficiently supported for an "
        "interpretable prospective transfer test, so the redesigned "
        "retinal experiment is retired."
    )


decision_payload = {
    "decision": (
        stage5_decision
    ),
    "interpretation": (
        interpretation
    ),
    "authorised_next_step": (
        authorised_next_step
    ),
    "endpoint_id": (
        ENDPOINT_ID
    ),
    "source_pass_rule": (
        SOURCE_PASS_RULE
    ),
    "axis_freeze_rule": (
        AXIS_FREEZE_RULE
    ),
    "passing_sources": (
        passing_sources
    ),
    "retired_sources": (
        retired_sources
    ),
    "authorised_edges": (
        authorised_edges
    ),
    "source_results": (
        source_summary
        .to_dict(
            orient="records"
        )
    ),
    "protocol_path": str(
        PROTOCOL_PATH
    ),
    "protocol_sha256": sha256_file(
        PROTOCOL_PATH
    ),
    "source_manifest_path": str(
        SOURCE_MANIFEST_PATH
    ),
    "source_manifest_sha256": sha256_file(
        SOURCE_MANIFEST_PATH
    ),
    "execution_device": (
        "cpu"
    ),
    "target_images_accessed": False,
    "target_sealed_label_files_accessed": False,
    "target_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_performance_observed": True,
    "source_axes_frozen_before_target_access": True,
}


with STAGE5_DECISION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


REPORT_PATH.write_text(
    "\n".join(
        [
            (
                "# Stage 5A — Source Recoverability "
                "and Axis Freeze"
            ),
            "",
            f"Decision: `{stage5_decision}`",
            "",
            interpretation,
            "",
            "## Execution",
            "",
            "- Device: `CPU`",
            "- Embedding checkpointing: `Enabled`",
            "",
            "## Frozen gate",
            "",
            (
                "- Development: patient-grouped "
                "five-fold OOF, eye-level AUC"
            ),
            (
                "- Validation: held-out source "
                "partition, eye-level AUC"
            ),
            (
                f"- Pass threshold: AUC >= "
                f"{AUC_THRESHOLD:.2f} and 95% CI "
                f"lower > {CI_LOWER_THRESHOLD:.2f}"
            ),
            (
                "- Both development and validation "
                "must pass"
            ),
            "",
            "## Safety boundary",
            "",
            "- Target images accessed: `False`",
            (
                "- Target sealed labels accessed: "
                "`False`"
            ),
            (
                "- Target performance observed: "
                "`False`"
            ),
            (
                "- Source-target transfer performance "
                "observed: `False`"
            ),
            (
                "- Source axes frozen before target "
                "access: `True`"
            ),
        ]
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": (
        sys.version
    ),
    "platform": (
        platform.platform()
    ),
    "numpy": (
        np.__version__
    ),
    "pandas": (
        pd.__version__
    ),
    "sklearn": (
        sklearn.__version__
    ),
    "torch": (
        torch.__version__
    ),
    "torchvision": (
        torchvision.__version__
    ),
    "device": (
        "cpu"
    ),
    "cpu_threads": (
        CPU_THREADS
    ),
    "batch_size": (
        BATCH_SIZE
    ),
    "random_seed": (
        RANDOM_SEED
    ),
    "protocol_sha256": sha256_file(
        PROTOCOL_PATH
    ),
}


with ENVIRONMENT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 10. Final display
# ============================================================

print(
    "\n================ SOURCE RECOVERABILITY "
    "SUMMARY ================"
)

display(
    source_summary[
        [
            "dataset",
            "development_images",
            "development_eyes",
            "development_patients",
            "development_eye_auc",
            "development_eye_auc_ci_lower_95",
            "development_eye_auc_ci_upper_95",
            "development_pass",
            "validation_images",
            "validation_eyes",
            "validation_patients",
            "validation_eye_auc",
            "validation_eye_auc_ci_lower_95",
            "validation_eye_auc_ci_upper_95",
            "validation_pass",
            "source_pass",
        ]
    ]
)


print(
    "\n================ DEVELOPMENT FOLD "
    "STABILITY ================"
)

display(
    fold_summary
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        folds=(
            "fold",
            "size",
        ),
        minimum_eye_auc=(
            "heldout_eye_auc",
            "min",
        ),
        median_eye_auc=(
            "heldout_eye_auc",
            "median",
        ),
        maximum_eye_auc=(
            "heldout_eye_auc",
            "max",
        ),
    )
)


print(
    "\n================ FROZEN SOURCE AXIS "
    "SUMMARY ================"
)

display(
    axis_summary
)


print(
    "\n================ STAGE 5A DECISION "
    "================"
)

print("Decision:")
print(
    stage5_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nPassing sources:")
print(
    passing_sources
)

print("\nRetired sources:")
print(
    retired_sources
)

print("\nAuthorised edges:")
print(
    json.dumps(
        authorised_edges,
        indent=2,
    )
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nExecution device:")
print(
    "CPU"
)

print("\nTarget images accessed:")
print(False)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nTarget performance observed:")
print(False)

print("\nSource-target transfer performance observed:")
print(False)

print(
    "\nStage 5A source recoverability and "
    "source-axis freeze completed and sealed."
)

================ STAGE 5A-CPU SOURCE RECOVERABILITY AND AXIS FREEZE ================

Imported Stage 4G decision:
PASS_CANONICAL_SOURCE_FINALISATION_ADVANCE_TO_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE

Execution device:
cpu

CPU threads:
2

Embedding batch size:
16

DataLoader workers:
1

Frozen source pass rule:
A source passes only if its fixed frozen-representation probe passes both checks: (1) patient-grouped five-fold development out-of-fold eye-level ROC AUC >= 0.70 with patient-bootstrap 95% CI lower bound > 0.55; and (2) the final axis fitted once on all development data achieves held-out validation eye-level ROC AUC >= 0.70 with patient-bootstrap 95% CI lower bound > 0.55. No architecture, C, class weight, threshold, preprocessing, or feature selection may be tuned.

Frozen axis rule:
For each source, fit one StandardScaler plus class-balanced L2-regularised logistic regression on all frozen development embeddings, weighting images so each eye contributes total training weight on

,dataset,source_partition,images,eyes,patients,negative_images,positive_images
0,DeepDRiD,development,1200,600,300,680,520
1,DeepDRiD,validation,400,200,100,220,180
2,EyePACS_2015,development,3350,3350,1675,2061,1289
3,EyePACS_2015,validation,1118,1118,559,674,444


ValueError: Object arrays cannot be saved when allow_pickle=False

In [41]:
#@title 05A-CPU-R1. Repair Unicode checkpoint IDs and continue Stage 5A

from pathlib import Path
from datetime import datetime, timezone
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet50, ResNet50_Weights
from tqdm.auto import tqdm

import hashlib
import json
import os
import platform
import random
import sys

import numpy as np
import pandas as pd
import sklearn
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision


# ============================================================
# 0. Confirm that the preceding Stage 5A-CPU setup remains loaded
# ============================================================

REQUIRED_RUNTIME_SYMBOLS = [
    "SOURCE_MANIFEST_PATH",
    "EMBEDDING_ROOT",
    "AXIS_ROOT",
    "RESULT_ROOT",
    "PROTOCOL_PATH",
    "STAGE5_DECISION_PATH",
    "FROZEN_BACKBONE",
    "FROZEN_FEATURE_DIMENSION",
    "EMBEDDING_NORMALISATION",
    "ENDPOINT_ID",
    "EXPECTED_DATASET_COUNTS",
    "EXPECTED_PARTITION_COUNTS",
    "EXPECTED_EYE_UNIT_COUNTS",
    "RANDOM_SEED",
    "N_FOLDS",
    "N_BOOTSTRAP",
    "AUC_THRESHOLD",
    "CI_LOWER_THRESHOLD",
    "SOURCE_PASS_RULE",
    "AXIS_FREEZE_RULE",
    "BATCH_SIZE",
    "NUM_WORKERS",
    "CHECKPOINT_EVERY_BATCHES",
    "CPU_THREADS",
    "sha256_file",
    "stable_table_sha256",
    "build_eye_unit_id",
    "create_probe",
    "eye_balanced_sample_weight",
    "aggregate_to_eye",
    "metric_record",
    "save_axis",
]

missing_symbols = [
    symbol
    for symbol in REQUIRED_RUNTIME_SYMBOLS
    if symbol not in globals()
]

assert not missing_symbols, (
    "The Colab runtime was restarted and the Stage 5A setup is no "
    "longer in memory. Missing symbols:\n"
    + "\n".join(missing_symbols)
)


FINAL_STAGE5_DECISIONS = {
    "PASS_BOTH_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE_ADVANCE_ALL_FOUR_EDGES_TO_BLIND_TRANSFER",
    "PARTIAL_PASS_RETAIN_ONLY_EDGES_FROM_RECOVERABLE_SOURCE",
    "FAIL_SOURCE_RECOVERABILITY_RETIRE_PROSPECTIVE_TEST",
}


if STAGE5_DECISION_PATH.is_file():
    with STAGE5_DECISION_PATH.open(
        "r",
        encoding="utf-8",
    ) as file:
        existing_stage5_decision = json.load(file)

    if existing_stage5_decision.get(
        "decision"
    ) in FINAL_STAGE5_DECISIONS:
        raise RuntimeError(
            "Stage 5A already has a sealed final decision. "
            "Do not overwrite or rerun it."
        )


print(
    "================ STAGE 5A-CPU-R1 "
    "CHECKPOINT REPAIR AND CONTINUATION ================"
)

print("\nRepair:")
print(
    "Image IDs will be stored as fixed-width NumPy Unicode, "
    "not object arrays and not pickle."
)

print("\nExecution device:")
print("CPU")

print("\nTarget images accessed:")
print(False)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nSource performance observed before this continuation:")
print(False)


# ============================================================
# 1. Reload and verify the final source manifest
# ============================================================

source_manifest = pd.read_csv(
    SOURCE_MANIFEST_PATH,
    low_memory=False,
)

assert (
    len(source_manifest)
    ==
    sum(EXPECTED_DATASET_COUNTS.values())
)

assert not source_manifest.duplicated(
    subset=[
        "dataset",
        "image_id",
    ]
).any()

source_manifest[
    "patient_id"
] = source_manifest[
    "patient_id"
].astype(str)

source_manifest[
    "eye_unit_id"
] = source_manifest.apply(
    build_eye_unit_id,
    axis=1,
)

assert source_manifest[
    "canonical_image_path"
].map(
    lambda value: Path(value).is_file()
).all()

assert (
    source_manifest[
        "moderate_or_worse_dr"
    ]
    ==
    (
        source_manifest[
            "dr_grade"
        ]
        >=
        2
    ).astype(int)
).all()


for dataset_name, expected_count in (
    EXPECTED_DATASET_COUNTS.items()
):
    dataset_rows = source_manifest[
        source_manifest[
            "dataset"
        ]
        ==
        dataset_name
    ]

    assert (
        len(dataset_rows)
        ==
        expected_count
    )


for (
    dataset_name,
    partition_name,
), expected_count in (
    EXPECTED_PARTITION_COUNTS.items()
):
    partition_rows = source_manifest[
        (
            source_manifest[
                "dataset"
            ]
            ==
            dataset_name
        )
        &
        (
            source_manifest[
                "source_partition"
            ]
            ==
            partition_name
        )
    ]

    assert (
        len(partition_rows)
        ==
        expected_count
    )

    assert (
        partition_rows[
            "eye_unit_id"
        ].nunique()
        ==
        EXPECTED_EYE_UNIT_COUNTS[
            (
                dataset_name,
                partition_name,
            )
        ]
    )


representation_inventory = (
    source_manifest
    .groupby(
        [
            "dataset",
            "source_partition",
        ],
        as_index=False,
    )
    .agg(
        images=(
            "image_id",
            "size",
        ),
        eyes=(
            "eye_unit_id",
            "nunique",
        ),
        patients=(
            "patient_id",
            "nunique",
        ),
        negative_images=(
            "moderate_or_worse_dr",
            lambda values: int(
                (values == 0).sum()
            ),
        ),
        positive_images=(
            "moderate_or_worse_dr",
            lambda values: int(
                (values == 1).sum()
            ),
        ),
    )
)


print(
    "\n================ SOURCE REPRESENTATION "
    "INVENTORY ================"
)

display(
    representation_inventory
)


# ============================================================
# 2. Frozen CPU ResNet50
# ============================================================

device = torch.device(
    "cpu"
)

torch.set_num_threads(
    CPU_THREADS
)

weights = (
    ResNet50_Weights
    .IMAGENET1K_V2
)

inference_transform = weights.transforms()

model = resnet50(
    weights=weights
)

model.fc = nn.Identity()
model.eval()

for parameter in model.parameters():
    parameter.requires_grad = False

model = model.to(
    device
)

ImageFile.LOAD_TRUNCATED_IMAGES = False


# ============================================================
# 3. Dataset for resumable extraction
# ============================================================

class CanonicalSourceDataset(Dataset):
    def __init__(
        self,
        dataframe,
        global_indices,
    ):
        self.dataframe = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.global_indices = np.asarray(
            global_indices,
            dtype=np.int64,
        )

    def __len__(self):
        return len(
            self.global_indices
        )

    def __getitem__(
        self,
        local_index,
    ):
        global_index = int(
            self.global_indices[
                local_index
            ]
        )

        row = self.dataframe.iloc[
            global_index
        ]

        image_path = Path(
            row[
                "canonical_image_path"
            ]
        )

        with Image.open(
            image_path
        ) as image:
            image.load()

            assert image.size == (
                768,
                768,
            )

            image = image.convert(
                "RGB"
            )

            image_tensor = inference_transform(
                image
            )

        return (
            image_tensor,
            global_index,
        )


def fixed_width_unicode_array(
    values,
):
    strings = [
        str(value)
        for value in values
    ]

    maximum_length = max(
        1,
        max(
            len(value)
            for value in strings
        ),
    )

    return np.asarray(
        strings,
        dtype=f"<U{maximum_length}",
    )


def remove_paths(
    paths,
):
    for path in paths:
        path = Path(path)

        if path.exists():
            path.unlink()


# ============================================================
# 4. Corrected resumable embedding extraction
# ============================================================

def load_or_extract_embeddings_repaired(
    dataframe,
    dataset_name,
):
    ordered = (
        dataframe
        .sort_values(
            [
                "source_partition",
                "image_id",
            ]
        )
        .reset_index(drop=True)
        .copy()
    )

    manifest_commitment = stable_table_sha256(
        ordered,
        [
            "dataset",
            "image_id",
            "canonical_sha256",
        ],
    )

    safe_name = dataset_name.replace(
        " ",
        "_",
    )

    embedding_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy"
    )

    completion_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_Completion_v0.1.npy"
    )

    image_id_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy"
    )

    metadata_path = (
        EMBEDDING_ROOT
        /
        f"{safe_name}_Canonical_ResNet50V2_L2_Embeddings_Metadata_v0.1.json"
    )

    checkpoint_paths = [
        embedding_path,
        completion_path,
        image_id_path,
        metadata_path,
    ]

    expected_shape = (
        len(ordered),
        FROZEN_FEATURE_DIMENSION,
    )

    expected_image_ids = (
        fixed_width_unicode_array(
            ordered[
                "image_id"
            ].tolist()
        )
    )

    resume_ok = False

    if all(
        path.is_file()
        for path in checkpoint_paths
    ):
        try:
            with metadata_path.open(
                "r",
                encoding="utf-8",
            ) as file:
                metadata = json.load(file)

            stored_ids = np.load(
                image_id_path,
                allow_pickle=False,
            ).astype(str)

            completed = np.load(
                completion_path,
                allow_pickle=False,
            ).astype(bool)

            stored_embeddings = np.load(
                embedding_path,
                mmap_mode="r",
                allow_pickle=False,
            )

            resume_ok = bool(
                metadata.get(
                    "manifest_commitment_sha256"
                )
                ==
                manifest_commitment
                and
                metadata.get(
                    "backbone"
                )
                ==
                FROZEN_BACKBONE
                and
                stored_embeddings.shape
                ==
                expected_shape
                and
                completed.shape
                ==
                (
                    len(ordered),
                )
                and
                np.array_equal(
                    stored_ids,
                    expected_image_ids.astype(str),
                )
            )

            del stored_embeddings

        except Exception:
            resume_ok = False

    if not resume_ok:
        partial_files = [
            str(path)
            for path in checkpoint_paths
            if path.exists()
        ]

        if partial_files:
            print(
                f"\nRemoving incomplete checkpoint files "
                f"for {dataset_name}:"
            )

            for path in partial_files:
                print(path)

        remove_paths(
            checkpoint_paths
        )

        embedding_memmap = (
            np.lib.format.open_memmap(
                embedding_path,
                mode="w+",
                dtype=np.float32,
                shape=expected_shape,
            )
        )

        embedding_memmap[:] = 0.0
        embedding_memmap.flush()

        del embedding_memmap

        completed = np.zeros(
            len(ordered),
            dtype=np.bool_,
        )

        np.save(
            completion_path,
            completed,
            allow_pickle=False,
        )

        # Critical repair:
        # fixed-width Unicode instead of dtype=object.
        np.save(
            image_id_path,
            expected_image_ids,
            allow_pickle=False,
        )

        metadata_payload = {
            "dataset": dataset_name,
            "status": "IN_PROGRESS",
            "images": int(
                len(ordered)
            ),
            "completed_images": 0,
            "embedding_shape": list(
                expected_shape
            ),
            "dtype": "float32",
            "image_id_dtype": str(
                expected_image_ids.dtype
            ),
            "backbone": (
                FROZEN_BACKBONE
            ),
            "embedding_normalisation": (
                EMBEDDING_NORMALISATION
            ),
            "manifest_commitment_sha256": (
                manifest_commitment
            ),
            "embedding_path": str(
                embedding_path
            ),
            "completion_path": str(
                completion_path
            ),
            "image_id_path": str(
                image_id_path
            ),
            "execution_device": "cpu",
            "target_images_accessed": False,
            "target_sealed_labels_accessed": False,
        }

        with metadata_path.open(
            "w",
            encoding="utf-8",
        ) as file:
            json.dump(
                metadata_payload,
                file,
                indent=2,
            )

    completed = np.load(
        completion_path,
        allow_pickle=False,
    ).astype(bool)

    missing_indices = np.flatnonzero(
        ~completed
    )

    print(
        f"\n{dataset_name}: completed embeddings "
        f"{int(completed.sum())}/{len(completed)}"
    )

    if len(missing_indices) > 0:
        embedding_memmap = (
            np.lib.format.open_memmap(
                embedding_path,
                mode="r+",
                dtype=np.float32,
                shape=expected_shape,
            )
        )

        extraction_dataset = (
            CanonicalSourceDataset(
                dataframe=ordered,
                global_indices=missing_indices,
            )
        )

        loader = DataLoader(
            extraction_dataset,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=NUM_WORKERS,
            pin_memory=False,
            drop_last=False,
            persistent_workers=(
                NUM_WORKERS > 0
            ),
        )

        with torch.inference_mode():
            for batch_number, (
                image_batch,
                global_index_batch,
            ) in enumerate(
                tqdm(
                    loader,
                    desc=(
                        f"CPU extracting "
                        f"{dataset_name} embeddings"
                    ),
                    total=len(loader),
                ),
                start=1,
            ):
                image_batch = image_batch.to(
                    device
                )

                feature_batch = model(
                    image_batch
                )

                feature_batch = F.normalize(
                    feature_batch,
                    p=2,
                    dim=1,
                )

                global_indices = (
                    global_index_batch
                    .numpy()
                    .astype(np.int64)
                )

                batch_embeddings = (
                    feature_batch
                    .detach()
                    .cpu()
                    .numpy()
                    .astype(np.float32)
                )

                embedding_memmap[
                    global_indices
                ] = batch_embeddings

                completed[
                    global_indices
                ] = True

                if (
                    batch_number
                    %
                    CHECKPOINT_EVERY_BATCHES
                    ==
                    0
                ):
                    embedding_memmap.flush()

                    np.save(
                        completion_path,
                        completed,
                        allow_pickle=False,
                    )

                    with metadata_path.open(
                        "r",
                        encoding="utf-8",
                    ) as file:
                        metadata_payload = json.load(
                            file
                        )

                    metadata_payload[
                        "completed_images"
                    ] = int(
                        completed.sum()
                    )

                    metadata_payload[
                        "last_checkpoint_utc"
                    ] = datetime.now(
                        timezone.utc
                    ).isoformat()

                    with metadata_path.open(
                        "w",
                        encoding="utf-8",
                    ) as file:
                        json.dump(
                            metadata_payload,
                            file,
                            indent=2,
                        )

        embedding_memmap.flush()

        np.save(
            completion_path,
            completed,
            allow_pickle=False,
        )

        del embedding_memmap

    assert completed.all(), (
        f"{dataset_name}: embedding extraction "
        "did not complete."
    )

    embeddings = np.load(
        embedding_path,
        mmap_mode="r",
        allow_pickle=False,
    )

    assert embeddings.shape == expected_shape
    assert np.isfinite(embeddings).all()

    row_norms = np.linalg.norm(
        embeddings,
        axis=1,
    )

    assert np.allclose(
        row_norms,
        1.0,
        atol=1e-5,
    )

    with metadata_path.open(
        "r",
        encoding="utf-8",
    ) as file:
        metadata_payload = json.load(file)

    metadata_payload.update(
        {
            "status": "COMPLETE",
            "completed_images": int(
                len(ordered)
            ),
            "completed_utc": datetime.now(
                timezone.utc
            ).isoformat(),
            "embedding_sha256": sha256_file(
                embedding_path
            ),
            "completion_sha256": sha256_file(
                completion_path
            ),
            "image_id_sha256": sha256_file(
                image_id_path
            ),
            "minimum_l2_norm": float(
                row_norms.min()
            ),
            "maximum_l2_norm": float(
                row_norms.max()
            ),
        }
    )

    with metadata_path.open(
        "w",
        encoding="utf-8",
    ) as file:
        json.dump(
            metadata_payload,
            file,
            indent=2,
        )

    print(
        f"{dataset_name}: embedding extraction complete."
    )

    return (
        ordered,
        embeddings,
        embedding_path,
        metadata_path,
    )


# ============================================================
# 5. Extract or resume both source embeddings
# ============================================================

dataset_assets = {}


for dataset_name in [
    "EyePACS_2015",
    "DeepDRiD",
]:
    dataset_frame = source_manifest[
        source_manifest[
            "dataset"
        ]
        ==
        dataset_name
    ].copy()

    (
        ordered_frame,
        embeddings,
        embedding_path,
        embedding_metadata_path,
    ) = load_or_extract_embeddings_repaired(
        dataframe=dataset_frame,
        dataset_name=dataset_name,
    )

    dataset_assets[
        dataset_name
    ] = {
        "frame": ordered_frame,
        "embeddings": embeddings,
        "embedding_path": embedding_path,
        "embedding_metadata_path": (
            embedding_metadata_path
        ),
    }


model = model.cpu()
del model


# ============================================================
# 6. Development OOF, axis freeze, then held-out validation
# ============================================================

source_summary_records = []
fold_metric_records = []
axis_summary_records = []


for source_index, dataset_name in enumerate(
    [
        "EyePACS_2015",
        "DeepDRiD",
    ],
    start=1,
):
    print(
        f"\n================ MODELLING "
        f"{dataset_name} ================"
    )

    frame = (
        dataset_assets[
            dataset_name
        ][
            "frame"
        ]
        .reset_index(drop=True)
        .copy()
    )

    embeddings = np.asarray(
        dataset_assets[
            dataset_name
        ][
            "embeddings"
        ],
        dtype=np.float32,
    )

    development_mask = (
        frame[
            "source_partition"
        ]
        ==
        "development"
    ).to_numpy()

    validation_mask = (
        frame[
            "source_partition"
        ]
        ==
        "validation"
    ).to_numpy()

    development_indices = np.flatnonzero(
        development_mask
    )

    validation_indices = np.flatnonzero(
        validation_mask
    )

    development_frame = (
        frame.iloc[
            development_indices
        ]
        .reset_index(drop=True)
        .copy()
    )

    validation_frame = (
        frame.iloc[
            validation_indices
        ]
        .reset_index(drop=True)
        .copy()
    )

    development_embeddings = embeddings[
        development_indices
    ]

    validation_embeddings = embeddings[
        validation_indices
    ]

    development_labels = development_frame[
        "moderate_or_worse_dr"
    ].to_numpy(
        dtype=np.int64
    )

    development_groups = development_frame[
        "patient_id"
    ].astype(str).to_numpy()

    assert (
        np.unique(
            development_labels
        ).size
        ==
        2
    )

    splitter = StratifiedGroupKFold(
        n_splits=N_FOLDS,
        shuffle=True,
        random_state=RANDOM_SEED,
    )

    development_oof_probability = np.full(
        len(development_frame),
        np.nan,
        dtype=np.float64,
    )

    development_fold = np.full(
        len(development_frame),
        -1,
        dtype=np.int64,
    )

    for fold_number, (
        training_rows,
        heldout_rows,
    ) in enumerate(
        splitter.split(
            development_embeddings,
            development_labels,
            groups=development_groups,
        ),
        start=1,
    ):
        assert not set(
            development_groups[
                training_rows
            ]
        ).intersection(
            set(
                development_groups[
                    heldout_rows
                ]
            )
        )

        probe = create_probe()

        training_weights = (
            eye_balanced_sample_weight(
                development_frame.iloc[
                    training_rows
                ][
                    "eye_unit_id"
                ].astype(str)
            )
        )

        probe.fit(
            development_embeddings[
                training_rows
            ],
            development_labels[
                training_rows
            ],
            logisticregression__sample_weight=(
                training_weights
            ),
        )

        heldout_probability = (
            probe.predict_proba(
                development_embeddings[
                    heldout_rows
                ]
            )[:, 1]
        )

        development_oof_probability[
            heldout_rows
        ] = heldout_probability

        development_fold[
            heldout_rows
        ] = fold_number

        fold_prediction_table = (
            development_frame.iloc[
                heldout_rows
            ][
                [
                    "dataset",
                    "image_id",
                    "patient_id",
                    "eye_unit_id",
                    "source_partition",
                    "moderate_or_worse_dr",
                ]
            ]
            .reset_index(drop=True)
            .copy()
        )

        fold_prediction_table[
            "probability"
        ] = heldout_probability

        fold_prediction_table[
            "fold"
        ] = fold_number

        fold_eye_table = aggregate_to_eye(
            fold_prediction_table
        )

        fold_auc = float(
            roc_auc_score(
                fold_eye_table[
                    "moderate_or_worse_dr"
                ],
                fold_eye_table[
                    "probability"
                ],
            )
        )

        print(
            f"{dataset_name} fold "
            f"{fold_number} eye AUC: "
            f"{fold_auc:.4f}"
        )

        fold_metric_records.append(
            {
                "dataset": dataset_name,
                "fold": int(
                    fold_number
                ),
                "training_images": int(
                    len(training_rows)
                ),
                "heldout_images": int(
                    len(heldout_rows)
                ),
                "training_patients": int(
                    pd.Series(
                        development_groups[
                            training_rows
                        ]
                    ).nunique()
                ),
                "heldout_patients": int(
                    pd.Series(
                        development_groups[
                            heldout_rows
                        ]
                    ).nunique()
                ),
                "heldout_eyes": int(
                    len(fold_eye_table)
                ),
                "heldout_eye_auc": fold_auc,
            }
        )

    assert np.isfinite(
        development_oof_probability
    ).all()

    assert (
        development_fold > 0
    ).all()

    development_prediction_table = (
        development_frame[
            [
                "dataset",
                "image_id",
                "patient_id",
                "eye_unit_id",
                "source_partition",
                "dr_grade",
                "moderate_or_worse_dr",
            ]
        ]
        .copy()
    )

    development_prediction_table[
        "probability"
    ] = development_oof_probability

    development_prediction_table[
        "fold"
    ] = development_fold

    development_eye_table = aggregate_to_eye(
        development_prediction_table
    )

    development_metrics = metric_record(
        eye_table=development_eye_table,
        bootstrap_seed=(
            RANDOM_SEED
            +
            source_index * 100
            +
            1
        ),
    )

    print(
        f"{dataset_name} development OOF AUC: "
        f"{development_metrics['eye_auc']:.4f} "
        f"[{development_metrics['eye_auc_ci_lower_95']:.4f}, "
        f"{development_metrics['eye_auc_ci_upper_95']:.4f}]"
    )

    # Fit and save exactly one final axis.
    final_probe = create_probe()

    final_training_weights = (
        eye_balanced_sample_weight(
            development_frame[
                "eye_unit_id"
            ].astype(str)
        )
    )

    final_probe.fit(
        development_embeddings,
        development_labels,
        logisticregression__sample_weight=(
            final_training_weights
        ),
    )

    development_manifest_sha256 = (
        stable_table_sha256(
            development_frame,
            [
                "dataset",
                "image_id",
                "canonical_sha256",
                "moderate_or_worse_dr",
            ],
        )
    )

    # Axis is written and hash-committed before validation.
    axis_artifacts = save_axis(
        probe=final_probe,
        dataset_name=dataset_name,
        development_manifest_sha256=(
            development_manifest_sha256
        ),
    )

    verification_rows = min(
        64,
        len(development_embeddings),
    )

    pipeline_decision = (
        final_probe.decision_function(
            development_embeddings[
                :verification_rows
            ]
        )
    )

    raw_decision = (
        development_embeddings[
            :verification_rows
        ]
        @
        axis_artifacts[
            "coefficient_raw"
        ]
        +
        axis_artifacts[
            "intercept_raw"
        ]
    )

    maximum_axis_equivalence_error = float(
        np.max(
            np.abs(
                pipeline_decision
                -
                raw_decision
            )
        )
    )

    assert (
        maximum_axis_equivalence_error
        <
        1e-6
    )

    # Held-out validation is observed only after axis freeze.
    validation_probability = (
        final_probe.predict_proba(
            validation_embeddings
        )[:, 1]
    )

    validation_prediction_table = (
        validation_frame[
            [
                "dataset",
                "image_id",
                "patient_id",
                "eye_unit_id",
                "source_partition",
                "dr_grade",
                "moderate_or_worse_dr",
            ]
        ]
        .copy()
    )

    validation_prediction_table[
        "probability"
    ] = validation_probability

    validation_prediction_table[
        "fold"
    ] = 0

    validation_eye_table = aggregate_to_eye(
        validation_prediction_table
    )

    validation_metrics = metric_record(
        eye_table=validation_eye_table,
        bootstrap_seed=(
            RANDOM_SEED
            +
            source_index * 100
            +
            2
        ),
    )

    print(
        f"{dataset_name} validation AUC: "
        f"{validation_metrics['eye_auc']:.4f} "
        f"[{validation_metrics['eye_auc_ci_lower_95']:.4f}, "
        f"{validation_metrics['eye_auc_ci_upper_95']:.4f}]"
    )

    development_pass = bool(
        development_metrics[
            "eye_auc"
        ]
        >=
        AUC_THRESHOLD
        and
        development_metrics[
            "eye_auc_ci_lower_95"
        ]
        >
        CI_LOWER_THRESHOLD
    )

    validation_pass = bool(
        validation_metrics[
            "eye_auc"
        ]
        >=
        AUC_THRESHOLD
        and
        validation_metrics[
            "eye_auc_ci_lower_95"
        ]
        >
        CI_LOWER_THRESHOLD
    )

    source_pass = bool(
        development_pass
        and
        validation_pass
    )

    safe_name = dataset_name.replace(
        " ",
        "_",
    )

    development_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Development_OOF_Image_Predictions_v0.1.csv"
    )

    development_eye_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Development_OOF_Eye_Predictions_v0.1.csv"
    )

    validation_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Validation_Image_Predictions_v0.1.csv"
    )

    validation_eye_prediction_path = (
        RESULT_ROOT
        /
        f"{safe_name}_Validation_Eye_Predictions_v0.1.csv"
    )

    development_prediction_table.to_csv(
        development_prediction_path,
        index=False,
    )

    development_eye_table.to_csv(
        development_eye_prediction_path,
        index=False,
    )

    validation_prediction_table.to_csv(
        validation_prediction_path,
        index=False,
    )

    validation_eye_table.to_csv(
        validation_eye_prediction_path,
        index=False,
    )

    source_summary_records.append(
        {
            "dataset": dataset_name,
            "development_images": int(
                len(development_frame)
            ),
            "development_eyes": int(
                len(development_eye_table)
            ),
            "development_patients": int(
                development_frame[
                    "patient_id"
                ].nunique()
            ),
            "development_eye_auc": (
                development_metrics[
                    "eye_auc"
                ]
            ),
            "development_eye_auc_ci_lower_95": (
                development_metrics[
                    "eye_auc_ci_lower_95"
                ]
            ),
            "development_eye_auc_ci_upper_95": (
                development_metrics[
                    "eye_auc_ci_upper_95"
                ]
            ),
            "development_pass": (
                development_pass
            ),
            "validation_images": int(
                len(validation_frame)
            ),
            "validation_eyes": int(
                len(validation_eye_table)
            ),
            "validation_patients": int(
                validation_frame[
                    "patient_id"
                ].nunique()
            ),
            "validation_eye_auc": (
                validation_metrics[
                    "eye_auc"
                ]
            ),
            "validation_eye_auc_ci_lower_95": (
                validation_metrics[
                    "eye_auc_ci_lower_95"
                ]
            ),
            "validation_eye_auc_ci_upper_95": (
                validation_metrics[
                    "eye_auc_ci_upper_95"
                ]
            ),
            "validation_eye_average_precision": (
                validation_metrics[
                    "eye_average_precision"
                ]
            ),
            "validation_eye_balanced_accuracy_at_0_5": (
                validation_metrics[
                    "eye_balanced_accuracy_at_0_5"
                ]
            ),
            "validation_eye_brier_score": (
                validation_metrics[
                    "eye_brier_score"
                ]
            ),
            "validation_pass": (
                validation_pass
            ),
            "source_pass": source_pass,
            "axis_npz_path": str(
                axis_artifacts[
                    "axis_npz_path"
                ]
            ),
            "axis_npz_sha256": (
                axis_artifacts[
                    "axis_npz_sha256"
                ]
            ),
            "axis_metadata_path": str(
                axis_artifacts[
                    "axis_json_path"
                ]
            ),
            "axis_metadata_sha256": (
                axis_artifacts[
                    "axis_json_sha256"
                ]
            ),
            "maximum_axis_equivalence_error": (
                maximum_axis_equivalence_error
            ),
        }
    )

    axis_summary_records.append(
        {
            "dataset": dataset_name,
            "development_images": int(
                len(development_frame)
            ),
            "development_eyes": int(
                development_frame[
                    "eye_unit_id"
                ].nunique()
            ),
            "development_patients": int(
                development_frame[
                    "patient_id"
                ].nunique()
            ),
            "raw_axis_l2_norm": (
                axis_artifacts[
                    "raw_axis_l2_norm"
                ]
            ),
            "intercept_raw": (
                axis_artifacts[
                    "intercept_raw"
                ]
            ),
            "maximum_pipeline_raw_axis_error": (
                maximum_axis_equivalence_error
            ),
            "axis_npz_sha256": (
                axis_artifacts[
                    "axis_npz_sha256"
                ]
            ),
            "axis_frozen_before_validation": True,
            "axis_frozen_before_target_access": True,
        }
    )


# ============================================================
# 7. Save summaries and seal Stage 5A decision
# ============================================================

source_summary = pd.DataFrame(
    source_summary_records
)

fold_summary = pd.DataFrame(
    fold_metric_records
)

axis_summary = pd.DataFrame(
    axis_summary_records
)


SOURCE_SUMMARY_PATH = (
    RESULT_ROOT
    / "Stage5A_Source_Recoverability_Summary_v0.1.csv"
)

FOLD_SUMMARY_PATH = (
    RESULT_ROOT
    / "Stage5A_Source_Development_Fold_Metrics_v0.1.csv"
)

AXIS_SUMMARY_PATH = (
    RESULT_ROOT
    / "Stage5A_Frozen_Source_Axis_Summary_v0.1.csv"
)

REPORT_PATH = (
    RESULT_ROOT
    / "Stage5A_Source_Recoverability_And_Axis_Freeze_Report_v0.1.md"
)

ENVIRONMENT_PATH = (
    RESULT_ROOT
    / "Stage5A_Environment_v0.1.json"
)


source_summary.to_csv(
    SOURCE_SUMMARY_PATH,
    index=False,
)

fold_summary.to_csv(
    FOLD_SUMMARY_PATH,
    index=False,
)

axis_summary.to_csv(
    AXIS_SUMMARY_PATH,
    index=False,
)


passing_sources = (
    source_summary.loc[
        source_summary[
            "source_pass"
        ],
        "dataset",
    ]
    .astype(str)
    .tolist()
)

retired_sources = [
    source
    for source in [
        "EyePACS_2015",
        "DeepDRiD",
    ]
    if source not in passing_sources
]

targets = [
    "APTOS_2019",
    "IDRiD",
]

authorised_edges = [
    {
        "source": source,
        "target": target,
    }
    for source in passing_sources
    for target in targets
]


if len(passing_sources) == 2:
    stage5_decision = (
        "PASS_BOTH_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE_"
        "ADVANCE_ALL_FOUR_EDGES_TO_BLIND_TRANSFER"
    )

    authorised_next_step = (
        "PRECOMMIT_BLIND_SOURCE_TO_TARGET_TRANSFER_"
        "AND_SCORE_LABEL_FREE_TARGETS"
    )

    interpretation = (
        "Both source datasets passed the frozen development OOF and "
        "held-out source-validation recoverability gates. Their exact "
        "development-fitted diagnostic axes were hash-committed before "
        "validation scoring. All four precommitted source-target edges "
        "remain authorised for the prospective blind transfer test."
    )

elif len(passing_sources) == 1:
    stage5_decision = (
        "PARTIAL_PASS_RETAIN_ONLY_EDGES_FROM_RECOVERABLE_SOURCE"
    )

    authorised_next_step = (
        "PRECOMMIT_AND_SCORE_ONLY_THE_TWO_EDGES_"
        "FROM_THE_PASSING_SOURCE"
    )

    interpretation = (
        "Only one source passed both frozen recoverability checks. "
        "Only the two precommitted edges originating from that source "
        "remain authorised. The other source and its two edges are "
        "retired before target transfer performance is observed."
    )

else:
    stage5_decision = (
        "FAIL_SOURCE_RECOVERABILITY_RETIRE_PROSPECTIVE_TEST"
    )

    authorised_next_step = (
        "DO_NOT_RUN_SOURCE_TO_TARGET_TRANSFER"
    )

    interpretation = (
        "Neither source passed both frozen recoverability checks. "
        "The source axes are not sufficiently supported for an "
        "interpretable prospective transfer test, so the redesigned "
        "retinal experiment is retired."
    )


decision_payload = {
    "decision": stage5_decision,
    "interpretation": interpretation,
    "authorised_next_step": (
        authorised_next_step
    ),
    "endpoint_id": ENDPOINT_ID,
    "source_pass_rule": (
        SOURCE_PASS_RULE
    ),
    "axis_freeze_rule": (
        AXIS_FREEZE_RULE
    ),
    "passing_sources": (
        passing_sources
    ),
    "retired_sources": (
        retired_sources
    ),
    "authorised_edges": (
        authorised_edges
    ),
    "source_results": (
        source_summary.to_dict(
            orient="records"
        )
    ),
    "protocol_path": str(
        PROTOCOL_PATH
    ),
    "protocol_sha256": sha256_file(
        PROTOCOL_PATH
    ),
    "source_manifest_path": str(
        SOURCE_MANIFEST_PATH
    ),
    "source_manifest_sha256": sha256_file(
        SOURCE_MANIFEST_PATH
    ),
    "execution_device": "cpu",
    "image_id_checkpoint_dtype": (
        "fixed_width_unicode"
    ),
    "target_images_accessed": False,
    "target_sealed_label_files_accessed": False,
    "target_performance_observed": False,
    "source_target_transfer_performance_observed": False,
    "source_performance_observed": True,
    "source_axes_frozen_before_target_access": True,
}


with STAGE5_DECISION_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        decision_payload,
        file,
        indent=2,
    )


REPORT_PATH.write_text(
    "\n".join(
        [
            (
                "# Stage 5A — Source Recoverability "
                "and Axis Freeze"
            ),
            "",
            f"Decision: `{stage5_decision}`",
            "",
            interpretation,
            "",
            "## Execution",
            "",
            "- Device: `CPU`",
            (
                "- Image-ID checkpoint dtype: "
                "`fixed-width Unicode`"
            ),
            "- Embedding checkpointing: `Enabled`",
            "",
            "## Safety boundary",
            "",
            "- Target images accessed: `False`",
            (
                "- Target sealed labels accessed: "
                "`False`"
            ),
            (
                "- Target performance observed: "
                "`False`"
            ),
            (
                "- Source-target transfer performance "
                "observed: `False`"
            ),
        ]
    ),
    encoding="utf-8",
)


environment_payload = {
    "timestamp_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "sklearn": sklearn.__version__,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "device": "cpu",
    "cpu_threads": CPU_THREADS,
    "batch_size": BATCH_SIZE,
    "random_seed": RANDOM_SEED,
    "protocol_sha256": sha256_file(
        PROTOCOL_PATH
    ),
}


with ENVIRONMENT_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        environment_payload,
        file,
        indent=2,
    )


# ============================================================
# 8. Final display
# ============================================================

print(
    "\n================ SOURCE RECOVERABILITY "
    "SUMMARY ================"
)

display(
    source_summary[
        [
            "dataset",
            "development_images",
            "development_eyes",
            "development_patients",
            "development_eye_auc",
            "development_eye_auc_ci_lower_95",
            "development_eye_auc_ci_upper_95",
            "development_pass",
            "validation_images",
            "validation_eyes",
            "validation_patients",
            "validation_eye_auc",
            "validation_eye_auc_ci_lower_95",
            "validation_eye_auc_ci_upper_95",
            "validation_pass",
            "source_pass",
        ]
    ]
)


print(
    "\n================ DEVELOPMENT FOLD "
    "STABILITY ================"
)

display(
    fold_summary
    .groupby(
        "dataset",
        as_index=False,
    )
    .agg(
        folds=(
            "fold",
            "size",
        ),
        minimum_eye_auc=(
            "heldout_eye_auc",
            "min",
        ),
        median_eye_auc=(
            "heldout_eye_auc",
            "median",
        ),
        maximum_eye_auc=(
            "heldout_eye_auc",
            "max",
        ),
    )
)


print(
    "\n================ FROZEN SOURCE AXIS "
    "SUMMARY ================"
)

display(
    axis_summary
)


print(
    "\n================ STAGE 5A DECISION "
    "================"
)

print("Decision:")
print(
    stage5_decision
)

print("\nInterpretation:")
print(
    interpretation
)

print("\nPassing sources:")
print(
    passing_sources
)

print("\nRetired sources:")
print(
    retired_sources
)

print("\nAuthorised edges:")
print(
    json.dumps(
        authorised_edges,
        indent=2,
    )
)

print("\nAuthorised next step:")
print(
    authorised_next_step
)

print("\nExecution device:")
print("CPU")

print("\nTarget images accessed:")
print(False)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nTarget performance observed:")
print(False)

print("\nSource-target transfer performance observed:")
print(False)

print(
    "\nStage 5A source recoverability and "
    "source-axis freeze completed and sealed."
)

================ STAGE 5A-CPU-R1 CHECKPOINT REPAIR AND CONTINUATION ================

Repair:
Image IDs will be stored as fixed-width NumPy Unicode, not object arrays and not pickle.

Execution device:
CPU

Target images accessed:
False

Target sealed-label files accessed:
False

Source performance observed before this continuation:
False

================ SOURCE REPRESENTATION INVENTORY ================


,dataset,source_partition,images,eyes,patients,negative_images,positive_images
0,DeepDRiD,development,1200,600,300,680,520
1,DeepDRiD,validation,400,200,100,220,180
2,EyePACS_2015,development,3350,3350,1675,2061,1289
3,EyePACS_2015,validation,1118,1118,559,674,444



Removing incomplete checkpoint files for EyePACS_2015:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage5_Source_Recoverability_And_Axis_Freeze_v0.1/01_Frozen_Embeddings/EyePACS_2015_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage5_Source_Recoverability_And_Axis_Freeze_v0.1/01_Frozen_Embeddings/EyePACS_2015_Canonical_ResNet50V2_L2_Completion_v0.1.npy
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage5_Source_Recoverability_And_Axis_Freeze_v0.1/01_Frozen_Embeddings/EyePACS_2015_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy

EyePACS_2015: completed embeddings 0/4468


CPU extracting EyePACS_2015 embeddings:   0%|          | 0/280 [00:00<?, ?it/s]

EyePACS_2015: embedding extraction complete.

DeepDRiD: completed embeddings 0/1600


CPU extracting DeepDRiD embeddings:   0%|          | 0/100 [00:00<?, ?it/s]

DeepDRiD: embedding extraction complete.

================ MODELLING EyePACS_2015 ================
EyePACS_2015 fold 1 eye AUC: 0.5920
EyePACS_2015 fold 2 eye AUC: 0.6057
EyePACS_2015 fold 3 eye AUC: 0.5855
EyePACS_2015 fold 4 eye AUC: 0.5737
EyePACS_2015 fold 5 eye AUC: 0.5910
EyePACS_2015 development OOF AUC: 0.5907 [0.5702, 0.6098]


AssertionError: 

In [2]:
#@title 05A-CPU-R1D. Diagnose frozen-axis equivalence without validation access

print(
    "================ STAGE 5A AXIS "
    "EQUIVALENCE DIAGNOSTIC ================"
)

assert "final_probe" in globals(), (
    "The runtime was restarted; final_probe is no longer available."
)

assert "axis_artifacts" in globals()
assert "development_embeddings" in globals()
assert dataset_name == "EyePACS_2015"

verification_rows = min(
    64,
    len(development_embeddings),
)

x32 = np.asarray(
    development_embeddings[:verification_rows],
    dtype=np.float32,
)

x64 = x32.astype(
    np.float64,
    copy=True,
)

scaler = final_probe.named_steps[
    "standardscaler"
]

classifier = final_probe.named_steps[
    "logisticregression"
]

mean64 = np.asarray(
    scaler.mean_,
    dtype=np.float64,
).reshape(-1)

scale64 = np.asarray(
    scaler.scale_,
    dtype=np.float64,
).reshape(-1)

coefficient_scaled64 = np.asarray(
    classifier.coef_[0],
    dtype=np.float64,
).reshape(-1)

intercept_scaled64 = float(
    classifier.intercept_[0]
)

coefficient_reconstructed64 = (
    coefficient_scaled64
    /
    scale64
)

intercept_reconstructed64 = float(
    intercept_scaled64
    -
    np.dot(
        mean64,
        coefficient_reconstructed64,
    )
)

coefficient_saved = np.asarray(
    axis_artifacts[
        "coefficient_raw"
    ]
).reshape(-1)

intercept_saved = float(
    np.asarray(
        axis_artifacts[
            "intercept_raw"
        ]
    ).reshape(-1)[0]
)

pipeline_decision32 = np.asarray(
    final_probe.decision_function(
        x32
    ),
    dtype=np.float64,
)

pipeline_decision64 = np.asarray(
    final_probe.decision_function(
        x64
    ),
    dtype=np.float64,
)

manual_scaled_decision64 = (
    (
        (x64 - mean64)
        /
        scale64
    )
    @
    coefficient_scaled64
    +
    intercept_scaled64
)

reconstructed_raw_decision64 = (
    x64
    @
    coefficient_reconstructed64
    +
    intercept_reconstructed64
)

saved_raw_decision_from_x32 = np.asarray(
    x32
    @
    coefficient_saved
    +
    intercept_saved,
    dtype=np.float64,
)

saved_raw_decision_from_x64 = (
    x64
    @
    coefficient_saved.astype(
        np.float64
    )
    +
    intercept_saved
)

diagnostic = pd.Series(
    {
        "dataset": dataset_name,
        "verification_rows": verification_rows,
        "embedding_dtype": str(x32.dtype),
        "scaler_mean_dtype": str(
            scaler.mean_.dtype
        ),
        "scaler_scale_dtype": str(
            scaler.scale_.dtype
        ),
        "classifier_coefficient_dtype": str(
            classifier.coef_.dtype
        ),
        "saved_raw_coefficient_dtype": str(
            coefficient_saved.dtype
        ),
        "original_maximum_axis_equivalence_error": float(
            maximum_axis_equivalence_error
        ),
        "pipeline_float32_vs_float64_error": float(
            np.max(
                np.abs(
                    pipeline_decision32
                    -
                    pipeline_decision64
                )
            )
        ),
        "pipeline64_vs_manual_scaled64_error": float(
            np.max(
                np.abs(
                    pipeline_decision64
                    -
                    manual_scaled_decision64
                )
            )
        ),
        "manual_scaled64_vs_reconstructed_raw64_error": float(
            np.max(
                np.abs(
                    manual_scaled_decision64
                    -
                    reconstructed_raw_decision64
                )
            )
        ),
        "pipeline32_vs_saved_raw_x32_error": float(
            np.max(
                np.abs(
                    pipeline_decision32
                    -
                    saved_raw_decision_from_x32
                )
            )
        ),
        "pipeline64_vs_saved_raw_x64_error": float(
            np.max(
                np.abs(
                    pipeline_decision64
                    -
                    saved_raw_decision_from_x64
                )
            )
        ),
        "saved_vs_reconstructed_coefficient_max_error": float(
            np.max(
                np.abs(
                    coefficient_saved.astype(
                        np.float64
                    )
                    -
                    coefficient_reconstructed64
                )
            )
        ),
        "saved_vs_reconstructed_intercept_error": float(
            abs(
                intercept_saved
                -
                intercept_reconstructed64
            )
        ),
    },
    name="value",
)

display(
    diagnostic.to_frame()
)

axis_npz_path = Path(
    axis_artifacts[
        "axis_npz_path"
    ]
)

print("\nFrozen NPZ array inventory:")

with np.load(
    axis_npz_path,
    allow_pickle=False,
) as frozen_axis:
    for key in frozen_axis.files:
        array = frozen_axis[key]

        print(
            key,
            "shape=",
            array.shape,
            "dtype=",
            array.dtype,
        )

print("\nTarget images accessed:", False)
print("Target sealed-label files accessed:", False)
print("Source validation performance accessed:", False)
print("Files modified by this diagnostic:", False)

================ STAGE 5A AXIS EQUIVALENCE DIAGNOSTIC ================


AssertionError: The runtime was restarted; final_probe is no longer available.

In [3]:
#@title 05A-CPU-R1D2. Rehydrate EyePACS probe and diagnose axis precision

from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# 0. Confirm that only the lightweight Stage 5A setup was rerun
# ============================================================

REQUIRED_SYMBOLS = [
    "SOURCE_MANIFEST_PATH",
    "EMBEDDING_ROOT",
    "AXIS_ROOT",
    "FROZEN_FEATURE_DIMENSION",
    "build_eye_unit_id",
    "stable_table_sha256",
    "create_probe",
    "eye_balanced_sample_weight",
    "save_axis",
]

missing_symbols = [
    symbol
    for symbol in REQUIRED_SYMBOLS
    if symbol not in globals()
]

assert not missing_symbols, (
    "First rerun the preceding Stage 5A-CPU setup cell. "
    "Missing symbols:\n"
    +
    "\n".join(missing_symbols)
)


print(
    "================ STAGE 5A REHYDRATED AXIS "
    "DIAGNOSTIC ================"
)

print("\nTarget images accessed:")
print(False)

print("\nTarget sealed-label files accessed:")
print(False)

print("\nSource validation performance accessed:")
print(False)

print("\nPersistent files modified:")
print(False)


# ============================================================
# 1. Reconstruct the exact EyePACS embedding/manifest alignment
# ============================================================

dataset_name = "EyePACS_2015"

source_manifest = pd.read_csv(
    SOURCE_MANIFEST_PATH,
    low_memory=False,
)

source_manifest[
    "patient_id"
] = source_manifest[
    "patient_id"
].astype(str)

source_manifest[
    "eye_unit_id"
] = source_manifest.apply(
    build_eye_unit_id,
    axis=1,
)

frame = (
    source_manifest[
        source_manifest[
            "dataset"
        ]
        ==
        dataset_name
    ]
    .sort_values(
        [
            "source_partition",
            "image_id",
        ]
    )
    .reset_index(drop=True)
    .copy()
)

safe_name = dataset_name.replace(
    " ",
    "_",
)

embedding_path = (
    EMBEDDING_ROOT
    /
    f"{safe_name}_Canonical_ResNet50V2_L2_Embeddings_v0.1.npy"
)

completion_path = (
    EMBEDDING_ROOT
    /
    f"{safe_name}_Canonical_ResNet50V2_L2_Completion_v0.1.npy"
)

image_id_path = (
    EMBEDDING_ROOT
    /
    f"{safe_name}_Canonical_ResNet50V2_L2_ImageIDs_v0.1.npy"
)

assert embedding_path.is_file()
assert completion_path.is_file()
assert image_id_path.is_file()

stored_image_ids = np.load(
    image_id_path,
    allow_pickle=False,
).astype(str)

expected_image_ids = (
    frame[
        "image_id"
    ]
    .astype(str)
    .to_numpy()
)

assert np.array_equal(
    stored_image_ids,
    expected_image_ids,
), "Stored embedding row order does not match the source manifest."

completion = np.load(
    completion_path,
    allow_pickle=False,
).astype(bool)

assert completion.shape == (
    len(frame),
)

assert completion.all(), (
    "The EyePACS embedding checkpoint is not complete."
)

embeddings = np.load(
    embedding_path,
    mmap_mode="r",
    allow_pickle=False,
)

assert embeddings.shape == (
    len(frame),
    FROZEN_FEATURE_DIMENSION,
)

assert np.isfinite(
    embeddings
).all()

development_indices = np.flatnonzero(
    (
        frame[
            "source_partition"
        ]
        ==
        "development"
    ).to_numpy()
)

development_frame = (
    frame.iloc[
        development_indices
    ]
    .reset_index(drop=True)
    .copy()
)

development_embeddings = np.asarray(
    embeddings[
        development_indices
    ],
    dtype=np.float32,
)

development_labels = development_frame[
    "moderate_or_worse_dr"
].to_numpy(
    dtype=np.int64,
)

assert len(
    development_frame
) == 3350

assert np.unique(
    development_labels
).size == 2

print("\nRehydrated EyePACS development images:")
print(len(development_frame))

print("\nEmbedding shape:")
print(development_embeddings.shape)

print("\nEmbedding dtype:")
print(development_embeddings.dtype)


# ============================================================
# 2. Refit exactly the development-only final probe
# ============================================================

final_probe = create_probe()

final_training_weights = (
    eye_balanced_sample_weight(
        development_frame[
            "eye_unit_id"
        ].astype(str)
    )
)

final_probe.fit(
    development_embeddings,
    development_labels,
    logisticregression__sample_weight=(
        final_training_weights
    ),
)

development_manifest_sha256 = (
    stable_table_sha256(
        development_frame,
        [
            "dataset",
            "image_id",
            "canonical_sha256",
            "moderate_or_worse_dr",
        ],
    )
)


# ============================================================
# 3. Run the existing save_axis implementation only in /tmp
# ============================================================

persistent_axis_root = AXIS_ROOT

temporary_axis_root = Path(
    "/tmp/stage5a_axis_precision_diagnostic"
)

temporary_axis_root.mkdir(
    parents=True,
    exist_ok=True,
)

try:
    AXIS_ROOT = temporary_axis_root

    diagnostic_axis_artifacts = save_axis(
        probe=final_probe,
        dataset_name=dataset_name,
        development_manifest_sha256=(
            development_manifest_sha256
        ),
    )

finally:
    AXIS_ROOT = persistent_axis_root


# ============================================================
# 4. Separate formula error from numerical precision
# ============================================================

verification_rows = min(
    64,
    len(development_embeddings),
)

x32 = np.asarray(
    development_embeddings[
        :verification_rows
    ],
    dtype=np.float32,
)

x64 = x32.astype(
    np.float64,
    copy=True,
)

scaler = final_probe.named_steps[
    "standardscaler"
]

classifier = final_probe.named_steps[
    "logisticregression"
]

mean64 = np.asarray(
    scaler.mean_,
    dtype=np.float64,
).reshape(-1)

scale64 = np.asarray(
    scaler.scale_,
    dtype=np.float64,
).reshape(-1)

coefficient_scaled64 = np.asarray(
    classifier.coef_[0],
    dtype=np.float64,
).reshape(-1)

intercept_scaled64 = float(
    classifier.intercept_[0]
)

coefficient_reconstructed64 = (
    coefficient_scaled64
    /
    scale64
)

intercept_reconstructed64 = float(
    intercept_scaled64
    -
    np.dot(
        mean64,
        coefficient_reconstructed64,
    )
)

coefficient_saved = np.asarray(
    diagnostic_axis_artifacts[
        "coefficient_raw"
    ]
).reshape(-1)

intercept_saved = float(
    np.asarray(
        diagnostic_axis_artifacts[
            "intercept_raw"
        ]
    ).reshape(-1)[0]
)

pipeline_decision32 = np.asarray(
    final_probe.decision_function(
        x32
    ),
    dtype=np.float64,
)

pipeline_decision64 = np.asarray(
    final_probe.decision_function(
        x64
    ),
    dtype=np.float64,
)

manual_scaled_decision64 = (
    (
        (x64 - mean64)
        /
        scale64
    )
    @
    coefficient_scaled64
    +
    intercept_scaled64
)

reconstructed_raw_decision64 = (
    x64
    @
    coefficient_reconstructed64
    +
    intercept_reconstructed64
)

saved_raw_decision_x32 = np.asarray(
    x32
    @
    coefficient_saved
    +
    intercept_saved,
    dtype=np.float64,
)

saved_raw_decision_x64 = (
    x64
    @
    coefficient_saved.astype(
        np.float64
    )
    +
    intercept_saved
)

diagnostic = pd.Series(
    {
        "dataset": dataset_name,
        "verification_rows": verification_rows,
        "embedding_dtype": str(
            x32.dtype
        ),
        "scaler_mean_dtype": str(
            scaler.mean_.dtype
        ),
        "scaler_scale_dtype": str(
            scaler.scale_.dtype
        ),
        "classifier_coefficient_dtype": str(
            classifier.coef_.dtype
        ),
        "returned_raw_coefficient_dtype": str(
            coefficient_saved.dtype
        ),
        "original_style_pipeline32_vs_saved_x32_error": float(
            np.max(
                np.abs(
                    pipeline_decision32
                    -
                    saved_raw_decision_x32
                )
            )
        ),
        "pipeline_float32_vs_float64_error": float(
            np.max(
                np.abs(
                    pipeline_decision32
                    -
                    pipeline_decision64
                )
            )
        ),
        "pipeline64_vs_manual_scaled64_error": float(
            np.max(
                np.abs(
                    pipeline_decision64
                    -
                    manual_scaled_decision64
                )
            )
        ),
        "manual_scaled64_vs_reconstructed_raw64_error": float(
            np.max(
                np.abs(
                    manual_scaled_decision64
                    -
                    reconstructed_raw_decision64
                )
            )
        ),
        "pipeline64_vs_saved_raw_x64_error": float(
            np.max(
                np.abs(
                    pipeline_decision64
                    -
                    saved_raw_decision_x64
                )
            )
        ),
        "saved_vs_reconstructed_coefficient_max_error": float(
            np.max(
                np.abs(
                    coefficient_saved.astype(
                        np.float64
                    )
                    -
                    coefficient_reconstructed64
                )
            )
        ),
        "saved_vs_reconstructed_intercept_error": float(
            abs(
                intercept_saved
                -
                intercept_reconstructed64
            )
        ),
    },
    name="value",
)

print(
    "\n================ NUMERICAL DIAGNOSTIC "
    "SUMMARY ================"
)

display(
    diagnostic.to_frame()
)


# ============================================================
# 5. Inspect the temporary NPZ storage dtype
# ============================================================

temporary_npz_path = Path(
    diagnostic_axis_artifacts[
        "axis_npz_path"
    ]
)

print(
    "\n================ TEMPORARY FROZEN NPZ "
    "ARRAY INVENTORY ================"
)

with np.load(
    temporary_npz_path,
    allow_pickle=False,
) as temporary_axis:
    for key in temporary_axis.files:
        array = temporary_axis[key]

        print(
            key,
            "shape=",
            array.shape,
            "dtype=",
            array.dtype,
        )


print("\n================ SAFETY CONFIRMATION ================")
print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)
print("Source validation performance accessed:", False)
print("Persistent axis directory modified:", False)
print("Diagnostic axis directory:", temporary_axis_root)

AssertionError: First rerun the preceding Stage 5A-CPU setup cell. Missing symbols:
SOURCE_MANIFEST_PATH
EMBEDDING_ROOT
AXIS_ROOT
FROZEN_FEATURE_DIMENSION
build_eye_unit_id
stable_table_sha256
create_probe
eye_balanced_sample_weight
save_axis

In [4]:
#@title 05A-CPU-R2. One-step runtime recovery, precision repair, and continuation

from google.colab import drive

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json
import os

import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


# ============================================================
# 0. Mount Drive
# ============================================================

drive.mount(
    "/content/drive",
    force_remount=False,
)

MYDRIVE_ROOT = Path(
    "/content/drive/MyDrive"
)

PROJECT_ROOT = (
    MYDRIVE_ROOT
    /
    "Cross-Modal_Diagnostic_Observability"
)

assert PROJECT_ROOT.is_dir(), (
    f"Project root not found: {PROJECT_ROOT}"
)


print(
    "================ STAGE 5A-CPU-R2 "
    "ONE-STEP RECOVERY ================"
)

print("\nRepair boundary:")
print(
    "Recover the exact saved Stage 5A setup and continuation "
    "from the user's notebook, then replace only the mixed-precision "
    "axis-equivalence assertion."
)

print("\nPreviously observed before this repair:")
print("EyePACS development OOF performance: True")
print("Source held-out validation performance: False")
print("Target images accessed during Stage 5A: False")
print("Target sealed-label files accessed: False")
print("Source-target transfer performance observed: False")


# ============================================================
# 1. Locate the exact original setup and R1 notebook cells
# ============================================================

def notebook_cell_source(
    cell,
):
    source = cell.get(
        "source",
        "",
    )

    if isinstance(
        source,
        str,
    ):
        return source

    return "".join(
        source
    )


search_roots = [
    PROJECT_ROOT,
    MYDRIVE_ROOT
    /
    "Colab Notebooks",
]

notebook_paths = set()

for search_root in search_roots:
    if not search_root.is_dir():
        continue

    for notebook_path in search_root.rglob(
        "*.ipynb"
    ):
        notebook_paths.add(
            notebook_path
        )


setup_candidates = []
continuation_candidates = []
same_notebook_pairs = []


for notebook_path in sorted(
    notebook_paths
):
    try:
        with notebook_path.open(
            "r",
            encoding="utf-8",
        ) as file:
            notebook = json.load(
                file
            )

    except Exception:
        continue

    notebook_setup_cells = []
    notebook_continuation_cells = []

    for cell_index, cell in enumerate(
        notebook.get(
            "cells",
            [],
        )
    ):
        if (
            cell.get(
                "cell_type"
            )
            !=
            "code"
        ):
            continue

        source = notebook_cell_source(
            cell
        )

        is_setup_cell = bool(
            "def create_probe"
            in
            source
            and
            "def save_axis"
            in
            source
            and
            "SOURCE_MANIFEST_PATH"
            in
            source
            and
            "AUC_THRESHOLD"
            in
            source
            and
            "CI_LOWER_THRESHOLD"
            in
            source
        )

        is_continuation_cell = bool(
            "STAGE 5A-CPU-R1"
            in
            source
            and
            "maximum_axis_equivalence_error"
            in
            source
            and
            "load_or_extract_embeddings_repaired"
            in
            source
        )

        if is_setup_cell:
            record = {
                "notebook_path": notebook_path,
                "cell_index": cell_index,
                "source": source,
                "modified_time": (
                    notebook_path.stat().st_mtime
                ),
            }

            setup_candidates.append(
                record
            )

            notebook_setup_cells.append(
                record
            )

        if is_continuation_cell:
            record = {
                "notebook_path": notebook_path,
                "cell_index": cell_index,
                "source": source,
                "modified_time": (
                    notebook_path.stat().st_mtime
                ),
            }

            continuation_candidates.append(
                record
            )

            notebook_continuation_cells.append(
                record
            )

    for setup_record in notebook_setup_cells:
        for continuation_record in (
            notebook_continuation_cells
        ):
            if (
                setup_record[
                    "cell_index"
                ]
                <
                continuation_record[
                    "cell_index"
                ]
            ):
                same_notebook_pairs.append(
                    (
                        setup_record,
                        continuation_record,
                    )
                )


if same_notebook_pairs:
    same_notebook_pairs.sort(
        key=lambda pair: (
            pair[1][
                "modified_time"
            ],
            pair[1][
                "cell_index"
            ],
        ),
        reverse=True,
    )

    (
        selected_setup,
        selected_continuation,
    ) = same_notebook_pairs[0]

else:
    assert setup_candidates, (
        "Could not find a saved Stage 5A setup cell containing "
        "create_probe() and save_axis() in the project or "
        "Colab Notebooks folders."
    )

    assert continuation_candidates, (
        "Could not find the saved 05A-CPU-R1 continuation cell."
    )

    setup_candidates.sort(
        key=lambda record: (
            record[
                "modified_time"
            ],
            record[
                "cell_index"
            ],
        ),
        reverse=True,
    )

    continuation_candidates.sort(
        key=lambda record: (
            record[
                "modified_time"
            ],
            record[
                "cell_index"
            ],
        ),
        reverse=True,
    )

    selected_setup = (
        setup_candidates[0]
    )

    selected_continuation = (
        continuation_candidates[0]
    )


print("\nRecovered setup notebook:")
print(
    selected_setup[
        "notebook_path"
    ]
)

print("\nRecovered setup cell index:")
print(
    selected_setup[
        "cell_index"
    ]
)

print("\nRecovered R1 notebook:")
print(
    selected_continuation[
        "notebook_path"
    ]
)

print("\nRecovered R1 cell index:")
print(
    selected_continuation[
        "cell_index"
    ]
)


# ============================================================
# 2. Execute the exact original lightweight setup
# ============================================================

setup_source = selected_setup[
    "source"
]

setup_sha256 = hashlib.sha256(
    setup_source.encode(
        "utf-8"
    )
).hexdigest()


print("\nExecuting recovered Stage 5A setup...")

setup_result = get_ipython().run_cell(
    setup_source,
    store_history=False,
)

if (
    setup_result.error_before_exec
    is not None
):
    raise setup_result.error_before_exec

if (
    setup_result.error_in_exec
    is not None
):
    raise setup_result.error_in_exec


REQUIRED_RUNTIME_SYMBOLS_R2 = [
    "SOURCE_MANIFEST_PATH",
    "EMBEDDING_ROOT",
    "AXIS_ROOT",
    "RESULT_ROOT",
    "PROTOCOL_PATH",
    "STAGE5_DECISION_PATH",
    "FROZEN_FEATURE_DIMENSION",
    "AUC_THRESHOLD",
    "CI_LOWER_THRESHOLD",
    "build_eye_unit_id",
    "stable_table_sha256",
    "create_probe",
    "eye_balanced_sample_weight",
    "save_axis",
]

missing_after_setup = [
    symbol
    for symbol in (
        REQUIRED_RUNTIME_SYMBOLS_R2
    )
    if symbol not in globals()
]

assert not missing_after_setup, (
    "Recovered setup did not define all required symbols:\n"
    +
    "\n".join(
        missing_after_setup
    )
)


# ============================================================
# 3. Patch only the faulty mixed-precision equivalence block
# ============================================================

original_r1_source = (
    selected_continuation[
        "source"
    ]
)

original_r1_sha256 = hashlib.sha256(
    original_r1_source.encode(
        "utf-8"
    )
).hexdigest()


start_marker = (
    "    verification_rows = min(\n"
)

end_marker = (
    "    # Held-out validation is observed only "
    "after axis freeze.\n"
)

assert start_marker in original_r1_source
assert end_marker in original_r1_source

block_start = original_r1_source.index(
    start_marker
)

block_end = original_r1_source.index(
    end_marker,
    block_start,
)


precision_repair_block = r'''    verification_rows = min(
        64,
        len(development_embeddings),
    )

    verification_embeddings32 = np.asarray(
        development_embeddings[
            :verification_rows
        ],
        dtype=np.float32,
    )

    verification_embeddings64 = (
        verification_embeddings32.astype(
            np.float64,
            copy=True,
        )
    )

    standardizer = final_probe.named_steps[
        "standardscaler"
    ]

    logistic_model = final_probe.named_steps[
        "logisticregression"
    ]

    scaler_mean64 = np.asarray(
        standardizer.mean_,
        dtype=np.float64,
    ).reshape(-1)

    scaler_scale64 = np.asarray(
        standardizer.scale_,
        dtype=np.float64,
    ).reshape(-1)

    scaled_coefficient64 = np.asarray(
        logistic_model.coef_[0],
        dtype=np.float64,
    ).reshape(-1)

    scaled_intercept64 = float(
        logistic_model.intercept_[0]
    )

    reconstructed_raw_coefficient64 = (
        scaled_coefficient64
        /
        scaler_scale64
    )

    reconstructed_raw_intercept64 = float(
        scaled_intercept64
        -
        np.dot(
            scaler_mean64,
            reconstructed_raw_coefficient64,
        )
    )

    # Exact mathematical check in a single float64 path.
    pipeline_decision64 = np.asarray(
        final_probe.decision_function(
            verification_embeddings64
        ),
        dtype=np.float64,
    )

    reconstructed_raw_decision64 = (
        verification_embeddings64
        @
        reconstructed_raw_coefficient64
        +
        reconstructed_raw_intercept64
    )

    maximum_axis_mathematical_error = float(
        np.max(
            np.abs(
                pipeline_decision64
                -
                reconstructed_raw_decision64
            )
        )
    )

    assert (
        maximum_axis_mathematical_error
        <
        1e-10
    ), (
        "The StandardScaler-to-raw-axis conversion is "
        "mathematically inconsistent."
    )

    stored_raw_coefficient = np.asarray(
        axis_artifacts[
            "coefficient_raw"
        ]
    ).reshape(-1)

    stored_raw_intercept = float(
        np.asarray(
            axis_artifacts[
                "intercept_raw"
            ]
        ).reshape(-1)[0]
    )

    # Check that the actually frozen axis is the same axis,
    # allowing only ordinary float32/float64 storage rounding.
    assert np.allclose(
        stored_raw_coefficient.astype(
            np.float64
        ),
        reconstructed_raw_coefficient64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The saved raw coefficient differs from the fitted "
        "pipeline by more than storage precision permits."
    )

    assert np.isclose(
        stored_raw_intercept,
        reconstructed_raw_intercept64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The saved raw intercept differs from the fitted "
        "pipeline by more than storage precision permits."
    )

    # Preserve the original mixed-precision quantity for reporting.
    pipeline_decision = np.asarray(
        final_probe.decision_function(
            verification_embeddings32
        ),
        dtype=np.float64,
    )

    raw_decision = np.asarray(
        verification_embeddings32
        @
        stored_raw_coefficient
        +
        stored_raw_intercept,
        dtype=np.float64,
    )

    maximum_axis_equivalence_error = float(
        np.max(
            np.abs(
                pipeline_decision
                -
                raw_decision
            )
        )
    )

    # This comparison crosses sklearn's float32 StandardScaler
    # path and NumPy's raw-axis dot-product path. It is therefore
    # checked with a numerical tolerance after the exact float64
    # formula and stored-axis identity checks have already passed.
    assert np.allclose(
        pipeline_decision,
        raw_decision,
        rtol=1e-5,
        atol=1e-4,
    ), (
        "The saved axis is not numerically equivalent to the "
        "pipeline even after accounting for mixed precision."
    )

    print(
        f"{dataset_name} float64 mathematical axis error: "
        f"{maximum_axis_mathematical_error:.3e}"
    )

    print(
        f"{dataset_name} mixed-precision saved-axis error: "
        f"{maximum_axis_equivalence_error:.3e}"
    )

'''


patched_r1_source = (
    original_r1_source[
        :block_start
    ]
    +
    precision_repair_block
    +
    original_r1_source[
        block_end:
    ]
)


old_observation_text = '''print(
    "\\nSource performance observed before this continuation:"
)
print(False)
'''

new_observation_text = '''print(
    "\\nSource development OOF performance observed before "
    "this precision repair:"
)
print(True)

print(
    "\\nSource held-out validation performance observed "
    "before this precision repair:"
)
print(False)
'''

if old_observation_text in patched_r1_source:
    patched_r1_source = (
        patched_r1_source.replace(
            old_observation_text,
            new_observation_text,
            1,
        )
    )


patched_r1_source = (
    patched_r1_source.replace(
        "STAGE 5A-CPU-R1 ",
        "STAGE 5A-CPU-R2 ",
        1,
    )
)


# ============================================================
# 4. Save the repair provenance before validation is observed
# ============================================================

REPAIR_RECORD_PATH = (
    RESULT_ROOT
    /
    "Stage5A_CPU_R2_Axis_Equivalence_Precision_Repair_v0.1.json"
)

repair_record = {
    "repair_id": (
        "STAGE5A_CPU_R2_AXIS_EQUIVALENCE_PRECISION_REPAIR"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "reason": (
        "The original mixed-precision equivalence assertion used "
        "a fixed absolute tolerance of 1e-6 between sklearn's "
        "float32 StandardScaler execution path and a NumPy raw-axis "
        "dot product. EyePACS development OOF was observed, but "
        "source held-out validation and all target transfer "
        "performance remained unobserved."
    ),
    "scientific_protocol_changed": False,
    "model_changed": False,
    "training_data_changed": False,
    "endpoint_changed": False,
    "source_split_changed": False,
    "target_data_access_changed": False,
    "repair_change": (
        "Replace the single mixed-precision 1e-6 assertion with "
        "three checks: exact float64 algebraic equivalence, stored "
        "axis coefficient/intercept identity within storage "
        "precision, and mixed-precision prediction equivalence "
        "using rtol=1e-5 and atol=1e-4."
    ),
    "source_development_oof_observed_before_repair": True,
    "source_validation_performance_observed_before_repair": False,
    "target_images_accessed_during_repair": False,
    "target_sealed_labels_accessed_during_repair": False,
    "source_target_transfer_performance_observed_before_repair": False,
    "recovered_setup_notebook": str(
        selected_setup[
            "notebook_path"
        ]
    ),
    "recovered_setup_cell_index": int(
        selected_setup[
            "cell_index"
        ]
    ),
    "recovered_setup_source_sha256": (
        setup_sha256
    ),
    "recovered_r1_notebook": str(
        selected_continuation[
            "notebook_path"
        ]
    ),
    "recovered_r1_cell_index": int(
        selected_continuation[
            "cell_index"
        ]
    ),
    "original_r1_source_sha256": (
        original_r1_sha256
    ),
}

with REPAIR_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_record,
        file,
        indent=2,
    )


print("\nRepair record saved:")
print(REPAIR_RECORD_PATH)

print("\nStarting repaired Stage 5A continuation...")
print(
    "Completed embedding checkpoints will be reused; "
    "image extraction should not restart."
)


# ============================================================
# 5. Execute the repaired original continuation
# ============================================================

continuation_result = get_ipython().run_cell(
    patched_r1_source,
    store_history=False,
)

if (
    continuation_result.error_before_exec
    is not None
):
    raise continuation_result.error_before_exec

if (
    continuation_result.error_in_exec
    is not None
):
    raise continuation_result.error_in_exec


print(
    "\n================ STAGE 5A-CPU-R2 "
    "RECOVERY FINISHED ================"
)

print("Repair record:")
print(REPAIR_RECORD_PATH)

print("Target images accessed by this recovery:")
print(False)

print("Target sealed-label files accessed:")
print(False)

Mounted at /content/drive
================ STAGE 5A-CPU-R2 ONE-STEP RECOVERY ================

Repair boundary:
Recover the exact saved Stage 5A setup and continuation from the user's notebook, then replace only the mixed-precision axis-equivalence assertion.

Previously observed before this repair:
EyePACS development OOF performance: True
Source held-out validation performance: False
Target images accessed during Stage 5A: False
Target sealed-label files accessed: False
Source-target transfer performance observed: False

Recovered setup notebook:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/05_Code/Retinal_DR/Retinal_DR_Prospective_Blind_Test_Design_v0.1.ipynb

Recovered setup cell index:
22

Recovered R1 notebook:
/content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/05_Code/Retinal_DR/Retinal_DR_Prospective_Blind_Test_Design_v0.1.ipynb

Recovered R1 cell index:
26

Executing recovered Stage 5A setup...
================ STAGE 5A-CPU SOURCE RECOVERABILITY AND AXI

,dataset,source_partition,images,eyes,patients,negative_images,positive_images
0,DeepDRiD,development,1200,600,300,680,520
1,DeepDRiD,validation,400,200,100,220,180
2,EyePACS_2015,development,3350,3350,1675,2061,1289
3,EyePACS_2015,validation,1118,1118,559,674,444


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 132MB/s]



EyePACS_2015: completed embeddings 4468/4468
EyePACS_2015: embedding extraction complete.

DeepDRiD: completed embeddings 1600/1600
DeepDRiD: embedding extraction complete.

================ MODELLING EyePACS_2015 ================
EyePACS_2015 fold 1 eye AUC: 0.5920
EyePACS_2015 fold 2 eye AUC: 0.6057
EyePACS_2015 fold 3 eye AUC: 0.5855
EyePACS_2015 fold 4 eye AUC: 0.5737
EyePACS_2015 fold 5 eye AUC: 0.5910
EyePACS_2015 development OOF AUC: 0.5907 [0.5702, 0.6098]


AssertionError: 

AssertionError: 

In [6]:
#@title 05A-CPU-R2C. Robust precision patch and finish Stage 5A

from pathlib import Path
from datetime import datetime, timezone

import json
import numpy as np


print(
    "================ STAGE 5A-CPU-R2C "
    "ROBUST PATCH ================"
)


# ============================================================
# 1. Numerical verifier used by the patched R1 continuation
# ============================================================

def verify_axis_equivalence_r2(
    probe,
    embeddings,
    artifacts,
    verification_rows=64,
):
    x32 = np.asarray(
        embeddings[
            :verification_rows
        ],
        dtype=np.float32,
    )

    x64 = x32.astype(
        np.float64,
        copy=True,
    )

    scaler = probe.named_steps[
        "standardscaler"
    ]

    classifier = probe.named_steps[
        "logisticregression"
    ]

    mean64 = np.asarray(
        scaler.mean_,
        dtype=np.float64,
    ).reshape(-1)

    scale64 = np.asarray(
        scaler.scale_,
        dtype=np.float64,
    ).reshape(-1)

    scaled_coefficient64 = np.asarray(
        classifier.coef_[0],
        dtype=np.float64,
    ).reshape(-1)

    scaled_intercept64 = float(
        classifier.intercept_[0]
    )

    reconstructed_coefficient64 = (
        scaled_coefficient64
        /
        scale64
    )

    reconstructed_intercept64 = float(
        scaled_intercept64
        -
        np.dot(
            mean64,
            reconstructed_coefficient64,
        )
    )

    pipeline64 = np.asarray(
        probe.decision_function(
            x64
        ),
        dtype=np.float64,
    )

    reconstructed64 = (
        x64
        @
        reconstructed_coefficient64
        +
        reconstructed_intercept64
    )

    mathematical_error = float(
        np.max(
            np.abs(
                pipeline64
                -
                reconstructed64
            )
        )
    )

    assert mathematical_error < 1e-10, (
        "The StandardScaler-to-raw-axis algebra is incorrect."
    )

    stored_coefficient = np.asarray(
        artifacts[
            "coefficient_raw"
        ]
    ).reshape(-1)

    stored_intercept = float(
        np.asarray(
            artifacts[
                "intercept_raw"
            ]
        ).reshape(-1)[0]
    )

    assert np.allclose(
        stored_coefficient.astype(
            np.float64
        ),
        reconstructed_coefficient64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored coefficient is not the fitted raw axis."
    )

    assert np.isclose(
        stored_intercept,
        reconstructed_intercept64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored intercept is not the fitted raw intercept."
    )

    pipeline32 = np.asarray(
        probe.decision_function(
            x32
        ),
        dtype=np.float64,
    )

    stored_raw32 = np.asarray(
        x32
        @
        stored_coefficient
        +
        stored_intercept,
        dtype=np.float64,
    )

    mixed_precision_error = float(
        np.max(
            np.abs(
                pipeline32
                -
                stored_raw32
            )
        )
    )

    assert np.allclose(
        pipeline32,
        stored_raw32,
        rtol=1e-5,
        atol=1e-4,
    ), (
        "The saved axis is inconsistent beyond ordinary "
        "mixed-precision rounding."
    )

    return (
        mathematical_error,
        mixed_precision_error,
    )


# ============================================================
# 2. Recover the exact R1 continuation source
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/"
    "Cross-Modal_Diagnostic_Observability/"
    "05_Code/Retinal_DR/"
    "Retinal_DR_Prospective_Blind_Test_Design_v0.1.ipynb"
)

R1_CELL_INDEX = 26

assert NOTEBOOK_PATH.is_file()

with NOTEBOOK_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook = json.load(file)

source_field = notebook[
    "cells"
][
    R1_CELL_INDEX
].get(
    "source",
    "",
)

if isinstance(
    source_field,
    str,
):
    original_source = source_field
else:
    original_source = "".join(
        source_field
    )

assert (
    "maximum_axis_equivalence_error = float("
    in
    original_source
)

assert (
    "# Held-out validation is observed only after axis freeze."
    in
    original_source
)


# ============================================================
# 3. Replace the old assertion by its position
# ============================================================

error_calculation_position = (
    original_source.index(
        "maximum_axis_equivalence_error = float("
    )
)

assertion_start = original_source.index(
    "    assert",
    error_calculation_position,
)

heldout_comment_position = (
    original_source.index(
        "    # Held-out validation is observed only "
        "after axis freeze.",
        assertion_start,
    )
)


replacement = r'''    (
        maximum_axis_mathematical_error,
        maximum_axis_equivalence_error,
    ) = verify_axis_equivalence_r2(
        probe=final_probe,
        embeddings=development_embeddings,
        artifacts=axis_artifacts,
        verification_rows=verification_rows,
    )

    print(
        f"{dataset_name} float64 mathematical axis error: "
        f"{maximum_axis_mathematical_error:.3e}"
    )

    print(
        f"{dataset_name} mixed-precision saved-axis error: "
        f"{maximum_axis_equivalence_error:.3e}"
    )

'''


patched_source = (
    original_source[
        :assertion_start
    ]
    +
    replacement
    +
    original_source[
        heldout_comment_position:
    ]
)

patched_source = patched_source.replace(
    "STAGE 5A-CPU-R1 ",
    "STAGE 5A-CPU-R2C ",
    1,
)


# Final checks that the old assertion is gone.
old_assertion_region = patched_source[
    error_calculation_position:
    patched_source.index(
        "    # Held-out validation is observed only "
        "after axis freeze.",
        error_calculation_position,
    )
]

assert (
    "1e-6"
    not in
    old_assertion_region
)

assert (
    "verify_axis_equivalence_r2"
    in
    old_assertion_region
)

print("Old assertion removed:", True)
print("Float64 mathematical check installed:", True)
print("Stored-axis identity check installed:", True)
print("Mixed-precision check installed:", True)


# ============================================================
# 4. Write the repair record before source validation
# ============================================================

REPAIR_RECORD_PATH = (
    RESULT_ROOT
    /
    "Stage5A_CPU_R2C_Axis_Equivalence_Repair_v0.1.json"
)

repair_record = {
    "repair_id": (
        "STAGE5A_CPU_R2C_AXIS_EQUIVALENCE_REPAIR"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scientific_protocol_changed": False,
    "model_changed": False,
    "training_data_changed": False,
    "source_partition_changed": False,
    "endpoint_changed": False,
    "repair": (
        "The mixed-precision 1e-6 assertion was replaced by "
        "an exact float64 algebra check, a stored-axis identity "
        "check, and a mixed-precision prediction-equivalence check."
    ),
    "eyepacs_development_oof_observed_before_repair": True,
    "source_validation_observed_before_repair": False,
    "target_images_accessed": False,
    "target_sealed_labels_accessed": False,
    "source_target_transfer_performance_observed": False,
}

with REPAIR_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_record,
        file,
        indent=2,
    )

print("Repair record saved:", REPAIR_RECORD_PATH)


# ============================================================
# 5. Execute the patched continuation
# ============================================================

print(
    "\nStarting repaired continuation. "
    "Existing embeddings will be reused."
)

result = get_ipython().run_cell(
    patched_source,
    store_history=False,
)

if result.error_before_exec is not None:
    raise result.error_before_exec

if result.error_in_exec is not None:
    raise result.error_in_exec


print(
    "\n================ STAGE 5A-CPU-R2C "
    "FINISHED ================"
)

print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)

================ STAGE 5A-CPU-R2C ROBUST PATCH ================


AssertionError: 

In [5]:
#@title 05A-CPU-R2B. Apply precision repair before execution and finish Stage 5A

from pathlib import Path
from datetime import datetime, timezone

import hashlib
import json

import numpy as np


# ============================================================
# 0. Confirm that the failed setup execution restored runtime
# ============================================================

REQUIRED_SYMBOLS = [
    "SOURCE_MANIFEST_PATH",
    "EMBEDDING_ROOT",
    "AXIS_ROOT",
    "RESULT_ROOT",
    "PROTOCOL_PATH",
    "STAGE5_DECISION_PATH",
    "FROZEN_FEATURE_DIMENSION",
    "create_probe",
    "save_axis",
]

missing_symbols = [
    symbol
    for symbol in REQUIRED_SYMBOLS
    if symbol not in globals()
]

assert not missing_symbols, (
    "Runtime symbols are missing. Do not reconnect before "
    "running this cell.\nMissing:\n"
    +
    "\n".join(missing_symbols)
)


print(
    "================ STAGE 5A-CPU-R2B "
    "PRECISION REPAIR ================"
)

print("\nRuntime recovered:", True)
print("Completed embeddings will be reused:", True)
print("Source validation observed before repair:", False)
print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)


# ============================================================
# 1. Read the exact original R1 continuation cell
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/"
    "Cross-Modal_Diagnostic_Observability/"
    "05_Code/Retinal_DR/"
    "Retinal_DR_Prospective_Blind_Test_Design_v0.1.ipynb"
)

R1_CELL_INDEX = 26

assert NOTEBOOK_PATH.is_file()

with NOTEBOOK_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook = json.load(file)

cell = notebook[
    "cells"
][
    R1_CELL_INDEX
]

assert cell[
    "cell_type"
] == "code"

source_field = cell.get(
    "source",
    "",
)

if isinstance(
    source_field,
    str,
):
    original_r1_source = source_field
else:
    original_r1_source = "".join(
        source_field
    )

assert (
    "STAGE 5A-CPU-R1"
    in
    original_r1_source
)

assert (
    "maximum_axis_equivalence_error"
    in
    original_r1_source
)

original_r1_sha256 = hashlib.sha256(
    original_r1_source.encode(
        "utf-8"
    )
).hexdigest()


# ============================================================
# 2. Replace only the faulty assertion
# ============================================================

old_assertion = '''    assert (
        maximum_axis_equivalence_error
        <
        1e-6
    )
'''

new_assertion = r'''    # R2 precision repair:
    # First verify the StandardScaler-to-raw-axis algebra in one
    # internally consistent float64 calculation path.

    standardizer = final_probe.named_steps[
        "standardscaler"
    ]

    logistic_model = final_probe.named_steps[
        "logisticregression"
    ]

    verification_embeddings64 = np.asarray(
        development_embeddings[
            :verification_rows
        ],
        dtype=np.float64,
    )

    scaler_mean64 = np.asarray(
        standardizer.mean_,
        dtype=np.float64,
    ).reshape(-1)

    scaler_scale64 = np.asarray(
        standardizer.scale_,
        dtype=np.float64,
    ).reshape(-1)

    scaled_coefficient64 = np.asarray(
        logistic_model.coef_[0],
        dtype=np.float64,
    ).reshape(-1)

    scaled_intercept64 = float(
        logistic_model.intercept_[0]
    )

    reconstructed_raw_coefficient64 = (
        scaled_coefficient64
        /
        scaler_scale64
    )

    reconstructed_raw_intercept64 = float(
        scaled_intercept64
        -
        np.dot(
            scaler_mean64,
            reconstructed_raw_coefficient64,
        )
    )

    pipeline_decision64 = np.asarray(
        final_probe.decision_function(
            verification_embeddings64
        ),
        dtype=np.float64,
    )

    reconstructed_raw_decision64 = (
        verification_embeddings64
        @
        reconstructed_raw_coefficient64
        +
        reconstructed_raw_intercept64
    )

    maximum_axis_mathematical_error = float(
        np.max(
            np.abs(
                pipeline_decision64
                -
                reconstructed_raw_decision64
            )
        )
    )

    assert (
        maximum_axis_mathematical_error
        <
        1e-10
    ), (
        "The fitted pipeline and reconstructed raw axis "
        "are not mathematically equivalent."
    )

    stored_raw_coefficient64 = np.asarray(
        axis_artifacts[
            "coefficient_raw"
        ],
        dtype=np.float64,
    ).reshape(-1)

    stored_raw_intercept64 = float(
        np.asarray(
            axis_artifacts[
                "intercept_raw"
            ]
        ).reshape(-1)[0]
    )

    # Verify that save_axis returned the same coefficient and
    # intercept, allowing only storage-dtype rounding.
    assert np.allclose(
        stored_raw_coefficient64,
        reconstructed_raw_coefficient64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored coefficient is not the fitted raw axis."
    )

    assert np.isclose(
        stored_raw_intercept64,
        reconstructed_raw_intercept64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored intercept is not the fitted raw intercept."
    )

    # The original comparison crosses sklearn's float32
    # StandardScaler path and NumPy's raw dot-product path.
    # After the two strict checks above pass, use an appropriate
    # mixed-precision prediction tolerance.
    assert np.allclose(
        pipeline_decision,
        raw_decision,
        rtol=1e-5,
        atol=1e-4,
    ), (
        "The frozen axis remains inconsistent even after "
        "accounting for ordinary mixed-precision rounding."
    )

    print(
        f"{dataset_name} float64 mathematical axis error: "
        f"{maximum_axis_mathematical_error:.3e}"
    )

    print(
        f"{dataset_name} mixed-precision saved-axis error: "
        f"{maximum_axis_equivalence_error:.3e}"
    )
'''

assert (
    original_r1_source.count(
        old_assertion
    )
    ==
    1
), (
    "The exact old assertion was not uniquely located."
)

patched_r1_source = (
    original_r1_source.replace(
        old_assertion,
        new_assertion,
        1,
    )
)

patched_r1_source = (
    patched_r1_source.replace(
        "STAGE 5A-CPU-R1 ",
        "STAGE 5A-CPU-R2B ",
        1,
    )
)


old_observation = '''print(
    "\\nSource performance observed before this continuation:"
)
print(False)
'''

new_observation = '''print(
    "\\nEyePACS development OOF observed before "
    "this precision repair:"
)
print(True)

print(
    "\\nSource held-out validation performance observed "
    "before this precision repair:"
)
print(False)
'''

if old_observation in patched_r1_source:
    patched_r1_source = (
        patched_r1_source.replace(
            old_observation,
            new_observation,
            1,
        )
    )


assert (
    "maximum_axis_equivalence_error\n"
    "        <\n"
    "        1e-6"
    not in
    patched_r1_source
)

assert (
    "maximum_axis_mathematical_error"
    in
    patched_r1_source
)

print("\nOld assertion removed:", True)
print("Strict float64 algebra check added:", True)
print("Stored-axis identity check added:", True)
print("Mixed-precision tolerance added:", True)


# ============================================================
# 3. Save repair record before validation scoring
# ============================================================

REPAIR_RECORD_PATH = (
    RESULT_ROOT
    /
    "Stage5A_CPU_R2B_Axis_Equivalence_Repair_v0.1.json"
)

repair_record = {
    "repair_id": (
        "STAGE5A_CPU_R2B_AXIS_EQUIVALENCE_REPAIR"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "original_notebook_path": str(
        NOTEBOOK_PATH
    ),
    "original_cell_index": int(
        R1_CELL_INDEX
    ),
    "original_cell_sha256": (
        original_r1_sha256
    ),
    "scientific_protocol_changed": False,
    "model_changed": False,
    "training_data_changed": False,
    "source_split_changed": False,
    "endpoint_changed": False,
    "repair": (
        "Replaced the mixed-precision absolute-error assertion "
        "of 1e-6 with an exact float64 algebra check, a stored-axis "
        "identity check, and an rtol=1e-5/atol=1e-4 numerical "
        "prediction-equivalence check."
    ),
    "eyepacs_development_oof_observed_before_repair": True,
    "source_validation_observed_before_repair": False,
    "target_images_accessed": False,
    "target_sealed_labels_accessed": False,
    "source_target_transfer_performance_observed": False,
}

with REPAIR_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_record,
        file,
        indent=2,
    )

print("\nRepair record:")
print(REPAIR_RECORD_PATH)


# ============================================================
# 4. Execute the patched continuation
# ============================================================

print(
    "\nStarting patched Stage 5A continuation. "
    "Existing embeddings should load immediately."
)

execution_result = get_ipython().run_cell(
    patched_r1_source,
    store_history=False,
)

if (
    execution_result.error_before_exec
    is not None
):
    raise execution_result.error_before_exec

if (
    execution_result.error_in_exec
    is not None
):
    raise execution_result.error_in_exec


print(
    "\n================ STAGE 5A-CPU-R2B "
    "FINISHED ================"
)

print("Repair record:")
print(REPAIR_RECORD_PATH)

print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)

================ STAGE 5A-CPU-R2B PRECISION REPAIR ================

Runtime recovered: True
Completed embeddings will be reused: True
Source validation observed before repair: False
Target images accessed: False
Target sealed-label files accessed: False


AssertionError: The exact old assertion was not uniquely located.

In [7]:
#@title 05A-CPU-R2C. Robust precision patch and finish Stage 5A

from pathlib import Path
from datetime import datetime, timezone

import json
import numpy as np


print(
    "================ STAGE 5A-CPU-R2C "
    "ROBUST PATCH ================"
)


# ============================================================
# 1. Numerical verifier used by the patched R1 continuation
# ============================================================

def verify_axis_equivalence_r2(
    probe,
    embeddings,
    artifacts,
    verification_rows=64,
):
    x32 = np.asarray(
        embeddings[
            :verification_rows
        ],
        dtype=np.float32,
    )

    x64 = x32.astype(
        np.float64,
        copy=True,
    )

    scaler = probe.named_steps[
        "standardscaler"
    ]

    classifier = probe.named_steps[
        "logisticregression"
    ]

    mean64 = np.asarray(
        scaler.mean_,
        dtype=np.float64,
    ).reshape(-1)

    scale64 = np.asarray(
        scaler.scale_,
        dtype=np.float64,
    ).reshape(-1)

    scaled_coefficient64 = np.asarray(
        classifier.coef_[0],
        dtype=np.float64,
    ).reshape(-1)

    scaled_intercept64 = float(
        classifier.intercept_[0]
    )

    reconstructed_coefficient64 = (
        scaled_coefficient64
        /
        scale64
    )

    reconstructed_intercept64 = float(
        scaled_intercept64
        -
        np.dot(
            mean64,
            reconstructed_coefficient64,
        )
    )

    pipeline64 = np.asarray(
        probe.decision_function(
            x64
        ),
        dtype=np.float64,
    )

    reconstructed64 = (
        x64
        @
        reconstructed_coefficient64
        +
        reconstructed_intercept64
    )

    mathematical_error = float(
        np.max(
            np.abs(
                pipeline64
                -
                reconstructed64
            )
        )
    )

    assert mathematical_error < 1e-10, (
        "The StandardScaler-to-raw-axis algebra is incorrect."
    )

    stored_coefficient = np.asarray(
        artifacts[
            "coefficient_raw"
        ]
    ).reshape(-1)

    stored_intercept = float(
        np.asarray(
            artifacts[
                "intercept_raw"
            ]
        ).reshape(-1)[0]
    )

    assert np.allclose(
        stored_coefficient.astype(
            np.float64
        ),
        reconstructed_coefficient64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored coefficient is not the fitted raw axis."
    )

    assert np.isclose(
        stored_intercept,
        reconstructed_intercept64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored intercept is not the fitted raw intercept."
    )

    pipeline32 = np.asarray(
        probe.decision_function(
            x32
        ),
        dtype=np.float64,
    )

    stored_raw32 = np.asarray(
        x32
        @
        stored_coefficient
        +
        stored_intercept,
        dtype=np.float64,
    )

    mixed_precision_error = float(
        np.max(
            np.abs(
                pipeline32
                -
                stored_raw32
            )
        )
    )

    assert np.allclose(
        pipeline32,
        stored_raw32,
        rtol=1e-5,
        atol=1e-4,
    ), (
        "The saved axis is inconsistent beyond ordinary "
        "mixed-precision rounding."
    )

    return (
        mathematical_error,
        mixed_precision_error,
    )


# ============================================================
# 2. Recover the exact R1 continuation source
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/"
    "Cross-Modal_Diagnostic_Observability/"
    "05_Code/Retinal_DR/"
    "Retinal_DR_Prospective_Blind_Test_Design_v0.1.ipynb"
)

R1_CELL_INDEX = 26

assert NOTEBOOK_PATH.is_file()

with NOTEBOOK_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook = json.load(file)

source_field = notebook[
    "cells"
][
    R1_CELL_INDEX
].get(
    "source",
    "",
)

if isinstance(
    source_field,
    str,
):
    original_source = source_field
else:
    original_source = "".join(
        source_field
    )

assert (
    "maximum_axis_equivalence_error = float("
    in
    original_source
)

assert (
    "# Held-out validation is observed only after axis freeze."
    in
    original_source
)


# ============================================================
# 3. Replace the old assertion by its position
# ============================================================

error_calculation_position = (
    original_source.index(
        "maximum_axis_equivalence_error = float("
    )
)

assertion_start = original_source.index(
    "    assert",
    error_calculation_position,
)

heldout_comment_position = (
    original_source.index(
        "    # Held-out validation is observed only "
        "after axis freeze.",
        assertion_start,
    )
)


replacement = r'''    (
        maximum_axis_mathematical_error,
        maximum_axis_equivalence_error,
    ) = verify_axis_equivalence_r2(
        probe=final_probe,
        embeddings=development_embeddings,
        artifacts=axis_artifacts,
        verification_rows=verification_rows,
    )

    print(
        f"{dataset_name} float64 mathematical axis error: "
        f"{maximum_axis_mathematical_error:.3e}"
    )

    print(
        f"{dataset_name} mixed-precision saved-axis error: "
        f"{maximum_axis_equivalence_error:.3e}"
    )

'''


patched_source = (
    original_source[
        :assertion_start
    ]
    +
    replacement
    +
    original_source[
        heldout_comment_position:
    ]
)

patched_source = patched_source.replace(
    "STAGE 5A-CPU-R1 ",
    "STAGE 5A-CPU-R2C ",
    1,
)


# Final checks that the old assertion is gone.
old_assertion_region = patched_source[
    error_calculation_position:
    patched_source.index(
        "    # Held-out validation is observed only "
        "after axis freeze.",
        error_calculation_position,
    )
]

assert (
    "1e-6"
    not in
    old_assertion_region
)

assert (
    "verify_axis_equivalence_r2"
    in
    old_assertion_region
)

print("Old assertion removed:", True)
print("Float64 mathematical check installed:", True)
print("Stored-axis identity check installed:", True)
print("Mixed-precision check installed:", True)


# ============================================================
# 4. Write the repair record before source validation
# ============================================================

REPAIR_RECORD_PATH = (
    RESULT_ROOT
    /
    "Stage5A_CPU_R2C_Axis_Equivalence_Repair_v0.1.json"
)

repair_record = {
    "repair_id": (
        "STAGE5A_CPU_R2C_AXIS_EQUIVALENCE_REPAIR"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scientific_protocol_changed": False,
    "model_changed": False,
    "training_data_changed": False,
    "source_partition_changed": False,
    "endpoint_changed": False,
    "repair": (
        "The mixed-precision 1e-6 assertion was replaced by "
        "an exact float64 algebra check, a stored-axis identity "
        "check, and a mixed-precision prediction-equivalence check."
    ),
    "eyepacs_development_oof_observed_before_repair": True,
    "source_validation_observed_before_repair": False,
    "target_images_accessed": False,
    "target_sealed_labels_accessed": False,
    "source_target_transfer_performance_observed": False,
}

with REPAIR_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_record,
        file,
        indent=2,
    )

print("Repair record saved:", REPAIR_RECORD_PATH)


# ============================================================
# 5. Execute the patched continuation
# ============================================================

print(
    "\nStarting repaired continuation. "
    "Existing embeddings will be reused."
)

result = get_ipython().run_cell(
    patched_source,
    store_history=False,
)

if result.error_before_exec is not None:
    raise result.error_before_exec

if result.error_in_exec is not None:
    raise result.error_in_exec


print(
    "\n================ STAGE 5A-CPU-R2C "
    "FINISHED ================"
)

print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)

================ STAGE 5A-CPU-R2C ROBUST PATCH ================


AssertionError: 

In [8]:
#@title 05A-CPU-R2D. AST-based assertion replacement and continuation

import ast
import json

from pathlib import Path
from datetime import datetime, timezone


print(
    "================ STAGE 5A-CPU-R2D "
    "AST PATCH ================"
)

assert (
    "verify_axis_equivalence_r2"
    in
    globals()
), (
    "The R2C verifier is missing. The runtime must not be "
    "reconnected between R2C and R2D."
)

assert "RESULT_ROOT" in globals()
assert "create_probe" in globals()
assert "save_axis" in globals()


# ============================================================
# 1. Reload the exact original continuation
# ============================================================

NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/"
    "Cross-Modal_Diagnostic_Observability/"
    "05_Code/Retinal_DR/"
    "Retinal_DR_Prospective_Blind_Test_Design_v0.1.ipynb"
)

R1_CELL_INDEX = 26

with NOTEBOOK_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook = json.load(file)

source_field = notebook[
    "cells"
][
    R1_CELL_INDEX
].get(
    "source",
    "",
)

if isinstance(
    source_field,
    str,
):
    original_source = source_field
else:
    original_source = "".join(
        source_field
    )


# ============================================================
# 2. Locate the assertion through the Python syntax tree
# ============================================================

syntax_tree = ast.parse(
    original_source
)

matching_assertions = []

for node in ast.walk(
    syntax_tree
):
    if not isinstance(
        node,
        ast.Assert,
    ):
        continue

    referenced_names = {
        child.id
        for child in ast.walk(
            node.test
        )
        if isinstance(
            child,
            ast.Name,
        )
    }

    if (
        "maximum_axis_equivalence_error"
        in
        referenced_names
    ):
        matching_assertions.append(
            node
        )

assert len(
    matching_assertions
) == 1, (
    "Expected exactly one assertion involving "
    "maximum_axis_equivalence_error, found "
    f"{len(matching_assertions)}."
)

assertion_node = (
    matching_assertions[0]
)

assert assertion_node.end_lineno is not None

source_lines = original_source.splitlines(
    keepends=True
)

original_assertion_text = "".join(
    source_lines[
        assertion_node.lineno - 1:
        assertion_node.end_lineno
    ]
)

print("\nLocated original assertion:")
print(original_assertion_text)


# ============================================================
# 3. Replace that exact syntax node
# ============================================================

replacement = r'''    (
        maximum_axis_mathematical_error,
        maximum_axis_equivalence_error,
    ) = verify_axis_equivalence_r2(
        probe=final_probe,
        embeddings=development_embeddings,
        artifacts=axis_artifacts,
        verification_rows=verification_rows,
    )

    print(
        f"{dataset_name} float64 mathematical axis error: "
        f"{maximum_axis_mathematical_error:.3e}"
    )

    print(
        f"{dataset_name} mixed-precision saved-axis error: "
        f"{maximum_axis_equivalence_error:.3e}"
    )
'''

patched_lines = (
    source_lines[
        :assertion_node.lineno - 1
    ]
    +
    [
        replacement
    ]
    +
    source_lines[
        assertion_node.end_lineno:
    ]
)

patched_source = "".join(
    patched_lines
)

patched_source = patched_source.replace(
    "STAGE 5A-CPU-R1 ",
    "STAGE 5A-CPU-R2D ",
    1,
)


# Confirm through a second AST parse that the old assertion is gone.
patched_tree = ast.parse(
    patched_source
)

remaining_assertions = []

for node in ast.walk(
    patched_tree
):
    if not isinstance(
        node,
        ast.Assert,
    ):
        continue

    referenced_names = {
        child.id
        for child in ast.walk(
            node.test
        )
        if isinstance(
            child,
            ast.Name,
        )
    }

    if (
        "maximum_axis_equivalence_error"
        in
        referenced_names
    ):
        remaining_assertions.append(
            node
        )

assert not remaining_assertions

assert (
    "verify_axis_equivalence_r2("
    in
    patched_source
)

print("AST assertion located:", True)
print("Old assertion removed:", True)
print("R2 verifier inserted:", True)
print("Patched source syntax valid:", True)


# ============================================================
# 4. Save repair record before validation
# ============================================================

REPAIR_RECORD_PATH = (
    RESULT_ROOT
    /
    "Stage5A_CPU_R2D_Axis_Equivalence_Repair_v0.1.json"
)

repair_record = {
    "repair_id": (
        "STAGE5A_CPU_R2D_AXIS_EQUIVALENCE_REPAIR"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "scientific_protocol_changed": False,
    "model_changed": False,
    "training_data_changed": False,
    "source_partition_changed": False,
    "endpoint_changed": False,
    "original_assertion": (
        original_assertion_text
    ),
    "repair": (
        "AST-located mixed-precision assertion replaced by "
        "the previously defined strict float64 algebra, "
        "stored-axis identity and numerical-equivalence verifier."
    ),
    "eyepacs_development_oof_observed_before_repair": True,
    "source_validation_observed_before_repair": False,
    "target_images_accessed": False,
    "target_sealed_labels_accessed": False,
    "source_target_transfer_performance_observed": False,
}

with REPAIR_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_record,
        file,
        indent=2,
    )

print("Repair record saved:", REPAIR_RECORD_PATH)


# ============================================================
# 5. Execute the already-patched continuation
# ============================================================

print(
    "\nStarting patched Stage 5A continuation. "
    "Completed embeddings will be reused."
)

result = get_ipython().run_cell(
    patched_source,
    store_history=False,
)

if result.error_before_exec is not None:
    raise result.error_before_exec

if result.error_in_exec is not None:
    raise result.error_in_exec


print(
    "\n================ STAGE 5A-CPU-R2D "
    "FINISHED ================"
)

print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)

================ STAGE 5A-CPU-R2D AST PATCH ================


AssertionError: Expected exactly one assertion involving maximum_axis_equivalence_error, found 0.

In [9]:
#@title 05A-CPU-R2E. Find the actual executable assertion, patch it, and finish

import ast
import json

from pathlib import Path
from datetime import datetime, timezone


print(
    "================ STAGE 5A-CPU-R2E "
    "ACTUAL-CELL PATCH ================"
)

assert (
    "verify_axis_equivalence_r2"
    in
    globals()
), (
    "The verifier defined by R2C is missing. "
    "Do not reconnect before running this cell."
)


NOTEBOOK_PATH = Path(
    "/content/drive/MyDrive/"
    "Cross-Modal_Diagnostic_Observability/"
    "05_Code/Retinal_DR/"
    "Retinal_DR_Prospective_Blind_Test_Design_v0.1.ipynb"
)

assert NOTEBOOK_PATH.is_file()

with NOTEBOOK_PATH.open(
    "r",
    encoding="utf-8",
) as file:
    notebook = json.load(file)


def get_cell_source(
    cell,
):
    source = cell.get(
        "source",
        "",
    )

    if isinstance(
        source,
        str,
    ):
        return source

    return "".join(
        source
    )


# ============================================================
# 1. Search every executable cell for the real AST assertion
# ============================================================

candidates = []

for cell_index, cell in enumerate(
    notebook.get(
        "cells",
        [],
    )
):
    if (
        cell.get(
            "cell_type"
        )
        !=
        "code"
    ):
        continue

    source = get_cell_source(
        cell
    )

    if (
        "maximum_axis_equivalence_error"
        not in
        source
    ):
        continue

    try:
        tree = ast.parse(
            source
        )
    except SyntaxError:
        continue

    matching_nodes = []

    for node in ast.walk(
        tree
    ):
        if not isinstance(
            node,
            ast.Assert,
        ):
            continue

        referenced_names = {
            child.id
            for child in ast.walk(
                node.test
            )
            if isinstance(
                child,
                ast.Name,
            )
        }

        if (
            "maximum_axis_equivalence_error"
            in
            referenced_names
        ):
            matching_nodes.append(
                node
            )

    if matching_nodes:
        candidates.append(
            {
                "cell_index": cell_index,
                "source": source,
                "nodes": matching_nodes,
                "contains_save_axis": (
                    "def save_axis"
                    in
                    source
                ),
                "contains_embedding_function": (
                    "load_or_extract_embeddings"
                    in
                    source
                ),
            }
        )


assert candidates, (
    "No executable notebook cell containing the real "
    "maximum_axis_equivalence_error assertion was found."
)

# Prefer the full executable Stage 5A cell containing both
# save_axis and embedding extraction.
candidates.sort(
    key=lambda record: (
        record[
            "contains_save_axis"
        ],
        record[
            "contains_embedding_function"
        ],
        -record[
            "cell_index"
        ],
    ),
    reverse=True,
)

selected = candidates[0]

assert len(
    selected[
        "nodes"
    ]
) == 1

selected_cell_index = selected[
    "cell_index"
]

original_source = selected[
    "source"
]

assertion_node = selected[
    "nodes"
][0]

assert assertion_node.end_lineno is not None

print("Actual executable cell found:", selected_cell_index)
print("Contains save_axis:", selected["contains_save_axis"])
print(
    "Contains embedding function:",
    selected["contains_embedding_function"],
)


# ============================================================
# 2. Replace the real AST assertion
# ============================================================

source_lines = original_source.splitlines(
    keepends=True
)

original_assertion = "".join(
    source_lines[
        assertion_node.lineno - 1:
        assertion_node.end_lineno
    ]
)

print("\nRemoving actual assertion:")
print(original_assertion)


replacement = r'''    (
        maximum_axis_mathematical_error,
        maximum_axis_equivalence_error,
    ) = verify_axis_equivalence_r2(
        probe=final_probe,
        embeddings=development_embeddings,
        artifacts=axis_artifacts,
        verification_rows=verification_rows,
    )

    print(
        f"{dataset_name} float64 mathematical axis error: "
        f"{maximum_axis_mathematical_error:.3e}"
    )

    print(
        f"{dataset_name} mixed-precision saved-axis error: "
        f"{maximum_axis_equivalence_error:.3e}"
    )
'''


patched_lines = (
    source_lines[
        :assertion_node.lineno - 1
    ]
    +
    [
        replacement
    ]
    +
    source_lines[
        assertion_node.end_lineno:
    ]
)

patched_source = "".join(
    patched_lines
)


# Confirm the patched source remains valid Python.
patched_tree = ast.parse(
    patched_source
)

remaining_old_assertions = []

for node in ast.walk(
    patched_tree
):
    if not isinstance(
        node,
        ast.Assert,
    ):
        continue

    names = {
        child.id
        for child in ast.walk(
            node.test
        )
        if isinstance(
            child,
            ast.Name,
        )
    }

    if (
        "maximum_axis_equivalence_error"
        in
        names
    ):
        remaining_old_assertions.append(
            node
        )

assert not remaining_old_assertions

assert (
    "verify_axis_equivalence_r2("
    in
    patched_source
)

print("\nActual assertion removed:", True)
print("Verifier inserted before execution:", True)
print("Patched cell syntax valid:", True)


# ============================================================
# 3. Save the final repair record
# ============================================================

REPAIR_RECORD_PATH = (
    RESULT_ROOT
    /
    "Stage5A_CPU_R2E_Actual_Cell_Repair_v0.1.json"
)

repair_record = {
    "repair_id": (
        "STAGE5A_CPU_R2E_ACTUAL_CELL_REPAIR"
    ),
    "created_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "notebook_path": str(
        NOTEBOOK_PATH
    ),
    "actual_executable_cell_index": int(
        selected_cell_index
    ),
    "original_assertion": (
        original_assertion
    ),
    "scientific_protocol_changed": False,
    "model_changed": False,
    "data_changed": False,
    "source_validation_observed_before_repair": False,
    "target_images_accessed": False,
    "target_sealed_labels_accessed": False,
}

with REPAIR_RECORD_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        repair_record,
        file,
        indent=2,
    )

print("Repair record:", REPAIR_RECORD_PATH)


# ============================================================
# 4. Execute the patched real Stage 5A cell
# ============================================================

print(
    "\nExecuting the patched actual Stage 5A cell. "
    "Existing embeddings will be reused."
)

result = get_ipython().run_cell(
    patched_source,
    store_history=False,
)

if result.error_before_exec is not None:
    raise result.error_before_exec

if result.error_in_exec is not None:
    raise result.error_in_exec


print(
    "\n================ STAGE 5A-CPU-R2E "
    "FINISHED ================"
)

print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)

================ STAGE 5A-CPU-R2E ACTUAL-CELL PATCH ================
Actual executable cell found: 22
Contains save_axis: True
Contains embedding function: True

Removing actual assertion:
    assert (
        maximum_axis_equivalence_error
        <
        1e-6
    )


Actual assertion removed: True
Verifier inserted before execution: True
Patched cell syntax valid: True
Repair record: /content/drive/MyDrive/Cross-Modal_Diagnostic_Observability/06_Data_Records/Retinal_DR/Prospective_Retinal_Blind_Test_v0.1/Stage5_Source_Recoverability_And_Axis_Freeze_v0.1/03_Results/Stage5A_CPU_R2E_Actual_Cell_Repair_v0.1.json

Executing the patched actual Stage 5A cell. Existing embeddings will be reused.
================ STAGE 5A-CPU SOURCE RECOVERABILITY AND AXIS FREEZE ================

Imported Stage 4G decision:
PASS_CANONICAL_SOURCE_FINALISATION_ADVANCE_TO_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE

Execution device:
cpu

CPU threads:
2

Embedding batch size:
16

DataLoader workers:
1

Frozen sourc

,dataset,source_partition,images,eyes,patients,negative_images,positive_images
0,DeepDRiD,development,1200,600,300,680,520
1,DeepDRiD,validation,400,200,100,220,180
2,EyePACS_2015,development,3350,3350,1675,2061,1289
3,EyePACS_2015,validation,1118,1118,559,674,444



EyePACS_2015: completed embeddings 4468/4468
EyePACS_2015: embedding extraction complete.

DeepDRiD: completed embeddings 1600/1600
DeepDRiD: embedding extraction complete.

================ MODELLING EyePACS_2015 ================
EyePACS_2015 fold 1 eye AUC: 0.5920
EyePACS_2015 fold 2 eye AUC: 0.6057
EyePACS_2015 fold 3 eye AUC: 0.5855
EyePACS_2015 fold 4 eye AUC: 0.5737
EyePACS_2015 fold 5 eye AUC: 0.5910
EyePACS_2015 development OOF AUC: 0.5907 [0.5702, 0.6098]


KeyError: 'standardscaler'

KeyError: 'standardscaler'

In [10]:
#@title 05A-CPU-R2F. Repair pipeline-step lookup and execute patched Stage 5A

import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler


print(
    "================ STAGE 5A-CPU-R2F "
    "DYNAMIC STEP LOOKUP ================"
)

assert "patched_source" in globals()
assert "final_probe" in globals()

print(
    "Actual fitted pipeline steps:",
    list(
        final_probe.named_steps.keys()
    ),
)


def verify_axis_equivalence_r2(
    probe,
    embeddings,
    artifacts,
    verification_rows=64,
):
    # Locate fitted components by class instead of assuming
    # sklearn-generated step names.
    scaler_candidates = [
        (
            step_name,
            step_object,
        )
        for (
            step_name,
            step_object,
        ) in probe.steps
        if isinstance(
            step_object,
            StandardScaler,
        )
    ]

    classifier_candidates = [
        (
            step_name,
            step_object,
        )
        for (
            step_name,
            step_object,
        ) in probe.steps
        if isinstance(
            step_object,
            LogisticRegression,
        )
    ]

    assert len(
        scaler_candidates
    ) == 1, (
        "Expected exactly one fitted StandardScaler; found "
        f"{len(scaler_candidates)}."
    )

    assert len(
        classifier_candidates
    ) == 1, (
        "Expected exactly one fitted LogisticRegression; found "
        f"{len(classifier_candidates)}."
    )

    (
        scaler_step_name,
        scaler,
    ) = scaler_candidates[0]

    (
        classifier_step_name,
        classifier,
    ) = classifier_candidates[0]

    x32 = np.asarray(
        embeddings[
            :verification_rows
        ],
        dtype=np.float32,
    )

    x64 = x32.astype(
        np.float64,
        copy=True,
    )

    mean64 = np.asarray(
        scaler.mean_,
        dtype=np.float64,
    ).reshape(-1)

    scale64 = np.asarray(
        scaler.scale_,
        dtype=np.float64,
    ).reshape(-1)

    scaled_coefficient64 = np.asarray(
        classifier.coef_[0],
        dtype=np.float64,
    ).reshape(-1)

    scaled_intercept64 = float(
        classifier.intercept_[0]
    )

    reconstructed_coefficient64 = (
        scaled_coefficient64
        /
        scale64
    )

    reconstructed_intercept64 = float(
        scaled_intercept64
        -
        np.dot(
            mean64,
            reconstructed_coefficient64,
        )
    )

    # Exact algebraic comparison using a float64 input path.
    pipeline64 = np.asarray(
        probe.decision_function(
            x64
        ),
        dtype=np.float64,
    )

    reconstructed64 = (
        x64
        @
        reconstructed_coefficient64
        +
        reconstructed_intercept64
    )

    mathematical_error = float(
        np.max(
            np.abs(
                pipeline64
                -
                reconstructed64
            )
        )
    )

    assert mathematical_error < 1e-10, (
        "The fitted pipeline and reconstructed raw axis "
        "are not mathematically equivalent."
    )

    stored_coefficient = np.asarray(
        artifacts[
            "coefficient_raw"
        ]
    ).reshape(-1)

    stored_intercept = float(
        np.asarray(
            artifacts[
                "intercept_raw"
            ]
        ).reshape(-1)[0]
    )

    # Confirm that the axis returned by save_axis is the same
    # fitted axis, allowing only storage-dtype rounding.
    assert np.allclose(
        stored_coefficient.astype(
            np.float64
        ),
        reconstructed_coefficient64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored raw coefficient differs from the fitted axis."
    )

    assert np.isclose(
        stored_intercept,
        reconstructed_intercept64,
        rtol=1e-6,
        atol=1e-8,
    ), (
        "The stored raw intercept differs from the fitted axis."
    )

    pipeline32 = np.asarray(
        probe.decision_function(
            x32
        ),
        dtype=np.float64,
    )

    stored_raw32 = np.asarray(
        x32
        @
        stored_coefficient
        +
        stored_intercept,
        dtype=np.float64,
    )

    mixed_precision_error = float(
        np.max(
            np.abs(
                pipeline32
                -
                stored_raw32
            )
        )
    )

    assert np.allclose(
        pipeline32,
        stored_raw32,
        rtol=1e-5,
        atol=1e-4,
    ), (
        "The saved axis differs beyond ordinary "
        "mixed-precision rounding."
    )

    print(
        "Resolved pipeline steps:",
        {
            "scaler": scaler_step_name,
            "classifier": classifier_step_name,
        },
    )

    return (
        mathematical_error,
        mixed_precision_error,
    )


print("Dynamic pipeline-step verifier installed:", True)
print("Source validation observed before execution:", False)
print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)


# Execute the already AST-patched Stage 5A source.
result = get_ipython().run_cell(
    patched_source,
    store_history=False,
)

if result.error_before_exec is not None:
    raise result.error_before_exec

if result.error_in_exec is not None:
    raise result.error_in_exec


print(
    "\n================ STAGE 5A-CPU-R2F "
    "FINISHED ================"
)

print("Target images accessed:", False)
print("Target sealed-label files accessed:", False)

================ STAGE 5A-CPU-R2F DYNAMIC STEP LOOKUP ================
Actual fitted pipeline steps: ['scaler', 'logisticregression']
Dynamic pipeline-step verifier installed: True
Source validation observed before execution: False
Target images accessed: False
Target sealed-label files accessed: False
================ STAGE 5A-CPU SOURCE RECOVERABILITY AND AXIS FREEZE ================

Imported Stage 4G decision:
PASS_CANONICAL_SOURCE_FINALISATION_ADVANCE_TO_SOURCE_RECOVERABILITY_AND_AXIS_FREEZE

Execution device:
cpu

CPU threads:
2

Embedding batch size:
16

DataLoader workers:
1

Frozen source pass rule:
A source passes only if its fixed frozen-representation probe passes both checks: (1) patient-grouped five-fold development out-of-fold eye-level ROC AUC >= 0.70 with patient-bootstrap 95% CI lower bound > 0.55; and (2) the final axis fitted once on all development data achieves held-out validation eye-level ROC AUC >= 0.70 with patient-bootstrap 95% CI lower bound > 0.55. No archi

,dataset,source_partition,images,eyes,patients,negative_images,positive_images
0,DeepDRiD,development,1200,600,300,680,520
1,DeepDRiD,validation,400,200,100,220,180
2,EyePACS_2015,development,3350,3350,1675,2061,1289
3,EyePACS_2015,validation,1118,1118,559,674,444



EyePACS_2015: completed embeddings 4468/4468
EyePACS_2015: embedding extraction complete.

DeepDRiD: completed embeddings 1600/1600
DeepDRiD: embedding extraction complete.

================ MODELLING EyePACS_2015 ================
EyePACS_2015 fold 1 eye AUC: 0.5920
EyePACS_2015 fold 2 eye AUC: 0.6057
EyePACS_2015 fold 3 eye AUC: 0.5855
EyePACS_2015 fold 4 eye AUC: 0.5737
EyePACS_2015 fold 5 eye AUC: 0.5910
EyePACS_2015 development OOF AUC: 0.5907 [0.5702, 0.6098]
Resolved pipeline steps: {'scaler': 'scaler', 'classifier': 'logisticregression'}
EyePACS_2015 float64 mathematical axis error: 7.105e-14
EyePACS_2015 mixed-precision saved-axis error: 1.881e-06
EyePACS_2015 validation AUC: 0.5965 [0.5606, 0.6335]

================ MODELLING DeepDRiD ================
DeepDRiD fold 1 eye AUC: 0.9161
DeepDRiD fold 2 eye AUC: 0.9109
DeepDRiD fold 3 eye AUC: 0.9070
DeepDRiD fold 4 eye AUC: 0.8666
DeepDRiD fold 5 eye AUC: 0.9245
DeepDRiD development OOF AUC: 0.9056 [0.8803, 0.9302]
Resolved pipel

,dataset,development_images,development_eyes,development_patients,development_eye_auc,development_eye_auc_ci_lower_95,development_eye_auc_ci_upper_95,development_pass,validation_images,validation_eyes,validation_patients,validation_eye_auc,validation_eye_auc_ci_lower_95,validation_eye_auc_ci_upper_95,validation_pass,source_pass
0,EyePACS_2015,3350,3350,1675,0.590653,0.57018,0.609792,False,1118,1118,559,0.596473,0.560566,0.633468,False,False
1,DeepDRiD,1200,600,300,0.905588,0.88025,0.930179,True,400,200,100,0.924646,0.876887,0.960957,True,True



================ DEVELOPMENT FOLD STABILITY ================


,dataset,folds,minimum_eye_auc,median_eye_auc,maximum_eye_auc
0,DeepDRiD,5,0.866626,0.910888,0.924491
1,EyePACS_2015,5,0.573731,0.591004,0.605689



================ FROZEN SOURCE AXIS SUMMARY ================


,dataset,development_images,development_eyes,development_patients,raw_axis_l2_norm,intercept_raw,maximum_pipeline_raw_axis_error,axis_npz_sha256,axis_frozen_before_validation,axis_frozen_before_target_access
0,EyePACS_2015,3350,3350,1675,42575.496667,-10.641917,1.880806e-06,373a0298d278bc1e2029fea3d764840de1592143d94161...,True,True
1,DeepDRiD,1200,600,300,21758.101645,0.880034,4.638489e-07,e63ba612c66cced743c7e6311f7e273c2e3815098e29c5...,True,True



================ STAGE 5A DECISION ================
Decision:
PARTIAL_PASS_RETAIN_ONLY_EDGES_FROM_RECOVERABLE_SOURCE

Interpretation:
Only one source passed both frozen recoverability checks. Only the two precommitted edges originating from that source remain authorised. The other source and its two edges are retired before target transfer performance is observed.

Passing sources:
['DeepDRiD']

Retired sources:
['EyePACS_2015']

Authorised edges:
[
  {
    "source": "DeepDRiD",
    "target": "APTOS_2019"
  },
  {
    "source": "DeepDRiD",
    "target": "IDRiD"
  }
]

Authorised next step:
PRECOMMIT_AND_SCORE_ONLY_THE_TWO_EDGES_FROM_THE_PASSING_SOURCE

Execution device:
CPU

Target images accessed:
False

Target sealed-label files accessed:
False

Target performance observed:
False

Source-target transfer performance observed:
False

Stage 5A source recoverability and source-axis freeze completed and sealed.

================ STAGE 5A-CPU-R2F FINISHED ================
Target images ac